# 09 — Criba SIREN k5: dobles con matching (cruces GARANTIZADOS)

C1/C2/C4 del 08 eran disjuntas (0 aristas cruzadas: 0 sin valor, mea culpa).
Estas SÍ: C5 874×2 (1748v/12616e/3694x), C6 509×2 (1018v/7428e/2544x),
C7 G3×2 (4262v/34179e/9119x). 0 = descartado DE VERDAD. Piso alto = kissat.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "jax[tpu]==0.7.2",
                           "-f", "https://storage.googleapis.com/jax-releases/libtpu_releases.html"])
subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "python-sat"])
import jax
print('jax', jax.__version__, jax.devices())
assert any('TPU' in str(d) for d in jax.devices()), 'SIN TPU: Entorno->Cambiar tipo (TPU v5e-1)'


## Archivos + humo


In [ ]:
NNJAX = "#!/usr/bin/env python3\n\"\"\"nn_jax.py \u2014 coloreo probabil\u00edstico Hadwiger-Nelson en JAX (CPU ac\u00e1, TPU en Colab).\n\nModelo (igual idea que lab/colG_pack.py, pero vectorizado + pmap/vmap):\n  logits (R, N, k) -> softmax -> p[r,i,c]. Loss[r] = media sobre aristas de\n  sum_c p[r,i,c]*p[r,j,c]  -  entropy_coef * entrop\u00eda_media (para no colapsar).\n  Adam a mano (sin optax) + jit + vmap sobre restarts. En TPU vmap corre en los\n  8 cores del host.\n\nValor honesto: BUSCA coloreos r\u00e1pido (heur\u00edstica, NO prueba). Si llega a\n0 violaciones con argmax, el grafo es k-coloreable (testigo de descarte para\ncandidatos grandes donde kissat tarda horas). Si queda en piso >0, NO prueba\nnada: el candidato va a kissat.\n\nFormatos .edge: 'e a b' o 'a b', 1-based por defecto (--zero-based para 0-based).\n\nEjemplos:\n  python nn_jax.py --demo\n  python nn_jax.py --edges /tmp/opencode/874.edge --k 5 --restarts 8 --steps 3000\n\"\"\"\n\nimport argparse\nimport json\nimport sys\nimport time\n\nimport jax\nimport jax.numpy as jnp\n\n\ndef load_edges(path, zero_based=False):\n    edges = []\n    seen = set()\n    with open(path) as f:\n        for line in f:\n            p = line.split()\n            if not p or p[0].startswith((\"c\", \"p\", \"#\")):\n                continue\n            if p[0] == \"e\" and len(p) >= 3:\n                a, b = int(p[1]), int(p[2])\n            elif len(p) >= 2:\n                try:\n                    a, b = int(p[0]), int(p[1])\n                except ValueError:\n                    continue\n            else:\n                continue\n            if not zero_based:\n                a, b = a - 1, b - 1\n            if a == b:\n                continue\n            if a > b:\n                a, b = b, a\n            if (a, b) not in seen:\n                seen.add((a, b))\n                edges.append((a, b))\n    return edges\n\n\ndef train(key, ei, ej, n, k, steps, lr, entropy_coef, temp_init):\n    rkey, skey = jax.random.split(key)\n    logits = jax.random.normal(rkey, (n, k)) * temp_init\n\n    m = jnp.zeros_like(logits)\n    v = jnp.zeros_like(logits)\n    b1, b2, eps = 0.9, 0.999, 1e-8\n\n    def loss_fn(lg, beta):\n        p = jax.nn.softmax(lg, axis=-1)\n        same = jnp.sum(p[ei] * p[ej], axis=-1).mean()\n        ent = -(p * jnp.log(p + 1e-12)).sum(-1).mean()\n        return same - beta * ent\n\n    # schedule coseno = lab/colG_pack.py::entropy_beta (el que plant\u00f3 874 en 0 en torch)\n    t_all = jnp.arange(steps)\n    betas = entropy_coef * 0.5 * (1.0 + jnp.cos(jnp.pi * t_all / steps))\n\n    @jax.jit\n    def step(carry, tb):\n        t, beta = tb\n        lg, m, v = carry\n        loss, g = jax.value_and_grad(loss_fn)(lg, beta)\n        m = b1 * m + (1 - b1) * g\n        v = b2 * v + (1 - b2) * g * g\n        mh = m / (1 - b1 ** (t + 1))\n        vh = v / (1 - b2 ** (t + 1))\n        lg = lg - lr * mh / (jnp.sqrt(vh) + eps)\n        return (lg, m, v), loss\n\n    (lg, _, _), losses = jax.lax.scan(step, (logits, m, v), (t_all, betas))\n    p = jax.nn.softmax(lg, axis=-1)\n    hard = jnp.argmax(p, axis=-1)\n    viol = jnp.sum(hard[ei] == hard[ej])\n    return losses[-1], viol, hard\n\n\ndef tabusearch(col, ei, ej, k, max_iters, seed=0, tenure_base=10):\n    \"\"\"TabuCol cl\u00e1sico (Hertz & de Werra 1987, coraz\u00f3n del SOTA 2025/TabuEdges):\n    mueve un v\u00e9rtice en conflicto al color menos conflictivo no-tab\u00fa (o por\n    aspiraci\u00f3n si mejora el r\u00e9cord). Supera al greedy: acepta empeorar para\n    salir de pozos. Numpy/CPU, ms por miles de iters.\"\"\"\n    import numpy as np\n\n    rng = np.random.default_rng(seed)\n    col = list(col)\n    ei_l = list(ei)\n    ej_l = list(ej)\n    n = max(len(col), max(ei_l) + 1, max(ej_l) + 1)\n    adj = [[] for _ in range(n)]\n    for a, b in zip(ei_l, ej_l):\n        adj[a].append(b)\n        adj[b].append(a)\n    cnt = [[0] * k for _ in range(n)]\n    for v in range(n):\n        for u in adj[v]:\n            cnt[v][col[u]] += 1\n    viol = sum(1 for a, b in zip(ei_l, ej_l) if col[a] == col[b])\n    tabu = [[-1] * k for _ in range(n)]\n    best_col, best_v = list(col), viol\n    it = 0\n    while it < max_iters and best_v > 0:\n        # v\u00e9rtices en conflicto\n        conf = [v for v in range(n) if cnt[v][col[v]] > 0]\n        if not conf:\n            best_col, best_v = list(col), 0\n            break\n        v = conf[int(rng.integers(0, len(conf)))]\n        cur = col[v]\n        # mejor movimiento (delta m\u00ednimo), respetando tab\u00fa salvo aspiraci\u00f3n\n        best_d, best_c = None, cur\n        for c in range(k):\n            if c == cur:\n                continue\n            d = cnt[v][c] - cnt[v][cur]\n            if tabu[v][c] > it and viol + d >= best_v:\n                continue\n            if best_d is None or d < best_d:\n                best_d, best_c = d, c\n        if best_d is None:\n            best_c = min((c for c in range(k) if c != cur), key=lambda c: cnt[v][c])\n            best_d = cnt[v][best_c] - cnt[v][cur]\n        col[v] = best_c\n        viol += best_d\n        for u in adj[v]:\n            cnt[u][cur] -= 1\n            cnt[u][best_c] += 1\n        tabu[v][cur] = it + tenure_base + int(rng.integers(0, 10)) + viol\n        if viol < best_v:\n            best_v, best_col = viol, list(col)\n        it += 1\n    return best_v, best_col, it\n\n\ndef slim_rounds(col, ei, ej, k, rounds, radius, cap, seed=0):\n    \"\"\"GC-SLIM 2023 (Schidler & Szeider) simplificado: por ronda, para v\u00e9rtices\n    en conflicto, toma bola radio `radius` (tope `cap` nodos), fija el borde y\n    re-colorea la bola \u00d3PTIMO con SAT (Glucose) minimizando conflictos v\u00eda\n    CardEnc por decisi\u00f3n. Exacto en chico, h\u00edbrido en grande.\"\"\"\n    from pysat.solvers import Glucose3\n    from pysat.card import CardEnc, EncType\n    import numpy as np\n\n    rng = np.random.default_rng(seed)\n    col = list(col)\n    ei_l = [int(x) for x in list(ei)]\n    ej_l = [int(x) for x in list(ej)]\n    n = max(len(col), max(ei_l) + 1, max(ej_l) + 1)\n    adj = [[] for _ in range(n)]\n    for a, b in zip(ei_l, ej_l):\n        adj[a].append(b)\n        adj[b].append(a)\n\n    def viol_of(c):\n        return sum(1 for a, b in zip(ei_l, ej_l) if c[a] == c[b])\n\n    for r in range(rounds):\n        cur = viol_of(col)\n        if cur == 0:\n            return 0, col, r\n        conf = [v for v in range(n) if any(col[v] == col[u] for u in adj[v])]\n        order = rng.permutation(conf).tolist()[:32]\n        improved = False\n        for v in order:\n            seen, frontier = {v}, [v]\n            for _ in range(radius):\n                nxt = []\n                for x in frontier:\n                    for u in adj[x]:\n                        if u not in seen:\n                            seen.add(u)\n                            nxt.append(u)\n                frontier = nxt\n            ball = sorted(seen)\n            if len(ball) > cap:\n                ball = sorted(rng.choice(ball, size=cap, replace=False).tolist())\n            bset = set(ball)\n            idx = {x: i for i, x in enumerate(ball)}\n            m = len(ball)\n\n            def V(x, c):\n                return idx[x] * k + c + 1\n\n            nvars = m * k\n            clauses = []\n            for x in ball:\n                clauses.append([V(x, c) for c in range(k)])\n                for a in range(k):\n                    for b in range(a + 1, k):\n                        clauses.append([-V(x, a), -V(x, b)])\n            for x in ball:  # borde fijo: prohibir colores de vecinos externos\n                for u in adj[x]:\n                    if u not in bset:\n                        clauses.append([-V(x, col[u])])\n            bedges = [(a, b) for a, b in zip(ei_l, ej_l) if a in bset and b in bset]\n            cur_in = sum(\n                1\n                for a, b in zip(ei_l, ej_l)\n                if (a in bset or b in bset) and col[a] == col[b]\n            )\n            if cur_in == 0:\n                continue\n            ind = []\n            for a, b in bedges:\n                e = nvars + len(ind) + 1\n                ind.append(e)\n                for c in range(k):\n                    clauses.append([-V(a, c), -V(b, c), e])\n            for target in range(cur_in - 1, -1, -1):\n                card = CardEnc.equals(\n                    lits=ind, bound=target, encoding=EncType.seqcounter\n                )\n                with Glucose3() as s:\n                    for cl in clauses:\n                        s.add_clause(cl)\n                    for cl in card.clauses:\n                        s.add_clause(cl)\n                    if not s.solve():\n                        continue\n                    model = set(x for x in s.get_model() or [] if x > 0)\n                    for x in ball:\n                        for c in range(k):\n                            if V(x, c) in model:\n                                col[x] = c\n                                break\n                    improved = True\n                    break\n            if improved:\n                break\n        if not improved:\n            return viol_of(col), col, r + 1\n    return viol_of(col), col, rounds\n\n\ndef refine_numpy(\n    hard,\n    ei,\n    ej,\n    k,\n    max_passes=50,\n    kicks=0,\n    kick_size=5,\n    seed=0,\n    tabu_iters=0,\n    slim_rounds_n=0,\n    slim_radius=2,\n):\n    \"\"\"Descenso greedy discreto (numpy, CPU) + basin hopping opcional.\n    Greedy solo remata (~40\u2192~35); con kicks (perturbaci\u00f3n aleatoria +\n    greedy, qued\u00e1ndose con lo mejor) escapa de \u00f3ptimos locales: 12\u21920.\"\"\"\n    import numpy as np\n\n    col0 = np.array(hard, dtype=np.int64).tolist()\n    ei_l = np.array(ei, dtype=np.int64).tolist()\n    ej_l = np.array(ej, dtype=np.int64).tolist()\n    n = max(len(col0), max(ei_l) + 1, max(ej_l) + 1)\n    adj = [[] for _ in range(n)]\n    for a, b in zip(ei_l, ej_l):\n        adj[a].append(b)\n        adj[b].append(a)\n\n    def viol_of(col):\n        return sum(1 for a, b in zip(ei_l, ej_l) if col[a] == col[b])\n\n    def greedy(col):\n        viol = viol_of(col)\n        for _ in range(max_passes):\n            improved = False\n            for v in range(n):\n                cur = col[v]\n                cnt = [0] * k\n                for u in adj[v]:\n                    cnt[col[u]] += 1\n                best_c = min(range(k), key=lambda c: cnt[c])\n                if cnt[best_c] < cnt[cur]:\n                    col[v] = best_c\n                    viol += cnt[best_c] - cnt[cur]\n                    improved = True\n            if not improved:\n                break\n        return col, viol\n\n    rng = np.random.default_rng(seed)\n    best_col, best_v = greedy(list(col0))\n    for _ in range(kicks):\n        cand = list(best_col)\n        for v in rng.integers(0, n, size=kick_size).tolist():\n            cand[v] = int(rng.integers(0, k))\n        cand, v = greedy(cand)\n        if v < best_v:\n            best_col, best_v = cand, v\n            if best_v == 0:\n                break\n    tabu_it = 0\n    if tabu_iters > 0 and best_v > 0:\n        best_v, best_col, tabu_it = tabusearch(\n            best_col, ei_l, ej_l, k, tabu_iters, seed=seed + 999\n        )\n    slim_it = 0\n    if slim_rounds_n > 0 and best_v > 0:\n        best_v, best_col, slim_it = slim_rounds(\n            best_col,\n            ei,\n            ej,\n            k,\n            slim_rounds_n,\n            slim_radius,\n            cap=120,\n            seed=seed + 7777,\n        )\n    return best_v, best_col, tabu_it, slim_it\n\n\ndef run(\n    edges,\n    k,\n    restarts,\n    steps,\n    lr,\n    entropy_coef,\n    temp_init,\n    seed,\n    kicks=0,\n    kick_size=5,\n    tabu_iters=0,\n    slim_rounds_n=0,\n    slim_radius=2,\n):\n    nodes = set()\n    for a, b in edges:\n        nodes.add(a)\n        nodes.add(b)\n    n = max(nodes) + 1\n    ei = jnp.array([a for a, _ in edges], dtype=jnp.int32)\n    ej = jnp.array([b for _, b in edges], dtype=jnp.int32)\n    keys = jax.random.split(jax.random.PRNGKey(seed), restarts)\n    vtrain = jax.vmap(\n        lambda key: train(key, ei, ej, n, k, steps, lr, entropy_coef, temp_init)\n    )\n    t0 = time.time()\n    losses, viols, hards = vtrain(keys)\n    dt = time.time() - t0\n    losses = [float(x) for x in losses]\n    viols = [int(x) for x in viols]\n    best = int(jnp.argmin(jnp.array(viols)))\n    t1 = time.time()\n    refined, refined_col, tabu_it, slim_it = refine_numpy(\n        hards[best],\n        ei,\n        ej,\n        k,\n        kicks=kicks,\n        kick_size=kick_size,\n        seed=seed,\n        tabu_iters=tabu_iters,\n        slim_rounds_n=slim_rounds_n,\n        slim_radius=slim_radius,\n    )\n    refine_s = round(time.time() - t1, 3)\n    return {\n        \"n\": n,\n        \"edges\": len(edges),\n        \"k\": k,\n        \"restarts\": restarts,\n        \"steps\": steps,\n        \"time_s\": round(dt, 3),\n        \"devices\": [str(d) for d in jax.devices()],\n        \"best_restart\": best,\n        \"best_violations\": viols[best],\n        \"refined_violations\": refined,\n        \"refined_coloring\": refined_col,\n        \"refine_s\": refine_s,\n        \"tabu_iters_done\": tabu_it,\n        \"slim_rounds_done\": slim_it,\n        \"best_loss\": losses[best],\n        \"all_violations\": viols,\n    }\n\n\ndef demo():\n    # tri\u00e1ngulo: k2 piso 1/3 (espejo UNSAT), k3 -> 0 (espejo SAT)\n    tri = [(0, 1), (1, 2), (0, 2)]\n    k2 = run(tri, 2, 4, 800, 0.05, 0.02, 1.0, 0)\n    k3 = run(tri, 3, 4, 800, 0.05, 0.02, 1.0, 1)\n    ok = k2[\"best_violations\"] >= 1 and k3[\"best_violations\"] == 0\n    result = {\"triangle_k2\": k2, \"triangle_k3\": k3, \"demo_ok\": bool(ok)}\n    print(json.dumps(result))\n    return 0 if ok else 1\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--demo\", action=\"store_true\")\n    ap.add_argument(\"--edges\", default=None)\n    ap.add_argument(\"--zero-based\", action=\"store_true\")\n    ap.add_argument(\"--k\", type=int, default=5)\n    ap.add_argument(\"--restarts\", type=int, default=8)\n    ap.add_argument(\"--steps\", type=int, default=3000)\n    ap.add_argument(\"--lr\", type=float, default=0.15)\n    ap.add_argument(\"--entropy\", type=float, default=0.05)\n    ap.add_argument(\"--temp\", type=float, default=1.0)\n    ap.add_argument(\"--seed\", type=int, default=0)\n    ap.add_argument(\"--kicks\", type=int, default=0)\n    ap.add_argument(\"--kick-size\", type=int, default=5)\n    ap.add_argument(\"--tabu-iters\", type=int, default=0)\n    ap.add_argument(\"--slim-rounds\", type=int, default=0)\n    ap.add_argument(\"--slim-radius\", type=int, default=2)\n    ap.add_argument(\n        \"--x64\",\n        action=\"store_true\",\n        help=\"float64 (igual que el torch ganador; en TPU emula lento: usar T4)\",\n    )\n    a = ap.parse_args()\n    if a.x64:\n        jax.config.update(\"jax_enable_x64\", True)\n    if a.demo:\n        return demo()\n    if not a.edges:\n        ap.error(\"--edges requerido sin --demo\")\n    edges = load_edges(a.edges, a.zero_based)\n    print(\n        json.dumps(\n            run(\n                edges,\n                a.k,\n                a.restarts,\n                a.steps,\n                a.lr,\n                a.entropy,\n                a.temp,\n                a.seed,\n                kicks=a.kicks,\n                kick_size=a.kick_size,\n                tabu_iters=a.tabu_iters,\n                slim_rounds_n=a.slim_rounds,\n                slim_radius=a.slim_radius,\n            )\n        )\n    )\n    return 0\n\n"
NNSIREN = "#!/usr/bin/env python3\n\"\"\"nn_siren.py \u2014 SIREN por coordenadas para coloreo HN (JAX, CPU ac\u00e1 / TPU all\u00e1).\n\nLiteratura (Mundinger et al. 2024/25): MLPs con activaci\u00f3n seno tienen bias\nespectral a soluciones estructuradas/espacialmente coherentes \u2014 justo lo que\nquiere un grafo unit-distance. Diferencia con nn_jax.py: ah\u00ed cada v\u00e9rtice tiene\nlogits LIBRES (N\u00d7k params, sin geometr\u00eda); ac\u00e1 UNA red coords\u2192colores (params\ncompartidos, la geometr\u00eda manda).\n\nRed: x(2) -> Linear(256, w0=30 seno) -> 2\u00d7 Linear(256, seno) -> Linear(k).\nLoss: media sobre aristas de sum_c p_i p_j - beta*entrop\u00eda (coseno, = colG).\nRefine: greedy + kicks + TabuCol + SLIM reusados de nn_jax (import).\n\nEjemplos:\n  python nn_siren.py --demo\n  python nn_siren.py --vtx 874.vtx.json --edges 874.edge --k 5 --restarts 8 --steps 8000\n\"\"\"\n\nimport argparse\nimport json\nimport sys\nimport time\n\nimport jax\nimport jax.numpy as jnp\n\nfrom nn_jax import load_edges, refine_numpy\n\n\ndef init_siren(key, widths, w0_first=5.0, w0=1.0):\n    params = []\n    keys = jax.random.split(key, len(widths) - 1)\n    for i, (din, dout) in enumerate(zip(widths[:-1], widths[1:])):\n        k1, k2 = jax.random.split(keys[i])\n        bound = jnp.sqrt(6.0 / din) / (w0_first if i == 0 else w0)\n        W = jax.random.uniform(k1, (din, dout), minval=-bound, maxval=bound)\n        b = jnp.zeros((dout,))\n        params.append((W, b))\n    return params\n\n\nW0F = 5.0\n\n\ndef forward(params, x, w0=1.0):\n    h = x\n    for i, (W, b) in enumerate(params):\n        h = h @ W + b\n        if i < len(params) - 1:\n            h = jnp.sin((W0F if i == 0 else w0) * h)\n    return h\n\n\ndef train(key, xs, ei, ej, k, steps, lr, entropy_coef, hidden=256, depth=3):\n    n = xs.shape[0]\n    widths = [2] + [hidden] * depth + [k]\n    params = init_siren(key, widths)\n    m = jax.tree.map(jnp.zeros_like, params)\n    v = jax.tree.map(jnp.zeros_like, params)\n    b1, b2, eps = 0.9, 0.999, 1e-8\n    t_all = jnp.arange(steps)\n    betas = entropy_coef * 0.5 * (1.0 + jnp.cos(jnp.pi * t_all / steps))\n\n    def loss_fn(pm, beta):\n        lg = forward(pm, xs)\n        p = jax.nn.softmax(lg, axis=-1)\n        same = jnp.sum(p[ei] * p[ej], axis=-1).mean()\n        ent = -(p * jnp.log(p + 1e-12)).sum(-1).mean()\n        return same - beta * ent\n\n    @jax.jit\n    def step(carry, tb):\n        t, beta = tb\n        pm, m, v = carry\n        loss, g = jax.value_and_grad(loss_fn)(pm, beta)\n        m = jax.tree.map(lambda a, b: b1 * a + (1 - b1) * b, m, g)\n        v = jax.tree.map(lambda a, b: b2 * a + (1 - b2) * b * b, v, g)\n        upd = jax.tree.map(\n            lambda mi, vi, gi: (\n                lr\n                * (mi / (1 - b1 ** (t + 1)))\n                / (jnp.sqrt(vi / (1 - b2 ** (t + 1))) + eps)\n            ),\n            m,\n            v,\n            g,\n        )\n        pm = jax.tree.map(lambda p, u: p - u, pm, upd)\n        return (pm, m, v), loss\n\n    (pm, _, _), losses = jax.lax.scan(step, (params, m, v), (t_all, betas))\n    lg = forward(pm, xs)\n    hard = jnp.argmax(jax.nn.softmax(lg, axis=-1), axis=-1)\n    viol = jnp.sum(hard[ei] == hard[ej])\n    return losses[-1], viol, hard\n\n\ndef run(\n    vtx,\n    edges,\n    k,\n    restarts,\n    steps,\n    lr,\n    entropy_coef,\n    seed,\n    kicks=0,\n    kick_size=5,\n    tabu_iters=0,\n    slim_rounds_n=0,\n    slim_radius=2,\n    hidden=256,\n    depth=3,\n):\n    import numpy as np\n\n    xs = jnp.array(np.array(vtx, dtype=np.float32))\n    # normalizar coords a [-1,1] para el seno\n    lo, hi = xs.min(), xs.max()\n    xs = 2 * (xs - lo) / (hi - lo + 1e-12) - 1\n    ei = jnp.array([a for a, _ in edges], dtype=jnp.int32)\n    ej = jnp.array([b for _, b in edges], dtype=jnp.int32)\n    n = xs.shape[0]\n    keys = jax.random.split(jax.random.PRNGKey(seed), restarts)\n    results = []\n    t0 = time.time()\n    for r in range(restarts):\n        loss, viol, hard = train(\n            keys[r], xs, ei, ej, k, steps, lr, entropy_coef, hidden, depth\n        )\n        results.append((float(loss), int(viol), np.array(hard)))\n    dt = time.time() - t0\n    results.sort(key=lambda t: t[1])\n    loss, viol, hard = results[0]\n    viols = [v for _, v, _ in results]\n    t1 = time.time()\n    # hard ya es np.array concreto (fuera del jit): refine corre en numpy\n    refined, refined_col, tabu_it, slim_it = refine_numpy(\n        hard,\n        ei,\n        ej,\n        k,\n        kicks=kicks,\n        kick_size=kick_size,\n        seed=seed,\n        tabu_iters=tabu_iters,\n        slim_rounds_n=slim_rounds_n,\n        slim_radius=slim_radius,\n    )\n    refine_s = round(time.time() - t1, 3)\n    return {\n        \"n\": n,\n        \"edges\": len(edges),\n        \"k\": k,\n        \"arch\": f\"siren-{hidden}x{depth}\",\n        \"restarts\": restarts,\n        \"steps\": steps,\n        \"time_s\": round(dt, 3),\n        \"devices\": [str(d) for d in jax.devices()],\n        \"best_violations\": viol,\n        \"refined_violations\": refined,\n        \"refined_coloring\": refined_col,\n        \"refine_s\": refine_s,\n        \"tabu_iters_done\": tabu_it,\n        \"slim_rounds_done\": slim_it,\n        \"best_loss\": loss,\n        \"all_violations\": viols,\n    }\n\n\ndef demo():\n    tri_v = [[0.0, 0.0], [1.0, 0.0], [0.5, 0.8660254]]\n    tri_e = [(0, 1), (1, 2), (0, 2)]\n    k2 = run(tri_v, tri_e, 2, 2, 2000, 0.001, 0.05, 0, hidden=64, depth=2)\n    k3 = run(tri_v, tri_e, 3, 2, 2000, 0.001, 0.05, 1, hidden=64, depth=2)\n    ok = k2[\"best_violations\"] >= 1 and k3[\"best_violations\"] == 0\n    result = {\"triangle_k2\": k2, \"triangle_k3\": k3, \"demo_ok\": bool(ok)}\n    print(json.dumps(result))\n    return 0 if ok else 1\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--demo\", action=\"store_true\")\n    ap.add_argument(\"--vtx\", default=None, help=\"JSON [[x,y],...]\")\n    ap.add_argument(\"--edges\", default=None)\n    ap.add_argument(\"--zero-based\", action=\"store_true\")\n    ap.add_argument(\"--k\", type=int, default=5)\n    ap.add_argument(\"--restarts\", type=int, default=8)\n    ap.add_argument(\"--steps\", type=int, default=8000)\n    ap.add_argument(\"--lr\", type=float, default=0.001)\n    ap.add_argument(\"--entropy\", type=float, default=0.05)\n    ap.add_argument(\"--seed\", type=int, default=0)\n    ap.add_argument(\"--kicks\", type=int, default=0)\n    ap.add_argument(\"--kick-size\", type=int, default=5)\n    ap.add_argument(\"--tabu-iters\", type=int, default=0)\n    ap.add_argument(\"--slim-rounds\", type=int, default=0)\n    ap.add_argument(\"--slim-radius\", type=int, default=2)\n    ap.add_argument(\"--hidden\", type=int, default=256)\n    ap.add_argument(\"--depth\", type=int, default=3)\n    a = ap.parse_args()\n    if a.demo:\n        return demo()\n    if not a.vtx or not a.edges:\n        ap.error(\"--vtx y --edges requeridos sin --demo\")\n    vtx = json.load(open(a.vtx))\n    edges = load_edges(a.edges, a.zero_based)\n    print(\n        json.dumps(\n            run(\n                vtx,\n                edges,\n                a.k,\n                a.restarts,\n                a.steps,\n                a.lr,\n                a.entropy,\n                a.seed,\n                kicks=a.kicks,\n                kick_size=a.kick_size,\n                tabu_iters=a.tabu_iters,\n                slim_rounds_n=a.slim_rounds,\n                slim_radius=a.slim_radius,\n                hidden=a.hidden,\n                depth=a.depth,\n            )\n        )\n    )\n    return 0\n\n"
open('/tmp/nn_jax.py','w').write(NNJAX)
open('/tmp/nn_siren.py','w').write(NNSIREN)
BUNDLE = "[{\"name\": \"C5_874x2+10\", \"n\": 1748, \"edges\": 12616, \"vtx\": [[0.0, 0.0], [1.0, 0.0], [-0.5, 0.8660254037844386], [-0.5, -0.8660254037844386], [-1.0, 0.0], [0.5, -0.8660254037844386], [0.5, 0.8660254037844386], [1.0620468872115023, 0.42072296649368973], [0.8953802205448357, 0.4453024372907488], [0.3333333333333333, 0.31325460539187205], [0.10461977945516428, 0.1320478318988769], [-0.16666666666666666, -0.5527707983925666], [-0.12409377442300477, -0.8414459329873795], [-0.06204688721150239, -0.9980732356833154], [0.8953802205448357, 0.7093981010885027], [0.728713553878169, 0.6848186302914435], [0.4574271077563381, 0.5773502691896258], [0.22871355387816905, 0.39614349569663065], [0.06204688721150239, 0.15662730269593603], [-0.0425728922436619, -0.2886751345948129], [0.0, -0.5773502691896258], [0.12409377442300477, -0.8414459329873795], [0.06204688721150239, -0.7339775718855618], [0.22871355387816905, -0.9734937648862564], [0.27128644612183095, 0.6848186302914435], [0.10461977945516428, 0.4453024372907488], [0.16666666666666666, 0.5527707983925666], [0.0425728922436619, 0.2886751345948129], [0.0425728922436619, -0.2886751345948129], [0.16666666666666666, -0.5527707983925666], [0.10461977945516428, -0.4453024372907488], [0.27128644612183095, -0.6848186302914435], [0.22871355387816905, 0.9734937648862564], [0.06204688721150239, 0.7339775718855618], [0.0, 0.5773502691896258], [-0.0425728922436619, 0.2886751345948129], [0.12409377442300477, -0.2640956637977538], [0.06204688721150239, -0.15662730269593603], [0.22871355387816905, -0.39614349569663065], [0.4574271077563381, -0.5773502691896258], [0.728713553878169, -0.6848186302914435], [0.8953802205448357, -0.7093981010885027], [-0.06204688721150239, 0.9980732356833154], [-0.16666666666666666, 0.5527707983925666], [-0.12409377442300477, 0.2640956637977538], [0.10461977945516428, -0.1320478318988769], [0.3333333333333333, -0.31325460539187205], [0.8953802205448357, -0.4453024372907488], [1.0620468872115023, -0.42072296649368973], [0.16666666666666666, 1.1301210675821924], [0.06204688721150239, 0.9980732356833154], [-0.06204688721150239, 0.7339775718855618], [-0.06204688721150239, 0.15662730269593603], [0.16666666666666666, -0.2395161930006946], [0.39538022054483574, -0.42072296649368973], [0.6666666666666666, -0.5281913275955074], [0.8333333333333334, -0.5527707983925666], [-0.16666666666666666, 1.1301210675821924], [-0.22871355387816905, 0.9734937648862564], [-0.27128644612183095, 0.6848186302914435], [-0.22871355387816905, 0.39614349569663065], [-0.10461977945516428, 0.1320478318988769], [-0.16666666666666666, 0.2395161930006946], [0.22871355387816905, -0.1812067734929951], [0.5, -0.2886751345948129], [0.7907604410896715, -0.31325460539187205], [0.6666666666666666, -0.31325460539187205], [0.9574271077563381, -0.2886751345948129], [-0.4574271077563381, 0.5773502691896258], [-0.3333333333333333, 0.31325460539187205], [-0.39538022054483574, 0.42072296649368973], [-0.22871355387816905, 0.1812067734929951], [0.27128644612183095, -0.10746836110181783], [0.5620468872115024, -0.1320478318988769], [0.43795311278849763, -0.1320478318988769], [0.728713553878169, -0.10746836110181783], [-0.728713553878169, 0.6848186302914435], [-0.6046197794551643, 0.42072296649368973], [-0.5, 0.2886751345948129], [-0.27128644612183095, 0.10746836110181783], [0.2907604410896714, -0.024579470797059095], [0.16666666666666666, -0.024579470797059095], [0.4574271077563381, 0.0], [0.728713553878169, 0.10746836110181783], [0.9574271077563381, 0.2886751345948129], [-0.8953802205448357, 0.4453024372907488], [-0.5620468872115024, 0.1320478318988769], [-0.2907604410896714, 0.024579470797059244], [0.16666666666666666, 0.024579470797059244], [0.43795311278849763, 0.1320478318988769], [0.6666666666666666, 0.31325460539187205], [0.8333333333333334, 0.5527707983925666], [-0.8953802205448357, 0.7093981010885027], [-0.8333333333333334, 0.5527707983925666], [-0.6666666666666666, 0.31325460539187205], [-0.43795311278849763, 0.1320478318988769], [-0.16666666666666666, 0.024579470797059244], [0.5620468872115024, 0.1320478318988769], [0.7907604410896715, 0.31325460539187205], [-1.0620468872115023, 0.42072296649368973], [-0.9574271077563381, 0.2886751345948129], [-0.728713553878169, 0.10746836110181783], [-0.4574271077563381, 0.0], [-0.16666666666666666, -0.024579470797059095], [0.27128644612183095, 0.10746836110181783], [0.5, 0.2886751345948129], [0.6666666666666666, 0.5281913275955076], [-0.728713553878169, -0.10746836110181783], [-0.43795311278849763, -0.1320478318988769], [-0.5620468872115024, -0.1320478318988769], [-0.27128644612183095, -0.10746836110181783], [0.22871355387816905, 0.1812067734929951], [0.39538022054483574, 0.42072296649368973], [-0.9574271077563381, -0.2886751345948129], [-0.6666666666666666, -0.31325460539187205], [-0.7907604410896715, -0.31325460539187205], [-0.5, -0.2886751345948129], [-0.22871355387816905, -0.1812067734929951], [0.16666666666666666, 0.2395161930006946], [-0.8333333333333334, -0.5527707983925666], [-0.6666666666666666, -0.5281913275955074], [-0.39538022054483574, -0.42072296649368973], [-0.16666666666666666, -0.2395161930006946], [-1.0620468872115023, -0.42072296649368973], [-0.8953802205448357, -0.4453024372907488], [-0.3333333333333333, -0.31325460539187205], [-0.10461977945516428, -0.1320478318988769], [-0.8953802205448357, -0.7093981010885027], [-0.728713553878169, -0.6848186302914435], [-0.4574271077563381, -0.5773502691896258], [-0.22871355387816905, -0.39614349569663065], [-0.06204688721150239, -0.15662730269593603], [-0.27128644612183095, -0.6848186302914435], [-0.10461977945516428, -0.4453024372907488], [-0.22871355387816905, -0.9734937648862564], [-0.06204688721150239, -0.7339775718855618], [0.06204688721150239, -0.9980732356833154], [-0.16666666666666666, -1.1301210675821924], [0.16666666666666666, -1.1301210675821924], [1.8953802205448358, 0.4453024372907488], [1.728713553878169, 0.6848186302914435], [1.457427107756338, 0.5773502691896258], [1.228713553878169, 0.39614349569663065], [1.0620468872115023, 0.15662730269593603], [1.0, -0.5773502691896258], [1.228713553878169, -0.9734937648862564], [1.5, 0.8660254037844386], [1.271286446121831, 0.6848186302914435], [1.1666666666666667, -0.5527707983925666], [1.228713553878169, 0.9734937648862564], [1.0620468872115023, 0.7339775718855618], [1.1240937744230048, 0.8414459329873795], [1.0, 0.5773502691896258], [1.1240937744230048, -0.2640956637977538], [1.0620468872115023, -0.15662730269593603], [1.228713553878169, -0.39614349569663065], [1.728713553878169, -0.6848186302914435], [0.9379531127884976, 0.9980732356833154], [1.8953802205448358, -0.4453024372907488], [1.0620468872115023, 0.9980732356833154], [1.0620468872115023, -0.10746836110181783], [1.3953802205448358, -0.42072296649368973], [0.8333333333333334, 1.1301210675821924], [0.771286446121831, 0.9734937648862564], [0.771286446121831, 0.39614349569663065], [0.8953802205448357, 0.1320478318988769], [1.228713553878169, -0.1812067734929951], [1.5, -0.2886751345948129], [1.7907604410896714, -0.31325460539187205], [1.957427107756338, -0.2886751345948129], [0.5425728922436619, 0.5773502691896258], [0.771286446121831, 0.1812067734929951], [1.5620468872115023, -0.1320478318988769], [1.728713553878169, -0.10746836110181783], [1.2907604410896714, -0.024579470797059095], [1.1666666666666667, -0.024579470797059095], [1.957427107756338, 0.2886751345948129], [1.1666666666666667, 0.024579470797059244], [0.10461977945516428, 0.7093981010885027], [0.22871355387816905, 0.4453024372907488], [0.8333333333333334, 0.024579470797059244], [1.1240937744230048, 0.0], [1.7907604410896714, 0.31325460539187205], [-0.06204688721150239, 0.42072296649368973], [0.5425728922436619, 0.0], [0.8333333333333334, -0.024579470797059095], [1.5, 0.2886751345948129], [1.228713553878169, 0.1812067734929951], [1.3953802205448358, 0.42072296649368973], [0.20923955891032855, -0.31325460539187205], [0.771286446121831, -0.1812067734929951], [0.3333333333333333, -0.5281913275955074], [0.8333333333333334, -0.2395161930006946], [-0.06204688721150239, -0.42072296649368973], [0.8953802205448357, -0.1320478318988769], [0.10461977945516428, -0.7093981010885027], [0.771286446121831, -0.39614349569663065], [1.0620468872115023, -0.9980732356833154], [1.1666666666666667, -1.1301210675821924], [0.6046197794551643, 0.9980732356833154], [0.3759062255769952, 0.024579470797059244], [1.228713553878169, 1.550844034075882], [0.9574271077563381, 1.4433756729740645], [0.5620468872115024, 1.0226527064803745], [0.6240937744230047, 0.024579470797059244], [1.0, 1.7320508075688772], [0.771286446121831, 1.550844034075882], [0.6046197794551643, 1.3113278410751874], [0.6666666666666666, 1.4187962021770053], [0.728713553878169, 1.839519168670695], [0.6240937744230047, 1.7074713367718182], [0.5, 1.4433756729740645], [0.4574271077563381, 1.1547005383792517], [0.6240937744230047, 0.6019297399866851], [0.5620468872115024, 0.7093981010885027], [0.728713553878169, 0.46988190808780805], [0.43795311278849763, 1.8640986394677543], [0.3333333333333333, 1.4187962021770053], [1.5620468872115023, 0.4453024372907488], [0.5620468872115024, 1.8640986394677543], [0.43795311278849763, 1.0226527064803745], [0.27128644612183095, 1.839519168670695], [0.22871355387816905, 1.550844034075882], [0.27128644612183095, 1.2621688994810691], [0.39538022054483574, 0.9980732356833154], [1.2907604410896714, 0.5527707983925666], [0.0, 1.7320508075688772], [0.0425728922436619, 1.4433756729740645], [0.10461977945516428, 1.2867483702781284], [0.771286446121831, 0.7585570426826208], [1.228713553878169, 0.7585570426826208], [-0.22871355387816905, 1.550844034075882], [-0.10461977945516428, 1.2867483702781284], [0.0, 1.1547005383792517], [0.7907604410896715, 0.8414459329873795], [0.9574271077563381, 0.8660254037844386], [1.457427107756338, 1.1547005383792517], [-0.39538022054483574, 1.3113278410751874], [1.3333333333333333, 1.4187962021770053], [-0.39538022054483574, 1.5754235048729412], [-0.3333333333333333, 1.4187962021770053], [0.3333333333333333, 0.8906048745814976], [0.6240937744230047, 0.8660254037844386], [-0.5620468872115024, 1.2867483702781284], [0.0425728922436619, 0.8660254037844386], [0.3333333333333333, 0.8414459329873795], [1.0, 1.1547005383792517], [1.1666666666666667, 1.394216731379946], [-0.22871355387816905, 0.7585570426826208], [0.22871355387816905, 0.7585570426826208], [0.8953802205448357, 1.2867483702781284], [0.6666666666666666, 1.1055415967851332], [-0.5620468872115024, 0.4453024372907488], [-0.27128644612183095, 0.42072296649368973], [0.39538022054483574, 0.7339775718855618], [0.5620468872115024, 0.9734937648862564], [-0.39538022054483574, 0.15662730269593603], [0.43795311278849763, 0.7093981010885027], [0.3333333333333333, -0.2640956637977538], [0.6666666666666666, -0.2640956637977538], [-0.39538022054483574, 0.9980732356833154], [-0.6240937744230047, 0.6019297399866851], [-0.6240937744230047, 0.024579470797059244], [0.39538022054483574, 1.5754235048729412], [-0.0425728922436619, 1.4433756729740645], [-0.27128644612183095, 1.2621688994810691], [-0.43795311278849763, 1.0226527064803745], [-0.3759062255769952, 0.024579470797059244], [-0.27128644612183095, 1.839519168670695], [-0.5, 1.4433756729740645], [-0.43795311278849763, 0.7093981010885027], [-0.27128644612183095, 0.46988190808780805], [0.39538022054483574, 0.15662730269593603], [-0.5620468872115024, 1.8640986394677543], [-0.6240937744230047, 1.7074713367718182], [-0.6240937744230047, 1.1301210675821924], [-0.39538022054483574, 0.7339775718855618], [0.5620468872115024, 0.4453024372907488], [-0.5620468872115024, 1.0226527064803745], [0.16666666666666666, 0.33783407618893097], [-0.728713553878169, 1.839519168670695], [-0.771286446121831, 1.550844034075882], [-0.728713553878169, 1.2621688994810691], [-0.6046197794551643, 0.9980732356833154], [-0.6666666666666666, 1.1055415967851332], [0.2907604410896714, 0.5527707983925666], [-1.0, 1.7320508075688772], [-0.9574271077563381, 1.4433756729740645], [-0.8953802205448357, 1.2867483702781284], [-1.228713553878169, 1.550844034075882], [-1.0, 1.1547005383792517], [-0.20923955891032855, 0.8414459329873795], [-0.3333333333333333, 0.8414459329873795], [-0.0425728922436619, 0.8660254037844386], [-1.0620468872115023, 0.9980732356833154], [-0.7907604410896715, 0.8906048745814976], [-1.457427107756338, 1.1547005383792517], [-1.228713553878169, 0.9734937648862564], [-0.9574271077563381, 0.8660254037844386], [-0.6666666666666666, 0.8414459329873795], [0.16666666666666666, 1.394216731379946], [-1.5, 0.8660254037844386], [-1.228713553878169, 0.7585570426826208], [-1.1666666666666667, 0.5527707983925666], [-1.2907604410896714, 0.5527707983925666], [-1.0, 0.5773502691896258], [-0.3333333333333333, 1.1055415967851332], [-1.3333333333333333, 0.31325460539187205], [-1.3953802205448358, 0.42072296649368973], [-0.728713553878169, 0.46988190808780805], [-0.771286446121831, 0.1812067734929951], [-0.6666666666666666, -0.2640956637977538], [-0.3333333333333333, -0.2640956637977538], [0.06204688721150239, 0.42072296649368973], [-0.8953802205448357, 0.1320478318988769], [-1.1666666666666667, -0.5527707983925666], [-1.1240937744230048, -0.8414459329873795], [-1.0620468872115023, -0.9980732356833154], [-0.10461977945516428, 0.7093981010885027], [-0.771286446121831, 0.39614349569663065], [-1.0, -0.5773502691896258], [-0.771286446121831, -0.39614349569663065], [-0.5425728922436619, -0.5773502691896258], [-0.10461977945516428, -0.7093981010885027], [-1.1240937744230048, 0.2640956637977538], [-0.8953802205448357, -0.1320478318988769], [0.06204688721150239, -0.42072296649368973], [-1.0620468872115023, 0.15662730269593603], [-1.1666666666666667, 1.1301210675821924], [-1.228713553878169, 0.39614349569663065], [-0.771286446121831, -0.1812067734929951], [-1.5, 0.2886751345948129], [-0.8333333333333334, -0.024579470797059095], [-1.8953802205448358, 0.4453024372907488], [-1.7907604410896714, 0.31325460539187205], [-0.8333333333333334, 0.024579470797059244], [-1.1666666666666667, 0.024579470797059244], [-0.20923955891032855, 0.31325460539187205], [-1.957427107756338, 0.2886751345948129], [-1.728713553878169, 0.10746836110181783], [-1.457427107756338, 0.0], [-1.1666666666666667, -0.024579470797059095], [-0.3333333333333333, 0.5281913275955076], [-1.4379531127884977, -0.1320478318988769], [-1.957427107756338, -0.2886751345948129], [-1.5, -0.2886751345948129], [-1.228713553878169, -0.1812067734929951], [-0.8333333333333334, 0.2395161930006946], [-1.3953802205448358, -0.42072296649368973], [-1.728713553878169, -0.6848186302914435], [-1.457427107756338, -0.5773502691896258], [-1.228713553878169, -0.39614349569663065], [-1.5, -0.8660254037844386], [-1.271286446121831, -0.6848186302914435], [-1.228713553878169, -0.9734937648862564], [-1.0620468872115023, -0.7339775718855618], [-0.9379531127884976, -0.9980732356833154], [0.5620468872115024, -0.4453024372907488], [0.27128644612183095, -0.42072296649368973], [-0.39538022054483574, -0.7339775718855618], [-0.5620468872115024, -0.9734937648862564], [-0.5620468872115024, -1.8640986394677543], [0.39538022054483574, -0.15662730269593603], [-0.27128644612183095, -0.46988190808780805], [-0.43795311278849763, -0.7093981010885027], [-0.5, -1.4433756729740645], [-0.3333333333333333, -1.4187962021770053], [-0.22871355387816905, -1.550844034075882], [0.0, -1.7320508075688772], [-0.27128644612183095, -1.2621688994810691], [0.22871355387816905, -1.550844034075882], [-0.6240937744230047, -0.6019297399866849], [-0.3333333333333333, 0.2640956637977538], [-0.5, 0.024579470797059244], [-0.5620468872115024, -0.7093981010885027], [-0.43795311278849763, -0.9734937648862564], [-0.10461977945516428, -1.2867483702781284], [0.16666666666666666, -1.394216731379946], [0.3333333333333333, -1.4187962021770053], [-0.6666666666666666, 0.2640956637977538], [-0.728713553878169, -0.46988190808780805], [-0.6046197794551643, -0.7339775718855618], [0.0, -1.1547005383792517], [-0.0425728922436619, -0.8660254037844386], [0.22871355387816905, -0.7585570426826208], [-0.6240937744230047, -0.8660254037844386], [-0.3333333333333333, -0.8414459329873795], [-1.3953802205448358, -0.15662730269593603], [-0.6666666666666666, -0.8414459329873795], [0.2907604410896714, -0.5527707983925666], [-0.9574271077563381, -0.8660254037844386], [-0.22871355387816905, -0.7585570426826208], [-1.457427107756338, -1.1547005383792517], [-1.0, -1.1547005383792517], [-0.3333333333333333, -0.6265092107837441], [-0.8953802205448357, -1.2867483702781284], [-0.6666666666666666, -1.1055415967851332], [-1.228713553878169, -1.550844034075882], [-0.9574271077563381, -1.4433756729740645], [-0.5620468872115024, -1.6000029756700005], [0.43795311278849763, -0.9734937648862564], [0.728713553878169, -0.46988190808780805], [0.5620468872115024, -0.7093981010885027], [0.5, -1.4433756729740645], [0.6240937744230047, -1.7074713367718184], [0.728713553878169, -1.839519168670695], [0.6240937744230047, -1.1301210675821924], [1.228713553878169, -1.550844034075882], [0.6666666666666666, 0.2640956637977538], [0.5, 0.024579470797059244], [0.43795311278849763, -0.7093981010885027], [0.5620468872115024, -0.9734937648862564], [1.3333333333333333, -1.4187962021770053], [0.3333333333333333, 0.2640956637977538], [0.39538022054483574, -0.7339775718855618], [1.0, -1.1547005383792517], [1.457427107756338, -1.1547005383792517], [0.9574271077563381, -0.8660254037844386], [0.20923955891032855, -0.8414459329873795], [-0.39538022054483574, -0.15662730269593603], [0.3333333333333333, -0.8414459329873795], [-0.5620468872115024, -0.4453024372907488], [0.3333333333333333, -0.8906048745814976], [-0.39538022054483574, -1.5754235048729412], [0.27128644612183095, -1.2621688994810691], [0.27128644612183095, -1.839519168670695], [0.5620468872115024, -1.8640986394677543], [0.125, -0.4841229182759271], [-0.7317627457812106, 0.031587850897493115], [0.10676274578121059, -1.4839566057252744], [-0.75, -0.9682458365518543], [0.9817627457812106, -0.9998336874493473], [0.1432372542187894, 0.5157107691734202], [0.8506093959854343, 0.39817091578874836], [0.6928765775201509, 0.33899079969018714], [0.4503606766986385, 0.176719969483233], [0.2650129329409796, -0.04865083246606342], [0.15261492529237544, -0.3179322324251918], [0.1442727108416574, -0.7752832643125013], [0.2467756787221902, -1.0484845199154107], [0.4237812080291313, -1.2804647498534238], [0.5538991012020394, -1.387475319605598], [0.5650218140583642, 0.5700745055132218], [0.4310879658569811, 0.4678803155198072], [0.24574022209932211, 0.24250951357051073], [0.13334221445071792, -0.02677188638861755], [0.1034641594472211, -0.3170357083102133], [0.4045084971874737, -0.9893044038168497], [0.6409453132693895, -1.1603114694263446], [0.5346263903603817, -1.0963149735690239], [0.7964150020235516, -1.225204489398644], [0.030839246570185292, 0.24642936921429182], [0.0009611915666883167, -0.04383445270730387], [0.003224321277809805, 0.08023868336355659], [0.022497032119467224, -0.21092166267301768], [0.3020055293069411, -0.7161031482139402], [0.5384423453888568, -0.8871102138234351], [0.6939120341430188, -0.9520032337957343], [-0.14616628273675578, 0.47840959915230497], [-0.17604433774025263, 0.18814577723070913], [-0.15450849718747373, 0.02105856726499541], [-0.0520055293069411, -0.25214268833791414], [0.36143681608191575, -0.6551299838854221], [0.25511789317290806, -0.5911334880281013], [0.5169065048360779, -0.7200230038577213], [0.8047572164742696, -0.7678534575113344], [1.0941607534298148, -0.7305522874902189], [1.2518935718950983, -0.6713721713916577], [-0.4124811538221686, 0.359152842840204], [-0.39094531326938964, 0.1920656328744903], [-0.2884423453888569, -0.08113562272841918], [-0.11143681608191576, -0.3131158526664322], [0.28046968875416206, -0.5490159382482264], [0.5683204003923537, -0.5968463919018394], [0.857723937347899, -0.5595452218807241], [1.1240388084333117, -0.4402884655686232], [1.257972656634695, -0.3380942755752087], [-0.2762841759096639, 0.5854201689044792], [-0.3038991012020394, 0.4192294830537437], [-0.2846263903603818, 0.12806913701716952], [-0.005117893172908117, -0.377112348523753], [0.3867886116631698, -0.6130124341055471], [0.6746393233013616, -0.66084288775916], [0.9640428602569067, -0.6235417177380449], [1.1217756787221902, -0.5643616016394837], [-0.5679508425763306, 0.4240458628125034], [-0.5464150020235516, 0.25695865284678954], [-0.4439120341430189, -0.016242602756119922], [-0.2669065048360779, -0.24822283269413298], [-0.030469688754162078, -0.4192298983036278], [-0.1367886116631698, -0.35523340244630724], [0.41285071163819165, -0.5319533719295401], [0.7022542485937369, -0.49465220190842485], [0.9685691196791494, -0.375395445596324], [0.8599870670590203, -0.4354720858098637], [1.1025029678805327, -0.2732012556029095], [-0.5547572164742696, -0.20039237904051999], [-0.3183204003923537, -0.37139944465001484], [-0.42463932330136145, -0.30740294879269414], [-0.16285071163819165, -0.43629246462231414], [0.41440353695554516, -0.4468217482548118], [0.680718408040958, -0.32756499194271105], [0.8146522562423409, -0.22537080194929646], [-0.8441607534298147, -0.23769354906163528], [-0.607723937347899, -0.4087006146711301], [-0.7140428602569067, -0.34470411881380936], [-0.45225424859373686, -0.47359363464342946], [-0.16440353695554513, -0.5214240882970425], [0.3913148710854127, -0.3648661619638263], [0.28273281846528353, -0.42494280217736596], [0.5252487192867958, -0.2626719719704118], [0.7105964630444549, -0.03730117002111536], [0.8229944706930589, 0.23198022993801304], [-0.8740388084333116, -0.527957370983231], [-0.4307184080409579, -0.6406808446091432], [-0.1413148710854127, -0.603379674588028], [0.25893384820138315, -0.38192872828251256], [0.4442815919590421, -0.1565579263332161], [0.5566795996076462, 0.11272347362591228], [0.5865576546111432, 0.402987295547508], [-1.0018935718950983, -0.2968736651601964], [-0.8717756787221903, -0.4038842349123706], [-0.6099870670590203, -0.5327737507419905], [-0.3221363554208287, -0.5806042043956036], [-0.03273281846528353, -0.5433030343744883], [0.5528636445791713, -0.09648128611967643], [0.6652616522277754, 0.17280011383945193], [-1.007972656634695, -0.6301515609766455], [-0.8525029678805328, -0.6950445809489448], [-0.5646522562423409, -0.7428750346025579], [-0.27524871928679584, -0.7055738645814424], [-0.008933848201383135, -0.5863171082693417], [-0.11751590082151231, -0.6463937484828813], [0.310347743757659, -0.2587521163266307], [0.42274575140626314, 0.010529283632497706], [-0.46059646304445484, -0.9309446665307389], [-0.3028636445791713, -0.8717645504321778], [-0.06034774375765902, -0.7094937202252236], [0.2373980076486042, -0.21484151831679874], [0.267276062652101, 0.07542230360479696], [-0.572994470693059, -1.2002260664898674], [-0.3066795996076463, -1.0809693101777664], [-0.4152616522277755, -1.1410459503913062], [-0.17274575140626314, -0.978775120184352], [0.012601992351395808, -0.7534043182350555], [0.1548780550034968, -0.1938590963543314], [-0.3365576546111431, -1.371233132099362], [-0.20262380640976, -1.2690389421059478], [-0.01727606265210106, -1.0436681401566512], [0.09512194499650313, -0.7743867401975227], [-0.6006093959854343, -1.3664167523406026], [-0.44287657752015086, -1.3072366362420416], [-0.015012932940979606, -0.9195950040857909], [0.09738507470762454, -0.6503136041266624], [-0.3150218140583642, -1.5383203420650762], [-0.18108796585698114, -1.4361261520716615], [0.004259777900677886, -1.2107553501223651], [0.11665778554928204, -0.9414739501632368], [0.1465358405527789, -0.6512101282416409], [0.2191607534298147, -1.214675205766146], [0.2490388084333117, -0.9244113838445503], [0.3961662827367558, -1.446655435704159], [0.4260443377402526, -1.1563916137825634], [0.6624811538221685, -1.3273986793920585], [0.5262841759096639, -1.5536660054563332], [0.8179508425763306, -1.3922916993643577], [1.5678765775201509, 0.8231137179661143], [1.306087965856981, 0.9520032337957343], [1.1207402220993221, 0.7266324318464378], [1.008342214450718, 0.4573510318873095], [0.9807272891583425, 0.29116034603657426], [1.2795084971874737, -0.5051814855409226], [1.5159453132693896, -0.6761885511504174], [1.4096263903603818, -0.6121920552930965], [1.6714150020235516, -0.7410815711227169], [1.0182372542187894, 0.9998336874493473], [0.9058392465701853, 0.7305522874902189], [0.8782243212778097, 0.5643616016394837], [1.413442345388857, -0.402987295547508], [1.8567627457812106, -0.5157107691734202], [0.7288337172632442, 0.9625325174282319], [0.6989556622597474, 0.6722686955066363], [0.7012187919708687, 0.7963418315774967], [0.7204915028125263, 0.5051814855409226], [1.1301178931729081, -0.10701056975217416], [1.3919065048360781, -0.23590008558179412], [1.6797572164742693, -0.28373053923540725], [1.9691607534298143, -0.24642936921429182], [0.4625188461778314, 0.8432757611161312], [1.4433204003923537, -0.11272347362591228], [1.9990388084333117, 0.04383445270730387], [0.5987158240903361, 1.0695430871804064], [0.5711008987979606, 0.9033524013296708], [1.1063189229090076, -0.06399649585732074], [1.5496393233013614, -0.176719969483233], [0.3070491574236694, 0.9081687810884306], [0.32858499797644836, 0.7410815711227169], [0.6080934951639222, 0.23590008558179412], [0.8445303112458379, 0.06489301997229924], [1.2878507116381919, -0.04783045365361301], [1.5772542485937369, -0.010529283632497706], [1.8435691196791495, 0.1087274726796031], [1.9775029678805327, 0.21092166267301768], [0.32024278352573043, 0.28373053923540725], [0.7121492883618084, 0.04783045365361301], [1.5557184080409578, 0.1565579263332161], [1.689652256242341, 0.2587521163266307], [1.875, 0.4841229182759271], [1.2663148710854124, 0.11925675631210082], [1.1577328184652835, 0.059180116098561164], [1.4002487192867958, 0.22145094630551537], [1.585596463044455, 0.4468217482548118], [1.697994470693059, 0.7161031482139402], [1.133933848201383, 0.10219418999341455], [1.3192815919590422, 0.32756499194271105], [1.461557654611143, 0.8871102138234351], [-0.12689357189509828, 0.18724925311573073], [0.10954324418681749, 0.016242187506235833], [0.8422671815347164, -0.059180116098561164], [1.108582052620129, 0.06007664021353968], [1.427863644579171, 0.38764163215625064], [1.5402616522277757, 0.6569230321153791], [-0.13297265663469482, -0.14602864270071844], [0.5997512807132042, -0.22145094630551537], [0.8660661517986169, -0.10219418999341455], [1.2977457514062631, 0.49465220190842485], [1.142276062652101, 0.5595452218807241], [0.4597383477722245, -0.6569230321153791], [0.8876019923513958, -0.26928139995912836], [0.67237619359024, -0.7849160238300206], [0.9701219449965031, -0.29026382192159567], [0.27439060401456566, -0.8822938340646754], [0.5407054750999783, -0.7630370777525747], [0.9723850747076246, -0.1661906858507353], [0.5599781859416358, -1.054197423789149], [0.9916577855492821, -0.4573510318873095], [1.2711662827367558, -0.9625325174282319], [1.5374811538221687, -0.8432757611161312], [1.6929508425763309, -0.9081687810884306], [0.1708521795111648, 0.6819014550241554], [0.44201846224792063, -0.28063106240407654], [0.44932522007577047, 1.4677140029691547], [0.26397747631811147, 1.242343201019858], [0.15157946866950742, 0.9730618010607297], [0.12170141366601044, 0.6827979791391341], [0.6591825674881789, -0.16047778197699722], [0.16147450843757882, 1.5155444566227676], [0.019198445785477802, 0.9559992347420433], [-0.1279290285179663, 1.478243286601652], [-0.1555439538103418, 1.312052600750917], [-0.13627124296868431, 1.0208922547143429], [-0.03376827508815161, 0.7476909991114332], [0.37967407030070527, 0.3447037035639253], [0.2733551473916976, 0.4087001994212461], [0.5351437590548672, 0.27981068359162614], [-0.3942438996033791, 1.3589865302895516], [-0.27020509117006747, 0.9186980647209282], [0.03691833130978169, 0.5797072650307409], [1.2762099108534841, 0.6617394118741386], [-0.28566184698324987, 1.4190631705030912], [-0.1638861682610597, 0.8547015688636073], [-0.5281777478047621, 1.256792340296137], [-0.42567477992422953, 0.9835910846932275], [-0.24866925061728842, 0.7516108547552143], [-0.012232434535372705, 0.5806037891457194], [-0.11855135744438039, 0.6446002850030402], [-0.7135254915624212, 1.0314215383468404], [-0.40640206908257204, 0.6924307386566534], [0.8328895104611305, 0.7744628855000508], [-0.5894866831291096, 0.5911330727782173], [-0.43401699437494745, 0.526240052805918], [0.40955212530420204, 0.634967525485521], [0.30097007268407294, 0.5748908852719814], [0.5434859735055853, 0.7371617154789356], [0.8412317249118484, 1.2318139173873603], [-0.8558015542145222, 0.47187631646611616], [0.6047949088299326, 1.4028209829968554], [-0.983656317676309, 0.7029600222891507], [-0.8535384245034008, 0.5959494525369768], [0.25181930683891857, 0.5757874093869599], [-0.9897354024159054, 0.3696821264727019], [-0.8342657136617433, 0.30478910650040253], [-0.25701146506800643, 0.29425982286790486], [0.009303406017406276, 0.4135165791800057], [-0.09927864660272297, 0.353439938966466], [0.44098300562505255, 1.0103629710818451], [0.4708610606285493, 1.3006267930034408], [-0.4423592088256654, 0.06888902091860831], [-0.04211048953886953, 0.29033996722412375], [0.17311530922228627, 0.805974591095016], [-0.582372141766645, -0.3665830648912553], [-0.3160572706812323, -0.24732630857915447], [0.11562232892641404, 0.34952008332268486], [-0.29678455983957475, -0.5384866546157286], [0.1647730947715684, 0.34862355920770643], [0.5445214301284533, -0.553832318006986], [0.5743994851319502, -0.2635684960853903], [0.83618809679512, -0.3924580119150103], [-0.7041478204888353, 0.19777853674822832], [-0.712490034939553, -0.25957249513908115], [-0.43298153775207937, -0.7647539806800037], [-0.29174093172284643, 1.0857852746866419], [-0.6110225236818886, 0.7582202827439309], [-0.7234205313304926, 0.4889388827848026], [-0.7532985863339895, 0.19867506086320685], [-0.21581743251182103, -0.6446007002529245], [-1.0029290285179664, 0.9941203683257251], [-1.0112712429686843, 0.5367693364384157], [-0.6016448526083025, -0.07542271885468106], [-0.3398562409451327, -0.20431223468430104], [0.3951308261138877, -0.15566140221823757], [-1.269243899603379, 0.8748636120136245], [-1.2477080590506002, 0.7077764020479106], [-0.9681995618631264, 0.2025949165069881], [-0.8380816686902183, 0.09558434675481386], [-0.5762930570270485, -0.03330516907480622], [0.4012099108534842, 0.1776164935982115], [-0.8618806389541187, 0.13859842064966724], [0.107280114475696, -0.10783094856462465], [-1.4031777478047622, 0.7726694220202098], [-1.3006747799242295, 0.4994681664173004], [-1.1236692506172885, 0.26748793647928726], [-0.8872324345353727, 0.09648087086979233], [-0.9935513574443804, 0.16047736672711319], [0.11180637389793897, 0.14031532357709628], [-1.5885254915624212, 0.5472986200709133], [-1.4115199622554802, 0.3153183901329002], [-1.175083146173564, 0.14431132452340542], [-1.7009234992110251, 0.27801722011178504], [-1.3090169943749475, 0.04211713452999082], [-0.46544787469579796, 0.15084460720959392], [-0.5740299273159271, 0.09076796699605423], [-0.3315140264944147, 0.2530387972030085], [-1.2874811538221687, -0.12497007543572303], [-0.9980776168666233, -0.08766890541460774], [-1.7092657136617433, -0.17933381177552463], [-1.4214150020235516, -0.22716426542913756], [-1.1320114650680064, -0.18986309540802226], [-0.8656965939825937, -0.07060633909592144], [-1.6067627457812106, -0.452535067378434], [-1.4297572164742693, -0.6845152973164472], [-1.272024398008986, -0.6253351812178859], [-1.0295084971874737, -0.46306435101093174], [-0.7018846907777138, 0.32185167281908883], [-1.1933204003923537, -0.855522362925942], [-1.2996393233013614, -0.7915258670686214], [-1.1717845598395746, -1.0226095728916558], [-0.7401049602319286, -0.42576318098981636], [-0.7102269052284317, -0.13549935906822067], [-0.6376019923513958, -0.698964436592726], [-0.33047856987154667, -1.0379552362829132], [-0.3006005148680498, -0.7476914143613174], [-0.038811903204880004, -0.8765809301909374], [-0.02439060401456561, -0.08595200248717878], [-0.7223850747076246, -0.802055150701119], [-0.3211008987979606, -1.8715982378815248], [-0.3099781859416358, 0.08595158723729475], [-0.4704915028125263, -1.473427322092777], [-0.619882106827092, -1.0752564063040284], [0.3768935718950983, -1.1554950896675849], [-0.9864368160819158, -0.7972387709423594], [-0.8563189229090077, -0.9042493406945334], [-0.5945303112458379, -1.0331388565241537], [0.1404567558131825, -0.9844880240580901], [0.38297265663469476, -0.8222171938511359], [-1.4429508425763304, -0.060077055463423644], [-1.141906504836078, -0.7323457509700602], [-0.46214928836180835, -1.0160762902054674], [-1.7191607534298148, -0.7218164673375624], [-1.3272542485937369, -0.9577165529193568], [-0.5922671815347165, -0.9090657204532931], [-1.7490388084333117, -1.012080289259158], [-1.5935691196791495, -1.0769733092314573], [-1.3057184080409578, -1.1248037628850704], [-0.8585820526201292, -1.0283224767653938], [-0.6160661517986169, -0.8660516465584397], [-0.9077328184652836, -1.0274259526504155], [-0.20973834777222447, -0.31132280443647514], [-1.7275029678805327, -1.179167499224872], [-1.439652256242341, -1.2269979528784851], [-0.8839338482013831, -1.0704400265452687], [-0.9925159008215122, -1.1305166667588085], [-1.447994470693059, -1.6843489847657944], [-1.2902616522277757, -1.6251688686672334], [-1.0477457514062631, -1.4628980384602792], [-0.7201219449965031, -0.6779820146302585], [-0.892276062652101, -1.5277910584325785], [-1.3178765775201509, -1.7913595545179686], [-1.056087965856981, -1.9202490703475885], [-0.8707402220993221, -1.6948782683982924], [-0.7583422144507179, -1.4255968684391638], [-0.7284641594472211, -1.135333046517568], [-0.7682372542187894, -1.9680795240012015], [-0.4788337172632442, -1.9307783539800862], [-0.21251884617783146, -1.8115215976679855], [-0.3487158240903361, -2.037788923732261], [0.832372141766645, -0.6016627716605989], [0.5660572706812323, -0.7209195279726996], [0.13437767107358597, -1.317765919874539], [0.1044996160700891, -1.608029741796135], [0.5356618469832499, -2.387309007054945], [0.5467845598395747, -0.42975918193612556], [0.11510496023192858, -1.026605573837965], [0.08522690522843161, -1.3168693957595607], [0.3862712429686843, -1.9891380912661971], [0.7781777478047621, -2.2250381768479914], [0.5202050911700674, -1.8869439012727824], [0.4138861682610597, -1.8229474054154615], [0.6756747799242295, -1.951836921245082], [0.9635254915624212, -1.9996673748986946], [0.23688063895411873, -1.5909671754774486], [0.49866925061728845, -1.7198566913070688], [0.7865199622554803, -1.7676871449606815], [1.0759234992110254, -1.7303859749395665], [-0.409182567488179, -0.807768054574857], [-0.12967407030070524, -1.3129495401157796], [-0.29452143012845333, -0.41441351854486824], [-0.3243994851319502, -0.7046773404664641], [-0.02335514739169753, -1.3769460359731003], [0.2130816686902183, -1.5479531015825954], [0.9458056060381171, -1.6233754051873923], [1.1035384245034008, -1.564195289088831], [-0.58618809679512, -0.5757878246368439], [-0.2851437590548673, -1.2480565201434803], [0.6840169943749475, -1.4944858893577724], [1.0842657136617433, -1.273034943052257], [0.5070114650680064, -1.2625056594197592], [0.6923592088256654, -1.0371348574704626], [-0.0018193068389186589, -1.5440332459388142], [-1.0201308261138877, -1.2967073526095438], [0.34927864660272295, -1.32168577551832], [0.647024398008986, -0.8270335736098954], [-1.0262099108534841, -1.629985248425993], [-0.29348597350558525, -1.7054075520307899], [-0.027171102420172472, -1.586150795718689], [-0.5912317249118484, -2.2000597539392146], [-0.19098300562505255, -1.9786088076336994], [-0.35479490882993253, -2.3710668195487097], [-0.22086106062854935, -2.268872629555295], [-0.03551331687089051, -2.0435018276059984], [0.07688469077771372, -1.7742204276468703], [-0.19932522007577047, -2.435959839521009], [-0.013977476318111451, -2.210589037571712], [0.20092349921102537, -2.2145088932154935], [0.9794996160700892, -1.1239068235202077], [0.9602269052284317, -0.8327464774836336], [1.2612712429686843, -1.50501517299027], [1.4977080590506002, -1.6760222385997647], [1.6531777478047622, -1.7409152585720642], [1.1118806389541187, -1.1068442572015218], [1.6615199622554802, -1.2835642266847545], [1.9509234992110251, -1.2462630566636392], [0.4658174325118212, -0.32364513629892994], [1.1372324345353728, -1.0647267074216467], [0.5804785698715468, 0.06970939973105887], [0.5506005148680498, -0.2205544221905369], [0.8516448526083025, -0.8928231176971733], [1.0880816686902184, -1.0638301833066681], [1.9785384245034008, -1.080072370812904], [0.28881190320488, -0.09166490636091684], [0.5898562409451328, -0.7639336018675532], [0.8262930570270485, -0.934940667477048], [1.5590169943749475, -1.0103629710818451], [1.9592657136617433, -0.7889120247763297], [0.142719885524304, -0.8604148879872295], [1.3820114650680064, -0.778382741143832], [0.138193626102061, -1.1085611601289505], [0.7154478746957981, -1.1190904437614482], [-0.14513082611388775, -0.8125844343336167], [1.2242786466027231, -0.8375628572423932], [-0.15120991085348415, -1.1458623301500657], [0.5417409317228464, -2.054031111238496], [0.9734205313304926, -1.4571847193366567], [1.519243899603379, -1.8431094485654789], [1.0, 0.0], [2.0, 0.0], [0.5, 0.8660254037844386], [0.5, -0.8660254037844386], [0.0, 0.0], [1.5, -0.8660254037844386], [1.5, 0.8660254037844386], [2.062046887211502, 0.42072296649368973], [1.8953802205448356, 0.4453024372907488], [1.3333333333333333, 0.31325460539187205], [1.1046197794551642, 0.1320478318988769], [0.8333333333333334, -0.5527707983925666], [0.8759062255769953, -0.8414459329873795], [0.9379531127884976, -0.9980732356833154], [1.8953802205448356, 0.7093981010885027], [1.728713553878169, 0.6848186302914435], [1.457427107756338, 0.5773502691896258], [1.228713553878169, 0.39614349569663065], [1.0620468872115023, 0.15662730269593603], [0.9574271077563381, -0.2886751345948129], [1.0, -0.5773502691896258], [1.1240937744230048, -0.8414459329873795], [1.0620468872115023, -0.7339775718855618], [1.228713553878169, -0.9734937648862564], [1.271286446121831, 0.6848186302914435], [1.1046197794551642, 0.4453024372907488], [1.1666666666666667, 0.5527707983925666], [1.042572892243662, 0.2886751345948129], [1.042572892243662, -0.2886751345948129], [1.1666666666666667, -0.5527707983925666], [1.1046197794551642, -0.4453024372907488], [1.271286446121831, -0.6848186302914435], [1.228713553878169, 0.9734937648862564], [1.0620468872115023, 0.7339775718855618], [1.0, 0.5773502691896258], [0.9574271077563381, 0.2886751345948129], [1.1240937744230048, -0.2640956637977538], [1.0620468872115023, -0.15662730269593603], [1.228713553878169, -0.39614349569663065], [1.457427107756338, -0.5773502691896258], [1.728713553878169, -0.6848186302914435], [1.8953802205448356, -0.7093981010885027], [0.9379531127884976, 0.9980732356833154], [0.8333333333333334, 0.5527707983925666], [0.8759062255769953, 0.2640956637977538], [1.1046197794551642, -0.1320478318988769], [1.3333333333333333, -0.31325460539187205], [1.8953802205448356, -0.4453024372907488], [2.062046887211502, -0.42072296649368973], [1.1666666666666667, 1.1301210675821924], [1.0620468872115023, 0.9980732356833154], [0.9379531127884976, 0.7339775718855618], [0.9379531127884976, 0.15662730269593603], [1.1666666666666667, -0.2395161930006946], [1.3953802205448358, -0.42072296649368973], [1.6666666666666665, -0.5281913275955074], [1.8333333333333335, -0.5527707983925666], [0.8333333333333334, 1.1301210675821924], [0.771286446121831, 0.9734937648862564], [0.728713553878169, 0.6848186302914435], [0.771286446121831, 0.39614349569663065], [0.8953802205448357, 0.1320478318988769], [0.8333333333333334, 0.2395161930006946], [1.228713553878169, -0.1812067734929951], [1.5, -0.2886751345948129], [1.7907604410896716, -0.31325460539187205], [1.6666666666666665, -0.31325460539187205], [1.957427107756338, -0.2886751345948129], [0.5425728922436619, 0.5773502691896258], [0.6666666666666667, 0.31325460539187205], [0.6046197794551642, 0.42072296649368973], [0.771286446121831, 0.1812067734929951], [1.271286446121831, -0.10746836110181783], [1.5620468872115025, -0.1320478318988769], [1.4379531127884977, -0.1320478318988769], [1.728713553878169, -0.10746836110181783], [0.27128644612183095, 0.6848186302914435], [0.3953802205448357, 0.42072296649368973], [0.5, 0.2886751345948129], [0.728713553878169, 0.10746836110181783], [1.2907604410896714, -0.024579470797059095], [1.1666666666666667, -0.024579470797059095], [1.457427107756338, 0.0], [1.728713553878169, 0.10746836110181783], [1.957427107756338, 0.2886751345948129], [0.10461977945516432, 0.4453024372907488], [0.4379531127884976, 0.1320478318988769], [0.7092395589103286, 0.024579470797059244], [1.1666666666666667, 0.024579470797059244], [1.4379531127884977, 0.1320478318988769], [1.6666666666666665, 0.31325460539187205], [1.8333333333333335, 0.5527707983925666], [0.10461977945516432, 0.7093981010885027], [0.16666666666666663, 0.5527707983925666], [0.33333333333333337, 0.31325460539187205], [0.5620468872115023, 0.1320478318988769], [0.8333333333333334, 0.024579470797059244], [1.5620468872115025, 0.1320478318988769], [1.7907604410896716, 0.31325460539187205], [-0.06204688721150231, 0.42072296649368973], [0.0425728922436619, 0.2886751345948129], [0.27128644612183095, 0.10746836110181783], [0.5425728922436619, 0.0], [0.8333333333333334, -0.024579470797059095], [1.271286446121831, 0.10746836110181783], [1.5, 0.2886751345948129], [1.6666666666666665, 0.5281913275955076], [0.27128644612183095, -0.10746836110181783], [0.5620468872115023, -0.1320478318988769], [0.4379531127884976, -0.1320478318988769], [0.728713553878169, -0.10746836110181783], [1.228713553878169, 0.1812067734929951], [1.3953802205448358, 0.42072296649368973], [0.0425728922436619, -0.2886751345948129], [0.33333333333333337, -0.31325460539187205], [0.20923955891032853, -0.31325460539187205], [0.5, -0.2886751345948129], [0.771286446121831, -0.1812067734929951], [1.1666666666666667, 0.2395161930006946], [0.16666666666666663, -0.5527707983925666], [0.33333333333333337, -0.5281913275955074], [0.6046197794551642, -0.42072296649368973], [0.8333333333333334, -0.2395161930006946], [-0.06204688721150231, -0.42072296649368973], [0.10461977945516432, -0.4453024372907488], [0.6666666666666667, -0.31325460539187205], [0.8953802205448357, -0.1320478318988769], [0.10461977945516432, -0.7093981010885027], [0.27128644612183095, -0.6848186302914435], [0.5425728922436619, -0.5773502691896258], [0.771286446121831, -0.39614349569663065], [0.9379531127884976, -0.15662730269593603], [0.728713553878169, -0.6848186302914435], [0.8953802205448357, -0.4453024372907488], [0.771286446121831, -0.9734937648862564], [0.9379531127884976, -0.7339775718855618], [1.0620468872115023, -0.9980732356833154], [0.8333333333333334, -1.1301210675821924], [1.1666666666666667, -1.1301210675821924], [2.895380220544836, 0.4453024372907488], [2.728713553878169, 0.6848186302914435], [2.457427107756338, 0.5773502691896258], [2.228713553878169, 0.39614349569663065], [2.062046887211502, 0.15662730269593603], [2.0, -0.5773502691896258], [2.228713553878169, -0.9734937648862564], [2.5, 0.8660254037844386], [2.271286446121831, 0.6848186302914435], [2.166666666666667, -0.5527707983925666], [2.228713553878169, 0.9734937648862564], [2.062046887211502, 0.7339775718855618], [2.124093774423005, 0.8414459329873795], [2.0, 0.5773502691896258], [2.124093774423005, -0.2640956637977538], [2.062046887211502, -0.15662730269593603], [2.228713553878169, -0.39614349569663065], [2.728713553878169, -0.6848186302914435], [1.9379531127884975, 0.9980732356833154], [2.895380220544836, -0.4453024372907488], [2.062046887211502, 0.9980732356833154], [2.062046887211502, -0.10746836110181783], [2.395380220544836, -0.42072296649368973], [1.8333333333333335, 1.1301210675821924], [1.771286446121831, 0.9734937648862564], [1.771286446121831, 0.39614349569663065], [1.8953802205448356, 0.1320478318988769], [2.228713553878169, -0.1812067734929951], [2.5, -0.2886751345948129], [2.790760441089671, -0.31325460539187205], [2.957427107756338, -0.2886751345948129], [1.542572892243662, 0.5773502691896258], [1.771286446121831, 0.1812067734929951], [2.562046887211502, -0.1320478318988769], [2.728713553878169, -0.10746836110181783], [2.290760441089671, -0.024579470797059095], [2.166666666666667, -0.024579470797059095], [2.957427107756338, 0.2886751345948129], [2.166666666666667, 0.024579470797059244], [1.1046197794551642, 0.7093981010885027], [1.228713553878169, 0.4453024372907488], [1.8333333333333335, 0.024579470797059244], [2.124093774423005, 0.0], [2.790760441089671, 0.31325460539187205], [0.9379531127884976, 0.42072296649368973], [1.542572892243662, 0.0], [1.8333333333333335, -0.024579470797059095], [2.5, 0.2886751345948129], [2.228713553878169, 0.1812067734929951], [2.395380220544836, 0.42072296649368973], [1.2092395589103286, -0.31325460539187205], [1.771286446121831, -0.1812067734929951], [1.3333333333333333, -0.5281913275955074], [1.8333333333333335, -0.2395161930006946], [0.9379531127884976, -0.42072296649368973], [1.8953802205448356, -0.1320478318988769], [1.1046197794551642, -0.7093981010885027], [1.771286446121831, -0.39614349569663065], [2.062046887211502, -0.9980732356833154], [2.166666666666667, -1.1301210675821924], [1.6046197794551644, 0.9980732356833154], [1.3759062255769952, 0.024579470797059244], [2.228713553878169, 1.550844034075882], [1.957427107756338, 1.4433756729740645], [1.5620468872115025, 1.0226527064803745], [1.6240937744230046, 0.024579470797059244], [2.0, 1.7320508075688772], [1.771286446121831, 1.550844034075882], [1.6046197794551644, 1.3113278410751874], [1.6666666666666665, 1.4187962021770053], [1.728713553878169, 1.839519168670695], [1.6240937744230046, 1.7074713367718182], [1.5, 1.4433756729740645], [1.457427107756338, 1.1547005383792517], [1.6240937744230046, 0.6019297399866851], [1.5620468872115025, 0.7093981010885027], [1.728713553878169, 0.46988190808780805], [1.4379531127884977, 1.8640986394677543], [1.3333333333333333, 1.4187962021770053], [2.562046887211502, 0.4453024372907488], [1.5620468872115025, 1.8640986394677543], [1.4379531127884977, 1.0226527064803745], [1.271286446121831, 1.839519168670695], [1.228713553878169, 1.550844034075882], [1.271286446121831, 1.2621688994810691], [1.3953802205448358, 0.9980732356833154], [2.290760441089671, 0.5527707983925666], [1.0, 1.7320508075688772], [1.042572892243662, 1.4433756729740645], [1.1046197794551642, 1.2867483702781284], [1.771286446121831, 0.7585570426826208], [2.228713553878169, 0.7585570426826208], [0.771286446121831, 1.550844034075882], [0.8953802205448357, 1.2867483702781284], [1.0, 1.1547005383792517], [1.7907604410896716, 0.8414459329873795], [1.957427107756338, 0.8660254037844386], [2.457427107756338, 1.1547005383792517], [0.6046197794551642, 1.3113278410751874], [2.333333333333333, 1.4187962021770053], [0.6046197794551642, 1.5754235048729412], [0.6666666666666667, 1.4187962021770053], [1.3333333333333333, 0.8906048745814976], [1.6240937744230046, 0.8660254037844386], [0.4379531127884976, 1.2867483702781284], [1.042572892243662, 0.8660254037844386], [1.3333333333333333, 0.8414459329873795], [2.0, 1.1547005383792517], [2.166666666666667, 1.394216731379946], [0.771286446121831, 0.7585570426826208], [1.228713553878169, 0.7585570426826208], [1.8953802205448356, 1.2867483702781284], [1.6666666666666665, 1.1055415967851332], [0.4379531127884976, 0.4453024372907488], [0.728713553878169, 0.42072296649368973], [1.3953802205448358, 0.7339775718855618], [1.5620468872115025, 0.9734937648862564], [0.6046197794551642, 0.15662730269593603], [1.4379531127884977, 0.7093981010885027], [1.3333333333333333, -0.2640956637977538], [1.6666666666666665, -0.2640956637977538], [0.6046197794551642, 0.9980732356833154], [0.37590622557699527, 0.6019297399866851], [0.37590622557699527, 0.024579470797059244], [1.3953802205448358, 1.5754235048729412], [0.9574271077563381, 1.4433756729740645], [0.728713553878169, 1.2621688994810691], [0.5620468872115023, 1.0226527064803745], [0.6240937744230048, 0.024579470797059244], [0.728713553878169, 1.839519168670695], [0.5, 1.4433756729740645], [0.5620468872115023, 0.7093981010885027], [0.728713553878169, 0.46988190808780805], [1.3953802205448358, 0.15662730269593603], [0.4379531127884976, 1.8640986394677543], [0.37590622557699527, 1.7074713367718182], [0.37590622557699527, 1.1301210675821924], [0.6046197794551642, 0.7339775718855618], [1.5620468872115025, 0.4453024372907488], [0.4379531127884976, 1.0226527064803745], [1.1666666666666667, 0.33783407618893097], [0.27128644612183095, 1.839519168670695], [0.22871355387816905, 1.550844034075882], [0.27128644612183095, 1.2621688994810691], [0.3953802205448357, 0.9980732356833154], [0.33333333333333337, 1.1055415967851332], [1.2907604410896714, 0.5527707983925666], [0.0, 1.7320508075688772], [0.0425728922436619, 1.4433756729740645], [0.10461977945516432, 1.2867483702781284], [-0.22871355387816905, 1.550844034075882], [0.0, 1.1547005383792517], [0.7907604410896715, 0.8414459329873795], [0.6666666666666667, 0.8414459329873795], [0.9574271077563381, 0.8660254037844386], [-0.06204688721150231, 0.9980732356833154], [0.20923955891032853, 0.8906048745814976], [-0.4574271077563381, 1.1547005383792517], [-0.22871355387816905, 0.9734937648862564], [0.0425728922436619, 0.8660254037844386], [0.33333333333333337, 0.8414459329873795], [1.1666666666666667, 1.394216731379946], [-0.5, 0.8660254037844386], [-0.22871355387816905, 0.7585570426826208], [-0.16666666666666674, 0.5527707983925666], [-0.29076044108967136, 0.5527707983925666], [0.0, 0.5773502691896258], [0.6666666666666667, 1.1055415967851332], [-0.33333333333333326, 0.31325460539187205], [-0.3953802205448358, 0.42072296649368973], [0.27128644612183095, 0.46988190808780805], [0.22871355387816905, 0.1812067734929951], [0.33333333333333337, -0.2640956637977538], [0.6666666666666667, -0.2640956637977538], [1.0620468872115023, 0.42072296649368973], [0.10461977945516432, 0.1320478318988769], [-0.16666666666666674, -0.5527707983925666], [-0.12409377442300484, -0.8414459329873795], [-0.06204688721150231, -0.9980732356833154], [0.8953802205448357, 0.7093981010885027], [0.22871355387816905, 0.39614349569663065], [0.0, -0.5773502691896258], [0.22871355387816905, -0.39614349569663065], [0.4574271077563381, -0.5773502691896258], [0.8953802205448357, -0.7093981010885027], [-0.12409377442300484, 0.2640956637977538], [0.10461977945516432, -0.1320478318988769], [1.0620468872115023, -0.42072296649368973], [-0.06204688721150231, 0.15662730269593603], [-0.16666666666666674, 1.1301210675821924], [-0.22871355387816905, 0.39614349569663065], [0.22871355387816905, -0.1812067734929951], [-0.5, 0.2886751345948129], [0.16666666666666663, -0.024579470797059095], [-0.8953802205448358, 0.4453024372907488], [-0.7907604410896714, 0.31325460539187205], [0.16666666666666663, 0.024579470797059244], [-0.16666666666666674, 0.024579470797059244], [0.7907604410896715, 0.31325460539187205], [-0.9574271077563381, 0.2886751345948129], [-0.728713553878169, 0.10746836110181783], [-0.4574271077563381, 0.0], [-0.16666666666666674, -0.024579470797059095], [0.6666666666666667, 0.5281913275955076], [-0.4379531127884977, -0.1320478318988769], [-0.9574271077563381, -0.2886751345948129], [-0.5, -0.2886751345948129], [-0.22871355387816905, -0.1812067734929951], [0.16666666666666663, 0.2395161930006946], [-0.3953802205448358, -0.42072296649368973], [-0.728713553878169, -0.6848186302914435], [-0.4574271077563381, -0.5773502691896258], [-0.22871355387816905, -0.39614349569663065], [-0.5, -0.8660254037844386], [-0.27128644612183095, -0.6848186302914435], [-0.22871355387816905, -0.9734937648862564], [-0.06204688721150231, -0.7339775718855618], [0.06204688721150242, -0.9980732356833154], [1.5620468872115025, -0.4453024372907488], [1.271286446121831, -0.42072296649368973], [0.6046197794551642, -0.7339775718855618], [0.4379531127884976, -0.9734937648862564], [0.4379531127884976, -1.8640986394677543], [1.3953802205448358, -0.15662730269593603], [0.728713553878169, -0.46988190808780805], [0.5620468872115023, -0.7093981010885027], [0.5, -1.4433756729740645], [0.6666666666666667, -1.4187962021770053], [0.771286446121831, -1.550844034075882], [1.0, -1.7320508075688772], [0.728713553878169, -1.2621688994810691], [1.228713553878169, -1.550844034075882], [0.37590622557699527, -0.6019297399866849], [0.6666666666666667, 0.2640956637977538], [0.5, 0.024579470797059244], [0.4379531127884976, -0.7093981010885027], [0.5620468872115023, -0.9734937648862564], [0.8953802205448357, -1.2867483702781284], [1.1666666666666667, -1.394216731379946], [1.3333333333333333, -1.4187962021770053], [0.33333333333333337, 0.2640956637977538], [0.27128644612183095, -0.46988190808780805], [0.3953802205448357, -0.7339775718855618], [1.0, -1.1547005383792517], [0.9574271077563381, -0.8660254037844386], [1.228713553878169, -0.7585570426826208], [0.37590622557699527, -0.8660254037844386], [0.6666666666666667, -0.8414459329873795], [-0.3953802205448358, -0.15662730269593603], [0.33333333333333337, -0.8414459329873795], [1.2907604410896714, -0.5527707983925666], [0.0425728922436619, -0.8660254037844386], [0.771286446121831, -0.7585570426826208], [-0.4574271077563381, -1.1547005383792517], [0.0, -1.1547005383792517], [0.6666666666666667, -0.6265092107837441], [0.10461977945516432, -1.2867483702781284], [0.33333333333333337, -1.1055415967851332], [-0.22871355387816905, -1.550844034075882], [0.0425728922436619, -1.4433756729740645], [0.4379531127884976, -1.6000029756700005], [1.4379531127884977, -0.9734937648862564], [1.728713553878169, -0.46988190808780805], [1.5620468872115025, -0.7093981010885027], [1.5, -1.4433756729740645], [1.6240937744230046, -1.7074713367718184], [1.728713553878169, -1.839519168670695], [1.6240937744230046, -1.1301210675821924], [2.228713553878169, -1.550844034075882], [1.6666666666666665, 0.2640956637977538], [1.5, 0.024579470797059244], [1.4379531127884977, -0.7093981010885027], [1.5620468872115025, -0.9734937648862564], [2.333333333333333, -1.4187962021770053], [1.3333333333333333, 0.2640956637977538], [1.3953802205448358, -0.7339775718855618], [2.0, -1.1547005383792517], [2.457427107756338, -1.1547005383792517], [1.957427107756338, -0.8660254037844386], [1.2092395589103286, -0.8414459329873795], [0.6046197794551642, -0.15662730269593603], [1.3333333333333333, -0.8414459329873795], [0.4379531127884976, -0.4453024372907488], [1.3333333333333333, -0.8906048745814976], [0.6046197794551642, -1.5754235048729412], [1.271286446121831, -1.2621688994810691], [1.271286446121831, -1.839519168670695], [1.5620468872115025, -1.8640986394677543], [1.125, -0.4841229182759271], [0.2682372542187894, 0.031587850897493115], [1.1067627457812106, -1.4839566057252744], [0.25, -0.9682458365518543], [1.9817627457812106, -0.9998336874493473], [1.1432372542187894, 0.5157107691734202], [1.8506093959854342, 0.39817091578874836], [1.6928765775201509, 0.33899079969018714], [1.4503606766986386, 0.176719969483233], [1.2650129329409796, -0.04865083246606342], [1.1526149252923754, -0.3179322324251918], [1.1442727108416575, -0.7752832643125013], [1.2467756787221902, -1.0484845199154107], [1.4237812080291312, -1.2804647498534238], [1.5538991012020396, -1.387475319605598], [1.565021814058364, 0.5700745055132218], [1.431087965856981, 0.4678803155198072], [1.2457402220993221, 0.24250951357051073], [1.133342214450718, -0.02677188638861755], [1.103464159447221, -0.3170357083102133], [1.4045084971874737, -0.9893044038168497], [1.6409453132693894, -1.1603114694263446], [1.5346263903603816, -1.0963149735690239], [1.7964150020235516, -1.225204489398644], [1.0308392465701852, 0.24642936921429182], [1.0009611915666883, -0.04383445270730387], [1.0032243212778098, 0.08023868336355659], [1.0224970321194673, -0.21092166267301768], [1.302005529306941, -0.7161031482139402], [1.538442345388857, -0.8871102138234351], [1.6939120341430187, -0.9520032337957343], [0.8538337172632442, 0.47840959915230497], [0.8239556622597474, 0.18814577723070913], [0.8454915028125263, 0.02105856726499541], [0.9479944706930589, -0.25214268833791414], [1.3614368160819157, -0.6551299838854221], [1.2551178931729081, -0.5911334880281013], [1.516906504836078, -0.7200230038577213], [1.8047572164742696, -0.7678534575113344], [2.094160753429815, -0.7305522874902189], [2.2518935718950983, -0.6713721713916577], [0.5875188461778313, 0.359152842840204], [0.6090546867306104, 0.1920656328744903], [0.7115576546111431, -0.08113562272841918], [0.8885631839180842, -0.3131158526664322], [1.280469688754162, -0.5490159382482264], [1.5683204003923537, -0.5968463919018394], [1.8577239373478989, -0.5595452218807241], [2.124038808433312, -0.4402884655686232], [2.2579726566346947, -0.3380942755752087], [0.7237158240903361, 0.5854201689044792], [0.6961008987979607, 0.4192294830537437], [0.7153736096396182, 0.12806913701716952], [0.9948821068270919, -0.377112348523753], [1.3867886116631698, -0.6130124341055471], [1.6746393233013617, -0.66084288775916], [1.9640428602569067, -0.6235417177380449], [2.12177567872219, -0.5643616016394837], [0.43204915742366945, 0.4240458628125034], [0.45358499797644836, 0.25695865284678954], [0.5560879658569811, -0.016242602756119922], [0.7330934951639221, -0.24822283269413298], [0.9695303112458379, -0.4192298983036278], [0.8632113883368302, -0.35523340244630724], [1.4128507116381916, -0.5319533719295401], [1.7022542485937369, -0.49465220190842485], [1.9685691196791493, -0.375395445596324], [1.8599870670590204, -0.4354720858098637], [2.1025029678805325, -0.2732012556029095], [0.44524278352573043, -0.20039237904051999], [0.6816795996076463, -0.37139944465001484], [0.5753606766986386, -0.30740294879269414], [0.8371492883618084, -0.43629246462231414], [1.4144035369555452, -0.4468217482548118], [1.680718408040958, -0.32756499194271105], [1.8146522562423408, -0.22537080194929646], [0.15583924657018533, -0.23769354906163528], [0.392276062652101, -0.4087006146711301], [0.28595713974309334, -0.34470411881380936], [0.5477457514062631, -0.47359363464342946], [0.8355964630444549, -0.5214240882970425], [1.3913148710854126, -0.3648661619638263], [1.2827328184652835, -0.42494280217736596], [1.5252487192867958, -0.2626719719704118], [1.710596463044455, -0.03730117002111536], [1.8229944706930588, 0.23198022993801304], [0.1259611915666884, -0.527957370983231], [0.5692815919590422, -0.6406808446091432], [0.8586851289145873, -0.603379674588028], [1.2589338482013832, -0.38192872828251256], [1.4442815919590422, -0.1565579263332161], [1.5566795996076461, 0.11272347362591228], [1.586557654611143, 0.402987295547508], [-0.0018935718950983116, -0.2968736651601964], [0.1282243212778097, -0.4038842349123706], [0.3900129329409797, -0.5327737507419905], [0.6778636445791713, -0.5806042043956036], [0.9672671815347165, -0.5433030343744883], [1.5528636445791713, -0.09648128611967643], [1.6652616522277754, 0.17280011383945193], [-0.00797265663469493, -0.6301515609766455], [0.1474970321194672, -0.6950445809489448], [0.43534774375765906, -0.7428750346025579], [0.7247512807132042, -0.7055738645814424], [0.9910661517986169, -0.5863171082693417], [0.8824840991784877, -0.6463937484828813], [1.310347743757659, -0.2587521163266307], [1.4227457514062631, 0.010529283632497706], [0.5394035369555452, -0.9309446665307389], [0.6971363554208287, -0.8717645504321778], [0.9396522562423409, -0.7094937202252236], [1.2373980076486042, -0.21484151831679874], [1.2672760626521011, 0.07542230360479696], [0.427005529306941, -1.2002260664898674], [0.6933204003923537, -1.0809693101777664], [0.5847383477722246, -1.1410459503913062], [0.8272542485937369, -0.978775120184352], [1.0126019923513958, -0.7534043182350555], [1.1548780550034967, -0.1938590963543314], [0.6634423453888569, -1.371233132099362], [0.79737619359024, -1.2690389421059478], [0.982723937347899, -1.0436681401566512], [1.095121944996503, -0.7743867401975227], [0.3993906040145657, -1.3664167523406026], [0.5571234224798491, -1.3072366362420416], [0.9849870670590204, -0.9195950040857909], [1.0973850747076246, -0.6503136041266624], [0.6849781859416357, -1.5383203420650762], [0.8189120341430188, -1.4361261520716615], [1.0042597779006779, -1.2107553501223651], [1.116657785549282, -0.9414739501632368], [1.146535840552779, -0.6512101282416409], [1.2191607534298148, -1.214675205766146], [1.2490388084333117, -0.9244113838445503], [1.3961662827367558, -1.446655435704159], [1.4260443377402525, -1.1563916137825634], [1.6624811538221684, -1.3273986793920585], [1.526284175909664, -1.5536660054563332], [1.8179508425763307, -1.3922916993643577], [2.567876577520151, 0.8231137179661143], [2.306087965856981, 0.9520032337957343], [2.120740222099322, 0.7266324318464378], [2.008342214450718, 0.4573510318873095], [1.9807272891583425, 0.29116034603657426], [2.2795084971874737, -0.5051814855409226], [2.5159453132693894, -0.6761885511504174], [2.4096263903603816, -0.6121920552930965], [2.6714150020235516, -0.7410815711227169], [2.018237254218789, 0.9998336874493473], [1.9058392465701854, 0.7305522874902189], [1.8782243212778096, 0.5643616016394837], [2.413442345388857, -0.402987295547508], [2.856762745781211, -0.5157107691734202], [1.7288337172632442, 0.9625325174282319], [1.6989556622597473, 0.6722686955066363], [1.7012187919708688, 0.7963418315774967], [1.7204915028125263, 0.5051814855409226], [2.1301178931729083, -0.10701056975217416], [2.3919065048360784, -0.23590008558179412], [2.679757216474269, -0.28373053923540725], [2.969160753429814, -0.24642936921429182], [1.4625188461778313, 0.8432757611161312], [2.443320400392354, -0.11272347362591228], [2.999038808433312, 0.04383445270730387], [1.598715824090336, 1.0695430871804064], [1.5711008987979604, 0.9033524013296708], [2.106318922909008, -0.06399649585732074], [2.5496393233013617, -0.176719969483233], [1.3070491574236693, 0.9081687810884306], [1.3285849979764484, 0.7410815711227169], [1.608093495163922, 0.23590008558179412], [1.8445303112458378, 0.06489301997229924], [2.2878507116381916, -0.04783045365361301], [2.5772542485937366, -0.010529283632497706], [2.8435691196791497, 0.1087274726796031], [2.9775029678805325, 0.21092166267301768], [1.3202427835257304, 0.28373053923540725], [1.7121492883618084, 0.04783045365361301], [2.555718408040958, 0.1565579263332161], [2.689652256242341, 0.2587521163266307], [2.875, 0.4841229182759271], [2.266314871085412, 0.11925675631210082], [2.1577328184652833, 0.059180116098561164], [2.400248719286796, 0.22145094630551537], [2.585596463044455, 0.4468217482548118], [2.697994470693059, 0.7161031482139402], [2.1339338482013828, 0.10219418999341455], [2.319281591959042, 0.32756499194271105], [2.461557654611143, 0.8871102138234351], [0.8731064281049017, 0.18724925311573073], [1.1095432441868174, 0.016242187506235833], [1.8422671815347162, -0.059180116098561164], [2.108582052620129, 0.06007664021353968], [2.427863644579171, 0.38764163215625064], [2.540261652227776, 0.6569230321153791], [0.8670273433653052, -0.14602864270071844], [1.5997512807132042, -0.22145094630551537], [1.8660661517986168, -0.10219418999341455], [2.2977457514062634, 0.49465220190842485], [2.142276062652101, 0.5595452218807241], [1.4597383477722246, -0.6569230321153791], [1.8876019923513958, -0.26928139995912836], [1.6723761935902401, -0.7849160238300206], [1.970121944996503, -0.29026382192159567], [1.2743906040145656, -0.8822938340646754], [1.5407054750999785, -0.7630370777525747], [1.9723850747076246, -0.1661906858507353], [1.559978185941636, -1.054197423789149], [1.991657785549282, -0.4573510318873095], [2.271166282736756, -0.9625325174282319], [2.537481153822169, -0.8432757611161312], [2.692950842576331, -0.9081687810884306], [1.1708521795111648, 0.6819014550241554], [1.4420184622479206, -0.28063106240407654], [1.4493252200757705, 1.4677140029691547], [1.2639774763181115, 1.242343201019858], [1.1515794686695073, 0.9730618010607297], [1.1217014136660104, 0.6827979791391341], [1.659182567488179, -0.16047778197699722], [1.1614745084375788, 1.5155444566227676], [1.0191984457854777, 0.9559992347420433], [0.8720709714820337, 1.478243286601652], [0.8444560461896582, 1.312052600750917], [0.8637287570313157, 1.0208922547143429], [0.9662317249118484, 0.7476909991114332], [1.3796740703007053, 0.3447037035639253], [1.2733551473916975, 0.4087001994212461], [1.535143759054867, 0.27981068359162614], [0.605756100396621, 1.3589865302895516], [0.7297949088299325, 0.9186980647209282], [1.0369183313097816, 0.5797072650307409], [2.276209910853484, 0.6617394118741386], [0.7143381530167501, 1.4190631705030912], [0.8361138317389403, 0.8547015688636073], [0.4718222521952379, 1.256792340296137], [0.5743252200757705, 0.9835910846932275], [0.7513307493827116, 0.7516108547552143], [0.9877675654646273, 0.5806037891457194], [0.8814486425556196, 0.6446002850030402], [0.2864745084375788, 1.0314215383468404], [0.593597930917428, 0.6924307386566534], [1.8328895104611305, 0.7744628855000508], [0.41051331687089043, 0.5911330727782173], [0.5659830056250525, 0.526240052805918], [1.409552125304202, 0.634967525485521], [1.300970072684073, 0.5748908852719814], [1.5434859735055853, 0.7371617154789356], [1.8412317249118484, 1.2318139173873603], [0.1441984457854778, 0.47187631646611616], [1.6047949088299327, 1.4028209829968554], [0.01634368232369099, 0.7029600222891507], [0.14646157549659922, 0.5959494525369768], [1.2518193068389185, 0.5757874093869599], [0.010264597584094592, 0.3696821264727019], [0.1657342863382567, 0.30478910650040253], [0.7429885349319936, 0.29425982286790486], [1.0093034060174062, 0.4135165791800057], [0.900721353397277, 0.353439938966466], [1.4409830056250525, 1.0103629710818451], [1.4708610606285493, 1.3006267930034408], [0.5576407911743346, 0.06888902091860831], [0.9578895104611305, 0.29033996722412375], [1.1731153092222864, 0.805974591095016], [0.417627858233355, -0.3665830648912553], [0.6839427293187677, -0.24732630857915447], [1.115622328926414, 0.34952008332268486], [0.7032154401604253, -0.5384866546157286], [1.1647730947715684, 0.34862355920770643], [1.5445214301284533, -0.553832318006986], [1.5743994851319503, -0.2635684960853903], [1.8361880967951199, -0.3924580119150103], [0.2958521795111647, 0.19777853674822832], [0.287509965060447, -0.25957249513908115], [0.5670184622479206, -0.7647539806800037], [0.7082590682771536, 1.0857852746866419], [0.3889774763181114, 0.7582202827439309], [0.27657946866950744, 0.4889388827848026], [0.2467014136660105, 0.19867506086320685], [0.7841825674881789, -0.6446007002529245], [-0.002929028517966392, 0.9941203683257251], [-0.011271242968684314, 0.5367693364384157], [0.39835514739169753, -0.07542271885468106], [0.6601437590548673, -0.20431223468430104], [1.3951308261138877, -0.15566140221823757], [-0.269243899603379, 0.8748636120136245], [-0.24770805905060023, 0.7077764020479106], [0.03180043813687361, 0.2025949165069881], [0.16191833130978173, 0.09558434675481386], [0.42370694297295153, -0.03330516907480622], [1.4012099108534841, 0.1776164935982115], [0.1381193610458813, 0.13859842064966724], [1.107280114475696, -0.10783094856462465], [-0.40317774780476223, 0.7726694220202098], [-0.30067477992422953, 0.4994681664173004], [-0.1236692506172885, 0.26748793647928726], [0.1127675654646273, 0.09648087086979233], [0.00644864255561961, 0.16047736672711319], [1.111806373897939, 0.14031532357709628], [-0.5885254915624212, 0.5472986200709133], [-0.41151996225548015, 0.3153183901329002], [-0.17508314617356402, 0.14431132452340542], [-0.7009234992110251, 0.27801722011178504], [-0.30901699437494745, 0.04211713452999082], [0.534552125304202, 0.15084460720959392], [0.42597007268407294, 0.09076796699605423], [0.6684859735055853, 0.2530387972030085], [-0.28748115382216866, -0.12497007543572303], [0.0019223831333766705, -0.08766890541460774], [-0.7092657136617433, -0.17933381177552463], [-0.42141500202355164, -0.22716426542913756], [-0.13201146506800643, -0.18986309540802226], [0.1343034060174063, -0.07060633909592144], [-0.6067627457812106, -0.452535067378434], [-0.42975721647426934, -0.6845152973164472], [-0.27202439800898603, -0.6253351812178859], [-0.029508497187473726, -0.46306435101093174], [0.29811530922228624, 0.32185167281908883], [-0.19332040039235365, -0.855522362925942], [-0.29963932330136145, -0.7915258670686214], [-0.17178455983957464, -1.0226095728916558], [0.2598950397680714, -0.42576318098981636], [0.2897730947715683, -0.13549935906822067], [0.3623980076486042, -0.698964436592726], [0.6695214301284533, -1.0379552362829132], [0.6993994851319503, -0.7476914143613174], [0.96118809679512, -0.8765809301909374], [0.9756093959854344, -0.08595200248717878], [0.2776149252923754, -0.802055150701119], [0.6788991012020393, -1.8715982378815248], [0.6900218140583643, 0.08595158723729475], [0.5295084971874737, -1.473427322092777], [0.380117893172908, -1.0752564063040284], [1.3768935718950983, -1.1554950896675849], [0.013563183918084198, -0.7972387709423594], [0.14368107709099232, -0.9042493406945334], [0.4054696887541621, -1.0331388565241537], [1.1404567558131826, -0.9844880240580901], [1.3829726566346947, -0.8222171938511359], [-0.44295084257633044, -0.060077055463423644], [-0.14190650483607792, -0.7323457509700602], [0.5378507116381916, -1.0160762902054674], [-0.7191607534298148, -0.7218164673375624], [-0.32725424859373686, -0.9577165529193568], [0.40773281846528353, -0.9090657204532931], [-0.7490388084333117, -1.012080289259158], [-0.5935691196791495, -1.0769733092314573], [-0.30571840804095785, -1.1248037628850704], [0.1414179473798708, -1.0283224767653938], [0.3839338482013831, -0.8660516465584397], [0.09226718153471636, -1.0274259526504155], [0.7902616522277756, -0.31132280443647514], [-0.7275029678805327, -1.179167499224872], [-0.43965225624234106, -1.2269979528784851], [0.1160661517986169, -1.0704400265452687], [0.007484099178487802, -1.1305166667588085], [-0.447994470693059, -1.6843489847657944], [-0.29026165222777567, -1.6251688686672334], [-0.04774575140626314, -1.4628980384602792], [0.27987805500349694, -0.6779820146302585], [0.10772393734789898, -1.5277910584325785], [-0.31787657752015086, -1.7913595545179686], [-0.05608796585698106, -1.9202490703475885], [0.12925977790067789, -1.6948782683982924], [0.24165778554928208, -1.4255968684391638], [0.2715358405527789, -1.135333046517568], [0.2317627457812106, -1.9680795240012015], [0.5211662827367558, -1.9307783539800862], [0.7874811538221685, -1.8115215976679855], [0.6512841759096639, -2.037788923732261], [1.832372141766645, -0.6016627716605989], [1.5660572706812323, -0.7209195279726996], [1.134377671073586, -1.317765919874539], [1.104499616070089, -1.608029741796135], [1.53566184698325, -2.387309007054945], [1.5467845598395749, -0.42975918193612556], [1.1151049602319285, -1.026605573837965], [1.0852269052284316, -1.3168693957595607], [1.3862712429686843, -1.9891380912661971], [1.778177747804762, -2.2250381768479914], [1.5202050911700673, -1.8869439012727824], [1.4138861682610597, -1.8229474054154615], [1.6756747799242295, -1.951836921245082], [1.9635254915624212, -1.9996673748986946], [1.2368806389541187, -1.5909671754774486], [1.4986692506172885, -1.7198566913070688], [1.7865199622554804, -1.7676871449606815], [2.0759234992110254, -1.7303859749395665], [0.590817432511821, -0.807768054574857], [0.8703259296992948, -1.3129495401157796], [0.7054785698715467, -0.41441351854486824], [0.6756005148680497, -0.7046773404664641], [0.9766448526083025, -1.3769460359731003], [1.2130816686902184, -1.5479531015825954], [1.945805606038117, -1.6233754051873923], [2.103538424503401, -1.564195289088831], [0.41381190320488004, -0.5757878246368439], [0.7148562409451327, -1.2480565201434803], [1.6840169943749475, -1.4944858893577724], [2.0842657136617433, -1.273034943052257], [1.5070114650680064, -1.2625056594197592], [1.6923592088256654, -1.0371348574704626], [0.9981806931610814, -1.5440332459388142], [-0.020130826113887723, -1.2967073526095438], [1.349278646602723, -1.32168577551832], [1.647024398008986, -0.8270335736098954], [-0.02620991085348412, -1.629985248425993], [0.7065140264944147, -1.7054075520307899], [0.9728288975798275, -1.586150795718689], [0.4087682750881516, -2.2000597539392146], [0.8090169943749475, -1.9786088076336994], [0.6452050911700675, -2.3710668195487097], [0.7791389393714506, -2.268872629555295], [0.9644866831291095, -2.0435018276059984], [1.0768846907777136, -1.7742204276468703], [0.8006747799242295, -2.435959839521009], [0.9860225236818886, -2.210589037571712], [1.2009234992110254, -2.2145088932154935], [1.9794996160700893, -1.1239068235202077], [1.9602269052284318, -0.8327464774836336], [2.261271242968684, -1.50501517299027], [2.4977080590506002, -1.6760222385997647], [2.6531777478047625, -1.7409152585720642], [2.1118806389541187, -1.1068442572015218], [2.66151996225548, -1.2835642266847545], [2.950923499211025, -1.2462630566636392], [1.4658174325118212, -0.32364513629892994], [2.137232434535373, -1.0647267074216467], [1.5804785698715467, 0.06970939973105887], [1.5506005148680497, -0.2205544221905369], [1.8516448526083025, -0.8928231176971733], [2.0880816686902186, -1.0638301833066681], [2.978538424503401, -1.080072370812904], [1.28881190320488, -0.09166490636091684], [1.589856240945133, -0.7639336018675532], [1.8262930570270486, -0.934940667477048], [2.5590169943749475, -1.0103629710818451], [2.9592657136617433, -0.7889120247763297], [1.142719885524304, -0.8604148879872295], [2.3820114650680067, -0.778382741143832], [1.138193626102061, -1.1085611601289505], [1.7154478746957982, -1.1190904437614482], [0.8548691738861123, -0.8125844343336167], [2.2242786466027233, -0.8375628572423932], [0.8487900891465159, -1.1458623301500657], [1.5417409317228463, -2.054031111238496], [1.9734205313304924, -1.4571847193366567], [2.519243899603379, -1.8431094485654789]], \"edge\": [[1, 2], [1, 3], [1, 4], [1, 5], [1, 6], [1, 7], [1, 9], [1, 14], [1, 16], [1, 24], [1, 33], [1, 41], [1, 43], [1, 48], [1, 51], [1, 57], [1, 59], [1, 68], [1, 77], [1, 85], [1, 86], [1, 92], [1, 94], [1, 101], [1, 114], [1, 120], [1, 125], [1, 129], [1, 135], [1, 137], [1, 875], [1, 877], [1, 878], [1, 886], [1, 894], [1, 910], [1, 918], [1, 934], [1, 1007], [1, 1008], [1, 1169], [1, 1172], [1, 1176], [1, 1192], [1, 1208], [1, 1213], [1, 1219], [1, 1224], [1, 1227], [1, 1229], [1, 1231], [1, 1305], [2, 6], [2, 7], [2, 25], [2, 26], [2, 27], [2, 28], [2, 29], [2, 30], [2, 31], [2, 32], [2, 140], [2, 141], [2, 146], [2, 147], [2, 150], [2, 157], [2, 158], [2, 159], [2, 160], [2, 164], [2, 170], [2, 177], [2, 198], [2, 428], [2, 432], [2, 433], [2, 452], [2, 453], [2, 454], [2, 455], [2, 456], [2, 457], [2, 458], [2, 567], [2, 568], [2, 575], [2, 576], [2, 580], [2, 581], [2, 588], [2, 589], [2, 591], [2, 593], [2, 597], [2, 603], [2, 608], [2, 613], [2, 616], [2, 637], [2, 638], [2, 876], [2, 877], [2, 878], [2, 879], [2, 880], [2, 881], [2, 883], [2, 888], [2, 890], [2, 898], [2, 907], [2, 915], [2, 917], [2, 922], [2, 925], [2, 931], [2, 933], [2, 942], [2, 951], [2, 959], [2, 960], [2, 966], [2, 968], [2, 975], [2, 988], [2, 994], [2, 999], [2, 1003], [2, 1009], [2, 1011], [3, 5], [3, 7], [3, 10], [3, 17], [3, 108], [3, 109], [3, 110], [3, 111], [3, 112], [3, 113], [3, 213], [3, 218], [3, 223], [3, 227], [3, 269], [3, 274], [3, 281], [3, 287], [3, 290], [3, 297], [3, 302], [3, 308], [3, 309], [3, 877], [3, 879], [3, 952], [3, 969], [3, 1156], [3, 1161], [3, 1185], [3, 1218], [4, 5], [4, 6], [4, 40], [4, 47], [4, 55], [4, 64], [4, 80], [4, 87], [4, 96], [4, 102], [4, 347], [4, 349], [4, 351], [4, 353], [4, 362], [4, 369], [4, 371], [4, 379], [4, 393], [4, 398], [4, 878], [4, 879], [4, 989], [4, 1197], [4, 1205], [4, 1214], [5, 12], [5, 20], [5, 36], [5, 44], [5, 60], [5, 133], [5, 134], [5, 295], [5, 298], [5, 302], [5, 318], [5, 334], [5, 339], [5, 345], [5, 350], [5, 353], [5, 355], [5, 357], [5, 431], [5, 879], [5, 1176], [5, 1178], [5, 1190], [5, 1227], [5, 1228], [6, 84], [6, 90], [6, 98], [6, 105], [6, 118], [6, 122], [6, 126], [6, 130], [6, 162], [6, 167], [6, 367], [6, 368], [6, 369], [6, 406], [6, 408], [6, 413], [6, 417], [6, 426], [6, 427], [6, 875], [6, 879], [6, 880], [6, 914], [6, 921], [6, 929], [6, 938], [6, 954], [6, 961], [6, 970], [6, 976], [6, 1221], [6, 1223], [6, 1225], [6, 1227], [6, 1236], [6, 1243], [6, 1245], [6, 1253], [6, 1267], [6, 1272], [7, 69], [7, 70], [7, 71], [7, 72], [7, 73], [7, 74], [7, 75], [7, 76], [7, 142], [7, 147], [7, 188], [7, 189], [7, 202], [7, 206], [7, 210], [7, 217], [7, 220], [7, 222], [7, 227], [7, 232], [7, 237], [7, 238], [7, 239], [7, 241], [7, 875], [7, 879], [7, 881], [7, 884], [7, 891], [7, 982], [7, 983], [7, 984], [7, 985], [7, 986], [7, 987], [7, 1087], [7, 1092], [7, 1097], [7, 1101], [7, 1143], [7, 1148], [7, 1155], [7, 1161], [7, 1164], [7, 1171], [7, 1176], [7, 1182], [7, 1183], [8, 11], [8, 33], [8, 57], [8, 82], [8, 145], [8, 179], [8, 259], [8, 314], [8, 358], [8, 882], [8, 886], [8, 895], [8, 967], [8, 1069], [8, 1110], [8, 1159], [8, 1186], [8, 1189], [8, 1207], [8, 1232], [8, 1264], [9, 50], [9, 51], [9, 52], [9, 53], [9, 54], [9, 55], [9, 56], [9, 57], [9, 140], [9, 162], [9, 174], [9, 203], [9, 209], [9, 879], [9, 883], [9, 886], [9, 929], [9, 950], [9, 1115], [9, 1139], [9, 1202], [10, 12], [10, 32], [10, 57], [10, 95], [10, 110], [10, 160], [10, 229], [10, 231], [10, 261], [10, 262], [10, 263], [10, 884], [10, 886], [10, 925], [10, 928], [10, 1003], [10, 1124], [10, 1163], [10, 1176], [10, 1190], [11, 13], [11, 50], [11, 57], [11, 77], [11, 116], [11, 153], [11, 155], [11, 200], [11, 261], [11, 315], [11, 360], [11, 421], [11, 885], [11, 886], [11, 909], [11, 912], [11, 1068], [11, 1135], [11, 1159], [11, 1188], [11, 1191], [11, 1234], [11, 1263], [11, 1265], [12, 57], [12, 71], [12, 76], [12, 95], [12, 98], [12, 314], [12, 315], [12, 316], [12, 317], [12, 318], [12, 368], [12, 379], [12, 886], [12, 969], [12, 970], [12, 985], [12, 1183], [12, 1272], [13, 53], [13, 57], [13, 114], [13, 197], [13, 201], [13, 263], [13, 317], [13, 886], [13, 887], [13, 1005], [13, 1012], [13, 1137], [13, 1189], [13, 1202], [13, 1219], [13, 1294], [14, 42], [14, 57], [14, 67], [14, 75], [14, 82], [14, 110], [14, 116], [14, 125], [14, 318], [14, 362], [14, 399], [14, 879], [14, 886], [14, 888], [14, 984], [14, 1000], [14, 1198], [14, 1207], [14, 1236], [14, 1242], [14, 1251], [15, 19], [15, 43], [15, 68], [15, 89], [15, 154], [15, 184], [15, 234], [15, 260], [15, 264], [15, 301], [15, 319], [15, 363], [15, 889], [15, 894], [15, 911], [15, 974], [15, 1079], [15, 1138], [15, 1165], [15, 1169], [15, 1187], [15, 1210], [15, 1237], [16, 58], [16, 59], [16, 60], [16, 61], [16, 62], [16, 63], [16, 64], [16, 65], [16, 66], [16, 67], [16, 68], [16, 141], [16, 167], [16, 202], [16, 223], [16, 879], [16, 890], [16, 894], [16, 938], [16, 957], [16, 972], [16, 991], [16, 1000], [16, 1097], [16, 1156], [16, 1172], [16, 1203], [16, 1204], [16, 1205], [17, 20], [17, 39], [17, 55], [17, 68], [17, 79], [17, 96], [17, 111], [17, 142], [17, 203], [17, 223], [17, 264], [17, 265], [17, 266], [17, 267], [17, 268], [17, 891], [17, 894], [17, 1139], [17, 1156], [17, 1176], [17, 1196], [17, 1206], [18, 21], [18, 40], [18, 68], [18, 77], [18, 102], [18, 117], [18, 143], [18, 152], [18, 155], [18, 266], [18, 300], [18, 301], [18, 320], [18, 364], [18, 402], [18, 892], [18, 894], [18, 912], [18, 1140], [18, 1195], [18, 1197], [18, 1214], [18, 1220], [18, 1238], [19, 22], [19, 58], [19, 68], [19, 86], [19, 121], [19, 144], [19, 204], [19, 234], [19, 235], [19, 267], [19, 300], [19, 322], [19, 326], [19, 365], [19, 403], [19, 893], [19, 894], [19, 1141], [19, 1165], [19, 1166], [19, 1193], [19, 1203], [19, 1208], [19, 1239], [20, 60], [20, 68], [20, 319], [20, 320], [20, 321], [20, 370], [20, 377], [20, 894], [21, 61], [21, 68], [21, 79], [21, 84], [21, 102], [21, 106], [21, 114], [21, 145], [21, 184], [21, 186], [21, 195], [21, 314], [21, 321], [21, 326], [21, 333], [21, 366], [21, 368], [21, 371], [21, 391], [21, 404], [21, 418], [21, 894], [21, 895], [21, 953], [21, 954], [21, 974], [21, 978], [21, 1001], [21, 1012], [21, 1194], [21, 1204], [21, 1206], [21, 1214], [21, 1219], [21, 1240], [21, 1244], [21, 1258], [21, 1272], [22, 62], [22, 68], [22, 120], [22, 205], [22, 268], [22, 322], [22, 405], [22, 894], [22, 896], [22, 1142], [23, 63], [23, 68], [23, 96], [23, 98], [23, 125], [23, 331], [23, 369], [23, 894], [23, 897], [23, 970], [23, 1251], [24, 49], [24, 68], [24, 76], [24, 83], [24, 89], [24, 111], [24, 117], [24, 121], [24, 129], [24, 146], [24, 406], [24, 879], [24, 894], [24, 898], [24, 985], [24, 1201], [24, 1210], [24, 1220], [24, 1224], [25, 29], [25, 47], [25, 65], [25, 77], [25, 87], [25, 103], [25, 118], [25, 143], [25, 148], [25, 150], [25, 190], [25, 191], [25, 207], [25, 232], [25, 276], [25, 875], [25, 885], [25, 892], [25, 899], [25, 907], [25, 924], [25, 988], [25, 989], [25, 990], [25, 991], [25, 992], [25, 993], [25, 1106], [25, 1164], [25, 1215], [25, 1221], [26, 30], [26, 86], [26, 108], [26, 122], [26, 144], [26, 151], [26, 158], [26, 163], [26, 192], [26, 193], [26, 208], [26, 218], [26, 228], [26, 238], [26, 276], [26, 296], [26, 875], [26, 893], [26, 900], [26, 908], [26, 917], [26, 932], [26, 994], [26, 995], [26, 996], [26, 997], [26, 1112], [26, 1162], [26, 1208], [26, 1223], [26, 1372], [27, 31], [27, 55], [27, 67], [27, 94], [27, 102], [27, 110], [27, 126], [27, 152], [27, 160], [27, 194], [27, 195], [27, 209], [27, 223], [27, 241], [27, 285], [27, 875], [27, 901], [27, 925], [27, 998], [27, 999], [27, 1000], [27, 1001], [27, 1115], [27, 1156], [27, 1214], [28, 32], [28, 101], [28, 130], [28, 153], [28, 164], [28, 196], [28, 197], [28, 224], [28, 229], [28, 875], [28, 902], [28, 909], [28, 926], [28, 933], [28, 1002], [28, 1003], [28, 1004], [28, 1005], [28, 1006], [28, 1157], [28, 1163], [28, 1171], [28, 1213], [28, 1225], [29, 69], [29, 114], [29, 145], [29, 165], [29, 171], [29, 179], [29, 425], [29, 875], [29, 887], [29, 895], [29, 903], [29, 919], [29, 927], [29, 935], [29, 943], [29, 951], [29, 967], [29, 1009], [29, 1010], [29, 1219], [29, 1267], [29, 1270], [30, 70], [30, 87], [30, 91], [30, 108], [30, 113], [30, 120], [30, 149], [30, 154], [30, 161], [30, 166], [30, 180], [30, 184], [30, 198], [30, 367], [30, 371], [30, 397], [30, 875], [30, 896], [30, 904], [30, 911], [30, 936], [30, 944], [30, 952], [30, 960], [30, 974], [30, 1011], [30, 1182], [30, 1241], [31, 71], [31, 102], [31, 125], [31, 155], [31, 379], [31, 875], [31, 888], [31, 897], [31, 905], [31, 912], [31, 937], [31, 945], [31, 968], [31, 1012], [31, 1183], [31, 1214], [31, 1273], [31, 1427], [32, 72], [32, 103], [32, 106], [32, 110], [32, 129], [32, 146], [32, 156], [32, 172], [32, 199], [32, 368], [32, 875], [32, 898], [32, 906], [32, 913], [32, 920], [32, 928], [32, 946], [32, 953], [32, 969], [32, 975], [32, 1013], [32, 1215], [32, 1224], [32, 1242], [32, 1272], [33, 77], [33, 78], [33, 79], [33, 80], [33, 81], [33, 82], [33, 83], [33, 84], [33, 85], [33, 150], [33, 210], [33, 269], [33, 283], [33, 879], [33, 907], [33, 910], [33, 954], [33, 1143], [33, 1188], [33, 1206], [33, 1207], [34, 37], [34, 74], [34, 85], [34, 86], [34, 109], [34, 123], [34, 151], [34, 227], [34, 251], [34, 311], [34, 908], [34, 910], [34, 983], [34, 1107], [34, 1161], [34, 1208], [34, 1218], [35, 39], [35, 65], [35, 76], [35, 85], [35, 101], [35, 108], [35, 117], [35, 131], [35, 153], [35, 163], [35, 166], [35, 181], [35, 194], [35, 212], [35, 223], [35, 232], [35, 236], [35, 270], [35, 283], [35, 299], [35, 306], [35, 315], [35, 327], [35, 336], [35, 909], [35, 910], [35, 932], [35, 936], [35, 971], [35, 985], [35, 991], [35, 998], [35, 1140], [35, 1144], [35, 1156], [35, 1164], [35, 1168], [35, 1196], [35, 1213], [35, 1220], [35, 1226], [36, 40], [36, 85], [36, 133], [36, 213], [36, 233], [36, 235], [36, 266], [36, 306], [36, 322], [36, 323], [36, 324], [36, 910], [36, 1166], [36, 1197], [36, 1228], [37, 78], [37, 85], [37, 120], [37, 154], [37, 214], [37, 311], [37, 319], [37, 336], [37, 407], [37, 910], [37, 911], [38, 42], [38, 85], [38, 125], [38, 138], [38, 155], [38, 215], [38, 271], [38, 315], [38, 320], [38, 383], [38, 389], [38, 910], [38, 912], [38, 1145], [38, 1198], [38, 1268], [39, 79], [39, 85], [39, 108], [39, 129], [39, 144], [39, 156], [39, 216], [39, 272], [39, 322], [39, 370], [39, 378], [39, 389], [39, 893], [39, 896], [39, 910], [39, 913], [39, 1146], [39, 1180], [39, 1206], [39, 1224], [39, 1244], [40, 80], [40, 85], [40, 109], [40, 113], [40, 117], [40, 175], [40, 323], [40, 370], [40, 371], [40, 910], [40, 914], [40, 952], [40, 955], [40, 1194], [40, 1218], [40, 1220], [40, 1227], [41, 85], [41, 91], [41, 99], [41, 106], [41, 112], [41, 123], [41, 127], [41, 131], [41, 133], [41, 135], [41, 138], [41, 157], [41, 173], [41, 188], [41, 371], [41, 408], [41, 879], [41, 910], [41, 915], [41, 944], [41, 948], [41, 953], [41, 957], [41, 986], [41, 1185], [41, 1212], [41, 1226], [41, 1228], [41, 1229], [41, 1245], [41, 1281], [42, 82], [42, 85], [42, 194], [42, 273], [42, 324], [42, 378], [42, 383], [42, 409], [42, 910], [42, 916], [42, 998], [42, 1147], [42, 1192], [42, 1207], [42, 1247], [42, 1268], [43, 86], [43, 87], [43, 88], [43, 89], [43, 90], [43, 91], [43, 92], [43, 158], [43, 207], [43, 217], [43, 251], [43, 274], [43, 288], [43, 295], [43, 879], [43, 917], [43, 918], [43, 944], [43, 961], [43, 1106], [43, 1107], [43, 1148], [43, 1193], [43, 1208], [43, 1209], [43, 1210], [44, 47], [44, 74], [44, 84], [44, 92], [44, 115], [44, 122], [44, 134], [44, 218], [44, 232], [44, 235], [44, 252], [44, 295], [44, 304], [44, 325], [44, 326], [44, 327], [44, 918], [44, 954], [44, 983], [44, 989], [44, 1164], [44, 1166], [44, 1181], [44, 1223], [45, 92], [45, 114], [45, 136], [45, 186], [45, 191], [45, 196], [45, 276], [45, 325], [45, 372], [45, 918], [45, 919], [45, 978], [45, 992], [45, 996], [45, 1002], [45, 1150], [45, 1219], [45, 1230], [45, 1246], [46, 49], [46, 92], [46, 129], [46, 139], [46, 144], [46, 145], [46, 245], [46, 246], [46, 277], [46, 326], [46, 893], [46, 895], [46, 918], [46, 920], [46, 1058], [46, 1151], [46, 1173], [46, 1174], [46, 1201], [46, 1209], [46, 1224], [46, 1271], [47, 87], [47, 92], [47, 115], [47, 175], [47, 198], [47, 372], [47, 918], [47, 921], [47, 951], [47, 955], [47, 993], [47, 1011], [47, 1178], [47, 1227], [47, 1259], [47, 1270], [48, 92], [48, 107], [48, 113], [48, 119], [48, 132], [48, 134], [48, 136], [48, 137], [48, 139], [48, 159], [48, 189], [48, 879], [48, 918], [48, 922], [48, 952], [48, 958], [48, 987], [48, 1217], [48, 1222], [48, 1230], [48, 1231], [48, 1241], [48, 1281], [49, 89], [49, 92], [49, 153], [49, 196], [49, 219], [49, 226], [49, 278], [49, 327], [49, 414], [49, 909], [49, 918], [49, 923], [49, 1002], [49, 1040], [49, 1152], [49, 1160], [49, 1200], [49, 1210], [49, 1254], [49, 1271], [49, 1292], [50, 53], [50, 77], [50, 152], [50, 153], [50, 253], [50, 273], [50, 373], [50, 409], [50, 909], [50, 924], [50, 1189], [50, 1202], [50, 1247], [51, 93], [51, 94], [51, 95], [51, 96], [51, 97], [51, 98], [51, 99], [51, 160], [51, 203], [51, 220], [51, 282], [51, 289], [51, 679], [51, 879], [51, 925], [51, 970], [51, 1139], [51, 1211], [51, 1212], [52, 54], [52, 75], [52, 101], [52, 110], [52, 172], [52, 209], [52, 227], [52, 289], [52, 926], [52, 946], [52, 984], [52, 1115], [52, 1161], [52, 1213], [53, 56], [53, 93], [53, 114], [53, 195], [53, 197], [53, 221], [53, 234], [53, 279], [53, 328], [53, 375], [53, 411], [53, 927], [53, 1001], [53, 1005], [53, 1153], [53, 1165], [53, 1167], [53, 1191], [53, 1219], [53, 1249], [54, 95], [54, 129], [54, 250], [54, 253], [54, 255], [54, 928], [54, 1224], [55, 96], [55, 110], [55, 146], [55, 162], [55, 373], [55, 374], [55, 375], [55, 376], [55, 377], [55, 378], [55, 379], [55, 898], [55, 929], [55, 968], [55, 1227], [55, 1251], [56, 97], [56, 135], [56, 216], [56, 280], [56, 378], [56, 930], [56, 1127], [56, 1146], [56, 1147], [56, 1154], [56, 1202], [56, 1211], [56, 1229], [56, 1252], [56, 1284], [56, 1286], [57, 174], [57, 379], [57, 413], [57, 879], [57, 884], [57, 931], [57, 945], [57, 950], [57, 969], [57, 972], [57, 1188], [57, 1189], [57, 1190], [57, 1191], [57, 1192], [57, 1242], [57, 1253], [58, 62], [58, 86], [58, 163], [58, 235], [58, 257], [58, 278], [58, 306], [58, 329], [58, 380], [58, 414], [58, 932], [58, 934], [58, 1166], [58, 1208], [58, 1254], [59, 100], [59, 101], [59, 102], [59, 103], [59, 104], [59, 105], [59, 106], [59, 107], [59, 164], [59, 209], [59, 222], [59, 281], [59, 298], [59, 879], [59, 933], [59, 934], [59, 945], [59, 953], [59, 976], [59, 1115], [59, 1140], [59, 1155], [59, 1213], [59, 1214], [59, 1215], [59, 1216], [59, 1217], [60, 64], [60, 83], [60, 98], [60, 117], [60, 126], [60, 223], [60, 282], [60, 298], [60, 329], [60, 330], [60, 331], [60, 934], [60, 970], [60, 1156], [60, 1205], [60, 1220], [61, 65], [61, 84], [61, 114], [61, 130], [61, 165], [61, 224], [61, 283], [61, 330], [61, 381], [61, 934], [61, 935], [61, 954], [61, 991], [61, 1157], [61, 1167], [61, 1195], [61, 1219], [61, 1225], [61, 1255], [61, 1294], [62, 66], [62, 100], [62, 120], [62, 166], [62, 225], [62, 284], [62, 306], [62, 307], [62, 382], [62, 384], [62, 387], [62, 415], [62, 934], [62, 936], [62, 1158], [62, 1203], [62, 1256], [63, 67], [63, 125], [63, 285], [63, 358], [63, 360], [63, 392], [63, 934], [63, 937], [63, 1000], [63, 1159], [64, 102], [64, 167], [64, 331], [64, 380], [64, 381], [64, 382], [64, 383], [64, 418], [64, 897], [64, 934], [64, 938], [64, 1214], [64, 1227], [64, 1258], [64, 1268], [65, 103], [65, 117], [65, 130], [65, 135], [65, 143], [65, 146], [65, 153], [65, 168], [65, 215], [65, 257], [65, 258], [65, 360], [65, 373], [65, 383], [65, 387], [65, 416], [65, 425], [65, 892], [65, 898], [65, 909], [65, 914], [65, 934], [65, 939], [65, 951], [65, 957], [65, 1145], [65, 1147], [65, 1180], [65, 1204], [65, 1215], [65, 1220], [65, 1225], [65, 1229], [65, 1244], [65, 1257], [65, 1268], [65, 1288], [65, 1289], [65, 1295], [66, 104], [66, 137], [66, 169], [66, 226], [66, 286], [66, 384], [66, 934], [66, 940], [66, 1160], [66, 1216], [66, 1231], [67, 126], [67, 392], [67, 880], [67, 901], [67, 934], [67, 941], [67, 968], [67, 972], [67, 1192], [67, 1251], [67, 1426], [68, 142], [68, 170], [68, 417], [68, 879], [68, 891], [68, 934], [68, 942], [68, 1193], [68, 1194], [68, 1195], [68, 1244], [68, 1251], [69, 73], [69, 90], [69, 106], [69, 114], [69, 122], [69, 131], [69, 171], [69, 201], [69, 221], [69, 224], [69, 228], [69, 232], [69, 240], [69, 288], [69, 877], [69, 943], [69, 953], [69, 961], [69, 982], [69, 988], [69, 1137], [69, 1150], [69, 1153], [69, 1157], [69, 1162], [69, 1164], [69, 1219], [69, 1223], [69, 1226], [70, 74], [70, 91], [70, 120], [70, 133], [70, 205], [70, 214], [70, 225], [70, 233], [70, 238], [70, 244], [70, 295], [70, 303], [70, 308], [70, 514], [70, 877], [70, 944], [70, 983], [70, 994], [70, 1142], [70, 1158], [70, 1228], [71, 75], [71, 98], [71, 125], [71, 130], [71, 215], [71, 229], [71, 241], [71, 259], [71, 289], [71, 298], [71, 309], [71, 877], [71, 945], [71, 970], [71, 984], [71, 999], [71, 1145], [71, 1159], [71, 1163], [71, 1186], [71, 1190], [71, 1225], [72, 76], [72, 129], [72, 172], [72, 216], [72, 234], [72, 260], [72, 299], [72, 877], [72, 946], [72, 985], [72, 1003], [72, 1146], [72, 1151], [72, 1165], [72, 1171], [72, 1187], [72, 1224], [73, 108], [73, 135], [73, 149], [73, 153], [73, 156], [73, 188], [73, 230], [73, 242], [73, 245], [73, 249], [73, 253], [73, 877], [73, 900], [73, 904], [73, 909], [73, 913], [73, 947], [73, 986], [73, 1009], [73, 1123], [73, 1154], [73, 1170], [73, 1173], [73, 1177], [73, 1229], [73, 1271], [74, 109], [74, 122], [74, 133], [74, 137], [74, 151], [74, 173], [74, 189], [74, 198], [74, 226], [74, 235], [74, 243], [74, 246], [74, 254], [74, 257], [74, 407], [74, 877], [74, 908], [74, 914], [74, 948], [74, 987], [74, 1011], [74, 1160], [74, 1166], [74, 1174], [74, 1178], [74, 1182], [74, 1218], [74, 1223], [74, 1228], [74, 1231], [74, 1271], [75, 110], [75, 130], [75, 162], [75, 877], [75, 884], [75, 888], [75, 901], [75, 906], [75, 926], [75, 929], [75, 949], [75, 1147], [75, 1167], [75, 1179], [75, 1183], [75, 1192], [75, 1225], [76, 111], [76, 131], [76, 142], [76, 146], [76, 174], [76, 219], [76, 231], [76, 236], [76, 250], [76, 877], [76, 891], [76, 898], [76, 950], [76, 1124], [76, 1152], [76, 1168], [76, 1180], [76, 1190], [76, 1226], [77, 114], [77, 115], [77, 116], [77, 117], [77, 118], [77, 119], [77, 232], [77, 290], [77, 341], [77, 347], [77, 879], [77, 951], [77, 1164], [77, 1189], [77, 1194], [77, 1219], [77, 1220], [77, 1221], [77, 1222], [78, 81], [78, 113], [78, 120], [78, 134], [78, 233], [78, 302], [78, 323], [78, 344], [78, 952], [79, 83], [79, 106], [79, 129], [79, 133], [79, 234], [79, 244], [79, 246], [79, 255], [79, 259], [79, 266], [79, 283], [79, 291], [79, 298], [79, 321], [79, 332], [79, 341], [79, 352], [79, 363], [79, 365], [79, 375], [79, 388], [79, 953], [79, 1165], [79, 1174], [79, 1186], [79, 1195], [79, 1196], [79, 1224], [79, 1228], [80, 84], [80, 304], [80, 307], [80, 321], [80, 330], [80, 347], [80, 358], [80, 384], [80, 385], [80, 954], [80, 1197], [80, 1227], [81, 115], [81, 137], [81, 175], [81, 235], [81, 292], [81, 323], [81, 365], [81, 380], [81, 955], [81, 1166], [81, 1231], [82, 116], [82, 145], [82, 176], [82, 293], [82, 333], [82, 375], [82, 381], [82, 895], [82, 956], [82, 1167], [82, 1188], [82, 1192], [82, 1198], [83, 117], [83, 133], [83, 225], [83, 226], [83, 236], [83, 294], [83, 384], [83, 418], [83, 957], [83, 1158], [83, 1160], [83, 1168], [83, 1206], [83, 1220], [83, 1228], [83, 1258], [83, 1264], [84, 118], [84, 134], [84, 150], [84, 252], [84, 385], [84, 418], [84, 878], [84, 907], [84, 914], [84, 958], [84, 1178], [84, 1181], [84, 1195], [84, 1204], [84, 1221], [84, 1232], [84, 1258], [84, 1259], [85, 177], [85, 213], [85, 237], [85, 251], [85, 879], [85, 914], [85, 959], [85, 1007], [85, 1087], [85, 1107], [85, 1109], [85, 1140], [85, 1180], [85, 1196], [85, 1197], [85, 1198], [86, 120], [86, 121], [86, 122], [86, 123], [86, 238], [86, 288], [86, 334], [86, 349], [86, 498], [86, 879], [86, 960], [86, 1169], [86, 1203], [86, 1223], [87, 90], [87, 113], [87, 136], [87, 276], [87, 295], [87, 307], [87, 349], [87, 356], [87, 359], [87, 363], [87, 386], [87, 387], [87, 951], [87, 952], [87, 961], [87, 989], [87, 994], [87, 1169], [87, 1227], [87, 1230], [88, 91], [88, 135], [88, 171], [88, 258], [88, 260], [88, 296], [88, 419], [88, 943], [88, 944], [88, 962], [88, 996], [88, 1169], [88, 1170], [88, 1187], [88, 1229], [89, 121], [89, 153], [89, 154], [89, 178], [89, 225], [89, 310], [89, 336], [89, 387], [89, 909], [89, 911], [89, 963], [89, 1158], [89, 1169], [89, 1193], [89, 1201], [89, 1209], [89, 1261], [90, 122], [90, 136], [90, 148], [90, 149], [90, 158], [90, 189], [90, 252], [90, 419], [90, 878], [90, 899], [90, 904], [90, 917], [90, 921], [90, 964], [90, 987], [90, 1010], [90, 1150], [90, 1169], [90, 1181], [90, 1223], [90, 1230], [90, 1233], [90, 1237], [90, 1260], [90, 1261], [91, 123], [91, 147], [91, 149], [91, 173], [91, 208], [91, 249], [91, 251], [91, 881], [91, 904], [91, 948], [91, 965], [91, 994], [91, 1007], [91, 1079], [91, 1088], [91, 1099], [91, 1107], [91, 1112], [91, 1118], [91, 1169], [91, 1177], [91, 1182], [91, 1388], [92, 173], [92, 207], [92, 218], [92, 239], [92, 879], [92, 921], [92, 948], [92, 958], [92, 966], [92, 989], [92, 996], [92, 1008], [92, 1092], [92, 1106], [92, 1109], [92, 1126], [92, 1169], [92, 1178], [92, 1199], [92, 1200], [92, 1201], [93, 97], [93, 114], [93, 179], [93, 234], [93, 240], [93, 312], [93, 314], [93, 388], [93, 420], [93, 967], [93, 1165], [93, 1202], [93, 1211], [93, 1219], [93, 1262], [94, 124], [94, 125], [94, 126], [94, 127], [94, 241], [94, 282], [94, 340], [94, 879], [94, 968], [95, 129], [95, 250], [95, 289], [95, 302], [95, 316], [95, 969], [95, 1190], [95, 1224], [96, 98], [96, 285], [96, 309], [96, 316], [96, 388], [96, 389], [96, 390], [96, 970], [96, 1227], [97, 99], [97, 124], [97, 135], [97, 181], [97, 215], [97, 216], [97, 242], [97, 261], [97, 306], [97, 337], [97, 389], [97, 421], [97, 971], [97, 1145], [97, 1146], [97, 1212], [97, 1229], [97, 1263], [98, 126], [98, 142], [98, 160], [98, 162], [98, 420], [98, 421], [98, 878], [98, 891], [98, 897], [98, 925], [98, 929], [98, 972], [98, 1159], [98, 1183], [98, 1190], [98, 1262], [98, 1263], [98, 1264], [99, 127], [99, 183], [99, 294], [99, 338], [99, 390], [99, 973], [99, 1007], [99, 1117], [99, 1211], [99, 1264], [100, 104], [100, 120], [100, 184], [100, 244], [100, 307], [100, 313], [100, 319], [100, 321], [100, 422], [100, 974], [100, 1172], [100, 1216], [101, 128], [101, 129], [101, 130], [101, 131], [101, 132], [101, 283], [101, 289], [101, 297], [101, 339], [101, 351], [101, 879], [101, 975], [101, 1171], [101, 1172], [101, 1180], [101, 1224], [101, 1225], [101, 1226], [102, 105], [102, 285], [102, 298], [102, 340], [102, 351], [102, 391], [102, 392], [102, 968], [102, 976], [102, 999], [102, 1172], [102, 1194], [102, 1195], [102, 1205], [102, 1227], [103, 106], [103, 135], [103, 185], [103, 245], [103, 261], [103, 299], [103, 305], [103, 341], [103, 391], [103, 951], [103, 953], [103, 977], [103, 991], [103, 1003], [103, 1127], [103, 1172], [103, 1173], [103, 1229], [103, 1265], [103, 1296], [104, 107], [104, 128], [104, 137], [104, 186], [104, 246], [104, 300], [104, 321], [104, 325], [104, 342], [104, 402], [104, 403], [104, 423], [104, 978], [104, 1128], [104, 1172], [104, 1174], [104, 1217], [104, 1231], [104, 1238], [104, 1239], [105, 130], [105, 143], [105, 145], [105, 164], [105, 167], [105, 392], [105, 422], [105, 423], [105, 878], [105, 892], [105, 895], [105, 901], [105, 905], [105, 933], [105, 938], [105, 979], [105, 1159], [105, 1172], [105, 1214], [105, 1225], [105, 1265], [105, 1266], [106, 131], [106, 142], [106, 145], [106, 150], [106, 156], [106, 187], [106, 224], [106, 234], [106, 247], [106, 277], [106, 293], [106, 313], [106, 403], [106, 411], [106, 420], [106, 891], [106, 895], [106, 907], [106, 913], [106, 957], [106, 980], [106, 1003], [106, 1007], [106, 1108], [106, 1118], [106, 1120], [106, 1129], [106, 1133], [106, 1140], [106, 1157], [106, 1165], [106, 1172], [106, 1195], [106, 1206], [106, 1215], [106, 1226], [106, 1237], [106, 1239], [106, 1249], [106, 1262], [107, 132], [107, 248], [107, 301], [107, 343], [107, 402], [107, 981], [107, 1008], [107, 1130], [107, 1172], [107, 1175], [107, 1216], [107, 1237], [107, 1238], [107, 1296], [108, 112], [108, 135], [108, 249], [108, 280], [108, 296], [108, 299], [108, 303], [108, 355], [108, 397], [108, 960], [108, 982], [108, 994], [108, 1176], [108, 1177], [108, 1180], [108, 1185], [108, 1196], [108, 1229], [109, 113], [109, 137], [109, 286], [109, 292], [109, 300], [109, 304], [109, 308], [109, 344], [109, 349], [109, 354], [109, 357], [109, 397], [109, 952], [109, 983], [109, 1176], [109, 1197], [109, 1231], [110, 273], [110, 293], [110, 305], [110, 309], [110, 318], [110, 351], [110, 968], [110, 969], [110, 984], [110, 1003], [110, 1176], [110, 1192], [111, 250], [111, 278], [111, 294], [111, 306], [111, 316], [111, 352], [111, 985], [111, 1176], [112, 133], [112, 151], [112, 154], [112, 188], [112, 213], [112, 234], [112, 236], [112, 310], [112, 311], [112, 312], [112, 908], [112, 911], [112, 986], [112, 1007], [112, 1165], [112, 1168], [112, 1176], [112, 1228], [113, 134], [113, 150], [113, 154], [113, 175], [113, 189], [113, 218], [113, 233], [113, 251], [113, 301], [113, 307], [113, 313], [113, 907], [113, 911], [113, 955], [113, 987], [113, 994], [113, 1008], [113, 1107], [113, 1176], [113, 1197], [113, 1218], [114, 135], [114, 136], [114, 345], [114, 393], [114, 396], [114, 879], [114, 988], [114, 1191], [114, 1195], [114, 1199], [114, 1202], [114, 1204], [114, 1229], [114, 1230], [114, 1267], [115, 119], [115, 137], [115, 304], [115, 353], [115, 385], [115, 396], [115, 989], [115, 1178], [115, 1222], [115, 1231], [116, 190], [116, 305], [116, 990], [116, 1179], [116, 1189], [116, 1192], [116, 1207], [116, 1265], [117, 271], [117, 273], [117, 306], [117, 330], [117, 341], [117, 346], [117, 351], [117, 355], [117, 370], [117, 383], [117, 394], [117, 414], [117, 415], [117, 421], [117, 991], [117, 1180], [117, 1194], [117, 1197], [117, 1254], [117, 1256], [117, 1263], [117, 1268], [118, 191], [118, 325], [118, 347], [118, 356], [118, 383], [118, 391], [118, 402], [118, 409], [118, 878], [118, 951], [118, 954], [118, 992], [118, 1234], [118, 1238], [118, 1247], [118, 1267], [118, 1268], [118, 1269], [119, 136], [119, 252], [119, 307], [119, 348], [119, 385], [119, 395], [119, 415], [119, 422], [119, 993], [119, 1008], [119, 1181], [119, 1230], [119, 1256], [119, 1269], [120, 137], [120, 308], [120, 367], [120, 879], [120, 994], [120, 1182], [120, 1208], [120, 1231], [121, 192], [121, 310], [121, 995], [121, 1208], [121, 1210], [122, 325], [122, 349], [122, 355], [122, 367], [122, 396], [122, 401], [122, 403], [122, 410], [122, 414], [122, 878], [122, 960], [122, 961], [122, 983], [122, 996], [122, 1178], [122, 1208], [122, 1235], [122, 1239], [122, 1248], [122, 1254], [122, 1270], [122, 1271], [123, 193], [123, 249], [123, 277], [123, 278], [123, 397], [123, 944], [123, 997], [123, 1007], [123, 1177], [123, 1208], [123, 1271], [124, 127], [124, 135], [124, 194], [124, 253], [124, 305], [124, 306], [124, 324], [124, 373], [124, 998], [124, 1211], [124, 1229], [125, 138], [125, 309], [125, 340], [125, 399], [125, 553], [125, 879], [125, 999], [125, 1183], [125, 1192], [126, 318], [126, 377], [126, 552], [126, 878], [126, 968], [126, 970], [126, 1000], [127, 138], [127, 195], [127, 255], [127, 293], [127, 294], [127, 321], [127, 328], [127, 1001], [127, 1007], [127, 1128], [127, 1212], [128, 132], [128, 137], [128, 196], [128, 257], [128, 325], [128, 327], [128, 380], [128, 383], [128, 424], [128, 1002], [128, 1213], [128, 1216], [128, 1231], [128, 1268], [129, 139], [129, 341], [129, 350], [129, 368], [129, 398], [129, 879], [129, 1003], [129, 1196], [129, 1200], [129, 1206], [129, 1209], [129, 1213], [129, 1272], [130, 351], [130, 368], [130, 399], [130, 424], [130, 425], [130, 878], [130, 975], [130, 976], [130, 984], [130, 991], [130, 1004], [130, 1183], [130, 1204], [130, 1213], [130, 1272], [130, 1273], [131, 197], [131, 310], [131, 317], [131, 328], [131, 352], [131, 425], [131, 953], [131, 985], [131, 1005], [131, 1007], [131, 1131], [131, 1180], [131, 1184], [131, 1213], [131, 1261], [132, 139], [132, 165], [132, 166], [132, 258], [132, 383], [132, 935], [132, 936], [132, 1006], [132, 1008], [132, 1213], [132, 1217], [132, 1261], [132, 1268], [133, 311], [133, 338], [133, 352], [133, 354], [133, 355], [133, 371], [133, 407], [133, 983], [133, 1007], [133, 1182], [133, 1185], [133, 1206], [134, 343], [134, 348], [134, 356], [134, 357], [134, 367], [134, 407], [134, 952], [134, 954], [134, 1008], [134, 1178], [135, 355], [135, 426], [135, 879], [135, 982], [135, 991], [135, 996], [135, 1007], [135, 1009], [135, 1211], [135, 1215], [135, 1219], [135, 1241], [135, 1244], [136, 191], [136, 356], [136, 369], [136, 396], [136, 400], [136, 961], [136, 992], [136, 1008], [136, 1010], [136, 1199], [136, 1219], [136, 1222], [136, 1241], [136, 1274], [137, 198], [137, 357], [137, 396], [137, 427], [137, 807], [137, 879], [137, 983], [137, 1008], [137, 1011], [137, 1216], [137, 1218], [138, 259], [138, 312], [138, 317], [138, 321], [138, 358], [138, 420], [138, 1007], [138, 1012], [138, 1186], [138, 1262], [139, 145], [139, 199], [139, 260], [139, 313], [139, 363], [139, 422], [139, 895], [139, 896], [139, 1008], [139, 1013], [139, 1187], [139, 1200], [139, 1224], [140, 160], [140, 161], [140, 162], [140, 875], [140, 924], [140, 925], [140, 926], [140, 927], [140, 928], [140, 929], [140, 930], [140, 931], [140, 1014], [140, 1036], [140, 1048], [140, 1077], [140, 1083], [141, 163], [141, 164], [141, 165], [141, 166], [141, 167], [141, 168], [141, 169], [141, 170], [141, 202], [141, 875], [141, 932], [141, 933], [141, 934], [141, 935], [141, 936], [141, 937], [141, 938], [141, 939], [141, 940], [141, 941], [141, 942], [141, 1015], [141, 1041], [141, 1076], [141, 1097], [142, 156], [142, 162], [142, 170], [142, 202], [142, 203], [142, 204], [142, 205], [142, 877], [142, 894], [142, 913], [142, 929], [142, 942], [142, 953], [142, 970], [142, 985], [142, 1016], [142, 1077], [142, 1097], [142, 1138], [142, 1139], [142, 1140], [142, 1141], [142, 1142], [143, 145], [143, 170], [143, 246], [143, 248], [143, 363], [143, 402], [143, 895], [143, 914], [143, 942], [143, 951], [143, 976], [143, 991], [143, 1017], [143, 1026], [143, 1029], [143, 1140], [143, 1174], [143, 1175], [143, 1194], [143, 1238], [143, 1276], [144, 163], [144, 170], [144, 192], [144, 204], [144, 246], [144, 247], [144, 403], [144, 889], [144, 896], [144, 932], [144, 942], [144, 960], [144, 995], [144, 1018], [144, 1078], [144, 1108], [144, 1109], [144, 1141], [144, 1174], [144, 1196], [144, 1200], [144, 1239], [144, 1277], [145, 165], [145, 170], [145, 187], [145, 404], [145, 408], [145, 425], [145, 892], [145, 935], [145, 942], [145, 953], [145, 958], [145, 976], [145, 980], [145, 988], [145, 1019], [145, 1058], [145, 1060], [145, 1069], [145, 1188], [145, 1195], [145, 1200], [145, 1207], [145, 1240], [145, 1242], [145, 1245], [145, 1265], [145, 1278], [145, 1292], [146, 170], [146, 174], [146, 178], [146, 192], [146, 379], [146, 406], [146, 425], [146, 875], [146, 923], [146, 942], [146, 950], [146, 957], [146, 963], [146, 985], [146, 991], [146, 995], [146, 1003], [146, 1020], [146, 1280], [147, 171], [147, 172], [147, 173], [147, 174], [147, 206], [147, 207], [147, 208], [147, 209], [147, 875], [147, 876], [147, 877], [147, 943], [147, 944], [147, 945], [147, 946], [147, 947], [147, 948], [147, 949], [147, 950], [147, 1016], [147, 1021], [147, 1062], [147, 1063], [147, 1076], [147, 1080], [147, 1084], [147, 1091], [147, 1094], [147, 1096], [147, 1101], [147, 1106], [147, 1111], [147, 1112], [147, 1113], [147, 1115], [148, 168], [148, 185], [148, 191], [148, 207], [148, 876], [148, 903], [148, 921], [148, 939], [148, 951], [148, 961], [148, 977], [148, 992], [148, 1017], [148, 1022], [148, 1024], [148, 1064], [148, 1065], [148, 1081], [148, 1106], [148, 1150], [149, 189], [149, 408], [149, 419], [149, 876], [149, 900], [149, 944], [149, 961], [149, 965], [149, 982], [149, 987], [149, 994], [149, 1023], [149, 1028], [149, 1035], [149, 1040], [149, 1054], [149, 1058], [149, 1072], [149, 1241], [149, 1245], [149, 1271], [150, 175], [150, 176], [150, 177], [150, 210], [150, 218], [150, 224], [150, 875], [150, 882], [150, 951], [150, 952], [150, 953], [150, 954], [150, 955], [150, 956], [150, 957], [150, 958], [150, 959], [150, 1024], [150, 1084], [150, 1143], [150, 1157], [151, 154], [151, 173], [151, 177], [151, 193], [151, 206], [151, 218], [151, 911], [151, 948], [151, 959], [151, 960], [151, 983], [151, 997], [151, 1025], [151, 1101], [151, 1125], [151, 1185], [152, 155], [152, 177], [152, 195], [152, 211], [152, 273], [152, 912], [152, 959], [152, 968], [152, 1001], [152, 1026], [152, 1085], [152, 1194], [153, 156], [153, 168], [153, 174], [153, 177], [153, 197], [153, 202], [153, 207], [153, 212], [153, 224], [153, 245], [153, 913], [153, 939], [153, 950], [153, 959], [153, 975], [153, 982], [153, 991], [153, 1005], [153, 1027], [153, 1037], [153, 1040], [153, 1055], [153, 1068], [153, 1086], [153, 1097], [153, 1106], [153, 1110], [153, 1144], [153, 1157], [153, 1173], [153, 1180], [153, 1189], [153, 1201], [153, 1210], [154, 177], [154, 214], [154, 407], [154, 908], [154, 952], [154, 959], [154, 994], [154, 1028], [154, 1088], [154, 1185], [154, 1193], [154, 1210], [154, 1281], [155, 177], [155, 215], [155, 416], [155, 421], [155, 916], [155, 959], [155, 999], [155, 1012], [155, 1029], [155, 1089], [155, 1145], [155, 1189], [155, 1194], [155, 1257], [155, 1263], [156, 177], [156, 216], [156, 273], [156, 421], [156, 891], [156, 909], [156, 953], [156, 959], [156, 982], [156, 1003], [156, 1018], [156, 1030], [156, 1090], [156, 1146], [156, 1196], [156, 1244], [156, 1252], [156, 1263], [157, 177], [157, 183], [157, 187], [157, 188], [157, 193], [157, 195], [157, 197], [157, 408], [157, 875], [157, 959], [157, 965], [157, 973], [157, 980], [157, 986], [157, 997], [157, 1001], [157, 1005], [157, 1007], [157, 1009], [157, 1012], [157, 1031], [157, 1047], [157, 1062], [157, 1245], [157, 1282], [158, 178], [158, 217], [158, 228], [158, 875], [158, 889], [158, 960], [158, 961], [158, 962], [158, 963], [158, 964], [158, 965], [158, 966], [158, 1032], [158, 1081], [158, 1091], [158, 1125], [158, 1148], [158, 1162], [158, 1169], [159, 189], [159, 198], [159, 199], [159, 875], [159, 966], [159, 981], [159, 987], [159, 993], [159, 1006], [159, 1008], [159, 1010], [159, 1011], [159, 1013], [159, 1033], [159, 1063], [160, 179], [160, 180], [160, 181], [160, 182], [160, 183], [160, 220], [160, 223], [160, 229], [160, 875], [160, 883], [160, 967], [160, 968], [160, 969], [160, 970], [160, 971], [160, 972], [160, 973], [160, 1034], [160, 1077], [160, 1094], [160, 1156], [160, 1163], [160, 1553], [161, 180], [161, 280], [161, 412], [161, 883], [161, 994], [161, 1035], [161, 1250], [161, 1286], [162, 409], [162, 410], [162, 411], [162, 412], [162, 413], [162, 878], [162, 883], [162, 891], [162, 901], [162, 970], [162, 984], [162, 1020], [162, 1036], [162, 1247], [162, 1248], [162, 1249], [162, 1250], [162, 1251], [162, 1252], [162, 1253], [163, 166], [163, 219], [163, 414], [163, 890], [163, 893], [163, 936], [163, 960], [163, 1037], [163, 1109], [163, 1131], [163, 1152], [163, 1180], [163, 1203], [163, 1254], [163, 1288], [164, 184], [164, 185], [164, 186], [164, 187], [164, 222], [164, 875], [164, 890], [164, 974], [164, 975], [164, 976], [164, 977], [164, 978], [164, 979], [164, 980], [164, 981], [164, 1038], [164, 1083], [164, 1096], [164, 1155], [164, 1172], [165, 168], [165, 224], [165, 890], [165, 895], [165, 939], [165, 958], [165, 988], [165, 1004], [165, 1039], [165, 1098], [165, 1157], [165, 1204], [165, 1255], [166, 169], [166, 184], [166, 225], [166, 252], [166, 327], [166, 415], [166, 418], [166, 890], [166, 896], [166, 932], [166, 940], [166, 974], [166, 994], [166, 1040], [166, 1099], [166, 1158], [166, 1180], [166, 1181], [166, 1256], [166, 1258], [166, 1261], [166, 1289], [167, 414], [167, 415], [167, 416], [167, 417], [167, 878], [167, 890], [167, 934], [167, 976], [167, 1041], [167, 1205], [167, 1254], [167, 1255], [167, 1256], [167, 1257], [167, 1292], [168, 185], [168, 409], [168, 416], [168, 890], [168, 899], [168, 909], [168, 935], [168, 977], [168, 991], [168, 1004], [168, 1009], [168, 1017], [168, 1020], [168, 1027], [168, 1042], [168, 1089], [168, 1131], [168, 1132], [168, 1234], [168, 1247], [168, 1257], [168, 1261], [168, 1290], [168, 1299], [169, 186], [169, 198], [169, 226], [169, 418], [169, 890], [169, 936], [169, 978], [169, 1011], [169, 1043], [169, 1100], [169, 1160], [169, 1258], [170, 417], [170, 875], [170, 889], [170, 890], [170, 891], [170, 892], [170, 893], [170, 894], [170, 895], [170, 896], [170, 897], [170, 898], [170, 1016], [170, 1044], [170, 1291], [171, 187], [171, 197], [171, 207], [171, 228], [171, 881], [171, 903], [171, 947], [171, 964], [171, 980], [171, 988], [171, 996], [171, 1005], [171, 1045], [171, 1075], [171, 1095], [171, 1098], [171, 1102], [171, 1106], [171, 1114], [171, 1162], [172, 174], [172, 245], [172, 247], [172, 881], [172, 906], [172, 950], [172, 1003], [172, 1046], [172, 1090], [172, 1108], [172, 1134], [172, 1173], [173, 198], [173, 881], [173, 908], [173, 918], [173, 944], [173, 983], [173, 996], [173, 1007], [173, 1011], [173, 1025], [173, 1047], [173, 1063], [173, 1072], [173, 1100], [173, 1109], [173, 1117], [173, 1120], [173, 1128], [173, 1131], [173, 1281], [174, 197], [174, 231], [174, 881], [174, 886], [174, 898], [174, 909], [174, 946], [174, 985], [174, 1005], [174, 1016], [174, 1020], [174, 1048], [174, 1093], [174, 1105], [174, 1110], [174, 1124], [175, 198], [175, 235], [175, 403], [175, 414], [175, 907], [175, 952], [175, 989], [175, 1011], [175, 1049], [175, 1109], [175, 1166], [175, 1197], [175, 1239], [175, 1254], [176, 190], [176, 411], [176, 882], [176, 888], [176, 907], [176, 916], [176, 990], [176, 1019], [176, 1050], [176, 1167], [176, 1207], [176, 1249], [176, 1255], [177, 237], [177, 875], [177, 907], [177, 908], [177, 909], [177, 910], [177, 911], [177, 912], [177, 913], [177, 914], [177, 915], [177, 916], [177, 1051], [177, 1087], [177, 1111], [177, 1125], [178, 192], [178, 258], [178, 359], [178, 889], [178, 898], [178, 917], [178, 923], [178, 995], [178, 1027], [178, 1028], [178, 1052], [178, 1099], [178, 1184], [178, 1210], [178, 1261], [179, 181], [179, 240], [179, 247], [179, 259], [179, 263], [179, 283], [179, 420], [179, 925], [179, 927], [179, 971], [179, 988], [179, 1053], [179, 1108], [179, 1114], [179, 1186], [179, 1188], [179, 1262], [179, 1294], [180, 182], [180, 284], [180, 390], [180, 925], [180, 994], [180, 1054], [181, 183], [181, 194], [181, 200], [181, 242], [181, 421], [181, 925], [181, 930], [181, 967], [181, 973], [181, 998], [181, 1009], [181, 1055], [181, 1089], [181, 1090], [181, 1116], [181, 1135], [181, 1180], [181, 1211], [181, 1263], [181, 1295], [182, 198], [182, 243], [182, 286], [182, 390], [182, 925], [182, 1011], [182, 1056], [182, 1117], [183, 195], [183, 236], [183, 915], [183, 925], [183, 971], [183, 1001], [183, 1057], [183, 1168], [183, 1212], [183, 1264], [184, 186], [184, 244], [184, 252], [184, 260], [184, 299], [184, 301], [184, 326], [184, 422], [184, 933], [184, 936], [184, 978], [184, 994], [184, 1058], [184, 1118], [184, 1181], [184, 1187], [184, 1193], [184, 1195], [184, 1296], [185, 187], [185, 200], [185, 245], [185, 899], [185, 906], [185, 933], [185, 939], [185, 980], [185, 1009], [185, 1059], [185, 1119], [185, 1135], [185, 1173], [185, 1179], [185, 1215], [185, 1265], [186, 196], [186, 198], [186, 246], [186, 423], [186, 933], [186, 940], [186, 974], [186, 981], [186, 1002], [186, 1011], [186, 1060], [186, 1120], [186, 1174], [186, 1195], [186, 1199], [186, 1216], [186, 1276], [186, 1277], [186, 1297], [187, 197], [187, 247], [187, 260], [187, 895], [187, 906], [187, 915], [187, 933], [187, 943], [187, 953], [187, 977], [187, 1005], [187, 1016], [187, 1019], [187, 1024], [187, 1030], [187, 1061], [187, 1098], [187, 1108], [187, 1121], [187, 1151], [187, 1167], [187, 1187], [187, 1277], [187, 1285], [187, 1294], [188, 237], [188, 247], [188, 255], [188, 259], [188, 877], [188, 915], [188, 982], [188, 1007], [188, 1025], [188, 1028], [188, 1062], [188, 1087], [188, 1108], [188, 1110], [188, 1184], [188, 1185], [188, 1186], [189, 239], [189, 248], [189, 251], [189, 252], [189, 256], [189, 258], [189, 260], [189, 877], [189, 904], [189, 914], [189, 922], [189, 952], [189, 961], [189, 983], [189, 1008], [189, 1024], [189, 1028], [189, 1049], [189, 1063], [189, 1092], [189, 1107], [189, 1125], [189, 1175], [189, 1181], [189, 1187], [190, 386], [190, 885], [190, 888], [190, 951], [190, 956], [190, 1064], [190, 1179], [191, 416], [191, 880], [191, 899], [191, 951], [191, 958], [191, 1065], [191, 1199], [191, 1221], [191, 1230], [191, 1257], [191, 1265], [191, 1276], [191, 1283], [192, 257], [192, 278], [192, 361], [192, 374], [192, 893], [192, 898], [192, 960], [192, 963], [192, 1066], [192, 1184], [193, 219], [193, 230], [193, 908], [193, 915], [193, 960], [193, 965], [193, 1067], [193, 1123], [193, 1151], [193, 1152], [193, 1271], [194, 195], [194, 253], [194, 315], [194, 378], [194, 391], [194, 409], [194, 968], [194, 971], [194, 1001], [194, 1009], [194, 1068], [194, 1127], [194, 1179], [194, 1180], [194, 1198], [194, 1247], [195, 236], [195, 255], [195, 314], [195, 915], [195, 968], [195, 973], [195, 998], [195, 1012], [195, 1069], [195, 1129], [195, 1167], [195, 1168], [195, 1195], [195, 1202], [196, 198], [196, 257], [196, 414], [196, 416], [196, 424], [196, 975], [196, 978], [196, 1006], [196, 1011], [196, 1070], [196, 1131], [196, 1199], [196, 1201], [196, 1254], [196, 1257], [196, 1298], [197, 425], [197, 909], [197, 915], [197, 943], [197, 950], [197, 975], [197, 980], [197, 1071], [197, 1184], [197, 1191], [197, 1202], [197, 1226], [197, 1299], [198, 371], [198, 427], [198, 875], [198, 922], [198, 940], [198, 948], [198, 955], [198, 978], [198, 983], [198, 989], [198, 994], [198, 1002], [198, 1072], [198, 1231], [198, 1270], [198, 1301], [198, 1681], [199, 260], [199, 419], [199, 920], [199, 922], [199, 1003], [199, 1006], [199, 1019], [199, 1073], [199, 1134], [199, 1187], [199, 1237], [199, 1296], [200, 201], [200, 232], [200, 261], [200, 884], [200, 885], [200, 971], [200, 977], [200, 1074], [200, 1086], [200, 1089], [200, 1137], [200, 1164], [200, 1179], [200, 1189], [201, 221], [201, 263], [201, 401], [201, 884], [201, 887], [201, 1053], [201, 1075], [201, 1133], [201, 1135], [201, 1153], [201, 1191], [201, 1233], [201, 1235], [202, 222], [202, 223], [202, 224], [202, 225], [202, 226], [202, 877], [202, 890], [202, 891], [202, 901], [202, 909], [202, 934], [202, 1034], [202, 1076], [202, 1110], [202, 1155], [202, 1156], [202, 1157], [202, 1158], [202, 1159], [202, 1160], [203, 216], [203, 227], [203, 234], [203, 250], [203, 265], [203, 891], [203, 1077], [203, 1146], [203, 1161], [203, 1165], [204, 205], [204, 238], [204, 267], [204, 272], [204, 277], [204, 410], [204, 891], [204, 893], [204, 1078], [204, 1138], [204, 1142], [204, 1248], [205, 225], [205, 268], [205, 272], [205, 313], [205, 319], [205, 412], [205, 891], [205, 896], [205, 1079], [205, 1141], [205, 1158], [205, 1182], [205, 1250], [206, 227], [206, 228], [206, 229], [206, 230], [206, 231], [206, 877], [206, 881], [206, 908], [206, 926], [206, 1077], [206, 1080], [206, 1123], [206, 1124], [206, 1125], [206, 1161], [206, 1162], [206, 1163], [206, 1541], [207, 232], [207, 245], [207, 881], [207, 899], [207, 909], [207, 918], [207, 943], [207, 951], [207, 1074], [207, 1081], [207, 1084], [207, 1126], [207, 1164], [207, 1169], [207, 1173], [208, 238], [208, 249], [208, 656], [208, 881], [208, 900], [208, 944], [208, 960], [208, 1078], [208, 1082], [208, 1091], [208, 1177], [209, 241], [209, 881], [209, 901], [209, 945], [209, 968], [209, 1083], [209, 1085], [209, 1094], [209, 1127], [209, 1128], [209, 1129], [209, 1130], [209, 1172], [209, 1353], [210, 232], [210, 233], [210, 234], [210, 235], [210, 236], [210, 237], [210, 269], [210, 877], [210, 907], [210, 1084], [210, 1087], [210, 1164], [210, 1165], [210, 1166], [210, 1167], [210, 1168], [211, 215], [211, 237], [211, 241], [211, 255], [211, 266], [211, 1085], [211, 1087], [211, 1145], [212, 216], [212, 231], [212, 237], [212, 249], [212, 261], [212, 270], [212, 278], [212, 909], [212, 1086], [212, 1087], [212, 1099], [212, 1116], [212, 1124], [212, 1127], [212, 1146], [212, 1171], [212, 1177], [212, 1180], [212, 1184], [213, 237], [213, 269], [213, 270], [213, 271], [213, 272], [213, 273], [213, 910], [213, 1087], [213, 1176], [213, 1185], [214, 233], [214, 237], [214, 264], [214, 911], [214, 1087], [214, 1088], [214, 1182], [215, 237], [215, 259], [215, 261], [215, 266], [215, 271], [215, 373], [215, 912], [215, 991], [215, 1087], [215, 1089], [215, 1147], [215, 1183], [215, 1186], [215, 1211], [216, 234], [216, 237], [216, 249], [216, 272], [216, 913], [216, 1078], [216, 1079], [216, 1087], [216, 1090], [216, 1139], [216, 1144], [216, 1165], [216, 1177], [216, 1196], [216, 1211], [217, 238], [217, 239], [217, 274], [217, 648], [217, 877], [217, 917], [217, 1091], [217, 1092], [217, 1138], [217, 1169], [217, 1170], [218, 239], [218, 274], [218, 275], [218, 276], [218, 277], [218, 278], [218, 907], [218, 908], [218, 918], [218, 952], [218, 960], [218, 1092], [218, 1176], [218, 1178], [219, 239], [219, 278], [219, 923], [219, 932], [219, 985], [219, 997], [219, 1066], [219, 1086], [219, 1092], [219, 1093], [219, 1131], [219, 1151], [219, 1201], [220, 240], [220, 241], [220, 242], [220, 243], [220, 877], [220, 925], [220, 1094], [221, 240], [221, 279], [221, 410], [221, 927], [221, 1095], [221, 1129], [221, 1137], [221, 1154], [221, 1202], [221, 1248], [222, 244], [222, 245], [222, 246], [222, 247], [222, 248], [222, 281], [222, 877], [222, 933], [222, 1096], [222, 1097], [222, 1103], [222, 1108], [222, 1171], [222, 1172], [222, 1173], [222, 1174], [222, 1175], [223, 236], [223, 281], [223, 282], [223, 283], [223, 284], [223, 285], [223, 286], [223, 925], [223, 934], [223, 968], [223, 1097], [223, 1168], [223, 1176], [223, 1180], [224, 275], [224, 283], [224, 414], [224, 907], [224, 909], [224, 935], [224, 953], [224, 975], [224, 1053], [224, 1097], [224, 1098], [224, 1204], [224, 1254], [225, 226], [225, 244], [225, 270], [225, 284], [225, 936], [225, 1054], [225, 1097], [225, 1099], [225, 1142], [225, 1160], [225, 1182], [225, 1210], [226, 246], [226, 286], [226, 414], [226, 940], [226, 983], [226, 1056], [226, 1097], [226, 1100], [226, 1158], [226, 1174], [226, 1201], [226, 1254], [227, 249], [227, 250], [227, 251], [227, 287], [227, 288], [227, 289], [227, 667], [227, 877], [227, 1101], [227, 1107], [227, 1139], [227, 1176], [227, 1177], [228, 230], [228, 247], [228, 288], [228, 296], [228, 917], [228, 943], [228, 960], [228, 1101], [228, 1102], [228, 1108], [228, 1123], [228, 1184], [229, 262], [229, 281], [229, 289], [229, 925], [229, 926], [229, 945], [229, 969], [229, 975], [229, 1101], [229, 1103], [229, 1183], [230, 249], [230, 947], [230, 965], [230, 982], [230, 997], [230, 1082], [230, 1086], [230, 1090], [230, 1101], [230, 1104], [230, 1162], [230, 1177], [231, 250], [231, 928], [231, 950], [231, 969], [231, 985], [231, 1077], [231, 1101], [231, 1105], [231, 1144], [231, 1184], [232, 252], [232, 290], [232, 295], [232, 299], [232, 877], [232, 951], [232, 1106], [232, 1135], [232, 1140], [232, 1143], [232, 1178], [232, 1179], [232, 1180], [232, 1181], [233, 235], [233, 251], [233, 287], [233, 295], [233, 952], [233, 1107], [233, 1143], [233, 1166], [233, 1182], [234, 236], [234, 247], [234, 281], [234, 288], [234, 291], [234, 299], [234, 310], [234, 953], [234, 1108], [234, 1139], [234, 1143], [234, 1146], [234, 1168], [234, 1185], [234, 1193], [234, 1202], [234, 1206], [235, 292], [235, 955], [235, 983], [235, 1109], [235, 1143], [235, 1178], [235, 1203], [236, 294], [236, 314], [236, 957], [236, 973], [236, 985], [236, 1001], [236, 1110], [236, 1143], [236, 1156], [236, 1165], [236, 1180], [236, 1185], [237, 877], [237, 910], [237, 959], [237, 986], [237, 1111], [237, 1143], [237, 1144], [237, 1145], [237, 1146], [237, 1147], [238, 303], [238, 877], [238, 960], [238, 1112], [238, 1141], [238, 1148], [238, 1182], [238, 1208], [239, 877], [239, 900], [239, 918], [239, 966], [239, 987], [239, 1024], [239, 1025], [239, 1113], [239, 1148], [239, 1149], [239, 1150], [239, 1151], [239, 1152], [240, 242], [240, 262], [240, 967], [240, 1114], [240, 1153], [241, 253], [241, 254], [241, 255], [241, 256], [241, 298], [241, 479], [241, 877], [241, 968], [241, 1115], [241, 1183], [242, 253], [242, 262], [242, 270], [242, 971], [242, 982], [242, 1054], [242, 1116], [242, 1154], [242, 1211], [243, 254], [243, 338], [243, 983], [243, 1117], [244, 246], [244, 264], [244, 974], [244, 1118], [244, 1155], [244, 1158], [244, 1174], [244, 1182], [244, 1206], [244, 1209], [245, 247], [245, 299], [245, 909], [245, 946], [245, 977], [245, 982], [245, 1058], [245, 1106], [245, 1108], [245, 1119], [245, 1155], [245, 1200], [245, 1209], [245, 1215], [246, 248], [246, 257], [246, 276], [246, 300], [246, 363], [246, 892], [246, 893], [246, 978], [246, 983], [246, 1120], [246, 1155], [246, 1160], [246, 1175], [246, 1200], [246, 1206], [246, 1216], [247, 889], [247, 893], [247, 927], [247, 946], [247, 953], [247, 967], [247, 980], [247, 986], [247, 1077], [247, 1084], [247, 1090], [247, 1110], [247, 1121], [247, 1155], [247, 1162], [247, 1165], [247, 1173], [247, 1184], [248, 258], [248, 301], [248, 889], [248, 892], [248, 981], [248, 987], [248, 1058], [248, 1122], [248, 1155], [248, 1174], [248, 1217], [249, 288], [249, 303], [249, 944], [249, 982], [249, 1112], [249, 1123], [249, 1144], [249, 1146], [249, 1161], [250, 270], [250, 310], [250, 985], [250, 1124], [250, 1139], [250, 1161], [251, 910], [251, 944], [251, 952], [251, 987], [251, 1084], [251, 1088], [251, 1109], [251, 1125], [251, 1161], [251, 1169], [252, 307], [252, 936], [252, 954], [252, 961], [252, 974], [252, 987], [252, 993], [252, 1126], [252, 1164], [252, 1178], [252, 1222], [253, 255], [253, 270], [253, 273], [253, 341], [253, 982], [253, 998], [253, 1127], [254, 256], [254, 342], [254, 983], [254, 1128], [255, 259], [255, 279], [255, 986], [255, 1001], [255, 1129], [255, 1186], [255, 1206], [256, 343], [256, 987], [256, 1130], [257, 258], [257, 276], [257, 278], [257, 352], [257, 387], [257, 983], [257, 991], [257, 995], [257, 1002], [257, 1131], [257, 1171], [257, 1174], [257, 1203], [258, 260], [258, 963], [258, 987], [258, 991], [258, 1006], [258, 1098], [258, 1099], [258, 1132], [258, 1149], [258, 1171], [258, 1175], [258, 1187], [258, 1288], [259, 263], [259, 312], [259, 375], [259, 425], [259, 967], [259, 986], [259, 1012], [259, 1133], [259, 1145], [259, 1183], [259, 1188], [259, 1206], [259, 1264], [260, 313], [260, 974], [260, 980], [260, 987], [260, 1013], [260, 1079], [260, 1134], [260, 1151], [260, 1193], [260, 1244], [260, 1277], [261, 263], [261, 290], [261, 305], [261, 315], [261, 969], [261, 1127], [261, 1135], [261, 1144], [261, 1145], [261, 1189], [261, 1211], [261, 1215], [262, 297], [262, 969], [262, 1136], [262, 1163], [263, 279], [263, 317], [263, 359], [263, 361], [263, 967], [263, 969], [263, 1137], [263, 1184], [263, 1186], [263, 1191], [264, 267], [264, 274], [264, 319], [264, 1138], [264, 1193], [265, 272], [265, 287], [265, 291], [265, 1139], [266, 290], [266, 298], [266, 306], [266, 320], [266, 373], [266, 1140], [266, 1145], [266, 1194], [266, 1206], [267, 268], [267, 374], [267, 1141], [268, 284], [268, 308], [268, 376], [268, 1142], [269, 290], [269, 291], [269, 292], [269, 293], [269, 294], [269, 1143], [269, 1176], [270, 272], [270, 297], [270, 303], [270, 306], [270, 310], [270, 1144], [270, 1158], [270, 1180], [271, 273], [271, 309], [271, 312], [271, 337], [271, 1145], [271, 1220], [272, 291], [272, 303], [272, 322], [272, 337], [272, 1141], [272, 1142], [272, 1146], [272, 1196], [273, 293], [273, 324], [273, 421], [273, 913], [273, 1147], [273, 1198], [273, 1220], [273, 1263], [274, 295], [274, 296], [274, 1148], [274, 1169], [274, 1176], [275, 277], [275, 297], [275, 1149], [275, 1157], [276, 325], [276, 951], [276, 960], [276, 1150], [276, 1174], [276, 1199], [277, 278], [277, 313], [277, 326], [277, 953], [277, 1118], [277, 1141], [277, 1151], [277, 1200], [278, 327], [278, 957], [278, 995], [278, 1099], [278, 1144], [278, 1152], [278, 1201], [278, 1203], [279, 280], [279, 328], [279, 374], [279, 1153], [279, 1184], [279, 1202], [280, 1154], [281, 297], [281, 298], [281, 299], [281, 300], [281, 301], [281, 1155], [281, 1156], [281, 1163], [281, 1165], [281, 1172], [281, 1176], [282, 294], [282, 302], [282, 306], [282, 1156], [283, 330], [283, 380], [283, 967], [283, 1156], [283, 1157], [283, 1180], [283, 1204], [283, 1206], [283, 1213], [284, 286], [284, 308], [284, 336], [284, 1156], [284, 1158], [285, 309], [285, 314], [285, 315], [285, 724], [285, 968], [285, 1156], [285, 1159], [285, 1214], [286, 300], [286, 327], [286, 380], [286, 1156], [286, 1160], [286, 1218], [287, 302], [287, 303], [287, 1161], [287, 1176], [288, 310], [288, 1161], [288, 1162], [288, 1165], [288, 1169], [288, 1177], [288, 1208], [289, 309], [289, 1161], [289, 1163], [289, 1183], [289, 1213], [290, 304], [290, 305], [290, 306], [290, 307], [290, 1164], [290, 1176], [291, 294], [291, 311], [291, 319], [291, 328], [291, 332], [291, 1165], [291, 1206], [292, 304], [292, 329], [292, 1166], [292, 1218], [293, 305], [293, 328], [293, 330], [293, 333], [293, 420], [293, 953], [293, 1167], [293, 1207], [293, 1262], [294, 306], [294, 311], [294, 1168], [294, 1212], [295, 319], [295, 334], [295, 335], [295, 336], [295, 1164], [295, 1169], [295, 1178], [295, 1182], [296, 735], [296, 960], [296, 1162], [296, 1170], [297, 310], [297, 339], [297, 1171], [297, 1176], [297, 1213], [298, 339], [298, 340], [298, 341], [298, 342], [298, 343], [298, 1172], [298, 1183], [298, 1206], [298, 1214], [299, 326], [299, 335], [299, 341], [299, 974], [299, 1164], [299, 1165], [299, 1173], [299, 1180], [299, 1215], [300, 301], [300, 326], [300, 332], [300, 342], [300, 1174], [300, 1194], [300, 1216], [300, 1218], [301, 343], [301, 952], [301, 974], [301, 1175], [301, 1193], [301, 1194], [301, 1217], [302, 311], [302, 344], [302, 1176], [303, 1177], [303, 1182], [304, 307], [304, 349], [304, 1178], [304, 1218], [305, 373], [305, 1179], [305, 1215], [306, 322], [306, 339], [306, 346], [306, 352], [306, 1180], [306, 1203], [306, 1211], [306, 1220], [307, 348], [307, 952], [307, 1181], [307, 1222], [308, 354], [308, 1176], [308, 1182], [308, 1218], [309, 312], [309, 316], [309, 351], [309, 1176], [309, 1183], [310, 352], [310, 1165], [310, 1184], [310, 1185], [310, 1210], [310, 1226], [311, 354], [311, 1185], [311, 1228], [312, 314], [312, 332], [312, 390], [312, 1185], [312, 1186], [313, 319], [313, 370], [313, 403], [313, 952], [313, 953], [313, 1142], [313, 1187], [313, 1239], [314, 315], [314, 333], [314, 358], [314, 390], [314, 1001], [314, 1168], [314, 1188], [314, 1190], [314, 1195], [315, 317], [315, 360], [315, 389], [315, 391], [315, 998], [315, 1180], [315, 1189], [315, 1190], [316, 398], [316, 1190], [317, 328], [317, 345], [317, 420], [317, 1190], [317, 1191], [317, 1226], [317, 1262], [318, 324], [318, 333], [318, 362], [318, 368], [318, 377], [318, 1190], [318, 1192], [318, 1272], [319, 336], [319, 363], [319, 1140], [319, 1142], [319, 1193], [320, 321], [320, 323], [320, 340], [320, 346], [320, 364], [320, 1194], [321, 330], [321, 332], [321, 340], [321, 345], [321, 366], [321, 370], [321, 384], [321, 398], [321, 1195], [321, 1206], [321, 1216], [322, 332], [322, 350], [322, 370], [322, 1196], [323, 344], [323, 346], [323, 353], [323, 1197], [324, 333], [324, 373], [324, 394], [324, 1198], [324, 1244], [325, 345], [325, 356], [325, 372], [325, 1178], [325, 1199], [325, 1216], [325, 1221], [325, 1223], [326, 327], [326, 335], [326, 350], [326, 397], [326, 974], [326, 1178], [326, 1195], [326, 1200], [327, 336], [327, 380], [327, 397], [327, 418], [327, 936], [327, 1178], [327, 1180], [327, 1201], [327, 1258], [328, 345], [328, 375], [328, 1202], [328, 1226], [329, 334], [329, 380], [329, 1203], [330, 345], [330, 351], [330, 381], [330, 420], [330, 1204], [330, 1220], [330, 1262], [331, 340], [331, 353], [331, 384], [331, 394], [331, 1205], [332, 350], [332, 354], [332, 1206], [333, 1195], [333, 1207], [334, 349], [334, 1208], [335, 336], [335, 350], [335, 1209], [336, 387], [336, 1180], [336, 1210], [337, 338], [337, 355], [337, 389], [337, 1211], [338, 390], [338, 1212], [338, 1228], [339, 350], [339, 351], [339, 352], [339, 1213], [340, 353], [340, 1214], [341, 355], [341, 391], [341, 422], [341, 1206], [341, 1215], [341, 1220], [341, 1224], [342, 343], [342, 357], [342, 364], [342, 365], [342, 1216], [343, 363], [343, 364], [343, 422], [343, 1217], [344, 357], [344, 1218], [345, 355], [345, 356], [345, 393], [345, 1219], [346, 380], [346, 382], [346, 389], [346, 394], [346, 1220], [347, 360], [347, 364], [347, 373], [347, 393], [347, 394], [347, 395], [347, 1221], [347, 1227], [348, 356], [348, 382], [348, 395], [348, 1222], [349, 361], [349, 365], [349, 374], [349, 380], [349, 396], [349, 397], [349, 1208], [349, 1218], [349, 1223], [349, 1227], [350, 398], [350, 1224], [351, 398], [351, 399], [351, 1213], [351, 1214], [351, 1220], [351, 1225], [351, 1227], [352, 387], [352, 1206], [352, 1226], [352, 1228], [353, 1227], [354, 1218], [354, 1228], [355, 367], [355, 370], [355, 1220], [355, 1223], [355, 1228], [355, 1229], [356, 367], [356, 400], [356, 1221], [356, 1230], [357, 1218], [357, 1231], [358, 360], [358, 379], [358, 404], [358, 420], [358, 957], [358, 1188], [358, 1232], [358, 1240], [358, 1262], [359, 361], [359, 379], [359, 386], [359, 885], [359, 963], [359, 1233], [360, 373], [360, 379], [360, 991], [360, 1189], [360, 1234], [361, 374], [361, 379], [361, 401], [361, 995], [361, 1235], [362, 379], [362, 386], [362, 1192], [362, 1227], [362, 1236], [363, 365], [363, 387], [363, 407], [363, 422], [363, 892], [363, 896], [363, 1174], [363, 1193], [363, 1206], [363, 1237], [364, 366], [364, 394], [364, 402], [364, 1194], [364, 1238], [365, 380], [365, 403], [365, 1206], [365, 1239], [366, 381], [366, 385], [366, 393], [366, 404], [366, 422], [366, 423], [366, 1195], [366, 1240], [367, 405], [367, 407], [367, 412], [367, 415], [367, 422], [367, 427], [367, 564], [367, 878], [367, 994], [367, 1223], [367, 1241], [367, 1250], [367, 1256], [368, 391], [368, 398], [368, 406], [368, 878], [368, 1003], [368, 1190], [368, 1195], [368, 1224], [368, 1225], [368, 1242], [368, 1244], [369, 385], [369, 392], [369, 396], [369, 399], [369, 810], [369, 878], [369, 1227], [369, 1230], [369, 1243], [369, 1251], [370, 398], [370, 403], [370, 405], [370, 1196], [370, 1197], [370, 1220], [370, 1239], [370, 1244], [371, 390], [371, 397], [371, 408], [371, 418], [371, 994], [371, 1007], [371, 1011], [371, 1195], [371, 1197], [371, 1227], [371, 1228], [371, 1245], [371, 1258], [372, 393], [372, 400], [372, 423], [372, 424], [372, 989], [372, 1199], [372, 1246], [372, 1270], [373, 375], [373, 409], [373, 991], [373, 1145], [373, 1247], [374, 376], [374, 410], [374, 995], [374, 1248], [375, 378], [375, 388], [375, 393], [375, 411], [375, 425], [375, 1186], [375, 1202], [375, 1206], [375, 1207], [375, 1249], [376, 412], [376, 1250], [377, 406], [377, 1251], [378, 389], [378, 998], [378, 1196], [378, 1198], [378, 1252], [379, 413], [379, 886], [379, 897], [379, 898], [379, 999], [379, 1190], [379, 1227], [379, 1253], [380, 382], [380, 414], [380, 1203], [380, 1205], [380, 1254], [381, 383], [381, 385], [381, 393], [381, 399], [381, 1204], [381, 1205], [381, 1207], [381, 1255], [381, 1268], [382, 415], [382, 1205], [382, 1256], [383, 391], [383, 394], [383, 399], [383, 402], [383, 406], [383, 416], [383, 418], [383, 991], [383, 1198], [383, 1205], [383, 1220], [383, 1221], [383, 1238], [383, 1257], [383, 1258], [384, 394], [384, 418], [384, 1258], [385, 954], [385, 1222], [385, 1259], [386, 990], [386, 1260], [387, 407], [387, 991], [387, 1210], [387, 1261], [388, 389], [388, 393], [388, 420], [388, 1206], [388, 1262], [389, 390], [389, 421], [389, 1196], [389, 1211], [389, 1263], [390, 1212], [390, 1264], [391, 998], [391, 1195], [391, 1214], [391, 1215], [391, 1221], [391, 1265], [391, 1268], [391, 1272], [392, 399], [392, 402], [392, 404], [392, 976], [392, 1000], [392, 1214], [392, 1238], [392, 1240], [392, 1266], [393, 400], [393, 1219], [393, 1227], [393, 1267], [394, 1220], [394, 1268], [395, 400], [395, 1222], [395, 1269], [396, 1219], [396, 1223], [396, 1230], [396, 1231], [396, 1270], [397, 841], [397, 994], [397, 1218], [397, 1271], [398, 1224], [398, 1227], [398, 1272], [399, 1192], [399, 1225], [399, 1268], [399, 1273], [400, 1230], [400, 1274], [401, 410], [401, 413], [401, 1066], [401, 1137], [401, 1223], [401, 1233], [401, 1248], [401, 1253], [401, 1275], [402, 404], [402, 417], [402, 892], [402, 1194], [402, 1216], [402, 1217], [402, 1221], [402, 1240], [402, 1268], [402, 1276], [403, 405], [403, 414], [403, 417], [403, 893], [403, 953], [403, 955], [403, 1216], [403, 1223], [403, 1237], [403, 1254], [403, 1277], [404, 417], [404, 895], [404, 1195], [404, 1238], [404, 1255], [404, 1259], [404, 1267], [404, 1278], [404, 1296], [404, 1297], [405, 415], [405, 417], [405, 896], [405, 1239], [405, 1256], [405, 1279], [406, 417], [406, 418], [406, 878], [406, 898], [406, 1258], [406, 1261], [406, 1268], [406, 1272], [406, 1280], [407, 911], [407, 914], [407, 983], [407, 1228], [407, 1281], [408, 425], [408, 426], [408, 878], [408, 895], [408, 904], [408, 914], [408, 915], [408, 1007], [408, 1072], [408, 1264], [408, 1271], [408, 1282], [408, 1292], [409, 411], [409, 924], [409, 929], [409, 939], [409, 998], [409, 1089], [409, 1140], [409, 1179], [409, 1198], [409, 1221], [409, 1234], [409, 1249], [409, 1283], [410, 412], [410, 929], [410, 1066], [410, 1141], [410, 1153], [410, 1223], [410, 1235], [410, 1250], [410, 1284], [411, 420], [411, 927], [411, 929], [411, 953], [411, 956], [411, 1133], [411, 1202], [411, 1247], [411, 1252], [411, 1262], [411, 1267], [411, 1285], [411, 1299], [412, 929], [412, 1142], [412, 1248], [412, 1286], [413, 878], [413, 886], [413, 905], [413, 929], [413, 931], [413, 1020], [413, 1232], [413, 1233], [413, 1234], [413, 1235], [413, 1236], [413, 1287], [414, 415], [414, 932], [414, 938], [414, 955], [414, 1002], [414, 1157], [414, 1160], [414, 1201], [414, 1203], [414, 1220], [414, 1223], [414, 1239], [414, 1256], [414, 1288], [415, 422], [415, 936], [415, 938], [415, 1220], [415, 1222], [415, 1254], [415, 1289], [416, 426], [416, 912], [416, 916], [416, 938], [416, 939], [416, 991], [416, 992], [416, 1002], [416, 1006], [416, 1255], [416, 1265], [416, 1268], [416, 1273], [416, 1276], [416, 1280], [416, 1290], [416, 1292], [417, 878], [417, 894], [417, 938], [417, 942], [417, 1237], [417, 1238], [417, 1239], [417, 1240], [417, 1291], [418, 936], [418, 940], [418, 954], [418, 957], [418, 1195], [418, 1205], [418, 1268], [418, 1292], [419, 426], [419, 903], [419, 904], [419, 961], [419, 962], [419, 1006], [419, 1013], [419, 1293], [420, 421], [420, 953], [420, 967], [420, 970], [420, 1249], [420, 1263], [420, 1267], [420, 1294], [421, 426], [421, 912], [421, 913], [421, 970], [421, 971], [421, 1189], [421, 1211], [421, 1220], [421, 1252], [421, 1262], [421, 1264], [421, 1295], [422, 423], [422, 974], [422, 976], [422, 1222], [422, 1256], [422, 1296], [423, 424], [423, 427], [423, 976], [423, 978], [423, 1216], [423, 1297], [424, 427], [424, 1002], [424, 1225], [424, 1298], [425, 895], [425, 898], [425, 988], [425, 991], [425, 1005], [425, 1070], [425, 1186], [425, 1225], [425, 1226], [425, 1245], [425, 1299], [426, 878], [426, 1009], [426, 1229], [426, 1245], [426, 1252], [426, 1257], [426, 1263], [426, 1265], [426, 1267], [426, 1300], [427, 878], [427, 1011], [427, 1231], [427, 1301], [428, 429], [428, 430], [428, 431], [428, 432], [428, 433], [428, 435], [428, 442], [428, 444], [428, 451], [428, 459], [428, 467], [428, 469], [428, 476], [428, 479], [428, 485], [428, 487], [428, 496], [428, 504], [428, 513], [428, 514], [428, 520], [428, 522], [428, 529], [428, 541], [428, 547], [428, 552], [428, 556], [428, 562], [428, 564], [428, 875], [428, 1302], [429, 431], [429, 433], [429, 437], [429, 445], [429, 536], [429, 537], [429, 538], [429, 539], [429, 540], [429, 652], [429, 657], [429, 661], [429, 663], [429, 667], [429, 707], [429, 712], [429, 720], [429, 726], [429, 729], [429, 736], [429, 740], [429, 741], [429, 745], [429, 746], [429, 1303], [430, 431], [430, 432], [430, 466], [430, 474], [430, 483], [430, 492], [430, 508], [430, 515], [430, 524], [430, 530], [430, 787], [430, 790], [430, 793], [430, 801], [430, 806], [430, 810], [430, 814], [430, 822], [430, 826], [430, 836], [430, 838], [430, 842], [430, 1304], [431, 440], [431, 462], [431, 471], [431, 488], [431, 560], [431, 561], [431, 734], [431, 737], [431, 740], [431, 756], [431, 769], [431, 772], [431, 779], [431, 783], [431, 788], [431, 789], [431, 793], [431, 794], [431, 795], [431, 1305], [432, 512], [432, 518], [432, 526], [432, 534], [432, 545], [432, 549], [432, 553], [432, 557], [432, 580], [432, 587], [432, 590], [432, 595], [432, 600], [432, 807], [432, 808], [432, 809], [432, 810], [432, 849], [432, 852], [432, 859], [432, 864], [432, 874], [432, 875], [432, 1306], [433, 497], [433, 498], [433, 499], [433, 500], [433, 501], [433, 502], [433, 503], [433, 569], [433, 576], [433, 627], [433, 642], [433, 647], [433, 649], [433, 656], [433, 660], [433, 662], [433, 667], [433, 675], [433, 676], [433, 677], [433, 679], [433, 682], [433, 875], [433, 1307], [434, 438], [434, 459], [434, 485], [434, 510], [434, 572], [434, 617], [434, 666], [434, 696], [434, 754], [434, 797], [434, 1308], [435, 478], [435, 479], [435, 480], [435, 481], [435, 482], [435, 483], [435, 484], [435, 485], [435, 567], [435, 595], [435, 607], [435, 643], [435, 661], [435, 1309], [436, 439], [436, 485], [436, 487], [436, 499], [436, 533], [436, 576], [436, 621], [436, 668], [436, 1310], [437, 440], [437, 458], [437, 485], [437, 523], [437, 537], [437, 593], [437, 661], [437, 668], [437, 669], [437, 699], [437, 700], [437, 701], [437, 1311], [438, 441], [438, 478], [438, 485], [438, 504], [438, 543], [438, 584], [438, 585], [438, 640], [438, 666], [438, 699], [438, 755], [438, 799], [438, 1312], [439, 480], [439, 485], [439, 529], [439, 605], [439, 617], [439, 619], [439, 700], [439, 1313], [440, 485], [440, 499], [440, 503], [440, 523], [440, 526], [440, 754], [440, 755], [440, 756], [440, 809], [440, 822], [440, 870], [440, 1314], [441, 481], [441, 485], [441, 541], [441, 636], [441, 641], [441, 701], [441, 1315], [442, 468], [442, 485], [442, 495], [442, 510], [442, 533], [442, 537], [442, 543], [442, 552], [442, 756], [442, 801], [442, 843], [442, 1316], [443, 447], [443, 469], [443, 496], [443, 517], [443, 623], [443, 671], [443, 698], [443, 702], [443, 757], [443, 802], [443, 1317], [444, 486], [444, 487], [444, 488], [444, 489], [444, 490], [444, 491], [444, 492], [444, 493], [444, 494], [444, 495], [444, 496], [444, 568], [444, 600], [444, 611], [444, 621], [444, 642], [444, 663], [444, 1318], [445, 465], [445, 483], [445, 496], [445, 507], [445, 524], [445, 538], [445, 569], [445, 643], [445, 663], [445, 702], [445, 703], [445, 704], [445, 705], [445, 706], [445, 1319], [446, 448], [446, 466], [446, 496], [446, 504], [446, 530], [446, 544], [446, 570], [446, 583], [446, 585], [446, 644], [446, 704], [446, 739], [446, 803], [446, 1320], [447, 449], [447, 486], [447, 496], [447, 514], [447, 548], [447, 645], [447, 671], [447, 672], [447, 705], [447, 739], [447, 763], [447, 804], [447, 846], [447, 1321], [448, 489], [448, 496], [448, 507], [448, 512], [448, 530], [448, 535], [448, 541], [448, 572], [448, 623], [448, 625], [448, 634], [448, 754], [448, 758], [448, 763], [448, 771], [448, 805], [448, 809], [448, 814], [448, 834], [448, 847], [448, 866], [448, 1322], [449, 490], [449, 496], [449, 547], [449, 573], [449, 646], [449, 706], [449, 848], [449, 1323], [450, 491], [450, 496], [450, 524], [450, 526], [450, 552], [450, 574], [450, 768], [450, 810], [450, 1324], [451, 477], [451, 496], [451, 503], [451, 511], [451, 517], [451, 538], [451, 544], [451, 548], [451, 556], [451, 575], [451, 806], [451, 849], [451, 1325], [452, 456], [452, 474], [452, 493], [452, 504], [452, 515], [452, 531], [452, 545], [452, 570], [452, 577], [452, 581], [452, 592], [452, 628], [452, 629], [452, 714], [452, 875], [452, 1326], [453, 457], [453, 475], [453, 514], [453, 536], [453, 549], [453, 582], [453, 589], [453, 596], [453, 630], [453, 631], [453, 648], [453, 657], [453, 676], [453, 714], [453, 735], [453, 875], [453, 1327], [454, 483], [454, 495], [454, 522], [454, 530], [454, 537], [454, 553], [454, 571], [454, 578], [454, 583], [454, 593], [454, 632], [454, 633], [454, 634], [454, 663], [454, 679], [454, 724], [454, 875], [454, 1328], [455, 458], [455, 529], [455, 557], [455, 584], [455, 597], [455, 635], [455, 636], [455, 664], [455, 668], [455, 682], [455, 875], [455, 1329], [456, 497], [456, 541], [456, 572], [456, 598], [456, 604], [456, 617], [456, 637], [456, 873], [456, 875], [456, 1330], [457, 498], [457, 515], [457, 519], [457, 536], [457, 540], [457, 547], [457, 573], [457, 579], [457, 594], [457, 599], [457, 618], [457, 623], [457, 638], [457, 807], [457, 814], [457, 841], [457, 875], [457, 1331], [458, 500], [458, 531], [458, 535], [458, 537], [458, 556], [458, 575], [458, 586], [458, 605], [458, 639], [458, 809], [458, 875], [458, 1332], [459, 504], [459, 505], [459, 506], [459, 507], [459, 508], [459, 509], [459, 510], [459, 511], [459, 512], [459, 513], [459, 581], [459, 649], [459, 707], [459, 722], [459, 1333], [460, 463], [460, 502], [460, 513], [460, 514], [460, 550], [460, 582], [460, 667], [460, 728], [460, 750], [460, 1334], [461, 465], [461, 493], [461, 503], [461, 513], [461, 529], [461, 536], [461, 544], [461, 558], [461, 584], [461, 596], [461, 599], [461, 619], [461, 632], [461, 644], [461, 651], [461, 663], [461, 674], [461, 708], [461, 722], [461, 738], [461, 743], [461, 755], [461, 765], [461, 776], [461, 1335], [462, 466], [462, 513], [462, 560], [462, 652], [462, 670], [462, 672], [462, 704], [462, 743], [462, 759], [462, 760], [462, 1336], [463, 505], [463, 513], [463, 547], [463, 653], [463, 750], [463, 757], [463, 776], [463, 1337], [464, 468], [464, 506], [464, 513], [464, 552], [464, 565], [464, 585], [464, 654], [464, 709], [464, 755], [464, 759], [464, 811], [464, 825], [464, 850], [464, 870], [464, 1338], [465, 507], [465, 513], [465, 536], [465, 556], [465, 573], [465, 586], [465, 655], [465, 710], [465, 812], [465, 821], [465, 1339], [466, 508], [466, 513], [466, 540], [466, 544], [466, 587], [466, 609], [466, 811], [466, 812], [466, 813], [466, 814], [466, 851], [466, 1340], [467, 513], [467, 519], [467, 527], [467, 535], [467, 539], [467, 550], [467, 554], [467, 558], [467, 560], [467, 562], [467, 565], [467, 588], [467, 606], [467, 611], [467, 814], [467, 852], [467, 1341], [468, 510], [468, 513], [468, 571], [468, 632], [468, 711], [468, 760], [468, 821], [468, 825], [468, 855], [468, 1342], [469, 514], [469, 515], [469, 516], [469, 517], [469, 518], [469, 519], [469, 520], [469, 589], [469, 656], [469, 712], [469, 727], [469, 734], [469, 1343], [470, 473], [470, 520], [470, 529], [470, 559], [470, 596], [470, 598], [470, 713], [470, 815], [470, 853], [470, 1344], [471, 474], [471, 502], [471, 512], [471, 520], [471, 542], [471, 549], [471, 561], [471, 657], [471, 672], [471, 690], [471, 734], [471, 761], [471, 762], [471, 763], [471, 764], [471, 765], [471, 1345], [472, 475], [472, 520], [472, 541], [472, 563], [472, 625], [472, 629], [472, 635], [472, 714], [472, 761], [472, 816], [472, 1346], [473, 477], [473, 520], [473, 556], [473, 566], [473, 572], [473, 683], [473, 684], [473, 716], [473, 763], [473, 854], [473, 1347], [474, 515], [474, 520], [474, 542], [474, 590], [474, 609], [474, 638], [474, 815], [474, 816], [474, 1348], [475, 516], [475, 520], [475, 549], [475, 562], [475, 580], [475, 604], [475, 606], [475, 615], [475, 1349], [476, 520], [476, 540], [476, 546], [476, 559], [476, 561], [476, 563], [476, 564], [476, 566], [476, 591], [476, 612], [476, 627], [476, 851], [476, 1350], [477, 517], [477, 520], [477, 584], [477, 635], [477, 659], [477, 717], [477, 765], [477, 860], [477, 1351], [478, 481], [478, 504], [478, 583], [478, 584], [478, 592], [478, 691], [478, 711], [478, 817], [478, 855], [478, 1352], [479, 521], [479, 522], [479, 523], [479, 524], [479, 525], [479, 526], [479, 527], [479, 593], [479, 643], [479, 660], [479, 721], [479, 1353], [480, 482], [480, 529], [480, 537], [480, 605], [480, 667], [480, 1354], [481, 484], [481, 521], [481, 541], [481, 634], [481, 636], [481, 671], [481, 673], [481, 718], [481, 819], [481, 857], [481, 1355], [482, 523], [482, 556], [482, 689], [482, 691], [482, 693], [482, 1356], [483, 524], [483, 537], [483, 571], [483, 575], [483, 595], [483, 817], [483, 818], [483, 819], [483, 820], [483, 821], [483, 822], [483, 1357], [484, 525], [484, 562], [484, 655], [484, 719], [484, 821], [484, 1358], [485, 607], [485, 621], [485, 822], [485, 859], [485, 1359], [486, 490], [486, 514], [486, 596], [486, 672], [486, 694], [486, 717], [486, 743], [486, 766], [486, 823], [486, 860], [486, 1360], [487, 528], [487, 529], [487, 530], [487, 531], [487, 532], [487, 533], [487, 534], [487, 535], [487, 597], [487, 644], [487, 662], [487, 720], [487, 737], [487, 1361], [488, 492], [488, 511], [488, 526], [488, 544], [488, 553], [488, 663], [488, 721], [488, 737], [488, 766], [488, 767], [488, 768], [488, 1362], [489, 493], [489, 512], [489, 541], [489, 557], [489, 598], [489, 664], [489, 673], [489, 722], [489, 767], [489, 792], [489, 824], [489, 861], [489, 1363], [490, 494], [490, 528], [490, 547], [490, 599], [490, 665], [490, 723], [490, 743], [490, 744], [490, 792], [490, 827], [490, 862], [490, 1364], [491, 495], [491, 552], [491, 666], [491, 724], [491, 797], [491, 799], [491, 1365], [492, 530], [492, 571], [492, 574], [492, 600], [492, 768], [492, 823], [492, 824], [492, 825], [492, 826], [492, 866], [492, 1366], [493, 531], [493, 544], [493, 557], [493, 562], [493, 570], [493, 575], [493, 584], [493, 587], [493, 601], [493, 611], [493, 654], [493, 694], [493, 695], [493, 799], [493, 817], [493, 825], [493, 863], [493, 873], [493, 1367], [494, 532], [494, 564], [494, 602], [494, 725], [494, 827], [494, 1368], [495, 533], [495, 553], [495, 578], [495, 580], [495, 621], [495, 1369], [496, 569], [496, 603], [496, 826], [496, 864], [496, 1370], [497, 501], [497, 518], [497, 535], [497, 541], [497, 549], [497, 558], [497, 604], [497, 641], [497, 664], [497, 678], [497, 727], [497, 741], [497, 1371], [498, 502], [498, 519], [498, 547], [498, 560], [498, 646], [498, 653], [498, 665], [498, 670], [498, 676], [498, 681], [498, 728], [498, 734], [498, 745], [498, 1208], [498, 1372], [499, 526], [499, 552], [499, 557], [499, 654], [499, 658], [499, 666], [499, 668], [499, 679], [499, 696], [499, 697], [499, 737], [499, 746], [499, 782], [499, 1373], [500, 503], [500, 556], [500, 605], [500, 655], [500, 671], [500, 682], [500, 698], [500, 738], [500, 1374], [501, 536], [501, 562], [501, 579], [501, 584], [501, 586], [501, 637], [501, 683], [501, 688], [501, 691], [501, 1375], [502, 549], [502, 560], [502, 564], [502, 582], [502, 587], [502, 606], [502, 627], [502, 638], [502, 672], [502, 680], [502, 684], [502, 692], [502, 694], [502, 1376], [503, 538], [503, 558], [503, 569], [503, 575], [503, 607], [503, 659], [503, 669], [503, 674], [503, 689], [503, 1377], [504, 541], [504, 542], [504, 543], [504, 544], [504, 545], [504, 546], [504, 729], [504, 769], [504, 774], [504, 1378], [505, 509], [505, 540], [505, 547], [505, 561], [505, 670], [505, 740], [505, 774], [505, 1379], [506, 510], [506, 552], [506, 824], [506, 830], [506, 865], [506, 1380], [507, 511], [507, 535], [507, 556], [507, 560], [507, 671], [507, 681], [507, 684], [507, 693], [507, 696], [507, 704], [507, 722], [507, 730], [507, 737], [507, 741], [507, 758], [507, 770], [507, 791], [507, 802], [507, 804], [507, 819], [507, 830], [507, 1381], [508, 512], [508, 744], [508, 758], [508, 767], [508, 797], [508, 827], [508, 828], [508, 1382], [509, 542], [509, 564], [509, 609], [509, 672], [509, 731], [509, 804], [509, 823], [509, 1383], [510, 543], [510, 571], [510, 572], [510, 610], [510, 673], [510, 732], [510, 771], [510, 819], [510, 824], [510, 1384], [511, 544], [511, 560], [511, 611], [511, 665], [511, 674], [511, 733], [511, 827], [511, 866], [511, 1385], [512, 545], [512, 561], [512, 581], [512, 587], [512, 612], [512, 690], [512, 828], [512, 865], [512, 866], [512, 1386], [513, 587], [513, 613], [513, 644], [513, 652], [513, 675], [513, 1387], [514, 547], [514, 548], [514, 549], [514, 550], [514, 676], [514, 727], [514, 772], [514, 787], [514, 1182], [514, 1388], [515, 518], [515, 540], [515, 563], [515, 714], [515, 734], [515, 741], [515, 744], [515, 774], [515, 787], [515, 798], [515, 802], [515, 829], [515, 1389], [516, 519], [516, 562], [516, 604], [516, 695], [516, 698], [516, 735], [516, 868], [516, 1390], [517, 548], [517, 584], [517, 614], [517, 665], [517, 748], [517, 749], [517, 776], [517, 1391], [518, 549], [518, 563], [518, 577], [518, 579], [518, 589], [518, 590], [518, 615], [518, 627], [518, 690], [518, 867], [518, 868], [518, 1392], [519, 550], [519, 576], [519, 579], [519, 606], [519, 648], [519, 688], [519, 1393], [520, 590], [520, 606], [520, 612], [520, 616], [520, 657], [520, 677], [520, 1394], [521, 525], [521, 541], [521, 617], [521, 671], [521, 678], [521, 751], [521, 754], [521, 830], [521, 869], [521, 1395], [522, 551], [522, 552], [522, 553], [522, 554], [522, 679], [522, 721], [522, 780], [522, 1396], [523, 556], [523, 689], [523, 740], [523, 1397], [524, 526], [524, 724], [524, 746], [524, 830], [524, 831], [524, 832], [524, 1398], [525, 527], [525, 551], [525, 562], [525, 619], [525, 654], [525, 655], [525, 699], [525, 743], [525, 777], [525, 1399], [526, 553], [526, 569], [526, 574], [526, 593], [526, 595], [526, 621], [526, 666], [526, 869], [526, 870], [526, 1400], [527, 554], [527, 622], [527, 733], [527, 778], [527, 832], [527, 1401], [528, 532], [528, 547], [528, 623], [528, 681], [528, 744], [528, 753], [528, 757], [528, 758], [528, 833], [528, 871], [528, 1402], [529, 555], [529, 556], [529, 557], [529, 558], [529, 559], [529, 682], [529, 722], [529, 736], [529, 779], [529, 790], [529, 1403], [530, 534], [530, 724], [530, 737], [530, 780], [530, 790], [530, 833], [530, 834], [530, 835], [530, 1404], [531, 535], [531, 562], [531, 624], [531, 683], [531, 699], [531, 738], [531, 742], [531, 834], [531, 1405], [532, 555], [532, 564], [532, 625], [532, 684], [532, 739], [532, 758], [532, 761], [532, 781], [532, 835], [532, 846], [532, 1406], [533, 685], [533, 782], [533, 855], [533, 857], [533, 1407], [534, 557], [534, 570], [534, 572], [534, 578], [534, 597], [534, 600], [534, 666], [534, 871], [534, 1408], [535, 558], [535, 569], [535, 572], [535, 581], [535, 586], [535, 611], [535, 626], [535, 644], [535, 664], [535, 671], [535, 686], [535, 716], [535, 732], [535, 753], [535, 846], [535, 857], [535, 869], [535, 1409], [536, 539], [536, 562], [536, 688], [536, 719], [536, 735], [536, 738], [536, 741], [536, 794], [536, 841], [536, 1410], [537, 711], [537, 732], [537, 742], [537, 746], [537, 756], [537, 790], [537, 1411], [538, 689], [538, 717], [538, 733], [538, 743], [538, 791], [538, 1412], [539, 560], [539, 582], [539, 652], [539, 671], [539, 674], [539, 748], [539, 750], [539, 751], [539, 1413], [540, 561], [540, 581], [540, 609], [540, 627], [540, 657], [540, 670], [540, 744], [540, 749], [540, 752], [540, 753], [540, 1414], [541, 562], [541, 563], [541, 741], [541, 783], [541, 836], [541, 840], [541, 1415], [542, 546], [542, 564], [542, 774], [542, 793], [542, 828], [542, 840], [542, 1416], [543, 628], [543, 742], [543, 784], [543, 1417], [544, 709], [544, 711], [544, 743], [544, 747], [544, 749], [544, 767], [544, 785], [544, 790], [544, 794], [544, 812], [544, 825], [544, 837], [544, 860], [544, 862], [544, 1418], [545, 629], [545, 761], [545, 825], [545, 834], [545, 855], [545, 1419], [546, 563], [546, 690], [546, 744], [546, 786], [546, 828], [546, 862], [546, 871], [546, 1420], [547, 564], [547, 745], [547, 774], [547, 807], [547, 838], [547, 844], [547, 1421], [548, 630], [548, 747], [548, 748], [548, 839], [548, 1422], [549, 761], [549, 787], [549, 794], [549, 807], [549, 840], [549, 845], [549, 846], [549, 856], [549, 860], [549, 1423], [550, 631], [550, 688], [550, 716], [550, 717], [550, 841], [550, 1424], [551, 554], [551, 562], [551, 632], [551, 691], [551, 742], [551, 743], [551, 760], [551, 817], [551, 1425], [552, 565], [552, 746], [552, 780], [552, 788], [552, 808], [552, 843], [552, 1426], [553, 756], [553, 782], [553, 808], [553, 1427], [554, 565], [554, 634], [554, 693], [554, 732], [554, 733], [554, 758], [554, 1428], [555, 559], [555, 564], [555, 635], [555, 694], [555, 747], [555, 761], [555, 765], [555, 823], [555, 825], [555, 872], [555, 1429], [556, 566], [556, 789], [556, 809], [556, 842], [556, 1430], [557, 782], [557, 790], [557, 809], [557, 843], [557, 853], [557, 872], [557, 873], [557, 1431], [558, 636], [558, 748], [558, 791], [558, 873], [558, 1432], [559, 566], [559, 598], [559, 599], [559, 695], [559, 749], [559, 792], [559, 825], [559, 1433], [560, 750], [560, 778], [560, 791], [560, 794], [560, 796], [560, 814], [560, 844], [560, 1434], [561, 786], [561, 792], [561, 795], [561, 807], [561, 813], [561, 1435], [562, 637], [562, 794], [562, 1436], [563, 629], [563, 810], [563, 840], [563, 1437], [564, 638], [564, 795], [564, 840], [564, 844], [564, 851], [564, 874], [564, 1438], [565, 696], [565, 751], [565, 758], [565, 796], [565, 797], [565, 869], [565, 870], [565, 1439], [566, 572], [566, 573], [566, 639], [566, 698], [566, 753], [566, 802], [566, 871], [566, 1440], [567, 592], [567, 593], [567, 594], [567, 595], [567, 875], [567, 1441], [568, 596], [568, 597], [568, 598], [568, 599], [568, 600], [568, 601], [568, 602], [568, 603], [568, 642], [568, 875], [568, 1442], [569, 586], [569, 595], [569, 603], [569, 642], [569, 643], [569, 644], [569, 645], [569, 646], [569, 1443], [570, 572], [570, 587], [570, 603], [570, 644], [570, 684], [570, 687], [570, 802], [570, 1444], [571, 574], [571, 603], [571, 1445], [572, 598], [572, 603], [572, 612], [572, 626], [572, 847], [572, 852], [572, 873], [572, 1446], [573, 599], [573, 603], [573, 646], [573, 802], [573, 848], [573, 1447], [574, 603], [574, 621], [574, 822], [574, 1448], [575, 603], [575, 607], [575, 611], [575, 614], [575, 630], [575, 822], [575, 849], [575, 873], [575, 875], [575, 1449], [576, 604], [576, 605], [576, 606], [576, 607], [576, 608], [576, 647], [576, 648], [576, 875], [576, 1450], [577, 590], [577, 601], [577, 608], [577, 624], [577, 629], [577, 1451], [578, 595], [578, 608], [578, 642], [578, 666], [578, 685], [578, 1452], [579, 608], [579, 627], [579, 852], [579, 868], [579, 1453], [580, 608], [580, 612], [580, 615], [580, 621], [580, 629], [580, 875], [580, 1454], [581, 609], [581, 610], [581, 611], [581, 612], [581, 613], [581, 649], [581, 657], [581, 664], [581, 875], [581, 1455], [582, 606], [582, 613], [582, 631], [582, 647], [582, 657], [582, 1456], [583, 585], [583, 613], [583, 634], [583, 650], [583, 711], [583, 1457], [584, 586], [584, 601], [584, 607], [584, 613], [584, 636], [584, 642], [584, 651], [584, 664], [584, 683], [584, 1458], [585, 613], [585, 654], [585, 850], [585, 863], [585, 1459], [586, 613], [586, 655], [586, 711], [586, 1460], [587, 613], [587, 627], [587, 850], [587, 851], [587, 852], [587, 1461], [588, 613], [588, 622], [588, 626], [588, 631], [588, 634], [588, 636], [588, 637], [588, 852], [588, 875], [588, 1462], [589, 614], [589, 615], [589, 616], [589, 656], [589, 875], [589, 1463], [590, 616], [590, 853], [590, 854], [590, 1464], [591, 616], [591, 627], [591, 638], [591, 639], [591, 875], [591, 1465], [592, 685], [592, 855], [592, 1466], [593, 617], [593, 618], [593, 619], [593, 620], [593, 621], [593, 622], [593, 660], [593, 663], [593, 668], [593, 875], [593, 1467], [594, 618], [594, 719], [594, 858], [594, 1468], [595, 855], [595, 856], [595, 857], [595, 858], [595, 859], [595, 1469], [596, 599], [596, 659], [596, 860], [596, 1470], [597, 623], [597, 624], [597, 625], [597, 626], [597, 662], [597, 875], [597, 1471], [598, 601], [598, 612], [598, 664], [598, 861], [598, 1472], [599, 602], [599, 623], [599, 665], [599, 690], [599, 765], [599, 862], [599, 866], [599, 1473], [600, 860], [600, 861], [600, 862], [600, 863], [600, 864], [600, 1474], [601, 624], [601, 637], [601, 855], [601, 863], [601, 1475], [602, 625], [602, 638], [602, 866], [602, 1476], [603, 864], [603, 875], [603, 1477], [604, 615], [604, 626], [604, 636], [604, 1478], [605, 607], [605, 683], [605, 686], [605, 1479], [606, 638], [606, 1480], [607, 636], [607, 669], [607, 1481], [608, 875], [608, 1482], [609, 638], [609, 672], [609, 846], [609, 860], [609, 1483], [610, 628], [610, 673], [610, 857], [610, 861], [610, 1484], [611, 674], [611, 717], [611, 797], [611, 866], [611, 1485], [612, 629], [612, 1486], [613, 675], [613, 875], [613, 1487], [614, 630], [614, 695], [614, 798], [614, 1488], [615, 1489], [616, 677], [616, 875], [616, 1490], [617, 619], [617, 678], [617, 686], [617, 696], [617, 701], [617, 722], [617, 869], [617, 1491], [618, 620], [618, 723], [618, 832], [618, 1492], [619, 622], [619, 632], [619, 637], [619, 640], [619, 1493], [620, 633], [620, 638], [620, 680], [620, 725], [620, 832], [620, 1494], [621, 1495], [622, 634], [622, 674], [622, 1496], [623, 625], [623, 681], [623, 690], [623, 698], [623, 738], [623, 763], [623, 867], [623, 871], [623, 1497], [624, 626], [624, 637], [624, 640], [624, 683], [624, 854], [624, 867], [624, 1498], [625, 635], [625, 638], [625, 684], [625, 854], [625, 1499], [626, 636], [626, 673], [626, 686], [626, 698], [626, 1500], [627, 677], [627, 687], [627, 690], [627, 695], [627, 697], [627, 698], [627, 1501], [628, 829], [628, 1502], [629, 863], [629, 1503], [630, 694], [630, 717], [630, 800], [630, 818], [630, 1504], [631, 659], [631, 1505], [632, 634], [632, 637], [632, 691], [632, 755], [632, 821], [632, 834], [632, 855], [632, 1506], [633, 638], [633, 692], [633, 725], [633, 835], [633, 1507], [634, 673], [634, 674], [634, 693], [634, 754], [634, 1508], [635, 638], [635, 694], [635, 815], [635, 860], [635, 863], [635, 872], [635, 1509], [636, 873], [636, 1510], [637, 875], [637, 1511], [638, 814], [638, 874], [638, 875], [638, 1512], [639, 698], [639, 868], [639, 1513], [640, 641], [640, 699], [640, 1514], [641, 701], [641, 845], [641, 1515], [642, 662], [642, 663], [642, 664], [642, 665], [642, 666], [642, 1516], [643, 655], [643, 667], [643, 671], [643, 689], [643, 703], [643, 1517], [644, 704], [644, 757], [644, 855], [644, 1518], [645, 646], [645, 676], [645, 705], [645, 710], [645, 716], [645, 856], [645, 1519], [646, 665], [646, 706], [646, 710], [646, 753], [646, 757], [646, 858], [646, 1520], [647, 667], [647, 668], [647, 669], [647, 1521], [648, 676], [648, 688], [648, 1148], [648, 1522], [649, 670], [649, 671], [649, 672], [649, 673], [649, 674], [649, 675], [649, 707], [649, 1523], [650, 654], [650, 675], [650, 679], [650, 693], [650, 704], [650, 1524], [651, 655], [651, 669], [651, 675], [651, 682], [651, 688], [651, 699], [651, 708], [651, 717], [651, 1525], [652, 675], [652, 707], [652, 708], [652, 709], [652, 710], [652, 711], [652, 1526], [653, 670], [653, 675], [653, 702], [653, 1527], [654, 675], [654, 696], [654, 699], [654, 704], [654, 709], [654, 817], [654, 1528], [655, 671], [655, 675], [655, 688], [655, 710], [655, 1529], [656, 676], [656, 677], [656, 712], [656, 1112], [656, 1530], [657, 677], [657, 712], [657, 713], [657, 714], [657, 715], [657, 716], [657, 717], [657, 1531], [658, 677], [658, 697], [658, 715], [658, 853], [658, 1532], [659, 677], [659, 717], [659, 1533], [660, 678], [660, 679], [660, 680], [660, 1534], [661, 669], [661, 718], [661, 719], [661, 1535], [662, 681], [662, 682], [662, 683], [662, 684], [662, 685], [662, 686], [662, 687], [662, 720], [662, 1536], [663, 674], [663, 720], [663, 721], [663, 722], [663, 723], [663, 724], [663, 725], [663, 1537], [664, 713], [664, 722], [664, 749], [664, 860], [664, 1538], [665, 681], [665, 708], [665, 723], [665, 749], [665, 1539], [666, 724], [666, 1540], [667, 688], [667, 689], [667, 726], [667, 727], [667, 728], [667, 1161], [667, 1541], [668, 700], [668, 720], [668, 1542], [669, 689], [669, 1543], [670, 672], [670, 726], [670, 734], [670, 1544], [671, 674], [671, 686], [671, 720], [671, 727], [671, 730], [671, 738], [671, 748], [671, 1545], [672, 731], [672, 1546], [673, 732], [673, 1547], [674, 733], [674, 754], [674, 1548], [675, 1549], [676, 1550], [677, 1551], [678, 700], [678, 1552], [679, 691], [679, 692], [679, 693], [679, 737], [679, 1553], [680, 692], [680, 778], [680, 1554], [681, 684], [681, 702], [681, 1555], [682, 694], [682, 695], [682, 736], [682, 1556], [683, 686], [683, 738], [683, 1557], [684, 687], [684, 694], [684, 714], [684, 739], [684, 802], [684, 1558], [685, 1559], [686, 1560], [687, 695], [687, 1561], [688, 727], [688, 1562], [689, 708], [689, 748], [689, 1563], [690, 744], [690, 1564], [691, 693], [691, 708], [691, 711], [691, 1565], [692, 781], [692, 1566], [693, 696], [693, 718], [693, 1567], [694, 695], [694, 714], [694, 717], [694, 747], [694, 791], [694, 1568], [695, 698], [695, 749], [695, 1569], [696, 701], [696, 751], [696, 819], [696, 873], [696, 1570], [697, 752], [697, 850], [697, 1571], [698, 753], [698, 867], [698, 1572], [699, 701], [699, 729], [699, 742], [699, 755], [699, 1573], [700, 736], [700, 1574], [701, 718], [701, 741], [701, 798], [701, 800], [701, 1575], [702, 705], [702, 712], [702, 757], [702, 1576], [703, 710], [703, 726], [703, 730], [703, 1577], [704, 729], [704, 737], [704, 743], [704, 817], [704, 1578], [705, 706], [705, 818], [705, 1579], [706, 723], [706, 745], [706, 820], [706, 1580], [707, 729], [707, 730], [707, 731], [707, 732], [707, 733], [707, 1581], [708, 710], [708, 736], [708, 743], [708, 748], [708, 1582], [709, 711], [709, 746], [709, 751], [709, 759], [709, 777], [709, 1583], [710, 730], [710, 777], [710, 1584], [711, 732], [711, 760], [711, 1585], [712, 734], [712, 735], [712, 1586], [713, 716], [713, 736], [713, 749], [713, 1587], [714, 741], [714, 761], [714, 1588], [715, 746], [715, 752], [715, 762], [715, 815], [715, 1589], [716, 717], [716, 753], [716, 763], [716, 1590], [717, 765], [717, 1591], [718, 719], [718, 741], [718, 818], [718, 1592], [719, 1593], [720, 736], [720, 737], [720, 738], [720, 739], [720, 1594], [721, 733], [721, 740], [721, 743], [721, 1595], [722, 741], [722, 767], [722, 823], [722, 1596], [723, 725], [723, 745], [723, 776], [723, 1597], [724, 746], [724, 754], [724, 755], [724, 1598], [725, 739], [725, 765], [725, 823], [725, 1599], [726, 740], [726, 1600], [727, 741], [727, 748], [727, 1601], [728, 745], [728, 750], [728, 1602], [729, 741], [729, 742], [729, 743], [729, 744], [729, 769], [729, 1603], [730, 733], [730, 750], [730, 757], [730, 770], [730, 1604], [731, 766], [731, 1605], [732, 742], [732, 767], [732, 771], [732, 869], [732, 1606], [733, 743], [733, 750], [733, 1607], [734, 757], [734, 772], [734, 773], [734, 774], [734, 775], [734, 776], [734, 1608], [735, 1609], [736, 747], [736, 748], [736, 749], [736, 779], [736, 1610], [737, 779], [737, 780], [737, 781], [737, 782], [737, 1611], [738, 763], [738, 773], [738, 1612], [739, 747], [739, 763], [739, 770], [739, 781], [739, 1613], [740, 750], [740, 1614], [741, 783], [741, 787], [741, 791], [741, 1615], [742, 784], [742, 817], [742, 1616], [743, 779], [743, 785], [743, 791], [743, 1617], [744, 786], [744, 1618], [745, 1619], [746, 751], [746, 752], [746, 788], [746, 790], [746, 1620], [747, 749], [747, 766], [747, 1621], [748, 791], [748, 1622], [749, 753], [749, 792], [749, 860], [749, 1623], [750, 1624], [751, 754], [751, 770], [751, 796], [751, 832], [751, 1625], [752, 811], [752, 1626], [753, 757], [753, 812], [753, 846], [753, 1627], [754, 755], [754, 771], [754, 797], [754, 832], [754, 1628], [755, 769], [755, 784], [755, 799], [755, 834], [755, 1629], [756, 760], [756, 764], [756, 771], [756, 775], [756, 782], [756, 784], [756, 788], [756, 801], [756, 809], [756, 1630], [757, 776], [757, 802], [757, 1631], [758, 767], [758, 770], [758, 780], [758, 783], [758, 805], [758, 812], [758, 827], [758, 842], [758, 844], [758, 1632], [759, 760], [759, 788], [759, 796], [759, 811], [759, 831], [759, 837], [759, 1633], [760, 771], [760, 817], [760, 837], [760, 1634], [761, 783], [761, 816], [761, 1635], [762, 764], [762, 788], [762, 865], [762, 1636], [763, 765], [763, 773], [763, 789], [763, 841], [763, 1637], [764, 775], [764, 854], [764, 1638], [765, 776], [765, 823], [765, 841], [765, 866], [765, 1639], [766, 772], [766, 823], [766, 1640], [767, 783], [767, 790], [767, 824], [767, 869], [767, 1641], [768, 780], [768, 793], [768, 827], [768, 837], [768, 1642], [769, 783], [769, 784], [769, 785], [769, 786], [769, 1643], [770, 789], [770, 1644], [771, 784], [771, 1645], [772, 787], [772, 1646], [773, 776], [773, 789], [773, 1647], [774, 793], [774, 1648], [775, 829], [775, 867], [775, 1649], [776, 1650], [777, 778], [777, 794], [777, 1651], [778, 832], [778, 1652], [779, 789], [779, 790], [779, 791], [779, 792], [779, 1653], [780, 793], [780, 1654], [781, 795], [781, 803], [781, 804], [781, 835], [781, 1655], [782, 817], [782, 819], [782, 1656], [783, 794], [783, 836], [783, 1657], [784, 834], [784, 1658], [785, 823], [785, 837], [785, 1659], [786, 833], [786, 1660], [787, 800], [787, 804], [787, 818], [787, 823], [787, 838], [787, 839], [787, 840], [787, 841], [787, 1661], [788, 796], [788, 1662], [789, 842], [789, 1663], [790, 815], [790, 842], [790, 843], [790, 1664], [791, 839], [791, 1665], [792, 837], [792, 1666], [793, 844], [793, 1667], [794, 807], [794, 812], [794, 1668], [795, 813], [795, 1669], [796, 830], [796, 831], [796, 1670], [797, 799], [797, 822], [797, 847], [797, 869], [797, 1671], [798, 800], [798, 822], [798, 829], [798, 1672], [799, 817], [799, 822], [799, 850], [799, 1673], [800, 818], [800, 822], [800, 845], [800, 1674], [801, 822], [801, 829], [801, 1675], [802, 804], [802, 826], [802, 871], [802, 1676], [803, 805], [803, 813], [803, 826], [803, 837], [803, 850], [803, 1677], [804, 823], [804, 826], [804, 839], [804, 846], [804, 1678], [805, 824], [805, 826], [805, 828], [805, 836], [805, 847], [805, 871], [805, 1679], [806, 826], [806, 827], [806, 831], [806, 837], [806, 839], [806, 842], [806, 849], [806, 1680], [807, 838], [807, 848], [807, 858], [807, 862], [807, 871], [807, 874], [807, 1231], [807, 1681], [808, 850], [808, 865], [808, 1682], [809, 834], [809, 842], [809, 849], [809, 854], [809, 867], [809, 1683], [810, 828], [810, 840], [810, 843], [810, 851], [810, 1684], [811, 850], [811, 1685], [812, 842], [812, 846], [812, 848], [812, 1686], [813, 837], [813, 851], [813, 1687], [814, 832], [814, 841], [814, 844], [814, 852], [814, 866], [814, 1688], [815, 853], [815, 860], [815, 861], [815, 1689], [816, 836], [816, 872], [816, 1690], [817, 819], [817, 855], [817, 1691], [818, 820], [818, 856], [818, 1692], [819, 821], [819, 830], [819, 836], [819, 857], [819, 873], [819, 1693], [820, 838], [820, 858], [820, 1694], [821, 1695], [822, 859], [822, 1696], [823, 860], [823, 1697], [824, 825], [824, 828], [824, 836], [824, 843], [824, 861], [824, 1698], [825, 834], [825, 837], [825, 843], [825, 849], [825, 851], [825, 863], [825, 866], [825, 1699], [826, 864], [826, 1700], [827, 837], [827, 844], [827, 866], [827, 1701], [828, 851], [828, 1702], [829, 1703], [830, 836], [830, 869], [830, 1704], [831, 870], [831, 1705], [832, 1706], [833, 835], [833, 838], [833, 871], [833, 1707], [834, 1708], [835, 1709], [836, 1710], [837, 1711], [838, 1712], [839, 1713], [840, 1714], [841, 1715], [842, 1716], [843, 1717], [844, 1718], [845, 856], [845, 859], [845, 1719], [846, 848], [846, 860], [846, 864], [846, 1720], [847, 861], [847, 864], [847, 1721], [848, 862], [848, 864], [848, 1722], [849, 864], [849, 866], [849, 870], [849, 1723], [850, 865], [850, 1724], [851, 1725], [852, 873], [852, 1726], [853, 854], [853, 1727], [854, 867], [854, 1728], [855, 857], [855, 1729], [856, 858], [856, 1730], [857, 869], [857, 1731], [858, 1732], [859, 1733], [860, 862], [860, 1734], [861, 863], [861, 1735], [862, 871], [862, 1736], [863, 1737], [864, 1738], [865, 1739], [866, 1740], [867, 1741], [868, 1742], [869, 1743], [870, 1744], [871, 1745], [872, 874], [872, 1746], [873, 1747], [874, 1748], [875, 876], [875, 877], [875, 878], [875, 879], [875, 880], [875, 881], [875, 883], [875, 888], [875, 890], [875, 898], [875, 907], [875, 915], [875, 917], [875, 922], [875, 925], [875, 931], [875, 933], [875, 942], [875, 951], [875, 959], [875, 960], [875, 966], [875, 968], [875, 975], [875, 988], [875, 994], [875, 999], [875, 1003], [875, 1009], [875, 1011], [876, 880], [876, 881], [876, 899], [876, 900], [876, 901], [876, 902], [876, 903], [876, 904], [876, 905], [876, 906], [876, 1014], [876, 1015], [876, 1020], [876, 1021], [876, 1024], [876, 1031], [876, 1032], [876, 1033], [876, 1034], [876, 1038], [876, 1044], [876, 1051], [876, 1072], [876, 1302], [876, 1306], [876, 1307], [876, 1326], [876, 1327], [876, 1328], [876, 1329], [876, 1330], [876, 1331], [876, 1332], [876, 1441], [876, 1442], [876, 1449], [876, 1450], [876, 1454], [876, 1455], [876, 1462], [876, 1463], [876, 1465], [876, 1467], [876, 1471], [876, 1477], [876, 1482], [876, 1487], [876, 1490], [876, 1511], [876, 1512], [877, 879], [877, 881], [877, 884], [877, 891], [877, 982], [877, 983], [877, 984], [877, 985], [877, 986], [877, 987], [877, 1087], [877, 1092], [877, 1097], [877, 1101], [877, 1143], [877, 1148], [877, 1155], [877, 1161], [877, 1164], [877, 1171], [877, 1176], [877, 1182], [877, 1183], [878, 879], [878, 880], [878, 914], [878, 921], [878, 929], [878, 938], [878, 954], [878, 961], [878, 970], [878, 976], [878, 1221], [878, 1223], [878, 1225], [878, 1227], [878, 1236], [878, 1243], [878, 1245], [878, 1253], [878, 1267], [878, 1272], [879, 886], [879, 894], [879, 910], [879, 918], [879, 934], [879, 1007], [879, 1008], [879, 1169], [879, 1172], [879, 1176], [879, 1192], [879, 1208], [879, 1213], [879, 1219], [879, 1224], [879, 1227], [879, 1229], [879, 1231], [879, 1305], [880, 958], [880, 964], [880, 972], [880, 979], [880, 992], [880, 996], [880, 1000], [880, 1004], [880, 1036], [880, 1041], [880, 1241], [880, 1242], [880, 1243], [880, 1280], [880, 1282], [880, 1287], [880, 1291], [880, 1300], [880, 1301], [881, 943], [881, 944], [881, 945], [881, 946], [881, 947], [881, 948], [881, 949], [881, 950], [881, 1016], [881, 1021], [881, 1062], [881, 1063], [881, 1076], [881, 1080], [881, 1084], [881, 1091], [881, 1094], [881, 1096], [881, 1101], [881, 1106], [881, 1111], [881, 1112], [881, 1113], [881, 1115], [882, 885], [882, 907], [882, 931], [882, 956], [882, 1019], [882, 1053], [882, 1133], [882, 1188], [882, 1232], [883, 924], [883, 925], [883, 926], [883, 927], [883, 928], [883, 929], [883, 930], [883, 931], [883, 1014], [883, 1036], [883, 1048], [883, 1077], [883, 1083], [884, 886], [884, 906], [884, 931], [884, 969], [884, 984], [884, 1034], [884, 1103], [884, 1105], [884, 1135], [884, 1136], [884, 1137], [885, 887], [885, 924], [885, 931], [885, 951], [885, 990], [885, 1027], [885, 1029], [885, 1074], [885, 1135], [885, 1189], [885, 1234], [885, 1295], [886, 931], [886, 945], [886, 950], [886, 969], [886, 972], [886, 1188], [886, 1189], [886, 1190], [886, 1191], [886, 1192], [886, 1242], [886, 1253], [887, 927], [887, 931], [887, 988], [887, 1071], [887, 1075], [887, 1137], [887, 1191], [888, 916], [888, 931], [888, 941], [888, 949], [888, 956], [888, 984], [888, 990], [888, 999], [888, 1192], [888, 1236], [888, 1273], [889, 893], [889, 917], [889, 942], [889, 963], [889, 1028], [889, 1058], [889, 1108], [889, 1134], [889, 1138], [889, 1175], [889, 1193], [889, 1237], [890, 932], [890, 933], [890, 934], [890, 935], [890, 936], [890, 937], [890, 938], [890, 939], [890, 940], [890, 941], [890, 942], [890, 1015], [890, 1041], [890, 1076], [890, 1097], [891, 894], [891, 913], [891, 929], [891, 942], [891, 953], [891, 970], [891, 985], [891, 1016], [891, 1077], [891, 1097], [891, 1138], [891, 1139], [891, 1140], [891, 1141], [891, 1142], [892, 895], [892, 914], [892, 942], [892, 951], [892, 976], [892, 991], [892, 1017], [892, 1026], [892, 1029], [892, 1140], [892, 1174], [892, 1175], [892, 1194], [892, 1238], [892, 1276], [893, 896], [893, 932], [893, 942], [893, 960], [893, 995], [893, 1018], [893, 1078], [893, 1108], [893, 1109], [893, 1141], [893, 1174], [893, 1196], [893, 1200], [893, 1239], [893, 1277], [894, 934], [894, 942], [894, 1193], [894, 1194], [894, 1195], [894, 1244], [894, 1251], [895, 935], [895, 942], [895, 953], [895, 958], [895, 976], [895, 980], [895, 988], [895, 1019], [895, 1058], [895, 1060], [895, 1069], [895, 1188], [895, 1195], [895, 1200], [895, 1207], [895, 1240], [895, 1242], [895, 1245], [895, 1265], [895, 1278], [895, 1292], [896, 936], [896, 942], [896, 994], [896, 1079], [896, 1142], [896, 1196], [896, 1279], [897, 937], [897, 942], [897, 970], [897, 972], [897, 999], [897, 1205], [897, 1243], [898, 923], [898, 942], [898, 950], [898, 957], [898, 963], [898, 985], [898, 991], [898, 995], [898, 1003], [898, 1020], [898, 1280], [899, 903], [899, 921], [899, 939], [899, 951], [899, 961], [899, 977], [899, 992], [899, 1017], [899, 1022], [899, 1024], [899, 1064], [899, 1065], [899, 1081], [899, 1106], [899, 1150], [900, 904], [900, 960], [900, 982], [900, 996], [900, 1018], [900, 1025], [900, 1032], [900, 1037], [900, 1066], [900, 1067], [900, 1082], [900, 1092], [900, 1102], [900, 1112], [900, 1150], [900, 1170], [901, 905], [901, 929], [901, 941], [901, 968], [901, 976], [901, 984], [901, 1000], [901, 1026], [901, 1034], [901, 1068], [901, 1069], [901, 1083], [901, 1097], [901, 1115], [901, 1159], [902, 906], [902, 975], [902, 1004], [902, 1027], [902, 1038], [902, 1070], [902, 1071], [902, 1098], [902, 1103], [903, 943], [903, 988], [903, 1019], [903, 1039], [903, 1045], [903, 1053], [903, 1299], [904, 944], [904, 961], [904, 965], [904, 982], [904, 987], [904, 994], [904, 1023], [904, 1028], [904, 1035], [904, 1040], [904, 1054], [904, 1058], [904, 1072], [904, 1241], [904, 1245], [904, 1271], [905, 945], [905, 976], [905, 999], [905, 1029], [905, 1253], [906, 946], [906, 977], [906, 980], [906, 984], [906, 1003], [906, 1020], [906, 1030], [906, 1046], [906, 1073], [906, 1242], [907, 951], [907, 952], [907, 953], [907, 954], [907, 955], [907, 956], [907, 957], [907, 958], [907, 959], [907, 1024], [907, 1084], [907, 1143], [907, 1157], [908, 911], [908, 948], [908, 959], [908, 960], [908, 983], [908, 997], [908, 1025], [908, 1101], [908, 1125], [908, 1185], [909, 913], [909, 939], [909, 950], [909, 959], [909, 975], [909, 982], [909, 991], [909, 1005], [909, 1027], [909, 1037], [909, 1040], [909, 1055], [909, 1068], [909, 1086], [909, 1097], [909, 1106], [909, 1110], [909, 1144], [909, 1157], [909, 1173], [909, 1180], [909, 1189], [909, 1201], [909, 1210], [910, 914], [910, 959], [910, 1007], [910, 1087], [910, 1107], [910, 1109], [910, 1140], [910, 1180], [910, 1196], [910, 1197], [910, 1198], [911, 952], [911, 959], [911, 994], [911, 1028], [911, 1088], [911, 1185], [911, 1193], [911, 1210], [911, 1281], [912, 916], [912, 959], [912, 999], [912, 1012], [912, 1029], [912, 1089], [912, 1145], [912, 1189], [912, 1194], [912, 1257], [912, 1263], [913, 953], [913, 959], [913, 982], [913, 1003], [913, 1018], [913, 1030], [913, 1090], [913, 1146], [913, 1196], [913, 1244], [913, 1252], [913, 1263], [914, 954], [914, 959], [914, 983], [914, 987], [914, 991], [914, 1049], [914, 1197], [914, 1244], [914, 1245], [915, 959], [915, 965], [915, 973], [915, 980], [915, 986], [915, 997], [915, 1001], [915, 1005], [915, 1007], [915, 1009], [915, 1012], [915, 1031], [915, 1047], [915, 1062], [915, 1245], [915, 1282], [916, 956], [916, 959], [916, 1068], [916, 1147], [916, 1198], [916, 1252], [916, 1257], [916, 1283], [917, 960], [917, 961], [917, 962], [917, 963], [917, 964], [917, 965], [917, 966], [917, 1032], [917, 1081], [917, 1091], [917, 1125], [917, 1148], [917, 1162], [917, 1169], [918, 921], [918, 948], [918, 958], [918, 966], [918, 989], [918, 996], [918, 1008], [918, 1092], [918, 1106], [918, 1109], [918, 1126], [918, 1169], [918, 1178], [918, 1199], [918, 1200], [918, 1201], [919, 966], [919, 988], [919, 1010], [919, 1060], [919, 1065], [919, 1070], [919, 1150], [919, 1199], [919, 1246], [920, 923], [920, 966], [920, 1003], [920, 1013], [920, 1018], [920, 1019], [920, 1119], [920, 1120], [920, 1151], [920, 1200], [921, 961], [921, 966], [921, 989], [921, 1049], [921, 1072], [921, 1246], [922, 966], [922, 981], [922, 987], [922, 993], [922, 1006], [922, 1008], [922, 1010], [922, 1011], [922, 1013], [922, 1033], [922, 1063], [923, 963], [923, 966], [923, 1027], [923, 1070], [923, 1093], [923, 1100], [923, 1152], [923, 1201], [923, 1288], [924, 927], [924, 951], [924, 1026], [924, 1027], [924, 1127], [924, 1147], [924, 1247], [924, 1283], [925, 967], [925, 968], [925, 969], [925, 970], [925, 971], [925, 972], [925, 973], [925, 1034], [925, 1077], [925, 1094], [925, 1156], [925, 1163], [925, 1553], [926, 928], [926, 949], [926, 975], [926, 984], [926, 1046], [926, 1083], [926, 1101], [926, 1163], [927, 930], [927, 967], [927, 988], [927, 1069], [927, 1071], [927, 1095], [927, 1108], [927, 1153], [927, 1202], [927, 1249], [927, 1285], [928, 969], [928, 1003], [928, 1124], [928, 1127], [928, 1129], [929, 970], [929, 984], [929, 1020], [929, 1036], [929, 1247], [929, 1248], [929, 1249], [929, 1250], [929, 1251], [929, 1252], [929, 1253], [930, 971], [930, 1009], [930, 1090], [930, 1154], [930, 1252], [931, 1048], [931, 1253], [931, 1287], [932, 936], [932, 960], [932, 1037], [932, 1109], [932, 1131], [932, 1152], [932, 1180], [932, 1203], [932, 1254], [932, 1288], [933, 974], [933, 975], [933, 976], [933, 977], [933, 978], [933, 979], [933, 980], [933, 981], [933, 1038], [933, 1083], [933, 1096], [933, 1155], [933, 1172], [934, 938], [934, 957], [934, 972], [934, 991], [934, 1000], [934, 1097], [934, 1156], [934, 1172], [934, 1203], [934, 1204], [934, 1205], [935, 939], [935, 958], [935, 988], [935, 1004], [935, 1039], [935, 1098], [935, 1157], [935, 1204], [935, 1255], [936, 940], [936, 974], [936, 994], [936, 1040], [936, 1099], [936, 1158], [936, 1180], [936, 1181], [936, 1256], [936, 1258], [936, 1261], [936, 1289], [937, 941], [937, 999], [937, 1159], [937, 1232], [937, 1234], [937, 1266], [938, 976], [938, 1041], [938, 1205], [938, 1254], [938, 1255], [938, 1256], [938, 1257], [938, 1292], [939, 977], [939, 991], [939, 1004], [939, 1009], [939, 1017], [939, 1020], [939, 1027], [939, 1042], [939, 1089], [939, 1131], [939, 1132], [939, 1234], [939, 1247], [939, 1257], [939, 1261], [939, 1290], [939, 1299], [940, 978], [940, 1011], [940, 1043], [940, 1100], [940, 1160], [940, 1258], [941, 1000], [941, 1266], [942, 1016], [942, 1044], [942, 1291], [943, 947], [943, 964], [943, 980], [943, 988], [943, 996], [943, 1005], [943, 1045], [943, 1075], [943, 1095], [943, 1098], [943, 1102], [943, 1106], [943, 1114], [943, 1162], [944, 948], [944, 965], [944, 994], [944, 1007], [944, 1079], [944, 1088], [944, 1099], [944, 1107], [944, 1112], [944, 1118], [944, 1169], [944, 1177], [944, 1182], [944, 1388], [945, 949], [945, 972], [945, 999], [945, 1004], [945, 1089], [945, 1103], [945, 1115], [945, 1133], [945, 1163], [945, 1172], [945, 1183], [946, 950], [946, 1003], [946, 1046], [946, 1090], [946, 1108], [946, 1134], [946, 1173], [947, 982], [947, 1009], [947, 1023], [947, 1027], [947, 1030], [947, 1062], [947, 1104], [947, 1116], [947, 1119], [947, 1123], [947, 1127], [948, 983], [948, 996], [948, 1007], [948, 1011], [948, 1025], [948, 1047], [948, 1063], [948, 1072], [948, 1100], [948, 1109], [948, 1117], [948, 1120], [948, 1128], [948, 1131], [948, 1281], [949, 984], [949, 1004], [949, 1036], [950, 985], [950, 1005], [950, 1016], [950, 1020], [950, 1048], [950, 1093], [950, 1105], [950, 1110], [950, 1124], [951, 988], [951, 989], [951, 990], [951, 991], [951, 992], [951, 993], [951, 1106], [951, 1164], [951, 1215], [951, 1221], [952, 955], [952, 987], [952, 994], [952, 1008], [952, 1107], [952, 1176], [952, 1197], [952, 1218], [953, 957], [953, 980], [953, 1003], [953, 1007], [953, 1108], [953, 1118], [953, 1120], [953, 1129], [953, 1133], [953, 1140], [953, 1157], [953, 1165], [953, 1172], [953, 1195], [953, 1206], [953, 1215], [953, 1226], [953, 1237], [953, 1239], [953, 1249], [953, 1262], [954, 958], [954, 1178], [954, 1181], [954, 1195], [954, 1204], [954, 1221], [954, 1232], [954, 1258], [954, 1259], [955, 989], [955, 1011], [955, 1049], [955, 1109], [955, 1166], [955, 1197], [955, 1239], [955, 1254], [956, 990], [956, 1019], [956, 1050], [956, 1167], [956, 1207], [956, 1249], [956, 1255], [957, 991], [957, 1007], [957, 1099], [957, 1100], [957, 1110], [957, 1168], [957, 1258], [957, 1292], [958, 992], [958, 1008], [958, 1024], [958, 1126], [958, 1259], [958, 1292], [959, 1051], [959, 1087], [959, 1111], [959, 1125], [960, 994], [960, 995], [960, 996], [960, 997], [960, 1112], [960, 1162], [960, 1208], [960, 1223], [960, 1372], [961, 964], [961, 987], [961, 1010], [961, 1150], [961, 1169], [961, 1181], [961, 1223], [961, 1230], [961, 1233], [961, 1237], [961, 1260], [961, 1261], [962, 965], [962, 1009], [962, 1045], [962, 1132], [962, 1134], [962, 1170], [962, 1293], [963, 995], [963, 1027], [963, 1028], [963, 1052], [963, 1099], [963, 1184], [963, 1210], [963, 1261], [964, 996], [964, 1010], [964, 1022], [964, 1023], [964, 1032], [964, 1063], [964, 1126], [964, 1293], [965, 997], [965, 1021], [965, 1023], [965, 1047], [965, 1082], [965, 1123], [965, 1125], [966, 1047], [966, 1081], [966, 1092], [966, 1113], [967, 971], [967, 988], [967, 1053], [967, 1108], [967, 1114], [967, 1186], [967, 1188], [967, 1262], [967, 1294], [968, 998], [968, 999], [968, 1000], [968, 1001], [968, 1115], [968, 1156], [968, 1214], [969, 1003], [969, 1124], [969, 1163], [969, 1176], [969, 1190], [970, 972], [970, 1159], [970, 1183], [970, 1190], [970, 1262], [970, 1263], [970, 1264], [971, 973], [971, 998], [971, 1009], [971, 1055], [971, 1089], [971, 1090], [971, 1116], [971, 1135], [971, 1180], [971, 1211], [971, 1263], [971, 1295], [972, 1000], [972, 1016], [972, 1034], [972, 1036], [972, 1294], [972, 1295], [973, 1001], [973, 1057], [973, 1168], [973, 1212], [973, 1264], [974, 978], [974, 994], [974, 1058], [974, 1118], [974, 1181], [974, 1187], [974, 1193], [974, 1195], [974, 1296], [975, 1002], [975, 1003], [975, 1004], [975, 1005], [975, 1006], [975, 1157], [975, 1163], [975, 1171], [975, 1213], [975, 1225], [976, 979], [976, 1159], [976, 1172], [976, 1214], [976, 1225], [976, 1265], [976, 1266], [977, 980], [977, 1009], [977, 1059], [977, 1119], [977, 1135], [977, 1173], [977, 1179], [977, 1215], [977, 1265], [978, 981], [978, 1002], [978, 1011], [978, 1060], [978, 1120], [978, 1174], [978, 1195], [978, 1199], [978, 1216], [978, 1276], [978, 1277], [978, 1297], [979, 1004], [979, 1017], [979, 1019], [979, 1038], [979, 1041], [979, 1266], [979, 1296], [979, 1297], [980, 1005], [980, 1016], [980, 1019], [980, 1024], [980, 1030], [980, 1061], [980, 1098], [980, 1108], [980, 1121], [980, 1151], [980, 1167], [980, 1187], [980, 1277], [980, 1285], [980, 1294], [981, 1006], [981, 1122], [981, 1175], [981, 1217], [981, 1276], [982, 986], [982, 1009], [982, 1123], [982, 1154], [982, 1170], [982, 1173], [982, 1177], [982, 1229], [982, 1271], [983, 987], [983, 1011], [983, 1160], [983, 1166], [983, 1174], [983, 1178], [983, 1182], [983, 1218], [983, 1223], [983, 1228], [983, 1231], [983, 1271], [984, 1147], [984, 1167], [984, 1179], [984, 1183], [984, 1192], [984, 1225], [985, 1124], [985, 1152], [985, 1168], [985, 1180], [985, 1190], [985, 1226], [986, 1007], [986, 1025], [986, 1028], [986, 1062], [986, 1087], [986, 1108], [986, 1110], [986, 1184], [986, 1185], [986, 1186], [987, 1008], [987, 1024], [987, 1028], [987, 1049], [987, 1063], [987, 1092], [987, 1107], [987, 1125], [987, 1175], [987, 1181], [987, 1187], [988, 1009], [988, 1010], [988, 1219], [988, 1267], [988, 1270], [989, 993], [989, 1011], [989, 1178], [989, 1227], [989, 1259], [989, 1270], [990, 1064], [990, 1179], [991, 1145], [991, 1147], [991, 1180], [991, 1204], [991, 1215], [991, 1220], [991, 1225], [991, 1229], [991, 1244], [991, 1257], [991, 1268], [991, 1288], [991, 1289], [991, 1295], [992, 1065], [992, 1199], [992, 1221], [992, 1230], [992, 1257], [992, 1265], [992, 1276], [992, 1283], [993, 1010], [993, 1126], [993, 1181], [993, 1222], [993, 1259], [993, 1269], [993, 1289], [993, 1296], [994, 1011], [994, 1182], [994, 1241], [995, 1066], [995, 1184], [996, 1199], [996, 1223], [996, 1229], [996, 1241], [996, 1270], [996, 1275], [996, 1277], [996, 1284], [996, 1288], [997, 1067], [997, 1123], [997, 1151], [997, 1152], [997, 1271], [998, 1001], [998, 1009], [998, 1068], [998, 1127], [998, 1179], [998, 1180], [998, 1198], [998, 1247], [999, 1012], [999, 1183], [999, 1214], [999, 1273], [999, 1427], [1000, 1192], [1000, 1251], [1000, 1426], [1001, 1012], [1001, 1069], [1001, 1129], [1001, 1167], [1001, 1168], [1001, 1195], [1001, 1202], [1002, 1006], [1002, 1011], [1002, 1070], [1002, 1131], [1002, 1199], [1002, 1201], [1002, 1254], [1002, 1257], [1002, 1298], [1003, 1013], [1003, 1215], [1003, 1224], [1003, 1242], [1003, 1272], [1004, 1225], [1004, 1242], [1004, 1273], [1004, 1298], [1004, 1299], [1005, 1071], [1005, 1184], [1005, 1191], [1005, 1202], [1005, 1226], [1005, 1299], [1006, 1013], [1006, 1039], [1006, 1040], [1006, 1132], [1006, 1257], [1007, 1185], [1007, 1212], [1007, 1226], [1007, 1228], [1007, 1229], [1007, 1245], [1007, 1281], [1008, 1217], [1008, 1222], [1008, 1230], [1008, 1231], [1008, 1241], [1008, 1281], [1009, 1229], [1009, 1300], [1010, 1065], [1010, 1230], [1010, 1243], [1010, 1270], [1010, 1274], [1011, 1072], [1011, 1231], [1011, 1270], [1011, 1301], [1011, 1681], [1012, 1133], [1012, 1186], [1012, 1191], [1012, 1195], [1012, 1232], [1012, 1294], [1013, 1019], [1013, 1073], [1013, 1134], [1013, 1187], [1013, 1237], [1013, 1296], [1014, 1034], [1014, 1035], [1014, 1036], [1015, 1037], [1015, 1038], [1015, 1039], [1015, 1040], [1015, 1041], [1015, 1042], [1015, 1043], [1015, 1044], [1015, 1076], [1016, 1030], [1016, 1036], [1016, 1044], [1016, 1076], [1016, 1077], [1016, 1078], [1016, 1079], [1017, 1019], [1017, 1044], [1017, 1120], [1017, 1122], [1017, 1237], [1017, 1276], [1018, 1037], [1018, 1044], [1018, 1066], [1018, 1078], [1018, 1120], [1018, 1121], [1018, 1277], [1019, 1039], [1019, 1044], [1019, 1061], [1019, 1278], [1019, 1282], [1019, 1299], [1020, 1044], [1020, 1048], [1020, 1052], [1020, 1066], [1020, 1253], [1020, 1280], [1020, 1299], [1021, 1045], [1021, 1046], [1021, 1047], [1021, 1048], [1021, 1080], [1021, 1081], [1021, 1082], [1021, 1083], [1022, 1042], [1022, 1059], [1022, 1065], [1022, 1081], [1023, 1063], [1023, 1282], [1023, 1293], [1024, 1049], [1024, 1050], [1024, 1051], [1024, 1084], [1024, 1092], [1024, 1098], [1025, 1028], [1025, 1047], [1025, 1051], [1025, 1067], [1025, 1080], [1025, 1092], [1026, 1029], [1026, 1051], [1026, 1069], [1026, 1085], [1026, 1147], [1027, 1030], [1027, 1042], [1027, 1048], [1027, 1051], [1027, 1071], [1027, 1076], [1027, 1081], [1027, 1086], [1027, 1098], [1027, 1119], [1028, 1051], [1028, 1088], [1028, 1281], [1029, 1051], [1029, 1089], [1029, 1290], [1029, 1295], [1030, 1051], [1030, 1090], [1030, 1147], [1030, 1295], [1031, 1051], [1031, 1057], [1031, 1061], [1031, 1062], [1031, 1067], [1031, 1069], [1031, 1071], [1031, 1282], [1032, 1052], [1032, 1091], [1032, 1102], [1033, 1063], [1033, 1072], [1033, 1073], [1034, 1053], [1034, 1054], [1034, 1055], [1034, 1056], [1034, 1057], [1034, 1094], [1034, 1097], [1034, 1103], [1035, 1054], [1035, 1154], [1035, 1286], [1036, 1283], [1036, 1284], [1036, 1285], [1036, 1286], [1036, 1287], [1037, 1040], [1037, 1093], [1037, 1288], [1038, 1058], [1038, 1059], [1038, 1060], [1038, 1061], [1038, 1096], [1039, 1042], [1039, 1098], [1040, 1043], [1040, 1058], [1040, 1099], [1040, 1126], [1040, 1201], [1040, 1289], [1040, 1292], [1041, 1288], [1041, 1289], [1041, 1290], [1041, 1291], [1042, 1059], [1042, 1283], [1042, 1290], [1043, 1060], [1043, 1072], [1043, 1100], [1043, 1292], [1044, 1291], [1045, 1061], [1045, 1071], [1045, 1081], [1045, 1102], [1046, 1048], [1046, 1119], [1046, 1121], [1047, 1072], [1048, 1071], [1048, 1105], [1049, 1072], [1049, 1109], [1049, 1277], [1049, 1288], [1050, 1064], [1050, 1285], [1051, 1111], [1052, 1066], [1052, 1132], [1052, 1233], [1053, 1055], [1053, 1114], [1053, 1121], [1053, 1133], [1053, 1137], [1053, 1157], [1053, 1294], [1054, 1056], [1054, 1158], [1054, 1264], [1055, 1057], [1055, 1068], [1055, 1074], [1055, 1116], [1055, 1295], [1056, 1072], [1056, 1117], [1056, 1160], [1056, 1264], [1057, 1069], [1057, 1110], [1058, 1060], [1058, 1118], [1058, 1126], [1058, 1134], [1058, 1173], [1058, 1175], [1058, 1200], [1058, 1296], [1059, 1061], [1059, 1074], [1059, 1119], [1060, 1070], [1060, 1072], [1060, 1120], [1060, 1297], [1061, 1071], [1061, 1121], [1061, 1134], [1062, 1111], [1062, 1121], [1062, 1129], [1062, 1133], [1063, 1113], [1063, 1122], [1063, 1125], [1063, 1126], [1063, 1130], [1063, 1132], [1063, 1134], [1064, 1260], [1065, 1290], [1066, 1131], [1066, 1152], [1066, 1235], [1066, 1248], [1067, 1093], [1067, 1104], [1068, 1069], [1068, 1127], [1068, 1189], [1068, 1252], [1068, 1265], [1068, 1283], [1069, 1110], [1069, 1129], [1069, 1188], [1070, 1072], [1070, 1131], [1070, 1288], [1070, 1290], [1070, 1298], [1071, 1299], [1072, 1245], [1072, 1301], [1073, 1134], [1073, 1293], [1074, 1075], [1074, 1106], [1074, 1135], [1075, 1095], [1075, 1137], [1075, 1275], [1076, 1096], [1076, 1097], [1076, 1098], [1076, 1099], [1076, 1100], [1077, 1090], [1077, 1101], [1077, 1108], [1077, 1124], [1077, 1139], [1078, 1079], [1078, 1112], [1078, 1141], [1078, 1146], [1078, 1151], [1078, 1284], [1079, 1099], [1079, 1142], [1079, 1146], [1079, 1187], [1079, 1193], [1079, 1286], [1080, 1101], [1080, 1102], [1080, 1103], [1080, 1104], [1080, 1105], [1081, 1106], [1081, 1119], [1082, 1112], [1082, 1123], [1082, 1530], [1083, 1115], [1084, 1106], [1084, 1107], [1084, 1108], [1084, 1109], [1084, 1110], [1084, 1111], [1084, 1143], [1085, 1089], [1085, 1111], [1085, 1115], [1085, 1129], [1085, 1140], [1086, 1090], [1086, 1105], [1086, 1111], [1086, 1123], [1086, 1135], [1086, 1144], [1086, 1152], [1087, 1111], [1087, 1143], [1087, 1144], [1087, 1145], [1087, 1146], [1087, 1147], [1088, 1107], [1088, 1111], [1088, 1138], [1089, 1111], [1089, 1133], [1089, 1135], [1089, 1140], [1089, 1145], [1089, 1247], [1090, 1108], [1090, 1111], [1090, 1123], [1090, 1146], [1091, 1112], [1091, 1113], [1091, 1148], [1091, 1522], [1092, 1113], [1092, 1148], [1092, 1149], [1092, 1150], [1092, 1151], [1092, 1152], [1093, 1113], [1093, 1152], [1094, 1114], [1094, 1115], [1094, 1116], [1094, 1117], [1095, 1114], [1095, 1153], [1095, 1284], [1096, 1118], [1096, 1119], [1096, 1120], [1096, 1121], [1096, 1122], [1096, 1155], [1097, 1110], [1097, 1155], [1097, 1156], [1097, 1157], [1097, 1158], [1097, 1159], [1097, 1160], [1098, 1149], [1098, 1157], [1098, 1288], [1099, 1100], [1099, 1118], [1099, 1144], [1099, 1158], [1100, 1120], [1100, 1160], [1100, 1288], [1101, 1123], [1101, 1124], [1101, 1125], [1101, 1161], [1101, 1162], [1101, 1163], [1101, 1541], [1102, 1104], [1102, 1121], [1102, 1162], [1102, 1170], [1103, 1136], [1103, 1155], [1103, 1163], [1104, 1123], [1105, 1124], [1106, 1126], [1106, 1164], [1106, 1169], [1106, 1173], [1107, 1109], [1107, 1125], [1107, 1161], [1107, 1169], [1108, 1110], [1108, 1121], [1108, 1155], [1108, 1162], [1108, 1165], [1108, 1173], [1108, 1184], [1109, 1166], [1110, 1168], [1110, 1188], [1112, 1177], [1114, 1116], [1114, 1136], [1115, 1127], [1115, 1128], [1115, 1129], [1115, 1130], [1115, 1172], [1115, 1353], [1116, 1127], [1116, 1136], [1116, 1144], [1117, 1128], [1117, 1212], [1118, 1120], [1118, 1138], [1119, 1121], [1119, 1173], [1120, 1122], [1120, 1131], [1120, 1150], [1120, 1174], [1120, 1237], [1122, 1132], [1122, 1175], [1123, 1162], [1123, 1177], [1124, 1144], [1124, 1184], [1126, 1181], [1127, 1129], [1127, 1144], [1127, 1147], [1127, 1215], [1128, 1130], [1128, 1216], [1129, 1133], [1129, 1153], [1130, 1217], [1131, 1132], [1131, 1150], [1131, 1152], [1131, 1226], [1131, 1261], [1132, 1134], [1133, 1137], [1133, 1186], [1133, 1249], [1133, 1299], [1134, 1187], [1135, 1137], [1135, 1164], [1135, 1179], [1135, 1189], [1136, 1171], [1137, 1153], [1137, 1191], [1137, 1233], [1137, 1235], [1138, 1141], [1138, 1148], [1138, 1193], [1139, 1146], [1139, 1161], [1139, 1165], [1140, 1164], [1140, 1172], [1140, 1180], [1140, 1194], [1140, 1247], [1141, 1142], [1141, 1248], [1142, 1158], [1142, 1182], [1142, 1250], [1143, 1164], [1143, 1165], [1143, 1166], [1143, 1167], [1143, 1168], [1144, 1146], [1144, 1171], [1144, 1177], [1144, 1180], [1144, 1184], [1145, 1147], [1145, 1183], [1145, 1186], [1145, 1211], [1146, 1165], [1146, 1177], [1146, 1196], [1146, 1211], [1147, 1167], [1147, 1198], [1147, 1295], [1148, 1169], [1148, 1170], [1149, 1151], [1149, 1171], [1150, 1199], [1151, 1152], [1151, 1187], [1151, 1200], [1152, 1201], [1153, 1154], [1153, 1202], [1153, 1248], [1155, 1171], [1155, 1172], [1155, 1173], [1155, 1174], [1155, 1175], [1156, 1168], [1156, 1176], [1156, 1180], [1157, 1204], [1157, 1254], [1158, 1160], [1158, 1182], [1158, 1210], [1159, 1183], [1159, 1188], [1159, 1189], [1159, 1598], [1160, 1174], [1160, 1201], [1160, 1254], [1161, 1176], [1161, 1177], [1162, 1184], [1163, 1183], [1164, 1178], [1164, 1179], [1164, 1180], [1164, 1181], [1165, 1168], [1165, 1185], [1165, 1193], [1165, 1202], [1165, 1206], [1166, 1178], [1166, 1203], [1167, 1179], [1167, 1202], [1167, 1204], [1167, 1207], [1167, 1294], [1168, 1180], [1168, 1185], [1169, 1193], [1169, 1208], [1169, 1209], [1169, 1210], [1170, 1609], [1171, 1184], [1171, 1213], [1172, 1213], [1172, 1214], [1172, 1215], [1172, 1216], [1172, 1217], [1173, 1200], [1173, 1209], [1173, 1215], [1174, 1175], [1174, 1200], [1174, 1206], [1174, 1216], [1175, 1217], [1176, 1185], [1176, 1218], [1178, 1181], [1178, 1223], [1179, 1247], [1180, 1196], [1180, 1213], [1180, 1220], [1180, 1226], [1181, 1222], [1182, 1228], [1183, 1186], [1183, 1190], [1183, 1225], [1184, 1226], [1185, 1228], [1186, 1188], [1186, 1206], [1186, 1264], [1187, 1193], [1187, 1244], [1187, 1277], [1188, 1189], [1188, 1207], [1188, 1232], [1188, 1264], [1189, 1191], [1189, 1234], [1189, 1263], [1189, 1265], [1190, 1272], [1191, 1202], [1191, 1219], [1191, 1294], [1192, 1198], [1192, 1207], [1192, 1236], [1192, 1242], [1192, 1251], [1193, 1210], [1193, 1237], [1194, 1195], [1194, 1197], [1194, 1214], [1194, 1220], [1194, 1238], [1195, 1204], [1195, 1206], [1195, 1214], [1195, 1219], [1195, 1240], [1195, 1244], [1195, 1258], [1195, 1272], [1196, 1206], [1196, 1224], [1196, 1244], [1197, 1218], [1197, 1220], [1197, 1227], [1198, 1207], [1198, 1247], [1198, 1268], [1199, 1219], [1199, 1230], [1199, 1246], [1200, 1201], [1200, 1209], [1200, 1224], [1200, 1271], [1201, 1210], [1201, 1254], [1201, 1271], [1201, 1292], [1202, 1219], [1202, 1249], [1203, 1208], [1203, 1254], [1204, 1219], [1204, 1225], [1204, 1255], [1204, 1294], [1205, 1214], [1205, 1227], [1205, 1258], [1205, 1268], [1206, 1224], [1206, 1228], [1208, 1223], [1209, 1210], [1209, 1224], [1210, 1261], [1211, 1212], [1211, 1229], [1211, 1263], [1212, 1264], [1213, 1224], [1213, 1225], [1213, 1226], [1214, 1227], [1215, 1229], [1215, 1265], [1215, 1296], [1216, 1217], [1216, 1231], [1216, 1238], [1216, 1239], [1217, 1237], [1217, 1238], [1217, 1296], [1218, 1231], [1219, 1229], [1219, 1230], [1219, 1267], [1220, 1254], [1220, 1256], [1220, 1263], [1220, 1268], [1221, 1234], [1221, 1238], [1221, 1247], [1221, 1267], [1221, 1268], [1221, 1269], [1222, 1230], [1222, 1256], [1222, 1269], [1223, 1235], [1223, 1239], [1223, 1248], [1223, 1254], [1223, 1270], [1223, 1271], [1224, 1272], [1225, 1272], [1225, 1273], [1226, 1261], [1229, 1241], [1229, 1244], [1230, 1241], [1230, 1274], [1232, 1234], [1232, 1253], [1232, 1278], [1232, 1294], [1233, 1235], [1233, 1253], [1233, 1260], [1234, 1247], [1234, 1253], [1235, 1248], [1235, 1253], [1235, 1275], [1236, 1253], [1236, 1260], [1237, 1239], [1237, 1261], [1237, 1281], [1237, 1296], [1238, 1240], [1238, 1268], [1238, 1276], [1239, 1254], [1239, 1277], [1240, 1255], [1240, 1259], [1240, 1267], [1240, 1278], [1240, 1296], [1240, 1297], [1241, 1279], [1241, 1281], [1241, 1286], [1241, 1289], [1241, 1296], [1241, 1301], [1241, 1438], [1242, 1265], [1242, 1272], [1242, 1280], [1243, 1259], [1243, 1266], [1243, 1270], [1243, 1273], [1243, 1684], [1244, 1272], [1244, 1277], [1244, 1279], [1245, 1264], [1245, 1271], [1245, 1282], [1245, 1292], [1246, 1267], [1246, 1274], [1246, 1297], [1246, 1298], [1247, 1249], [1247, 1283], [1248, 1250], [1248, 1284], [1249, 1252], [1249, 1262], [1249, 1267], [1249, 1285], [1249, 1299], [1250, 1286], [1251, 1280], [1252, 1263], [1253, 1287], [1254, 1256], [1254, 1288], [1255, 1257], [1255, 1259], [1255, 1267], [1255, 1273], [1256, 1289], [1257, 1265], [1257, 1268], [1257, 1273], [1257, 1276], [1257, 1280], [1257, 1290], [1257, 1292], [1258, 1268], [1258, 1292], [1261, 1281], [1262, 1263], [1262, 1267], [1262, 1294], [1263, 1264], [1263, 1295], [1266, 1273], [1266, 1276], [1266, 1278], [1267, 1274], [1269, 1274], [1271, 1715], [1275, 1284], [1275, 1287], [1276, 1278], [1276, 1291], [1277, 1279], [1277, 1288], [1277, 1291], [1278, 1291], [1279, 1289], [1279, 1291], [1280, 1291], [1280, 1292], [1282, 1299], [1282, 1300], [1283, 1285], [1284, 1286], [1285, 1294], [1288, 1289], [1289, 1296], [1290, 1300], [1293, 1300], [1294, 1295], [1295, 1300], [1296, 1297], [1297, 1298], [1297, 1301], [1298, 1301], [1302, 1303], [1302, 1304], [1302, 1305], [1302, 1306], [1302, 1307], [1302, 1309], [1302, 1316], [1302, 1318], [1302, 1325], [1302, 1333], [1302, 1341], [1302, 1343], [1302, 1350], [1302, 1353], [1302, 1359], [1302, 1361], [1302, 1370], [1302, 1378], [1302, 1387], [1302, 1388], [1302, 1394], [1302, 1396], [1302, 1403], [1302, 1415], [1302, 1421], [1302, 1426], [1302, 1430], [1302, 1436], [1302, 1438], [1303, 1305], [1303, 1307], [1303, 1311], [1303, 1319], [1303, 1410], [1303, 1411], [1303, 1412], [1303, 1413], [1303, 1414], [1303, 1526], [1303, 1531], [1303, 1535], [1303, 1537], [1303, 1541], [1303, 1581], [1303, 1586], [1303, 1594], [1303, 1600], [1303, 1603], [1303, 1610], [1303, 1614], [1303, 1615], [1303, 1619], [1303, 1620], [1304, 1305], [1304, 1306], [1304, 1340], [1304, 1348], [1304, 1357], [1304, 1366], [1304, 1382], [1304, 1389], [1304, 1398], [1304, 1404], [1304, 1661], [1304, 1664], [1304, 1667], [1304, 1675], [1304, 1680], [1304, 1684], [1304, 1688], [1304, 1696], [1304, 1700], [1304, 1710], [1304, 1712], [1304, 1716], [1305, 1314], [1305, 1336], [1305, 1345], [1305, 1362], [1305, 1434], [1305, 1435], [1305, 1608], [1305, 1611], [1305, 1614], [1305, 1630], [1305, 1643], [1305, 1646], [1305, 1653], [1305, 1657], [1305, 1662], [1305, 1663], [1305, 1667], [1305, 1668], [1305, 1669], [1306, 1386], [1306, 1392], [1306, 1400], [1306, 1408], [1306, 1419], [1306, 1423], [1306, 1427], [1306, 1431], [1306, 1454], [1306, 1461], [1306, 1464], [1306, 1469], [1306, 1474], [1306, 1681], [1306, 1682], [1306, 1683], [1306, 1684], [1306, 1723], [1306, 1726], [1306, 1733], [1306, 1738], [1306, 1748], [1307, 1371], [1307, 1372], [1307, 1373], [1307, 1374], [1307, 1375], [1307, 1376], [1307, 1377], [1307, 1443], [1307, 1450], [1307, 1501], [1307, 1516], [1307, 1521], [1307, 1523], [1307, 1530], [1307, 1534], [1307, 1536], [1307, 1541], [1307, 1549], [1307, 1550], [1307, 1551], [1307, 1553], [1307, 1556], [1308, 1312], [1308, 1333], [1308, 1359], [1308, 1384], [1308, 1446], [1308, 1491], [1308, 1540], [1308, 1570], [1308, 1628], [1308, 1671], [1309, 1352], [1309, 1353], [1309, 1354], [1309, 1355], [1309, 1356], [1309, 1357], [1309, 1358], [1309, 1359], [1309, 1441], [1309, 1469], [1309, 1481], [1309, 1517], [1309, 1535], [1310, 1313], [1310, 1359], [1310, 1361], [1310, 1373], [1310, 1407], [1310, 1450], [1310, 1495], [1310, 1542], [1311, 1314], [1311, 1332], [1311, 1359], [1311, 1397], [1311, 1411], [1311, 1467], [1311, 1535], [1311, 1542], [1311, 1543], [1311, 1573], [1311, 1574], [1311, 1575], [1312, 1315], [1312, 1352], [1312, 1359], [1312, 1378], [1312, 1417], [1312, 1458], [1312, 1459], [1312, 1514], [1312, 1540], [1312, 1573], [1312, 1629], [1312, 1673], [1313, 1354], [1313, 1359], [1313, 1403], [1313, 1479], [1313, 1491], [1313, 1493], [1313, 1574], [1314, 1359], [1314, 1373], [1314, 1377], [1314, 1397], [1314, 1400], [1314, 1628], [1314, 1629], [1314, 1630], [1314, 1683], [1314, 1696], [1314, 1744], [1315, 1355], [1315, 1359], [1315, 1415], [1315, 1510], [1315, 1515], [1315, 1575], [1316, 1342], [1316, 1359], [1316, 1369], [1316, 1384], [1316, 1407], [1316, 1411], [1316, 1417], [1316, 1426], [1316, 1630], [1316, 1675], [1316, 1717], [1317, 1321], [1317, 1343], [1317, 1370], [1317, 1391], [1317, 1497], [1317, 1545], [1317, 1572], [1317, 1576], [1317, 1631], [1317, 1676], [1318, 1360], [1318, 1361], [1318, 1362], [1318, 1363], [1318, 1364], [1318, 1365], [1318, 1366], [1318, 1367], [1318, 1368], [1318, 1369], [1318, 1370], [1318, 1442], [1318, 1474], [1318, 1485], [1318, 1495], [1318, 1516], [1318, 1537], [1319, 1339], [1319, 1357], [1319, 1370], [1319, 1381], [1319, 1398], [1319, 1412], [1319, 1443], [1319, 1517], [1319, 1537], [1319, 1576], [1319, 1577], [1319, 1578], [1319, 1579], [1319, 1580], [1320, 1322], [1320, 1340], [1320, 1370], [1320, 1378], [1320, 1404], [1320, 1418], [1320, 1444], [1320, 1457], [1320, 1459], [1320, 1518], [1320, 1578], [1320, 1613], [1320, 1677], [1321, 1323], [1321, 1360], [1321, 1370], [1321, 1388], [1321, 1422], [1321, 1519], [1321, 1545], [1321, 1546], [1321, 1579], [1321, 1613], [1321, 1637], [1321, 1678], [1321, 1720], [1322, 1363], [1322, 1370], [1322, 1381], [1322, 1386], [1322, 1404], [1322, 1409], [1322, 1415], [1322, 1446], [1322, 1497], [1322, 1499], [1322, 1508], [1322, 1628], [1322, 1632], [1322, 1637], [1322, 1645], [1322, 1679], [1322, 1683], [1322, 1688], [1322, 1708], [1322, 1721], [1322, 1740], [1323, 1364], [1323, 1370], [1323, 1421], [1323, 1447], [1323, 1520], [1323, 1580], [1323, 1722], [1324, 1365], [1324, 1370], [1324, 1398], [1324, 1400], [1324, 1426], [1324, 1448], [1324, 1642], [1324, 1684], [1325, 1351], [1325, 1370], [1325, 1377], [1325, 1385], [1325, 1391], [1325, 1412], [1325, 1418], [1325, 1422], [1325, 1430], [1325, 1449], [1325, 1680], [1325, 1723], [1326, 1330], [1326, 1348], [1326, 1367], [1326, 1378], [1326, 1389], [1326, 1405], [1326, 1419], [1326, 1444], [1326, 1451], [1326, 1455], [1326, 1466], [1326, 1502], [1326, 1503], [1326, 1588], [1327, 1331], [1327, 1349], [1327, 1388], [1327, 1410], [1327, 1423], [1327, 1456], [1327, 1463], [1327, 1470], [1327, 1504], [1327, 1505], [1327, 1522], [1327, 1531], [1327, 1550], [1327, 1588], [1327, 1609], [1328, 1357], [1328, 1369], [1328, 1396], [1328, 1404], [1328, 1411], [1328, 1427], [1328, 1445], [1328, 1452], [1328, 1457], [1328, 1467], [1328, 1506], [1328, 1507], [1328, 1508], [1328, 1537], [1328, 1553], [1328, 1598], [1329, 1332], [1329, 1403], [1329, 1431], [1329, 1458], [1329, 1471], [1329, 1509], [1329, 1510], [1329, 1538], [1329, 1542], [1329, 1556], [1330, 1371], [1330, 1415], [1330, 1446], [1330, 1472], [1330, 1478], [1330, 1491], [1330, 1511], [1330, 1747], [1331, 1372], [1331, 1389], [1331, 1393], [1331, 1410], [1331, 1414], [1331, 1421], [1331, 1447], [1331, 1453], [1331, 1468], [1331, 1473], [1331, 1492], [1331, 1497], [1331, 1512], [1331, 1681], [1331, 1688], [1331, 1715], [1332, 1374], [1332, 1405], [1332, 1409], [1332, 1411], [1332, 1430], [1332, 1449], [1332, 1460], [1332, 1479], [1332, 1513], [1332, 1683], [1333, 1378], [1333, 1379], [1333, 1380], [1333, 1381], [1333, 1382], [1333, 1383], [1333, 1384], [1333, 1385], [1333, 1386], [1333, 1387], [1333, 1455], [1333, 1523], [1333, 1581], [1333, 1596], [1334, 1337], [1334, 1376], [1334, 1387], [1334, 1388], [1334, 1424], [1334, 1456], [1334, 1541], [1334, 1602], [1334, 1624], [1335, 1339], [1335, 1367], [1335, 1377], [1335, 1387], [1335, 1403], [1335, 1410], [1335, 1418], [1335, 1432], [1335, 1458], [1335, 1470], [1335, 1473], [1335, 1493], [1335, 1506], [1335, 1518], [1335, 1525], [1335, 1537], [1335, 1548], [1335, 1582], [1335, 1596], [1335, 1612], [1335, 1617], [1335, 1629], [1335, 1639], [1335, 1650], [1336, 1340], [1336, 1387], [1336, 1434], [1336, 1526], [1336, 1544], [1336, 1546], [1336, 1578], [1336, 1617], [1336, 1633], [1336, 1634], [1337, 1379], [1337, 1387], [1337, 1421], [1337, 1527], [1337, 1624], [1337, 1631], [1337, 1650], [1338, 1342], [1338, 1380], [1338, 1387], [1338, 1426], [1338, 1439], [1338, 1459], [1338, 1528], [1338, 1583], [1338, 1629], [1338, 1633], [1338, 1685], [1338, 1699], [1338, 1724], [1338, 1744], [1339, 1381], [1339, 1387], [1339, 1410], [1339, 1430], [1339, 1447], [1339, 1460], [1339, 1529], [1339, 1584], [1339, 1686], [1339, 1695], [1340, 1382], [1340, 1387], [1340, 1414], [1340, 1418], [1340, 1461], [1340, 1483], [1340, 1685], [1340, 1686], [1340, 1687], [1340, 1688], [1340, 1725], [1341, 1387], [1341, 1393], [1341, 1401], [1341, 1409], [1341, 1413], [1341, 1424], [1341, 1428], [1341, 1432], [1341, 1434], [1341, 1436], [1341, 1439], [1341, 1462], [1341, 1480], [1341, 1485], [1341, 1688], [1341, 1726], [1342, 1384], [1342, 1387], [1342, 1445], [1342, 1506], [1342, 1585], [1342, 1634], [1342, 1695], [1342, 1699], [1342, 1729], [1343, 1388], [1343, 1389], [1343, 1390], [1343, 1391], [1343, 1392], [1343, 1393], [1343, 1394], [1343, 1463], [1343, 1530], [1343, 1586], [1343, 1601], [1343, 1608], [1344, 1347], [1344, 1394], [1344, 1403], [1344, 1433], [1344, 1470], [1344, 1472], [1344, 1587], [1344, 1689], [1344, 1727], [1345, 1348], [1345, 1376], [1345, 1386], [1345, 1394], [1345, 1416], [1345, 1423], [1345, 1435], [1345, 1531], [1345, 1546], [1345, 1564], [1345, 1608], [1345, 1635], [1345, 1636], [1345, 1637], [1345, 1638], [1345, 1639], [1346, 1349], [1346, 1394], [1346, 1415], [1346, 1437], [1346, 1499], [1346, 1503], [1346, 1509], [1346, 1588], [1346, 1635], [1346, 1690], [1347, 1351], [1347, 1394], [1347, 1430], [1347, 1440], [1347, 1446], [1347, 1557], [1347, 1558], [1347, 1590], [1347, 1637], [1347, 1728], [1348, 1389], [1348, 1394], [1348, 1416], [1348, 1464], [1348, 1483], [1348, 1512], [1348, 1689], [1348, 1690], [1349, 1390], [1349, 1394], [1349, 1423], [1349, 1436], [1349, 1454], [1349, 1478], [1349, 1480], [1349, 1489], [1350, 1394], [1350, 1414], [1350, 1420], [1350, 1433], [1350, 1435], [1350, 1437], [1350, 1438], [1350, 1440], [1350, 1465], [1350, 1486], [1350, 1501], [1350, 1725], [1351, 1391], [1351, 1394], [1351, 1458], [1351, 1509], [1351, 1533], [1351, 1591], [1351, 1639], [1351, 1734], [1352, 1355], [1352, 1378], [1352, 1457], [1352, 1458], [1352, 1466], [1352, 1565], [1352, 1585], [1352, 1691], [1352, 1729], [1353, 1395], [1353, 1396], [1353, 1397], [1353, 1398], [1353, 1399], [1353, 1400], [1353, 1401], [1353, 1467], [1353, 1517], [1353, 1534], [1353, 1595], [1354, 1356], [1354, 1403], [1354, 1411], [1354, 1479], [1354, 1541], [1355, 1358], [1355, 1395], [1355, 1415], [1355, 1508], [1355, 1510], [1355, 1545], [1355, 1547], [1355, 1592], [1355, 1693], [1355, 1731], [1356, 1397], [1356, 1430], [1356, 1563], [1356, 1565], [1356, 1567], [1357, 1398], [1357, 1411], [1357, 1445], [1357, 1449], [1357, 1469], [1357, 1691], [1357, 1692], [1357, 1693], [1357, 1694], [1357, 1695], [1357, 1696], [1358, 1399], [1358, 1436], [1358, 1529], [1358, 1593], [1358, 1695], [1359, 1481], [1359, 1495], [1359, 1696], [1359, 1733], [1360, 1364], [1360, 1388], [1360, 1470], [1360, 1546], [1360, 1568], [1360, 1591], [1360, 1617], [1360, 1640], [1360, 1697], [1360, 1734], [1361, 1402], [1361, 1403], [1361, 1404], [1361, 1405], [1361, 1406], [1361, 1407], [1361, 1408], [1361, 1409], [1361, 1471], [1361, 1518], [1361, 1536], [1361, 1594], [1361, 1611], [1362, 1366], [1362, 1385], [1362, 1400], [1362, 1418], [1362, 1427], [1362, 1537], [1362, 1595], [1362, 1611], [1362, 1640], [1362, 1641], [1362, 1642], [1363, 1367], [1363, 1386], [1363, 1415], [1363, 1431], [1363, 1472], [1363, 1538], [1363, 1547], [1363, 1596], [1363, 1641], [1363, 1666], [1363, 1698], [1363, 1735], [1364, 1368], [1364, 1402], [1364, 1421], [1364, 1473], [1364, 1539], [1364, 1597], [1364, 1617], [1364, 1618], [1364, 1666], [1364, 1701], [1364, 1736], [1365, 1369], [1365, 1426], [1365, 1540], [1365, 1598], [1365, 1671], [1365, 1673], [1366, 1404], [1366, 1445], [1366, 1448], [1366, 1474], [1366, 1642], [1366, 1697], [1366, 1698], [1366, 1699], [1366, 1700], [1366, 1740], [1367, 1405], [1367, 1418], [1367, 1431], [1367, 1436], [1367, 1444], [1367, 1449], [1367, 1458], [1367, 1461], [1367, 1475], [1367, 1485], [1367, 1528], [1367, 1568], [1367, 1569], [1367, 1673], [1367, 1691], [1367, 1699], [1367, 1737], [1367, 1747], [1368, 1406], [1368, 1438], [1368, 1476], [1368, 1599], [1368, 1701], [1369, 1407], [1369, 1427], [1369, 1452], [1369, 1454], [1369, 1495], [1370, 1443], [1370, 1477], [1370, 1700], [1370, 1738], [1371, 1375], [1371, 1392], [1371, 1409], [1371, 1415], [1371, 1423], [1371, 1432], [1371, 1478], [1371, 1515], [1371, 1538], [1371, 1552], [1371, 1601], [1371, 1615], [1372, 1376], [1372, 1393], [1372, 1421], [1372, 1434], [1372, 1520], [1372, 1527], [1372, 1539], [1372, 1544], [1372, 1550], [1372, 1555], [1372, 1602], [1372, 1608], [1372, 1619], [1373, 1400], [1373, 1426], [1373, 1431], [1373, 1528], [1373, 1532], [1373, 1540], [1373, 1542], [1373, 1553], [1373, 1570], [1373, 1571], [1373, 1611], [1373, 1620], [1373, 1656], [1374, 1377], [1374, 1430], [1374, 1479], [1374, 1529], [1374, 1545], [1374, 1556], [1374, 1572], [1374, 1612], [1375, 1410], [1375, 1436], [1375, 1453], [1375, 1458], [1375, 1460], [1375, 1511], [1375, 1557], [1375, 1562], [1375, 1565], [1376, 1423], [1376, 1434], [1376, 1438], [1376, 1456], [1376, 1461], [1376, 1480], [1376, 1501], [1376, 1512], [1376, 1546], [1376, 1554], [1376, 1558], [1376, 1566], [1376, 1568], [1377, 1412], [1377, 1432], [1377, 1443], [1377, 1449], [1377, 1481], [1377, 1533], [1377, 1543], [1377, 1548], [1377, 1563], [1378, 1415], [1378, 1416], [1378, 1417], [1378, 1418], [1378, 1419], [1378, 1420], [1378, 1603], [1378, 1643], [1378, 1648], [1379, 1383], [1379, 1414], [1379, 1421], [1379, 1435], [1379, 1544], [1379, 1614], [1379, 1648], [1380, 1384], [1380, 1426], [1380, 1698], [1380, 1704], [1380, 1739], [1381, 1385], [1381, 1409], [1381, 1430], [1381, 1434], [1381, 1545], [1381, 1555], [1381, 1558], [1381, 1567], [1381, 1570], [1381, 1578], [1381, 1596], [1381, 1604], [1381, 1611], [1381, 1615], [1381, 1632], [1381, 1644], [1381, 1665], [1381, 1676], [1381, 1678], [1381, 1693], [1381, 1704], [1382, 1386], [1382, 1618], [1382, 1632], [1382, 1641], [1382, 1671], [1382, 1701], [1382, 1702], [1383, 1416], [1383, 1438], [1383, 1483], [1383, 1546], [1383, 1605], [1383, 1678], [1383, 1697], [1384, 1417], [1384, 1445], [1384, 1446], [1384, 1484], [1384, 1547], [1384, 1606], [1384, 1645], [1384, 1693], [1384, 1698], [1385, 1418], [1385, 1434], [1385, 1485], [1385, 1539], [1385, 1548], [1385, 1607], [1385, 1701], [1385, 1740], [1386, 1419], [1386, 1435], [1386, 1455], [1386, 1461], [1386, 1486], [1386, 1564], [1386, 1702], [1386, 1739], [1386, 1740], [1387, 1461], [1387, 1487], [1387, 1518], [1387, 1526], [1387, 1549], [1388, 1421], [1388, 1422], [1388, 1423], [1388, 1424], [1388, 1550], [1388, 1601], [1388, 1646], [1388, 1661], [1389, 1392], [1389, 1414], [1389, 1437], [1389, 1588], [1389, 1608], [1389, 1615], [1389, 1618], [1389, 1648], [1389, 1661], [1389, 1672], [1389, 1676], [1389, 1703], [1390, 1393], [1390, 1436], [1390, 1478], [1390, 1569], [1390, 1572], [1390, 1609], [1390, 1742], [1391, 1422], [1391, 1458], [1391, 1488], [1391, 1539], [1391, 1622], [1391, 1623], [1391, 1650], [1392, 1423], [1392, 1437], [1392, 1451], [1392, 1453], [1392, 1463], [1392, 1464], [1392, 1489], [1392, 1501], [1392, 1564], [1392, 1741], [1392, 1742], [1393, 1424], [1393, 1450], [1393, 1453], [1393, 1480], [1393, 1522], [1393, 1562], [1394, 1464], [1394, 1480], [1394, 1486], [1394, 1490], [1394, 1531], [1394, 1551], [1395, 1399], [1395, 1415], [1395, 1491], [1395, 1545], [1395, 1552], [1395, 1625], [1395, 1628], [1395, 1704], [1395, 1743], [1396, 1425], [1396, 1426], [1396, 1427], [1396, 1428], [1396, 1553], [1396, 1595], [1396, 1654], [1397, 1430], [1397, 1563], [1397, 1614], [1398, 1400], [1398, 1598], [1398, 1620], [1398, 1704], [1398, 1705], [1398, 1706], [1399, 1401], [1399, 1425], [1399, 1436], [1399, 1493], [1399, 1528], [1399, 1529], [1399, 1573], [1399, 1617], [1399, 1651], [1400, 1427], [1400, 1443], [1400, 1448], [1400, 1467], [1400, 1469], [1400, 1495], [1400, 1540], [1400, 1743], [1400, 1744], [1401, 1428], [1401, 1496], [1401, 1607], [1401, 1652], [1401, 1706], [1402, 1406], [1402, 1421], [1402, 1497], [1402, 1555], [1402, 1618], [1402, 1627], [1402, 1631], [1402, 1632], [1402, 1707], [1402, 1745], [1403, 1429], [1403, 1430], [1403, 1431], [1403, 1432], [1403, 1433], [1403, 1556], [1403, 1596], [1403, 1610], [1403, 1653], [1403, 1664], [1404, 1408], [1404, 1598], [1404, 1611], [1404, 1654], [1404, 1664], [1404, 1707], [1404, 1708], [1404, 1709], [1405, 1409], [1405, 1436], [1405, 1498], [1405, 1557], [1405, 1573], [1405, 1612], [1405, 1616], [1405, 1708], [1406, 1429], [1406, 1438], [1406, 1499], [1406, 1558], [1406, 1613], [1406, 1632], [1406, 1635], [1406, 1655], [1406, 1709], [1406, 1720], [1407, 1559], [1407, 1656], [1407, 1729], [1407, 1731], [1408, 1431], [1408, 1444], [1408, 1446], [1408, 1452], [1408, 1471], [1408, 1474], [1408, 1540], [1408, 1745], [1409, 1432], [1409, 1443], [1409, 1446], [1409, 1455], [1409, 1460], [1409, 1485], [1409, 1500], [1409, 1518], [1409, 1538], [1409, 1545], [1409, 1560], [1409, 1590], [1409, 1606], [1409, 1627], [1409, 1720], [1409, 1731], [1409, 1743], [1410, 1413], [1410, 1436], [1410, 1562], [1410, 1593], [1410, 1609], [1410, 1612], [1410, 1615], [1410, 1668], [1410, 1715], [1411, 1585], [1411, 1606], [1411, 1616], [1411, 1620], [1411, 1630], [1411, 1664], [1412, 1563], [1412, 1591], [1412, 1607], [1412, 1617], [1412, 1665], [1413, 1434], [1413, 1456], [1413, 1526], [1413, 1545], [1413, 1548], [1413, 1622], [1413, 1624], [1413, 1625], [1414, 1435], [1414, 1455], [1414, 1483], [1414, 1501], [1414, 1531], [1414, 1544], [1414, 1618], [1414, 1623], [1414, 1626], [1414, 1627], [1415, 1436], [1415, 1437], [1415, 1615], [1415, 1657], [1415, 1710], [1415, 1714], [1416, 1420], [1416, 1438], [1416, 1648], [1416, 1667], [1416, 1702], [1416, 1714], [1417, 1502], [1417, 1616], [1417, 1658], [1418, 1583], [1418, 1585], [1418, 1617], [1418, 1621], [1418, 1623], [1418, 1641], [1418, 1659], [1418, 1664], [1418, 1668], [1418, 1686], [1418, 1699], [1418, 1711], [1418, 1734], [1418, 1736], [1419, 1503], [1419, 1635], [1419, 1699], [1419, 1708], [1419, 1729], [1420, 1437], [1420, 1564], [1420, 1618], [1420, 1660], [1420, 1702], [1420, 1736], [1420, 1745], [1421, 1438], [1421, 1619], [1421, 1648], [1421, 1681], [1421, 1712], [1421, 1718], [1422, 1504], [1422, 1621], [1422, 1622], [1422, 1713], [1423, 1635], [1423, 1661], [1423, 1668], [1423, 1681], [1423, 1714], [1423, 1719], [1423, 1720], [1423, 1730], [1423, 1734], [1424, 1505], [1424, 1562], [1424, 1590], [1424, 1591], [1424, 1715], [1425, 1428], [1425, 1436], [1425, 1506], [1425, 1565], [1425, 1616], [1425, 1617], [1425, 1634], [1425, 1691], [1426, 1439], [1426, 1620], [1426, 1654], [1426, 1662], [1426, 1682], [1426, 1717], [1427, 1630], [1427, 1656], [1427, 1682], [1428, 1439], [1428, 1508], [1428, 1567], [1428, 1606], [1428, 1607], [1428, 1632], [1429, 1433], [1429, 1438], [1429, 1509], [1429, 1568], [1429, 1621], [1429, 1635], [1429, 1639], [1429, 1697], [1429, 1699], [1429, 1746], [1430, 1440], [1430, 1663], [1430, 1683], [1430, 1716], [1431, 1656], [1431, 1664], [1431, 1683], [1431, 1717], [1431, 1727], [1431, 1746], [1431, 1747], [1432, 1510], [1432, 1622], [1432, 1665], [1432, 1747], [1433, 1440], [1433, 1472], [1433, 1473], [1433, 1569], [1433, 1623], [1433, 1666], [1433, 1699], [1434, 1624], [1434, 1652], [1434, 1665], [1434, 1668], [1434, 1670], [1434, 1688], [1434, 1718], [1435, 1660], [1435, 1666], [1435, 1669], [1435, 1681], [1435, 1687], [1436, 1511], [1436, 1668], [1437, 1503], [1437, 1684], [1437, 1714], [1438, 1512], [1438, 1669], [1438, 1714], [1438, 1718], [1438, 1725], [1438, 1748], [1439, 1570], [1439, 1625], [1439, 1632], [1439, 1670], [1439, 1671], [1439, 1743], [1439, 1744], [1440, 1446], [1440, 1447], [1440, 1513], [1440, 1572], [1440, 1627], [1440, 1676], [1440, 1745], [1441, 1466], [1441, 1467], [1441, 1468], [1441, 1469], [1442, 1470], [1442, 1471], [1442, 1472], [1442, 1473], [1442, 1474], [1442, 1475], [1442, 1476], [1442, 1477], [1442, 1516], [1443, 1460], [1443, 1469], [1443, 1477], [1443, 1516], [1443, 1517], [1443, 1518], [1443, 1519], [1443, 1520], [1444, 1446], [1444, 1461], [1444, 1477], [1444, 1518], [1444, 1558], [1444, 1561], [1444, 1676], [1445, 1448], [1445, 1477], [1446, 1472], [1446, 1477], [1446, 1486], [1446, 1500], [1446, 1721], [1446, 1726], [1446, 1747], [1447, 1473], [1447, 1477], [1447, 1520], [1447, 1676], [1447, 1722], [1448, 1477], [1448, 1495], [1448, 1696], [1449, 1477], [1449, 1481], [1449, 1485], [1449, 1488], [1449, 1504], [1449, 1696], [1449, 1723], [1449, 1747], [1450, 1478], [1450, 1479], [1450, 1480], [1450, 1481], [1450, 1482], [1450, 1521], [1450, 1522], [1451, 1464], [1451, 1475], [1451, 1482], [1451, 1498], [1451, 1503], [1452, 1469], [1452, 1482], [1452, 1516], [1452, 1540], [1452, 1559], [1453, 1482], [1453, 1501], [1453, 1726], [1453, 1742], [1454, 1482], [1454, 1486], [1454, 1489], [1454, 1495], [1454, 1503], [1455, 1483], [1455, 1484], [1455, 1485], [1455, 1486], [1455, 1487], [1455, 1523], [1455, 1531], [1455, 1538], [1456, 1480], [1456, 1487], [1456, 1505], [1456, 1521], [1456, 1531], [1457, 1459], [1457, 1487], [1457, 1508], [1457, 1524], [1457, 1585], [1458, 1460], [1458, 1475], [1458, 1481], [1458, 1487], [1458, 1510], [1458, 1516], [1458, 1525], [1458, 1538], [1458, 1557], [1459, 1487], [1459, 1528], [1459, 1724], [1459, 1737], [1460, 1487], [1460, 1529], [1460, 1585], [1461, 1487], [1461, 1501], [1461, 1724], [1461, 1725], [1461, 1726], [1462, 1487], [1462, 1496], [1462, 1500], [1462, 1505], [1462, 1508], [1462, 1510], [1462, 1511], [1462, 1726], [1463, 1488], [1463, 1489], [1463, 1490], [1463, 1530], [1464, 1490], [1464, 1727], [1464, 1728], [1465, 1490], [1465, 1501], [1465, 1512], [1465, 1513], [1466, 1559], [1466, 1729], [1467, 1491], [1467, 1492], [1467, 1493], [1467, 1494], [1467, 1495], [1467, 1496], [1467, 1534], [1467, 1537], [1467, 1542], [1468, 1492], [1468, 1593], [1468, 1732], [1469, 1729], [1469, 1730], [1469, 1731], [1469, 1732], [1469, 1733], [1470, 1473], [1470, 1533], [1470, 1734], [1471, 1497], [1471, 1498], [1471, 1499], [1471, 1500], [1471, 1536], [1472, 1475], [1472, 1486], [1472, 1538], [1472, 1735], [1473, 1476], [1473, 1497], [1473, 1539], [1473, 1564], [1473, 1639], [1473, 1736], [1473, 1740], [1474, 1734], [1474, 1735], [1474, 1736], [1474, 1737], [1474, 1738], [1475, 1498], [1475, 1511], [1475, 1729], [1475, 1737], [1476, 1499], [1476, 1512], [1476, 1740], [1477, 1738], [1478, 1489], [1478, 1500], [1478, 1510], [1479, 1481], [1479, 1557], [1479, 1560], [1480, 1512], [1481, 1510], [1481, 1543], [1483, 1512], [1483, 1546], [1483, 1720], [1483, 1734], [1484, 1502], [1484, 1547], [1484, 1731], [1484, 1735], [1485, 1548], [1485, 1591], [1485, 1671], [1485, 1740], [1486, 1503], [1487, 1549], [1488, 1504], [1488, 1569], [1488, 1672], [1490, 1551], [1491, 1493], [1491, 1552], [1491, 1560], [1491, 1570], [1491, 1575], [1491, 1596], [1491, 1743], [1492, 1494], [1492, 1597], [1492, 1706], [1493, 1496], [1493, 1506], [1493, 1511], [1493, 1514], [1494, 1507], [1494, 1512], [1494, 1554], [1494, 1599], [1494, 1706], [1496, 1508], [1496, 1548], [1497, 1499], [1497, 1555], [1497, 1564], [1497, 1572], [1497, 1612], [1497, 1637], [1497, 1741], [1497, 1745], [1498, 1500], [1498, 1511], [1498, 1514], [1498, 1557], [1498, 1728], [1498, 1741], [1499, 1509], [1499, 1512], [1499, 1558], [1499, 1728], [1500, 1510], [1500, 1547], [1500, 1560], [1500, 1572], [1501, 1551], [1501, 1561], [1501, 1564], [1501, 1569], [1501, 1571], [1501, 1572], [1502, 1703], [1503, 1737], [1504, 1568], [1504, 1591], [1504, 1674], [1504, 1692], [1505, 1533], [1506, 1508], [1506, 1511], [1506, 1565], [1506, 1629], [1506, 1695], [1506, 1708], [1506, 1729], [1507, 1512], [1507, 1566], [1507, 1599], [1507, 1709], [1508, 1547], [1508, 1548], [1508, 1567], [1508, 1628], [1509, 1512], [1509, 1568], [1509, 1689], [1509, 1734], [1509, 1737], [1509, 1746], [1510, 1747], [1512, 1688], [1512, 1748], [1513, 1572], [1513, 1742], [1514, 1515], [1514, 1573], [1515, 1575], [1515, 1719], [1516, 1536], [1516, 1537], [1516, 1538], [1516, 1539], [1516, 1540], [1517, 1529], [1517, 1541], [1517, 1545], [1517, 1563], [1517, 1577], [1518, 1578], [1518, 1631], [1518, 1729], [1519, 1520], [1519, 1550], [1519, 1579], [1519, 1584], [1519, 1590], [1519, 1730], [1520, 1539], [1520, 1580], [1520, 1584], [1520, 1627], [1520, 1631], [1520, 1732], [1521, 1541], [1521, 1542], [1521, 1543], [1522, 1550], [1522, 1562], [1523, 1544], [1523, 1545], [1523, 1546], [1523, 1547], [1523, 1548], [1523, 1549], [1523, 1581], [1524, 1528], [1524, 1549], [1524, 1553], [1524, 1567], [1524, 1578], [1525, 1529], [1525, 1543], [1525, 1549], [1525, 1556], [1525, 1562], [1525, 1573], [1525, 1582], [1525, 1591], [1526, 1549], [1526, 1581], [1526, 1582], [1526, 1583], [1526, 1584], [1526, 1585], [1527, 1544], [1527, 1549], [1527, 1576], [1528, 1549], [1528, 1570], [1528, 1573], [1528, 1578], [1528, 1583], [1528, 1691], [1529, 1545], [1529, 1549], [1529, 1562], [1529, 1584], [1530, 1550], [1530, 1551], [1530, 1586], [1531, 1551], [1531, 1586], [1531, 1587], [1531, 1588], [1531, 1589], [1531, 1590], [1531, 1591], [1532, 1551], [1532, 1571], [1532, 1589], [1532, 1727], [1533, 1551], [1533, 1591], [1534, 1552], [1534, 1553], [1534, 1554], [1535, 1543], [1535, 1592], [1535, 1593], [1536, 1555], [1536, 1556], [1536, 1557], [1536, 1558], [1536, 1559], [1536, 1560], [1536, 1561], [1536, 1594], [1537, 1548], [1537, 1594], [1537, 1595], [1537, 1596], [1537, 1597], [1537, 1598], [1537, 1599], [1538, 1587], [1538, 1596], [1538, 1623], [1538, 1734], [1539, 1555], [1539, 1582], [1539, 1597], [1539, 1623], [1540, 1598], [1541, 1562], [1541, 1563], [1541, 1600], [1541, 1601], [1541, 1602], [1542, 1574], [1542, 1594], [1543, 1563], [1544, 1546], [1544, 1600], [1544, 1608], [1545, 1548], [1545, 1560], [1545, 1594], [1545, 1601], [1545, 1604], [1545, 1612], [1545, 1622], [1546, 1605], [1547, 1606], [1548, 1607], [1548, 1628], [1552, 1574], [1553, 1565], [1553, 1566], [1553, 1567], [1553, 1611], [1554, 1566], [1554, 1652], [1555, 1558], [1555, 1576], [1556, 1568], [1556, 1569], [1556, 1610], [1557, 1560], [1557, 1612], [1558, 1561], [1558, 1568], [1558, 1588], [1558, 1613], [1558, 1676], [1561, 1569], [1562, 1601], [1563, 1582], [1563, 1622], [1564, 1618], [1565, 1567], [1565, 1582], [1565, 1585], [1566, 1655], [1567, 1570], [1567, 1592], [1568, 1569], [1568, 1588], [1568, 1591], [1568, 1621], [1568, 1665], [1569, 1572], [1569, 1623], [1570, 1575], [1570, 1625], [1570, 1693], [1570, 1747], [1571, 1626], [1571, 1724], [1572, 1627], [1572, 1741], [1573, 1575], [1573, 1603], [1573, 1616], [1573, 1629], [1574, 1610], [1575, 1592], [1575, 1615], [1575, 1672], [1575, 1674], [1576, 1579], [1576, 1586], [1576, 1631], [1577, 1584], [1577, 1600], [1577, 1604], [1578, 1603], [1578, 1611], [1578, 1617], [1578, 1691], [1579, 1580], [1579, 1692], [1580, 1597], [1580, 1619], [1580, 1694], [1581, 1603], [1581, 1604], [1581, 1605], [1581, 1606], [1581, 1607], [1582, 1584], [1582, 1610], [1582, 1617], [1582, 1622], [1583, 1585], [1583, 1620], [1583, 1625], [1583, 1633], [1583, 1651], [1584, 1604], [1584, 1651], [1585, 1606], [1585, 1634], [1586, 1608], [1586, 1609], [1587, 1590], [1587, 1610], [1587, 1623], [1588, 1615], [1588, 1635], [1589, 1620], [1589, 1626], [1589, 1636], [1589, 1689], [1590, 1591], [1590, 1627], [1590, 1637], [1591, 1639], [1592, 1593], [1592, 1615], [1592, 1692], [1594, 1610], [1594, 1611], [1594, 1612], [1594, 1613], [1595, 1607], [1595, 1614], [1595, 1617], [1596, 1615], [1596, 1641], [1596, 1697], [1597, 1599], [1597, 1619], [1597, 1650], [1598, 1620], [1598, 1628], [1598, 1629], [1599, 1613], [1599, 1639], [1599, 1697], [1600, 1614], [1601, 1615], [1601, 1622], [1602, 1619], [1602, 1624], [1603, 1615], [1603, 1616], [1603, 1617], [1603, 1618], [1603, 1643], [1604, 1607], [1604, 1624], [1604, 1631], [1604, 1644], [1605, 1640], [1606, 1616], [1606, 1641], [1606, 1645], [1606, 1743], [1607, 1617], [1607, 1624], [1608, 1631], [1608, 1646], [1608, 1647], [1608, 1648], [1608, 1649], [1608, 1650], [1610, 1621], [1610, 1622], [1610, 1623], [1610, 1653], [1611, 1653], [1611, 1654], [1611, 1655], [1611, 1656], [1612, 1637], [1612, 1647], [1613, 1621], [1613, 1637], [1613, 1644], [1613, 1655], [1614, 1624], [1615, 1657], [1615, 1661], [1615, 1665], [1616, 1658], [1616, 1691], [1617, 1653], [1617, 1659], [1617, 1665], [1618, 1660], [1620, 1625], [1620, 1626], [1620, 1662], [1620, 1664], [1621, 1623], [1621, 1640], [1622, 1665], [1623, 1627], [1623, 1666], [1623, 1734], [1625, 1628], [1625, 1644], [1625, 1670], [1625, 1706], [1626, 1685], [1627, 1631], [1627, 1686], [1627, 1720], [1628, 1629], [1628, 1645], [1628, 1671], [1628, 1706], [1629, 1643], [1629, 1658], [1629, 1673], [1629, 1708], [1630, 1634], [1630, 1638], [1630, 1645], [1630, 1649], [1630, 1656], [1630, 1658], [1630, 1662], [1630, 1675], [1630, 1683], [1631, 1650], [1631, 1676], [1632, 1641], [1632, 1644], [1632, 1654], [1632, 1657], [1632, 1679], [1632, 1686], [1632, 1701], [1632, 1716], [1632, 1718], [1633, 1634], [1633, 1662], [1633, 1670], [1633, 1685], [1633, 1705], [1633, 1711], [1634, 1645], [1634, 1691], [1634, 1711], [1635, 1657], [1635, 1690], [1636, 1638], [1636, 1662], [1636, 1739], [1637, 1639], [1637, 1647], [1637, 1663], [1637, 1715], [1638, 1649], [1638, 1728], [1639, 1650], [1639, 1697], [1639, 1715], [1639, 1740], [1640, 1646], [1640, 1697], [1641, 1657], [1641, 1664], [1641, 1698], [1641, 1743], [1642, 1654], [1642, 1667], [1642, 1701], [1642, 1711], [1643, 1657], [1643, 1658], [1643, 1659], [1643, 1660], [1644, 1663], [1645, 1658], [1646, 1661], [1647, 1650], [1647, 1663], [1648, 1667], [1649, 1703], [1649, 1741], [1651, 1652], [1651, 1668], [1652, 1706], [1653, 1663], [1653, 1664], [1653, 1665], [1653, 1666], [1654, 1667], [1655, 1669], [1655, 1677], [1655, 1678], [1655, 1709], [1656, 1691], [1656, 1693], [1657, 1668], [1657, 1710], [1658, 1708], [1659, 1697], [1659, 1711], [1660, 1707], [1661, 1674], [1661, 1678], [1661, 1692], [1661, 1697], [1661, 1712], [1661, 1713], [1661, 1714], [1661, 1715], [1662, 1670], [1663, 1716], [1664, 1689], [1664, 1716], [1664, 1717], [1665, 1713], [1666, 1711], [1667, 1718], [1668, 1681], [1668, 1686], [1669, 1687], [1670, 1704], [1670, 1705], [1671, 1673], [1671, 1696], [1671, 1721], [1671, 1743], [1672, 1674], [1672, 1696], [1672, 1703], [1673, 1691], [1673, 1696], [1673, 1724], [1674, 1692], [1674, 1696], [1674, 1719], [1675, 1696], [1675, 1703], [1676, 1678], [1676, 1700], [1676, 1745], [1677, 1679], [1677, 1687], [1677, 1700], [1677, 1711], [1677, 1724], [1678, 1697], [1678, 1700], [1678, 1713], [1678, 1720], [1679, 1698], [1679, 1700], [1679, 1702], [1679, 1710], [1679, 1721], [1679, 1745], [1680, 1700], [1680, 1701], [1680, 1705], [1680, 1711], [1680, 1713], [1680, 1716], [1680, 1723], [1681, 1712], [1681, 1722], [1681, 1732], [1681, 1736], [1681, 1745], [1681, 1748], [1682, 1724], [1682, 1739], [1683, 1708], [1683, 1716], [1683, 1723], [1683, 1728], [1683, 1741], [1684, 1702], [1684, 1714], [1684, 1717], [1684, 1725], [1685, 1724], [1686, 1716], [1686, 1720], [1686, 1722], [1687, 1711], [1687, 1725], [1688, 1706], [1688, 1715], [1688, 1718], [1688, 1726], [1688, 1740], [1689, 1727], [1689, 1734], [1689, 1735], [1690, 1710], [1690, 1746], [1691, 1693], [1691, 1729], [1692, 1694], [1692, 1730], [1693, 1695], [1693, 1704], [1693, 1710], [1693, 1731], [1693, 1747], [1694, 1712], [1694, 1732], [1696, 1733], [1697, 1734], [1698, 1699], [1698, 1702], [1698, 1710], [1698, 1717], [1698, 1735], [1699, 1708], [1699, 1711], [1699, 1717], [1699, 1723], [1699, 1725], [1699, 1737], [1699, 1740], [1700, 1738], [1701, 1711], [1701, 1718], [1701, 1740], [1702, 1725], [1704, 1710], [1704, 1743], [1705, 1744], [1707, 1709], [1707, 1712], [1707, 1745], [1719, 1730], [1719, 1733], [1720, 1722], [1720, 1734], [1720, 1738], [1721, 1735], [1721, 1738], [1722, 1736], [1722, 1738], [1723, 1738], [1723, 1740], [1723, 1744], [1724, 1739], [1726, 1747], [1727, 1728], [1728, 1741], [1729, 1731], [1730, 1732], [1731, 1743], [1734, 1736], [1735, 1737], [1736, 1745], [1746, 1748]], \"cross\": 3694}, {\"name\": \"C6_509x2+10\", \"n\": 1018, \"edges\": 7428, \"vtx\": [[0.0, 0.0], [0.06204688721150239, 0.15662730269593603], [0.10461977945516428, 0.1320478318988769], [0.16666666666666666, 0.024579470797059244], [0.16666666666666666, -0.024579470797059095], [0.10461977945516428, -0.1320478318988769], [0.06204688721150239, -0.15662730269593603], [-0.06204688721150239, -0.15662730269593603], [-0.10461977945516428, -0.1320478318988769], [-0.16666666666666666, -0.024579470797059095], [-0.16666666666666666, 0.024579470797059244], [-0.10461977945516428, 0.1320478318988769], [-0.06204688721150239, 0.15662730269593603], [0.0, 0.2640956637977538], [0.22871355387816905, -0.1320478318988769], [-0.22871355387816905, -0.1320478318988769], [0.0425728922436619, 0.2886751345948129], [0.27128644612183095, -0.10746836110181783], [0.22871355387816905, -0.1812067734929951], [-0.22871355387816905, -0.1812067734929951], [-0.27128644612183095, -0.10746836110181783], [-0.0425728922436619, 0.2886751345948129], [0.16666666666666666, 0.2886751345948129], [0.3333333333333333, 0.0], [0.16666666666666666, -0.2886751345948129], [-0.16666666666666666, -0.2886751345948129], [-0.3333333333333333, 0.0], [-0.16666666666666666, 0.2886751345948129], [0.06204688721150239, 0.42072296649368973], [0.3333333333333333, 0.2640956637977538], [0.39538022054483574, 0.15662730269593603], [0.39538022054483574, -0.15662730269593603], [0.3333333333333333, -0.2640956637977538], [0.06204688721150239, -0.42072296649368973], [-0.06204688721150239, -0.42072296649368973], [-0.3333333333333333, -0.2640956637977538], [-0.39538022054483574, -0.15662730269593603], [-0.39538022054483574, 0.15662730269593603], [-0.3333333333333333, 0.2640956637977538], [-0.06204688721150239, 0.42072296649368973], [0.22871355387816905, 0.39614349569663065], [0.3333333333333333, 0.31325460539187205], [0.43795311278849763, 0.1320478318988769], [0.4574271077563381, 0.0], [0.22871355387816905, -0.39614349569663065], [0.10461977945516428, -0.4453024372907488], [-0.10461977945516428, -0.4453024372907488], [-0.22871355387816905, -0.39614349569663065], [-0.4574271077563381, 0.0], [-0.43795311278849763, 0.1320478318988769], [-0.3333333333333333, 0.31325460539187205], [-0.22871355387816905, 0.39614349569663065], [0.0, 0.5773502691896258], [0.16666666666666666, 0.5527707983925666], [0.39538022054483574, 0.42072296649368973], [0.5, 0.2886751345948129], [0.5620468872115024, 0.1320478318988769], [0.5620468872115024, -0.1320478318988769], [0.5, -0.2886751345948129], [0.39538022054483574, -0.42072296649368973], [0.16666666666666666, -0.5527707983925666], [0.0, -0.5773502691896258], [-0.16666666666666666, -0.5527707983925666], [-0.39538022054483574, -0.42072296649368973], [-0.5, -0.2886751345948129], [-0.5620468872115024, -0.1320478318988769], [-0.5620468872115024, 0.1320478318988769], [-0.5, 0.2886751345948129], [-0.39538022054483574, 0.42072296649368973], [-0.16666666666666666, 0.5527707983925666], [0.16666666666666666, 0.6019297399866851], [0.6046197794551643, -0.15662730269593603], [0.43795311278849763, -0.4453024372907488], [-0.43795311278849763, -0.4453024372907488], [-0.6046197794551643, -0.15662730269593603], [-0.16666666666666666, 0.6019297399866851], [0.10461977945516428, 0.7093981010885027], [0.5620468872115024, 0.4453024372907488], [0.6666666666666666, 0.2640956637977538], [0.6666666666666666, -0.2640956637977538], [0.5620468872115024, -0.4453024372907488], [0.10461977945516428, -0.7093981010885027], [-0.10461977945516428, -0.7093981010885027], [-0.5620468872115024, -0.4453024372907488], [-0.6666666666666666, -0.2640956637977538], [-0.6666666666666666, 0.2640956637977538], [-0.5620468872115024, 0.4453024372907488], [-0.10461977945516428, 0.7093981010885027], [0.4574271077563381, 0.5773502691896258], [0.6046197794551643, 0.42072296649368973], [0.6666666666666666, 0.31325460539187205], [0.728713553878169, 0.10746836110181783], [0.27128644612183095, -0.6848186302914435], [0.06204688721150239, -0.7339775718855618], [-0.06204688721150239, -0.7339775718855618], [-0.27128644612183095, -0.6848186302914435], [-0.728713553878169, 0.10746836110181783], [-0.6666666666666666, 0.31325460539187205], [-0.6046197794551643, 0.42072296649368973], [-0.4574271077563381, 0.5773502691896258], [0.22871355387816905, 0.7093981010885027], [0.5, 0.5527707983925666], [0.728713553878169, 0.15662730269593603], [0.728713553878169, -0.15662730269593603], [0.5, -0.5527707983925666], [0.22871355387816905, -0.7093981010885027], [-0.22871355387816905, -0.7093981010885027], [-0.5, -0.5527707983925666], [-0.728713553878169, -0.15662730269593603], [-0.728713553878169, 0.15662730269593603], [-0.5, 0.5527707983925666], [-0.22871355387816905, 0.7093981010885027], [0.0, 0.7922869913932613], [0.6861406616345072, -0.39614349569663065], [-0.6861406616345072, -0.39614349569663065], [0.43795311278849763, 0.7093981010885027], [0.8333333333333334, 0.024579470797059244], [0.7907604410896715, -0.2640956637977538], [0.6240937744230047, -0.5527707983925666], [0.39538022054483574, -0.7339775718855618], [-0.39538022054483574, -0.7339775718855618], [-0.6240937744230047, -0.5527707983925666], [-0.7907604410896715, -0.2640956637977538], [-0.8333333333333334, 0.024579470797059244], [-0.43795311278849763, 0.7093981010885027], [0.0, 0.8414459329873795], [0.728713553878169, -0.42072296649368973], [-0.728713553878169, -0.42072296649368973], [0.6666666666666666, 0.5281913275955076], [0.7907604410896715, 0.31325460539187205], [0.12409377442300477, -0.8414459329873795], [-0.12409377442300477, -0.8414459329873795], [-0.7907604410896715, 0.31325460539187205], [-0.6666666666666666, 0.5281913275955076], [0.3333333333333333, 0.8414459329873795], [0.5620468872115024, 0.7093981010885027], [0.8953802205448357, 0.1320478318988769], [0.8953802205448357, -0.1320478318988769], [0.5620468872115024, -0.7093981010885027], [0.3333333333333333, -0.8414459329873795], [-0.3333333333333333, -0.8414459329873795], [-0.5620468872115024, -0.7093981010885027], [-0.8953802205448357, -0.1320478318988769], [-0.8953802205448357, 0.1320478318988769], [-0.5620468872115024, 0.7093981010885027], [-0.3333333333333333, 0.8414459329873795], [0.3333333333333333, 0.8906048745814976], [-0.3333333333333333, 0.8906048745814976], [0.06204688721150239, 0.9980732356833154], [0.22871355387816905, 0.9734937648862564], [0.5, 0.8660254037844386], [0.728713553878169, 0.6848186302914435], [0.9574271077563381, 0.2886751345948129], [1.0, 0.0], [0.9574271077563381, -0.2886751345948129], [0.8953802205448357, -0.4453024372907488], [0.8333333333333334, -0.5527707983925666], [0.728713553878169, -0.6848186302914435], [0.5, -0.8660254037844386], [0.22871355387816905, -0.9734937648862564], [-0.22871355387816905, -0.9734937648862564], [-0.5, -0.8660254037844386], [-0.728713553878169, -0.6848186302914435], [-0.8333333333333334, -0.5527707983925666], [-0.8953802205448357, -0.4453024372907488], [-0.9574271077563381, -0.2886751345948129], [-1.0, 0.0], [-0.9574271077563381, 0.2886751345948129], [-0.728713553878169, 0.6848186302914435], [-0.5, 0.8660254037844386], [-0.22871355387816905, 0.9734937648862564], [-0.06204688721150239, 0.9980732356833154], [0.39538022054483574, 0.9980732356833154], [0.6666666666666666, 0.8414459329873795], [1.0620468872115023, 0.15662730269593603], [1.0620468872115023, -0.15662730269593603], [0.6666666666666666, -0.8414459329873795], [0.39538022054483574, -0.9980732356833154], [-0.39538022054483574, -0.9980732356833154], [-0.6666666666666666, -0.8414459329873795], [-1.0620468872115023, -0.15662730269593603], [-1.0620468872115023, 0.15662730269593603], [-0.6666666666666666, 0.8414459329873795], [-0.39538022054483574, 0.9980732356833154], [0.9574271077563381, 0.5527707983925666], [0.0, -1.1055415967851332], [-0.9574271077563381, 0.5527707983925666], [0.16666666666666666, 1.1301210675821924], [0.8953802205448357, 0.7093981010885027], [1.0620468872115023, 0.42072296649368973], [1.0620468872115023, -0.42072296649368973], [0.8953802205448357, -0.7093981010885027], [0.16666666666666666, -1.1301210675821924], [-0.16666666666666666, -1.1301210675821924], [-0.8953802205448357, -0.7093981010885027], [-1.0620468872115023, -0.42072296649368973], [-1.0620468872115023, 0.42072296649368973], [-0.8953802205448357, 0.7093981010885027], [-0.16666666666666666, 1.1301210675821924], [0.0, 1.1547005383792517], [0.3333333333333333, 1.1055415967851332], [0.7907604410896715, 0.8414459329873795], [1.0, 0.5773502691896258], [1.1240937744230048, 0.2640956637977538], [1.1240937744230048, -0.2640956637977538], [1.0, -0.5773502691896258], [0.7907604410896715, -0.8414459329873795], [0.3333333333333333, -1.1055415967851332], [0.0, -1.1547005383792517], [-0.3333333333333333, -1.1055415967851332], [-0.7907604410896715, -0.8414459329873795], [-1.0, -0.5773502691896258], [-1.1240937744230048, -0.2640956637977538], [-1.1240937744230048, 0.2640956637977538], [-1.0, 0.5773502691896258], [-0.7907604410896715, 0.8414459329873795], [-0.3333333333333333, 1.1055415967851332], [0.8333333333333334, 0.8168664621903204], [0.2907604410896714, -1.1301210675821924], [-0.2907604410896714, -1.1301210675821924], [-0.8333333333333334, 0.8168664621903204], [1.228713553878169, -0.1812067734929951], [0.771286446121831, -0.9734937648862564], [-0.771286446121831, -0.9734937648862564], [-1.228713553878169, -0.1812067734929951], [0.10461977945516428, 1.2867483702781284], [0.27128644612183095, 1.2621688994810691], [0.9574271077563381, 0.8660254037844386], [1.228713553878169, 0.39614349569663065], [1.228713553878169, -0.39614349569663065], [1.1666666666666667, -0.5527707983925666], [1.0620468872115023, -0.7339775718855618], [0.9574271077563381, -0.8660254037844386], [0.27128644612183095, -1.2621688994810691], [-0.27128644612183095, -1.2621688994810691], [-0.9574271077563381, -0.8660254037844386], [-1.0620468872115023, -0.7339775718855618], [-1.1666666666666667, -0.5527707983925666], [-1.228713553878169, -0.39614349569663065], [-1.228713553878169, 0.39614349569663065], [-0.9574271077563381, 0.8660254037844386], [-0.27128644612183095, 1.2621688994810691], [-0.10461977945516428, 1.2867483702781284], [1.1861406616345072, 0.6848186302914435], [0.0, -1.369637260582887], [-1.1861406616345072, 0.6848186302914435], [0.5620468872115024, 1.2867483702781284], [1.3953802205448358, -0.15662730269593603], [1.2907604410896714, -0.5527707983925666], [1.1240937744230048, -0.8414459329873795], [0.8333333333333334, -1.1301210675821924], [-0.8333333333333334, -1.1301210675821924], [-1.1240937744230048, -0.8414459329873795], [-1.2907604410896714, -0.5527707983925666], [-1.3953802205448358, -0.15662730269593603], [-0.5620468872115024, 1.2867483702781284], [1.1666666666666667, 0.8168664621903204], [1.2907604410896714, 0.6019297399866851], [0.12409377442300477, -1.4187962021770053], [-0.12409377442300477, -1.4187962021770053], [-1.2907604410896714, 0.6019297399866851], [-1.1666666666666667, 0.8168664621903204], [0.728713553878169, 1.2621688994810691], [1.0620468872115023, 0.9980732356833154], [1.3953802205448358, 0.42072296649368973], [1.457427107756338, 0.0], [0.728713553878169, -1.2621688994810691], [0.3333333333333333, -1.4187962021770053], [-0.3333333333333333, -1.4187962021770053], [-0.728713553878169, -1.2621688994810691], [-1.457427107756338, 0.0], [-1.3953802205448358, 0.42072296649368973], [-1.0620468872115023, 0.9980732356833154], [-0.728713553878169, 1.2621688994810691], [0.5, 1.4433756729740645], [1.0, 1.1547005383792517], [1.5, 0.2886751345948129], [1.5, -0.2886751345948129], [1.0, -1.1547005383792517], [0.5, -1.4433756729740645], [-0.5, -1.4433756729740645], [-1.0, -1.1547005383792517], [-1.5, -0.2886751345948129], [-1.5, 0.2886751345948129], [-1.0, 1.1547005383792517], [-0.5, 1.4433756729740645], [0.22871355387816905, 1.550844034075882], [0.8953802205448357, 1.2867483702781284], [1.5620468872115023, 0.1320478318988769], [1.457427107756338, -0.5773502691896258], [1.228713553878169, -0.9734937648862564], [0.6666666666666666, -1.4187962021770053], [-0.6666666666666666, -1.4187962021770053], [-1.228713553878169, -0.9734937648862564], [-1.457427107756338, -0.5773502691896258], [-1.5620468872115023, 0.1320478318988769], [-0.8953802205448357, 1.2867483702781284], [-0.22871355387816905, 1.550844034075882], [0.8333333333333334, 1.394216731379946], [1.6240937744230048, 0.024579470797059244], [0.7907604410896715, -1.4187962021770053], [-0.7907604410896715, -1.4187962021770053], [-1.6240937744230048, 0.024579470797059244], [-0.8333333333333334, 1.394216731379946], [1.3953802205448358, 0.9980732356833154], [1.5620468872115023, 0.7093981010885027], [0.16666666666666666, -1.7074713367718184], [-0.16666666666666666, -1.7074713367718184], [-1.5620468872115023, 0.7093981010885027], [-1.3953802205448358, 0.9980732356833154], [0.0, 1.7320508075688772], [0.5, 1.6583123951777], [1.1861406616345072, 1.2621688994810691], [1.5, 0.8660254037844386], [1.6861406616345072, 0.39614349569663065], [1.6861406616345072, -0.39614349569663065], [1.5, -0.8660254037844386], [1.1861406616345072, -1.2621688994810691], [0.5, -1.6583123951777], [0.0, -1.7320508075688772], [-0.5, -1.6583123951777], [-1.1861406616345072, -1.2621688994810691], [-1.5, -0.8660254037844386], [-1.6861406616345072, -0.39614349569663065], [-1.6861406616345072, 0.39614349569663065], [-1.5, 0.8660254037844386], [-1.1861406616345072, 1.2621688994810691], [-0.5, 1.6583123951777], [0.6666666666666666, 1.682891865974759], [1.7907604410896714, -0.2640956637977538], [1.1240937744230048, -1.4187962021770053], [-1.1240937744230048, -1.4187962021770053], [-1.7907604410896714, -0.2640956637977538], [-0.6666666666666666, 1.682891865974759], [1.1666666666666667, 1.394216731379946], [1.7907604410896714, 0.31325460539187205], [0.6240937744230047, -1.7074713367718184], [-0.6240937744230047, -1.7074713367718184], [-1.7907604410896714, 0.31325460539187205], [-1.1666666666666667, 1.394216731379946], [0.27128644612183095, 1.839519168670695], [1.728713553878169, -0.6848186302914435], [-1.728713553878169, -0.6848186302914435], [-0.27128644612183095, 1.839519168670695], [1.6861406616345072, 0.9734937648862564], [0.0, -1.9469875297725128], [-1.6861406616345072, 0.9734937648862564], [0.16666666666666666, 1.971567000569572], [0.728713553878169, 1.839519168670695], [1.957427107756338, -0.2886751345948129], [1.6240937744230048, -1.1301210675821924], [1.228713553878169, -1.550844034075882], [-1.228713553878169, -1.550844034075882], [-1.6240937744230048, -1.1301210675821924], [-1.957427107756338, -0.2886751345948129], [-0.728713553878169, 1.839519168670695], [-0.16666666666666666, 1.971567000569572], [0.4574271077563381, 1.9469875297725128], [1.457427107756338, 1.369637260582887], [1.9148542155126762, 0.5773502691896258], [1.9148542155126762, -0.5773502691896258], [1.457427107756338, -1.369637260582887], [0.4574271077563381, -1.9469875297725128], [-0.4574271077563381, -1.9469875297725128], [-1.457427107756338, -1.369637260582887], [-1.9148542155126762, -0.5773502691896258], [-1.9148542155126762, 0.5773502691896258], [-1.457427107756338, 1.369637260582887], [-0.4574271077563381, 1.9469875297725128], [0.3333333333333333, -1.9961464713666308], [-0.3333333333333333, -1.9961464713666308], [0.0, 2.5243377989621383], [2.186140661634507, -1.2621688994810691], [-2.186140661634507, -1.2621688994810691], [0.1770055293069411, -0.23198022993801304], [-0.28940353695554516, -0.03730117002111536], [-0.28785071163819165, 0.04783045365361301], [0.008342214450717917, 0.4573510318873095], [0.40024871928679584, 0.22145094630551537], [0.3919065048360779, -0.23590008558179412], [-0.008342214450717917, -0.4573510318873095], [-0.40024871928679584, -0.22145094630551537], [-0.3919065048360779, 0.23590008558179412], [-0.2795084971874737, 0.5051814855409226], [0.29774575140626314, 0.49465220190842485], [0.5772542485937369, -0.010529283632497706], [0.2795084971874737, -0.5051814855409226], [-0.29774575140626314, -0.49465220190842485], [-0.5772542485937369, 0.010529283632497706], [-0.09416075342981471, 0.7305522874902189], [0.12074022209932211, 0.7266324318464378], [0.5855964630444549, 0.4468217482548118], [0.6896522562423409, 0.2587521163266307], [0.6797572164742696, -0.28373053923540725], [0.5689120341430188, -0.4678803155198072], [0.09416075342981471, -0.7305522874902189], [-0.6797572164742696, 0.28373053923540725], [-0.27116628273675575, 0.9625325174282319], [0.3060879658569811, 0.9520032337957343], [0.6979944706930589, 0.7161031482139402], [0.9775029678805328, 0.21092166267301768], [0.9691607534298147, -0.24642936921429182], [0.6714150020235516, -0.7410815711227169], [0.27116628273675575, -0.9625325174282319], [-0.3060879658569811, -0.9520032337957343], [-0.6979944706930589, -0.7161031482139402], [-0.9775029678805328, -0.21092166267301768], [-0.9691607534298147, 0.24642936921429182], [-0.6714150020235516, 0.7410815711227169], [-0.5590169943749475, 1.0103629710818451], [1.1545084971874737, -0.02105856726499541], [-0.5954915028125263, -0.9893044038168497], [-0.08581853897909686, 1.1879033193775286], [0.9858451823312505, 0.6682726945603271], [1.089900975529137, 0.4802030626321461], [0.9608185389790969, -0.7037804011016013], [-0.9858451823312505, -0.6682726945603271], [-0.37366925061728845, 1.2357337730311415], [0.41848597350558525, 1.2212846337548626], [0.8833422144507179, 0.9414739501632368], [1.266906504836078, 0.24822283269413298], [1.2570114650680064, -0.29425982286790486], [0.8484205313304926, -0.9730618010607297], [0.37366925061728845, -1.2357337730311415], [-0.41848597350558525, -1.2212846337548626], [-0.8833422144507179, -0.9414739501632368], [-1.266906504836078, -0.24822283269413298], [-1.2570114650680064, 0.29425982286790486], [-0.8484205313304926, 0.9730618010607297], [-0.6630727875728337, 1.1984326030100263], [0.7063366851437769, 1.1734541801012497], [1.3694094727166106, -0.02497842290877643], [0.6630727875728337, -1.1984326030100263], [-0.7063366851437769, -1.1734541801012497], [-1.3694094727166106, 0.02497842290877643], [-0.2628240682860378, 1.4198835493155417], [0.31443018030769904, 1.409354265683044], [1.0982431899798548, 0.9375540945194555], [1.3777516871673285, 0.4323726089785331], [1.3610672582658925, -0.48232945479608597], [1.0633215068596293, -0.9769816567045108], [0.2628240682860378, -1.4198835493155417], [-0.31443018030769904, -1.409354265683044], [-1.0982431899798548, -0.9375540945194555], [-1.3777516871673285, -0.4323726089785331], [-1.3610672582658925, 0.48232945479608597], [-1.0633215068596293, 0.9769816567045108], [0.02657946866950733, 1.4571847193366567], [1.2752487192867958, 0.7055738645814424], [1.2486692506172885, -0.7516108547552143], [-0.02657946866950733, -1.4571847193366567], [-1.2752487192867958, -0.7055738645814424], [-1.2486692506172885, 0.7516108547552143], [0.6038337172632442, 1.446655435704159], [1.5547572164742693, 0.20039237904051999], [-0.6038337172632442, -1.446655435704159], [-0.9957402220993221, -1.2107553501223651], [-1.5547572164742693, -0.20039237904051999], [-0.9509234992110254, 1.2462630566636392], [-0.3653270361665706, 1.693084804918451], [0.4268281879563032, 1.6786356656421724], [1.2835909337375138, 1.162924896468752], [1.6671552241228738, 0.46967377899964835], [1.6489179699040846, -0.5301599084496991], [1.2403270361665706, -1.208961886642524], [0.3653270361665706, -1.693084804918451], [-0.4268281879563032, -1.6786356656421724], [-1.2835909337375138, -1.162924896468752], [-1.6671552241228738, -0.46967377899964835], [-1.6489179699040846, 0.5301599084496991], [-1.2403270361665706, 1.208961886642524], [0.03492168312022524, 1.9145357512239665], [1.6405757554533664, -0.9875109403370085], [-0.03492168312022524, -1.9145357512239665], [-1.6405757554533664, 0.9875109403370085], [-0.9425812847603074, 1.7036140885509485], [1.00408243655004, 1.6681063820096744], [1.9466637213103475, -0.03550770654127413], [0.9425812847603074, -1.7036140885509485], [-1.00408243655004, -1.6681063820096744], [-1.9466637213103475, 0.03550770654127413], [-0.5423325654735115, 1.9250650348564637], [0.6121759317139622, 1.9040064675914685], [1.3959889413861177, 1.4322062964278803], [1.9550059357610656, 0.42184332534603536], [1.9383215068596293, -0.49285873842858363], [1.3428300040471033, -1.4821631422454338], [0.5423325654735115, -1.9250650348564637], [-0.6121759317139622, -1.9040064675914685], [-1.3959889413861177, -1.4322062964278803], [-1.9550059357610656, -0.42184332534603536], [-1.9383215068596293, 0.49285873842858363], [-1.3428300040471033, 1.4821631422454338], [-0.6448355333540443, 2.1982662904593737], [2.226172218497821, -0.5406891920821968], [-0.7245739393625664, -2.1732878675505973], [-2.226172218497821, 0.5406891920821968], [-0.2445868140672484, 2.419717236764889], [0.3326674345264884, 2.4091879531323914], [1.9732431899798548, 1.4216770127953826], [2.2527516871673288, 0.9164955272544603], [2.2178300040471033, -0.9980402239695061], [1.9200842526408402, -1.4926924258779308], [0.2445868140672484, -2.419717236764889], [-0.3326674345264884, -2.4091879531323914], [-1.9732431899798548, -1.4216770127953826], [-2.2527516871673288, -0.9164955272544603], [-2.2178300040471033, 0.9980402239695061], [-1.9200842526408402, 1.4926924258779308], [1.0, 0.0], [1.0620468872115023, 0.15662730269593603], [1.1046197794551642, 0.1320478318988769], [1.1666666666666667, 0.024579470797059244], [1.1666666666666667, -0.024579470797059095], [1.1046197794551642, -0.1320478318988769], [1.0620468872115023, -0.15662730269593603], [0.9379531127884976, -0.15662730269593603], [0.8953802205448357, -0.1320478318988769], [0.8333333333333334, -0.024579470797059095], [0.8333333333333334, 0.024579470797059244], [0.8953802205448357, 0.1320478318988769], [0.9379531127884976, 0.15662730269593603], [1.0, 0.2640956637977538], [1.228713553878169, -0.1320478318988769], [0.771286446121831, -0.1320478318988769], [1.042572892243662, 0.2886751345948129], [1.271286446121831, -0.10746836110181783], [1.228713553878169, -0.1812067734929951], [0.771286446121831, -0.1812067734929951], [0.728713553878169, -0.10746836110181783], [0.9574271077563381, 0.2886751345948129], [1.1666666666666667, 0.2886751345948129], [1.3333333333333333, 0.0], [1.1666666666666667, -0.2886751345948129], [0.8333333333333334, -0.2886751345948129], [0.6666666666666667, 0.0], [0.8333333333333334, 0.2886751345948129], [1.0620468872115023, 0.42072296649368973], [1.3333333333333333, 0.2640956637977538], [1.3953802205448358, 0.15662730269593603], [1.3953802205448358, -0.15662730269593603], [1.3333333333333333, -0.2640956637977538], [1.0620468872115023, -0.42072296649368973], [0.9379531127884976, -0.42072296649368973], [0.6666666666666667, -0.2640956637977538], [0.6046197794551642, -0.15662730269593603], [0.6046197794551642, 0.15662730269593603], [0.6666666666666667, 0.2640956637977538], [0.9379531127884976, 0.42072296649368973], [1.228713553878169, 0.39614349569663065], [1.3333333333333333, 0.31325460539187205], [1.4379531127884977, 0.1320478318988769], [1.457427107756338, 0.0], [1.228713553878169, -0.39614349569663065], [1.1046197794551642, -0.4453024372907488], [0.8953802205448357, -0.4453024372907488], [0.771286446121831, -0.39614349569663065], [0.5425728922436619, 0.0], [0.5620468872115023, 0.1320478318988769], [0.6666666666666667, 0.31325460539187205], [0.771286446121831, 0.39614349569663065], [1.0, 0.5773502691896258], [1.1666666666666667, 0.5527707983925666], [1.3953802205448358, 0.42072296649368973], [1.5, 0.2886751345948129], [1.5620468872115025, 0.1320478318988769], [1.5620468872115025, -0.1320478318988769], [1.5, -0.2886751345948129], [1.3953802205448358, -0.42072296649368973], [1.1666666666666667, -0.5527707983925666], [1.0, -0.5773502691896258], [0.8333333333333334, -0.5527707983925666], [0.6046197794551642, -0.42072296649368973], [0.5, -0.2886751345948129], [0.4379531127884976, -0.1320478318988769], [0.4379531127884976, 0.1320478318988769], [0.5, 0.2886751345948129], [0.6046197794551642, 0.42072296649368973], [0.8333333333333334, 0.5527707983925666], [1.1666666666666667, 0.6019297399866851], [1.6046197794551644, -0.15662730269593603], [1.4379531127884977, -0.4453024372907488], [0.5620468872115023, -0.4453024372907488], [0.3953802205448357, -0.15662730269593603], [0.8333333333333334, 0.6019297399866851], [1.1046197794551642, 0.7093981010885027], [1.5620468872115025, 0.4453024372907488], [1.6666666666666665, 0.2640956637977538], [1.6666666666666665, -0.2640956637977538], [1.5620468872115025, -0.4453024372907488], [1.1046197794551642, -0.7093981010885027], [0.8953802205448357, -0.7093981010885027], [0.4379531127884976, -0.4453024372907488], [0.33333333333333337, -0.2640956637977538], [0.33333333333333337, 0.2640956637977538], [0.4379531127884976, 0.4453024372907488], [0.8953802205448357, 0.7093981010885027], [1.457427107756338, 0.5773502691896258], [1.6046197794551644, 0.42072296649368973], [1.6666666666666665, 0.31325460539187205], [1.728713553878169, 0.10746836110181783], [1.271286446121831, -0.6848186302914435], [1.0620468872115023, -0.7339775718855618], [0.9379531127884976, -0.7339775718855618], [0.728713553878169, -0.6848186302914435], [0.27128644612183095, 0.10746836110181783], [0.33333333333333337, 0.31325460539187205], [0.3953802205448357, 0.42072296649368973], [0.5425728922436619, 0.5773502691896258], [1.228713553878169, 0.7093981010885027], [1.5, 0.5527707983925666], [1.728713553878169, 0.15662730269593603], [1.728713553878169, -0.15662730269593603], [1.5, -0.5527707983925666], [1.228713553878169, -0.7093981010885027], [0.771286446121831, -0.7093981010885027], [0.5, -0.5527707983925666], [0.27128644612183095, -0.15662730269593603], [0.27128644612183095, 0.15662730269593603], [0.5, 0.5527707983925666], [0.771286446121831, 0.7093981010885027], [1.0, 0.7922869913932613], [1.6861406616345072, -0.39614349569663065], [0.31385933836549285, -0.39614349569663065], [1.4379531127884977, 0.7093981010885027], [1.8333333333333335, 0.024579470797059244], [1.7907604410896716, -0.2640956637977538], [1.6240937744230046, -0.5527707983925666], [1.3953802205448358, -0.7339775718855618], [0.6046197794551642, -0.7339775718855618], [0.37590622557699527, -0.5527707983925666], [0.20923955891032853, -0.2640956637977538], [0.16666666666666663, 0.024579470797059244], [0.5620468872115023, 0.7093981010885027], [1.0, 0.8414459329873795], [1.728713553878169, -0.42072296649368973], [0.27128644612183095, -0.42072296649368973], [1.6666666666666665, 0.5281913275955076], [1.7907604410896716, 0.31325460539187205], [1.1240937744230048, -0.8414459329873795], [0.8759062255769953, -0.8414459329873795], [0.20923955891032853, 0.31325460539187205], [0.33333333333333337, 0.5281913275955076], [1.3333333333333333, 0.8414459329873795], [1.5620468872115025, 0.7093981010885027], [1.8953802205448356, 0.1320478318988769], [1.8953802205448356, -0.1320478318988769], [1.5620468872115025, -0.7093981010885027], [1.3333333333333333, -0.8414459329873795], [0.6666666666666667, -0.8414459329873795], [0.4379531127884976, -0.7093981010885027], [0.10461977945516432, -0.1320478318988769], [0.10461977945516432, 0.1320478318988769], [0.4379531127884976, 0.7093981010885027], [0.6666666666666667, 0.8414459329873795], [1.3333333333333333, 0.8906048745814976], [0.6666666666666667, 0.8906048745814976], [1.0620468872115023, 0.9980732356833154], [1.228713553878169, 0.9734937648862564], [1.5, 0.8660254037844386], [1.728713553878169, 0.6848186302914435], [1.957427107756338, 0.2886751345948129], [2.0, 0.0], [1.957427107756338, -0.2886751345948129], [1.8953802205448356, -0.4453024372907488], [1.8333333333333335, -0.5527707983925666], [1.728713553878169, -0.6848186302914435], [1.5, -0.8660254037844386], [1.228713553878169, -0.9734937648862564], [0.771286446121831, -0.9734937648862564], [0.5, -0.8660254037844386], [0.27128644612183095, -0.6848186302914435], [0.16666666666666663, -0.5527707983925666], [0.10461977945516432, -0.4453024372907488], [0.0425728922436619, -0.2886751345948129], [0.0, 0.0], [0.0425728922436619, 0.2886751345948129], [0.27128644612183095, 0.6848186302914435], [0.5, 0.8660254037844386], [0.771286446121831, 0.9734937648862564], [0.9379531127884976, 0.9980732356833154], [1.3953802205448358, 0.9980732356833154], [1.6666666666666665, 0.8414459329873795], [2.062046887211502, 0.15662730269593603], [2.062046887211502, -0.15662730269593603], [1.6666666666666665, -0.8414459329873795], [1.3953802205448358, -0.9980732356833154], [0.6046197794551642, -0.9980732356833154], [0.33333333333333337, -0.8414459329873795], [-0.06204688721150231, -0.15662730269593603], [-0.06204688721150231, 0.15662730269593603], [0.33333333333333337, 0.8414459329873795], [0.6046197794551642, 0.9980732356833154], [1.957427107756338, 0.5527707983925666], [1.0, -1.1055415967851332], [0.0425728922436619, 0.5527707983925666], [1.1666666666666667, 1.1301210675821924], [1.8953802205448356, 0.7093981010885027], [2.062046887211502, 0.42072296649368973], [2.062046887211502, -0.42072296649368973], [1.8953802205448356, -0.7093981010885027], [1.1666666666666667, -1.1301210675821924], [0.8333333333333334, -1.1301210675821924], [0.10461977945516432, -0.7093981010885027], [-0.06204688721150231, -0.42072296649368973], [-0.06204688721150231, 0.42072296649368973], [0.10461977945516432, 0.7093981010885027], [0.8333333333333334, 1.1301210675821924], [1.0, 1.1547005383792517], [1.3333333333333333, 1.1055415967851332], [1.7907604410896716, 0.8414459329873795], [2.0, 0.5773502691896258], [2.124093774423005, 0.2640956637977538], [2.124093774423005, -0.2640956637977538], [2.0, -0.5773502691896258], [1.7907604410896716, -0.8414459329873795], [1.3333333333333333, -1.1055415967851332], [1.0, -1.1547005383792517], [0.6666666666666667, -1.1055415967851332], [0.20923955891032853, -0.8414459329873795], [0.0, -0.5773502691896258], [-0.12409377442300484, -0.2640956637977538], [-0.12409377442300484, 0.2640956637977538], [0.0, 0.5773502691896258], [0.20923955891032853, 0.8414459329873795], [0.6666666666666667, 1.1055415967851332], [1.8333333333333335, 0.8168664621903204], [1.2907604410896714, -1.1301210675821924], [0.7092395589103286, -1.1301210675821924], [0.16666666666666663, 0.8168664621903204], [2.228713553878169, -0.1812067734929951], [1.771286446121831, -0.9734937648862564], [0.22871355387816905, -0.9734937648862564], [-0.22871355387816905, -0.1812067734929951], [1.1046197794551642, 1.2867483702781284], [1.271286446121831, 1.2621688994810691], [1.957427107756338, 0.8660254037844386], [2.228713553878169, 0.39614349569663065], [2.228713553878169, -0.39614349569663065], [2.166666666666667, -0.5527707983925666], [2.062046887211502, -0.7339775718855618], [1.957427107756338, -0.8660254037844386], [1.271286446121831, -1.2621688994810691], [0.728713553878169, -1.2621688994810691], [0.0425728922436619, -0.8660254037844386], [-0.06204688721150231, -0.7339775718855618], [-0.16666666666666674, -0.5527707983925666], [-0.22871355387816905, -0.39614349569663065], [-0.22871355387816905, 0.39614349569663065], [0.0425728922436619, 0.8660254037844386], [0.728713553878169, 1.2621688994810691], [0.8953802205448357, 1.2867483702781284], [2.186140661634507, 0.6848186302914435], [1.0, -1.369637260582887], [-0.18614066163450715, 0.6848186302914435], [1.5620468872115025, 1.2867483702781284], [2.395380220544836, -0.15662730269593603], [2.290760441089671, -0.5527707983925666], [2.124093774423005, -0.8414459329873795], [1.8333333333333335, -1.1301210675821924], [0.16666666666666663, -1.1301210675821924], [-0.12409377442300484, -0.8414459329873795], [-0.29076044108967136, -0.5527707983925666], [-0.3953802205448358, -0.15662730269593603], [0.4379531127884976, 1.2867483702781284], [2.166666666666667, 0.8168664621903204], [2.290760441089671, 0.6019297399866851], [1.1240937744230048, -1.4187962021770053], [0.8759062255769953, -1.4187962021770053], [-0.29076044108967136, 0.6019297399866851], [-0.16666666666666674, 0.8168664621903204], [1.728713553878169, 1.2621688994810691], [2.062046887211502, 0.9980732356833154], [2.395380220544836, 0.42072296649368973], [2.457427107756338, 0.0], [1.728713553878169, -1.2621688994810691], [1.3333333333333333, -1.4187962021770053], [0.6666666666666667, -1.4187962021770053], [0.27128644612183095, -1.2621688994810691], [-0.4574271077563381, 0.0], [-0.3953802205448358, 0.42072296649368973], [-0.06204688721150231, 0.9980732356833154], [0.27128644612183095, 1.2621688994810691], [1.5, 1.4433756729740645], [2.0, 1.1547005383792517], [2.5, 0.2886751345948129], [2.5, -0.2886751345948129], [2.0, -1.1547005383792517], [1.5, -1.4433756729740645], [0.5, -1.4433756729740645], [0.0, -1.1547005383792517], [-0.5, -0.2886751345948129], [-0.5, 0.2886751345948129], [0.0, 1.1547005383792517], [0.5, 1.4433756729740645], [1.228713553878169, 1.550844034075882], [1.8953802205448356, 1.2867483702781284], [2.562046887211502, 0.1320478318988769], [2.457427107756338, -0.5773502691896258], [2.228713553878169, -0.9734937648862564], [1.6666666666666665, -1.4187962021770053], [0.33333333333333337, -1.4187962021770053], [-0.22871355387816905, -0.9734937648862564], [-0.4574271077563381, -0.5773502691896258], [-0.5620468872115023, 0.1320478318988769], [0.10461977945516432, 1.2867483702781284], [0.771286446121831, 1.550844034075882], [1.8333333333333335, 1.394216731379946], [2.624093774423005, 0.024579470797059244], [1.7907604410896716, -1.4187962021770053], [0.20923955891032853, -1.4187962021770053], [-0.6240937744230048, 0.024579470797059244], [0.16666666666666663, 1.394216731379946], [2.395380220544836, 0.9980732356833154], [2.562046887211502, 0.7093981010885027], [1.1666666666666667, -1.7074713367718184], [0.8333333333333334, -1.7074713367718184], [-0.5620468872115023, 0.7093981010885027], [-0.3953802205448358, 0.9980732356833154], [1.0, 1.7320508075688772], [1.5, 1.6583123951777], [2.186140661634507, 1.2621688994810691], [2.5, 0.8660254037844386], [2.686140661634507, 0.39614349569663065], [2.686140661634507, -0.39614349569663065], [2.5, -0.8660254037844386], [2.186140661634507, -1.2621688994810691], [1.5, -1.6583123951777], [1.0, -1.7320508075688772], [0.5, -1.6583123951777], [-0.18614066163450715, -1.2621688994810691], [-0.5, -0.8660254037844386], [-0.6861406616345072, -0.39614349569663065], [-0.6861406616345072, 0.39614349569663065], [-0.5, 0.8660254037844386], [-0.18614066163450715, 1.2621688994810691], [0.5, 1.6583123951777], [1.6666666666666665, 1.682891865974759], [2.790760441089671, -0.2640956637977538], [2.124093774423005, -1.4187962021770053], [-0.12409377442300484, -1.4187962021770053], [-0.7907604410896714, -0.2640956637977538], [0.33333333333333337, 1.682891865974759], [2.166666666666667, 1.394216731379946], [2.790760441089671, 0.31325460539187205], [1.6240937744230046, -1.7074713367718184], [0.37590622557699527, -1.7074713367718184], [-0.7907604410896714, 0.31325460539187205], [-0.16666666666666674, 1.394216731379946], [1.271286446121831, 1.839519168670695], [2.728713553878169, -0.6848186302914435], [-0.728713553878169, -0.6848186302914435], [0.728713553878169, 1.839519168670695], [2.686140661634507, 0.9734937648862564], [1.0, -1.9469875297725128], [-0.6861406616345072, 0.9734937648862564], [1.1666666666666667, 1.971567000569572], [1.728713553878169, 1.839519168670695], [2.957427107756338, -0.2886751345948129], [2.624093774423005, -1.1301210675821924], [2.228713553878169, -1.550844034075882], [-0.22871355387816905, -1.550844034075882], [-0.6240937744230048, -1.1301210675821924], [-0.9574271077563381, -0.2886751345948129], [0.27128644612183095, 1.839519168670695], [0.8333333333333334, 1.971567000569572], [1.457427107756338, 1.9469875297725128], [2.457427107756338, 1.369637260582887], [2.914854215512676, 0.5773502691896258], [2.914854215512676, -0.5773502691896258], [2.457427107756338, -1.369637260582887], [1.457427107756338, -1.9469875297725128], [0.5425728922436619, -1.9469875297725128], [-0.4574271077563381, -1.369637260582887], [-0.9148542155126762, -0.5773502691896258], [-0.9148542155126762, 0.5773502691896258], [-0.4574271077563381, 1.369637260582887], [0.5425728922436619, 1.9469875297725128], [1.3333333333333333, -1.9961464713666308], [0.6666666666666667, -1.9961464713666308], [1.0, 2.5243377989621383], [3.186140661634507, -1.2621688994810691], [-1.1861406616345072, -1.2621688994810691], [1.177005529306941, -0.23198022993801304], [0.7105964630444548, -0.03730117002111536], [0.7121492883618084, 0.04783045365361301], [1.008342214450718, 0.4573510318873095], [1.4002487192867958, 0.22145094630551537], [1.391906504836078, -0.23590008558179412], [0.9916577855492821, -0.4573510318873095], [0.5997512807132042, -0.22145094630551537], [0.6080934951639221, 0.23590008558179412], [0.7204915028125263, 0.5051814855409226], [1.2977457514062631, 0.49465220190842485], [1.5772542485937369, -0.010529283632497706], [1.2795084971874737, -0.5051814855409226], [0.7022542485937369, -0.49465220190842485], [0.42274575140626314, 0.010529283632497706], [0.9058392465701853, 0.7305522874902189], [1.1207402220993221, 0.7266324318464378], [1.585596463044455, 0.4468217482548118], [1.6896522562423408, 0.2587521163266307], [1.6797572164742696, -0.28373053923540725], [1.5689120341430187, -0.4678803155198072], [1.0941607534298148, -0.7305522874902189], [0.32024278352573043, 0.28373053923540725], [0.7288337172632442, 0.9625325174282319], [1.306087965856981, 0.9520032337957343], [1.6979944706930588, 0.7161031482139402], [1.977502967880533, 0.21092166267301768], [1.9691607534298146, -0.24642936921429182], [1.6714150020235516, -0.7410815711227169], [1.2711662827367558, -0.9625325174282319], [0.6939120341430189, -0.9520032337957343], [0.30200552930694113, -0.7161031482139402], [0.022497032119467186, -0.21092166267301768], [0.03083924657018533, 0.24642936921429182], [0.32858499797644836, 0.7410815711227169], [0.44098300562505255, 1.0103629710818451], [2.1545084971874737, -0.02105856726499541], [0.4045084971874737, -0.9893044038168497], [0.9141814610209031, 1.1879033193775286], [1.9858451823312504, 0.6682726945603271], [2.0899009755291367, 0.4802030626321461], [1.9608185389790969, -0.7037804011016013], [0.014154817668749486, -0.6682726945603271], [0.6263307493827115, 1.2357337730311415], [1.4184859735055853, 1.2212846337548626], [1.883342214450718, 0.9414739501632368], [2.266906504836078, 0.24822283269413298], [2.2570114650680067, -0.29425982286790486], [1.8484205313304924, -0.9730618010607297], [1.3736692506172885, -1.2357337730311415], [0.5815140264944147, -1.2212846337548626], [0.11665778554928208, -0.9414739501632368], [-0.2669065048360779, -0.24822283269413298], [-0.25701146506800643, 0.29425982286790486], [0.15157946866950744, 0.9730618010607297], [0.3369272124271663, 1.1984326030100263], [1.706336685143777, 1.1734541801012497], [2.369409472716611, -0.02497842290877643], [1.6630727875728337, -1.1984326030100263], [0.2936633148562231, -1.1734541801012497], [-0.3694094727166106, 0.02497842290877643], [0.7371759317139621, 1.4198835493155417], [1.314430180307699, 1.409354265683044], [2.098243189979855, 0.9375540945194555], [2.3777516871673283, 0.4323726089785331], [2.3610672582658925, -0.48232945479608597], [2.063321506859629, -0.9769816567045108], [1.2628240682860379, -1.4198835493155417], [0.685569819692301, -1.409354265683044], [-0.09824318997985482, -0.9375540945194555], [-0.37775168716732854, -0.4323726089785331], [-0.3610672582658925, 0.48232945479608597], [-0.06332150685962934, 0.9769816567045108], [1.0265794686695073, 1.4571847193366567], [2.275248719286796, 0.7055738645814424], [2.2486692506172883, -0.7516108547552143], [0.9734205313304927, -1.4571847193366567], [-0.27524871928679584, -0.7055738645814424], [-0.2486692506172885, 0.7516108547552143], [1.6038337172632442, 1.446655435704159], [2.554757216474269, 0.20039237904051999], [0.3961662827367558, -1.446655435704159], [0.004259777900677886, -1.2107553501223651], [-0.5547572164742693, -0.20039237904051999], [0.04907650078897463, 1.2462630566636392], [0.6346729638334294, 1.693084804918451], [1.4268281879563032, 1.6786356656421724], [2.2835909337375138, 1.162924896468752], [2.6671552241228738, 0.46967377899964835], [2.6489179699040846, -0.5301599084496991], [2.240327036166571, -1.208961886642524], [1.3653270361665706, -1.693084804918451], [0.5731718120436968, -1.6786356656421724], [-0.28359093373751376, -1.162924896468752], [-0.6671552241228738, -0.46967377899964835], [-0.6489179699040846, 0.5301599084496991], [-0.24032703616657058, 1.208961886642524], [1.0349216831202253, 1.9145357512239665], [2.6405757554533666, -0.9875109403370085], [0.9650783168797747, -1.9145357512239665], [-0.6405757554533664, 0.9875109403370085], [0.05741871523969255, 1.7036140885509485], [2.00408243655004, 1.6681063820096744], [2.9466637213103475, -0.03550770654127413], [1.9425812847603074, -1.7036140885509485], [-0.004082436550040036, -1.6681063820096744], [-0.9466637213103475, 0.03550770654127413], [0.4576674345264885, 1.9250650348564637], [1.6121759317139621, 1.9040064675914685], [2.3959889413861175, 1.4322062964278803], [2.955005935761066, 0.42184332534603536], [2.938321506859629, -0.49285873842858363], [2.3428300040471033, -1.4821631422454338], [1.5423325654735116, -1.9250650348564637], [0.38782406828603777, -1.9040064675914685], [-0.39598894138611773, -1.4322062964278803], [-0.9550059357610656, -0.42184332534603536], [-0.9383215068596293, 0.49285873842858363], [-0.3428300040471033, 1.4821631422454338], [0.3551644666459557, 2.1982662904593737], [3.226172218497821, -0.5406891920821968], [0.2754260606374336, -2.1732878675505973], [-1.2261722184978212, 0.5406891920821968], [0.7554131859327516, 2.419717236764889], [1.3326674345264884, 2.4091879531323914], [2.973243189979855, 1.4216770127953826], [3.2527516871673288, 0.9164955272544603], [3.2178300040471033, -0.9980402239695061], [2.92008425264084, -1.4926924258779308], [1.2445868140672485, -2.419717236764889], [0.6673325654735116, -2.4091879531323914], [-0.9732431899798548, -1.4216770127953826], [-1.2527516871673288, -0.9164955272544603], [-1.2178300040471033, 0.9980402239695061], [-0.9200842526408402, 1.4926924258779308]], \"edge\": [[1, 149], [1, 150], [1, 151], [1, 152], [1, 153], [1, 154], [1, 155], [1, 156], [1, 157], [1, 158], [1, 159], [1, 160], [1, 161], [1, 162], [1, 163], [1, 164], [1, 165], [1, 166], [1, 167], [1, 168], [1, 169], [1, 170], [1, 171], [1, 172], [1, 398], [1, 399], [1, 400], [1, 401], [1, 402], [1, 403], [1, 404], [1, 405], [1, 406], [1, 407], [1, 408], [1, 409], [1, 510], [1, 531], [1, 556], [1, 572], [1, 579], [1, 605], [1, 671], [1, 679], [1, 733], [1, 782], [1, 803], [1, 832], [1, 835], [1, 852], [1, 864], [2, 131], [2, 139], [2, 143], [2, 155], [2, 175], [2, 183], [2, 189], [2, 199], [2, 200], [2, 202], [2, 511], [2, 597], [2, 794], [3, 121], [3, 122], [3, 128], [3, 132], [3, 140], [3, 144], [3, 157], [3, 169], [3, 176], [3, 184], [3, 188], [3, 190], [3, 201], [3, 203], [3, 218], [3, 512], [3, 516], [3, 538], [3, 544], [3, 562], [3, 572], [3, 630], [3, 689], [3, 693], [3, 745], [3, 762], [3, 819], [4, 123], [4, 124], [4, 128], [4, 133], [4, 141], [4, 145], [4, 148], [4, 160], [4, 172], [4, 173], [4, 177], [4, 189], [4, 191], [4, 203], [4, 205], [4, 513], [4, 543], [4, 562], [4, 597], [4, 650], [4, 657], [4, 733], [4, 782], [4, 818], [4, 842], [4, 848], [5, 134], [5, 142], [5, 146], [5, 150], [5, 174], [5, 178], [5, 190], [5, 192], [5, 204], [5, 206], [5, 514], [5, 538], [5, 571], [5, 592], [5, 655], [6, 133], [6, 135], [6, 143], [6, 163], [6, 175], [6, 179], [6, 191], [6, 193], [6, 206], [6, 208], [6, 511], [6, 515], [6, 543], [6, 549], [6, 571], [6, 579], [6, 688], [6, 692], [6, 750], [6, 761], [6, 848], [6, 852], [7, 125], [7, 126], [7, 134], [7, 136], [7, 144], [7, 153], [7, 165], [7, 176], [7, 180], [7, 192], [7, 194], [7, 207], [7, 209], [7, 219], [7, 516], [7, 531], [7, 592], [7, 634], [7, 771], [7, 791], [8, 116], [8, 126], [8, 129], [8, 137], [8, 145], [8, 156], [8, 168], [8, 177], [8, 181], [8, 193], [8, 195], [8, 209], [8, 211], [8, 220], [8, 517], [8, 521], [8, 537], [8, 556], [8, 561], [8, 616], [8, 650], [8, 654], [8, 730], [8, 761], [8, 791], [8, 818], [9, 130], [9, 138], [9, 146], [9, 158], [9, 178], [9, 182], [9, 194], [9, 196], [9, 210], [9, 212], [9, 518], [9, 605], [10, 129], [10, 135], [10, 139], [10, 171], [10, 179], [10, 183], [10, 195], [10, 197], [10, 212], [10, 214], [10, 519], [10, 692], [11, 117], [11, 118], [11, 127], [11, 130], [11, 136], [11, 140], [11, 147], [11, 149], [11, 161], [11, 180], [11, 184], [11, 196], [11, 198], [11, 213], [11, 215], [11, 520], [11, 634], [11, 689], [11, 803], [11, 819], [12, 119], [12, 120], [12, 127], [12, 131], [12, 137], [12, 141], [12, 152], [12, 164], [12, 173], [12, 181], [12, 197], [12, 199], [12, 215], [12, 217], [12, 221], [12, 521], [13, 132], [13, 138], [13, 142], [13, 166], [13, 174], [13, 182], [13, 188], [13, 198], [13, 200], [13, 216], [13, 518], [13, 522], [13, 535], [13, 557], [13, 585], [13, 621], [13, 651], [13, 655], [13, 762], [13, 794], [13, 864], [14, 94], [14, 95], [14, 106], [14, 107], [14, 127], [14, 128], [14, 185], [14, 187], [14, 189], [14, 198], [14, 218], [14, 221], [14, 523], [14, 535], [14, 597], [14, 746], [15, 98], [15, 99], [15, 110], [15, 111], [15, 126], [15, 128], [15, 185], [15, 186], [15, 190], [15, 193], [15, 219], [15, 524], [15, 538], [15, 761], [16, 90], [16, 91], [16, 102], [16, 103], [16, 126], [16, 127], [16, 186], [16, 187], [16, 194], [16, 197], [16, 220], [16, 525], [16, 560], [16, 578], [16, 620], [17, 82], [17, 93], [17, 115], [17, 123], [17, 154], [17, 168], [17, 203], [17, 216], [17, 226], [17, 227], [17, 510], [17, 517], [17, 526], [17, 557], [17, 562], [17, 672], [17, 680], [17, 704], [17, 783], [17, 804], [17, 806], [17, 833], [17, 836], [17, 842], [17, 856], [17, 876], [18, 87], [18, 100], [18, 115], [18, 122], [18, 147], [18, 151], [18, 161], [18, 203], [18, 208], [18, 223], [18, 230], [18, 231], [18, 527], [18, 554], [18, 562], [18, 570], [18, 670], [18, 679], [18, 750], [18, 803], [18, 833], [19, 86], [19, 97], [19, 113], [19, 152], [19, 162], [19, 204], [19, 209], [19, 222], [19, 232], [19, 233], [19, 528], [19, 603], [19, 730], [19, 791], [19, 832], [20, 79], [20, 92], [20, 113], [20, 159], [20, 169], [20, 209], [20, 214], [20, 225], [20, 236], [20, 237], [20, 529], [20, 548], [20, 630], [20, 671], [20, 678], [20, 771], [20, 791], [21, 78], [21, 89], [21, 114], [21, 119], [21, 148], [21, 160], [21, 170], [21, 210], [21, 215], [21, 224], [21, 238], [21, 239], [21, 530], [21, 733], [21, 835], [22, 83], [22, 96], [22, 114], [22, 118], [22, 153], [22, 167], [22, 202], [22, 215], [22, 242], [22, 243], [22, 531], [23, 82], [23, 85], [23, 106], [23, 109], [23, 176], [23, 183], [23, 226], [23, 516], [23, 532], [23, 635], [23, 704], [23, 806], [24, 84], [24, 87], [24, 108], [24, 111], [24, 173], [24, 178], [24, 231], [24, 533], [24, 563], [24, 570], [25, 77], [25, 86], [25, 101], [25, 110], [25, 175], [25, 180], [25, 232], [25, 511], [25, 523], [25, 534], [25, 603], [25, 707], [26, 79], [26, 88], [26, 103], [26, 112], [26, 177], [26, 182], [26, 237], [26, 535], [26, 548], [26, 650], [27, 78], [27, 81], [27, 102], [27, 105], [27, 179], [27, 184], [27, 238], [27, 536], [27, 583], [27, 617], [27, 620], [27, 819], [28, 80], [28, 83], [28, 104], [28, 107], [28, 174], [28, 181], [28, 243], [28, 537], [28, 545], [28, 655], [29, 62], [29, 63], [29, 74], [29, 81], [29, 85], [29, 118], [29, 138], [29, 144], [29, 190], [29, 198], [29, 228], [29, 247], [29, 518], [29, 538], [29, 583], [29, 721], [29, 747], [29, 849], [30, 64], [30, 65], [30, 82], [30, 86], [30, 120], [30, 139], [30, 145], [30, 191], [30, 199], [30, 222], [30, 227], [30, 257], [30, 528], [30, 534], [30, 539], [30, 543], [30, 610], [30, 704], [30, 708], [30, 783], [30, 792], [30, 818], [31, 65], [31, 66], [31, 83], [31, 87], [31, 125], [31, 140], [31, 146], [31, 188], [31, 192], [31, 230], [31, 258], [31, 540], [31, 554], [31, 592], [31, 610], [31, 689], [31, 792], [32, 67], [32, 68], [32, 75], [32, 84], [32, 88], [32, 135], [32, 141], [32, 189], [32, 193], [32, 229], [32, 248], [32, 250], [32, 541], [32, 550], [32, 597], [32, 615], [32, 640], [32, 692], [32, 730], [32, 761], [32, 793], [32, 805], [33, 68], [33, 69], [33, 76], [33, 77], [33, 85], [33, 122], [33, 136], [33, 142], [33, 190], [33, 194], [33, 234], [33, 249], [33, 251], [33, 532], [33, 538], [33, 542], [33, 585], [33, 615], [33, 634], [33, 703], [33, 707], [33, 779], [33, 781], [33, 793], [33, 812], [34, 53], [34, 70], [34, 78], [34, 86], [34, 124], [34, 137], [34, 143], [34, 191], [34, 195], [34, 224], [34, 233], [34, 259], [34, 521], [34, 543], [34, 724], [35, 53], [35, 54], [35, 79], [35, 87], [35, 117], [35, 138], [35, 144], [35, 192], [35, 196], [35, 223], [35, 236], [35, 260], [35, 518], [35, 520], [35, 544], [35, 548], [35, 592], [35, 596], [35, 670], [35, 719], [35, 724], [35, 841], [36, 55], [36, 56], [36, 71], [36, 80], [36, 88], [36, 119], [36, 139], [36, 145], [36, 193], [36, 197], [36, 235], [36, 252], [36, 254], [36, 545], [36, 577], [36, 608], [36, 761], [36, 818], [37, 56], [37, 57], [37, 72], [37, 77], [37, 81], [37, 140], [37, 146], [37, 194], [37, 198], [37, 240], [37, 253], [37, 255], [37, 546], [37, 559], [37, 577], [37, 583], [37, 643], [37, 651], [37, 689], [37, 707], [37, 771], [37, 863], [38, 58], [38, 59], [38, 78], [38, 82], [38, 116], [38, 135], [38, 141], [38, 195], [38, 199], [38, 239], [38, 261], [38, 547], [38, 574], [38, 654], [38, 692], [38, 704], [39, 59], [39, 60], [39, 79], [39, 83], [39, 121], [39, 136], [39, 142], [39, 188], [39, 196], [39, 225], [39, 242], [39, 262], [39, 548], [39, 574], [39, 634], [40, 61], [40, 62], [40, 73], [40, 80], [40, 84], [40, 123], [40, 137], [40, 143], [40, 189], [40, 197], [40, 241], [40, 256], [40, 519], [40, 521], [40, 525], [40, 545], [40, 549], [40, 593], [40, 597], [40, 673], [40, 680], [40, 721], [40, 726], [40, 763], [40, 765], [40, 813], [40, 842], [41, 62], [41, 65], [41, 75], [41, 97], [41, 155], [41, 169], [41, 176], [41, 183], [41, 229], [41, 242], [41, 244], [41, 263], [41, 516], [41, 550], [41, 721], [41, 751], [41, 792], [41, 813], [42, 63], [42, 66], [42, 93], [42, 98], [42, 157], [42, 170], [42, 184], [42, 226], [42, 247], [42, 258], [42, 264], [42, 524], [42, 551], [42, 572], [42, 658], [42, 672], [42, 747], [42, 806], [42, 812], [42, 819], [42, 835], [43, 64], [43, 67], [43, 95], [43, 100], [43, 159], [43, 172], [43, 177], [43, 231], [43, 248], [43, 257], [43, 265], [43, 541], [43, 552], [43, 564], [43, 570], [43, 604], [43, 650], [43, 671], [43, 681], [43, 720], [43, 726], [43, 746], [43, 763], [43, 782], [43, 805], [44, 65], [44, 68], [44, 74], [44, 96], [44, 150], [44, 160], [44, 173], [44, 178], [44, 228], [44, 233], [44, 244], [44, 249], [44, 266], [44, 553], [44, 733], [44, 792], [44, 793], [45, 53], [45, 68], [45, 71], [45, 89], [45, 153], [45, 163], [45, 175], [45, 180], [45, 230], [45, 235], [45, 245], [45, 250], [45, 267], [45, 511], [45, 531], [45, 554], [45, 640], [45, 724], [45, 744], [45, 793], [45, 852], [46, 54], [46, 69], [46, 90], [46, 97], [46, 154], [46, 165], [46, 176], [46, 232], [46, 251], [46, 260], [46, 268], [46, 510], [46, 516], [46, 555], [46, 578], [46, 603], [46, 643], [46, 696], [46, 703], [46, 781], [46, 802], [46, 841], [46, 863], [47, 55], [47, 70], [47, 92], [47, 99], [47, 156], [47, 167], [47, 181], [47, 237], [47, 252], [47, 259], [47, 269], [47, 556], [47, 608], [48, 53], [48, 56], [48, 76], [48, 100], [48, 158], [48, 168], [48, 177], [48, 182], [48, 234], [48, 239], [48, 245], [48, 253], [48, 270], [48, 530], [48, 547], [48, 557], [48, 577], [48, 605], [48, 650], [48, 724], [48, 770], [48, 779], [48, 874], [49, 56], [49, 59], [49, 73], [49, 93], [49, 161], [49, 171], [49, 179], [49, 184], [49, 236], [49, 241], [49, 246], [49, 254], [49, 271], [49, 558], [49, 574], [49, 577], [49, 593], [49, 596], [49, 631], [49, 672], [49, 678], [49, 745], [49, 750], [49, 803], [49, 819], [49, 856], [50, 57], [50, 60], [50, 89], [50, 94], [50, 149], [50, 162], [50, 180], [50, 238], [50, 255], [50, 262], [50, 272], [50, 559], [50, 832], [51, 58], [51, 61], [51, 91], [51, 96], [51, 151], [51, 164], [51, 173], [51, 243], [51, 256], [51, 261], [51, 273], [51, 560], [51, 673], [51, 679], [52, 59], [52, 62], [52, 72], [52, 92], [52, 152], [52, 166], [52, 174], [52, 181], [52, 227], [52, 240], [52, 246], [52, 274], [52, 546], [52, 561], [52, 574], [52, 655], [52, 721], [52, 783], [52, 804], [52, 849], [52, 864], [52, 877], [53, 59], [53, 65], [53, 117], [53, 124], [53, 137], [53, 144], [53, 153], [53, 168], [53, 203], [53, 215], [53, 228], [53, 241], [53, 263], [53, 274], [53, 275], [53, 286], [53, 287], [53, 298], [53, 520], [53, 521], [53, 530], [53, 531], [53, 562], [53, 574], [53, 705], [53, 708], [53, 748], [53, 751], [53, 792], [53, 795], [54, 60], [54, 66], [54, 97], [54, 138], [54, 154], [54, 204], [54, 216], [54, 264], [54, 287], [54, 510], [54, 518], [54, 563], [54, 658], [54, 674], [54, 705], [55, 61], [55, 67], [55, 99], [55, 119], [55, 145], [55, 156], [55, 170], [55, 205], [55, 217], [55, 243], [55, 265], [55, 288], [55, 524], [55, 556], [55, 564], [55, 659], [55, 673], [55, 752], [55, 805], [55, 813], [55, 818], [55, 835], [56, 62], [56, 68], [56, 93], [56, 100], [56, 139], [56, 146], [56, 158], [56, 171], [56, 200], [56, 206], [56, 227], [56, 230], [56, 247], [56, 248], [56, 263], [56, 266], [56, 276], [56, 277], [56, 541], [56, 542], [56, 553], [56, 554], [56, 565], [56, 571], [56, 598], [56, 605], [56, 644], [56, 651], [56, 659], [56, 672], [56, 709], [56, 721], [56, 748], [56, 751], [56, 764], [56, 765], [56, 780], [56, 783], [56, 793], [56, 794], [57, 63], [57, 69], [57, 94], [57, 140], [57, 149], [57, 159], [57, 201], [57, 207], [57, 232], [57, 249], [57, 264], [57, 289], [57, 566], [57, 569], [57, 598], [57, 603], [57, 658], [57, 671], [57, 689], [57, 747], [57, 764], [57, 771], [57, 781], [58, 64], [58, 70], [58, 96], [58, 135], [58, 151], [58, 202], [58, 208], [58, 265], [58, 290], [58, 564], [58, 567], [58, 679], [58, 692], [59, 65], [59, 116], [59, 121], [59, 136], [59, 141], [59, 152], [59, 161], [59, 203], [59, 209], [59, 229], [59, 234], [59, 266], [59, 267], [59, 278], [59, 279], [59, 290], [59, 291], [59, 539], [59, 540], [59, 550], [59, 553], [59, 562], [59, 568], [59, 629], [59, 634], [59, 649], [59, 654], [59, 669], [59, 678], [59, 718], [59, 724], [59, 744], [59, 749], [59, 779], [59, 780], [59, 791], [59, 792], [59, 803], [59, 804], [60, 66], [60, 89], [60, 142], [60, 162], [60, 204], [60, 210], [60, 268], [60, 291], [60, 569], [60, 669], [60, 802], [60, 832], [61, 67], [61, 91], [61, 123], [61, 137], [61, 154], [61, 164], [61, 205], [61, 211], [61, 231], [61, 250], [61, 269], [61, 292], [61, 510], [61, 521], [61, 560], [61, 570], [61, 608], [61, 640], [61, 695], [61, 706], [61, 778], [61, 805], [61, 842], [62, 68], [62, 92], [62, 97], [62, 138], [62, 143], [62, 155], [62, 166], [62, 206], [62, 212], [62, 233], [62, 236], [62, 251], [62, 252], [62, 267], [62, 270], [62, 280], [62, 281], [62, 518], [62, 519], [62, 571], [62, 577], [62, 703], [62, 706], [62, 744], [62, 749], [62, 790], [62, 793], [62, 862], [62, 864], [63, 69], [63, 98], [63, 118], [63, 144], [63, 157], [63, 167], [63, 207], [63, 213], [63, 238], [63, 253], [63, 268], [63, 293], [63, 530], [63, 536], [63, 559], [63, 572], [63, 607], [63, 719], [63, 781], [63, 802], [63, 862], [64, 70], [64, 100], [64, 139], [64, 159], [64, 208], [64, 214], [64, 269], [64, 294], [64, 573], [64, 576], [64, 595], [64, 671], [65, 120], [65, 125], [65, 140], [65, 145], [65, 160], [65, 169], [65, 209], [65, 215], [65, 235], [65, 240], [65, 270], [65, 271], [65, 282], [65, 283], [65, 294], [65, 295], [65, 574], [65, 595], [65, 689], [65, 724], [65, 733], [65, 791], [65, 818], [66, 93], [66, 146], [66, 170], [66, 210], [66, 216], [66, 272], [66, 295], [66, 575], [66, 607], [66, 672], [66, 835], [66, 863], [67, 95], [67, 141], [67, 162], [67, 172], [67, 211], [67, 217], [67, 237], [67, 254], [67, 273], [67, 296], [67, 576], [67, 584], [67, 608], [67, 637], [67, 673], [67, 678], [67, 730], [67, 746], [67, 782], [67, 832], [68, 89], [68, 96], [68, 135], [68, 142], [68, 150], [68, 163], [68, 200], [68, 212], [68, 239], [68, 242], [68, 255], [68, 256], [68, 271], [68, 274], [68, 284], [68, 285], [68, 577], [68, 584], [68, 594], [68, 692], [68, 721], [68, 794], [68, 852], [69, 90], [69, 122], [69, 136], [69, 151], [69, 165], [69, 201], [69, 213], [69, 226], [69, 272], [69, 297], [69, 559], [69, 575], [69, 578], [69, 594], [69, 634], [69, 674], [69, 679], [69, 747], [69, 804], [69, 806], [69, 849], [70, 92], [70, 143], [70, 167], [70, 202], [70, 214], [70, 273], [70, 298], [70, 536], [70, 579], [70, 726], [71, 80], [71, 110], [71, 133], [71, 175], [71, 256], [71, 276], [71, 288], [71, 511], [71, 545], [71, 580], [71, 709], [71, 752], [71, 848], [72, 77], [72, 107], [72, 132], [72, 174], [72, 218], [72, 251], [72, 277], [72, 289], [72, 565], [72, 566], [72, 581], [72, 586], [72, 590], [72, 649], [72, 655], [72, 703], [72, 707], [72, 749], [72, 762], [72, 764], [73, 84], [73, 102], [73, 129], [73, 179], [73, 220], [73, 248], [73, 280], [73, 292], [73, 533], [73, 541], [73, 549], [73, 582], [73, 620], [73, 629], [73, 702], [73, 706], [73, 778], [73, 780], [73, 790], [73, 811], [74, 81], [74, 111], [74, 134], [74, 178], [74, 219], [74, 255], [74, 281], [74, 293], [74, 583], [75, 88], [75, 106], [75, 131], [75, 183], [75, 221], [75, 252], [75, 284], [75, 296], [75, 584], [76, 85], [76, 103], [76, 130], [76, 182], [76, 247], [76, 285], [76, 297], [76, 585], [76, 594], [76, 748], [77, 110], [77, 117], [77, 190], [77, 198], [77, 264], [77, 274], [77, 276], [77, 299], [77, 520], [77, 522], [77, 523], [77, 538], [77, 546], [77, 586], [77, 594], [77, 658], [77, 675], [77, 709], [77, 722], [77, 764], [77, 812], [77, 843], [78, 105], [78, 119], [78, 148], [78, 191], [78, 199], [78, 266], [78, 275], [78, 305], [78, 543], [78, 553], [78, 587], [78, 617], [78, 682], [78, 795], [79, 112], [79, 188], [79, 192], [79, 263], [79, 278], [79, 306], [79, 568], [79, 569], [79, 588], [79, 592], [79, 630], [79, 645], [79, 651], [79, 697], [79, 705], [79, 734], [79, 751], [79, 771], [80, 107], [80, 116], [80, 189], [80, 193], [80, 265], [80, 267], [80, 277], [80, 300], [80, 564], [80, 565], [80, 580], [80, 589], [80, 597], [80, 628], [80, 648], [80, 654], [80, 702], [80, 706], [80, 744], [80, 761], [80, 763], [81, 102], [81, 121], [81, 190], [81, 194], [81, 266], [81, 268], [81, 280], [81, 301], [81, 538], [81, 553], [81, 590], [81, 620], [81, 643], [81, 687], [81, 728], [81, 764], [81, 790], [81, 802], [82, 109], [82, 123], [82, 191], [82, 195], [82, 270], [82, 279], [82, 307], [82, 517], [82, 519], [82, 543], [82, 547], [82, 591], [82, 595], [82, 677], [82, 718], [82, 723], [82, 842], [83, 104], [83, 118], [83, 192], [83, 196], [83, 267], [83, 282], [83, 308], [83, 592], [83, 744], [84, 111], [84, 120], [84, 193], [84, 197], [84, 269], [84, 271], [84, 281], [84, 302], [84, 584], [84, 593], [84, 606], [84, 706], [84, 761], [85, 106], [85, 125], [85, 194], [85, 198], [85, 270], [85, 272], [85, 284], [85, 303], [85, 594], [86, 101], [86, 195], [86, 199], [86, 274], [86, 283], [86, 309], [86, 595], [86, 730], [87, 108], [87, 122], [87, 147], [87, 188], [87, 196], [87, 271], [87, 286], [87, 310], [87, 596], [87, 705], [88, 103], [88, 124], [88, 189], [88, 197], [88, 263], [88, 273], [88, 285], [88, 304], [88, 584], [88, 597], [88, 751], [89, 114], [89, 155], [89, 170], [89, 242], [89, 287], [89, 313], [89, 598], [89, 793], [89, 835], [90, 157], [90, 171], [90, 226], [90, 289], [90, 299], [90, 306], [90, 314], [90, 542], [90, 555], [90, 566], [90, 572], [90, 599], [90, 631], [90, 645], [90, 660], [90, 674], [90, 710], [90, 722], [90, 735], [90, 781], [90, 806], [91, 158], [91, 172], [91, 231], [91, 288], [91, 300], [91, 305], [91, 314], [91, 567], [91, 570], [91, 600], [91, 605], [91, 660], [91, 673], [91, 682], [91, 752], [91, 765], [91, 770], [91, 782], [92, 113], [92, 150], [92, 159], [92, 233], [92, 290], [92, 315], [92, 590], [92, 601], [92, 659], [92, 671], [92, 721], [92, 726], [92, 734], [92, 749], [93, 115], [93, 154], [93, 163], [93, 230], [93, 291], [93, 319], [93, 510], [93, 515], [93, 554], [93, 577], [93, 602], [93, 607], [93, 642], [93, 669], [93, 677], [93, 702], [93, 754], [93, 780], [93, 830], [93, 833], [93, 852], [93, 862], [93, 863], [93, 874], [94, 155], [94, 165], [94, 232], [94, 293], [94, 301], [94, 308], [94, 320], [94, 559], [94, 603], [95, 156], [95, 166], [95, 237], [95, 292], [95, 302], [95, 307], [95, 320], [95, 529], [95, 535], [95, 556], [95, 576], [95, 604], [95, 723], [95, 778], [95, 805], [95, 864], [96, 114], [96, 158], [96, 167], [96, 239], [96, 294], [96, 321], [96, 605], [96, 793], [97, 113], [97, 162], [97, 171], [97, 236], [97, 295], [97, 325], [97, 606], [97, 674], [97, 721], [97, 832], [98, 149], [98, 163], [98, 238], [98, 297], [98, 303], [98, 310], [98, 326], [98, 607], [98, 747], [98, 852], [99, 150], [99, 164], [99, 243], [99, 296], [99, 304], [99, 309], [99, 326], [99, 608], [100, 115], [100, 151], [100, 166], [100, 227], [100, 298], [100, 327], [100, 576], [100, 577], [100, 609], [100, 675], [100, 679], [100, 748], [100, 783], [100, 833], [100, 864], [101, 104], [101, 175], [101, 204], [101, 511], [101, 610], [102, 111], [102, 127], [102, 184], [102, 217], [102, 305], [102, 524], [102, 533], [102, 583], [102, 593], [102, 611], [102, 637], [102, 682], [102, 710], [102, 819], [103, 106], [103, 126], [103, 177], [103, 207], [103, 306], [103, 612], [103, 615], [103, 645], [103, 650], [104, 174], [104, 202], [104, 610], [104, 613], [104, 648], [104, 655], [105, 108], [105, 179], [105, 210], [105, 533], [105, 596], [105, 614], [105, 687], [105, 717], [106, 128], [106, 176], [106, 205], [106, 307], [106, 516], [106, 615], [107, 110], [107, 127], [107, 181], [107, 213], [107, 308], [107, 545], [107, 546], [107, 616], [107, 619], [108, 178], [108, 208], [108, 617], [109, 112], [109, 183], [109, 216], [109, 618], [109, 704], [110, 126], [110, 180], [110, 211], [110, 309], [110, 619], [110, 707], [111, 128], [111, 173], [111, 201], [111, 310], [111, 620], [112, 182], [112, 214], [112, 547], [112, 548], [112, 618], [112, 621], [113, 312], [113, 328], [113, 622], [113, 734], [113, 837], [114, 316], [114, 318], [114, 623], [115, 322], [115, 324], [115, 624], [115, 672], [115, 677], [115, 720], [115, 831], [115, 874], [115, 876], [115, 883], [116, 145], [116, 265], [116, 305], [116, 329], [116, 335], [116, 513], [116, 517], [116, 539], [116, 545], [116, 564], [116, 574], [116, 625], [116, 632], [116, 682], [116, 690], [116, 736], [116, 818], [117, 140], [117, 147], [117, 223], [117, 264], [117, 306], [117, 330], [117, 336], [117, 626], [117, 627], [117, 636], [117, 639], [117, 645], [117, 649], [117, 656], [117, 658], [117, 670], [117, 689], [117, 693], [117, 705], [117, 707], [117, 722], [117, 724], [118, 136], [118, 219], [118, 258], [118, 267], [118, 330], [118, 627], [118, 634], [118, 728], [118, 744], [118, 747], [119, 141], [119, 259], [119, 266], [119, 331], [119, 553], [119, 608], [119, 628], [119, 768], [120, 137], [120, 222], [120, 269], [120, 307], [120, 331], [120, 337], [120, 521], [120, 528], [120, 595], [120, 629], [120, 768], [120, 792], [121, 144], [121, 225], [121, 268], [121, 308], [121, 332], [121, 338], [121, 574], [121, 583], [121, 630], [121, 653], [121, 802], [122, 140], [122, 260], [122, 271], [122, 332], [122, 594], [122, 631], [122, 653], [122, 689], [122, 781], [122, 841], [123, 145], [123, 220], [123, 261], [123, 270], [123, 333], [123, 632], [123, 633], [123, 673], [123, 677], [123, 704], [123, 706], [123, 770], [123, 818], [124, 141], [124, 148], [124, 224], [124, 273], [124, 309], [124, 333], [124, 339], [124, 633], [124, 724], [125, 136], [125, 272], [125, 310], [125, 334], [125, 340], [125, 634], [125, 792], [126, 185], [126, 187], [126, 288], [126, 297], [126, 299], [126, 304], [126, 537], [126, 635], [126, 690], [126, 752], [127, 185], [127, 186], [127, 289], [127, 292], [127, 300], [127, 301], [127, 566], [127, 620], [127, 636], [127, 778], [128, 186], [128, 187], [128, 293], [128, 296], [128, 302], [128, 303], [128, 633], [128, 637], [128, 653], [129, 156], [129, 171], [129, 248], [129, 335], [129, 541], [129, 556], [129, 593], [129, 638], [129, 690], [129, 813], [130, 149], [130, 158], [130, 247], [130, 249], [130, 336], [130, 605], [130, 639], [131, 155], [131, 164], [131, 250], [131, 252], [131, 337], [131, 640], [132, 157], [132, 166], [132, 251], [132, 253], [132, 338], [132, 546], [132, 557], [132, 572], [132, 641], [132, 653], [132, 691], [132, 703], [132, 812], [132, 847], [132, 864], [133, 163], [133, 172], [133, 254], [133, 256], [133, 339], [133, 633], [133, 642], [133, 652], [133, 750], [133, 763], [133, 782], [133, 852], [134, 150], [134, 165], [134, 255], [134, 340], [134, 643], [134, 849], [135, 175], [135, 183], [135, 229], [135, 256], [135, 335], [135, 341], [135, 511], [135, 519], [135, 532], [135, 550], [135, 584], [135, 618], [135, 644], [135, 652], [135, 793], [135, 865], [136, 176], [136, 184], [136, 242], [136, 306], [136, 516], [136, 540], [136, 574], [136, 594], [136, 645], [136, 781], [136, 819], [136, 843], [136, 849], [137, 173], [137, 177], [137, 233], [137, 250], [137, 305], [137, 628], [137, 629], [137, 636], [137, 640], [137, 646], [137, 650], [137, 661], [137, 673], [137, 682], [137, 690], [137, 706], [137, 708], [137, 724], [137, 726], [137, 730], [138, 174], [138, 178], [138, 228], [138, 251], [138, 336], [138, 342], [138, 639], [138, 647], [138, 655], [138, 667], [138, 687], [138, 691], [138, 703], [138, 705], [138, 719], [138, 721], [139, 175], [139, 179], [139, 235], [139, 248], [139, 337], [139, 511], [139, 541], [139, 577], [139, 595], [139, 648], [140, 176], [140, 180], [140, 230], [140, 249], [140, 308], [140, 516], [140, 520], [140, 534], [140, 554], [140, 559], [140, 619], [140, 649], [140, 653], [140, 728], [140, 764], [140, 792], [140, 817], [140, 863], [141, 177], [141, 181], [141, 239], [141, 254], [141, 307], [141, 574], [141, 584], [141, 633], [141, 650], [141, 805], [142, 178], [142, 182], [142, 234], [142, 255], [142, 338], [142, 594], [142, 618], [142, 651], [142, 691], [142, 779], [142, 793], [143, 179], [143, 183], [143, 241], [143, 252], [143, 339], [143, 343], [143, 652], [143, 706], [143, 721], [144, 180], [144, 184], [144, 236], [144, 253], [144, 310], [144, 653], [144, 705], [144, 724], [144, 747], [144, 771], [144, 819], [145, 173], [145, 181], [145, 227], [145, 309], [145, 595], [145, 608], [145, 619], [145, 633], [145, 654], [145, 690], [145, 783], [145, 792], [145, 813], [145, 842], [146, 174], [146, 182], [146, 240], [146, 247], [146, 340], [146, 344], [146, 577], [146, 655], [146, 764], [147, 258], [147, 286], [147, 520], [147, 656], [148, 261], [148, 275], [148, 633], [148, 657], [148, 795], [149, 185], [149, 198], [149, 264], [149, 297], [149, 334], [149, 357], [149, 559], [149, 658], [149, 676], [150, 153], [150, 169], [150, 190], [150, 244], [150, 274], [150, 313], [150, 328], [150, 344], [150, 348], [150, 349], [150, 358], [150, 531], [150, 538], [150, 659], [150, 676], [150, 793], [150, 853], [151, 154], [151, 170], [151, 265], [151, 298], [151, 311], [151, 314], [151, 341], [151, 349], [151, 510], [151, 530], [151, 551], [151, 564], [151, 575], [151, 598], [151, 660], [151, 676], [151, 781], [151, 796], [151, 820], [151, 835], [151, 853], [151, 865], [152, 155], [152, 171], [152, 199], [152, 222], [152, 266], [152, 287], [152, 289], [152, 312], [152, 315], [152, 329], [152, 345], [152, 359], [152, 528], [152, 553], [152, 566], [152, 574], [152, 661], [152, 676], [152, 749], [152, 796], [152, 837], [153, 158], [153, 192], [153, 263], [153, 288], [153, 290], [153, 313], [153, 316], [153, 330], [153, 345], [153, 360], [153, 592], [153, 605], [153, 623], [153, 627], [153, 662], [153, 676], [153, 711], [153, 724], [153, 751], [153, 752], [154, 159], [154, 223], [154, 264], [154, 291], [154, 314], [154, 317], [154, 342], [154, 350], [154, 658], [154, 659], [154, 660], [154, 661], [154, 662], [154, 663], [154, 664], [154, 665], [154, 666], [154, 667], [154, 668], [154, 669], [154, 670], [154, 671], [154, 672], [154, 673], [154, 674], [154, 675], [154, 676], [154, 677], [154, 678], [154, 679], [154, 680], [154, 681], [154, 907], [154, 908], [154, 909], [154, 910], [154, 911], [154, 912], [154, 913], [154, 914], [154, 915], [154, 916], [154, 917], [154, 918], [155, 160], [155, 189], [155, 244], [155, 267], [155, 315], [155, 318], [155, 350], [155, 361], [155, 597], [155, 598], [155, 664], [155, 676], [155, 716], [155, 721], [155, 733], [155, 744], [156, 185], [156, 193], [156, 265], [156, 292], [156, 331], [156, 351], [156, 564], [156, 579], [156, 601], [156, 608], [156, 665], [156, 676], [156, 690], [156, 746], [156, 761], [156, 768], [156, 778], [157, 186], [157, 190], [157, 268], [157, 289], [157, 330], [157, 538], [157, 551], [157, 566], [157, 578], [157, 607], [157, 627], [157, 653], [157, 666], [157, 676], [157, 716], [157, 722], [157, 747], [157, 762], [157, 777], [157, 802], [158, 161], [158, 194], [158, 245], [158, 266], [158, 316], [158, 319], [158, 342], [158, 351], [158, 352], [158, 362], [158, 531], [158, 553], [158, 560], [158, 567], [158, 577], [158, 623], [158, 667], [158, 676], [158, 748], [158, 803], [158, 830], [159, 162], [159, 222], [159, 269], [159, 290], [159, 317], [159, 320], [159, 352], [159, 510], [159, 528], [159, 559], [159, 569], [159, 576], [159, 606], [159, 668], [159, 676], [159, 734], [159, 777], [159, 804], [159, 829], [159, 832], [159, 862], [160, 163], [160, 191], [160, 224], [160, 270], [160, 291], [160, 293], [160, 318], [160, 321], [160, 331], [160, 346], [160, 363], [160, 530], [160, 543], [160, 633], [160, 669], [160, 676], [160, 768], [160, 792], [160, 852], [161, 166], [161, 196], [161, 223], [161, 267], [161, 292], [161, 294], [161, 319], [161, 322], [161, 332], [161, 346], [161, 364], [161, 573], [161, 574], [161, 605], [161, 670], [161, 676], [161, 744], [161, 778], [161, 780], [161, 830], [161, 864], [161, 883], [162, 167], [162, 225], [162, 268], [162, 295], [162, 320], [162, 323], [162, 353], [162, 671], [162, 676], [162, 802], [162, 805], [163, 168], [163, 193], [163, 245], [163, 271], [163, 321], [163, 324], [163, 343], [163, 353], [163, 354], [163, 365], [163, 652], [163, 672], [163, 676], [163, 733], [163, 761], [163, 793], [163, 848], [163, 862], [164, 186], [164, 197], [164, 269], [164, 296], [164, 333], [164, 673], [164, 676], [165, 187], [165, 194], [165, 272], [165, 293], [165, 332], [165, 354], [165, 674], [165, 676], [165, 781], [166, 169], [166, 198], [166, 246], [166, 270], [166, 322], [166, 325], [166, 355], [166, 366], [166, 675], [166, 676], [166, 691], [166, 721], [166, 723], [166, 746], [166, 749], [166, 762], [166, 803], [166, 883], [167, 170], [167, 224], [167, 273], [167, 294], [167, 323], [167, 326], [167, 343], [167, 355], [167, 676], [167, 747], [167, 832], [167, 835], [168, 171], [168, 195], [168, 274], [168, 295], [168, 297], [168, 324], [168, 327], [168, 333], [168, 347], [168, 367], [168, 676], [168, 677], [168, 690], [168, 722], [168, 724], [168, 748], [168, 804], [168, 852], [169, 188], [169, 225], [169, 271], [169, 296], [169, 298], [169, 325], [169, 328], [169, 334], [169, 347], [169, 368], [169, 653], [169, 676], [169, 678], [169, 734], [169, 792], [169, 864], [170, 272], [170, 287], [170, 311], [170, 326], [170, 344], [170, 356], [170, 607], [170, 608], [170, 676], [170, 679], [171, 197], [171, 246], [171, 263], [171, 312], [171, 327], [171, 341], [171, 356], [171, 357], [171, 369], [171, 577], [171, 578], [171, 606], [171, 676], [171, 680], [171, 751], [171, 780], [171, 837], [171, 865], [172, 187], [172, 189], [172, 273], [172, 288], [172, 329], [172, 348], [172, 560], [172, 576], [172, 579], [172, 597], [172, 633], [172, 676], [172, 681], [172, 752], [172, 805], [172, 807], [172, 848], [173, 256], [173, 286], [173, 305], [173, 348], [173, 521], [173, 633], [173, 682], [173, 796], [173, 818], [174, 277], [174, 349], [174, 514], [174, 518], [174, 540], [174, 546], [174, 565], [174, 575], [174, 683], [174, 691], [174, 749], [174, 756], [174, 849], [174, 853], [175, 250], [175, 276], [175, 350], [175, 640], [175, 648], [175, 652], [175, 664], [175, 684], [175, 692], [175, 698], [175, 708], [175, 709], [175, 711], [176, 218], [176, 251], [176, 279], [176, 306], [176, 634], [176, 635], [176, 643], [176, 645], [176, 653], [176, 662], [176, 674], [176, 685], [176, 689], [176, 701], [176, 703], [176, 716], [176, 718], [176, 728], [177, 220], [177, 248], [177, 278], [177, 307], [177, 351], [177, 513], [177, 521], [177, 541], [177, 547], [177, 568], [177, 576], [177, 628], [177, 633], [177, 686], [177, 690], [177, 748], [177, 763], [177, 816], [178, 249], [178, 281], [178, 352], [178, 370], [178, 518], [178, 687], [179, 254], [179, 280], [179, 353], [179, 371], [179, 593], [179, 617], [179, 652], [179, 688], [179, 780], [179, 790], [180, 219], [180, 255], [180, 283], [180, 308], [180, 354], [180, 689], [181, 221], [181, 252], [181, 282], [181, 309], [181, 690], [181, 749], [181, 818], [182, 253], [182, 285], [182, 355], [182, 691], [182, 748], [183, 284], [183, 356], [183, 692], [183, 813], [184, 247], [184, 275], [184, 310], [184, 357], [184, 596], [184, 607], [184, 620], [184, 634], [184, 653], [184, 693], [184, 780], [184, 795], [184, 812], [184, 843], [185, 556], [185, 694], [186, 572], [186, 695], [187, 696], [187, 782], [188, 203], [188, 216], [188, 334], [188, 548], [188, 562], [188, 653], [188, 691], [188, 697], [189, 200], [189, 205], [189, 248], [189, 300], [189, 329], [189, 535], [189, 541], [189, 545], [189, 584], [189, 612], [189, 633], [189, 698], [189, 706], [189, 772], [189, 782], [189, 794], [189, 813], [190, 201], [190, 206], [190, 247], [190, 249], [190, 299], [190, 330], [190, 571], [190, 572], [190, 583], [190, 590], [190, 594], [190, 627], [190, 647], [190, 653], [190, 699], [190, 707], [190, 737], [190, 756], [191, 203], [191, 208], [191, 331], [191, 562], [191, 579], [191, 587], [191, 595], [191, 633], [191, 646], [191, 652], [191, 700], [191, 704], [191, 733], [191, 742], [191, 768], [192, 204], [192, 209], [192, 330], [192, 531], [192, 537], [192, 540], [192, 548], [192, 613], [192, 627], [192, 701], [192, 705], [192, 776], [192, 791], [192, 817], [193, 206], [193, 211], [193, 250], [193, 252], [193, 302], [193, 331], [193, 371], [193, 545], [193, 556], [193, 571], [193, 584], [193, 640], [193, 652], [193, 690], [193, 702], [193, 768], [193, 852], [193, 880], [194, 207], [194, 212], [194, 251], [194, 253], [194, 301], [194, 332], [194, 370], [194, 583], [194, 594], [194, 605], [194, 703], [194, 764], [195, 209], [195, 214], [195, 333], [195, 690], [195, 704], [195, 791], [196, 210], [196, 215], [196, 332], [196, 705], [196, 803], [197, 212], [197, 217], [197, 254], [197, 256], [197, 304], [197, 333], [197, 706], [198, 200], [198, 213], [198, 255], [198, 303], [198, 334], [198, 691], [198, 707], [198, 764], [198, 794], [198, 849], [198, 864], [199, 202], [199, 215], [199, 329], [199, 595], [199, 708], [200, 228], [200, 241], [200, 276], [200, 285], [200, 349], [200, 356], [200, 577], [200, 585], [200, 597], [200, 691], [200, 709], [200, 793], [200, 853], [201, 328], [201, 357], [201, 538], [201, 559], [201, 563], [201, 606], [201, 653], [201, 710], [201, 781], [201, 796], [201, 866], [202, 243], [202, 315], [202, 349], [202, 711], [202, 853], [203, 227], [203, 230], [203, 275], [203, 278], [203, 543], [203, 544], [203, 554], [203, 557], [203, 568], [203, 574], [203, 626], [203, 633], [203, 646], [203, 653], [203, 662], [203, 677], [203, 712], [203, 724], [203, 737], [203, 750], [203, 772], [203, 783], [203, 784], [203, 795], [203, 796], [203, 807], [204, 232], [204, 313], [204, 350], [204, 592], [204, 603], [204, 664], [204, 713], [205, 318], [205, 351], [205, 597], [205, 608], [205, 633], [205, 662], [205, 673], [205, 714], [206, 229], [206, 234], [206, 277], [206, 280], [206, 350], [206, 352], [206, 538], [206, 549], [206, 550], [206, 561], [206, 565], [206, 577], [206, 601], [206, 606], [206, 647], [206, 652], [206, 664], [206, 675], [206, 715], [206, 721], [206, 742], [206, 745], [206, 760], [206, 761], [206, 776], [206, 779], [206, 789], [206, 790], [207, 316], [207, 559], [207, 623], [207, 716], [207, 747], [208, 231], [208, 321], [208, 352], [208, 543], [208, 570], [208, 652], [208, 717], [209, 233], [209, 236], [209, 279], [209, 282], [209, 574], [209, 592], [209, 690], [209, 718], [209, 734], [209, 792], [209, 817], [210, 238], [210, 319], [210, 353], [210, 617], [210, 719], [210, 830], [211, 324], [211, 673], [211, 675], [211, 690], [211, 720], [211, 761], [211, 805], [212, 235], [212, 240], [212, 281], [212, 284], [212, 353], [212, 355], [212, 721], [212, 793], [213, 322], [213, 354], [213, 722], [213, 747], [213, 781], [213, 863], [213, 883], [214, 237], [214, 327], [214, 355], [214, 723], [214, 734], [215, 239], [215, 242], [215, 283], [215, 286], [215, 724], [215, 792], [216, 226], [216, 325], [216, 356], [216, 677], [216, 691], [216, 725], [216, 806], [217, 312], [217, 348], [217, 608], [217, 620], [217, 726], [217, 805], [217, 837], [218, 289], [218, 516], [218, 529], [218, 546], [218, 566], [218, 588], [218, 653], [218, 727], [218, 771], [218, 772], [218, 843], [219, 293], [219, 728], [220, 292], [220, 593], [220, 632], [220, 650], [220, 690], [220, 729], [220, 778], [220, 842], [221, 296], [221, 730], [222, 257], [222, 279], [222, 595], [222, 606], [222, 622], [222, 661], [222, 671], [222, 713], [222, 718], [222, 731], [222, 741], [222, 742], [223, 260], [223, 278], [223, 342], [223, 510], [223, 520], [223, 527], [223, 558], [223, 568], [223, 667], [223, 675], [223, 705], [223, 732], [223, 776], [223, 801], [223, 803], [223, 828], [223, 831], [223, 841], [223, 855], [223, 873], [224, 259], [224, 283], [224, 343], [224, 733], [225, 262], [225, 282], [225, 734], [225, 832], [226, 264], [226, 297], [226, 356], [226, 578], [226, 585], [226, 607], [226, 635], [226, 658], [226, 677], [226, 735], [226, 781], [226, 820], [226, 866], [227, 257], [227, 274], [227, 369], [227, 561], [227, 562], [227, 577], [227, 586], [227, 595], [227, 659], [227, 677], [227, 736], [227, 749], [227, 796], [227, 818], [227, 856], [228, 266], [228, 287], [228, 336], [228, 349], [228, 360], [228, 518], [228, 530], [228, 553], [228, 639], [228, 724], [228, 737], [228, 794], [228, 853], [229, 263], [229, 290], [229, 335], [229, 350], [229, 359], [229, 571], [229, 574], [229, 584], [229, 606], [229, 664], [229, 678], [229, 685], [229, 692], [229, 738], [229, 751], [229, 753], [229, 772], [230, 258], [230, 267], [230, 362], [230, 562], [230, 577], [230, 580], [230, 598], [230, 662], [230, 672], [230, 684], [230, 689], [230, 739], [230, 744], [230, 754], [230, 759], [230, 776], [231, 265], [231, 292], [231, 352], [231, 549], [231, 560], [231, 564], [231, 576], [231, 600], [231, 632], [231, 646], [231, 663], [231, 673], [231, 714], [231, 720], [231, 740], [231, 759], [231, 778], [231, 801], [232, 268], [232, 289], [232, 350], [232, 523], [232, 559], [232, 566], [232, 664], [232, 674], [232, 741], [232, 802], [232, 810], [232, 817], [232, 829], [233, 259], [233, 266], [233, 361], [233, 521], [233, 553], [233, 721], [233, 742], [233, 791], [233, 810], [234, 270], [234, 291], [234, 338], [234, 352], [234, 364], [234, 557], [234, 571], [234, 574], [234, 591], [234, 594], [234, 632], [234, 669], [234, 675], [234, 743], [234, 748], [234, 804], [234, 816], [234, 855], [235, 267], [235, 294], [235, 337], [235, 353], [235, 363], [235, 744], [235, 792], [236, 260], [236, 271], [236, 366], [236, 705], [236, 721], [236, 734], [236, 745], [236, 780], [236, 791], [236, 841], [236, 862], [237, 269], [237, 296], [237, 355], [237, 734], [237, 746], [237, 805], [238, 272], [238, 293], [238, 353], [238, 747], [239, 261], [239, 270], [239, 365], [239, 748], [239, 793], [240, 274], [240, 295], [240, 340], [240, 355], [240, 368], [240, 749], [240, 764], [240, 792], [241, 271], [241, 298], [241, 339], [241, 356], [241, 367], [241, 706], [241, 724], [241, 750], [241, 780], [241, 794], [242, 262], [242, 263], [242, 358], [242, 634], [242, 751], [242, 793], [243, 273], [243, 288], [243, 349], [243, 608], [243, 752], [243, 853], [244, 477], [244, 753], [245, 479], [245, 605], [245, 748], [245, 754], [245, 852], [246, 481], [246, 749], [246, 755], [246, 780], [246, 864], [246, 877], [247, 258], [247, 344], [247, 357], [247, 538], [247, 577], [247, 607], [247, 756], [247, 819], [248, 257], [248, 351], [248, 576], [248, 577], [248, 584], [248, 593], [248, 597], [248, 644], [248, 650], [248, 698], [248, 702], [248, 738], [248, 757], [248, 759], [249, 301], [249, 336], [249, 352], [249, 538], [249, 559], [249, 594], [249, 639], [249, 689], [249, 758], [249, 810], [250, 300], [250, 337], [250, 350], [250, 511], [250, 521], [250, 584], [250, 664], [250, 673], [250, 759], [250, 761], [250, 846], [251, 260], [251, 342], [251, 370], [251, 516], [251, 518], [251, 525], [251, 542], [251, 546], [251, 590], [251, 594], [251, 667], [251, 674], [251, 716], [251, 721], [251, 760], [251, 762], [251, 810], [251, 841], [251, 879], [252, 259], [252, 343], [252, 371], [252, 721], [252, 761], [253, 303], [253, 338], [253, 355], [253, 747], [253, 748], [253, 762], [253, 764], [253, 812], [254, 302], [254, 339], [254, 353], [254, 763], [254, 780], [254, 805], [254, 848], [255, 262], [255, 354], [255, 764], [255, 793], [256, 261], [256, 341], [256, 348], [256, 692], [256, 706], [256, 765], [256, 793], [256, 848], [256, 865], [257, 329], [257, 517], [257, 528], [257, 541], [257, 576], [257, 595], [257, 646], [257, 730], [257, 766], [257, 783], [257, 838], [258, 330], [258, 554], [258, 607], [258, 627], [258, 767], [259, 331], [259, 768], [260, 332], [260, 630], [260, 631], [260, 670], [260, 674], [260, 703], [260, 705], [260, 769], [260, 817], [261, 333], [261, 770], [261, 842], [262, 334], [262, 771], [263, 298], [263, 306], [263, 345], [263, 531], [263, 548], [263, 550], [263, 577], [263, 598], [263, 645], [263, 724], [263, 771], [263, 772], [263, 867], [264, 287], [264, 289], [264, 336], [264, 510], [264, 520], [264, 559], [264, 566], [264, 607], [264, 639], [264, 694], [264, 707], [264, 773], [264, 806], [264, 843], [264, 866], [265, 288], [265, 290], [265, 335], [265, 545], [265, 556], [265, 570], [265, 576], [265, 608], [265, 628], [265, 654], [265, 665], [265, 679], [265, 714], [265, 726], [265, 752], [265, 774], [265, 797], [266, 291], [266, 305], [266, 345], [266, 574], [266, 577], [266, 583], [266, 605], [266, 659], [266, 669], [266, 682], [266, 687], [266, 737], [266, 742], [266, 753], [266, 758], [266, 775], [267, 290], [267, 308], [267, 346], [267, 545], [267, 554], [267, 574], [267, 648], [267, 721], [267, 776], [267, 803], [267, 846], [267, 862], [267, 872], [268, 291], [268, 293], [268, 338], [268, 572], [268, 583], [268, 603], [268, 637], [268, 669], [268, 674], [268, 728], [268, 747], [268, 777], [268, 832], [268, 863], [269, 292], [269, 294], [269, 337], [269, 671], [269, 673], [269, 778], [270, 295], [270, 307], [270, 346], [270, 704], [270, 721], [270, 733], [270, 748], [270, 779], [270, 792], [270, 842], [270, 864], [271, 294], [271, 310], [271, 347], [271, 780], [271, 792], [271, 793], [271, 852], [272, 295], [272, 297], [272, 340], [272, 781], [272, 835], [273, 296], [273, 298], [273, 339], [273, 782], [273, 805], [274, 287], [274, 309], [274, 347], [274, 707], [274, 724], [274, 730], [274, 749], [274, 783], [274, 793], [275, 286], [275, 305], [275, 562], [275, 596], [275, 656], [275, 682], [275, 724], [275, 784], [275, 819], [276, 277], [276, 341], [276, 511], [276, 522], [276, 565], [276, 577], [276, 698], [276, 707], [276, 737], [276, 750], [276, 785], [276, 794], [276, 858], [276, 865], [277, 342], [277, 545], [277, 546], [277, 557], [277, 558], [277, 571], [277, 577], [277, 602], [277, 609], [277, 648], [277, 655], [277, 667], [277, 680], [277, 709], [277, 715], [277, 736], [277, 739], [277, 756], [277, 757], [277, 772], [277, 775], [277, 785], [277, 786], [278, 279], [278, 306], [278, 547], [278, 548], [278, 558], [278, 561], [278, 562], [278, 574], [278, 625], [278, 630], [278, 645], [278, 650], [278, 661], [278, 670], [278, 712], [278, 718], [278, 738], [278, 743], [278, 775], [278, 776], [278, 787], [278, 788], [278, 799], [278, 800], [279, 307], [279, 516], [279, 517], [279, 528], [279, 529], [279, 568], [279, 574], [279, 701], [279, 704], [279, 742], [279, 745], [279, 788], [279, 791], [280, 281], [280, 371], [280, 571], [280, 583], [280, 593], [280, 687], [280, 721], [280, 789], [280, 879], [281, 370], [281, 721], [281, 790], [282, 283], [282, 308], [282, 791], [282, 792], [283, 309], [283, 792], [284, 285], [284, 343], [284, 793], [285, 344], [285, 793], [285, 794], [286, 310], [286, 724], [286, 795], [287, 313], [287, 356], [287, 372], [287, 658], [287, 724], [287, 796], [287, 835], [288, 311], [288, 348], [288, 531], [288, 537], [288, 560], [288, 564], [288, 608], [288, 711], [288, 782], [288, 797], [288, 858], [289, 317], [289, 546], [289, 559], [289, 572], [289, 578], [289, 603], [289, 649], [289, 658], [289, 668], [289, 710], [289, 716], [289, 741], [289, 758], [289, 773], [289, 798], [290, 315], [290, 352], [290, 373], [290, 531], [290, 550], [290, 564], [290, 574], [290, 662], [290, 671], [290, 744], [290, 799], [290, 827], [291, 319], [291, 350], [291, 373], [291, 510], [291, 513], [291, 530], [291, 553], [291, 574], [291, 664], [291, 672], [291, 700], [291, 733], [291, 779], [291, 800], [291, 802], [291, 827], [291, 830], [291, 840], [291, 855], [291, 872], [292, 317], [292, 351], [292, 556], [292, 570], [292, 573], [292, 593], [292, 629], [292, 668], [292, 673], [292, 746], [292, 801], [292, 803], [292, 846], [293, 323], [293, 354], [293, 733], [293, 747], [293, 802], [294, 321], [294, 355], [294, 374], [294, 792], [294, 803], [295, 325], [295, 353], [295, 374], [295, 792], [295, 804], [295, 832], [295, 863], [296, 323], [296, 805], [297, 311], [297, 357], [297, 781], [297, 806], [298, 327], [298, 349], [298, 372], [298, 678], [298, 679], [298, 693], [298, 724], [298, 726], [298, 751], [298, 807], [298, 853], [299, 306], [299, 538], [299, 561], [299, 578], [299, 643], [299, 645], [299, 683], [299, 707], [299, 808], [299, 849], [299, 858], [300, 305], [300, 545], [300, 560], [300, 597], [300, 640], [300, 682], [300, 759], [300, 809], [301, 308], [301, 583], [301, 810], [302, 307], [302, 745], [302, 746], [302, 761], [302, 763], [302, 811], [303, 310], [303, 812], [304, 309], [304, 813], [305, 329], [305, 513], [305, 521], [305, 533], [305, 553], [305, 560], [305, 620], [305, 646], [305, 654], [305, 765], [305, 795], [305, 814], [305, 857], [306, 330], [306, 516], [306, 520], [306, 542], [306, 548], [306, 568], [306, 578], [306, 627], [306, 634], [306, 685], [306, 693], [306, 751], [306, 815], [307, 331], [307, 650], [307, 704], [307, 718], [307, 746], [307, 768], [307, 816], [308, 332], [308, 689], [308, 744], [308, 817], [309, 333], [309, 818], [310, 334], [310, 819], [311, 679], [311, 752], [311, 820], [311, 835], [312, 328], [312, 359], [312, 369], [312, 372], [312, 622], [312, 659], [312, 678], [312, 710], [312, 821], [312, 867], [312, 877], [312, 881], [313, 315], [313, 358], [313, 360], [313, 531], [313, 822], [314, 510], [314, 527], [314, 560], [314, 567], [314, 578], [314, 609], [314, 663], [314, 679], [314, 774], [314, 807], [314, 820], [314, 823], [314, 850], [314, 858], [315, 359], [315, 361], [315, 822], [315, 824], [316, 318], [316, 360], [316, 362], [316, 373], [316, 530], [316, 531], [316, 598], [316, 605], [316, 825], [316, 827], [317, 510], [317, 529], [317, 552], [317, 566], [317, 573], [317, 601], [317, 663], [317, 671], [317, 731], [317, 778], [317, 799], [317, 826], [317, 829], [317, 861], [318, 361], [318, 363], [318, 373], [318, 623], [318, 733], [318, 827], [319, 321], [319, 362], [319, 364], [319, 605], [319, 669], [319, 672], [319, 717], [319, 803], [319, 828], [319, 872], [319, 874], [320, 671], [320, 746], [320, 829], [320, 832], [321, 363], [321, 365], [321, 733], [321, 830], [321, 852], [322, 324], [322, 364], [322, 366], [322, 374], [322, 803], [322, 804], [322, 831], [322, 833], [322, 862], [322, 864], [323, 832], [324, 365], [324, 367], [324, 374], [324, 833], [324, 852], [324, 883], [325, 327], [325, 366], [325, 368], [325, 834], [325, 864], [326, 835], [327, 367], [327, 369], [327, 834], [327, 836], [328, 358], [328, 368], [328, 372], [328, 837], [329, 597], [329, 654], [329, 682], [329, 730], [329, 782], [329, 838], [330, 520], [330, 531], [330, 538], [330, 572], [330, 592], [330, 645], [330, 728], [330, 767], [330, 776], [330, 839], [331, 543], [331, 556], [331, 628], [331, 733], [331, 742], [331, 761], [331, 840], [332, 803], [332, 841], [333, 842], [334, 771], [334, 843], [335, 341], [335, 549], [335, 550], [335, 564], [335, 638], [335, 654], [335, 692], [335, 698], [335, 813], [335, 844], [335, 865], [336, 342], [336, 518], [336, 520], [336, 585], [336, 658], [336, 667], [336, 756], [336, 758], [336, 845], [337, 371], [337, 640], [337, 846], [338, 370], [338, 762], [338, 779], [338, 802], [338, 847], [339, 343], [339, 848], [340, 344], [340, 849], [341, 356], [341, 679], [341, 680], [341, 692], [341, 709], [341, 725], [341, 735], [341, 750], [341, 796], [341, 850], [341, 881], [342, 352], [342, 510], [342, 518], [342, 557], [342, 565], [342, 600], [342, 605], [342, 639], [342, 662], [342, 670], [342, 703], [342, 754], [342, 775], [342, 825], [342, 828], [342, 851], [342, 860], [342, 861], [342, 871], [343, 353], [343, 852], [344, 349], [344, 835], [344, 853], [345, 431], [345, 531], [345, 553], [345, 751], [345, 854], [345, 867], [346, 433], [346, 733], [346, 744], [346, 803], [346, 855], [347, 435], [347, 856], [348, 726], [348, 752], [348, 782], [348, 857], [348, 881], [349, 372], [349, 655], [349, 659], [349, 679], [349, 756], [349, 794], [349, 849], [349, 858], [350, 373], [350, 510], [350, 511], [350, 550], [350, 571], [350, 598], [350, 603], [350, 640], [350, 661], [350, 669], [350, 698], [350, 753], [350, 776], [350, 824], [350, 827], [350, 859], [350, 870], [351, 541], [351, 556], [351, 567], [351, 605], [351, 650], [351, 714], [351, 778], [351, 799], [351, 860], [352, 370], [352, 373], [352, 570], [352, 571], [352, 605], [352, 667], [352, 671], [352, 742], [352, 779], [352, 827], [352, 861], [353, 371], [353, 374], [353, 832], [353, 852], [353, 862], [354, 852], [354, 863], [355, 374], [355, 864], [356, 372], [356, 794], [356, 806], [356, 813], [356, 835], [356, 865], [357, 819], [357, 866], [358, 482], [358, 867], [359, 483], [359, 550], [359, 753], [359, 837], [359, 868], [360, 484], [360, 531], [360, 623], [360, 869], [361, 485], [361, 870], [362, 486], [362, 554], [362, 605], [362, 623], [362, 830], [362, 871], [363, 487], [363, 733], [363, 872], [364, 488], [364, 779], [364, 803], [364, 830], [364, 873], [364, 883], [365, 489], [365, 852], [365, 874], [366, 490], [366, 864], [366, 875], [366, 883], [367, 491], [367, 876], [368, 492], [368, 877], [369, 493], [369, 783], [369, 837], [369, 856], [369, 878], [370, 703], [370, 879], [371, 761], [371, 790], [371, 880], [372, 837], [372, 853], [372, 866], [372, 881], [373, 623], [373, 664], [373, 669], [373, 714], [373, 825], [373, 870], [373, 872], [373, 882], [374, 883], [375, 390], [375, 397], [375, 406], [375, 411], [375, 423], [375, 884], [376, 392], [376, 394], [376, 398], [376, 412], [376, 427], [376, 885], [377, 393], [377, 395], [377, 405], [377, 410], [377, 417], [377, 428], [377, 886], [378, 387], [378, 388], [378, 394], [378, 401], [378, 408], [378, 414], [378, 420], [378, 429], [378, 430], [378, 431], [378, 436], [378, 437], [378, 448], [378, 887], [379, 388], [379, 389], [379, 396], [379, 398], [379, 403], [379, 419], [379, 422], [379, 431], [379, 432], [379, 438], [379, 439], [379, 449], [379, 888], [380, 384], [380, 389], [380, 391], [380, 400], [380, 405], [380, 415], [380, 421], [380, 424], [380, 432], [380, 433], [380, 440], [380, 441], [380, 450], [380, 889], [381, 384], [381, 385], [381, 393], [381, 397], [381, 402], [381, 407], [381, 416], [381, 417], [381, 423], [381, 426], [381, 433], [381, 434], [381, 442], [381, 443], [381, 451], [381, 890], [382, 385], [382, 386], [382, 390], [382, 395], [382, 404], [382, 409], [382, 425], [382, 428], [382, 434], [382, 435], [382, 444], [382, 445], [382, 452], [382, 891], [383, 386], [383, 387], [383, 392], [383, 399], [383, 406], [383, 413], [383, 418], [383, 427], [383, 430], [383, 435], [383, 446], [383, 447], [383, 453], [383, 892], [384, 386], [384, 388], [384, 393], [384, 400], [384, 407], [384, 419], [384, 428], [384, 448], [384, 453], [384, 459], [384, 893], [385, 387], [385, 389], [385, 395], [385, 397], [385, 402], [385, 409], [385, 410], [385, 411], [385, 418], [385, 421], [385, 448], [385, 449], [385, 454], [385, 894], [386, 388], [386, 390], [386, 399], [386, 404], [386, 420], [386, 423], [386, 449], [386, 450], [386, 455], [386, 895], [387, 389], [387, 392], [387, 401], [387, 406], [387, 411], [387, 412], [387, 422], [387, 425], [387, 450], [387, 451], [387, 896], [388, 394], [388, 403], [388, 408], [388, 424], [388, 427], [388, 451], [388, 452], [388, 456], [388, 457], [388, 897], [389, 391], [389, 396], [389, 398], [389, 405], [389, 410], [389, 412], [389, 426], [389, 429], [389, 452], [389, 453], [389, 458], [389, 898], [390, 408], [390, 420], [390, 447], [390, 454], [390, 459], [390, 460], [390, 899], [391, 401], [391, 415], [391, 429], [391, 438], [391, 461], [391, 900], [392, 398], [392, 413], [392, 422], [392, 437], [392, 454], [392, 455], [392, 462], [392, 901], [393, 403], [393, 416], [393, 419], [393, 440], [393, 463], [393, 902], [394, 400], [394, 414], [394, 424], [394, 439], [394, 455], [394, 464], [394, 903], [395, 405], [395, 421], [395, 442], [395, 465], [395, 904], [396, 402], [396, 426], [396, 441], [396, 456], [396, 466], [396, 905], [397, 406], [397, 417], [397, 418], [397, 445], [397, 458], [397, 459], [397, 470], [397, 906], [398, 400], [398, 408], [398, 431], [398, 453], [398, 454], [398, 461], [398, 471], [398, 472], [398, 476], [398, 482], [398, 676], [398, 907], [399, 401], [399, 409], [399, 430], [399, 449], [399, 460], [399, 462], [399, 472], [399, 477], [399, 483], [399, 676], [399, 908], [400, 402], [400, 432], [400, 448], [400, 455], [400, 461], [400, 463], [400, 477], [400, 484], [400, 676], [400, 909], [401, 403], [401, 431], [401, 450], [401, 462], [401, 464], [401, 478], [401, 485], [401, 676], [401, 910], [402, 404], [402, 433], [402, 449], [402, 463], [402, 465], [402, 473], [402, 478], [402, 486], [402, 676], [402, 911], [403, 405], [403, 432], [403, 451], [403, 464], [403, 466], [403, 473], [403, 479], [403, 487], [403, 676], [403, 912], [404, 406], [404, 434], [404, 450], [404, 456], [404, 465], [404, 467], [404, 474], [404, 479], [404, 488], [404, 676], [404, 913], [405, 407], [405, 433], [405, 452], [405, 466], [405, 468], [405, 474], [405, 480], [405, 489], [405, 676], [405, 914], [406, 408], [406, 435], [406, 451], [406, 458], [406, 467], [406, 469], [406, 480], [406, 490], [406, 676], [406, 915], [407, 409], [407, 434], [407, 453], [407, 457], [407, 468], [407, 470], [407, 481], [407, 491], [407, 676], [407, 916], [408, 430], [408, 452], [408, 459], [408, 469], [408, 471], [408, 475], [408, 481], [408, 492], [408, 676], [408, 917], [409, 435], [409, 448], [409, 460], [409, 470], [409, 475], [409, 476], [409, 493], [409, 676], [409, 918], [410, 419], [410, 428], [410, 919], [411, 420], [411, 423], [411, 920], [412, 424], [412, 427], [412, 921], [413, 420], [413, 447], [413, 476], [413, 483], [413, 922], [414, 422], [414, 437], [414, 477], [414, 485], [414, 923], [415, 419], [415, 440], [415, 478], [415, 484], [415, 924], [416, 421], [416, 442], [416, 479], [416, 486], [416, 925], [417, 428], [417, 443], [417, 480], [417, 491], [417, 926], [418, 453], [418, 454], [418, 493], [418, 494], [418, 927], [419, 449], [419, 484], [419, 928], [420, 448], [420, 455], [420, 483], [420, 929], [421, 450], [421, 486], [421, 930], [422, 449], [422, 485], [422, 495], [422, 931], [423, 451], [423, 488], [423, 932], [424, 450], [424, 456], [424, 487], [424, 933], [425, 452], [425, 490], [425, 496], [425, 934], [426, 451], [426, 458], [426, 489], [426, 935], [427, 453], [427, 457], [427, 492], [427, 936], [428, 452], [428, 459], [428, 491], [428, 497], [428, 937], [429, 448], [429, 482], [429, 938], [430, 437], [430, 446], [430, 472], [430, 475], [430, 494], [430, 939], [431, 436], [431, 439], [431, 472], [431, 940], [432, 438], [432, 441], [432, 473], [432, 495], [432, 941], [433, 440], [433, 443], [433, 473], [433, 474], [433, 942], [434, 442], [434, 445], [434, 474], [434, 496], [434, 943], [435, 444], [435, 447], [435, 475], [435, 497], [435, 944], [436, 471], [436, 483], [436, 498], [436, 945], [437, 462], [437, 482], [437, 499], [437, 946], [438, 461], [438, 485], [438, 500], [438, 947], [439, 464], [439, 484], [439, 501], [439, 948], [440, 463], [440, 487], [440, 502], [440, 949], [441, 466], [441, 486], [441, 503], [441, 950], [442, 465], [442, 489], [442, 504], [442, 951], [443, 468], [443, 488], [443, 505], [443, 952], [444, 467], [444, 491], [444, 506], [444, 953], [445, 470], [445, 490], [445, 507], [445, 954], [446, 469], [446, 493], [446, 508], [446, 955], [447, 460], [447, 492], [447, 509], [447, 956], [448, 459], [448, 476], [448, 477], [448, 494], [448, 498], [448, 499], [448, 957], [449, 454], [449, 477], [449, 478], [449, 500], [449, 501], [449, 958], [450, 455], [450, 478], [450, 479], [450, 495], [450, 502], [450, 503], [450, 959], [451, 457], [451, 479], [451, 480], [451, 496], [451, 504], [451, 505], [451, 960], [452, 456], [452, 480], [452, 481], [452, 506], [452, 507], [452, 961], [453, 458], [453, 476], [453, 481], [453, 497], [453, 508], [453, 509], [453, 962], [454, 460], [454, 499], [454, 963], [455, 462], [455, 495], [455, 501], [455, 964], [456, 466], [456, 505], [456, 965], [457, 469], [457, 496], [457, 506], [457, 966], [458, 468], [458, 497], [458, 507], [458, 967], [459, 470], [459, 494], [459, 509], [459, 968], [460, 471], [460, 483], [460, 493], [460, 499], [460, 969], [461, 462], [461, 482], [461, 484], [461, 498], [461, 970], [462, 483], [462, 485], [462, 501], [462, 971], [463, 464], [463, 484], [463, 486], [463, 500], [463, 972], [464, 485], [464, 487], [464, 503], [464, 973], [465, 466], [465, 486], [465, 488], [465, 502], [465, 974], [466, 487], [466, 489], [466, 505], [466, 975], [467, 468], [467, 488], [467, 490], [467, 504], [467, 976], [468, 489], [468, 491], [468, 507], [468, 977], [469, 470], [469, 490], [469, 492], [469, 506], [469, 978], [470, 491], [470, 493], [470, 509], [470, 979], [471, 482], [471, 492], [471, 508], [471, 980], [472, 476], [472, 477], [472, 981], [473, 478], [473, 479], [473, 982], [474, 479], [474, 480], [474, 983], [475, 476], [475, 481], [475, 984], [476, 498], [476, 509], [476, 985], [477, 499], [477, 500], [477, 986], [478, 501], [478, 502], [478, 987], [479, 503], [479, 504], [479, 988], [480, 505], [480, 506], [480, 989], [481, 507], [481, 508], [481, 990], [482, 499], [482, 991], [483, 498], [483, 992], [484, 501], [484, 993], [485, 495], [485, 500], [485, 994], [486, 503], [486, 995], [487, 502], [487, 996], [488, 505], [488, 997], [489, 504], [489, 998], [490, 496], [490, 507], [490, 999], [491, 497], [491, 506], [491, 1000], [492, 509], [492, 1001], [493, 494], [493, 508], [493, 1002], [494, 499], [494, 1003], [495, 503], [495, 1004], [496, 504], [496, 1005], [497, 509], [497, 1006], [498, 1007], [499, 1008], [500, 1009], [501, 1010], [502, 1011], [503, 1012], [504, 1013], [505, 1014], [506, 1015], [507, 1016], [508, 1017], [509, 1018], [510, 658], [510, 659], [510, 660], [510, 661], [510, 662], [510, 663], [510, 664], [510, 665], [510, 666], [510, 667], [510, 668], [510, 669], [510, 670], [510, 671], [510, 672], [510, 673], [510, 674], [510, 675], [510, 676], [510, 677], [510, 678], [510, 679], [510, 680], [510, 681], [510, 907], [510, 908], [510, 909], [510, 910], [510, 911], [510, 912], [510, 913], [510, 914], [510, 915], [510, 916], [510, 917], [510, 918], [511, 640], [511, 648], [511, 652], [511, 664], [511, 684], [511, 692], [511, 698], [511, 708], [511, 709], [511, 711], [512, 630], [512, 631], [512, 637], [512, 641], [512, 649], [512, 653], [512, 666], [512, 678], [512, 685], [512, 693], [512, 697], [512, 699], [512, 710], [512, 712], [512, 727], [513, 632], [513, 633], [513, 637], [513, 642], [513, 650], [513, 654], [513, 657], [513, 669], [513, 681], [513, 682], [513, 686], [513, 698], [513, 700], [513, 712], [513, 714], [514, 643], [514, 651], [514, 655], [514, 659], [514, 683], [514, 687], [514, 699], [514, 701], [514, 713], [514, 715], [515, 642], [515, 644], [515, 652], [515, 672], [515, 684], [515, 688], [515, 700], [515, 702], [515, 715], [515, 717], [516, 634], [516, 635], [516, 643], [516, 645], [516, 653], [516, 662], [516, 674], [516, 685], [516, 689], [516, 701], [516, 703], [516, 716], [516, 718], [516, 728], [517, 625], [517, 635], [517, 638], [517, 646], [517, 654], [517, 665], [517, 677], [517, 686], [517, 690], [517, 702], [517, 704], [517, 718], [517, 720], [517, 729], [518, 639], [518, 647], [518, 655], [518, 667], [518, 687], [518, 691], [518, 703], [518, 705], [518, 719], [518, 721], [519, 638], [519, 644], [519, 648], [519, 680], [519, 688], [519, 692], [519, 704], [519, 706], [519, 721], [519, 723], [520, 626], [520, 627], [520, 636], [520, 639], [520, 645], [520, 649], [520, 656], [520, 658], [520, 670], [520, 689], [520, 693], [520, 705], [520, 707], [520, 722], [520, 724], [521, 628], [521, 629], [521, 636], [521, 640], [521, 646], [521, 650], [521, 661], [521, 673], [521, 682], [521, 690], [521, 706], [521, 708], [521, 724], [521, 726], [521, 730], [522, 641], [522, 647], [522, 651], [522, 675], [522, 683], [522, 691], [522, 697], [522, 707], [522, 709], [522, 725], [523, 603], [523, 604], [523, 615], [523, 616], [523, 636], [523, 637], [523, 694], [523, 696], [523, 698], [523, 707], [523, 727], [523, 730], [524, 607], [524, 608], [524, 619], [524, 620], [524, 635], [524, 637], [524, 694], [524, 695], [524, 699], [524, 702], [524, 728], [525, 599], [525, 600], [525, 611], [525, 612], [525, 635], [525, 636], [525, 695], [525, 696], [525, 703], [525, 706], [525, 729], [526, 591], [526, 602], [526, 624], [526, 632], [526, 663], [526, 677], [526, 712], [526, 725], [526, 735], [526, 736], [527, 596], [527, 609], [527, 624], [527, 631], [527, 656], [527, 660], [527, 670], [527, 712], [527, 717], [527, 732], [527, 739], [527, 740], [528, 595], [528, 606], [528, 622], [528, 661], [528, 671], [528, 713], [528, 718], [528, 731], [528, 741], [528, 742], [529, 588], [529, 601], [529, 622], [529, 668], [529, 678], [529, 718], [529, 723], [529, 734], [529, 745], [529, 746], [530, 587], [530, 598], [530, 623], [530, 628], [530, 657], [530, 669], [530, 679], [530, 719], [530, 724], [530, 733], [530, 747], [530, 748], [531, 592], [531, 605], [531, 623], [531, 627], [531, 662], [531, 676], [531, 711], [531, 724], [531, 751], [531, 752], [532, 591], [532, 594], [532, 615], [532, 618], [532, 685], [532, 692], [532, 735], [533, 593], [533, 596], [533, 617], [533, 620], [533, 682], [533, 687], [533, 740], [534, 586], [534, 595], [534, 610], [534, 619], [534, 684], [534, 689], [534, 741], [535, 588], [535, 597], [535, 612], [535, 621], [535, 686], [535, 691], [535, 746], [536, 587], [536, 590], [536, 611], [536, 614], [536, 688], [536, 693], [536, 747], [537, 589], [537, 592], [537, 613], [537, 616], [537, 683], [537, 690], [537, 752], [538, 571], [538, 572], [538, 583], [538, 590], [538, 594], [538, 627], [538, 647], [538, 653], [538, 699], [538, 707], [538, 737], [538, 756], [539, 573], [539, 574], [539, 591], [539, 595], [539, 629], [539, 648], [539, 654], [539, 700], [539, 708], [539, 731], [539, 736], [539, 766], [540, 574], [540, 575], [540, 592], [540, 596], [540, 634], [540, 649], [540, 655], [540, 697], [540, 701], [540, 739], [540, 767], [541, 576], [541, 577], [541, 584], [541, 593], [541, 597], [541, 644], [541, 650], [541, 698], [541, 702], [541, 738], [541, 757], [541, 759], [542, 577], [542, 578], [542, 585], [542, 586], [542, 594], [542, 631], [542, 645], [542, 651], [542, 699], [542, 703], [542, 743], [542, 758], [542, 760], [543, 562], [543, 579], [543, 587], [543, 595], [543, 633], [543, 646], [543, 652], [543, 700], [543, 704], [543, 733], [543, 742], [543, 768], [544, 562], [544, 563], [544, 588], [544, 596], [544, 626], [544, 647], [544, 653], [544, 701], [544, 705], [544, 732], [544, 745], [544, 769], [545, 564], [545, 565], [545, 580], [545, 589], [545, 597], [545, 628], [545, 648], [545, 654], [545, 702], [545, 706], [545, 744], [545, 761], [545, 763], [546, 565], [546, 566], [546, 581], [546, 586], [546, 590], [546, 649], [546, 655], [546, 703], [546, 707], [546, 749], [546, 762], [546, 764], [547, 567], [547, 568], [547, 587], [547, 591], [547, 625], [547, 644], [547, 650], [547, 704], [547, 708], [547, 748], [547, 770], [548, 568], [548, 569], [548, 588], [548, 592], [548, 630], [548, 645], [548, 651], [548, 697], [548, 705], [548, 734], [548, 751], [548, 771], [549, 570], [549, 571], [549, 582], [549, 589], [549, 593], [549, 632], [549, 646], [549, 652], [549, 698], [549, 706], [549, 750], [549, 765], [550, 571], [550, 574], [550, 584], [550, 606], [550, 664], [550, 678], [550, 685], [550, 692], [550, 738], [550, 751], [550, 753], [550, 772], [551, 572], [551, 575], [551, 602], [551, 607], [551, 666], [551, 679], [551, 693], [551, 735], [551, 756], [551, 767], [551, 773], [552, 573], [552, 576], [552, 604], [552, 609], [552, 668], [552, 681], [552, 686], [552, 740], [552, 757], [552, 766], [552, 774], [553, 574], [553, 577], [553, 583], [553, 605], [553, 659], [553, 669], [553, 682], [553, 687], [553, 737], [553, 742], [553, 753], [553, 758], [553, 775], [554, 562], [554, 577], [554, 580], [554, 598], [554, 662], [554, 672], [554, 684], [554, 689], [554, 739], [554, 744], [554, 754], [554, 759], [554, 776], [555, 563], [555, 578], [555, 599], [555, 606], [555, 663], [555, 674], [555, 685], [555, 741], [555, 760], [555, 769], [555, 777], [556, 564], [556, 579], [556, 601], [556, 608], [556, 665], [556, 676], [556, 690], [556, 746], [556, 761], [556, 768], [556, 778], [557, 562], [557, 565], [557, 585], [557, 609], [557, 667], [557, 677], [557, 686], [557, 691], [557, 743], [557, 748], [557, 754], [557, 762], [557, 779], [558, 565], [558, 568], [558, 582], [558, 602], [558, 670], [558, 680], [558, 688], [558, 693], [558, 745], [558, 750], [558, 755], [558, 763], [558, 780], [559, 566], [559, 569], [559, 598], [559, 603], [559, 658], [559, 671], [559, 689], [559, 747], [559, 764], [559, 771], [559, 781], [560, 567], [560, 570], [560, 600], [560, 605], [560, 660], [560, 673], [560, 682], [560, 752], [560, 765], [560, 770], [560, 782], [561, 568], [561, 571], [561, 581], [561, 601], [561, 661], [561, 675], [561, 683], [561, 690], [561, 736], [561, 749], [561, 755], [561, 783], [562, 568], [562, 574], [562, 626], [562, 633], [562, 646], [562, 653], [562, 662], [562, 677], [562, 712], [562, 724], [562, 737], [562, 750], [562, 772], [562, 783], [562, 784], [562, 795], [562, 796], [562, 807], [563, 569], [563, 575], [563, 606], [563, 647], [563, 663], [563, 713], [563, 725], [563, 773], [563, 796], [564, 570], [564, 576], [564, 608], [564, 628], [564, 654], [564, 665], [564, 679], [564, 714], [564, 726], [564, 752], [564, 774], [564, 797], [565, 571], [565, 577], [565, 602], [565, 609], [565, 648], [565, 655], [565, 667], [565, 680], [565, 709], [565, 715], [565, 736], [565, 739], [565, 756], [565, 757], [565, 772], [565, 775], [565, 785], [565, 786], [566, 572], [566, 578], [566, 603], [566, 649], [566, 658], [566, 668], [566, 710], [566, 716], [566, 741], [566, 758], [566, 773], [566, 798], [567, 573], [567, 579], [567, 605], [567, 644], [567, 660], [567, 711], [567, 717], [567, 774], [567, 799], [568, 574], [568, 625], [568, 630], [568, 645], [568, 650], [568, 661], [568, 670], [568, 712], [568, 718], [568, 738], [568, 743], [568, 775], [568, 776], [568, 787], [568, 788], [568, 799], [568, 800], [569, 575], [569, 598], [569, 651], [569, 671], [569, 713], [569, 719], [569, 777], [569, 800], [570, 576], [570, 600], [570, 632], [570, 646], [570, 663], [570, 673], [570, 714], [570, 720], [570, 740], [570, 759], [570, 778], [570, 801], [571, 577], [571, 601], [571, 606], [571, 647], [571, 652], [571, 664], [571, 675], [571, 715], [571, 721], [571, 742], [571, 745], [571, 760], [571, 761], [571, 776], [571, 779], [571, 789], [571, 790], [572, 578], [572, 607], [572, 627], [572, 653], [572, 666], [572, 676], [572, 716], [572, 722], [572, 747], [572, 762], [572, 777], [572, 802], [573, 579], [573, 609], [573, 648], [573, 668], [573, 717], [573, 723], [573, 778], [573, 803], [574, 629], [574, 634], [574, 649], [574, 654], [574, 669], [574, 678], [574, 718], [574, 724], [574, 744], [574, 749], [574, 779], [574, 780], [574, 791], [574, 792], [574, 803], [574, 804], [575, 602], [575, 655], [575, 679], [575, 719], [575, 725], [575, 781], [575, 804], [576, 604], [576, 650], [576, 671], [576, 681], [576, 720], [576, 726], [576, 746], [576, 763], [576, 782], [576, 805], [577, 598], [577, 605], [577, 644], [577, 651], [577, 659], [577, 672], [577, 709], [577, 721], [577, 748], [577, 751], [577, 764], [577, 765], [577, 780], [577, 783], [577, 793], [577, 794], [578, 599], [578, 631], [578, 645], [578, 660], [578, 674], [578, 710], [578, 722], [578, 735], [578, 781], [578, 806], [579, 601], [579, 652], [579, 676], [579, 711], [579, 723], [579, 782], [579, 807], [580, 589], [580, 619], [580, 642], [580, 684], [580, 765], [580, 785], [580, 797], [581, 586], [581, 616], [581, 641], [581, 683], [581, 727], [581, 760], [581, 786], [581, 798], [582, 593], [582, 611], [582, 638], [582, 688], [582, 729], [582, 757], [582, 789], [582, 801], [583, 590], [583, 620], [583, 643], [583, 687], [583, 728], [583, 764], [583, 790], [583, 802], [584, 597], [584, 615], [584, 640], [584, 692], [584, 730], [584, 761], [584, 793], [584, 805], [585, 594], [585, 612], [585, 639], [585, 691], [585, 756], [585, 794], [585, 806], [586, 619], [586, 626], [586, 699], [586, 707], [586, 773], [586, 783], [586, 785], [586, 808], [587, 614], [587, 628], [587, 657], [587, 700], [587, 708], [587, 775], [587, 784], [587, 814], [588, 621], [588, 697], [588, 701], [588, 772], [588, 787], [588, 815], [589, 616], [589, 625], [589, 698], [589, 702], [589, 774], [589, 776], [589, 786], [589, 809], [590, 611], [590, 630], [590, 699], [590, 703], [590, 775], [590, 777], [590, 789], [590, 810], [591, 618], [591, 632], [591, 700], [591, 704], [591, 779], [591, 788], [591, 816], [592, 613], [592, 627], [592, 701], [592, 705], [592, 776], [592, 791], [592, 817], [593, 620], [593, 629], [593, 702], [593, 706], [593, 778], [593, 780], [593, 790], [593, 811], [594, 615], [594, 634], [594, 703], [594, 707], [594, 779], [594, 781], [594, 793], [594, 812], [595, 610], [595, 704], [595, 708], [595, 783], [595, 792], [595, 818], [596, 617], [596, 631], [596, 656], [596, 697], [596, 705], [596, 780], [596, 795], [596, 819], [597, 612], [597, 633], [597, 698], [597, 706], [597, 772], [597, 782], [597, 794], [597, 813], [598, 623], [598, 664], [598, 679], [598, 751], [598, 796], [598, 822], [599, 666], [599, 680], [599, 735], [599, 798], [599, 808], [599, 815], [599, 823], [600, 667], [600, 681], [600, 740], [600, 797], [600, 809], [600, 814], [600, 823], [601, 622], [601, 659], [601, 668], [601, 742], [601, 799], [601, 824], [602, 624], [602, 663], [602, 672], [602, 739], [602, 800], [602, 828], [603, 664], [603, 674], [603, 741], [603, 802], [603, 810], [603, 817], [603, 829], [604, 665], [604, 675], [604, 746], [604, 801], [604, 811], [604, 816], [604, 829], [605, 623], [605, 667], [605, 676], [605, 748], [605, 803], [605, 830], [606, 622], [606, 671], [606, 680], [606, 745], [606, 804], [606, 834], [607, 658], [607, 672], [607, 747], [607, 806], [607, 812], [607, 819], [607, 835], [608, 659], [608, 673], [608, 752], [608, 805], [608, 813], [608, 818], [608, 835], [609, 624], [609, 660], [609, 675], [609, 736], [609, 807], [609, 836], [610, 613], [610, 684], [610, 713], [611, 620], [611, 636], [611, 693], [611, 726], [611, 814], [612, 615], [612, 635], [612, 686], [612, 716], [612, 815], [613, 683], [613, 711], [614, 617], [614, 688], [614, 719], [615, 637], [615, 685], [615, 714], [615, 816], [616, 619], [616, 636], [616, 690], [616, 722], [616, 817], [617, 687], [617, 717], [618, 621], [618, 692], [618, 725], [619, 635], [619, 689], [619, 720], [619, 818], [620, 637], [620, 682], [620, 710], [620, 819], [621, 691], [621, 723], [622, 821], [622, 837], [623, 825], [623, 827], [624, 831], [624, 833], [625, 654], [625, 774], [625, 814], [625, 838], [625, 844], [626, 649], [626, 656], [626, 732], [626, 773], [626, 815], [626, 839], [626, 845], [627, 645], [627, 728], [627, 767], [627, 776], [627, 839], [628, 650], [628, 768], [628, 775], [628, 840], [629, 646], [629, 731], [629, 778], [629, 816], [629, 840], [629, 846], [630, 653], [630, 734], [630, 777], [630, 817], [630, 841], [630, 847], [631, 649], [631, 769], [631, 780], [631, 841], [632, 654], [632, 729], [632, 770], [632, 779], [632, 842], [633, 650], [633, 657], [633, 733], [633, 782], [633, 818], [633, 842], [633, 848], [634, 645], [634, 781], [634, 819], [634, 843], [634, 849], [635, 694], [635, 696], [635, 797], [635, 806], [635, 808], [635, 813], [636, 694], [636, 695], [636, 798], [636, 801], [636, 809], [636, 810], [637, 695], [637, 696], [637, 802], [637, 805], [637, 811], [637, 812], [638, 665], [638, 680], [638, 757], [638, 844], [639, 658], [639, 667], [639, 756], [639, 758], [639, 845], [640, 664], [640, 673], [640, 759], [640, 761], [640, 846], [641, 666], [641, 675], [641, 760], [641, 762], [641, 847], [642, 672], [642, 681], [642, 763], [642, 765], [642, 848], [643, 659], [643, 674], [643, 764], [643, 849], [644, 684], [644, 692], [644, 738], [644, 765], [644, 844], [644, 850], [645, 685], [645, 693], [645, 751], [645, 815], [646, 682], [646, 686], [646, 742], [646, 759], [646, 814], [647, 683], [647, 687], [647, 737], [647, 760], [647, 845], [647, 851], [648, 684], [648, 688], [648, 744], [648, 757], [648, 846], [649, 685], [649, 689], [649, 739], [649, 758], [649, 817], [650, 686], [650, 690], [650, 748], [650, 763], [650, 816], [651, 687], [651, 691], [651, 743], [651, 764], [651, 847], [652, 688], [652, 692], [652, 750], [652, 761], [652, 848], [652, 852], [653, 689], [653, 693], [653, 745], [653, 762], [653, 819], [654, 682], [654, 690], [654, 736], [654, 818], [655, 683], [655, 691], [655, 749], [655, 756], [655, 849], [655, 853], [656, 767], [656, 795], [657, 770], [657, 784], [658, 694], [658, 707], [658, 773], [658, 806], [658, 843], [658, 866], [659, 662], [659, 678], [659, 699], [659, 753], [659, 783], [659, 822], [659, 837], [659, 853], [659, 857], [659, 858], [659, 867], [660, 663], [660, 679], [660, 774], [660, 807], [660, 820], [660, 823], [660, 850], [660, 858], [661, 664], [661, 680], [661, 708], [661, 731], [661, 775], [661, 796], [661, 798], [661, 821], [661, 824], [661, 838], [661, 854], [661, 868], [662, 667], [662, 701], [662, 772], [662, 797], [662, 799], [662, 822], [662, 825], [662, 839], [662, 854], [662, 869], [663, 668], [663, 732], [663, 773], [663, 800], [663, 823], [663, 826], [663, 851], [663, 859], [664, 669], [664, 698], [664, 753], [664, 776], [664, 824], [664, 827], [664, 859], [664, 870], [665, 694], [665, 702], [665, 774], [665, 801], [665, 840], [665, 860], [666, 695], [666, 699], [666, 777], [666, 798], [666, 839], [667, 670], [667, 703], [667, 754], [667, 775], [667, 825], [667, 828], [667, 851], [667, 860], [667, 861], [667, 871], [668, 671], [668, 731], [668, 778], [668, 799], [668, 826], [668, 829], [668, 861], [669, 672], [669, 700], [669, 733], [669, 779], [669, 800], [669, 802], [669, 827], [669, 830], [669, 840], [669, 855], [669, 872], [670, 675], [670, 705], [670, 732], [670, 776], [670, 801], [670, 803], [670, 828], [670, 831], [670, 841], [670, 855], [670, 873], [671, 676], [671, 734], [671, 777], [671, 804], [671, 829], [671, 832], [671, 862], [672, 677], [672, 702], [672, 754], [672, 780], [672, 830], [672, 833], [672, 852], [672, 862], [672, 863], [672, 874], [673, 695], [673, 706], [673, 778], [673, 805], [673, 842], [674, 696], [674, 703], [674, 781], [674, 802], [674, 841], [674, 863], [675, 678], [675, 707], [675, 755], [675, 779], [675, 831], [675, 834], [675, 864], [675, 875], [676, 679], [676, 733], [676, 782], [676, 803], [676, 832], [676, 835], [676, 852], [676, 864], [677, 680], [677, 704], [677, 783], [677, 804], [677, 806], [677, 833], [677, 836], [677, 842], [677, 856], [677, 876], [678, 697], [678, 734], [678, 780], [678, 805], [678, 807], [678, 834], [678, 837], [678, 843], [678, 856], [678, 877], [679, 781], [679, 796], [679, 820], [679, 835], [679, 853], [679, 865], [680, 706], [680, 755], [680, 772], [680, 821], [680, 836], [680, 850], [680, 865], [680, 866], [680, 878], [681, 696], [681, 698], [681, 782], [681, 797], [681, 838], [681, 857], [682, 765], [682, 795], [682, 814], [682, 857], [683, 786], [683, 858], [684, 759], [684, 785], [684, 859], [685, 727], [685, 760], [685, 788], [685, 815], [686, 729], [686, 757], [686, 787], [686, 816], [686, 860], [687, 758], [687, 790], [687, 861], [687, 879], [688, 763], [688, 789], [688, 862], [688, 880], [689, 728], [689, 764], [689, 792], [689, 817], [689, 863], [690, 730], [690, 761], [690, 791], [690, 818], [691, 762], [691, 794], [691, 864], [692, 793], [692, 865], [693, 756], [693, 784], [693, 819], [693, 866], [697, 712], [697, 725], [697, 843], [698, 709], [698, 714], [698, 757], [698, 809], [698, 838], [699, 710], [699, 715], [699, 756], [699, 758], [699, 808], [699, 839], [700, 712], [700, 717], [700, 840], [701, 713], [701, 718], [701, 839], [702, 715], [702, 720], [702, 759], [702, 761], [702, 811], [702, 840], [702, 880], [703, 716], [703, 721], [703, 760], [703, 762], [703, 810], [703, 841], [703, 879], [704, 718], [704, 723], [704, 842], [705, 719], [705, 724], [705, 841], [706, 721], [706, 726], [706, 763], [706, 765], [706, 813], [706, 842], [707, 709], [707, 722], [707, 764], [707, 812], [707, 843], [708, 711], [708, 724], [708, 838], [709, 737], [709, 750], [709, 785], [709, 794], [709, 858], [709, 865], [710, 837], [710, 866], [711, 752], [711, 824], [711, 858], [712, 736], [712, 739], [712, 784], [712, 787], [713, 741], [713, 822], [713, 859], [714, 827], [714, 860], [715, 738], [715, 743], [715, 786], [715, 789], [715, 859], [715, 861], [716, 825], [717, 740], [717, 830], [717, 861], [718, 742], [718, 745], [718, 788], [718, 791], [719, 747], [719, 828], [719, 862], [720, 833], [721, 744], [721, 749], [721, 790], [721, 793], [721, 862], [721, 864], [722, 831], [722, 863], [723, 746], [723, 836], [723, 864], [724, 748], [724, 751], [724, 792], [724, 795], [725, 735], [725, 834], [725, 865], [726, 821], [726, 857], [727, 798], [728, 802], [729, 801], [730, 805], [731, 766], [731, 788], [732, 769], [732, 787], [732, 851], [733, 768], [733, 792], [733, 852], [734, 771], [734, 791], [735, 773], [735, 806], [735, 865], [736, 766], [736, 783], [736, 878], [737, 775], [737, 796], [737, 845], [737, 858], [737, 869], [738, 772], [738, 799], [738, 844], [738, 859], [738, 868], [739, 767], [739, 776], [739, 871], [740, 774], [740, 801], [740, 861], [741, 777], [741, 798], [741, 859], [742, 768], [742, 775], [742, 870], [743, 779], [743, 800], [743, 847], [743, 861], [743, 873], [744, 776], [744, 803], [744, 846], [744, 862], [744, 872], [745, 769], [745, 780], [745, 875], [746, 778], [746, 805], [746, 864], [747, 781], [747, 802], [747, 862], [748, 770], [748, 779], [748, 874], [749, 783], [749, 804], [749, 849], [749, 864], [749, 877], [750, 780], [750, 807], [750, 848], [750, 865], [750, 876], [751, 771], [751, 772], [751, 867], [752, 782], [752, 797], [752, 858], [753, 986], [754, 988], [755, 990], [756, 767], [756, 853], [756, 866], [757, 766], [757, 860], [758, 810], [758, 845], [758, 861], [759, 809], [759, 846], [759, 859], [760, 769], [760, 851], [760, 879], [761, 768], [761, 852], [761, 880], [762, 812], [762, 847], [762, 864], [763, 811], [763, 848], [763, 862], [764, 771], [764, 863], [765, 770], [765, 850], [765, 857], [766, 838], [767, 839], [768, 840], [769, 841], [770, 842], [771, 843], [772, 807], [772, 815], [772, 854], [773, 796], [773, 798], [773, 845], [774, 797], [774, 799], [774, 844], [775, 800], [775, 814], [775, 854], [776, 799], [776, 817], [776, 855], [777, 800], [777, 802], [777, 847], [778, 801], [778, 803], [778, 846], [779, 804], [779, 816], [779, 855], [780, 803], [780, 819], [780, 856], [781, 804], [781, 806], [781, 849], [782, 805], [782, 807], [782, 848], [783, 796], [783, 818], [783, 856], [784, 795], [784, 814], [785, 786], [785, 850], [786, 851], [787, 788], [787, 815], [788, 816], [789, 790], [789, 880], [790, 879], [791, 792], [791, 817], [792, 818], [793, 794], [793, 852], [794, 853], [795, 819], [796, 822], [796, 865], [796, 881], [797, 820], [797, 857], [798, 826], [799, 824], [799, 861], [799, 882], [800, 828], [800, 859], [800, 882], [801, 826], [801, 860], [802, 832], [802, 863], [803, 830], [803, 864], [803, 883], [804, 834], [804, 862], [804, 883], [805, 832], [806, 820], [806, 866], [807, 836], [807, 858], [807, 881], [808, 815], [809, 814], [810, 817], [811, 816], [812, 819], [813, 818], [814, 838], [815, 839], [816, 840], [817, 841], [818, 842], [819, 843], [821, 837], [821, 868], [821, 878], [821, 881], [822, 824], [822, 867], [822, 869], [824, 868], [824, 870], [825, 827], [825, 869], [825, 871], [825, 882], [827, 870], [827, 872], [827, 882], [828, 830], [828, 871], [828, 873], [830, 872], [830, 874], [831, 833], [831, 873], [831, 875], [831, 883], [833, 874], [833, 876], [833, 883], [834, 836], [834, 875], [834, 877], [836, 876], [836, 878], [837, 867], [837, 877], [837, 881], [844, 850], [845, 851], [846, 880], [847, 879], [848, 852], [849, 853], [850, 865], [851, 861], [852, 862], [853, 858], [854, 940], [855, 942], [856, 944], [858, 881], [859, 882], [861, 879], [861, 882], [862, 880], [862, 883], [864, 883], [865, 881], [867, 991], [868, 992], [869, 993], [870, 994], [871, 995], [872, 996], [873, 997], [874, 998], [875, 999], [876, 1000], [877, 1001], [878, 1002], [884, 899], [884, 906], [884, 915], [884, 920], [884, 932], [885, 901], [885, 903], [885, 907], [885, 921], [885, 936], [886, 902], [886, 904], [886, 914], [886, 919], [886, 926], [886, 937], [887, 896], [887, 897], [887, 903], [887, 910], [887, 917], [887, 923], [887, 929], [887, 938], [887, 939], [887, 940], [887, 945], [887, 946], [887, 957], [888, 897], [888, 898], [888, 905], [888, 907], [888, 912], [888, 928], [888, 931], [888, 940], [888, 941], [888, 947], [888, 948], [888, 958], [889, 893], [889, 898], [889, 900], [889, 909], [889, 914], [889, 924], [889, 930], [889, 933], [889, 941], [889, 942], [889, 949], [889, 950], [889, 959], [890, 893], [890, 894], [890, 902], [890, 906], [890, 911], [890, 916], [890, 925], [890, 926], [890, 932], [890, 935], [890, 942], [890, 943], [890, 951], [890, 952], [890, 960], [891, 894], [891, 895], [891, 899], [891, 904], [891, 913], [891, 918], [891, 934], [891, 937], [891, 943], [891, 944], [891, 953], [891, 954], [891, 961], [892, 895], [892, 896], [892, 901], [892, 908], [892, 915], [892, 922], [892, 927], [892, 936], [892, 939], [892, 944], [892, 955], [892, 956], [892, 962], [893, 895], [893, 897], [893, 902], [893, 909], [893, 916], [893, 928], [893, 937], [893, 957], [893, 962], [893, 968], [894, 896], [894, 898], [894, 904], [894, 906], [894, 911], [894, 918], [894, 919], [894, 920], [894, 927], [894, 930], [894, 957], [894, 958], [894, 963], [895, 897], [895, 899], [895, 908], [895, 913], [895, 929], [895, 932], [895, 958], [895, 959], [895, 964], [896, 898], [896, 901], [896, 910], [896, 915], [896, 920], [896, 921], [896, 931], [896, 934], [896, 959], [896, 960], [897, 903], [897, 912], [897, 917], [897, 933], [897, 936], [897, 960], [897, 961], [897, 965], [897, 966], [898, 900], [898, 905], [898, 907], [898, 914], [898, 919], [898, 921], [898, 935], [898, 938], [898, 961], [898, 962], [898, 967], [899, 917], [899, 929], [899, 956], [899, 963], [899, 968], [899, 969], [900, 910], [900, 924], [900, 938], [900, 947], [900, 970], [901, 907], [901, 922], [901, 931], [901, 946], [901, 963], [901, 964], [901, 971], [902, 912], [902, 925], [902, 928], [902, 949], [902, 972], [903, 909], [903, 923], [903, 933], [903, 948], [903, 964], [903, 973], [904, 914], [904, 930], [904, 951], [904, 974], [905, 911], [905, 935], [905, 950], [905, 965], [905, 975], [906, 915], [906, 926], [906, 927], [906, 954], [906, 967], [906, 968], [906, 979], [907, 909], [907, 917], [907, 940], [907, 962], [907, 963], [907, 970], [907, 980], [907, 981], [907, 985], [907, 991], [908, 910], [908, 918], [908, 939], [908, 958], [908, 969], [908, 971], [908, 981], [908, 986], [908, 992], [909, 911], [909, 941], [909, 957], [909, 964], [909, 970], [909, 972], [909, 986], [909, 993], [910, 912], [910, 940], [910, 959], [910, 971], [910, 973], [910, 987], [910, 994], [911, 913], [911, 942], [911, 958], [911, 972], [911, 974], [911, 982], [911, 987], [911, 995], [912, 914], [912, 941], [912, 960], [912, 973], [912, 975], [912, 982], [912, 988], [912, 996], [913, 915], [913, 943], [913, 959], [913, 965], [913, 974], [913, 976], [913, 983], [913, 988], [913, 997], [914, 916], [914, 942], [914, 961], [914, 975], [914, 977], [914, 983], [914, 989], [914, 998], [915, 917], [915, 944], [915, 960], [915, 967], [915, 976], [915, 978], [915, 989], [915, 999], [916, 918], [916, 943], [916, 962], [916, 966], [916, 977], [916, 979], [916, 990], [916, 1000], [917, 939], [917, 961], [917, 968], [917, 978], [917, 980], [917, 984], [917, 990], [917, 1001], [918, 944], [918, 957], [918, 969], [918, 979], [918, 984], [918, 985], [918, 1002], [919, 928], [919, 937], [920, 929], [920, 932], [921, 933], [921, 936], [922, 929], [922, 956], [922, 985], [922, 992], [923, 931], [923, 946], [923, 986], [923, 994], [924, 928], [924, 949], [924, 987], [924, 993], [925, 930], [925, 951], [925, 988], [925, 995], [926, 937], [926, 952], [926, 989], [926, 1000], [927, 962], [927, 963], [927, 1002], [927, 1003], [928, 958], [928, 993], [929, 957], [929, 964], [929, 992], [930, 959], [930, 995], [931, 958], [931, 994], [931, 1004], [932, 960], [932, 997], [933, 959], [933, 965], [933, 996], [934, 961], [934, 999], [934, 1005], [935, 960], [935, 967], [935, 998], [936, 962], [936, 966], [936, 1001], [937, 961], [937, 968], [937, 1000], [937, 1006], [938, 957], [938, 991], [939, 946], [939, 955], [939, 981], [939, 984], [939, 1003], [940, 945], [940, 948], [940, 981], [941, 947], [941, 950], [941, 982], [941, 1004], [942, 949], [942, 952], [942, 982], [942, 983], [943, 951], [943, 954], [943, 983], [943, 1005], [944, 953], [944, 956], [944, 984], [944, 1006], [945, 980], [945, 992], [945, 1007], [946, 971], [946, 991], [946, 1008], [947, 970], [947, 994], [947, 1009], [948, 973], [948, 993], [948, 1010], [949, 972], [949, 996], [949, 1011], [950, 975], [950, 995], [950, 1012], [951, 974], [951, 998], [951, 1013], [952, 977], [952, 997], [952, 1014], [953, 976], [953, 1000], [953, 1015], [954, 979], [954, 999], [954, 1016], [955, 978], [955, 1002], [955, 1017], [956, 969], [956, 1001], [956, 1018], [957, 968], [957, 985], [957, 986], [957, 1003], [957, 1007], [957, 1008], [958, 963], [958, 986], [958, 987], [958, 1009], [958, 1010], [959, 964], [959, 987], [959, 988], [959, 1004], [959, 1011], [959, 1012], [960, 966], [960, 988], [960, 989], [960, 1005], [960, 1013], [960, 1014], [961, 965], [961, 989], [961, 990], [961, 1015], [961, 1016], [962, 967], [962, 985], [962, 990], [962, 1006], [962, 1017], [962, 1018], [963, 969], [963, 1008], [964, 971], [964, 1004], [964, 1010], [965, 975], [965, 1014], [966, 978], [966, 1005], [966, 1015], [967, 977], [967, 1006], [967, 1016], [968, 979], [968, 1003], [968, 1018], [969, 980], [969, 992], [969, 1002], [969, 1008], [970, 971], [970, 991], [970, 993], [970, 1007], [971, 992], [971, 994], [971, 1010], [972, 973], [972, 993], [972, 995], [972, 1009], [973, 994], [973, 996], [973, 1012], [974, 975], [974, 995], [974, 997], [974, 1011], [975, 996], [975, 998], [975, 1014], [976, 977], [976, 997], [976, 999], [976, 1013], [977, 998], [977, 1000], [977, 1016], [978, 979], [978, 999], [978, 1001], [978, 1015], [979, 1000], [979, 1002], [979, 1018], [980, 991], [980, 1001], [980, 1017], [981, 985], [981, 986], [982, 987], [982, 988], [983, 988], [983, 989], [984, 985], [984, 990], [985, 1007], [985, 1018], [986, 1008], [986, 1009], [987, 1010], [987, 1011], [988, 1012], [988, 1013], [989, 1014], [989, 1015], [990, 1016], [990, 1017], [991, 1008], [992, 1007], [993, 1010], [994, 1004], [994, 1009], [995, 1012], [996, 1011], [997, 1014], [998, 1013], [999, 1005], [999, 1016], [1000, 1006], [1000, 1015], [1001, 1018], [1002, 1003], [1002, 1017], [1003, 1008], [1004, 1012], [1005, 1013], [1006, 1018]], \"cross\": 2544}, {\"name\": \"C7_G3x2+10\", \"n\": 4262, \"edges\": 34179, \"vtx\": [[-1.0, 1.0408340855860843e-16], [-0.3765101981412665, -0.7818314824680297], [0.41239324334649896, -0.16731430613464227], [0.9123932433464991, 0.6987110976497966], [0.33967068575833653, -0.121038176231964], [0.500000000000002, 0.8660254037844397], [-0.13974330102097354, -0.5098611692036452], [0.7952434915444768, -0.1551786383000625], [1.2952434915444773, 0.7108467654843759], [0.7225209339563146, -0.10890250839738473], [-0.04442719421385943, -0.2947551744109042], [0.26122556688144516, 0.6573878766243993], [-0.3622642349772881, -0.12444360584363032], [-0.500000000000002, 0.8660254037844397], [-0.8396706857583365, 0.9870635800164027], [-0.3396706857583362, 1.8530889838008409], [0.5612981821440828, 2.2869727229183994], [-0.17175368968574334, 1.6067999851474801], [-1.137735765022712, 0.9904690096280693], [-0.6377357650227119, 1.8564944134125079], [0.05076724427057089, 2.581727897329304], [-1.3653410243663948, 0.9308737486442044], [-0.38793457358534633, 1.1422422183164662], [-1.3602566989790266, 1.3758865729880843], [-1.5, 0.8660254037844387], [-0.7110965585122344, 1.480542580117826], [0.28890344148776537, 1.4805425801178262], [-1.6234898018587332, 0.7818314824680299], [-0.7632331028797068, 0.27197031326438426], [-0.26323310287970647, 1.1379957170488229], [-0.1885030092932825, 0.14079191986764267], [-1.7330518718298262, 0.6801727377709195], [-0.8727951728507999, 0.17031156856727397], [0.12720482714920012, 0.17031156856727386], [-0.4962849747095332, 0.9521430510353037], [-1.774657478323787, 0.6323810491128204], [-1.3510950612024157, -0.27348587201980457], [-1.488830826225128, 0.716983137608264], [-0.4888308262251281, 0.7169831376082637], [-1.8262387743159947, 0.5633200580636221], [-1.326238774315995, 1.4293454618480608], [-0.8262387743159948, 0.563320058063622], [-5.551115123125783e-17, 2.7755575615628914e-17], [-1.900968867902419, 0.433883739117558], [-0.9659820753369686, 0.7885662700211411], [0.011424375444079815, 0.9999347396934029], [-1.4009688679024193, 1.2999091429019964], [-0.42356241712137077, 1.5112776125742586], [-0.43498679256545064, 0.511342872880856], [-1.97232212539368, 0.23364435467161776], [-1.7498011914373657, 1.2085722668534413], [-1.3262387743159947, 0.3027053457208162], [-0.3262387743159949, 0.30270534572081664], [-2.3376631497600746, 1.164518103315822], [-1.477406450781048, 0.6546569341121767], [-0.4774064507810482, 0.6546569341121768], [-1.9888308262251284, 0.14904226617617466], [-1.0114243754440801, 0.3604107358484366], [-0.5114243754440801, 1.226436139632875], [-0.28890344148776576, 0.25150822745105145], [-1.9009688679024195, -0.43388373911755773], [-1.4774064507810483, -1.339750660250183], [-0.9774064507810484, -0.4737252564657444], [-1.826238774315995, -0.5633200580636221], [-0.9659820753369682, -1.0731812272672676], [-0.8056527610953048, -0.0861176472508646], [-1.7330518718298265, -0.6801727377709192], [-1.0612981821440828, -1.4209473191339608], [-0.16032931424166352, -0.9870635800164026], [-1.572722557588163, -0.8197492738817607], [-0.6377357650227122, -0.46506674297817735], [-0.13773576502271212, 0.4009586608062611], [-0.3602566989790263, 1.3758865729880845], [-1.4123932433464992, 0.16731430613464254], [-0.9123932433464992, 1.033339709919081], [-1.4979924640017386, 0.17745452329941977], [-0.5205860132206903, 0.38882299297168155], [-1.7952434915444768, 0.15517863830006318], [-0.8602566989790262, 0.5098611692036462], [-1.365341024366395, -0.9308737486442041], [-1.6491601404667924, 0.028004089944210164], [-0.6491601404667923, 0.028004089944210206], [-0.4888308262251283, 1.0150676699606132], [-1.0114243754440797, -0.9999347396934024], [-0.05585156965793914, -1.2946899141043064], [0.9215548811231093, -1.0833214444320447], [0.6377357650227125, -0.12444360584363029], [-0.5114243754440798, -0.13390933590896398], [0.4659820753369688, 0.07745913376329817], [-1.7444762472739057, -0.31976200192248294], [-1.2444762472739062, 0.5462634018619557], [-0.925269906413576, -0.9972037971811801], [-0.06501320743454925, -1.5070649663848257], [-0.6885030092932829, -0.7252334839167959], [-0.7774790660436861, -0.9749279121818236], [-0.27747906604368583, -0.1089025083973848], [0.6999273847373626, 0.102465961274877], [-0.6346589756336057, -0.9308737486442045], [0.30032781693184485, -0.5761912177406217], [-0.13465897563360546, -0.06484834485976568], [0.48883082622512825, 0.7169831376082639], [-0.5764375828786295, -0.905866921132625], [0.21246585860913658, -0.2913497447992376], [0.7124658586091364, 0.5746756589852008], [-0.651167676465053, 0.09133687604855498], [-0.5, -0.8660254037844386], [0.3716810744231065, -0.3759518332946817], [0.7952434915444775, -1.2818187544273068], [0.9555728057861408, -0.294755174410904], [1.455572805786141, 0.5712702293735346], [0.6234898018587334, -0.7818314824680297], [-0.1511676764650539, -0.1494504333552093], [0.21417334790134157, 0.7814233152889947], [-0.21618088389960288, 0.2052320975483732], [0.7838191161003969, 0.2052320975483733], [1.772649942325526, 0.056189831372198684], [0.7838191161003976, -0.09285243480397559], [-1.1511676764650534, -0.14945043335520924], [-0.7858266520986583, 0.7814233152889951], [-0.9235624171213702, 1.7718923249170642], [0.0764375828786295, 1.7718923249170642], [-0.32824631031425633, -0.7407745813630414], [0.5727225575881629, -0.30689084224548346], [0.4349867925654509, 0.6835781673825857], [0.5953161068071144, 1.6706417473989887], [0.36025669897902635, -0.015541097446245433], [1.093308570808853, -0.695713835217165], [1.5933085708088526, 0.17031156856727378], [-0.2535162167278322, 0.2564292158181386], [0.7464837832721678, 0.25642921581813877], [1.0521365443674724, 1.2085722668534418], [0.2774790660436852, 1.8409533159662623], [-0.40297640390068035, 0.2564292158181386], [0.49799246400173847, -0.1774545232994193], [-0.0747300935864243, -0.9972037971811801], [-0.4659820753369682, 0.24969442826502775], [0.5303028993725654, 0.16357678101416318], [-0.042419658215597456, -0.6561724928675973], [-0.6120654264146533, 0.2181032572253728], [-0.11206542641465322, 1.0841286610098113], [-0.8451172982444796, 0.40395592323889207], [-0.40096886790241915, 1.2999091429019969], [-0.26694812817017377, -0.6801727377709192], [0.7330518718298262, -0.6801727377709192], [-0.12720482714920012, -0.17031156856727353], [0.3727951728508001, 0.6957138352171646], [0.7104583226108747, -0.4688042680986573], [1.3989613319041574, 0.2564292158181387], [1.8989613319041574, 1.1224546196025773], [1.326238774315995, 0.30270534572081653], [0.4215548811231089, 0.04506074614587685], [-0.5050843253873684, 0.4210125794405588], [0.4949156746126313, 0.42101257944055864], [-0.5507672442705709, 0.27870510081749506], [-0.050767244270570666, 1.1447305046019336], [0.17175368968574356, 2.119658416783757], [-0.1737612256840051, -0.5633200580636221], [0.8262387743159949, -0.5633200580636224], [0.6885030092932831, 0.4271489515644469], [0.06501320743454958, -0.35468253090358276], [0.7612255668814454, -0.20863752716003892], [1.2612255668814456, 0.6573878766243992], [2.1214822658604717, 0.14752670742075402], [1.1234898018587338, 0.0841939213164091], [0.326238774315995, 0.3027053457208164], [-0.6717536896857434, 0.23937255961647197], [-1.2952434915444768, 1.021204042084502], [0.8602566989790266, -0.5098611692036455], [0.9349867925654507, -1.5070649663848257], [0.03401792466303144, -1.0731812272672676], [0.19434723890469496, -0.0861176472508646], [0.8376631497600747, -0.29849269953138335], [0.21417334790134163, 0.4833387829366464], [0.9888308262251284, -0.14904226617617378], [0.36025669897902657, 0.35616423458079316], [-0.4659820753369681, -0.20715582348282907], [0.4349867925654513, -0.6410395626003871], [0.16590946007433094, 0.44228188183165795], [-0.406813097513832, -0.3774673920501024], [0.2649405921719119, -1.1182419734131441], [0.02058601322068998, 0.4772024108127572], [1.02058601322069, 0.47720241081275716], [1.095316106807114, -0.520001386368423], [1.400968867902419, 0.43214166466688086], [-0.09903113209758085, -0.4338837391175581], [0.9009688679024193, -0.43388373911755784], [-0.07135325749126087, -0.20023938444594025], [0.6171497518020221, 0.5249940994708557], [-0.3716810744231066, 0.37595183329468146], [0.7726499423255255, 0.056189831372198684], [1.2726499423255255, -0.8098355724122399], [0.2838191161003971, -0.9588778385884145], [0.40096886790241926, 0.43214166466688053], [0.0953161068071145, -0.5200013863684229], [-0.1885030092932825, 0.4388764522199912], [-0.3262387743159946, 1.4293454618480608], [0.3451172982444797, 0.46206948054554675], [-0.3879345735853468, -0.2181032572253725], [0.11206542641465372, 0.647922146559066], [0.932979256567189, -0.08338670473864229], [1.7218826980549546, 0.5311304715947451], [1.710458322610875, -0.4688042680986573], [0.7444762472739062, 0.3197620019224833], [1.744476247273906, 0.31976200192248305], [0.8178370407634286, 0.6957138352171652], [0.45557280578614073, 0.5712702293735343], [0.3178370407634288, 1.5617392390016034], [-0.3056527610953047, 0.7799077565335738], [-0.1821629592365714, 0.6957138352171652], [-1.137735765022712, 0.4009586608062609], [-0.011169173774871477, -0.1490422661761745], [0.6773338355184111, 0.5761912177406217], [0.05384403365967794, -0.2056402647274082], [0.9888308262251286, 0.14904226617617455], [1.4888308262251284, 1.015067669960613], [-0.0037150252904665226, -0.08611764725086457], [0.49628497470953326, -0.9521430510353031], [0.21246585860913614, 0.006734787553111188], [0.07473009358642424, 0.9972037971811805], [0.5747300935864244, 1.8632292009656188], [0.9962849747095334, -0.08611764725086468], [0.06964576819905577, 0.28983418604381717], [0.4349867925654507, 1.2207079346880212], [0.9774064507810483, 0.21136846967226194], [1.837663149760075, -0.29849269953138363], [0.8488323235349462, -0.14945043335520936], [-0.12348980185873365, 0.08419392131640882], [0.8262387743159947, 0.5633200580636217], [0.2535162167278321, -0.25642921581813855], [-0.24648378327216797, 0.6095961879663001], [0.42526990641357554, -0.13117839339674142], [-0.9774064507810483, -0.21136846967226244], [-0.6120654264146541, -1.1422422183164669], [-0.11206542641465389, -0.2762168145320285], [0.13773576502271223, -0.990469009628069], [0.9555728057861401, 0.2947551744109042], [1.4555728057861406, 1.160780578195343], [-0.011169173774871477, 0.1490422661761745], [-0.5838917313630343, -0.670707007705586], [-1.3585492096868212, -0.03832595859276547], [-0.35854920968682125, -0.03832595859276561], [-0.02259354921895146, 0.2113684696722622], [0.9123932433464989, 0.5660510005758448], [0.8376631497600747, 1.563254797757025], [-0.13465897563360546, 1.7968991524286428], [0.8653410243663949, 1.7968991524286428], [-0.04442719421385921, 0.2947551744109042], [-0.6171497518020221, -0.5249940994708562], [-0.11714975180202214, 0.34103130431358264], [0.043179562439641606, 1.3280948843299853], [-0.4009688679024195, 0.43214166466688053], [-0.06501320743454941, 0.354682530903583], [-1.02058601322069, 0.6494377053144871], [-0.1489049387975837, 1.1395112758042438], [0.8510950612024164, 1.1395112758042438], [-0.09903113209758085, 0.433883739117558], [0.9009688679024193, 0.43388373911755773], [0.1263113895786323, 1.0662647882303784], [1.0612981821440826, 1.420947319133961], [0.1346589756336044, 1.7968991524286428], [-0.5431795624396419, -0.4620694805455466], [-0.3828502481979778, 0.5249940994708562], [0.1171497518020218, 1.3910195032552948], [-0.45557280578614073, 0.5712702293735346], [-0.12831892557689362, 0.49007357048975686], [0.8716810744231063, 0.4900735704897571], [0.6491601404667919, 1.4650014826715805], [-0.27747906604368566, 1.8409533159662625], [0.7225209339563148, 1.8409533159662623], [-0.1737612256840051, 0.5633200580636218], [-0.7464837832721678, -0.2564292158181386], [1.2498011914373655, -0.34254686306900334], [0.2774790660436859, -0.10890250839738491], [-0.21109655851223452, 0.6145171763333875], [-1.1666693642983752, 0.31976200192248344], [-0.9441484303420609, 1.294689914104307], [-0.7838191161003975, 2.28175349412071], [-0.26694812817017366, 0.6801727377709194], [0.5933085708088528, 0.1703115685672738], [1.0933085708088526, 1.0363369723517124], [0.137735765022712, 1.3310921467626167], [1.0727225575881627, 1.6857746776661993], [-0.3114969907067173, 0.7252334839167962], [0.47740645078104804, 1.3397506602501836], [0.402676357194623, 0.3425468630690035], [-0.42356241712137177, 0.9058669211326256], [0.11206542641465378, -0.18063343721582928], [-0.6625920519091332, 0.45174761189699153], [0.2723947406563174, 0.8064301428005746], [-0.3765101981412665, 0.7818314824680299], [0.6234898018587338, 0.7818314824680297], [0.3396706857583362, 1.740709321056444], [-0.025670338608059062, 0.8098355724122399], [0.42663920651047776, 0.49007357048975675], [0.9266392065104776, 1.3560989742741956], [0.19358733468065137, 0.6759262365032759], [0.8653410243663953, -0.06484834485976558], [0.4888308262251284, 1.0150676699606132], [0.9123932433464996, 0.1092007488279878], [1.4123932433464996, -0.7568246549564508], [0.47740645078104854, 1.0773938734567006], [0.11206542641465389, 2.008267622100905], [0.9723221253936802, 1.4984064528972594], [-1.1102230246251565e-16, 1.7320508075688776], [0.45557280578614057, 1.160780578195343], [0.011424375444079593, 0.26482735853223816], [-0.5612981821440829, -0.5549219153495224], [0.23305187182982612, 1.546198141555358], [0.0727225575881627, 0.5591345615389549], [1.072722557588163, 0.5591345615389549], [0.18850300929328268, 1.5912588877012346], [1.1773338355184113, 1.4422166215250598], [1.1659094600743316, 0.4422818818316576], [-1.3716810744231065, 0.37595183329468174], [-0.41610826863696604, 0.6707070077055863], [0.5727225575881627, 0.8197492738817607], [-1.6773338355184109, -0.5761912177406214], [-0.634658975633605, 0.9308737486442042], [-0.6460833510776844, -0.06906099104919824], [-0.1460833510776846, 0.7969644127352402], [0.8313230997033638, 1.0083328824075022], [-0.6943472389046955, 0.9521430510353035], [-0.19434723890469552, 0.08611764725086496], [-0.034017924663031884, 1.073181227267268], [0.4659820753369681, 1.9392066310517064], [1.326238774315995, 1.4293454618480608], [0.30565276109530476, 0.9521430510353034], [-0.52058601322069, 1.5154631090989255], [0.21246585860913592, 2.1956358468698447], [-0.7774790660436857, 0.9749279121818235], [-0.22363503238400761, 0.14230747862306364], [-1.2124658586091361, 0.29134974479923825], [-0.7889034414877658, -0.614517176333387], [0.22252093395631428, 0.9749279121818236], [1.2188059086658476, 0.8888102649309589], [0.6460833510776848, 0.06906099104919844], [1.4723221253936798, 0.6323810491128203], [0.14916014046679216, 0.8380213138402284], [0.5727225575881629, -0.06784560729239662], [0.7952434915444779, 0.907082304889427], [0.1603293142416634, 0.9870635800164026], [-0.5727225575881629, 0.30689084224548346], [-0.7952434915444769, 1.2818187544273072], [0.137735765022712, 1.198432049688665], [-0.5953161068071143, 0.5182593119177455], [0.36025669897902635, 0.8130144863286499], [0.06129818214408289, 1.4209473191339608], [0.9215548811231091, 0.9110861499303154], [0.2330518718298265, 0.18585266601351919], [0.03201038866477002, 1.4771371505061597], [1.0320103886647702, 1.4771371505061597], [1.3376631497600748, 2.429280201541463], [0.5114243754440798, 1.865960143477841], [-0.05076724427057083, 1.6015807563497906], [0.8209138301525354, 2.0916543268395476], [0.8094894547084561, 1.091719587146145], [0.171753689685743, 0.6266528441679668], [1.1491601404667917, 0.8380213138402286], [0.025670338608057897, 0.9222152351566368], [0.997992464001738, 0.6885708804850192], [-0.5340179246630319, 1.9392066310517064], [-1.434986792565451, 2.3730903701692645], [-0.4349867925654509, 2.3730903701692645], [0.42526990641357554, 1.8632292009656188], [-0.9774064507810484, 1.9775325896444724], [-0.5538440336596776, 1.071665668511847], [-1.1773338355184109, 0.2898341860438171], [-0.925269906413576, 0.9972037971811802], [0.05213654436747228, 1.2085722668534422], [1.0501290083692107, 1.271905052957787], [0.12348980185873348, 1.647856886252469], [1.123489801858734, 1.6478568862524683], [-0.42526990641357587, 1.8632292009656188], [0.5521365443674728, 2.074597670637881], [0.7746574783237867, 1.0996697584560573], [-0.7649405921719126, 1.984267377197583], [-0.09318690248616879, 1.2434927958345414], [0.7330518718298262, 0.6801727377709192], [1.2330518718298265, 1.5461981415553576], [-1.0747300935864241, 0.9972037971811801], [-0.09732364280537575, 1.2085722668534422], [0.40267635719462436, 2.0745976706378806], [0.9026763571946244, 1.2085722668534422], [-0.31149699070671666, 0.4271489515644471], [-0.1511676764650533, 1.4142125315808498], [-0.774657478323787, 0.6323810491128203], [-0.14145079031317864, 0.9043513623772046], [0.8585492096868212, 0.9043513623772044], [-0.09702359609931932, 0.6095961879663001], [0.5747300935864245, -0.13117839339674128], [-0.13773576502271195, 0.9904690096280693], [-0.3602566989790262, 0.015541097446245516], [0.31149699070671766, -0.725233483916796], [-0.16032931424166375, 1.2018374793003312], [-1.1491601404667926, 1.3508797454765058], [-0.22252093395631428, 0.974927912181824], [-0.4046838931928858, 1.6706417473989887], [-0.688503009293283, 2.6295195859874028], [-0.4659820753369688, 1.6545916738055797], [-0.44923275572942933, 1.7157024935448653], [-1.0727225575881625, 0.9338710110768353], [-0.07473009358642402, 0.9972037971811804], [-1.5933085708088526, 1.5617392390016036], [-0.8602566989790262, 0.8815665012306841], [-0.7141733479013409, 0.08460208849544386], [0.2858266520986591, 0.08460208849544351], [0.651167676465054, 1.0154758371396482], [-0.12348980185873315, 1.6478568862524685], [-1.222520933956314, 0.9749279121818235], [-0.36226423497728755, 0.4650667429781781], [0.637735765022712, 0.46506674297817807], [-0.3178370407634287, 0.17031156856727397], [-0.45557280578614046, 1.1607805781953429], [-0.3508398595332076, 1.4650014826715805], [-1.083891731363034, 0.7848287449006612], [-0.08389173136303418, 0.7848287449006615], [-1.2838191161003976, 0.9588778385884146], [-0.42356241712137066, 0.4490166693847688], [-0.3488323235349464, -0.5481871277964114], [0.47740645078104826, -1.1115071858600332], [-0.2949882898752685, 0.8098355724122399], [0.20501171012473118, 1.6758609761966783], [-0.7838191161003969, 1.8249032423728528], [-0.28753414139086375, 0.8727601913375498], [0.7124658586091364, 0.8727601913375497], [-0.21417334790134135, 1.2487120246322316], [-0.2838191161003974, 0.9588778385884142], [0.6717536896857436, 1.2536330129993187], [0.44923275572942956, 0.2787051008174948], [1.4266392065104776, 0.49007357048975664], [-0.30641266531934885, 1.1702463082606762], [0.6935873346806509, 1.1702463082606762], [1.6491601404667917, 1.4650014826715805], [-0.3282463103142565, 1.253633012999319], [1.5727225575881625, 0.8197492738817604], [1.0000000000000002, 5.551115123125783e-17], [-0.4575803417844021, 1.5221978966520362], [-1.030302899372565, 0.7024486227702758], [-0.03401792466303144, 0.6163309755194112], [-1.4215548811231093, 1.9493468482164835], [-0.4215548811231091, 1.9493468482164833], [0.4046838931928857, 1.3860267901528616], [-0.6935873346806509, 0.19009916728116277], [-1.665909460074331, 0.42374352195278064], [-0.6659094600743309, 0.42374352195278053], [0.16666936429837537, -0.31976200192248283], [0.47232212539367985, 0.6323810491128207], [-0.8653410243663948, 1.7968991524286428], [-0.19358733468065098, 1.0561245710656013], [0.1717536896857434, 0.12525082242139707], [-0.8282463103142561, 0.12525082242139712], [-0.7535162167278322, 1.1224546196025773], [0.24648378327216736, 1.122454619602577], [-0.5650132074345493, 1.2207079346880216], [-0.19967218306815515, 0.28983418604381717], [0.8003278169318454, 0.28983418604381694], [-0.022593549218951958, 2.2057760640346222], [0.977406450781048, 2.2057760640346222], [-0.8114969907067172, 1.5912588877012348], [-0.13974330102097332, 0.8504843063381932], [0.8602566989790263, 0.8504843063381932], [-1.5114243754440795, -0.13390933590896384], [-2.2444762472739055, 0.5462634018619559], [-1.4555728057861403, 1.160780578195343], [-0.6272048271491997, 0.6957138352171655], [0.2444762472739066, 1.1857874057069224], [1.2444762472739066, 1.1857874057069218], [1.2330518718298267, 0.18585266601351913], [-0.6234898018587333, 0.7818314824680299], [0.24819127256437312, 1.2719050529577869], [0.23676689712029325, 0.27197031326438403], [-0.9349867925654506, 1.5070649663848263], [-0.5114243754440795, 0.6011980452522012], [0.34883232353494664, 0.09133687604855514], [-1.1234898018587332, 1.6478568862524687], [-0.34883232353494575, 1.0154758371396482], [-1.3376631497600746, 1.164518103315823], [-2.6214822658604717, 0.7184986963636855], [-1.7612255668814452, 0.20863752716003972], [-1.2612255668814454, 1.0746629309444782], [-0.7367668971202928, 0.594055090520055], [-0.7481912725643722, -0.4058796491733475], [0.1234898018587336, 0.08419392131640932], [-1.2330518718298262, 1.546198141555358], [-0.37279517285079944, 1.0363369723517124], [-0.9962849747095328, 1.8181684548197425], [-0.744476247273906, -0.31976200192248305], [0.21109655851223474, -0.6145171763333873], [-1.309489454708455, -0.22569418336170557], [-1.3209138301525345, -1.225628923055108], [-0.4492327557294286, -0.7355553525653513], [-1.274657478323787, -0.2336443546716181], [-1.052136544367473, -1.2085722668534418], [-0.5521365443674728, -0.3425468630690034], [0.09702359609931932, 1.1224546196025775], [-0.5747300935864244, 1.8632292009656184], [-1.699927384737363, 1.6295848462940006], [-1.0281736950516192, 0.8888102649309592], [-0.7225209339563148, 1.8409533159662628], [-1.9123932433464987, 1.6228500587408896], [-0.912393243346499, 1.6228500587408892], [0.08389173136303424, 1.536732411490025], [-1.552136544367473, -0.34254686306900317], [-1.2464837832721685, 0.6095961879663003], [-0.5747300935864247, -0.1311783933967412], [-0.8488323235349464, 0.7746885277358839], [-1.4215548811231091, -0.045060746145876365], [-0.4252699064135755, -0.13117839339674114], [-1.4026763571946237, -0.342546863069003], [-0.9026763571946238, 0.5234785407154355], [-0.9009688679024193, 0.43388373911755845], [-0.5953161068071142, 1.3860267901528613], [0.40096886790241903, 1.2999091429019967], [1.4009688679024188, 1.2999091429019969], [-0.9235624171213708, 0.6452522087898201], [0.07643758287862912, 0.6452522087898199], [-0.7498011914373657, 1.208572266853442], [-1.5953161068071147, 1.3860267901528616], [-1.5205860132206905, 0.38882299297168155], [-0.6603293142416642, -0.121038176231964], [-1.9266392065104776, 0.3759518332946819], [-1.4266392065104776, -0.49007357048975675], [-0.44923275572942933, -0.2787051008174948], [0.05076724427057089, 0.5873203029669437], [-0.23305187182982634, 1.546198141555358], [-0.9266392065104776, 0.3759518332946818], [-1.2104583226108747, 1.3348296718830959], [-0.22162749638574636, 1.1857874057069218], [-0.2330518718298265, 0.18585266601351913], [-0.9492327557294293, 0.5873203029669438], [-0.7330518718298262, 0.6801727377709197], [-1.0256703386080586, 0.8098355724122399], [-0.5256703386080586, -0.05618983137219871], [-0.3653410243663947, 0.930873748644204], [-2.210458322610875, 1.3348296718830963], [-2.221882698054954, 0.3348949321896937], [-1.4329792565671888, 0.9494121085230813], [-1.5612981821440834, -0.5549219153495226], [-0.626311389578633, -0.20023938444593964], [-1.4009688679024195, 0.43214166466688075], [-1.9555728057861406, 0.29475517441090404], [-1.7952434915444773, 1.281818754427307], [-1.2952434915444768, 2.147844158211745], [-0.8716810744231062, 1.2419772370791207], [-1.47232212539368, -0.6323810491128208], [-1.2498011914373657, 0.3425468630690028], [-0.8262387743159945, -0.5633200580636222], [-1.1120654264146534, -0.2762168145320277], [-0.4885756245559201, 0.5056146679360022], [0.5114243754440799, 0.5056146679360023], [1.4888308262251286, 0.7169831376082643], [-0.9723221253936799, 0.2336443546716181], [-1.337663149760075, -0.6972293939725863], [-1.4123932433464992, 0.299974403208594], [-0.9949156746126316, 0.4450128243438798], [0.005084325387368249, 0.4450128243438797], [-0.9215548811231091, 0.8209646576385619], [-0.05992888204718094, 0.7996953552474628], [0.6285741272461018, 1.524928839164259], [-1.146083351077685, 0.7969644127352395], [-1.7188059086658476, -0.02278486114652012], [-1.4962849747095337, 0.9521430510353031], [-1.3488323235349466, 1.0154758371396477], [-0.6603293142416637, 1.7407093210564437], [-0.16032931424166358, 2.6067347248408828], [-0.9888308262251289, 0.1490422661761744], [-0.011424375444080148, 0.3604107358484364], [0.6120654264146532, 1.1422422183164664], [-0.7050117101247315, -0.8098355724122399], [-1.199927384737363, 0.7635594425095622], [-0.699927384737363, 1.6295848462940006], [-0.27636496761599183, 0.7237179251613756], [-1.4888308262251286, 1.015067669960613], [-0.9349867925654504, 0.18244723640185304], [-0.7124658586091361, 1.1573751485836765], [-1.9888308262251286, -0.1490422661761746], [-1.128574127246102, -0.6589034353798201], [-0.4400711179528194, 0.06633004853697605], [-0.3653410243663954, -0.930873748644204], [0.13465897563360496, -0.06484834485976551], [-0.9888308262251282, -0.1490422661761747], [-0.3003278169318457, 0.5761912177406215], [-1.2726499423255258, 0.8098355724122394], [-1.4123932433464992, 0.7568246549564503], [-1.1171497518020221, 0.34103130431358225], [-0.42864674250873924, 1.0662647882303784], [-1.300327816931846, 0.5761912177406215], [-1.8003278169318457, 1.4422166215250602], [-0.8003278169318457, 1.44221662152506], [-1.7726499423255255, 1.675860976196678], [-0.9979924640017385, 1.0434799270838582], [-2.2726499423255255, 0.8098355724122397], [-1.2838191161003971, 0.6607933062360654], [-0.2838191161003971, 0.6607933062360654], [-1.9555728057861408, -0.29475517441090404], [-1.0953161068071142, -0.8046163436145497], [-0.7225209339563141, -0.10890250839738451], [-1.4555728057861408, 0.5712702293735347], [-0.6666693642983751, 1.1857874057069218], [-0.44414843034206086, 2.1607153178887453], [-0.9046838931928859, -0.5200013863684223], [-0.0784451188768911, -1.0833214444320443], [-1.188503009293283, 0.43887645221999205], [-0.9009688679024193, -0.43388373911755795], [-0.4774064507810485, -1.3397506602501832], [0.423562417121371, -0.9058669211326249], [-1.8376631497600744, -1.5632547977570246], [-1.5320103886647698, -0.6111117467217211], [-0.5320103886647699, -0.6111117467217211], [-0.7330518718298266, -0.6801727377709195], [-1.5933085708088528, -0.17031156856727347], [-1.0933085708088528, 0.6957138352171652], [-1.2330518718298262, 0.18585266601351924], [-0.7889034414877657, 1.0818058856766242], [0.1885030092932829, 1.2931743553488855], [-1.572722557588163, 0.3068908422454836], [-1.149160140466792, -0.5989760788871417], [-0.1935873346806512, -0.3042209044762374], [-1.6234898018587332, -0.78183148246803], [-0.6234898018587336, -0.7818314824680298], [-1.550129008369211, -0.4058796491733483], [-0.7612255668814454, 0.20863752716003942], [-0.6254973378569948, -0.718498696363685], [-0.5507672442705707, -1.7157024935448653], [-0.050767244270570555, -0.8496770897604266], [-0.6885030092932827, -0.42714895156444704], [0.2889034414877658, -0.21578048189218502], [0.15116767646505347, 0.7746885277358841], [-1.3178370407634288, 0.17031156856727347], [-0.8178370407634288, -0.6957138352171651], [-0.9555728057861406, 0.2947551744109043], [-0.7124658586091362, -1.3296104430854059], [0.02058601322069009, -0.6494377053144864], [-0.9349867925654506, -0.35468253090358237], [-0.6717536896857437, -1.2536330129993185], [-0.5970235960993195, -0.25642921581813843], [0.07473009358642435, -0.9972037971811798], [-0.7838191161003973, -0.2052320975483729], [0.2050117101247313, -0.05618983137219846], [0.7050117101247313, 0.80983557241224], [-1.072722557588163, 0.04627612990267821], [-1.8989613319041574, 0.6095961879663001], [-1.0727225575881625, 1.1729162460299218], [-2.072722557588163, 0.046276129902678265], [-1.1717536896857434, -0.38760760921487986], [-0.1717536896857435, -0.38760760921488], [-2.3989613319041574, -0.2564292158181384], [-2.093308570808853, 0.6957138352171652], [-0.43035423180094456, -0.5761912177406212], [-1.0538440336596775, 0.20564026472740876], [-0.07643758287862934, 0.4170087343996709], [-0.8653410243663954, -0.06484834485976533], [-0.7050117101247316, 0.9222152351566375], [0.2838191161003968, 1.071257501332812], [-0.7141733479013416, 0.3826866208477929], [-1.2225209339563141, -0.9749279121818237], [-0.22252093395631434, -0.9749279121818235], [0.6491601404667919, -0.4848543416920664], [-0.33967068575833625, -0.3358120755158922], [-1.360256698979026, 0.015541097446245572], [0.3282463103142565, 0.7407745813630415], [-2.1491601404667917, -0.5989760788871418], [-1.193587334680651, -0.3042209044762376], [-0.21618088389960255, -0.09285243480397559], [-1.074730093586425, -0.99720379718118], [-0.28582665209865954, -0.3826866208477926], [-0.5696457681990562, 0.5761912177406217], [0.36534102436639415, 0.9308737486442046], [-1.212465858609137, -0.0067347875531108825], [-0.7124658586091368, 0.8592906162313276], [-0.2124658586091368, -0.006734787553111021], [-0.3396706857583358, -1.740709321056444], [-0.8396706857583365, -0.8746839172720056], [-1.1234898018587332, 0.08419392131640888], [-0.03401792466303122, -0.7885662700211404], [0.18850300929328268, -1.763494182202964], [-0.09531610680711461, -0.8046163436145499], [-0.076437582878629, -0.6452522087898195], [-0.9026763571946238, -1.2085722668534415], [-0.4026763571946238, -0.34254686306900334], [-0.27837250361425325, -0.31976200192248316], [-1.2050117101247309, 0.05618983137219871], [-0.2050117101247314, 0.05618983137219885], [0.22162749638574653, 0.5462634018619553], [-0.35109506120241574, -0.27348587201980507], [-0.7889034414877653, -0.025006827511578927], [-1.2889034414877651, 0.8410185762728597], [-0.28890344148776537, 0.8410185762728599], [-1.295243491544477, -0.04105690110498799], [-0.3064126653193485, -0.19009916728116263], [-0.6717536896857433, 0.7407745813630418], [-0.23676689712029342, -0.27197031326438403], [0.263233102879707, 0.5940550905200546], [0.12549733785699502, 1.5845241001481236], [0.22252093395631378, -0.9749279121818235], [0.5281736950516185, -0.02278486114652012], [1.1999273847373622, -0.7635594425095615], [1.2746574783237867, 0.23364435467161834], [0.19992738473736216, -0.7635594425095615], [-0.08389173136303474, 0.19531839607885293], [-0.9555728057861412, -0.29475517441090393], [0.01142437544407926, -0.36041073584843614], [-0.7216274963857465, 0.31976200192248344], [0.2746574783237869, 0.23364435467161865], [-1.331323099703364, -0.14230747862306378], [-0.3539166489223155, 0.06906099104919843], [0.14608335107768444, 0.935086394833637], [0.36534102436639454, -0.9308737486442042], [1.300327816931845, -0.5761912177406218], [0.525670338608058, 0.05618983137219857], [0.5538440336596774, 0.6603851390570306], [-0.06964576819905577, 1.4422166215250602], [-0.4743296613919418, 0.05618983137219856], [0.16032931424166275, -0.8746839172720055], [-0.7723947406563177, 0.059595260983864716], [0.16259205190913284, 0.41427779188744773], [-0.6120654264146538, 1.0466588410002682], [0.28381911610039756, -1.4157280903362703], [0.4441484303420607, -0.42866451031986774], [0.6666693642983751, 0.5462634018619559], [0.4123932433464996, -0.7568246549564508], [0.28890344148776587, 0.8910322312960177], [0.2952434915444775, -0.41579335064286815], [1.2952434915444773, -0.41579335064286815], [0.5205860132206902, 0.216587698469952], [0.249801191437366, -0.3425468630690033], [-0.5764375828786288, 0.22077319499461875], [0.42356241712137094, 0.22077319499461845], [-0.07643758287862923, -0.03984151734818661], [0.7498011914373661, 0.5234785407154352], [0.326238774315995, 1.4293454618480606], [-0.2110965585122342, 0.025006827511579177], [0.7889034414877657, 0.025006827511579004], [1.0114243754440801, 0.9999347396934024], [-0.41610826863696543, 0.08119665888377782], [0.5838917313630345, 0.08119665888377747], [-0.14916014046679194, -0.5989760788871418], [-0.860256698979026, 0.05301091745578923], [0.0953161068071145, 0.3477660918666935], [-0.6377357650227118, -0.3324066459042258], [1.365341024366395, 0.9308737486442042], [0.4387018178559172, 1.3068255819388859], [-0.5612981821440828, 1.306825581938886], [0.23676689712029264, 1.6323157888062223], [-0.7632331028797072, 1.6323157888062227], [0.7774790660436857, 0.9749279121818235], [-0.19992738473736282, 0.7635594425095614], [0.7889034414877656, 0.6145171763333871], [-1.7612255668814454, -0.3808728216617685], [0.3765101981412662, -0.7818314824680296], [-0.2952434915444775, -0.04105690110498805], [-0.011424375444079926, -0.9999347396934024], [0.4999999999999999, -0.8660254037844385], [-0.4266392065104778, -0.49007357048975686], [0.6346589756336046, -0.930873748644204], [0.7774790660436853, -0.9749279121818235], [0.839670685758336, -0.9870635800164028], [1.4123932433464985, -0.16731430613464207], [0.9252699064135756, -0.9972037971811801], [1.0747300935864244, -0.99720379718118], [0.30007261526263695, -0.3648227480683601], [1.2838191161003976, -0.9588778385884142], [0.328246310314257, -1.2536330129993183], [0.6603293142416637, -1.7407093210564437], [0.6717536896857434, -0.7407745813630412], [1.3653410243663948, -0.9308737486442044], [0.693587334680651, -0.19009916728116283], [1.8262387743159945, -0.5633200580636224], [2.398961331904157, 0.2564292158181384], [1.900968867902419, -0.433883739117558], [1.912393243346499, 0.5660510005758443], [1.9555728057861406, -0.2947551744109044], [2.2612255668814454, 0.657387876624399], [1.4349867925654505, 1.2207079346880212], [1.9962849747095335, -0.08611764725086496], [1.221627496385746, 0.546263401861955], [1.0612981821440826, -0.44080017815444755], [0.06129818214408267, -0.44080017815444733], [1.2632331028797068, -0.7662903850217839], [0.4885756245559194, -0.13390933590896392], [1.9888308262251284, 0.14904226617617422], [1.8716810744231063, 0.49007357048975697], [0.1999273847373626, 1.230848151852799], [1.6491601404667915, -0.4848543416920663], [1.826238774315995, 0.5633200580636218], [1.837663149760075, 1.5632547977570244], [1.7330518718298267, 0.6801727377709189], [1.6234898018587343, 0.7818314824680292], [0.8488323235349469, 1.4142125315808494], [0.6460833510776847, 1.1957011071764425], [-0.35391664892231534, 1.1957011071764427], [1.5, 0.8660254037844386], [0.6397433010209735, 1.3758865729880843], [1.212465858609136, 2.195635846869845], [0.43035423180094434, 0.5761912177406215], [1.205011710124731, -0.05618983137219864], [0.21618088389960244, -0.20523209754837296], [-0.5727225575881634, -0.8197492738817602], [1.3056527610953046, 0.9521430510353034], [0.4794139867793099, 1.5154631090989257], [-0.3094894547084559, 0.9009459327655384], [0.3333306357016248, 1.1857874057069218], [0.5558515696579394, 2.1607153178887453], [0.4046838931928858, 0.5182593119177452], [-0.09531610680711422, 1.3842847157021838], [0.4794139867793098, 0.38882299297168166], [0.516749319607539, 0.33762587470191624], [0.09318690248616801, 1.2434927958345416], [1.0894718771957015, 1.1573751485836765], [0.6171497518020219, 0.2269095671185075], [-0.3828502481979781, 0.2269095671185078], [1.2225209339563143, 0.9749279121818235], [0.7110965585122342, 0.8410185762728597], [0.5507672442705707, 1.715702493544865], [0.25019880856263454, 1.2085722668534418], [0.3879345735853462, 0.21810325722537272], [0.5340179246630317, 0.2496944282650275], [1.0747300935864244, 0.9972037971811801], [0.30007261526263695, 1.6295848462940001], [0.6285741272461021, 0.16458336362242032], [0.2632331028797066, -0.7662903850217838], [1.4123932433464992, 1.432076404360283], [0.03401792466303166, 0.7885662700211403], [1.030302899372565, 0.7024486227702753], [1.3359556604678697, 1.6545916738055788], [0.9252699064135758, 0.9972037971811802], [0.7090890225139733, 0.9043513623772046], [0.06501320743454952, 1.507064966384826], [0.5650132074345495, 2.3730903701692645], [1.188503009293283, 1.5912588877012346], [-0.0727225575881627, 0.9338710110768353], [0.8828502481979779, 0.639115836665931], [-0.11714975180202214, 0.6391158366659312], [0.10572537635794199, 1.715702493544865], [-0.8209138301525356, 2.0916543268395467], [-0.14916014046679194, 1.3508797454765054], [-0.6491601404667919, 2.2169051492609437], [0.7216274963857461, -0.3197620019224832], [0.6346589756336051, 0.9308737486442042], [-0.33766314976007483, 1.1645181033158225], [0.5764375828786292, 0.9058669211326252], [1.0764375828786292, 1.7718923249170637], [0.38793457358534633, 1.0466588410002677], [0.5953161068071144, 1.2137914956511322], [1.1717536896857441, 0.12525082242139743], [1.7952434915444784, 0.9070823048894266], [1.2889034414877656, 1.480542580117826], [0.3765101981412665, 0.7818314824680297], [1.3727951728507999, 0.6957138352171649], [1.212465858609136, -0.2913497447992379], [-0.44641116800953107, 1.3731556304758614], [0.26694812817017377, 0.6801727377709195], [1.263233102879707, 0.5940550905200546], [0.22534252167621271, 0.6323810491128201], [0.44786345563252716, -0.3425468630690035], [-0.4123932433464992, 0.1673143061346422], [0.1737612256840052, 0.5633200580636222], [-0.48656808855765865, 0.442281881831658], [1.1717536896857437, 0.6266528441679664], [0.13974330102097354, 0.5098611692036457], [0.09903113209758097, 0.4338837391175583], [1.0953161068071144, 0.3477660918666934], [0.07336079348952229, 0.3759518332946816], [0.8064126653193484, -0.3042209044762379], [-1.084146933032243, -0.9536586097907247], [0.04442719421385932, 0.29475517441090426], [0.05585156965793936, 1.2946899141043067], [0.027677874606320207, 0.2336443546716182], [-0.8989613319041576, 0.6095961879662999], [0.002007535998261867, 1.0434799270838573], [-0.07272255758816248, 0.0462761299026771], [0.6004004321944825, 1.053393628553379], [0.4400711179528186, 0.0663300485369761], [-0.3345863603709689, 0.6987110976497962], [0.011169173774871477, 0.1490422661761744], [0.4492327557294291, 2.4676061601342294], [0.26494059217191235, 0.7435055238752639], [-0.02058601322068987, 0.6494377053144871], [0.011169173774871366, -0.14904226617617433], [0.8828502481979776, 0.34103130431358286], [0.022593549218951625, -0.21136846967226214], [0.5225935492189516, 0.6546569341121764], [-0.4123932433464992, 0.2999744032085939], [-1.507709350153613, -0.04779168865809913], [0.04442719421385943, -0.2947551744109042], [0.48857562455591974, 0.6011980452522006], [0.8539166489223146, -0.3296757033920037], [-0.8791352229075113, 1.47713715050616], [0.27239474065631675, 1.3959404916223823], [-0.6831780651298238, 1.101185317211478], [-0.21246585860913658, -0.4635850393009673], [-0.6067404822511944, 0.3860920504594589], [-0.38421954829488003, 0.7715096138194739], [-0.795243491544477, 0.15517863830006307], [-0.8616259990759287, 0.021269302391099143], [-1.7218826980549546, 0.5311304715947449], [-0.8502016236318488, 1.021204042084502], [-0.12857412724610207, 0.701442040162018], [-0.9123932433464992, -0.5660510005758446], [-0.837663149760075, -1.5632547977570248], [-0.7238902340532167, -0.96919970723697], [0.09903113209758097, -0.43388373911755834], [-1.1265665912478409, 1.1395112758042434], [-0.8359556604678695, -0.7885662700211409], [-0.5303028993725647, 0.16357678101416262], [0.1737612256840051, -0.5633200580636218], [-0.4979924640017388, 0.1774545232994197], [0.28582665209865854, 0.38268662084779287], [-0.21246585860913658, 0.2913497447992379], [-0.46065713117350926, 0.1262574050296542], [-1.4329792565671893, -1.0449954858392791], [-0.6999273847373626, -0.36482274806836024], [-0.7612255668814456, -0.38087282166176895], [0.22760525934368292, -0.529915087837943], [0.26694812817017344, -0.680172737770919], [0.27837250361425325, 0.31976200192248344], [-0.5933085708088528, -0.17031156856727336], [0.36226423497728766, -0.4650667429781777], [0.32292136615079714, 0.2747012557766064], [1.2238902340532163, 1.8352251110214084], [-0.053844033659677715, 0.20564026472740826], [-0.11206542641465367, 0.1806334372158292], [0.03401792466303177, 1.1157198320494666], [-0.4215548811231091, -0.04506074614587652], [-0.5894718771957019, -0.2913497447992377], [0.40681309751383155, -0.3774673920501026], [-0.016749319607539226, 0.5283995290825226], [0.682162959236571, 0.17031156856727392], [0.9659820753369686, -0.7885662700211403], [0.8056527610953046, 0.08611764725086497], [-0.09531610680711444, -0.3477660918666934], [-0.2746574783237874, -0.23364435467161848], [0.34883232353494664, 0.5481871277964108], [-1.3989613319041574, -0.2564292158181387], [-0.47232212539368, -0.6323810491128203], [-0.1666693642983752, 0.3197620019224833], [1.371681074423106, -0.3759518332946814], [1.1491601404667913, -1.350879745476505], [0.21109655851223375, -0.025006827511578844], [-0.6491601404667928, 0.4848543416920668], [-0.7255977233454218, 0.4450128243438803], [-0.3376631497600755, -0.697229393972586], [0.23505940782808699, 0.12251987990917482], [1.1885030092932825, -1.7634941822029642], [0.35391664892231445, -0.06906099104919822], [1.2255977233454207, 0.4210125794405589], [1.1491601404667917, 0.3811710620923724], [1.3502016236318477, -0.1551786383000627], [2.221882698054954, 0.3348949321896944], [1.3616259990759274, 0.8447561013933401], [1.339670685758336, -0.12103817623196422], [2.072722557588163, 0.5591345615389546], [-0.061298182144083, -1.420947319133961], [0.2443545789512218, -0.4688042680986576], [0.05076724427057033, -1.6015807563497901], [1.5727225575881627, -0.3068908422454839], [0.5953161068071144, -0.5182593119177461], [1.497992464001738, -0.1774545232994193], [0.40297640390068046, -0.2564292158181387], [0.09732364280537587, -1.2085722668534422], [0.9235624171213709, -0.6452522087898205], [0.9349867925654507, 0.354682530903582], [1.137735765022712, -0.9904690096280693], [1.637735765022712, -0.12444360584363057], [0.9659820753369683, 0.616330975519411], [0.4659820753369682, -0.2496944282650277], [0.3114969907067172, -0.4271489515644471], [0.8114969907067172, 0.4388764522199915], [0.4492327557294291, -1.7157024935448653], [1.3209138301525354, -1.225628923055108], [0.3942746236420577, -0.8496770897604264], [1.7612255668814463, -0.20863752716003997], [0.9865680885576589, 0.42374352195278003], [1.2225209339563143, -0.9749279121818235], [0.28753414139086375, -1.329610443085406], [1.7838191161003976, -0.09285243480397565], [0.8282463103142569, -0.38760760921487986], [1.2952434915444775, 0.04105690110498822], [0.30641266531934885, 0.19009916728116263], [0.9464111680095313, -0.5071302266914232], [0.6120654264146539, -0.21810325722537272], [-0.16259205190913373, 0.4142777918874474], [-1.1349141773028135, 0.6479221465590657], [0.38285024819797864, -0.524994099470856], [1.3791352229075113, -0.611111746721721], [0.29498828987526904, -0.8098355724122398], [-0.6773338355184109, -0.5761912177406214], [-0.7952434915444773, 0.8249685026794502], [1.553844033659678, 0.6603851390570303], [0.4387018178559171, -0.5549219153495226], [0.8094894547084555, -0.034920528981099166], [-0.1460833510776851, -0.32967570339200336], [1.6234898018587336, -0.7818314824680297], [1.6349141773028135, 0.21810325722537272], [0.6625920519091336, 0.4517476118969909], [0.1511676764650539, 0.31783827598802716], [0.5507672442705712, -0.7355553525653515], [0.7632331028797072, -0.2719703132643841], [-0.20908902251397277, -0.038325958592765774], [0.6511676764650538, -0.5481871277964115], [0.15116767646505358, -1.4142125315808503], [0.5696457681990559, -0.5761912177406214], [1.7330518718298262, -0.6801727377709196], [1.4026763571946237, 0.3425468630690028], [1.477406450781048, 1.339750660250183], [0.714173347901341, -0.38268662084779326], [2.7726499423255255, 0.05618983137219913], [1.795243491544477, -0.15517863830006298], [1.4441484303420604, -0.4286645103198682], [-0.04241965821559779, -0.3580879605152489], [1.0205860132206899, -0.6494377053144871], [1.9888308262251286, -0.1490422661761741], [1.6265665912478406, -0.27348587201980457], [1.4349867925654505, 0.6835781673825858], [2.2612255668814454, 1.2468982254462078], [1.2612255668814456, 1.246898225446208], [1.2141733479013412, 0.48333878293664595], [0.5424196582155975, 1.2241133642996873], [1.0114243754440801, -0.3604107358484362], [1.5114243754440801, 0.5056146679360024], [2.007709350153613, 0.9138170924425375], [1.9215548811231091, 0.9110861499303151], [1.0696457681990557, 0.2898341860438167], [0.44615596634032206, 1.0716656685118466], [1.4962849747095333, -0.9521430510353035], [1.4215548811231091, 0.045060746145876684], [1.921554881123109, -1.0833214444320451], [0.9441484303420606, -1.2946899141043073], [1.2141733479013412, 0.7814233152889942], [0.17707863384920197, 0.5913241480078317], [1.9555728057861406, 0.29475517441090454], [0.960657131173509, 0.7397679987547845], [1.1666693642983748, -0.3197620019224827], [2.0933085708088526, -0.6957138352171647], [1.3602566989790263, -0.015541097446245239], [0.6885030092932826, 0.7252334839167962], [1.9009688679024195, 0.43388373911755734], [0.9743296613919418, 0.809835572412239], [1.1120654264146537, -0.1806334372158299], [0.6120654264146539, 0.685391966568609], [2.177333835518411, 1.4422166215250605], [1.6773338355184113, 0.5761912177406214], [1.1999273847373626, 1.2308481518527987], [1.932979256567189, 1.9110208896237175], [1.0970235960993189, 1.122454619602577], [1.997992464001738, 0.6885708804850192], [0.9161082686369656, 0.7848287449006615], [1.0538440336596775, -0.20564026472740773], [0.9161082686369656, 0.19531839607885296], [0.4161082686369655, -0.6707070077055857], [1.1831780651298234, -0.23515991342703885], [0.8539166489223151, 0.79696441273524], [0.2556454210487782, 1.3348296718830954], [0.806412665319349, 1.0561245710656006], [1.2889034414877663, -0.21578048189218588], [0.3488323235349471, 2.280237935365288], [0.8345863603709684, 0.16731430613464196], [0.059928882047181053, 0.799695355247462], [-0.10040043219448291, -0.18736822476894088], [1.5841469330322422, 1.819684013575163], [0.9492327557294289, 1.6015807563497906], [0.7124658586091359, 1.3296104430854063], [0.5970235960993191, 0.256429215818139], [0.5838917313630339, 0.6707070077055864], [0.8842195482948796, 0.09451578996496474], [1.106740482251194, 0.4799333533249797], [-0.07594388448119127, -0.3822568447683964], [0.3168447780834016, 0.5373719208542967], [0.3350820323021908, 1.5372056083036445], [0.2308092051387587, 0.5426568777150699], [-0.10676274578120926, 1.4839566057252764], [-0.00043991124290687633, -0.029658539477047496], [0.6459635903293279, 0.7333372096298221], [0.6642008445481182, 1.7331708970791695], [0.5599280173846859, 0.7386221664905949], [-0.021176059724390184, 0.20470391775276092], [-0.21468416624935482, 1.1858025940892052], [-0.3817352039833296, 0.19985434453854972], [-0.9817627457812128, 0.9998336874493474], [-1.3375719509199695, 0.9412997280102047], [-1.3193346967011794, 1.9411334154595516], [-0.7410399992606262, 2.756961364792093], [-1.0531731763703398, 1.806923007804652], [-1.6000275417978813, 0.799979342910798], [-1.5817902875790917, 1.7998130303601454], [-1.3304523051126862, 2.7677124148881638], [-1.7703307120606795, 0.6376445671815032], [-1.0174283879965142, 1.2957768414385697], [-1.981322834538304, 1.029492226926395], [-1.8567627457812106, 0.5157107691734204], [-1.4639740832166177, 1.4353395347961135], [-0.5889740832166179, 1.9194624530720406], [-1.9240561155188085, 0.38225684476839666], [-0.9244960267617153, 0.3525983052913488], [-0.9062587725429255, 1.3524319927406963], [-0.35810072824761585, 0.5160572211972563], [-1.9707075985924825, 0.24026393411166813], [-0.9711475098353892, 0.21060539463462036], [-0.09614750983538911, 0.6947283129105475], [-1.0202036253541977, 1.076985157678944], [-1.9839754524922046, 0.17830397890333544], [-1.1748074000826492, -0.40927330363890774], [-1.7748349418805305, 0.39070603927188946], [-0.8998349418805304, 0.8748289575478163], [-1.9956750779596208, 0.09290392419108484], [-1.9774378237408317, 1.0927376116404321], [-1.1206750779596208, 0.5770268424670117], [-0.125, 0.4841229182759271], [-1.99840082138868, -0.056531405876814156], [-1.351997319816445, 0.7064643432300559], [-0.59909499575228, 1.3645966174871222], [-1.9801635671698907, 0.943302281572533], [-1.2272612431057255, 1.6014345558296], [-0.7531662473534457, 0.7209608566184047], [-1.9638944465417894, -0.2662846145121748], [-2.2411735752841357, 0.6945047925713366], [-1.4320055228745803, 0.10692751002909329], [-0.5570055228745807, 0.5910504283050206], [-2.7342251586024684, 0.3713599526693283], [-1.7346650698453754, 0.34170141319228087], [-0.8596650698453756, 0.825824331468208], [-1.9373817497946546, -0.3483036823691525], [-1.1844794257304896, 0.30982859188791406], [-1.1662421715117, 1.3096622793372612], [-0.4995514083458037, 0.5643278401025814], [-1.578294697440554, -0.8158279493325407], [-0.7691266450309985, -1.403405231874784], [-0.7508893908122092, -0.4035715444254367], [-1.4502427770933701, -0.8929061774202539], [-0.45068268833627667, -0.9225647168973016], [-0.7882546392562463, 0.018735011112903308], [-1.3121331771097138, -0.9500383569874409], [-0.36572274652058423, -1.2730047590668185], [0.2125719509199695, -0.45717680973427743], [-1.1042728271634323, -0.9945487305885747], [-0.4578693255911974, -0.23155298148170442], [-0.4396320713724078, 0.7682807059676429], [-1.1063228345383038, 1.5136151452023225], [-1.4418447780834018, -0.05324900257836951], [-1.4236075238646122, 0.9465846848709777], [-1.5216532076824998, -0.08581685706494896], [-0.7687508836183348, 0.5723154171921175], [-1.7709635903293282, -0.24921429135389445], [-1.1245600887570935, 0.5137814577529756], [-0.8690160805805116, -0.9913844929458544], [-1.581572544655896, -0.2897697229300104], [-0.7065725446558959, 0.19435319534591683], [-1.044144495575865, 1.135652923356122], [-0.5259050042477202, -0.8804736992111949], [0.4529189360278896, -0.6757697814584337], [1.2058212600920548, -0.017637507201367142], [0.49326479601667095, 0.6839772628144771], [-0.5076677500289308, 0.11935998823815233], [0.2452345740352344, 0.7774922624952193], [-1.4966126028402025, -0.6402097650995265], [-1.4783753486214133, 0.35962392234982055], [-0.4518419557046903, -0.8363747715434399], [0.547718133052403, -0.8660333110204874], [-0.3763379824664057, -0.48377646625209114], [-0.3333092368341041, -0.7453344392346801], [-0.31507198261531455, 0.25449924821466746], [0.4378303414488507, 0.9126315224717341], [-0.22966928793932095, -0.6376445671815035], [0.4167342136329142, 0.12535118192536632], [-0.2114320337205312, 0.3621891202678441], [-0.044380995986556204, 1.3481373698184997], [-0.1908319475904452, -0.5875772825422435], [0.20195671497414813, 0.33205148308044996], [0.22019396919293754, 1.3318851705297972], [-0.7389899918857545, 0.24879748900119653], [-0.1432372542187894, -0.5157107691734202], [0.3822278387860243, 0.3351043905607271], [1.1913958911955795, -0.25247289198151635], [0.85382394027561, 0.6888268360286884], [0.8720611944943997, 1.6886605234780356], [0.7990561155188085, 0.10186607350753074], [-0.18491933697339635, 0.2801700524108659], [-0.3159032563928841, 1.2715545453567203], [-0.41351583540116077, 0.5590428832418086], [0.46148416459883923, 1.0431658015177356], [1.398865914393494, 1.3914694838868884], [0.6057937182941737, 0.7823418357094307], [-1.059919336973396, -0.2039528658650609], [-1.1909032563928843, 0.7874316270807937], [-1.7909307981907654, 1.5874109699915913], [-0.9159307981907656, 2.0715338882675183], [-0.05358956941087056, -0.32296640207937743], [0.5247051280296833, 0.4928615472531632], [-0.07532241376819793, 1.2928408901639612], [-0.41289436468816765, 2.234140618174166], [0.19774841305553514, 0.6449329824486408], [1.1684560116480176, 0.4046690483369727], [1.1866932658668068, 1.4045027357863202], [-0.4709699499299379, 0.5857654714442478], [0.40403005007006176, 1.0698883897201752], [0.21052194354509757, 2.050987066056619], [-0.7734535089471074, 2.2292910449599543], [-0.6017476137061801, 0.5134083694640623], [0.39665320768249934, 0.5699397753408764], [0.29238038051906745, -0.42460895524769837], [-0.6536171112087517, 0.47701294083142043], [0.25982376836423593, 0.8839843888777517], [0.15555094120080337, -0.11056434171082297], [-0.7661460334862543, 0.3786483679364666], [-0.7479087792674648, 1.378482055385814], [-1.0600419563771784, 0.42844369839837315], [-1.1051635671698907, 1.4274251998484604], [-0.029292401407517654, -0.24026393411166802], [0.8457075985924822, 0.2438589841642591], [-0.153852490164611, 0.27351752364130716], [-0.13561523594582114, 1.2733512110906542], [0.7236099226566475, 0.41786834014539886], [0.974947905123053, 1.385767724673417], [0.9931851593418424, 2.385601412122764], [0.8889123321784103, 1.39105268153419], [0.22204558105888772, 0.7276354504163504], [-0.7707706233035863, 0.6079860277044553], [0.10422937669641352, 1.092108945980382], [-0.7418488654829017, 0.46135083590417614], [-0.7236116112641121, 1.4611845233535234], [-1.0008907400064584, 2.4219739304370345], [-0.004324922040378931, -0.09290392419108473], [0.8706750779596213, 0.3912189940848422], [0.27064753616174, 1.1911983369956403], [0.10359649842776486, 0.20525008744498457], [0.6420785795318555, 0.6700918249157853], [0.6603158337506452, 1.6699255123651322], [1.6598759225077386, 1.6402669728880848], [0.8172933697375984, 1.1016997609568786], [0.01391233217841048, 0.9069297632582625], [-0.8286702205917302, 0.3683625513270564], [-1.752726336110539, 0.7506193960954533], [0.8745600887570932, 0.45446437879887946], [1.4227181330524035, -0.3819103927445604], [0.4243173116637231, -0.43844179862137456], [0.08674536074375339, 0.5028579293888303], [0.752462412821258, 0.628473734780019], [-0.17159370269755025, 1.0107305795484156], [0.8123817497946539, 0.8324266006450805], [0.017797342975882646, 0.9701751479722998], [-0.43244543411748704, 0.07726897055204576], [0.5659553872711931, 0.1338003764288601], [-0.19394801776787152, 0.9514401368593963], [-0.29822084493130396, -0.043108593729178135], [0.6481895856578259, -0.36607499580855574], [-0.33801186216287604, 0.911641188533157], [0.536988137837124, 1.395764106809084], [1.0851461821324335, 0.5593893352656442], [0.8916380756074689, 1.540488011602089], [-0.0015991786113199735, 0.0565314058768141], [0.8734008213886799, 0.5406543241527415], [-0.09049362515310944, 0.27436970964056656], [0.16084435731329627, 1.242269094168585], [-0.6322278387860243, 0.633141445991127], [0.5238659143934938, 0.9073465656109612], [1.3806286601747044, 0.391635796437541], [0.5875564640753841, -0.21749185173991714], [0.01663807560746955, 1.0563650933261615], [0.21014618213243397, 0.07526641698971731], [-0.5024102819429498, 0.7768811870055613], [-1.1024378237408312, 1.5768605299163594], [-0.046720789404032126, 1.0555129073269014], [-0.3588539665137459, 0.10547455033946071], [-0.34061671229495616, 1.1053082377888082], [0.7317262643397751, 0.8628361920098276], [1.124514926904368, 1.7824649576325209], [1.5986099226566477, 0.901991258421326], [0.3716126028402025, 1.1243326833754541], [1.2466126028402025, 1.608455601651381], [0.2537963984777285, 1.4888061789394862], [-0.002938805505600439, 1.2045376052021082], [-0.6029663473034818, 2.004516948112906], [-0.7700173850374568, 1.0185686985622504], [-0.6212036015222716, 1.004683260663559], [-1.3146320713724076, 0.28415778769171557], [-0.0626182502053454, 0.3483036823691527], [0.18871973226105987, 1.3162030668971711], [0.021668694527085197, 0.3302548173465155], [0.6680721960993203, 1.0932505664533854], [0.6863094503181097, 2.0930842539027323], [-0.08655912042701286, 0.4069714480463311], [0.7702036253541975, -0.10873932112708906], [0.05764716127881364, 0.5928754488887549], [-0.5423803805190678, 1.3928547917995528], [-0.524143126300278, 2.3926884792489], [0.7884408795729869, 0.8910943663222579], [-0.20437532478948683, 0.7714449436103628], [-0.3353592442089752, 1.762829436556217], [0.627902324064165, 1.1422551925329938], [1.6274624128212585, 1.1125966530559461], [0.6900806630266039, 0.764292970686793], [-0.2738137835151856, 0.49800835617461875], [0.32524277709337013, 1.3770290956961806], [0.2209699499299378, 0.3824803651076063], [-0.6357927958512728, 0.8981911342810266], [0.31061763473785664, 0.5752247322016489], [-0.8779023240641648, -0.17400935598113992], [-0.10757161200348575, -0.8116539231626435], [-0.089334357784696, 0.18817976428670377], [0.4750275417978813, -0.3158564246348705], [0.5684284698501358, 1.2046483912477703], [0.5866657240689257, 2.2044820786971178], [-0.20692780390067966, 0.6091276481774581], [-0.311200631064112, -0.3854210824111165], [-1.2951760835563166, -0.207117103507781], [-0.42017608355631664, 0.277005814768146], [-0.24709767593583487, 0.658132274257067], [0.39930582563639994, 1.4211280233639365], [-0.14885221865891007, 2.257502794907376], [-1.1127466652006992, 1.9912181803952014], [-0.23774666520069876, 2.475341098671129], [-0.3065715301498637, 0.7205254729718433], [-0.41084435731329616, -0.27402325761673113], [-0.39260710309450686, 0.7258104298326165], [-0.730179054014476, 1.6671101578428211], [-0.6850574432217644, 0.6681286563927336], [-0.3535964984277651, 0.76299574910687], [-1.332420438703375, 0.558291831354109], [-0.8069553456985612, 1.4091069910882559], [0.0680446543014388, 1.8932299093641831], [-0.4217053025594465, 0.8158279493325407], [0.45329469744055384, 1.2999508676084677], [-0.5306807550516509, 1.4782548465118033], [0.11572274652058434, 2.2412505956186726], [-0.8770934578418905, 2.121601172906777], [-0.3765836917667349, -0.18315355211754678], [-0.7141556426867038, 0.7581461758926582], [-0.6959183884679147, 1.7579798633420054], [-0.800191215631347, 0.7634311327534312], [-0.4745349069951863, 0.8508151597341471], [0.4004650930048135, 1.3349380780100744], [-0.2662256701610827, 2.080272517244754], [-1.259041874523557, 1.960623094532859], [-0.3840418745235564, 2.4447460128087863], [-0.5497572229066297, 0.8929061774202537], [-0.654030050070062, -0.10164255316832073], [1.134410829502925, 0.789451813153937], [0.1705163829611358, 0.5231671986417633], [-0.6072113374354073, 0.9196287656226934], [-1.3006398072855436, 0.19910329265085], [-1.5779189360278898, 1.1598926997343613], [-1.9154908869478593, 2.101192427744566], [-0.6878668228902864, 0.9500383569874409], [0.311693265866807, 0.920379817510393], [0.329930520085596, 1.9202135049597404], [-0.6488934201900136, 1.7155095872069794], [-0.002489918617778253, 2.478505336313849], [-0.7486620175335947, 0.9678993845280186], [-0.3558733549690021, 1.8875281501507115], [0.06150702555006471, 0.9787962766270857], [-0.9341680524095561, 1.0717002008181704], [0.060506034875960735, 0.3803221019857749], [-0.923469417616244, 0.5586260808891106], [-0.277065916044009, 1.3216218299959805], [-0.8329489622660251, 0.9859482495506557], [0.04205103773397534, 1.470071167826583], [-0.6705054263414091, 2.171685937842427], [-0.5395215069219211, 1.1803014448965723], [0.01105345858126383, 1.1194831101612428], [0.0292907128000528, 2.1193167976105904], [-0.28284246430966054, 1.169278440623149], [0.6635679662794691, 0.8463120385437717], [-0.18869054968189047, 1.6089613356268053], [0.6204775027276652, 1.021384053084562], [1.4772402485088758, 0.5056732839111419], [-0.22886042171704535, 1.657965961706414], [-0.9991911337777248, 2.2956105288879174], [0.0003689549793683966, 2.2659519894108695], [-0.9635254915624214, 1.9996673748986948], [-0.2883342759310744, 1.7203591604211907], [-0.2432126651383626, 0.7213776589711032], [-0.3474854923017947, -0.27317107161747123], [-0.669629568671497, 1.949872044436788], [-0.33205761775152753, 1.0085723164265832], [0.5429423822484727, 1.4926952347025102], [-0.7304247633148052, 1.9677330719773656], [0.20695698647984972, 2.316036754346518], [0.6810519822321293, 1.4355630551353236], [-1.5072278387860245, 0.14901852771520013], [-0.8137993689358884, 0.8695440006870436], [-0.020727172836567886, 1.4786716488645018], [-1.3137197322610596, -0.8320801486212437], [-1.1309839194194884, 0.9913844929458544], [-0.6568889236672084, 0.11091079373465972], [-0.638651669448419, 1.1107444811840068], [0.11425065461574624, 1.7688767554410736], [-1.1935081065249646, 0.9810986763364441], [-0.33674536074375416, 0.46538790716302403], [-0.6743173116637237, 1.406687635173229], [-0.656080057444934, 2.4065213226225763], [0.3434800313121593, 2.3768627831455285], [-0.3185081065249643, 1.4652215946123714], [-1.3141831844845853, 1.558125518803456], [-1.002050007374872, 2.5081638757908964], [-1.2772791287423462, 0.9607894070835113], [-0.3895749651794934, 0.5003751175646306], [-1.326956714974148, 0.152071435195478], [-0.5177886625645933, -0.4355058473467658], [-0.40227912874234617, 1.4449123253594385], [0.5111617508306407, 1.8518837734057694], [0.4068889236672084, 0.8573350428171949], [0.8571317007605783, 1.7502412202374487], [-0.4001902011253149, 1.2896034103793577], [0.40897785128424013, 0.7020261278371143], [0.13169872254189463, 1.6628155349206262], [-0.4625719509199695, 1.4254226462861315], [-0.7747051280296832, 0.47538428929869103], [-1.4413958911955793, 1.2207187285333712], [-0.5846696268558044, 1.5994320022672717], [-0.896802803965518, 0.6493936452798308], [-0.20337433411538175, 1.3699191182516741], [-0.7592772534794158, 1.7571276773427456], [0.24028283527767735, 1.727469137865698], [-0.01105514718872791, 0.7595697533376793], [-0.8121068579151557, 1.792114887744352], [0.06289314208484464, 2.276237806020279], [-0.13061496444012022, 3.257336482356723], [-0.5808577415334902, 2.3644303049364694], [-0.9447832883553768, 1.8609284936328983], [-0.41931819535056336, 2.7117436533670456], [0.05477680040171684, 1.8312699541558506], [-0.2780925251894809, 1.1155940543982181], [0.47480979887468466, 1.7737263286552847], [-0.5490039846405008, 1.3034888482780485], [0.41489046190128853, 1.5697734627902236], [-1.5310800574449341, 1.9223984043466489], [-2.529480878833614, 1.8658669984698348], [-1.6544808788336138, 2.349989916745762], [-0.6549207900765208, 2.320331377268714], [-1.9375994927178508, 1.741279070921003], [-1.1284314403082956, 1.1537017883787595], [-1.2954824780422702, 0.16775353882810362], [-1.4173803805190677, 0.9087318735236254], [-0.6644780564549027, 1.566864147780692], [0.17810449631523784, 2.1054313597118988], [-0.814711708047236, 1.9857819370000034], [0.06028829195276497, 2.4699048552759306], [-1.3991431263002783, 1.9085655609729728], [-0.646240802236113, 2.56669783523004], [0.020449960929783018, 1.8213633959953597], [-1.7549523314390372, 1.85003160153383], [-0.8085419008499075, 1.5270651994544526], [0.18713317710971378, 1.434161275263368], [0.20537043132850363, 2.433994962712715], [-1.5481580442953098, 0.8363747715434398], [-0.7952557202311448, 1.4945070458005065], [-0.7770184660123551, 2.494340733249854], [0.07974427976885545, 1.9786299640764338], [-0.6043524638382598, 0.7070754187197132], [-0.9419244147582294, 1.648375146729918], [-1.1089754524922046, 0.6624268971792624], [-0.6865866622248942, 1.2069507909571289], [0.18841333777510605, 1.6910737092330557], [-0.5050151320750302, 0.9705482362612123], [0.4413952985140994, 0.647581834181835], [-0.7250275417978813, 1.284102261186725], [-0.4477484130555348, 0.32331285410321364], [0.49866201753359474, 0.00034645202383609686], [-0.8471252177337163, 1.4581116171678645], [-1.7845069675283711, 1.1098079347987118], [-0.7916907631658965, 1.2294573575106074], [-1.2878943646881678, 1.7500176998982389], [-2.000450828763552, 2.451632469914083], [-1.3337600655976558, 1.706298030679403], [-1.3486895593314758, 1.7678787274388161], [-1.5157405970654505, 0.7819304778881605], [-0.6731580442953098, 1.3204976898193672], [-2.275218757429228, 1.0792875573883018], [-1.3045111588367455, 0.8390236232766338], [-0.7908594893883258, 0.21240206036855439], [0.08414051061167438, 0.6965249786444813], [-0.046843408807814235, 1.687909471590336], [-1.0308188613000189, 1.8662134504936714], [-1.6666907631658958, 0.74533443923468], [-0.6671306744088028, 0.7156758997576323], [0.20786932559119697, 1.1997988180335593], [-0.48555914425893953, 0.47927334506171587], [-1.0855866860568204, 1.2792526879725137], [-1.1412256701610821, 1.5961495989688272], [-1.4533588472707961, 0.6461112419813864], [-0.5783588472707962, 1.1302341602573138], [-1.7125564640753845, 0.701614770015844], [-0.7129963753182907, 0.6719562305387963], [-0.16483833102298084, -0.16441854100464354], [0.8308367469366398, -0.2573224651957281], [-0.7751747142807294, 1.049918452384997], [-0.7569374600619401, 2.049752139834344], [-1.6943192098565945, 1.7014484574651914], [-0.799115584502397, 1.1085862180621753], [0.07588441549760327, 1.5927091363381023], [-0.916931788864871, 1.4730597136262071], [-0.8375564640753842, 1.1857376882917712], [-0.14412799422524802, 1.9062631612636145], [0.1331511345170988, 0.9454737541801033], [0.8860534585812636, 1.6036060284371698], [-0.9596541400112191, 1.3597470442729107], [-0.08465414001121918, 1.8438699625488377], [0.6087743298389174, 2.564395435520681], [-1.019127994225248, 1.4221402429876877], [0.8542728271634323, 1.9627945671404285], [0.75, 0.9682458365518543], [-1.2623136869820137, 1.5945209474360977], [-1.366586514145446, 0.5999722168475234], [-0.45314563457245893, 1.0069436648938543], [-2.3125840058732656, 1.5015941129266417], [-1.4375840058732652, 1.9857170312025687], [-0.44190892791364433, 1.8928131070114844], [-0.8239202814715497, 0.31467816510212565], [-1.7878147280133392, 0.048393550589950984], [-0.9128147280133391, 0.532516468865878], [0.17563980728554363, 0.2850196256250778], [-0.017868299239421237, 1.2661183019615219], [-1.75209345784189, 1.6374782546308504], [-0.8056830272527602, 1.314511852551473], [-0.03535231519208126, 0.6768672853699697], [-0.9103523151920808, 0.19274436709404283], [-1.3277326957111486, 1.1014762406176681], [-0.4527326957111488, 1.585599158893595], [-1.2103592442089757, 1.2787065182802901], [-0.4400285321482965, 0.6410619510987867], [0.43497146785170404, 1.125184869374714], [-1.2126361007502127, 2.4032389193241315], [-0.33763610075021266, 2.887361837600059], [-1.605424763314805, 1.4836101537014388], [-0.6590143327256752, 1.1606437516220613], [0.21598566727432456, 1.6447666698979884], [-1.3826677500289306, -0.3647629300377745], [-2.353375348621413, -0.12449899592610603], [-1.9605866860568204, 0.7951297696965868], [-1.0106152359458214, 0.789228292814728], [-0.4851501429410078, 1.6400434525488752], [0.3898498570589928, 2.1241663708248018], [0.8639448528112723, 1.2436926716136065], [-1.0490561155188085, 0.8663797630443237], [-0.5235910225139949, 1.7171949227784709], [-0.04949602676171527, 0.8367212235672757], [-1.6727181330524035, 1.3501562292964153], [-0.8635500806428481, 0.762578946754172], [0.13601000811424524, 0.7329204072771237], [-1.905818861300019, 1.3820905322177441], [-0.921843408807814, 1.203786553314409], [-1.859225158602469, 0.8554828709452564], [-2.7666386682889494, -0.15631036716280944], [-1.767078579531856, -0.18596890663985743], [-1.7488413253130668, 0.8138647808094897], [-1.0572667190194953, 0.6472353821579995], [-0.5831717232672152, -0.23323831705319542], [-0.05770663026240197, 0.6175768426809514], [-1.952470344373693, 1.2400976215610156], [-0.9529102556165994, 1.2104390820839677], [-1.876966371135408, 1.5926959268523646], [-0.6216126028402027, -0.1560868468235997], [0.3572113374354071, 0.04861707092916123], [-1.161539546182927, -0.347313348430575], [-0.687444550430647, -1.2277870476417698], [-0.16197945742583375, -0.37697188790762304], [-1.127212706710994, -0.3374067902700848], [-0.4605219435450981, -1.0827412295047651], [-0.4422846893263084, -0.0829075420554177], [-0.5835103594873912, 1.51324205691341], [-1.5299207900765204, 1.8362084589927867], [-2.4013558330112725, 1.0870358524269605], [-1.4549454024221427, 0.7640694503475833], [-1.6484535089471075, 1.7451681266840273], [-2.5840029942900857, 0.9782833218141334], [-1.7090029942900857, 1.46240624009006], [-0.7955621147170988, 1.8693776881363913], [-1.3172846893263088, -0.5670304603313447], [-1.5107927958512732, 0.41406821600509947], [-0.5643823652621436, 0.09110181392572195], [-1.2427727538954558, 0.7510361984481514], [-1.3470455810588882, -0.24351253214042295], [-0.4336047014859006, 0.16345891590590805], [-1.1865070255500656, -0.49467335835115844], [-1.1682697713312762, 0.5051603290981888], [-1.1234008213886804, 0.42759151239911325], [-1.3169089279136443, 1.4086901887355572], [-0.4034680483406572, 1.815661636781888], [0.4715319516593426, 2.2997845550578155], [-1.245498497324515, 0.6016008683802526], [-0.3704984973245149, 1.0857237866561795], [-1.366173575284136, 1.1786277108472643], [-2.1919089279136448, 0.9245672704596299], [-1.643750883618335, 0.08819249891619024], [-0.6441907948612418, 0.05853395943914243], [-1.9928162043624742, -0.11964942271189521], [-1.1360534585812636, -0.6353601918853156], [-0.3831511345170986, 0.022772082371751157], [-0.36491388029830885, 1.0226057698210984], [-1.077470344373693, 1.7242205398369426], [-1.1178162043624742, 0.3644734955640318], [-1.830372668437858, 1.0660882655798756], [-0.8929909186432037, 1.4143919479490288], [-0.41889592289092425, 0.5339182487378334], [-1.2399138802983092, 0.5384828515451713], [-1.0957075985924825, 0.7243868523875955], [-1.4145215069219206, 0.6961785266206454], [-0.5577587611407101, 0.18046775744722518], [-0.8953307120606793, 1.12176748545743], [-2.7053726684378585, 0.5819653473039489], [-2.231277672685578, -0.298508351907246], [-1.8384890101209856, 0.6211204137154474], [-1.2224854923017952, -0.7572939898933988], [-0.5760819907295602, 0.005701759213471047], [-1.5600574432217644, 0.18400573811680662], [-1.9788239402756098, -0.20470391775276103], [-2.3163958911955795, 0.7365958102574436], [-2.2981586369767895, 1.736429497706791], [-1.4889905845672349, 1.148852215164548], [-1.1071317007605788, -0.7819953836855948], [-1.384410829502925, 0.1787940233979164], [-0.5752427770933697, -0.4087832591443266], [-0.964334357784696, -0.29594315398922244], [-0.7972833200507213, 0.6900050955614333], [0.07771667994927878, 1.1741280138373607], [0.8306190040134438, 1.8322602880944272], [-1.0888944465417894, 0.2178383037637527], [-0.9579105271223011, -0.773546189182102], [-1.506068571417611, 0.06282858236133801], [-1.2109921224776243, 0.391847659744892], [-0.33599212247762444, 0.8759705780208189], [-1.3288083268400985, 0.756321155308924], [-0.5645886209053892, 1.154843408851762], [-0.31325063843898393, 2.1227427933797807], [-1.5136516694484192, 0.6266215629080789], [-1.6179244966118516, -0.36792716768049477], [-1.895203625354198, 0.592862239403016], [-1.7968434088078147, 0.7196636350384811], [-1.5455054263414092, 1.6875630195664992], [-1.5272681721226196, 2.6873967070158473], [-1.0623817497946548, 0.13581923590677417], [-0.3094794257304895, 0.793951510163841], [-0.1424283879965147, 1.779899759714497], [-0.34982528571927063, -0.5657955341090698], [-1.544593087230062, 0.5713250832535407], [-1.5263558330112723, 1.5711587707028878], [-0.7171877806017171, 0.9835814881606446], [-1.9191444955758652, 0.6515300050801944], [-1.031440332013012, 0.1911157155613138], [-1.3087194607553583, 1.151905122644825], [-1.7930721960993203, -0.6091276481774582], [-0.7935121073422271, -0.638786187654506], [-0.5421741248758216, 0.3291131968735125], [0.005983919419487949, -0.5072615746699273], [0.024221173638277804, 0.4925721127794201], [-0.91807219609932, -0.12500472990153094], [-0.6667342136329149, 0.8428946546264874], [-1.6306286601747042, 0.5766100401143127], [-1.7272402485088751, 0.4625725526407124], [-1.2676071030945066, 0.24168751155668894], [-1.0162691206281012, 1.2095868960847076], [-1.541734213632915, 0.35877173635056014], [-2.3984969594141257, 0.8744825055239804], [-1.5234969594141252, 1.3586054237999077], [-2.487391405955915, 1.0923208092877332], [-1.5034159534637102, 0.9140168303843983], [-2.505628660174704, 0.09248712183838592], [-1.5682469103800498, 0.4407908042075387], [-0.6932469103800498, 0.9249137224834658], [-1.6934284698501365, -0.7205254729718433], [-0.693868381093043, -0.7501840124488911], [-0.7044836170388644, 0.03904428036583647], [-1.675191215631347, 0.2793082144775042], [-1.282402553066754, 1.1989369801001972], [-1.5596816818091002, 2.1597263871837087], [-0.6648538178675667, -0.4088565012862095], [0.33082126009205415, -0.501760425477294], [-1.3774102819429506, 0.29275826872963473], [-0.7032946974405536, -0.3317050310566136], [0.10587335496900141, -0.919282313598857], [0.684168052409555, -0.10345436426631605], [-0.97614778134109, -1.7733798766314486], [-1.1696558878660548, -0.7922812002950046], [-0.2946558878660548, -0.30815828201907747], [-0.4371331771097138, -0.4659154387115139], [-1.436693265866807, -0.43625689923446564], [-1.4184560116480176, 0.5635767882148818], [-1.2938959228909241, 0.04979533046190654], [-1.3390175336836363, 1.048776831911994], [-0.5861152096194707, 1.7069091061690602], [-1.6497051280296833, -0.00873862897723604], [-0.840537075620128, -0.5963159115194795], [-0.14710860576999163, 0.124209561452364], [-1.1670510377339745, -0.9859482495506557], [-0.2920510377339749, -0.5018253312747287], [-1.2848672420964486, -0.6214747539866241], [-0.892078579531856, 0.29815401163606936], [-0.32446848496383396, -0.44738103761944914], [0.22368955933147583, -1.283755809162889], [0.24192681355026524, -0.2839221217135416], [-0.5206475361617396, -0.2229525004437858], [0.2322547879024257, 0.43517977381328093], [-0.36777275389545605, 1.2351591167240785], [-1.3605591442589393, -0.004849573214211672], [-0.5037963984777287, -0.5205603423876318], [-1.1038239402756098, 0.2794190005231663], [-0.1047127384063391, -1.0242072700656222], [0.20742043870337445, -0.07416891307818113], [-0.7714035015722353, -0.27887283083094233], [-0.10587200577475231, -0.9380173247117605], [-0.5232523862938199, -0.029285451188134998], [0.4231580442953098, -0.3522518532675123], [-0.7114841645988397, -0.07491996496588124], [0.08158803150048088, 0.5342076832115769], [0.0998252857192703, 1.534041370660924], [-1.0860355729446431, 0.005284956860772828], [-2.0817106509042635, 0.09818888105185758], [-1.6314678738108932, 0.9910950584721111], [-1.9610355729446431, -0.4788379614151543], [-0.9626347515559628, -0.42230655553833996], [-0.08763475155596256, 0.06181636273758695], [-2.099947905123053, -0.9016448063974897], [-2.293456011648018, 0.07945386993895454], [-0.2226125790082768, -0.22838874383898444], [-1.1466686945270852, 0.15386810092941236], [-0.3937663704629204, 0.812000375186479], [-0.8507788263617226, 0.008449194503492963], [-1.188350777281692, 0.9497489225136978], [-0.3952785811823717, 1.5588765706911558], [-0.9351690430836606, 0.4732260261768595], [-0.7227208712576535, -0.9607894070835115], [0.15227912874234617, -0.47666648880758417], [0.6777442217471596, 0.37414867092656307], [-0.2596375280474945, 0.025844988557410298], [-1.3227484130555347, -0.16081006417271335], [-0.19641043058912933, 1.291212238631232], [-1.7155370756201278, -1.0804388297954066], [-1.0221086057699913, -0.3599133568235632], [-0.2692062817058263, 0.29821891743350354], [-0.5826196194809332, -0.9087318735236257], [-0.18983095691634044, 0.010896892099067357], [-0.9023874209917241, 0.7125116621149117], [-0.2559839194194893, 1.4755074112217814], [-1.1826471612788145, -0.10875253061282796], [-1.164409907060025, 0.8910811568365193], [-0.3076471612788142, 0.37537038766309916], [0.42050542634140964, -1.2034401012905722], [-0.4362573194398015, -0.6877293321171526], [-1.1488137835151853, 0.013885437898691858], [0.22699731981644478, -0.222341424954128], [0.8936880829823406, -0.9676758641888081], [0.18113161890695673, -0.2660610941729644], [0.1204984973245149, -0.11747795010432482], [-0.3297442797688551, -1.0103841275245788], [-0.3115070255500656, -0.010550440075231704], [-0.21377182713800646, 0.06956465777624604], [-1.2065880315004807, -0.050084764935649256], [-0.3315880315004811, 0.43403815334027773], [-0.19553457291921705, 1.069398345225593], [-0.29980740008264906, 0.07484961463701892], [-0.8031841329900662, 0.08031570787231652], [-1.6599468787712768, 0.5960264770457369], [-0.7849468787712771, 1.080149395321664], [-1.2384614683231046, -0.17885892919535074], [-0.30107971852844984, 0.16944475317380184], [-1.0714104305891294, 0.8070893203553051], [-0.20050397323828495, 0.13152461298457843], [-0.1822667190194951, 1.131358300433926], [-0.7822942608173764, 1.9313376433447238], [0.5416907631658958, -0.2612115209587529], [0.3481826566409312, 0.7198871553776912], [1.2945930872300608, 0.3969207532983138], [0.8772127067109938, 1.305652626821939], [0.41959308723006083, -0.08720216497761346], [-0.29296337684532314, 0.6144126050382309], [-0.818428469850137, -0.23640255469591626], [0.05947942573048848, 0.174294326388013], [-0.9112281728619935, 0.4145582604996815], [0.002212706710993828, 0.8215297085460124], [-1.2210134003969568, -0.2849201497157994], [-0.4681110763327917, 0.37321212454126756], [-0.4498738221140024, 1.373045811990615], [0.6453307120606793, -0.15352164890557596], [1.2917342136329144, 0.6094741002012934], [0.30775876114070955, 0.7877780791046286], [0.039905948745874165, 1.3300885047958628], [-0.8841501667729342, 1.712345349564259], [-0.5672412388592901, 0.3036551608287016], [0.43874268056019794, -0.2036064138412257], [-0.829696829737202, 0.1623347757292943], [-0.18329332816496724, 0.9253305248361643], [-1.1672687806571715, 1.1036345037395], [0.8087281411666485, -0.6172358220192908], [0.47115619024667876, 0.3240639059909136], [0.19387706150433281, 1.284853313074425], [0.6022402485088758, 0.02155036563521462], [-0.30357861279114373, 1.4036408978529584], [0.33463314541436895, 0.26323787719190395], [1.2096331454143692, 0.747360795467831], [0.2256576929221641, 0.925664774371166], [0.2594108295029256, 0.3053288948780105], [-0.7362642484566954, 0.39823281906909513], [0.13873575154330475, 0.8823557373450219], [-0.17259469337165545, 0.41225640490710386], [0.277648083721715, 1.3051625823273576], [-0.5315199686878403, 1.8927398648696012], [-0.32181586700993337, 0.4038072104036111], [0.5531841329900669, 0.8879301286795381], [0.2759050042477207, 1.8487195357630493], [-0.5284038985104142, 0.35372244546796155], [0.34659610148958575, 0.8378453637438884], [0.034462924379872195, -0.11219299324355247], [-0.9033884116658287, 0.11403748747360085], [-0.20995994181569277, 0.8345629604454441], [-0.5220931189254062, -0.11547539654199662], [0.6190160805805118, 1.9596303294977087], [-0.3738001237819627, 1.839980906785813], [-1.2488001237819626, 1.3558579885098863], [-0.7080704482444843, 2.027023514666384], [-1.5830704482444844, 1.5429005963904574], [0.08330923683410374, 1.713580275786534], [-0.6695930872300614, 1.0554480015294674], [0.2677886625645931, 1.4037516838986202], [-1.4816831091063825, -0.7017904618589397], [0.5829489622660247, -0.017702412998801487], [-0.3634614683231049, 0.3052639890805761], [0.34909499575227954, -0.3963507809352678], [0.7317627457812104, -0.03158785089749294], [-0.2610534585812637, -0.15123727360938855], [0.880983919419488, -0.023138656394000146], [1.0272791287423457, 0.007456429468342685], [1.087571950919969, 0.02694610854164925], [1.1918447780834005, 1.0214948391302239], [1.1673803805190675, 0.059513963028228756], [1.29815804429531, 0.13187106500841472], [0.31418259180310515, 0.3101750439117493], [1.4625564640753845, 0.2666310665360103], [0.7691279942252482, -0.4538944064358328], [1.2955054263414088, -0.7193171830146451], [0.8214104305891292, 0.1611565161965498], [1.5203307120606797, 0.33060126937035117], [0.57392028147155, 0.6535676714497287], [1.7456750779596204, 0.8753419123607691], [1.8499479051230527, 1.8698906429493436], [1.7484008213886795, 1.0247772424286683], [1.2743058256364002, 1.9052509416398635], [1.72882394027561, 1.172949754304615], [1.5353158337506452, 2.154048430641059], [0.5396407557910246, 2.246952354832144], [1.6634408795729878, 1.3752172845981852], [0.6794654270807827, 1.5535212635015196], [1.0170373780007522, 0.6122215354913153], [0.142037378000752, 0.1280986172153884], [1.3513077024632736, 0.4251789276107444], [0.3673322499710685, 0.6034829065140791], [1.5430721960993203, 1.577373484729312], [1.2754650930048137, 1.8190609962860016], [-0.5459453375843163, 1.6579044800894522], [1.552744221747159, 0.85827158920249], [1.20024277709337, 1.861152013972108], [0.7261477813410906, 2.7416257131833035], [1.0621331771097142, 1.9182841935392947], [0.9170510377339758, 1.9541940861025098], [-0.06692441475822897, 2.1324980650058447], [-0.13854337719904242, 1.8431451444285334], [-1.0135433771990425, 1.3590222261526064], [0.7682372542187894, 1.9680795240012015], [-0.23132283453830393, 1.9977380634782493], [-0.12705000737487238, 2.992286794066824], [-0.027387420991723532, 1.196634580390839], [0.9565880315004809, 1.0183306014875038], [0.16351583540116033, 0.40920295331004575], [-0.229272827163433, -0.5104258123126475], [0.5564918934750356, 1.949344512888298], [-0.4391831844845855, 2.042248437079383], [-0.8319718470491786, 1.12261967145669], [-0.407402553066754, 1.6830598983761242], [-0.6846816818090999, 2.6438493054596357], [-0.021802803965517703, 1.1335165635557576], [-0.8785655497467283, 1.649227332729178], [0.10624911638166523, 1.0564383354680449], [0.16370323091044225, 1.0297157472656053], [-0.6454648214991129, 1.6172930298078492], [0.2679760580738746, 2.02426447785418], [0.3051539110086301, 0.9814451283602802], [-0.5698460889913701, 0.4973222100843533], [0.47272087125765383, 1.9290352436353655], [0.09005312122872255, 1.564272313597591], [-0.4736895593314757, 2.252001645714743], [-0.49117357528413563, 1.6627506291231915], [0.10885396651374557, 0.8627712862123935], [0.22138288879124857, 0.9611358591073472], [0.33261961948093277, 1.8769777100754799], [-0.6513558330112722, 2.0552816889788144], [0.3453237830437852, 0.9324405022806718], [0.47630770246327314, -0.058943990665182744], [0.4175430798551898, 2.420961710813284], [-0.47699731981644455, 1.1905872615059825], [0.4364435597565426, 1.597558709552313], [0.2429354532315784, 2.5786573858887576], [0.2018419557046902, 1.804620608095294], [0.057635673998863934, 1.6187166072528705], [-0.7977181330524031, 1.8342791475723421], [-0.7794808788336136, 2.8341128350216893], [0.14457523668519512, 2.451855990253293], [-0.6407405970654505, 1.2660533961640876], [0.33808334321015954, 1.4707573139168484], [-0.5369166567898408, 0.9866343956409214], [-0.8631011937550258, 2.0365466778659114], [-1.8559173981175, 1.9168972551540162], [-0.9095069675283703, 1.5939308530746388], [-1.7662697133095808, 2.1096416222480587], [0.6612281728619929, 0.5536875760521727], [-0.020330712060679712, 1.6058904037333572], [-0.9842251586024691, 1.339605789221183], [-0.05916805240955503, 1.5558231190940979], [-0.040930798190765616, 2.5556568065434453], [-0.2922687806571711, 1.5877574220154267], [-0.19172268759690336, 1.8343966478947915], [0.8396476848079191, 1.1609902036458974], [1.0066987225418949, 2.1469384531965527], [0.2860259167833825, 2.403585371347968], [-0.17405611551880862, 1.3505026813202505], [0.7393847640541784, 1.7574741293665814], [1.0769567149741475, 0.8161744013563763], [-1.1803858830813343, 1.4695162175345666], [-0.22070759859248246, 1.2085097706635224], [0.6927332809805049, 1.615481218709853], [-0.23397545249220464, 1.1465498154551894], [0.4327153106736916, 0.4012153762205093], [-0.5668447780834016, 0.4308739156975573], [-0.24567507795962107, 1.061149760742939], [-0.7648658728208624, 0.6355608019061545], [0.5969074748105201, 1.5997169726741451], [-0.24956008875709323, 0.9979043760289021], [-0.24840082138868003, 0.9117144306750404], [0.6650400581843074, 1.3186858787213713], [-0.2428162043624742, 0.8485964138399587], [0.7278913942300083, 0.6083324797282905], [-0.6119405771923634, -0.8751887423504227], [-0.22882394027560993, 0.7635419187990933], [-0.7029189360278894, 1.6440156180102883], [-0.21389444654178935, 0.70196122203968], [-1.2067106509042635, 0.5823117993277844], [-0.6284159534637095, 1.3981397486603249], [-0.21103557294464204, 0.4894078751366992], [-0.10962161937835779, 1.6965099526282543], [0.2279503315416116, 0.7552102246180495], [-0.7560251209505935, 0.9335142035213844], [-0.18738174979465438, 0.6199421541827015], [-0.9265460341365874, 2.8607621810822454], [-0.2531250458223415, 1.2629540643188015], [-0.4574204387033747, 1.0424147496300362], [-0.04307219609932056, 0.35911818837439624], [0.4823928969054929, 1.2099333481085437], [-0.0029023240641649073, 0.3101135622947875], [0.015334930154624615, 1.309947249744135], [-0.631068571417611, 0.546951500637265], [-1.421108629601918, -0.28761145980817837], [0.056571530149863714, 0.24772036358001093], [0.011449919357151561, 1.2467018650300983], [0.7817806314178308, 0.6090572978485951], [-1.609359268040902, 1.351008415295675], [-0.562461386469483, 1.837443385215062], [-1.255889856319619, 1.1169179122432187], [-0.08647548418754991, -0.02437358261627498], [-0.8428139321613648, 0.5282164895243621], [-0.8346975904782373, 0.9731843413887911], [-0.8959635903293285, 0.23490862692203252], [-0.8892197059347098, 0.08560066473308897], [-1.8887797946918026, 0.11525920421013724], [-1.3633147016869895, 0.9660743639442844], [-0.5770865288249956, 1.0356390217205296], [-0.6493058256364004, -0.4528821868120822], [-0.10114778134109059, -1.289256958355522], [-0.28919216413682847, -0.7143786781777088], [0.1717053025594466, 0.15241788721931337], [-1.662409291592536, 0.9357985788175719], [-0.4746981990127883, -0.6105778618875563], [-0.6682063055377527, 0.37052081444888807], [0.29975722290662965, 0.07533965913160051], [-0.6466532076825, 0.398306061210978], [-0.060169043083660445, 0.9573489444527866], [-0.45195671497414813, 0.6361943534714046], [-0.5891990931537225, 0.3715834730085387], [-0.8729505853066085, -1.1239862313516182], [-0.5608174081968944, -0.17394787436417766], [-0.6066831091063826, -0.2176675435830131], [0.3306986406882719, 0.13063613878614], [0.4378668228902858, 0.018207479564413487], [-0.03622817286199376, 0.8986811787756086], [-0.5616932658668071, 0.047866019041461566], [0.4171306744088026, 0.2525699367942223], [0.02456702178131498, 0.8808201512350309], [0.057429418355617656, 2.68245820217891], [-0.2716686945270851, 0.637991019205339], [-0.3105060348759606, 0.5879237345660794], [-0.6353798569899621, 1.476846628280768], [-0.47204558105888805, 0.24061038613550406], [-0.4997388038550856, -0.056184953852978003], [0.41370207571790174, 0.35078649419335284], [-0.3954659766916533, 0.9383637767355963], [0.38944085574106024, 0.9633962633376428], [1.1019973198164448, 0.261781493321799], [0.538254639256246, 0.9495108254389512], [-0.04004005818430734, 0.13368287610641022], [-0.25221270671099427, 0.1467161280058418], [-0.0851616689770186, 1.1326643775564973], [-1.2249479051230527, -0.41752188812156277], [-0.23213170076057899, -0.29787246540966744], [-0.4256398072855435, 0.6832262109267769], [1.257227838786024, 0.8192273088366542], [1.5345069675283698, -0.1415620982468575], [0.07181586700993292, 0.5644386261482432], [-0.9277442217471605, 0.594097165625291], [-0.975338915118816, 0.5222306522564679], [-0.08291052712230162, -0.28942327090617476], [0.021362300041130267, 0.7051254596823999], [1.768688082982341, -0.48355294591288134], [0.2181110763327907, 0.5950337120105863], [0.743576169337604, 1.4458488717447335], [0.6959814759659488, 1.3739823583759105], [1.1315519559057776, 1.0020051600569175], [1.6570170489105913, 1.8528203197910647], [0.6574569601534979, 1.8824788592681125], [1.1058092051387587, 1.0267797959909966], [1.417942382248473, 1.9768181529784372], [0.5092772534794157, -0.7888818407908914], [0.31576914695445124, 0.1922168355455527], [0.6947832883553762, -0.8926826570810439], [1.3997051280296833, 0.9769844655290901], [0.6468028039655183, 0.31885219127202313], [1.2716532076824993, 1.0540626936168032], [0.35174761370618013, 0.4548374670877919], [0.5452557202311448, -0.5262612092486523], [0.9954984973245151, 0.3666449681716013], [0.5214035015722356, 1.2471186673827963], [1.3500275417978815, 0.1682664936410564], [1.3682647960166707, 1.1681001810904037], [0.42185436542754084, 1.4910665831697814], [0.40361711120875166, 0.4912328957204339], [0.35435246383826025, 0.2611704178321412], [0.37258971805704966, 1.2610041052814884], [1.0986895593314756, -0.799632890886962], [1.6241546523362889, 0.05118226884718524], [0.6313384479738151, -0.06846715386471014], [1.5170785795318569, 1.1542147431917118], [0.5331031270396522, 1.3325187220950467], [1.4166907631658963, 0.22291139731717446], [0.7702872615936611, -0.5400843517896953], [1.4807937182941742, 1.2664647539853577], [0.7873652484440377, 0.5459392810135145], [0.9884614683231048, 1.1471047657472053], [0.05107971852844995, 0.7988010833780526], [0.9486231373001242, 0.49856330646663494], [0.5161460334862547, 0.589597468615388], [-0.4678294190059502, 0.7679014475187227], [-1.4317238655477396, 0.5016168330065485], [0.4641556426867042, 0.2100996606591965], [1.377596522259691, 0.617071108705527], [0.5251747142807299, -0.08167261583314223], [-0.43871973226105965, -0.34795723034531645], [-1.2202242141043151, 0.8209747582539961], [0.9149059487458746, 1.8142114230717898], [0.5275145076982053, 0.21095184665845568], [0.6002091012679673, 0.8454598525445117], [-0.09321936858216884, 0.12493437957266845], [1.674056115518809, 0.5859889917834579], [1.1999611197665292, 1.466462690994653], [0.23606667322473962, 1.2001780764824785], [-0.1466010768041912, 0.8354151464447039], [0.7130205425741665, 0.10715103036830398], [0.6744960267617155, 0.6156475312605059], [-0.2893984197800741, 0.34936291674833153], [0.7101616689770194, 0.3197043772712836], [0.6919244147582297, -0.680129310178064], [0.6523874209917238, 0.2557341744369427], [1.7207075985924827, 0.7279819024401859], [0.9365070255500658, 1.4629191949030127], [0.5191266450309988, 2.3716510684266385], [0.6851690430836603, 0.4950198103749942], [2.2738659143934936, 1.8755924021628159], [1.520963590329329, 1.217460127905749], [1.3461561902466785, 0.8081868242668402], [0.011241387505469325, 0.15025962409748178], [1.082420438703375, 0.4099540051977453], [1.6873817497946542, 1.3165495189210072], [1.4306465458113249, 1.0322809451836297], [0.7996775862318015, 1.7769638084398882], [1.2499203633251716, 2.6698699858601422], [0.37492036332517165, 2.1857470675842148], [0.7034062973024497, 1.4948534978243422], [-0.2430041332866799, 1.8178198999037196], [0.9344794257304896, 0.6584172446639405], [0.952716679949279, 1.658250932113288], [1.1893458838207076, 2.25569098380938], [1.1152828352776778, 2.2115920561416247], [0.6706246752105134, 1.2555678618862895], [-0.2534314403082958, 1.637824706654686], [1.6452036253541977, 0.3753835971488375], [1.0970455810588882, 1.2117583686922775], [2.080821260092055, 0.4664854110745593], [1.3279189360278898, -0.19164686318250757], [0.5590967436071159, 1.7556774636326469], [-0.25632976756252623, 1.08725937276617], [1.443428469850136, 1.6887713095236978], [0.35743634737251173, 1.596496050992663], [1.0506398072855427, 0.7691425439010047], [2.0434560116480176, 0.8887919666128999], [1.072748413055535, 1.1290559007245682], [0.12633798246640526, 1.4520223028039454], [1.3282946974405538, 1.7840737858843945], [0.33547849307807986, 1.664424363172499], [0.935506034875961, 0.8644450202617014], [0.07874328909475037, 1.380155789435122], [1.0819569864798488, 2.8001596726224456], [1.06371973226106, 1.8003259851730982], [0.3290546624156838, 2.1420273983653795], [0.641187839525398, 3.0920657553528197], [0.2914896405126086, 1.9973649751893363], [1.2898904619012885, 2.0538963810661506], [0.29664115272920366, 1.6143570785332406], [0.8966686945270848, 0.8143777356224429], [0.5820366231546772, 1.0985355233141583], [0.5637993689358878, 0.09870183586481085], [1.0241271105384078, 0.851161611757983], [0.23634833055158078, 1.5948673994599336], [-0.5475318927356614, 1.775862688455648], [0.0693169727472398, 1.7986347708273995], [1.1072547879024266, 0.9193026920892076], [-0.9236871605394392, 2.6482088341792647], [0.5242623751693827, 1.0345653204798166], [-0.4597130773228222, 1.2128692993831516], [-0.12214112640285302, 0.27156957137294646], [0.3801778314111519, 2.8432682663516236], [-0.06978328835537706, 2.3450514119088255], [-0.14528726159366145, 1.9924531066174764], [0.2732523862938192, 0.9975312877399896], [0.06120063106411133, 1.3536669189629709], [0.6029348446970264, 0.9948951826124105], [0.6110511863801538, 1.4398630344768393], [0.0, 1.0408340855860843e-16], [0.6234898018587335, -0.7818314824680297], [1.412393243346499, -0.16731430613464227], [1.9123932433464992, 0.6987110976497966], [1.3396706857583365, -0.121038176231964], [1.500000000000002, 0.8660254037844397], [0.8602566989790265, -0.5098611692036452], [1.7952434915444768, -0.1551786383000625], [2.2952434915444773, 0.7108467654843759], [1.7225209339563146, -0.10890250839738473], [0.9555728057861406, -0.2947551744109042], [1.2612255668814452, 0.6573878766243993], [0.6377357650227119, -0.12444360584363032], [0.499999999999998, 0.8660254037844397], [0.16032931424166352, 0.9870635800164027], [0.6603293142416637, 1.8530889838008409], [1.5612981821440828, 2.2869727229183994], [0.8282463103142567, 1.6067999851474801], [-0.137735765022712, 0.9904690096280693], [0.3622642349772881, 1.8564944134125079], [1.0507672442705709, 2.581727897329304], [-0.3653410243663948, 0.9308737486442044], [0.6120654264146537, 1.1422422183164662], [-0.36025669897902657, 1.3758865729880843], [-0.5, 0.8660254037844387], [0.2889034414877656, 1.480542580117826], [1.2889034414877654, 1.4805425801178262], [-0.6234898018587332, 0.7818314824680299], [0.2367668971202932, 0.27197031326438426], [0.7367668971202935, 1.1379957170488229], [0.8114969907067175, 0.14079191986764267], [-0.7330518718298262, 0.6801727377709195], [0.12720482714920012, 0.17031156856727397], [1.1272048271492001, 0.17031156856727386], [0.5037150252904667, 0.9521430510353037], [-0.774657478323787, 0.6323810491128204], [-0.35109506120241574, -0.27348587201980457], [-0.48883082622512797, 0.716983137608264], [0.5111691737748719, 0.7169831376082637], [-0.8262387743159947, 0.5633200580636221], [-0.3262387743159949, 1.4293454618480608], [0.1737612256840052, 0.563320058063622], [1.0, 2.7755575615628914e-17], [-0.900968867902419, 0.433883739117558], [0.03401792466303144, 0.7885662700211411], [1.0114243754440797, 0.9999347396934029], [-0.40096886790241926, 1.2999091429019964], [0.5764375828786292, 1.5112776125742586], [0.5650132074345493, 0.511342872880856], [-0.97232212539368, 0.23364435467161776], [-0.7498011914373657, 1.2085722668534413], [-0.3262387743159947, 0.3027053457208162], [0.6737612256840051, 0.30270534572081664], [-1.3376631497600746, 1.164518103315822], [-0.47740645078104804, 0.6546569341121767], [0.5225935492189517, 0.6546569341121768], [-0.9888308262251284, 0.14904226617617466], [-0.011424375444080148, 0.3604107358484366], [0.48857562455591985, 1.226436139632875], [0.7110965585122342, 0.25150822745105145], [-0.9009688679024195, -0.43388373911755773], [-0.47740645078104826, -1.339750660250183], [0.022593549218951625, -0.4737252564657444], [-0.8262387743159949, -0.5633200580636221], [0.03401792466303177, -1.0731812272672676], [0.19434723890469519, -0.0861176472508646], [-0.7330518718298265, -0.6801727377709192], [-0.06129818214408278, -1.4209473191339608], [0.8396706857583365, -0.9870635800164026], [-0.5727225575881629, -0.8197492738817607], [0.36226423497728777, -0.46506674297817735], [0.8622642349772879, 0.4009586608062611], [0.6397433010209737, 1.3758865729880845], [-0.4123932433464992, 0.16731430613464254], [0.08760675665350082, 1.033339709919081], [-0.4979924640017386, 0.17745452329941977], [0.4794139867793097, 0.38882299297168155], [-0.7952434915444768, 0.15517863830006318], [0.13974330102097376, 0.5098611692036462], [-0.36534102436639504, -0.9308737486442041], [-0.6491601404667924, 0.028004089944210164], [0.35083985953320773, 0.028004089944210206], [0.5111691737748717, 1.0150676699606132], [-0.011424375444079704, -0.9999347396934024], [0.9441484303420609, -1.2946899141043064], [1.9215548811231093, -1.0833214444320447], [1.6377357650227125, -0.12444360584363029], [0.4885756245559202, -0.13390933590896398], [1.4659820753369688, 0.07745913376329817], [-0.7444762472739057, -0.31976200192248294], [-0.24447624727390616, 0.5462634018619557], [0.07473009358642402, -0.9972037971811801], [0.9349867925654507, -1.5070649663848257], [0.3114969907067171, -0.7252334839167959], [0.2225209339563139, -0.9749279121818236], [0.7225209339563141, -0.1089025083973848], [1.6999273847373626, 0.102465961274877], [0.36534102436639426, -0.9308737486442045], [1.3003278169318448, -0.5761912177406217], [0.8653410243663946, -0.06484834485976568], [1.4888308262251282, 0.7169831376082639], [0.42356241712137055, -0.905866921132625], [1.2124658586091366, -0.2913497447992376], [1.7124658586091364, 0.5746756589852008], [0.348832323534947, 0.09133687604855498], [0.5, -0.8660254037844386], [1.3716810744231065, -0.3759518332946817], [1.7952434915444775, -1.2818187544273068], [1.9555728057861408, -0.294755174410904], [2.455572805786141, 0.5712702293735346], [1.6234898018587334, -0.7818314824680297], [0.8488323235349461, -0.1494504333552093], [1.2141733479013417, 0.7814233152889947], [0.7838191161003971, 0.2052320975483732], [1.783819116100397, 0.2052320975483733], [2.772649942325526, 0.056189831372198684], [1.7838191161003976, -0.09285243480397559], [-0.15116767646505336, -0.14945043335520924], [0.21417334790134168, 0.7814233152889951], [0.07643758287862978, 1.7718923249170642], [1.0764375828786295, 1.7718923249170642], [0.6717536896857437, -0.7407745813630414], [1.572722557588163, -0.30689084224548346], [1.434986792565451, 0.6835781673825857], [1.5953161068071144, 1.6706417473989887], [1.3602566989790263, -0.015541097446245433], [2.093308570808853, -0.695713835217165], [2.5933085708088526, 0.17031156856727378], [0.7464837832721678, 0.2564292158181386], [1.7464837832721678, 0.25642921581813877], [2.0521365443674724, 1.2085722668534418], [1.2774790660436852, 1.8409533159662623], [0.5970235960993197, 0.2564292158181386], [1.4979924640017384, -0.1774545232994193], [0.9252699064135756, -0.9972037971811801], [0.5340179246630318, 0.24969442826502775], [1.5303028993725654, 0.16357678101416318], [0.9575803417844025, -0.6561724928675973], [0.38793457358534666, 0.2181032572253728], [0.8879345735853468, 1.0841286610098113], [0.15488270175552044, 0.40395592323889207], [0.5990311320975809, 1.2999091429019969], [0.7330518718298262, -0.6801727377709192], [1.7330518718298262, -0.6801727377709192], [0.8727951728507999, -0.17031156856727353], [1.3727951728508, 0.6957138352171646], [1.7104583226108747, -0.4688042680986573], [2.3989613319041574, 0.2564292158181387], [2.8989613319041574, 1.1224546196025773], [2.326238774315995, 0.30270534572081653], [1.421554881123109, 0.04506074614587685], [0.49491567461263164, 0.4210125794405588], [1.4949156746126313, 0.42101257944055864], [0.4492327557294291, 0.27870510081749506], [0.9492327557294293, 1.1447305046019336], [1.1717536896857434, 2.119658416783757], [0.8262387743159949, -0.5633200580636221], [1.826238774315995, -0.5633200580636224], [1.6885030092932831, 0.4271489515644469], [1.0650132074345495, -0.35468253090358276], [1.7612255668814454, -0.20863752716003892], [2.2612255668814454, 0.6573878766243992], [3.1214822658604717, 0.14752670742075402], [2.123489801858734, 0.0841939213164091], [1.326238774315995, 0.3027053457208164], [0.32824631031425655, 0.23937255961647197], [-0.2952434915444768, 1.021204042084502], [1.8602566989790266, -0.5098611692036455], [1.9349867925654507, -1.5070649663848257], [1.0340179246630314, -1.0731812272672676], [1.194347238904695, -0.0861176472508646], [1.8376631497600746, -0.29849269953138335], [1.2141733479013417, 0.4833387829366464], [1.9888308262251284, -0.14904226617617378], [1.3602566989790266, 0.35616423458079316], [0.5340179246630319, -0.20715582348282907], [1.4349867925654514, -0.6410395626003871], [1.165909460074331, 0.44228188183165795], [0.593186902486168, -0.3774673920501024], [1.264940592171912, -1.1182419734131441], [1.02058601322069, 0.4772024108127572], [2.02058601322069, 0.47720241081275716], [2.0953161068071138, -0.520001386368423], [2.400968867902419, 0.43214166466688086], [0.9009688679024191, -0.4338837391175581], [1.9009688679024193, -0.43388373911755784], [0.9286467425087391, -0.20023938444594025], [1.6171497518020221, 0.5249940994708557], [0.6283189255768934, 0.37595183329468146], [1.7726499423255255, 0.056189831372198684], [2.2726499423255255, -0.8098355724122399], [1.2838191161003971, -0.9588778385884145], [1.4009688679024193, 0.43214166466688053], [1.0953161068071144, -0.5200013863684229], [0.8114969907067175, 0.4388764522199912], [0.6737612256840054, 1.4293454618480608], [1.3451172982444797, 0.46206948054554675], [0.6120654264146532, -0.2181032572253725], [1.1120654264146537, 0.647922146559066], [1.932979256567189, -0.08338670473864229], [2.7218826980549546, 0.5311304715947451], [2.710458322610875, -0.4688042680986573], [1.7444762472739062, 0.3197620019224833], [2.744476247273906, 0.31976200192248305], [1.8178370407634286, 0.6957138352171652], [1.4555728057861408, 0.5712702293735343], [1.3178370407634288, 1.5617392390016034], [0.6943472389046953, 0.7799077565335738], [0.8178370407634286, 0.6957138352171652], [-0.137735765022712, 0.4009586608062609], [0.9888308262251285, -0.1490422661761745], [1.677333835518411, 0.5761912177406217], [1.053844033659678, -0.2056402647274082], [1.9888308262251286, 0.14904226617617455], [2.4888308262251284, 1.015067669960613], [0.9962849747095335, -0.08611764725086457], [1.4962849747095333, -0.9521430510353031], [1.2124658586091361, 0.006734787553111188], [1.0747300935864241, 0.9972037971811805], [1.5747300935864244, 1.8632292009656188], [1.9962849747095333, -0.08611764725086468], [1.0696457681990559, 0.28983418604381717], [1.4349867925654507, 1.2207079346880212], [1.9774064507810483, 0.21136846967226194], [2.837663149760075, -0.29849269953138363], [1.8488323235349462, -0.14945043335520936], [0.8765101981412664, 0.08419392131640882], [1.8262387743159947, 0.5633200580636217], [1.253516216727832, -0.25642921581813855], [0.753516216727832, 0.6095961879663001], [1.4252699064135754, -0.13117839339674142], [0.022593549218951736, -0.21136846967226244], [0.3879345735853459, -1.1422422183164669], [0.8879345735853461, -0.2762168145320285], [1.1377357650227122, -0.990469009628069], [1.9555728057861401, 0.2947551744109042], [2.4555728057861406, 1.160780578195343], [0.9888308262251285, 0.1490422661761745], [0.4161082686369657, -0.670707007705586], [-0.35854920968682125, -0.03832595859276547], [0.6414507903131788, -0.03832595859276561], [0.9774064507810485, 0.2113684696722622], [1.9123932433464987, 0.5660510005758448], [1.8376631497600746, 1.563254797757025], [0.8653410243663946, 1.7968991524286428], [1.865341024366395, 1.7968991524286428], [0.9555728057861408, 0.2947551744109042], [0.38285024819797786, -0.5249940994708562], [0.8828502481979779, 0.34103130431358264], [1.0431795624396416, 1.3280948843299853], [0.5990311320975805, 0.43214166466688053], [0.9349867925654506, 0.354682530903583], [-0.02058601322069009, 0.6494377053144871], [0.8510950612024163, 1.1395112758042438], [1.8510950612024164, 1.1395112758042438], [0.9009688679024191, 0.433883739117558], [1.9009688679024193, 0.43388373911755773], [1.1263113895786323, 1.0662647882303784], [2.061298182144083, 1.420947319133961], [1.1346589756336045, 1.7968991524286428], [0.45682043756035806, -0.4620694805455466], [0.6171497518020221, 0.5249940994708562], [1.117149751802022, 1.3910195032552948], [0.5444271942138592, 0.5712702293735346], [0.8716810744231064, 0.49007357048975686], [1.8716810744231063, 0.4900735704897571], [1.649160140466792, 1.4650014826715805], [0.7225209339563143, 1.8409533159662625], [1.7225209339563148, 1.8409533159662623], [0.8262387743159949, 0.5633200580636218], [0.2535162167278322, -0.2564292158181386], [2.2498011914373652, -0.34254686306900334], [1.2774790660436859, -0.10890250839738491], [0.7889034414877655, 0.6145171763333875], [-0.1666693642983752, 0.31976200192248344], [0.05585156965793914, 1.294689914104307], [0.21618088389960255, 2.28175349412071], [0.7330518718298263, 0.6801727377709194], [1.5933085708088528, 0.1703115685672738], [2.0933085708088526, 1.0363369723517124], [1.137735765022712, 1.3310921467626167], [2.072722557588163, 1.6857746776661993], [0.6885030092932827, 0.7252334839167962], [1.477406450781048, 1.3397506602501836], [1.402676357194623, 0.3425468630690035], [0.5764375828786282, 0.9058669211326256], [1.1120654264146537, -0.18063343721582928], [0.3374079480908668, 0.45174761189699153], [1.2723947406563174, 0.8064301428005746], [0.6234898018587335, 0.7818314824680299], [1.6234898018587338, 0.7818314824680297], [1.3396706857583363, 1.740709321056444], [0.974329661391941, 0.8098355724122399], [1.4266392065104778, 0.49007357048975675], [1.9266392065104776, 1.3560989742741956], [1.1935873346806514, 0.6759262365032759], [1.8653410243663953, -0.06484834485976558], [1.4888308262251284, 1.0150676699606132], [1.9123932433464996, 0.1092007488279878], [2.4123932433464996, -0.7568246549564508], [1.4774064507810485, 1.0773938734567006], [1.112065426414654, 2.008267622100905], [1.9723221253936802, 1.4984064528972594], [0.9999999999999999, 1.7320508075688776], [1.4555728057861406, 1.160780578195343], [1.0114243754440797, 0.26482735853223816], [0.4387018178559171, -0.5549219153495224], [1.2330518718298262, 1.546198141555358], [1.0727225575881627, 0.5591345615389549], [2.072722557588163, 0.5591345615389549], [1.1885030092932827, 1.5912588877012346], [2.1773338355184113, 1.4422166215250598], [2.1659094600743316, 0.4422818818316576], [-0.3716810744231065, 0.37595183329468174], [0.583891731363034, 0.6707070077055863], [1.5727225575881627, 0.8197492738817607], [-0.6773338355184109, -0.5761912177406214], [0.36534102436639504, 0.9308737486442042], [0.35391664892231556, -0.06906099104919824], [0.8539166489223153, 0.7969644127352402], [1.8313230997033638, 1.0083328824075022], [0.3056527610953045, 0.9521430510353035], [0.8056527610953045, 0.08611764725086496], [0.9659820753369681, 1.073181227267268], [1.4659820753369681, 1.9392066310517064], [2.326238774315995, 1.4293454618480608], [1.3056527610953048, 0.9521430510353034], [0.47941398677931, 1.5154631090989255], [1.212465858609136, 2.1956358468698447], [0.22252093395631434, 0.9749279121818235], [0.7763649676159924, 0.14230747862306364], [-0.21246585860913614, 0.29134974479923825], [0.2110965585122342, -0.614517176333387], [1.2225209339563143, 0.9749279121818236], [2.2188059086658476, 0.8888102649309589], [1.6460833510776847, 0.06906099104919844], [2.4723221253936796, 0.6323810491128203], [1.1491601404667922, 0.8380213138402284], [1.572722557588163, -0.06784560729239662], [1.795243491544478, 0.907082304889427], [1.1603293142416633, 0.9870635800164026], [0.42727744241183707, 0.30689084224548346], [0.20475650845552307, 1.2818187544273072], [1.137735765022712, 1.198432049688665], [0.40468389319288567, 0.5182593119177455], [1.3602566989790263, 0.8130144863286499], [1.0612981821440828, 1.4209473191339608], [1.9215548811231091, 0.9110861499303154], [1.2330518718298265, 0.18585266601351919], [1.03201038866477, 1.4771371505061597], [2.0320103886647702, 1.4771371505061597], [2.3376631497600746, 2.429280201541463], [1.5114243754440797, 1.865960143477841], [0.9492327557294291, 1.6015807563497906], [1.8209138301525354, 2.0916543268395476], [1.8094894547084561, 1.091719587146145], [1.171753689685743, 0.6266528441679668], [2.1491601404667917, 0.8380213138402286], [1.025670338608058, 0.9222152351566368], [1.997992464001738, 0.6885708804850192], [0.4659820753369681, 1.9392066310517064], [-0.4349867925654509, 2.3730903701692645], [0.5650132074345491, 2.3730903701692645], [1.4252699064135754, 1.8632292009656188], [0.022593549218951625, 1.9775325896444724], [0.4461559663403224, 1.071665668511847], [-0.17733383551841087, 0.2898341860438171], [0.07473009358642402, 0.9972037971811802], [1.0521365443674724, 1.2085722668534422], [2.0501290083692107, 1.271905052957787], [1.1234898018587334, 1.647856886252469], [2.1234898018587343, 1.6478568862524683], [0.5747300935864241, 1.8632292009656188], [1.5521365443674728, 2.074597670637881], [1.7746574783237867, 1.0996697584560573], [0.23505940782808743, 1.984267377197583], [0.9068130975138312, 1.2434927958345414], [1.7330518718298262, 0.6801727377709192], [2.2330518718298267, 1.5461981415553576], [-0.07473009358642413, 0.9972037971811801], [0.9026763571946242, 1.2085722668534422], [1.4026763571946244, 2.0745976706378806], [1.9026763571946244, 1.2085722668534422], [0.6885030092932833, 0.4271489515644471], [0.8488323235349466, 1.4142125315808498], [0.22534252167621305, 0.6323810491128203], [0.8585492096868214, 0.9043513623772046], [1.8585492096868212, 0.9043513623772044], [0.9029764039006807, 0.6095961879663001], [1.5747300935864246, -0.13117839339674128], [0.862264234977288, 0.9904690096280693], [0.6397433010209739, 0.015541097446245516], [1.3114969907067175, -0.725233483916796], [0.8396706857583363, 1.2018374793003312], [-0.1491601404667926, 1.3508797454765058], [0.7774790660436857, 0.974927912181824], [0.5953161068071142, 1.6706417473989887], [0.311496990706717, 2.6295195859874028], [0.5340179246630312, 1.6545916738055797], [0.5507672442705707, 1.7157024935448653], [-0.07272255758816248, 0.9338710110768353], [0.925269906413576, 0.9972037971811804], [-0.5933085708088526, 1.5617392390016036], [0.13974330102097376, 0.8815665012306841], [0.2858266520986591, 0.08460208849544386], [1.2858266520986592, 0.08460208849544351], [1.651167676465054, 1.0154758371396482], [0.8765101981412668, 1.6478568862524685], [-0.2225209339563139, 0.9749279121818235], [0.6377357650227125, 0.4650667429781781], [1.637735765022712, 0.46506674297817807], [0.6821629592365712, 0.17031156856727397], [0.5444271942138595, 1.1607805781953429], [0.6491601404667924, 1.4650014826715805], [-0.08389173136303407, 0.7848287449006612], [0.9161082686369658, 0.7848287449006615], [-0.28381911610039756, 0.9588778385884146], [0.5764375828786293, 0.4490166693847688], [0.6511676764650536, -0.5481871277964114], [1.4774064507810483, -1.1115071858600332], [0.7050117101247315, 0.8098355724122399], [1.2050117101247313, 1.6758609761966783], [0.2161808838996031, 1.8249032423728528], [0.7124658586091362, 0.8727601913375498], [1.7124658586091364, 0.8727601913375497], [0.7858266520986587, 1.2487120246322316], [0.7161808838996027, 0.9588778385884142], [1.6717536896857434, 1.2536330129993187], [1.4492327557294296, 0.2787051008174948], [2.426639206510478, 0.49007357048975664], [0.6935873346806511, 1.1702463082606762], [1.693587334680651, 1.1702463082606762], [2.6491601404667917, 1.4650014826715805], [0.6717536896857434, 1.253633012999319], [2.5727225575881625, 0.8197492738817604], [2.0, 5.551115123125783e-17], [0.5424196582155979, 1.5221978966520362], [-0.030302899372564918, 0.7024486227702758], [0.9659820753369686, 0.6163309755194112], [-0.42155488112310935, 1.9493468482164835], [0.5784451188768909, 1.9493468482164833], [1.4046838931928858, 1.3860267901528616], [0.3064126653193491, 0.19009916728116277], [-0.6659094600743309, 0.42374352195278064], [0.33409053992566906, 0.42374352195278053], [1.1666693642983754, -0.31976200192248283], [1.4723221253936798, 0.6323810491128207], [0.13465897563360518, 1.7968991524286428], [0.8064126653193491, 1.0561245710656013], [1.1717536896857434, 0.12525082242139707], [0.1717536896857439, 0.12525082242139712], [0.2464837832721678, 1.1224546196025773], [1.2464837832721674, 1.122454619602577], [0.4349867925654507, 1.2207079346880216], [0.8003278169318448, 0.28983418604381717], [1.8003278169318455, 0.28983418604381694], [0.977406450781048, 2.2057760640346222], [1.977406450781048, 2.2057760640346222], [0.18850300929328279, 1.5912588877012348], [0.8602566989790267, 0.8504843063381932], [1.8602566989790263, 0.8504843063381932], [-0.5114243754440795, -0.13390933590896384], [-1.2444762472739055, 0.5462634018619559], [-0.45557280578614034, 1.160780578195343], [0.3727951728508003, 0.6957138352171655], [1.2444762472739066, 1.1857874057069224], [2.244476247273907, 1.1857874057069218], [2.2330518718298267, 0.18585266601351913], [0.37651019814126674, 0.7818314824680299], [1.2481912725643731, 1.2719050529577869], [1.2367668971202932, 0.27197031326438403], [0.06501320743454941, 1.5070649663848263], [0.4885756245559205, 0.6011980452522012], [1.3488323235349466, 0.09133687604855514], [-0.12348980185873315, 1.6478568862524687], [0.6511676764650542, 1.0154758371396482], [-0.3376631497600746, 1.164518103315823], [-1.6214822658604717, 0.7184986963636855], [-0.7612255668814452, 0.20863752716003972], [-0.2612255668814454, 1.0746629309444782], [0.26323310287970725, 0.594055090520055], [0.25180872743562777, -0.4058796491733475], [1.1234898018587336, 0.08419392131640932], [-0.23305187182982623, 1.546198141555358], [0.6272048271492006, 1.0363369723517124], [0.0037150252904671888, 1.8181684548197425], [0.25552375272609396, -0.31976200192248305], [1.2110965585122346, -0.6145171763333873], [-0.309489454708455, -0.22569418336170557], [-0.3209138301525345, -1.225628923055108], [0.5507672442705713, -0.7355553525653513], [-0.27465747832378695, -0.2336443546716181], [-0.05213654436747306, -1.2085722668534418], [0.44786345563252716, -0.3425468630690034], [1.0970235960993193, 1.1224546196025775], [0.42526990641357565, 1.8632292009656184], [-0.699927384737363, 1.6295848462940006], [-0.028173695051619152, 0.8888102649309592], [0.2774790660436852, 1.8409533159662628], [-0.9123932433464987, 1.6228500587408896], [0.08760675665350104, 1.6228500587408892], [1.0838917313630343, 1.536732411490025], [-0.5521365443674731, -0.34254686306900317], [-0.24648378327216847, 0.6095961879663003], [0.4252699064135753, -0.1311783933967412], [0.15116767646505358, 0.7746885277358839], [-0.4215548811231091, -0.045060746145876365], [0.5747300935864246, -0.13117839339674114], [-0.4026763571946237, -0.342546863069003], [0.0973236428053762, 0.5234785407154355], [0.09903113209758074, 0.43388373911755845], [0.4046838931928858, 1.3860267901528613], [1.400968867902419, 1.2999091429019967], [2.400968867902419, 1.2999091429019969], [0.07643758287862923, 0.6452522087898201], [1.0764375828786292, 0.6452522087898199], [0.2501988085626343, 1.208572266853442], [-0.5953161068071147, 1.3860267901528616], [-0.5205860132206905, 0.38882299297168155], [0.3396706857583358, -0.121038176231964], [-0.9266392065104776, 0.3759518332946819], [-0.4266392065104776, -0.49007357048975675], [0.5507672442705707, -0.2787051008174948], [1.0507672442705709, 0.5873203029669437], [0.7669481281701737, 1.546198141555358], [0.0733607934895224, 0.3759518332946818], [-0.21045832261087472, 1.3348296718830959], [0.7783725036142537, 1.1857874057069218], [0.7669481281701735, 0.18585266601351913], [0.050767244270570666, 0.5873203029669438], [0.26694812817017377, 0.6801727377709197], [-0.025670338608058563, 0.8098355724122399], [0.47432966139194144, -0.05618983137219871], [0.6346589756336053, 0.930873748644204], [-1.210458322610875, 1.3348296718830963], [-1.2218826980549542, 0.3348949321896937], [-0.43297925656718883, 0.9494121085230813], [-0.5612981821440834, -0.5549219153495226], [0.37368861042136703, -0.20023938444593964], [-0.4009688679024195, 0.43214166466688075], [-0.9555728057861406, 0.29475517441090404], [-0.7952434915444773, 1.281818754427307], [-0.2952434915444768, 2.147844158211745], [0.12831892557689384, 1.2419772370791207], [-0.47232212539368, -0.6323810491128208], [-0.24980119143736568, 0.3425468630690028], [0.17376122568400554, -0.5633200580636222], [-0.11206542641465345, -0.2762168145320277], [0.5114243754440799, 0.5056146679360022], [1.51142437544408, 0.5056146679360023], [2.4888308262251284, 0.7169831376082643], [0.027677874606320096, 0.2336443546716181], [-0.33766314976007505, -0.6972293939725863], [-0.4123932433464992, 0.299974403208594], [0.00508432538736836, 0.4450128243438798], [1.0050843253873682, 0.4450128243438797], [0.07844511887689087, 0.8209646576385619], [0.9400711179528191, 0.7996953552474628], [1.6285741272461018, 1.524928839164259], [-0.1460833510776851, 0.7969644127352395], [-0.7188059086658476, -0.02278486114652012], [-0.4962849747095337, 0.9521430510353031], [-0.34883232353494664, 1.0154758371396477], [0.33967068575833625, 1.7407093210564437], [0.8396706857583365, 2.6067347248408828], [0.011169173774871144, 0.1490422661761744], [0.9885756245559199, 0.3604107358484364], [1.6120654264146532, 1.1422422183164664], [0.2949882898752685, -0.8098355724122399], [-0.19992738473736305, 0.7635594425095622], [0.30007261526263695, 1.6295848462940006], [0.7236350323840082, 0.7237179251613756], [-0.48883082622512863, 1.015067669960613], [0.06501320743454964, 0.18244723640185304], [0.28753414139086386, 1.1573751485836765], [-0.9888308262251286, -0.1490422661761746], [-0.12857412724610207, -0.6589034353798201], [0.5599288820471806, 0.06633004853697605], [0.6346589756336046, -0.930873748644204], [1.134658975633605, -0.06484834485976551], [0.01116917377487181, -0.1490422661761747], [0.6996721830681543, 0.5761912177406215], [-0.27264994232552575, 0.8098355724122394], [-0.4123932433464992, 0.7568246549564503], [-0.11714975180202214, 0.34103130431358225], [0.5713532574912608, 1.0662647882303784], [-0.30032781693184596, 0.5761912177406215], [-0.8003278169318457, 1.4422166215250602], [0.19967218306815426, 1.44221662152506], [-0.7726499423255255, 1.675860976196678], [0.0020075359982615337, 1.0434799270838582], [-1.2726499423255255, 0.8098355724122397], [-0.2838191161003971, 0.6607933062360654], [0.7161808838996029, 0.6607933062360654], [-0.9555728057861408, -0.29475517441090404], [-0.09531610680711422, -0.8046163436145497], [0.2774790660436859, -0.10890250839738451], [-0.4555728057861408, 0.5712702293735347], [0.3333306357016249, 1.1857874057069218], [0.5558515696579391, 2.1607153178887453], [0.09531610680711411, -0.5200013863684223], [0.9215548811231089, -1.0833214444320443], [-0.1885030092932829, 0.43887645221999205], [0.09903113209758074, -0.43388373911755795], [0.5225935492189515, -1.3397506602501832], [1.423562417121371, -0.9058669211326249], [-0.8376631497600744, -1.5632547977570246], [-0.5320103886647698, -0.6111117467217211], [0.4679896113352301, -0.6111117467217211], [0.26694812817017344, -0.6801727377709195], [-0.5933085708088528, -0.17031156856727347], [-0.0933085708088528, 0.6957138352171652], [-0.23305187182982623, 0.18585266601351924], [0.2110965585122343, 1.0818058856766242], [1.188503009293283, 1.2931743553488855], [-0.5727225575881629, 0.3068908422454836], [-0.14916014046679194, -0.5989760788871417], [0.8064126653193489, -0.3042209044762374], [-0.6234898018587332, -0.78183148246803], [0.3765101981412664, -0.7818314824680298], [-0.550129008369211, -0.4058796491733483], [0.23877443311855462, 0.20863752716003942], [0.3745026621430052, -0.718498696363685], [0.44923275572942933, -1.7157024935448653], [0.9492327557294294, -0.8496770897604266], [0.3114969907067173, -0.42714895156444704], [1.2889034414877658, -0.21578048189218502], [1.1511676764650534, 0.7746885277358841], [-0.3178370407634288, 0.17031156856727347], [0.18216295923657122, -0.6957138352171651], [0.04442719421385943, 0.2947551744109043], [0.28753414139086375, -1.3296104430854059], [1.02058601322069, -0.6494377053144864], [0.06501320743454941, -0.35468253090358237], [0.32824631031425633, -1.2536330129993185], [0.40297640390068046, -0.25642921581813843], [1.0747300935864244, -0.9972037971811798], [0.21618088389960266, -0.2052320975483729], [1.2050117101247313, -0.05618983137219846], [1.7050117101247313, 0.80983557241224], [-0.07272255758816293, 0.04627612990267821], [-0.8989613319041574, 0.6095961879663001], [-0.07272255758816248, 1.1729162460299218], [-1.072722557588163, 0.046276129902678265], [-0.17175368968574345, -0.38760760921487986], [0.8282463103142566, -0.38760760921488], [-1.3989613319041574, -0.2564292158181384], [-1.093308570808853, 0.6957138352171652], [0.5696457681990554, -0.5761912177406212], [-0.05384403365967749, 0.20564026472740876], [0.9235624171213707, 0.4170087343996709], [0.13465897563360463, -0.06484834485976533], [0.2949882898752684, 0.9222152351566375], [1.2838191161003967, 1.071257501332812], [0.28582665209865843, 0.3826866208477929], [-0.22252093395631412, -0.9749279121818237], [0.7774790660436857, -0.9749279121818235], [1.649160140466792, -0.4848543416920664], [0.6603293142416637, -0.3358120755158922], [-0.3602566989790259, 0.015541097446245572], [1.3282463103142566, 0.7407745813630415], [-1.1491601404667917, -0.5989760788871418], [-0.19358733468065092, -0.3042209044762376], [0.7838191161003975, -0.09285243480397559], [-0.07473009358642502, -0.99720379718118], [0.7141733479013405, -0.3826866208477926], [0.4303542318009438, 0.5761912177406217], [1.3653410243663942, 0.9308737486442046], [-0.21246585860913703, -0.0067347875531108825], [0.2875341413908632, 0.8592906162313276], [0.7875341413908632, -0.006734787553111021], [0.6603293142416642, -1.740709321056444], [0.16032931424166352, -0.8746839172720056], [-0.12348980185873315, 0.08419392131640888], [0.9659820753369688, -0.7885662700211404], [1.1885030092932827, -1.763494182202964], [0.9046838931928853, -0.8046163436145499], [0.923562417121371, -0.6452522087898195], [0.0973236428053762, -1.2085722668534415], [0.5973236428053762, -0.34254686306900334], [0.7216274963857467, -0.31976200192248316], [-0.20501171012473085, 0.05618983137219871], [0.7949882898752686, 0.05618983137219885], [1.2216274963857465, 0.5462634018619553], [0.6489049387975843, -0.27348587201980507], [0.21109655851223474, -0.025006827511578927], [-0.28890344148776514, 0.8410185762728597], [0.7110965585122346, 0.8410185762728599], [-0.29524349154447704, -0.04105690110498799], [0.6935873346806515, -0.19009916728116263], [0.32824631031425666, 0.7407745813630418], [0.7632331028797066, -0.27197031326438403], [1.263233102879707, 0.5940550905200546], [1.125497337856995, 1.5845241001481236], [1.222520933956314, -0.9749279121818235], [1.5281736950516185, -0.02278486114652012], [2.199927384737362, -0.7635594425095615], [2.2746574783237867, 0.23364435467161834], [1.1999273847373622, -0.7635594425095615], [0.9161082686369653, 0.19531839607885293], [0.04442719421385877, -0.29475517441090393], [1.0114243754440793, -0.36041073584843614], [0.2783725036142535, 0.31976200192248344], [1.274657478323787, 0.23364435467161865], [-0.33132309970336404, -0.14230747862306378], [0.6460833510776844, 0.06906099104919843], [1.1460833510776844, 0.935086394833637], [1.3653410243663946, -0.9308737486442042], [2.300327816931845, -0.5761912177406218], [1.525670338608058, 0.05618983137219857], [1.5538440336596775, 0.6603851390570306], [0.9303542318009442, 1.4422166215250602], [0.5256703386080581, 0.05618983137219856], [1.1603293142416629, -0.8746839172720055], [0.22760525934368225, 0.059595260983864716], [1.1625920519091328, 0.41427779188744773], [0.3879345735853462, 1.0466588410002682], [1.2838191161003976, -1.4157280903362703], [1.4441484303420606, -0.42866451031986774], [1.6666693642983752, 0.5462634018619559], [1.4123932433464996, -0.7568246549564508], [1.2889034414877658, 0.8910322312960177], [1.2952434915444775, -0.41579335064286815], [2.2952434915444773, -0.41579335064286815], [1.52058601322069, 0.216587698469952], [1.2498011914373661, -0.3425468630690033], [0.4235624171213712, 0.22077319499461875], [1.423562417121371, 0.22077319499461845], [0.9235624171213708, -0.03984151734818661], [1.7498011914373661, 0.5234785407154352], [1.326238774315995, 1.4293454618480606], [0.7889034414877658, 0.025006827511579177], [1.7889034414877658, 0.025006827511579004], [2.01142437544408, 0.9999347396934024], [0.5838917313630345, 0.08119665888377782], [1.5838917313630345, 0.08119665888377747], [0.8508398595332081, -0.5989760788871418], [0.139743301020974, 0.05301091745578923], [1.0953161068071144, 0.3477660918666935], [0.3622642349772882, -0.3324066459042258], [2.365341024366395, 0.9308737486442042], [1.4387018178559172, 1.3068255819388859], [0.4387018178559172, 1.306825581938886], [1.2367668971202925, 1.6323157888062223], [0.23676689712029275, 1.6323157888062227], [1.7774790660436857, 0.9749279121818235], [0.8000726152626372, 0.7635594425095614], [1.7889034414877656, 0.6145171763333871], [-0.7612255668814454, -0.3808728216617685], [1.3765101981412662, -0.7818314824680296], [0.7047565084555225, -0.04105690110498805], [0.9885756245559201, -0.9999347396934024], [1.5, -0.8660254037844385], [0.5733607934895222, -0.49007357048975686], [1.6346589756336045, -0.930873748644204], [1.7774790660436852, -0.9749279121818235], [1.839670685758336, -0.9870635800164028], [2.4123932433464983, -0.16731430613464207], [1.9252699064135756, -0.9972037971811801], [2.0747300935864246, -0.99720379718118], [1.300072615262637, -0.3648227480683601], [2.2838191161003976, -0.9588778385884142], [1.328246310314257, -1.2536330129993183], [1.6603293142416637, -1.7407093210564437], [1.6717536896857434, -0.7407745813630412], [2.365341024366395, -0.9308737486442044], [1.6935873346806511, -0.19009916728116283], [2.8262387743159945, -0.5633200580636224], [3.398961331904157, 0.2564292158181384], [2.900968867902419, -0.433883739117558], [2.912393243346499, 0.5660510005758443], [2.9555728057861406, -0.2947551744109044], [3.2612255668814454, 0.657387876624399], [2.4349867925654505, 1.2207079346880212], [2.9962849747095337, -0.08611764725086496], [2.221627496385746, 0.546263401861955], [2.061298182144083, -0.44080017815444755], [1.0612981821440828, -0.44080017815444733], [2.263233102879707, -0.7662903850217839], [1.4885756245559194, -0.13390933590896392], [2.9888308262251284, 0.14904226617617422], [2.8716810744231065, 0.49007357048975697], [1.1999273847373626, 1.230848151852799], [2.6491601404667913, -0.4848543416920663], [2.826238774315995, 0.5633200580636218], [2.837663149760075, 1.5632547977570244], [2.7330518718298267, 0.6801727377709189], [2.6234898018587343, 0.7818314824680292], [1.8488323235349469, 1.4142125315808494], [1.6460833510776847, 1.1957011071764425], [0.6460833510776847, 1.1957011071764427], [2.5, 0.8660254037844386], [1.6397433010209737, 1.3758865729880843], [2.2124658586091357, 2.195635846869845], [1.4303542318009443, 0.5761912177406215], [2.2050117101247313, -0.05618983137219864], [1.2161808838996024, -0.20523209754837296], [0.4272774424118366, -0.8197492738817602], [2.3056527610953044, 0.9521430510353034], [1.47941398677931, 1.5154631090989257], [0.6905105452915441, 0.9009459327655384], [1.3333306357016248, 1.1857874057069218], [1.5558515696579394, 2.1607153178887453], [1.4046838931928858, 0.5182593119177452], [0.9046838931928858, 1.3842847157021838], [1.47941398677931, 0.38882299297168166], [1.516749319607539, 0.33762587470191624], [1.093186902486168, 1.2434927958345416], [2.0894718771957015, 1.1573751485836765], [1.617149751802022, 0.2269095671185075], [0.6171497518020219, 0.2269095671185078], [2.2225209339563143, 0.9749279121818235], [1.7110965585122342, 0.8410185762728597], [1.5507672442705707, 1.715702493544865], [1.2501988085626345, 1.2085722668534418], [1.3879345735853463, 0.21810325722537272], [1.5340179246630317, 0.2496944282650275], [2.0747300935864246, 0.9972037971811801], [1.300072615262637, 1.6295848462940001], [1.628574127246102, 0.16458336362242032], [1.2632331028797066, -0.7662903850217838], [2.412393243346499, 1.432076404360283], [1.0340179246630317, 0.7885662700211403], [2.0303028993725647, 0.7024486227702753], [2.33595566046787, 1.6545916738055788], [1.9252699064135759, 0.9972037971811802], [1.7090890225139734, 0.9043513623772046], [1.0650132074345495, 1.507064966384826], [1.5650132074345495, 2.3730903701692645], [2.188503009293283, 1.5912588877012346], [0.9272774424118373, 0.9338710110768353], [1.8828502481979779, 0.639115836665931], [0.8828502481979779, 0.6391158366659312], [1.105725376357942, 1.715702493544865], [0.1790861698474644, 2.0916543268395467], [0.8508398595332081, 1.3508797454765054], [0.35083985953320806, 2.2169051492609437], [1.721627496385746, -0.3197620019224832], [1.634658975633605, 0.9308737486442042], [0.6623368502399252, 1.1645181033158225], [1.5764375828786292, 0.9058669211326252], [2.0764375828786292, 1.7718923249170637], [1.3879345735853463, 1.0466588410002677], [1.5953161068071144, 1.2137914956511322], [2.171753689685744, 0.12525082242139743], [2.795243491544478, 0.9070823048894266], [2.288903441487766, 1.480542580117826], [1.3765101981412666, 0.7818314824680297], [2.3727951728508, 0.6957138352171649], [2.2124658586091357, -0.2913497447992379], [0.5535888319904689, 1.3731556304758614], [1.2669481281701738, 0.6801727377709195], [2.263233102879707, 0.5940550905200546], [1.2253425216762128, 0.6323810491128201], [1.4478634556325272, -0.3425468630690035], [0.5876067566535008, 0.1673143061346422], [1.173761225684005, 0.5633200580636222], [0.5134319114423413, 0.442281881831658], [2.171753689685744, 0.6266528441679664], [1.1397433010209737, 0.5098611692036457], [1.099031132097581, 0.4338837391175583], [2.0953161068071147, 0.3477660918666934], [1.0733607934895222, 0.3759518332946816], [1.8064126653193484, -0.3042209044762379], [-0.08414693303224308, -0.9536586097907247], [1.0444271942138594, 0.29475517441090426], [1.0558515696579394, 1.2946899141043067], [1.0276778746063202, 0.2336443546716182], [0.10103866809584239, 0.6095961879662999], [1.0020075359982619, 1.0434799270838573], [0.9272774424118375, 0.0462761299026771], [1.6004004321944825, 1.053393628553379], [1.4400711179528187, 0.0663300485369761], [0.6654136396290311, 0.6987110976497962], [1.0111691737748716, 0.1490422661761744], [1.4492327557294291, 2.4676061601342294], [1.2649405921719123, 0.7435055238752639], [0.9794139867793101, 0.6494377053144871], [1.0111691737748714, -0.14904226617617433], [1.8828502481979776, 0.34103130431358286], [1.0225935492189517, -0.21136846967226214], [1.5225935492189517, 0.6546569341121764], [0.5876067566535008, 0.2999744032085939], [-0.507709350153613, -0.04779168865809913], [1.0444271942138594, -0.2947551744109042], [1.4885756245559199, 0.6011980452522006], [1.8539166489223144, -0.3296757033920037], [0.12086477709248866, 1.47713715050616], [1.2723947406563167, 1.3959404916223823], [0.3168219348701762, 1.101185317211478], [0.7875341413908634, -0.4635850393009673], [0.39325951774880563, 0.3860920504594589], [0.61578045170512, 0.7715096138194739], [0.20475650845552296, 0.15517863830006307], [0.13837400092407126, 0.021269302391099143], [-0.7218826980549546, 0.5311304715947449], [0.1497983763681512, 1.021204042084502], [0.8714258727538979, 0.701442040162018], [0.08760675665350082, -0.5660510005758446], [0.16233685023992495, -1.5632547977570248], [0.27610976594678327, -0.96919970723697], [1.099031132097581, -0.43388373911755834], [-0.12656659124784087, 1.1395112758042434], [0.1640443395321305, -0.7885662700211409], [0.4696971006274353, 0.16357678101416262], [1.173761225684005, -0.5633200580636218], [0.5020075359982612, 0.1774545232994197], [1.2858266520986585, 0.38268662084779287], [0.7875341413908634, 0.2913497447992379], [0.5393428688264907, 0.1262574050296542], [-0.4329792565671893, -1.0449954858392791], [0.3000726152626374, -0.36482274806836024], [0.2387744331185544, -0.38087282166176895], [1.227605259343683, -0.529915087837943], [1.2669481281701733, -0.680172737770919], [1.2783725036142533, 0.31976200192248344], [0.4066914291911472, -0.17031156856727336], [1.3622642349772875, -0.4650667429781777], [1.3229213661507973, 0.2747012557766064], [2.2238902340532163, 1.8352251110214084], [0.9461559663403223, 0.20564026472740826], [0.8879345735853463, 0.1806334372158292], [1.0340179246630319, 1.1157198320494666], [0.5784451188768909, -0.04506074614587652], [0.41052812280429807, -0.2913497447992377], [1.4068130975138315, -0.3774673920501026], [0.9832506803924608, 0.5283995290825226], [1.682162959236571, 0.17031156856727392], [1.9659820753369686, -0.7885662700211403], [1.8056527610953046, 0.08611764725086497], [0.9046838931928856, -0.3477660918666934], [0.7253425216762126, -0.23364435467161848], [1.3488323235349466, 0.5481871277964108], [-0.3989613319041574, -0.2564292158181387], [0.52767787460632, -0.6323810491128203], [0.8333306357016248, 0.3197620019224833], [2.371681074423106, -0.3759518332946814], [2.1491601404667913, -1.350879745476505], [1.2110965585122337, -0.025006827511578844], [0.3508398595332072, 0.4848543416920668], [0.27440227665457817, 0.4450128243438803], [0.6623368502399245, -0.697229393972586], [1.235059407828087, 0.12251987990917482], [2.1885030092932825, -1.7634941822029642], [1.3539166489223144, -0.06906099104919822], [2.2255977233454205, 0.4210125794405589], [2.1491601404667917, 0.3811710620923724], [2.3502016236318477, -0.1551786383000627], [3.221882698054954, 0.3348949321896944], [2.3616259990759274, 0.8447561013933401], [2.3396706857583363, -0.12103817623196422], [3.072722557588163, 0.5591345615389546], [0.938701817855917, -1.420947319133961], [1.2443545789512218, -0.4688042680986576], [1.0507672442705704, -1.6015807563497901], [2.572722557588163, -0.3068908422454839], [1.5953161068071144, -0.5182593119177461], [2.497992464001738, -0.1774545232994193], [1.4029764039006805, -0.2564292158181387], [1.0973236428053759, -1.2085722668534422], [1.9235624171213708, -0.6452522087898205], [1.9349867925654507, 0.354682530903582], [2.137735765022712, -0.9904690096280693], [2.637735765022712, -0.12444360584363057], [1.9659820753369683, 0.616330975519411], [1.4659820753369681, -0.2496944282650277], [1.311496990706717, -0.4271489515644471], [1.811496990706717, 0.4388764522199915], [1.4492327557294291, -1.7157024935448653], [2.320913830152535, -1.225628923055108], [1.3942746236420578, -0.8496770897604264], [2.7612255668814463, -0.20863752716003997], [1.9865680885576589, 0.42374352195278003], [2.2225209339563143, -0.9749279121818235], [1.2875341413908639, -1.329610443085406], [2.7838191161003976, -0.09285243480397565], [1.828246310314257, -0.38760760921487986], [2.2952434915444773, 0.04105690110498822], [1.3064126653193489, 0.19009916728116263], [1.9464111680095313, -0.5071302266914232], [1.612065426414654, -0.21810325722537272], [0.8374079480908663, 0.4142777918874474], [-0.13491417730281352, 0.6479221465590657], [1.3828502481979785, -0.524994099470856], [2.3791352229075113, -0.611111746721721], [1.2949882898752691, -0.8098355724122398], [0.32266616448158913, -0.5761912177406214], [0.20475650845552273, 0.8249685026794502], [2.553844033659678, 0.6603851390570303], [1.4387018178559172, -0.5549219153495226], [1.8094894547084555, -0.034920528981099166], [0.8539166489223149, -0.32967570339200336], [2.6234898018587334, -0.7818314824680297], [2.6349141773028135, 0.21810325722537272], [1.6625920519091335, 0.4517476118969909], [1.1511676764650538, 0.31783827598802716], [1.5507672442705713, -0.7355553525653515], [1.7632331028797072, -0.2719703132643841], [0.7909109774860272, -0.038325958592765774], [1.6511676764650538, -0.5481871277964115], [1.1511676764650536, -1.4142125315808503], [1.5696457681990559, -0.5761912177406214], [2.7330518718298262, -0.6801727377709196], [2.4026763571946237, 0.3425468630690028], [2.4774064507810483, 1.339750660250183], [1.714173347901341, -0.38268662084779326], [3.7726499423255255, 0.05618983137219913], [2.7952434915444773, -0.15517863830006298], [2.4441484303420604, -0.4286645103198682], [0.9575803417844022, -0.3580879605152489], [2.02058601322069, -0.6494377053144871], [2.9888308262251284, -0.1490422661761741], [2.6265665912478404, -0.27348587201980457], [2.4349867925654505, 0.6835781673825858], [3.2612255668814454, 1.2468982254462078], [2.2612255668814454, 1.246898225446208], [2.2141733479013412, 0.48333878293664595], [1.5424196582155973, 1.2241133642996873], [2.01142437544408, -0.3604107358484362], [2.51142437544408, 0.5056146679360024], [3.007709350153613, 0.9138170924425375], [2.921554881123109, 0.9110861499303151], [2.069645768199056, 0.2898341860438167], [1.446155966340322, 1.0716656685118466], [2.4962849747095333, -0.9521430510353035], [2.421554881123109, 0.045060746145876684], [2.921554881123109, -1.0833214444320451], [1.9441484303420606, -1.2946899141043073], [2.2141733479013412, 0.7814233152889942], [1.1770786338492019, 0.5913241480078317], [2.9555728057861406, 0.29475517441090454], [1.960657131173509, 0.7397679987547845], [2.1666693642983748, -0.3197620019224827], [3.0933085708088526, -0.6957138352171647], [2.3602566989790263, -0.015541097446245239], [1.6885030092932825, 0.7252334839167962], [2.9009688679024195, 0.43388373911755734], [1.9743296613919417, 0.809835572412239], [2.1120654264146537, -0.1806334372158299], [1.612065426414654, 0.685391966568609], [3.177333835518411, 1.4422166215250605], [2.6773338355184113, 0.5761912177406214], [2.1999273847373626, 1.2308481518527987], [2.9329792565671893, 1.9110208896237175], [2.097023596099319, 1.122454619602577], [2.997992464001738, 0.6885708804850192], [1.9161082686369655, 0.7848287449006615], [2.0538440336596775, -0.20564026472740773], [1.9161082686369655, 0.19531839607885296], [1.4161082686369655, -0.6707070077055857], [2.183178065129823, -0.23515991342703885], [1.8539166489223151, 0.79696441273524], [1.2556454210487782, 1.3348296718830954], [1.8064126653193489, 1.0561245710656006], [2.2889034414877663, -0.21578048189218588], [1.348832323534947, 2.280237935365288], [1.8345863603709684, 0.16731430613464196], [1.059928882047181, 0.799695355247462], [0.8995995678055171, -0.18736822476894088], [2.584146933032242, 1.819684013575163], [1.949232755729429, 1.6015807563497906], [1.712465858609136, 1.3296104430854063], [1.597023596099319, 0.256429215818139], [1.5838917313630339, 0.6707070077055864], [1.8842195482948796, 0.09451578996496474], [2.106740482251194, 0.4799333533249797], [0.9240561155188087, -0.3822568447683964], [1.3168447780834016, 0.5373719208542967], [1.3350820323021908, 1.5372056083036445], [1.2308092051387587, 0.5426568777150699], [0.8932372542187907, 1.4839566057252764], [0.9995600887570931, -0.029658539477047496], [1.645963590329328, 0.7333372096298221], [1.6642008445481182, 1.7331708970791695], [1.559928017384686, 0.7386221664905949], [0.9788239402756098, 0.20470391775276092], [0.7853158337506452, 1.1858025940892052], [0.6182647960166704, 0.19985434453854972], [0.01823725421878719, 0.9998336874493474], [-0.3375719509199695, 0.9412997280102047], [-0.3193346967011794, 1.9411334154595516], [0.25896000073937375, 2.756961364792093], [-0.0531731763703398, 1.806923007804652], [-0.6000275417978813, 0.799979342910798], [-0.5817902875790917, 1.7998130303601454], [-0.3304523051126862, 2.7677124148881638], [-0.7703307120606795, 0.6376445671815032], [-0.01742838799651425, 1.2957768414385697], [-0.981322834538304, 1.029492226926395], [-0.8567627457812106, 0.5157107691734204], [-0.4639740832166177, 1.4353395347961135], [0.4110259167833821, 1.9194624530720406], [-0.9240561155188085, 0.38225684476839666], [0.07550397323828473, 0.3525983052913488], [0.09374122745707447, 1.3524319927406963], [0.6418992717523841, 0.5160572211972563], [-0.9707075985924825, 0.24026393411166813], [0.028852490164610778, 0.21060539463462036], [0.9038524901646109, 0.6947283129105475], [-0.02020362535419773, 1.076985157678944], [-0.9839754524922046, 0.17830397890333544], [-0.17480740008264917, -0.40927330363890774], [-0.7748349418805305, 0.39070603927188946], [0.10016505811946963, 0.8748289575478163], [-0.9956750779596208, 0.09290392419108484], [-0.9774378237408317, 1.0927376116404321], [-0.12067507795962085, 0.5770268424670117], [0.875, 0.4841229182759271], [-0.9984008213886799, -0.056531405876814156], [-0.351997319816445, 0.7064643432300559], [0.40090500424772, 1.3645966174871222], [-0.9801635671698907, 0.943302281572533], [-0.22726124310572549, 1.6014345558296], [0.24683375264655427, 0.7209608566184047], [-0.9638944465417894, -0.2662846145121748], [-1.2411735752841357, 0.6945047925713366], [-0.43200552287458027, 0.10692751002909329], [0.4429944771254193, 0.5910504283050206], [-1.7342251586024684, 0.3713599526693283], [-0.7346650698453754, 0.34170141319228087], [0.1403349301546244, 0.825824331468208], [-0.9373817497946546, -0.3483036823691525], [-0.1844794257304896, 0.30982859188791406], [-0.16624217151169995, 1.3096622793372612], [0.5004485916541963, 0.5643278401025814], [-0.5782946974405541, -0.8158279493325407], [0.23087335496900152, -1.403405231874784], [0.24911060918779082, -0.4035715444254367], [-0.45024277709337013, -0.8929061774202539], [0.5493173116637233, -0.9225647168973016], [0.21174536074375372, 0.018735011112903308], [-0.3121331771097138, -0.9500383569874409], [0.6342772534794158, -1.2730047590668185], [1.2125719509199695, -0.45717680973427743], [-0.10427282716343234, -0.9945487305885747], [0.5421306744088026, -0.23155298148170442], [0.5603679286275922, 0.7682807059676429], [-0.10632283453830382, 1.5136151452023225], [-0.44184477808340183, -0.05324900257836951], [-0.4236075238646122, 0.9465846848709777], [-0.5216532076824998, -0.08581685706494896], [0.23124911638166523, 0.5723154171921175], [-0.7709635903293282, -0.24921429135389445], [-0.12456008875709346, 0.5137814577529756], [0.1309839194194884, -0.9913844929458544], [-0.5815725446558959, -0.2897697229300104], [0.2934274553441041, 0.19435319534591683], [-0.04414449557586497, 1.135652923356122], [0.47409499575227976, -0.8804736992111949], [1.4529189360278896, -0.6757697814584337], [2.205821260092055, -0.017637507201367142], [1.493264796016671, 0.6839772628144771], [0.49233224997106917, 0.11935998823815233], [1.2452345740352344, 0.7774922624952193], [-0.4966126028402025, -0.6402097650995265], [-0.4783753486214133, 0.35962392234982055], [0.5481580442953097, -0.8363747715434399], [1.547718133052403, -0.8660333110204874], [0.6236620175335943, -0.48377646625209114], [0.6666907631658959, -0.7453344392346801], [0.6849280173846855, 0.25449924821466746], [1.4378303414488507, 0.9126315224717341], [0.770330712060679, -0.6376445671815035], [1.4167342136329142, 0.12535118192536632], [0.7885679662794688, 0.3621891202678441], [0.9556190040134438, 1.3481373698184997], [0.8091680524095548, -0.5875772825422435], [1.2019567149741481, 0.33205148308044996], [1.2201939691929375, 1.3318851705297972], [0.26101000811424546, 0.24879748900119653], [0.8567627457812106, -0.5157107691734202], [1.3822278387860243, 0.3351043905607271], [2.1913958911955795, -0.25247289198151635], [1.85382394027561, 0.6888268360286884], [1.8720611944943997, 1.6886605234780356], [1.7990561155188085, 0.10186607350753074], [0.8150806630266036, 0.2801700524108659], [0.6840967436071159, 1.2715545453567203], [0.5864841645988392, 0.5590428832418086], [1.4614841645988392, 1.0431658015177356], [2.398865914393494, 1.3914694838868884], [1.6057937182941737, 0.7823418357094307], [-0.05991933697339591, -0.2039528658650609], [-0.1909032563928843, 0.7874316270807937], [-0.7909307981907654, 1.5874109699915913], [0.08406920180923438, 2.0715338882675183], [0.9464104305891294, -0.32296640207937743], [1.5247051280296833, 0.4928615472531632], [0.9246775862318021, 1.2928408901639612], [0.5871056353118324, 2.234140618174166], [1.1977484130555351, 0.6449329824486408], [2.1684560116480176, 0.4046690483369727], [2.186693265866807, 1.4045027357863202], [0.5290300500700621, 0.5857654714442478], [1.4040300500700618, 1.0698883897201752], [1.2105219435450976, 2.050987066056619], [0.2265464910528926, 2.2292910449599543], [0.39825238629381987, 0.5134083694640623], [1.3966532076824993, 0.5699397753408764], [1.2923803805190675, -0.42460895524769837], [0.34638288879124834, 0.47701294083142043], [1.259823768364236, 0.8839843888777517], [1.1555509412008034, -0.11056434171082297], [0.23385396651374568, 0.3786483679364666], [0.2520912207325352, 1.378482055385814], [-0.06004195637717835, 0.42844369839837315], [-0.10516356716989073, 1.4274251998484604], [0.9707075985924823, -0.24026393411166802], [1.8457075985924822, 0.2438589841642591], [0.846147509835389, 0.27351752364130716], [0.8643847640541789, 1.2733512110906542], [1.7236099226566475, 0.41786834014539886], [1.974947905123053, 1.385767724673417], [1.9931851593418424, 2.385601412122764], [1.8889123321784103, 1.39105268153419], [1.2220455810588877, 0.7276354504163504], [0.22922937669641374, 0.6079860277044553], [1.1042293766964135, 1.092108945980382], [0.25815113451709826, 0.46135083590417614], [0.2763883887358879, 1.4611845233535234], [-0.0008907400064583904, 2.4219739304370345], [0.9956750779596211, -0.09290392419108473], [1.8706750779596213, 0.3912189940848422], [1.27064753616174, 1.1911983369956403], [1.1035964984277649, 0.20525008744498457], [1.6420785795318555, 0.6700918249157853], [1.6603158337506452, 1.6699255123651322], [2.6598759225077386, 1.6402669728880848], [1.8172933697375984, 1.1016997609568786], [1.0139123321784105, 0.9069297632582625], [0.17132977940826977, 0.3683625513270564], [-0.752726336110539, 0.7506193960954533], [1.8745600887570932, 0.45446437879887946], [2.4227181330524035, -0.3819103927445604], [1.424317311663723, -0.43844179862137456], [1.0867453607437534, 0.5028579293888303], [1.752462412821258, 0.628473734780019], [0.8284062973024497, 1.0107305795484156], [1.812381749794654, 0.8324266006450805], [1.0177973429758826, 0.9701751479722998], [0.567554565882513, 0.07726897055204576], [1.565955387271193, 0.1338003764288601], [0.8060519822321285, 0.9514401368593963], [0.701779155068696, -0.043108593729178135], [1.648189585657826, -0.36607499580855574], [0.661988137837124, 0.911641188533157], [1.536988137837124, 1.395764106809084], [2.0851461821324335, 0.5593893352656442], [1.8916380756074689, 1.540488011602089], [0.99840082138868, 0.0565314058768141], [1.87340082138868, 0.5406543241527415], [0.9095063748468906, 0.27436970964056656], [1.1608443573132963, 1.242269094168585], [0.3677721612139757, 0.633141445991127], [1.5238659143934938, 0.9073465656109612], [2.3806286601747044, 0.391635796437541], [1.587556464075384, -0.21749185173991714], [1.0166380756074695, 1.0563650933261615], [1.210146182132434, 0.07526641698971731], [0.4975897180570502, 0.7768811870055613], [-0.10243782374083121, 1.5768605299163594], [0.9532792105959679, 1.0555129073269014], [0.6411460334862541, 0.10547455033946071], [0.6593832877050438, 1.1053082377888082], [1.731726264339775, 0.8628361920098276], [2.124514926904368, 1.7824649576325209], [2.5986099226566477, 0.901991258421326], [1.3716126028402025, 1.1243326833754541], [2.2466126028402025, 1.608455601651381], [1.2537963984777285, 1.4888061789394862], [0.9970611944943996, 1.2045376052021082], [0.39703365269651825, 2.004516948112906], [0.22998261496254324, 1.0185686985622504], [0.3787963984777284, 1.004683260663559], [-0.31463207137240756, 0.28415778769171557], [0.9373817497946546, 0.3483036823691527], [1.1887197322610599, 1.3162030668971711], [1.0216686945270852, 0.3302548173465155], [1.6680721960993203, 1.0932505664533854], [1.6863094503181097, 2.0930842539027323], [0.9134408795729871, 0.4069714480463311], [1.7702036253541975, -0.10873932112708906], [1.0576471612788136, 0.5928754488887549], [0.4576196194809322, 1.3928547917995528], [0.47585687369972196, 2.3926884792489], [1.788440879572987, 0.8910943663222579], [0.7956246752105132, 0.7714449436103628], [0.6646407557910248, 1.762829436556217], [1.627902324064165, 1.1422551925329938], [2.6274624128212585, 1.1125966530559461], [1.6900806630266039, 0.764292970686793], [0.7261862164848144, 0.49800835617461875], [1.3252427770933701, 1.3770290956961806], [1.2209699499299378, 0.3824803651076063], [0.3642072041487272, 0.8981911342810266], [1.3106176347378566, 0.5752247322016489], [0.1220976759358352, -0.17400935598113992], [0.8924283879965142, -0.8116539231626435], [0.910665642215304, 0.18817976428670377], [1.4750275417978813, -0.3158564246348705], [1.5684284698501358, 1.2046483912477703], [1.5866657240689257, 2.2044820786971178], [0.7930721960993203, 0.6091276481774581], [0.688799368935888, -0.3854210824111165], [-0.29517608355631664, -0.207117103507781], [0.5798239164436834, 0.277005814768146], [0.7529023240641651, 0.658132274257067], [1.3993058256364, 1.4211280233639365], [0.8511477813410899, 2.257502794907376], [-0.1127466652006992, 1.9912181803952014], [0.7622533347993012, 2.475341098671129], [0.6934284698501363, 0.7205254729718433], [0.5891556426867038, -0.27402325761673113], [0.6073928969054931, 0.7258104298326165], [0.269820945985524, 1.6671101578428211], [0.31494255677823557, 0.6681286563927336], [0.6464035015722349, 0.76299574910687], [-0.3324204387033749, 0.558291831354109], [0.1930446543014388, 1.4091069910882559], [1.0680446543014388, 1.8932299093641831], [0.5782946974405535, 0.8158279493325407], [1.4532946974405538, 1.2999508676084677], [0.4693192449483491, 1.4782548465118033], [1.1157227465205843, 2.2412505956186726], [0.12290654215810948, 2.121601172906777], [0.6234163082332651, -0.18315355211754678], [0.28584435731329616, 0.7581461758926582], [0.30408161153208535, 1.7579798633420054], [0.19980878436865301, 0.7634311327534312], [0.5254650930048137, 0.8508151597341471], [1.4004650930048135, 1.3349380780100744], [0.7337743298389173, 2.080272517244754], [-0.259041874523557, 1.960623094532859], [0.6159581254764436, 2.4447460128087863], [0.45024277709337035, 0.8929061774202537], [0.345969949929938, -0.10164255316832073], [2.134410829502925, 0.789451813153937], [1.1705163829611358, 0.5231671986417633], [0.39278866256459266, 0.9196287656226934], [-0.3006398072855436, 0.19910329265085], [-0.5779189360278898, 1.1598926997343613], [-0.9154908869478593, 2.101192427744566], [0.31213317710971356, 0.9500383569874409], [1.311693265866807, 0.920379817510393], [1.329930520085596, 1.9202135049597404], [0.3511065798099864, 1.7155095872069794], [0.9975100813822217, 2.478505336313849], [0.25133798246640526, 0.9678993845280186], [0.6441266450309979, 1.8875281501507115], [1.0615070255500647, 0.9787962766270857], [0.06583194759044386, 1.0717002008181704], [1.0605060348759607, 0.3803221019857749], [0.07653058238375599, 0.5586260808891106], [0.722934083955991, 1.3216218299959805], [0.1670510377339749, 0.9859482495506557], [1.0420510377339753, 1.470071167826583], [0.3294945736585909, 2.171685937842427], [0.46047849307807887, 1.1803014448965723], [1.0110534585812638, 1.1194831101612428], [1.0292907128000528, 2.1193167976105904], [0.7171575356903395, 1.169278440623149], [1.6635679662794691, 0.8463120385437717], [0.8113094503181095, 1.6089613356268053], [1.6204775027276652, 1.021384053084562], [2.477240248508876, 0.5056732839111419], [0.7711395782829547, 1.657965961706414], [0.0008088662222751619, 2.2956105288879174], [1.0003689549793684, 2.2659519894108695], [0.0364745084375786, 1.9996673748986948], [0.7116657240689256, 1.7203591604211907], [0.7567873348616374, 0.7213776589711032], [0.6525145076982053, -0.27317107161747123], [0.33037043132850297, 1.949872044436788], [0.6679423822484725, 1.0085723164265832], [1.5429423822484727, 1.4926952347025102], [0.2695752366851948, 1.9677330719773656], [1.2069569864798497, 2.316036754346518], [1.6810519822321293, 1.4355630551353236], [-0.5072278387860245, 0.14901852771520013], [0.18620063106411155, 0.8695440006870436], [0.9792728271634321, 1.4786716488645018], [-0.31371973226105965, -0.8320801486212437], [-0.1309839194194884, 0.9913844929458544], [0.3431110763327916, 0.11091079373465972], [0.361348330551581, 1.1107444811840068], [1.1142506546157462, 1.7688767554410736], [-0.19350810652496464, 0.9810986763364441], [0.6632546392562458, 0.46538790716302403], [0.32568268833627634, 1.406687635173229], [0.34391994255506597, 2.4065213226225763], [1.3434800313121593, 2.3768627831455285], [0.6814918934750357, 1.4652215946123714], [-0.31418318448458527, 1.558125518803456], [-0.0020500073748719316, 2.5081638757908964], [-0.27727912874234617, 0.9607894070835113], [0.6104250348205066, 0.5003751175646306], [-0.3269567149741479, 0.152071435195478], [0.4822113374354067, -0.4355058473467658], [0.5977208712576538, 1.4449123253594385], [1.5111617508306407, 1.8518837734057694], [1.4068889236672084, 0.8573350428171949], [1.8571317007605783, 1.7502412202374487], [0.5998097988746851, 1.2896034103793577], [1.4089778512842401, 0.7020261278371143], [1.1316987225418946, 1.6628155349206262], [0.5374280490800305, 1.4254226462861315], [0.22529487197031683, 0.47538428929869103], [-0.4413958911955793, 1.2207187285333712], [0.41533037314419563, 1.5994320022672717], [0.10319719603448196, 0.6493936452798308], [0.7966256658846183, 1.3699191182516741], [0.24072274652058423, 1.7571276773427456], [1.2402828352776774, 1.727469137865698], [0.9889448528112721, 0.7595697533376793], [0.1878931420848443, 1.792114887744352], [1.0628931420848446, 2.276237806020279], [0.8693850355598798, 3.257336482356723], [0.41914225846650977, 2.3644303049364694], [0.05521671164462316, 1.8609284936328983], [0.5806818046494366, 2.7117436533670456], [1.0547768004017168, 1.8312699541558506], [0.7219074748105191, 1.1155940543982181], [1.4748097988746847, 1.7737263286552847], [0.4509960153594992, 1.3034888482780485], [1.4148904619012885, 1.5697734627902236], [-0.5310800574449341, 1.9223984043466489], [-1.5294808788336138, 1.8658669984698348], [-0.6544808788336138, 2.349989916745762], [0.3450792099234792, 2.320331377268714], [-0.9375994927178508, 1.741279070921003], [-0.12843144030829556, 1.1537017883787595], [-0.29548247804227024, 0.16775353882810362], [-0.4173803805190677, 0.9087318735236254], [0.33552194354509735, 1.566864147780692], [1.1781044963152378, 2.1054313597118988], [0.18528829195276397, 1.9857819370000034], [1.060288291952765, 2.4699048552759306], [-0.39914312630027826, 1.9085655609729728], [0.353759197763887, 2.56669783523004], [1.020449960929783, 1.8213633959953597], [-0.7549523314390372, 1.85003160153383], [0.1914580991500925, 1.5270651994544526], [1.1871331771097138, 1.434161275263368], [1.2053704313285036, 2.433994962712715], [-0.5481580442953098, 0.8363747715434398], [0.20474427976885523, 1.4945070458005065], [0.22298153398764486, 2.494340733249854], [1.0797442797688555, 1.9786299640764338], [0.3956475361617402, 0.7070754187197132], [0.05807558524177059, 1.648375146729918], [-0.10897545249220464, 0.6624268971792624], [0.3134133377751058, 1.2069507909571289], [1.188413337775106, 1.6910737092330557], [0.49498486792496976, 0.9705482362612123], [1.4413952985140994, 0.647581834181835], [0.2749724582021187, 1.284102261186725], [0.5522515869444652, 0.32331285410321364], [1.4986620175335947, 0.00034645202383609686], [0.1528747822662837, 1.4581116171678645], [-0.7845069675283711, 1.1098079347987118], [0.20830923683410352, 1.2294573575106074], [-0.28789436468816776, 1.7500176998982389], [-1.0004508287635518, 2.451632469914083], [-0.3337600655976558, 1.706298030679403], [-0.34868955933147583, 1.7678787274388161], [-0.5157405970654505, 0.7819304778881605], [0.3268419557046902, 1.3204976898193672], [-1.275218757429228, 1.0792875573883018], [-0.3045111588367455, 0.8390236232766338], [0.20914051061167416, 0.21240206036855439], [1.0841405106116744, 0.6965249786444813], [0.9531565911921858, 1.687909471590336], [-0.030818861300018874, 1.8662134504936714], [-0.6666907631658958, 0.74533443923468], [0.3328693255911972, 0.7156758997576323], [1.207869325591197, 1.1997988180335593], [0.5144408557410605, 0.47927334506171587], [-0.0855866860568204, 1.2792526879725137], [-0.14122567016108212, 1.5961495989688272], [-0.4533588472707961, 0.6461112419813864], [0.42164115272920377, 1.1302341602573138], [-0.7125564640753845, 0.701614770015844], [0.28700362468170926, 0.6719562305387963], [0.8351616689770192, -0.16441854100464354], [1.8308367469366398, -0.2573224651957281], [0.22482528571927063, 1.049918452384997], [0.24306253993805993, 2.049752139834344], [-0.6943192098565945, 1.7014484574651914], [0.20088441549760305, 1.1085862180621753], [1.0758844154976033, 1.5927091363381023], [0.08306821113512897, 1.4730597136262071], [0.1624435359246158, 1.1857376882917712], [0.855872005774752, 1.9062631612636145], [1.1331511345170988, 0.9454737541801033], [1.8860534585812636, 1.6036060284371698], [0.04034585998878093, 1.3597470442729107], [0.9153458599887808, 1.8438699625488377], [1.6087743298389174, 2.564395435520681], [-0.019127994225248024, 1.4221402429876877], [1.8542728271634323, 1.9627945671404285], [1.75, 0.9682458365518543], [-0.26231368698201374, 1.5945209474360977], [-0.3665865141454461, 0.5999722168475234], [0.5468543654275411, 1.0069436648938543], [-1.3125840058732656, 1.5015941129266417], [-0.4375840058732652, 1.9857170312025687], [0.5580910720863557, 1.8928131070114844], [0.17607971852845028, 0.31467816510212565], [-0.7878147280133392, 0.048393550589950984], [0.08718527198666093, 0.532516468865878], [1.1756398072855436, 0.2850196256250778], [0.9821317007605788, 1.2661183019615219], [-0.7520934578418901, 1.6374782546308504], [0.1943169727472398, 1.314511852551473], [0.9646476848079187, 0.6768672853699697], [0.08964768480791918, 0.19274436709404283], [-0.3277326957111486, 1.1014762406176681], [0.5472673042888512, 1.585599158893595], [-0.21035924420897567, 1.2787065182802901], [0.5599714678517035, 0.6410619510987867], [1.434971467851704, 1.125184869374714], [-0.21263610075021266, 2.4032389193241315], [0.6623638992497873, 2.887361837600059], [-0.6054247633148051, 1.4836101537014388], [0.3409856672743248, 1.1606437516220613], [1.2159856672743246, 1.6447666698979884], [-0.3826677500289306, -0.3647629300377745], [-1.3533753486214128, -0.12449899592610603], [-0.9605866860568204, 0.7951297696965868], [-0.010615235945821366, 0.789228292814728], [0.5148498570589922, 1.6400434525488752], [1.3898498570589928, 2.1241663708248018], [1.8639448528112723, 1.2436926716136065], [-0.04905611551880851, 0.8663797630443237], [0.4764089774860051, 1.7171949227784709], [0.9505039732382847, 0.8367212235672757], [-0.6727181330524035, 1.3501562292964153], [0.1364499193571519, 0.762578946754172], [1.1360100081142452, 0.7329204072771237], [-0.9058188613000191, 1.3820905322177441], [0.07815659119218599, 1.203786553314409], [-0.8592251586024691, 0.8554828709452564], [-1.7666386682889494, -0.15631036716280944], [-0.767078579531856, -0.18596890663985743], [-0.7488413253130668, 0.8138647808094897], [-0.057266719019495316, 0.6472353821579995], [0.4168282767327848, -0.23323831705319542], [0.942293369737598, 0.6175768426809514], [-0.952470344373693, 1.2400976215610156], [0.047089744383400634, 1.2104390820839677], [-0.8769663711354081, 1.5926959268523646], [0.3783873971597973, -0.1560868468235997], [1.3572113374354071, 0.04861707092916123], [-0.16153954618292699, -0.347313348430575], [0.312555449569353, -1.2277870476417698], [0.8380205425741662, -0.37697188790762304], [-0.12721270671099405, -0.3374067902700848], [0.5394780564549019, -1.0827412295047651], [0.5577153106736916, -0.0829075420554177], [0.41648964051260884, 1.51324205691341], [-0.5299207900765204, 1.8362084589927867], [-1.4013558330112725, 1.0870358524269605], [-0.45494540242214265, 0.7640694503475833], [-0.6484535089471075, 1.7451681266840273], [-1.5840029942900857, 0.9782833218141334], [-0.7090029942900857, 1.46240624009006], [0.2044378852829012, 1.8693776881363913], [-0.3172846893263088, -0.5670304603313447], [-0.5107927958512732, 0.41406821600509947], [0.4356176347378564, 0.09110181392572195], [-0.24277275389545583, 0.7510361984481514], [-0.34704558105888816, -0.24351253214042295], [0.5663952985140994, 0.16345891590590805], [-0.1865070255500656, -0.49467335835115844], [-0.16826977133127619, 0.5051603290981888], [-0.12340082138868036, 0.42759151239911325], [-0.31690892791364433, 1.4086901887355572], [0.5965319516593428, 1.815661636781888], [1.4715319516593426, 2.2997845550578155], [-0.2454984973245149, 0.6016008683802526], [0.6295015026754851, 1.0857237866561795], [-0.36617357528413597, 1.1786277108472643], [-1.1919089279136448, 0.9245672704596299], [-0.643750883618335, 0.08819249891619024], [0.35580920513875824, 0.05853395943914243], [-0.9928162043624742, -0.11964942271189521], [-0.1360534585812636, -0.6353601918853156], [0.6168488654829014, 0.022772082371751157], [0.6350861197016912, 1.0226057698210984], [-0.07747034437369305, 1.7242205398369426], [-0.1178162043624742, 0.3644734955640318], [-0.8303726684378581, 1.0660882655798756], [0.10700908135679632, 1.4143919479490288], [0.5811040771090757, 0.5339182487378334], [-0.23991388029830918, 0.5384828515451713], [-0.09570759859248246, 0.7243868523875955], [-0.4145215069219206, 0.6961785266206454], [0.4422412388592899, 0.18046775744722518], [0.10466928793932073, 1.12176748545743], [-1.7053726684378585, 0.5819653473039489], [-1.2312776726855779, -0.298508351907246], [-0.8384890101209856, 0.6211204137154474], [-0.22248549230179515, -0.7572939898933988], [0.42391800927043977, 0.005701759213471047], [-0.5600574432217644, 0.18400573811680662], [-0.9788239402756098, -0.20470391775276103], [-1.3163958911955795, 0.7365958102574436], [-1.2981586369767895, 1.736429497706791], [-0.4889905845672349, 1.148852215164548], [-0.10713170076057876, -0.7819953836855948], [-0.38441082950292493, 0.1787940233979164], [0.4247572229066303, -0.4087832591443266], [0.035665642215303994, -0.29594315398922244], [0.20271667994927867, 0.6900050955614333], [1.0777166799492788, 1.1741280138373607], [1.8306190040134438, 1.8322602880944272], [-0.08889444654178935, 0.2178383037637527], [0.04208947287769893, -0.773546189182102], [-0.506068571417611, 0.06282858236133801], [-0.21099212247762433, 0.391847659744892], [0.6640078775223756, 0.8759705780208189], [-0.32880832684009853, 0.756321155308924], [0.4354113790946108, 1.154843408851762], [0.6867493615610161, 2.1227427933797807], [-0.5136516694484192, 0.6266215629080789], [-0.6179244966118516, -0.36792716768049477], [-0.895203625354198, 0.592862239403016], [-0.7968434088078147, 0.7196636350384811], [-0.5455054263414092, 1.6875630195664992], [-0.5272681721226196, 2.6873967070158473], [-0.06238174979465483, 0.13581923590677417], [0.6905205742695105, 0.793951510163841], [0.8575716120034853, 1.779899759714497], [0.6501747142807294, -0.5657955341090698], [-0.544593087230062, 0.5713250832535407], [-0.5263558330112723, 1.5711587707028878], [0.28281221939828294, 0.9835814881606446], [-0.9191444955758652, 0.6515300050801944], [-0.03144033201301211, 0.1911157155613138], [-0.3087194607553583, 1.151905122644825], [-0.7930721960993203, -0.6091276481774582], [0.2064878926577729, -0.638786187654506], [0.4578258751241784, 0.3291131968735125], [1.005983919419488, -0.5072615746699273], [1.0242211736382778, 0.4925721127794201], [0.08192780390068, -0.12500472990153094], [0.33326578636708515, 0.8428946546264874], [-0.6306286601747042, 0.5766100401143127], [-0.7272402485088751, 0.4625725526407124], [-0.26760710309450664, 0.24168751155668894], [-0.016269120628101152, 1.2095868960847076], [-0.5417342136329151, 0.35877173635056014], [-1.3984969594141257, 0.8744825055239804], [-0.5234969594141252, 1.3586054237999077], [-1.4873914059559148, 1.0923208092877332], [-0.5034159534637102, 0.9140168303843983], [-1.505628660174704, 0.09248712183838592], [-0.5682469103800498, 0.4407908042075387], [0.3067530896199502, 0.9249137224834658], [-0.6934284698501365, -0.7205254729718433], [0.30613161890695695, -0.7501840124488911], [0.2955163829611356, 0.03904428036583647], [-0.6751912156313471, 0.2793082144775042], [-0.282402553066754, 1.1989369801001972], [-0.5596816818091002, 2.1597263871837087], [0.3351461821324333, -0.4088565012862095], [1.3308212600920541, -0.501760425477294], [-0.37741028194295057, 0.29275826872963473], [0.2967053025594464, -0.3317050310566136], [1.1058733549690014, -0.919282313598857], [1.684168052409555, -0.10345436426631605], [0.023852218658909963, -1.7733798766314486], [-0.1696558878660548, -0.7922812002950046], [0.7053441121339452, -0.30815828201907747], [0.5628668228902862, -0.4659154387115139], [-0.436693265866807, -0.43625689923446564], [-0.4184560116480176, 0.5635767882148818], [-0.29389592289092414, 0.04979533046190654], [-0.3390175336836363, 1.048776831911994], [0.4138847903805293, 1.7069091061690602], [-0.6497051280296833, -0.00873862897723604], [0.15946292437987197, -0.5963159115194795], [0.8528913942300084, 0.124209561452364], [-0.16705103773397445, -0.9859482495506557], [0.7079489622660251, -0.5018253312747287], [-0.28486724209644865, -0.6214747539866241], [0.10792142046814401, 0.29815401163606936], [0.675531515036166, -0.44738103761944914], [1.2236895593314758, -1.283755809162889], [1.2419268135502652, -0.2839221217135416], [0.47935246383826036, -0.2229525004437858], [1.2322547879024257, 0.43517977381328093], [0.632227246104544, 1.2351591167240785], [-0.3605591442589393, -0.004849573214211672], [0.4962036015222713, -0.5205603423876318], [-0.10382394027560982, 0.2794190005231663], [0.8952872615936609, -1.0242072700656222], [1.2074204387033745, -0.07416891307818113], [0.22859649842776475, -0.27887283083094233], [0.8941279942252477, -0.9380173247117605], [0.47674761370618013, -0.029285451188134998], [1.4231580442953098, -0.3522518532675123], [0.2885158354011603, -0.07491996496588124], [1.0815880315004809, 0.5342076832115769], [1.0998252857192703, 1.534041370660924], [-0.08603557294464315, 0.005284956860772828], [-1.0817106509042635, 0.09818888105185758], [-0.6314678738108932, 0.9910950584721111], [-0.9610355729446431, -0.4788379614151543], [0.03736524844403721, -0.42230655553833996], [0.9123652484440374, 0.06181636273758695], [-1.0999479051230532, -0.9016448063974897], [-1.293456011648018, 0.07945386993895454], [0.7773874209917232, -0.22838874383898444], [-0.1466686945270852, 0.15386810092941236], [0.6062336295370796, 0.812000375186479], [0.14922117363827736, 0.008449194503492963], [-0.18835077728169192, 0.9497489225136978], [0.6047214188176283, 1.5588765706911558], [0.06483095691633944, 0.4732260261768595], [0.2772791287423465, -0.9607894070835115], [1.1522791287423462, -0.47666648880758417], [1.6777442217471596, 0.37414867092656307], [0.7403624719525055, 0.025844988557410298], [-0.3227484130555347, -0.16081006417271335], [0.8035895694108707, 1.291212238631232], [-0.7155370756201278, -1.0804388297954066], [-0.022108605769991296, -0.3599133568235632], [0.7307937182941737, 0.29821891743350354], [0.4173803805190668, -0.9087318735236257], [0.8101690430836596, 0.010896892099067357], [0.09761257900827591, 0.7125116621149117], [0.7440160805805107, 1.4755074112217814], [-0.18264716127881453, -0.10875253061282796], [-0.1644099070600249, 0.8910811568365193], [0.6923528387211858, 0.37537038766309916], [1.4205054263414096, -1.2034401012905722], [0.5637426805601985, -0.6877293321171526], [-0.14881378351518526, 0.013885437898691858], [1.2269973198164448, -0.222341424954128], [1.8936880829823406, -0.9676758641888081], [1.1811316189069567, -0.2660610941729644], [1.120498497324515, -0.11747795010432482], [0.6702557202311449, -1.0103841275245788], [0.6884929744499344, -0.010550440075231704], [0.7862281728619935, 0.06956465777624604], [-0.20658803150048066, -0.050084764935649256], [0.6684119684995189, 0.43403815334027773], [0.804465427080783, 1.069398345225593], [0.7001925999173509, 0.07484961463701892], [0.1968158670099338, 0.08031570787231652], [-0.6599468787712768, 0.5960264770457369], [0.2150531212287229, 1.080149395321664], [-0.23846146832310455, -0.17885892919535074], [0.6989202814715502, 0.16944475317380184], [-0.07141043058912944, 0.8070893203553051], [0.799496026761715, 0.13152461298457843], [0.8177332809805049, 1.131358300433926], [0.2177057391826236, 1.9313376433447238], [1.5416907631658958, -0.2612115209587529], [1.3481826566409312, 0.7198871553776912], [2.294593087230061, 0.3969207532983138], [1.8772127067109938, 1.305652626821939], [1.4195930872300608, -0.08720216497761346], [0.7070366231546769, 0.6144126050382309], [0.18157153014986305, -0.23640255469591626], [1.0594794257304885, 0.174294326388013], [0.08877182713800646, 0.4145582604996815], [1.0022127067109938, 0.8215297085460124], [-0.22101340039695683, -0.2849201497157994], [0.5318889236672083, 0.37321212454126756], [0.5501261778859976, 1.373045811990615], [1.6453307120606793, -0.15352164890557596], [2.2917342136329144, 0.6094741002012934], [1.3077587611407095, 0.7877780791046286], [1.0399059487458742, 1.3300885047958628], [0.11584983322706577, 1.712345349564259], [0.4327587611407099, 0.3036551608287016], [1.438742680560198, -0.2036064138412257], [0.17030317026279795, 0.1623347757292943], [0.8167066718350328, 0.9253305248361643], [-0.16726878065717155, 1.1036345037395], [1.8087281411666485, -0.6172358220192908], [1.4711561902466788, 0.3240639059909136], [1.1938770615043328, 1.284853313074425], [1.6022402485088758, 0.02155036563521462], [0.6964213872088563, 1.4036408978529584], [1.334633145414369, 0.26323787719190395], [2.209633145414369, 0.747360795467831], [1.225657692922164, 0.925664774371166], [1.2594108295029256, 0.3053288948780105], [0.26373575154330464, 0.39823281906909513], [1.1387357515433048, 0.8823557373450219], [0.8274053066283446, 0.41225640490710386], [1.277648083721715, 1.3051625823273576], [0.46848003131215965, 1.8927398648696012], [0.6781841329900666, 0.4038072104036111], [1.5531841329900669, 0.8879301286795381], [1.2759050042477207, 1.8487195357630493], [0.47159610148958575, 0.35372244546796155], [1.3465961014895858, 0.8378453637438884], [1.0344629243798722, -0.11219299324355247], [0.09661158833417127, 0.11403748747360085], [0.7900400581843072, 0.8345629604454441], [0.4779068810745938, -0.11547539654199662], [1.6190160805805118, 1.9596303294977087], [0.6261998762180373, 1.839980906785813], [-0.2488001237819626, 1.3558579885098863], [0.2919295517555157, 2.027023514666384], [-0.5830704482444844, 1.5429005963904574], [1.0833092368341037, 1.713580275786534], [0.3304069127699386, 1.0554480015294674], [1.267788662564593, 1.4037516838986202], [-0.48168310910638246, -0.7017904618589397], [1.5829489622660247, -0.017702412998801487], [0.6365385316768951, 0.3052639890805761], [1.3490949957522795, -0.3963507809352678], [1.7317627457812104, -0.03158785089749294], [0.7389465414187363, -0.15123727360938855], [1.880983919419488, -0.023138656394000146], [2.0272791287423457, 0.007456429468342685], [2.087571950919969, 0.02694610854164925], [2.1918447780834005, 1.0214948391302239], [2.1673803805190675, 0.059513963028228756], [2.29815804429531, 0.13187106500841472], [1.3141825918031051, 0.3101750439117493], [2.4625564640753845, 0.2666310665360103], [1.7691279942252482, -0.4538944064358328], [2.2955054263414088, -0.7193171830146451], [1.8214104305891292, 0.1611565161965498], [2.5203307120606797, 0.33060126937035117], [1.57392028147155, 0.6535676714497287], [2.7456750779596204, 0.8753419123607691], [2.8499479051230527, 1.8698906429493436], [2.7484008213886795, 1.0247772424286683], [2.2743058256364, 1.9052509416398635], [2.72882394027561, 1.172949754304615], [2.535315833750645, 2.154048430641059], [1.5396407557910246, 2.246952354832144], [2.663440879572988, 1.3752172845981852], [1.6794654270807827, 1.5535212635015196], [2.0170373780007522, 0.6122215354913153], [1.142037378000752, 0.1280986172153884], [2.3513077024632736, 0.4251789276107444], [1.3673322499710685, 0.6034829065140791], [2.5430721960993203, 1.577373484729312], [2.2754650930048137, 1.8190609962860016], [0.4540546624156837, 1.6579044800894522], [2.552744221747159, 0.85827158920249], [2.20024277709337, 1.861152013972108], [1.7261477813410906, 2.7416257131833035], [2.062133177109714, 1.9182841935392947], [1.9170510377339758, 1.9541940861025098], [0.933075585241771, 2.1324980650058447], [0.8614566228009576, 1.8431451444285334], [-0.013543377199042528, 1.3590222261526064], [1.7682372542187894, 1.9680795240012015], [0.7686771654616961, 1.9977380634782493], [0.8729499926251276, 2.992286794066824], [0.9726125790082765, 1.196634580390839], [1.9565880315004809, 1.0183306014875038], [1.1635158354011603, 0.40920295331004575], [0.770727172836567, -0.5104258123126475], [1.5564918934750356, 1.949344512888298], [0.5608168155154145, 2.042248437079383], [0.1680281529508214, 1.12261967145669], [0.592597446933246, 1.6830598983761242], [0.31531831819090006, 2.6438493054596357], [0.9781971960344823, 1.1335165635557576], [0.12143445025327171, 1.649227332729178], [1.1062491163816652, 1.0564383354680449], [1.1637032309104423, 1.0297157472656053], [0.3545351785008871, 1.6172930298078492], [1.2679760580738746, 2.02426447785418], [1.30515391100863, 0.9814451283602802], [0.4301539110086299, 0.4973222100843533], [1.4727208712576538, 1.9290352436353655], [1.0900531212287226, 1.564272313597591], [0.5263104406685243, 2.252001645714743], [0.5088264247158644, 1.6627506291231915], [1.1088539665137456, 0.8627712862123935], [1.2213828887912486, 0.9611358591073472], [1.3326196194809328, 1.8769777100754799], [0.3486441669887278, 2.0552816889788144], [1.3453237830437852, 0.9324405022806718], [1.4763077024632731, -0.058943990665182744], [1.4175430798551898, 2.420961710813284], [0.5230026801835554, 1.1905872615059825], [1.4364435597565426, 1.597558709552313], [1.2429354532315784, 2.5786573858887576], [1.2018419557046902, 1.804620608095294], [1.057635673998864, 1.6187166072528705], [0.20228186694759687, 1.8342791475723421], [0.2205191211663864, 2.8341128350216893], [1.1445752366851951, 2.451855990253293], [0.3592594029345495, 1.2660533961640876], [1.3380833432101595, 1.4707573139168484], [0.4630833432101592, 0.9866343956409214], [0.1368988062449742, 2.0365466778659114], [-0.8559173981174999, 1.9168972551540162], [0.09049303247162965, 1.5939308530746388], [-0.7662697133095808, 2.1096416222480587], [1.6612281728619929, 0.5536875760521727], [0.9796692879393203, 1.6058904037333572], [0.015774841397530937, 1.339605789221183], [0.940831947590445, 1.5558231190940979], [0.9590692018092344, 2.5556568065434453], [0.7077312193428289, 1.5877574220154267], [0.8082773124030966, 1.8343966478947915], [1.839647684807919, 1.1609902036458974], [2.006698722541895, 2.1469384531965527], [1.2860259167833825, 2.403585371347968], [0.8259438844811914, 1.3505026813202505], [1.7393847640541784, 1.7574741293665814], [2.0769567149741475, 0.8161744013563763], [-0.1803858830813343, 1.4695162175345666], [0.7792924014075175, 1.2085097706635224], [1.692733280980505, 1.615481218709853], [0.7660245475077954, 1.1465498154551894], [1.4327153106736916, 0.4012153762205093], [0.4331552219165984, 0.4308739156975573], [0.7543249220403789, 1.061149760742939], [0.23513412717913762, 0.6355608019061545], [1.59690747481052, 1.5997169726741451], [0.7504399112429068, 0.9979043760289021], [0.75159917861132, 0.9117144306750404], [1.6650400581843074, 1.3186858787213713], [0.7571837956375258, 0.8485964138399587], [1.7278913942300083, 0.6083324797282905], [0.3880594228076366, -0.8751887423504227], [0.7711760597243901, 0.7635419187990933], [0.29708106397211065, 1.6440156180102883], [0.7861055534582106, 0.70196122203968], [-0.20671065090426355, 0.5823117993277844], [0.3715840465362905, 1.3981397486603249], [0.788964427055358, 0.4894078751366992], [0.8903783806216422, 1.6965099526282543], [1.2279503315416116, 0.7552102246180495], [0.2439748790494065, 0.9335142035213844], [0.8126182502053456, 0.6199421541827015], [0.07345396586341257, 2.8607621810822454], [0.7468749541776585, 1.2629540643188015], [0.5425795612966253, 1.0424147496300362], [0.9569278039006794, 0.35911818837439624], [1.482392896905493, 1.2099333481085437], [0.9970976759358351, 0.3101135622947875], [1.0153349301546246, 1.309947249744135], [0.36893142858238903, 0.546951500637265], [-0.42110862960191797, -0.28761145980817837], [1.0565715301498637, 0.24772036358001093], [1.0114499193571516, 1.2467018650300983], [1.7817806314178308, 0.6090572978485951], [-0.6093592680409019, 1.351008415295675], [0.43753861353051704, 1.837443385215062], [-0.255889856319619, 1.1169179122432187], [0.9135245158124501, -0.02437358261627498], [0.1571860678386352, 0.5282164895243621], [0.16530240952176267, 0.9731843413887911], [0.10403640967067151, 0.23490862692203252], [0.11078029406529022, 0.08560066473308897], [-0.8887797946918026, 0.11525920421013724], [-0.36331470168698954, 0.9660743639442844], [0.42291347117500444, 1.0356390217205296], [0.3506941743635996, -0.4528821868120822], [0.8988522186589094, -1.289256958355522], [0.7108078358631715, -0.7143786781777088], [1.1717053025594466, 0.15241788721931337], [-0.6624092915925359, 0.9357985788175719], [0.5253018009872117, -0.6105778618875563], [0.33179369446224727, 0.37052081444888807], [1.2997572229066297, 0.07533965913160051], [0.3533467923175, 0.398306061210978], [0.9398309569163396, 0.9573489444527866], [0.5480432850258519, 0.6361943534714046], [0.41080090684627746, 0.3715834730085387], [0.12704941469339148, -1.1239862313516182], [0.4391825918031056, -0.17394787436417766], [0.39331689089361743, -0.2176675435830131], [1.330698640688272, 0.13063613878614], [1.4378668228902858, 0.018207479564413487], [0.9637718271380062, 0.8986811787756086], [0.4383067341331929, 0.047866019041461566], [1.4171306744088026, 0.2525699367942223], [1.024567021781315, 0.8808201512350309], [1.0574294183556177, 2.68245820217891], [0.7283313054729149, 0.637991019205339], [0.6894939651240394, 0.5879237345660794], [0.36462014301003787, 1.476846628280768], [0.527954418941112, 0.24061038613550406], [0.5002611961449144, -0.056184953852978003], [1.4137020757179017, 0.35078649419335284], [0.6045340233083467, 0.9383637767355963], [1.3894408557410602, 0.9633962633376428], [2.1019973198164448, 0.261781493321799], [1.538254639256246, 0.9495108254389512], [0.9599599418156927, 0.13368287610641022], [0.7477872932890057, 0.1467161280058418], [0.9148383310229814, 1.1326643775564973], [-0.22494790512305274, -0.41752188812156277], [0.767868299239421, -0.29787246540966744], [0.5743601927144565, 0.6832262109267769], [2.257227838786024, 0.8192273088366542], [2.53450696752837, -0.1415620982468575], [1.071815867009933, 0.5644386261482432], [0.07225577825283946, 0.594097165625291], [0.024661084881184014, 0.5222306522564679], [0.9170894728776984, -0.28942327090617476], [1.0213623000411303, 0.7051254596823999], [2.768688082982341, -0.48355294591288134], [1.2181110763327907, 0.5950337120105863], [1.743576169337604, 1.4458488717447335], [1.6959814759659488, 1.3739823583759105], [2.1315519559057776, 1.0020051600569175], [2.6570170489105913, 1.8528203197910647], [1.6574569601534979, 1.8824788592681125], [2.1058092051387587, 1.0267797959909966], [2.417942382248473, 1.9768181529784372], [1.5092772534794157, -0.7888818407908914], [1.3157691469544512, 0.1922168355455527], [1.6947832883553762, -0.8926826570810439], [2.3997051280296833, 0.9769844655290901], [1.6468028039655183, 0.31885219127202313], [2.2716532076824993, 1.0540626936168032], [1.3517476137061801, 0.4548374670877919], [1.5452557202311448, -0.5262612092486523], [1.9954984973245151, 0.3666449681716013], [1.5214035015722356, 1.2471186673827963], [2.3500275417978815, 0.1682664936410564], [2.3682647960166707, 1.1681001810904037], [1.4218543654275408, 1.4910665831697814], [1.4036171112087517, 0.4912328957204339], [1.3543524638382602, 0.2611704178321412], [1.3725897180570497, 1.2610041052814884], [2.0986895593314756, -0.799632890886962], [2.624154652336289, 0.05118226884718524], [1.631338447973815, -0.06846715386471014], [2.517078579531857, 1.1542147431917118], [1.5331031270396522, 1.3325187220950467], [2.4166907631658963, 0.22291139731717446], [1.7702872615936611, -0.5400843517896953], [2.480793718294174, 1.2664647539853577], [1.7873652484440377, 0.5459392810135145], [1.9884614683231048, 1.1471047657472053], [1.05107971852845, 0.7988010833780526], [1.9486231373001242, 0.49856330646663494], [1.5161460334862547, 0.589597468615388], [0.5321705809940498, 0.7679014475187227], [-0.43172386554773956, 0.5016168330065485], [1.4641556426867042, 0.2100996606591965], [2.377596522259691, 0.617071108705527], [1.52517471428073, -0.08167261583314223], [0.5612802677389404, -0.34795723034531645], [-0.22022421410431514, 0.8209747582539961], [1.9149059487458746, 1.8142114230717898], [1.5275145076982053, 0.21095184665845568], [1.6002091012679673, 0.8454598525445117], [0.9067806314178312, 0.12493437957266845], [2.674056115518809, 0.5859889917834579], [2.199961119766529, 1.466462690994653], [1.2360666732247396, 1.2001780764824785], [0.8533989231958088, 0.8354151464447039], [1.7130205425741665, 0.10715103036830398], [1.6744960267617155, 0.6156475312605059], [0.7106015802199259, 0.34936291674833153], [1.7101616689770194, 0.3197043772712836], [1.6919244147582297, -0.680129310178064], [1.6523874209917238, 0.2557341744369427], [2.7207075985924827, 0.7279819024401859], [1.9365070255500658, 1.4629191949030127], [1.5191266450309988, 2.3716510684266385], [1.6851690430836603, 0.4950198103749942], [3.2738659143934936, 1.8755924021628159], [2.520963590329329, 1.217460127905749], [2.3461561902466785, 0.8081868242668402], [1.0112413875054693, 0.15025962409748178], [2.082420438703375, 0.4099540051977453], [2.687381749794654, 1.3165495189210072], [2.430646545811325, 1.0322809451836297], [1.7996775862318015, 1.7769638084398882], [2.2499203633251716, 2.6698699858601422], [1.3749203633251716, 2.1857470675842148], [1.7034062973024497, 1.4948534978243422], [0.7569958667133201, 1.8178198999037196], [1.9344794257304896, 0.6584172446639405], [1.952716679949279, 1.658250932113288], [2.1893458838207076, 2.25569098380938], [2.115282835277678, 2.2115920561416247], [1.6706246752105134, 1.2555678618862895], [0.7465685596917042, 1.637824706654686], [2.6452036253541977, 0.3753835971488375], [2.097045581058888, 1.2117583686922775], [3.080821260092055, 0.4664854110745593], [2.32791893602789, -0.19164686318250757], [1.559096743607116, 1.7556774636326469], [0.7436702324374738, 1.08725937276617], [2.443428469850136, 1.6887713095236978], [1.3574363473725117, 1.596496050992663], [2.0506398072855427, 0.7691425439010047], [3.0434560116480176, 0.8887919666128999], [2.072748413055535, 1.1290559007245682], [1.1263379824664053, 1.4520223028039454], [2.328294697440554, 1.7840737858843945], [1.3354784930780799, 1.664424363172499], [1.935506034875961, 0.8644450202617014], [1.0787432890947504, 1.380155789435122], [2.081956986479849, 2.8001596726224456], [2.06371973226106, 1.8003259851730982], [1.3290546624156838, 2.1420273983653795], [1.641187839525398, 3.0920657553528197], [1.2914896405126086, 1.9973649751893363], [2.2898904619012885, 2.0538963810661506], [1.2966411527292037, 1.6143570785332406], [1.8966686945270848, 0.8143777356224429], [1.5820366231546772, 1.0985355233141583], [1.5637993689358878, 0.09870183586481085], [2.0241271105384078, 0.851161611757983], [1.2363483305515808, 1.5948673994599336], [0.4524681072643386, 1.775862688455648], [1.0693169727472398, 1.7986347708273995], [2.1072547879024266, 0.9193026920892076], [0.07631283946056078, 2.6482088341792647], [1.5242623751693827, 1.0345653204798166], [0.5402869226771778, 1.2128692993831516], [0.877858873597147, 0.27156957137294646], [1.380177831411152, 2.8432682663516236], [0.9302167116446229, 2.3450514119088255], [0.8547127384063385, 1.9924531066174764], [1.2732523862938192, 0.9975312877399896], [1.0612006310641113, 1.3536669189629709], [1.6029348446970264, 0.9948951826124105], [1.6110511863801538, 1.4398630344768393]], \"edge\": [[1, 2], [1, 7], [1, 11], [1, 14], [1, 15], [1, 19], [1, 22], [1, 25], [1, 28], [1, 32], [1, 36], [1, 40], [1, 43], [1, 44], [1, 50], [1, 57], [1, 61], [1, 64], [1, 67], [1, 70], [1, 80], [1, 84], [1, 92], [1, 95], [1, 98], [1, 102], [1, 106], [1, 122], [1, 143], [1, 157], [1, 185], [1, 211], [1, 216], [1, 238], [1, 242], [1, 247], [1, 252], [1, 256], [1, 265], [1, 270], [1, 274], [1, 278], [1, 283], [1, 290], [1, 318], [1, 322], [1, 330], [1, 368], [1, 380], [1, 409], [1, 417], [1, 521], [1, 541], [1, 576], [1, 595], [1, 619], [1, 656], [1, 665], [1, 1067], [1, 1072], [1, 1076], [1, 1079], [1, 1080], [1, 1084], [1, 1087], [1, 1090], [1, 1093], [1, 1097], [1, 1101], [1, 1105], [1, 1108], [1, 1109], [1, 1115], [1, 1122], [1, 1126], [1, 1129], [1, 1132], [1, 1135], [1, 1145], [1, 1149], [1, 1157], [1, 1160], [1, 1163], [1, 1167], [1, 1171], [1, 1187], [1, 1208], [1, 1222], [1, 1250], [1, 1276], [1, 1281], [1, 1303], [1, 1307], [1, 1312], [1, 1317], [1, 1321], [1, 1330], [1, 1335], [1, 1339], [1, 1343], [1, 1348], [1, 1355], [1, 1383], [1, 1387], [1, 1395], [1, 1433], [1, 1445], [1, 1474], [1, 1482], [1, 1586], [1, 1606], [1, 1641], [1, 1660], [1, 1684], [1, 1721], [1, 1730], [1, 2132], [1, 2156], [2, 3], [2, 80], [2, 111], [2, 114], [2, 118], [2, 238], [2, 483], [2, 2133], [3, 4], [3, 69], [3, 261], [3, 274], [3, 316], [3, 410], [3, 758], [3, 2134], [4, 5], [4, 27], [4, 172], [4, 281], [4, 282], [4, 306], [4, 369], [4, 416], [4, 483], [4, 758], [4, 2134], [4, 2135], [4, 2157], [4, 2546], [4, 2651], [5, 6], [5, 49], [5, 69], [5, 77], [5, 82], [5, 170], [5, 180], [5, 293], [5, 307], [5, 310], [5, 313], [5, 416], [5, 420], [5, 491], [5, 520], [5, 594], [5, 609], [5, 640], [5, 698], [5, 724], [5, 740], [5, 757], [5, 858], [5, 951], [5, 2136], [5, 2145], [5, 2196], [5, 2200], [5, 2212], [5, 2309], [5, 2440], [5, 2546], [5, 2552], [5, 2650], [5, 2663], [5, 2669], [5, 2724], [5, 2739], [5, 2835], [5, 2854], [6, 10], [6, 14], [6, 18], [6, 31], [6, 35], [6, 39], [6, 43], [6, 49], [6, 53], [6, 56], [6, 60], [6, 73], [6, 83], [6, 87], [6, 110], [6, 117], [6, 121], [6, 132], [6, 142], [6, 150], [6, 164], [6, 171], [6, 184], [6, 189], [6, 196], [6, 215], [6, 220], [6, 227], [6, 231], [6, 237], [6, 246], [6, 251], [6, 260], [6, 264], [6, 269], [6, 273], [6, 282], [6, 297], [6, 304], [6, 326], [6, 337], [6, 353], [6, 364], [6, 372], [6, 379], [6, 390], [6, 408], [6, 413], [6, 430], [6, 436], [6, 468], [6, 514], [6, 525], [6, 529], [6, 551], [6, 580], [6, 698], [6, 774], [6, 780], [6, 792], [6, 830], [6, 845], [6, 847], [6, 871], [6, 2132], [6, 2137], [6, 2144], [6, 2147], [6, 2151], [6, 2155], [6, 2156], [6, 2158], [6, 2169], [6, 2172], [6, 2174], [6, 2178], [6, 2183], [6, 2186], [6, 2197], [6, 2219], [6, 2227], [6, 2231], [6, 2232], [6, 2251], [6, 2296], [6, 2306], [6, 2324], [6, 2337], [6, 2354], [6, 2376], [6, 2399], [6, 2425], [6, 2429], [6, 2432], [6, 2435], [6, 2436], [6, 2439], [6, 2442], [6, 2445], [6, 2480], [6, 2502], [6, 2504], [6, 2554], [6, 2578], [6, 2579], [6, 2581], [6, 2595], [6, 2606], [6, 2614], [6, 2615], [6, 2627], [6, 2630], [6, 2636], [6, 2639], [6, 2644], [6, 2651], [6, 2671], [6, 2693], [6, 2704], [6, 2728], [6, 2729], [6, 2744], [6, 2772], [6, 2783], [6, 2795], [6, 2805], [6, 2835], [6, 2860], [6, 2862], [7, 8], [7, 29], [7, 33], [7, 65], [7, 84], [7, 93], [7, 152], [7, 168], [7, 172], [7, 175], [7, 178], [7, 181], [7, 265], [7, 279], [7, 410], [7, 418], [7, 548], [7, 577], [7, 596], [7, 632], [7, 718], [7, 2138], [8, 9], [8, 116], [8, 181], [8, 252], [8, 316], [8, 357], [8, 618], [8, 678], [8, 695], [8, 1008], [8, 2139], [8, 2226], [8, 2321], [8, 2794], [9, 10], [9, 163], [9, 175], [9, 223], [9, 282], [9, 728], [9, 949], [9, 1008], [9, 2139], [9, 2140], [9, 2227], [9, 2585], [9, 2858], [10, 72], [10, 96], [10, 110], [10, 529], [10, 695], [10, 737], [10, 784], [10, 891], [10, 974], [10, 1020], [10, 2141], [10, 2145], [10, 2226], [10, 2228], [10, 2337], [10, 2341], [10, 2681], [10, 2744], [10, 2748], [10, 2789], [10, 2826], [10, 3102], [11, 12], [11, 85], [11, 95], [11, 109], [11, 200], [11, 203], [11, 206], [11, 209], [11, 278], [11, 2142], [12, 13], [12, 161], [12, 162], [12, 200], [12, 322], [12, 370], [12, 371], [12, 454], [12, 481], [12, 726], [12, 2143], [13, 14], [13, 37], [13, 87], [13, 119], [13, 235], [13, 240], [13, 279], [13, 294], [13, 424], [13, 460], [13, 596], [13, 602], [13, 603], [13, 629], [13, 721], [13, 2144], [13, 2156], [14, 16], [14, 20], [14, 24], [14, 25], [14, 27], [14, 38], [14, 41], [14, 43], [14, 47], [14, 52], [14, 55], [14, 66], [14, 88], [14, 96], [14, 100], [14, 101], [14, 120], [14, 165], [14, 175], [14, 193], [14, 206], [14, 223], [14, 245], [14, 268], [14, 294], [14, 298], [14, 301], [14, 304], [14, 305], [14, 308], [14, 311], [14, 314], [14, 349], [14, 371], [14, 373], [14, 423], [14, 447], [14, 448], [14, 450], [14, 464], [14, 475], [14, 483], [14, 484], [14, 496], [14, 499], [14, 505], [14, 508], [14, 513], [14, 520], [14, 540], [14, 562], [14, 573], [14, 597], [14, 598], [14, 613], [14, 641], [14, 652], [14, 664], [14, 674], [14, 704], [14, 729], [14, 731], [14, 2132], [14, 2145], [14, 2157], [14, 2582], [14, 2585], [14, 2590], [14, 2593], [14, 2775], [15, 16], [15, 36], [15, 74], [15, 114], [15, 181], [15, 252], [15, 338], [15, 341], [15, 344], [15, 347], [15, 350], [15, 354], [15, 361], [15, 365], [15, 376], [15, 444], [15, 542], [15, 616], [15, 682], [15, 713], [15, 735], [15, 2146], [16, 17], [16, 75], [16, 223], [16, 341], [16, 359], [16, 543], [16, 643], [16, 653], [16, 654], [16, 876], [16, 2146], [16, 2147], [16, 2156], [16, 2585], [17, 18], [17, 259], [17, 263], [17, 295], [17, 347], [17, 363], [17, 379], [17, 513], [17, 825], [17, 1053], [17, 2148], [17, 2439], [17, 2478], [17, 2493], [18, 21], [18, 56], [18, 199], [18, 230], [18, 296], [18, 389], [18, 404], [18, 439], [18, 565], [18, 591], [18, 643], [18, 661], [18, 691], [18, 988], [18, 2145], [18, 2149], [18, 2186], [18, 2635], [18, 2822], [19, 20], [19, 136], [19, 209], [19, 254], [19, 274], [19, 365], [19, 384], [19, 387], [19, 391], [19, 394], [19, 397], [19, 400], [19, 405], [19, 440], [19, 479], [19, 500], [19, 521], [19, 648], [19, 660], [19, 669], [19, 715], [19, 2150], [20, 21], [20, 27], [20, 207], [20, 391], [20, 403], [20, 404], [20, 480], [20, 615], [20, 670], [20, 923], [20, 2150], [20, 2151], [20, 2156], [20, 2157], [21, 269], [21, 311], [21, 400], [21, 458], [21, 694], [21, 814], [21, 835], [21, 2152], [21, 2399], [21, 2531], [21, 2588], [21, 2590], [22, 23], [22, 54], [22, 57], [22, 152], [22, 290], [22, 443], [22, 448], [22, 534], [22, 2153], [23, 24], [23, 58], [23, 197], [23, 242], [23, 302], [23, 443], [23, 447], [23, 568], [23, 2154], [24, 54], [24, 73], [24, 79], [24, 210], [24, 314], [24, 361], [24, 362], [24, 397], [24, 444], [24, 465], [24, 481], [24, 519], [24, 555], [24, 648], [24, 2155], [24, 2156], [25, 26], [25, 451], [25, 454], [25, 459], [25, 462], [25, 644], [25, 2156], [26, 27], [26, 274], [26, 457], [26, 466], [26, 497], [26, 549], [26, 593], [26, 2157], [27, 104], [27, 173], [27, 274], [27, 294], [27, 458], [27, 467], [27, 550], [27, 571], [27, 594], [27, 599], [27, 600], [27, 717], [27, 748], [27, 847], [27, 874], [27, 876], [27, 920], [27, 927], [27, 1016], [27, 1032], [27, 2156], [27, 2158], [27, 2405], [27, 2588], [27, 2597], [27, 2628], [27, 2680], [27, 2724], [28, 29], [28, 318], [28, 465], [28, 469], [28, 472], [28, 475], [28, 478], [28, 576], [28, 2159], [29, 30], [29, 94], [29, 173], [29, 349], [29, 357], [29, 402], [29, 471], [29, 479], [29, 503], [29, 561], [29, 562], [29, 577], [29, 623], [29, 702], [29, 2160], [29, 2511], [30, 31], [30, 175], [30, 364], [30, 471], [30, 475], [30, 480], [30, 486], [30, 550], [30, 827], [30, 983], [30, 2160], [30, 2161], [30, 2627], [30, 2680], [31, 63], [31, 94], [31, 104], [31, 117], [31, 177], [31, 366], [31, 367], [31, 393], [31, 404], [31, 456], [31, 557], [31, 625], [31, 717], [31, 923], [31, 2145], [31, 2162], [31, 2225], [31, 2586], [31, 2795], [32, 33], [32, 90], [32, 276], [32, 330], [32, 481], [32, 484], [32, 489], [32, 531], [32, 595], [32, 2163], [33, 34], [33, 90], [33, 213], [33, 416], [33, 460], [33, 483], [33, 485], [33, 491], [33, 538], [33, 562], [33, 575], [33, 586], [33, 596], [33, 2164], [33, 2546], [34, 35], [34, 141], [34, 164], [34, 168], [34, 307], [34, 461], [34, 485], [34, 487], [34, 531], [34, 677], [34, 895], [34, 997], [34, 1047], [34, 1048], [34, 2138], [34, 2163], [34, 2165], [34, 2221], [34, 2344], [34, 2547], [34, 2591], [34, 2614], [34, 2616], [34, 2622], [34, 2669], [34, 2693], [34, 2706], [34, 2717], [34, 2727], [35, 121], [35, 241], [35, 441], [35, 442], [35, 471], [35, 486], [35, 562], [35, 671], [35, 742], [35, 744], [35, 795], [35, 804], [35, 871], [35, 895], [35, 916], [35, 924], [35, 2145], [35, 2160], [35, 2164], [35, 2166], [35, 2251], [35, 2371], [35, 2476], [35, 2571], [35, 2617], [35, 2638], [35, 2643], [35, 2692], [35, 2772], [35, 2798], [35, 2800], [35, 2834], [35, 2874], [35, 2876], [35, 3015], [36, 37], [36, 118], [36, 119], [36, 386], [36, 492], [36, 497], [36, 500], [36, 503], [36, 2167], [37, 38], [37, 102], [37, 288], [37, 685], [37, 735], [37, 2168], [38, 39], [38, 366], [38, 405], [38, 475], [38, 500], [38, 576], [38, 581], [38, 589], [38, 590], [38, 652], [38, 2156], [38, 2169], [38, 2707], [38, 2719], [39, 406], [39, 408], [39, 501], [39, 502], [39, 580], [39, 581], [39, 685], [39, 879], [39, 1024], [39, 2145], [39, 2168], [39, 2170], [39, 2497], [39, 2536], [39, 2606], [39, 2631], [39, 2707], [39, 2712], [39, 2720], [39, 2721], [39, 2783], [40, 41], [40, 42], [40, 61], [40, 368], [40, 506], [40, 509], [40, 647], [40, 2171], [41, 42], [41, 196], [41, 373], [41, 510], [41, 540], [41, 603], [41, 642], [41, 791], [41, 2156], [41, 2171], [41, 2172], [42, 43], [42, 196], [42, 219], [42, 328], [42, 357], [42, 377], [42, 479], [42, 604], [42, 626], [42, 628], [42, 680], [42, 716], [42, 857], [42, 934], [42, 2132], [42, 2172], [42, 2173], [42, 2192], [42, 2499], [42, 2637], [42, 2640], [42, 2778], [43, 46], [43, 69], [43, 79], [43, 94], [43, 106], [43, 109], [43, 111], [43, 135], [43, 144], [43, 158], [43, 168], [43, 174], [43, 186], [43, 192], [43, 214], [43, 219], [43, 221], [43, 224], [43, 228], [43, 232], [43, 235], [43, 236], [43, 257], [43, 266], [43, 286], [43, 291], [43, 316], [43, 327], [43, 333], [43, 334], [43, 341], [43, 378], [43, 386], [43, 391], [43, 396], [43, 402], [43, 427], [43, 436], [43, 469], [43, 511], [43, 526], [43, 531], [43, 534], [43, 547], [43, 552], [43, 566], [43, 579], [43, 581], [43, 604], [43, 606], [43, 610], [43, 620], [43, 631], [43, 634], [43, 637], [43, 657], [43, 668], [43, 691], [43, 695], [43, 701], [43, 708], [43, 748], [43, 752], [43, 753], [43, 765], [43, 798], [43, 962], [43, 1036], [43, 2133], [43, 2138], [43, 2142], [43, 2145], [43, 2146], [43, 2150], [43, 2153], [43, 2156], [43, 2159], [43, 2163], [43, 2167], [43, 2171], [43, 2174], [43, 2175], [43, 2181], [43, 2188], [43, 2192], [43, 2195], [43, 2198], [43, 2201], [43, 2211], [43, 2215], [43, 2223], [43, 2226], [43, 2229], [43, 2233], [43, 2237], [43, 2253], [43, 2274], [43, 2288], [43, 2316], [43, 2342], [43, 2347], [43, 2369], [43, 2373], [43, 2378], [43, 2383], [43, 2387], [43, 2396], [43, 2401], [43, 2405], [43, 2409], [43, 2414], [43, 2421], [43, 2449], [43, 2453], [43, 2461], [43, 2499], [43, 2511], [43, 2540], [43, 2548], [43, 2652], [43, 2672], [43, 2707], [43, 2726], [43, 2750], [43, 2787], [43, 2796], [43, 3198], [43, 3203], [43, 3207], [43, 3210], [43, 3211], [43, 3215], [43, 3218], [43, 3221], [43, 3224], [43, 3228], [43, 3232], [43, 3236], [43, 3239], [43, 3240], [43, 3246], [43, 3253], [43, 3257], [43, 3260], [43, 3263], [43, 3266], [43, 3276], [43, 3280], [43, 3288], [43, 3291], [43, 3294], [43, 3298], [43, 3302], [43, 3318], [43, 3339], [43, 3353], [43, 3381], [43, 3407], [43, 3412], [43, 3434], [43, 3438], [43, 3443], [43, 3448], [43, 3452], [43, 3461], [43, 3466], [43, 3470], [43, 3474], [43, 3479], [43, 3486], [43, 3514], [43, 3518], [43, 3526], [43, 3564], [43, 3576], [43, 3605], [43, 3613], [43, 3717], [43, 3737], [43, 3772], [43, 3791], [43, 3815], [43, 3852], [43, 3861], [44, 45], [44, 47], [44, 64], [44, 380], [44, 511], [44, 515], [44, 518], [44, 2175], [45, 46], [45, 232], [45, 252], [45, 399], [45, 564], [45, 705], [45, 823], [45, 2176], [46, 79], [46, 139], [46, 203], [46, 243], [46, 244], [46, 266], [46, 276], [46, 353], [46, 515], [46, 564], [46, 703], [46, 706], [46, 732], [46, 734], [46, 745], [46, 2132], [46, 2177], [46, 2396], [47, 48], [47, 52], [47, 142], [47, 399], [47, 496], [47, 511], [47, 554], [47, 585], [47, 586], [47, 592], [47, 2156], [47, 2175], [47, 2178], [48, 49], [48, 167], [48, 301], [48, 353], [48, 382], [48, 515], [48, 516], [48, 2179], [48, 2646], [49, 89], [49, 160], [49, 167], [49, 281], [49, 480], [49, 554], [49, 589], [49, 615], [49, 634], [49, 711], [49, 712], [49, 725], [49, 891], [49, 2145], [49, 2180], [49, 2651], [49, 2719], [49, 2765], [50, 51], [50, 54], [50, 417], [50, 545], [50, 548], [50, 552], [50, 555], [50, 560], [50, 563], [50, 2181], [51, 52], [51, 120], [51, 330], [51, 452], [51, 478], [51, 479], [51, 506], [51, 515], [51, 517], [51, 546], [51, 2182], [52, 53], [52, 64], [52, 176], [52, 452], [52, 508], [52, 547], [52, 553], [52, 706], [52, 2156], [52, 2183], [52, 2195], [53, 138], [53, 142], [53, 336], [53, 390], [53, 453], [53, 517], [53, 547], [53, 591], [53, 906], [53, 914], [53, 926], [53, 942], [53, 995], [53, 2145], [53, 2178], [53, 2182], [53, 2184], [53, 2195], [53, 2307], [53, 2583], [53, 2639], [53, 2678], [53, 2684], [53, 2837], [54, 55], [54, 444], [54, 477], [54, 563], [54, 2185], [55, 56], [55, 66], [55, 105], [55, 152], [55, 154], [55, 232], [55, 497], [55, 503], [55, 509], [55, 535], [55, 548], [55, 549], [55, 663], [55, 727], [55, 2156], [55, 2186], [56, 153], [56, 171], [56, 232], [56, 234], [56, 399], [56, 429], [56, 474], [56, 477], [56, 494], [56, 527], [56, 537], [56, 550], [56, 554], [56, 571], [56, 618], [56, 659], [56, 680], [56, 728], [56, 740], [56, 881], [56, 882], [56, 1053], [56, 2145], [56, 2185], [56, 2187], [56, 2197], [56, 2236], [56, 2283], [56, 2285], [56, 2363], [56, 2628], [56, 2634], [56, 2640], [56, 2666], [56, 2679], [56, 2680], [56, 2794], [56, 2858], [57, 58], [57, 566], [57, 570], [57, 573], [57, 619], [57, 2188], [58, 59], [58, 90], [58, 112], [58, 242], [58, 333], [58, 395], [58, 396], [58, 460], [58, 567], [58, 681], [58, 692], [58, 2189], [58, 2249], [58, 2540], [59, 60], [59, 91], [59, 132], [59, 166], [59, 301], [59, 302], [59, 567], [59, 573], [59, 684], [59, 693], [59, 837], [59, 2189], [59, 2190], [59, 2594], [59, 2612], [59, 2630], [60, 63], [60, 75], [60, 87], [60, 91], [60, 97], [60, 333], [60, 426], [60, 488], [60, 491], [60, 575], [60, 654], [60, 719], [60, 720], [60, 1040], [60, 2144], [60, 2145], [60, 2191], [60, 2464], [60, 2594], [61, 62], [61, 332], [61, 540], [61, 601], [61, 604], [61, 665], [61, 2192], [62, 63], [62, 94], [62, 102], [62, 333], [62, 503], [62, 509], [62, 601], [62, 605], [62, 624], [62, 645], [62, 662], [62, 749], [62, 2193], [63, 510], [63, 540], [63, 605], [63, 625], [63, 729], [63, 2193], [63, 2194], [64, 65], [64, 92], [64, 444], [64, 519], [64, 547], [64, 607], [64, 2195], [65, 66], [65, 93], [65, 170], [65, 176], [65, 520], [65, 607], [65, 705], [65, 2196], [66, 171], [66, 181], [66, 194], [66, 208], [66, 209], [66, 317], [66, 444], [66, 446], [66, 450], [66, 479], [66, 498], [66, 633], [66, 739], [66, 749], [66, 833], [66, 927], [66, 2156], [66, 2197], [66, 2582], [66, 2732], [67, 68], [67, 95], [67, 541], [67, 610], [67, 613], [67, 616], [67, 2198], [68, 69], [68, 93], [68, 122], [68, 248], [68, 307], [68, 522], [68, 538], [68, 604], [68, 953], [68, 2192], [68, 2199], [68, 2669], [69, 108], [69, 123], [69, 520], [69, 533], [69, 634], [69, 638], [69, 639], [69, 757], [69, 2132], [69, 2200], [69, 2651], [70, 71], [70, 74], [70, 76], [70, 78], [70, 102], [70, 239], [70, 248], [70, 271], [70, 632], [70, 635], [70, 638], [70, 641], [70, 644], [70, 647], [70, 798], [70, 2201], [70, 2775], [71, 72], [71, 74], [71, 79], [71, 210], [71, 252], [71, 611], [71, 695], [71, 918], [71, 2202], [71, 2226], [71, 2341], [72, 73], [72, 75], [72, 205], [72, 210], [72, 223], [72, 407], [72, 612], [72, 641], [72, 733], [72, 737], [72, 743], [72, 913], [72, 914], [72, 918], [72, 1057], [72, 2155], [72, 2202], [72, 2203], [72, 2227], [72, 2340], [72, 2585], [72, 2607], [72, 2748], [72, 2775], [72, 2779], [72, 2863], [72, 2880], [73, 77], [73, 79], [73, 125], [73, 146], [73, 167], [73, 189], [73, 277], [73, 325], [73, 329], [73, 353], [73, 363], [73, 445], [73, 477], [73, 556], [73, 559], [73, 568], [73, 612], [73, 651], [73, 693], [73, 793], [73, 860], [73, 941], [73, 2145], [73, 2154], [73, 2185], [73, 2204], [73, 2210], [73, 2341], [73, 2445], [73, 2492], [73, 2493], [73, 2528], [73, 2575], [73, 2596], [73, 2612], [73, 2650], [73, 2686], [73, 2779], [74, 75], [74, 333], [74, 469], [74, 473], [74, 477], [74, 494], [74, 735], [74, 856], [74, 2159], [74, 2185], [74, 2205], [74, 2634], [75, 250], [75, 408], [75, 641], [75, 856], [75, 1058], [75, 2205], [75, 2206], [75, 2341], [75, 2606], [75, 2775], [76, 77], [76, 368], [76, 477], [76, 533], [76, 547], [76, 591], [76, 592], [76, 636], [76, 647], [76, 691], [76, 740], [76, 907], [76, 2185], [76, 2195], [76, 2207], [76, 2650], [77, 103], [77, 124], [77, 167], [77, 276], [77, 327], [77, 369], [77, 489], [77, 507], [77, 512], [77, 519], [77, 547], [77, 609], [77, 806], [77, 2155], [77, 2195], [77, 2208], [77, 2453], [77, 2649], [77, 2651], [77, 2739], [78, 79], [78, 167], [78, 409], [78, 478], [78, 515], [78, 634], [78, 894], [78, 2209], [79, 145], [79, 324], [79, 472], [79, 611], [79, 692], [79, 860], [79, 2132], [79, 2155], [79, 2210], [80, 81], [80, 576], [80, 579], [80, 649], [80, 652], [80, 682], [80, 2211], [81, 82], [81, 105], [81, 166], [81, 417], [81, 482], [81, 520], [81, 532], [81, 573], [81, 592], [81, 596], [81, 655], [81, 2212], [82, 83], [82, 293], [82, 401], [82, 427], [82, 474], [82, 553], [82, 579], [82, 583], [82, 621], [82, 677], [82, 901], [82, 908], [82, 1030], [82, 2211], [82, 2213], [82, 2236], [82, 2297], [82, 2548], [82, 2613], [82, 2651], [82, 2663], [82, 2704], [82, 2723], [82, 2727], [82, 2786], [83, 227], [83, 260], [83, 367], [83, 566], [83, 573], [83, 683], [83, 819], [83, 875], [83, 908], [83, 2145], [83, 2188], [83, 2190], [83, 2212], [83, 2214], [83, 2419], [83, 2579], [83, 2667], [83, 2697], [83, 2702], [83, 2705], [83, 2786], [83, 2805], [83, 2813], [84, 85], [84, 88], [84, 90], [84, 198], [84, 319], [84, 462], [84, 607], [84, 672], [84, 675], [84, 678], [84, 681], [84, 686], [84, 689], [84, 752], [84, 2215], [84, 2593], [85, 86], [85, 144], [85, 333], [85, 446], [85, 493], [85, 618], [85, 659], [85, 681], [85, 719], [85, 726], [85, 1028], [85, 2216], [85, 2274], [85, 2733], [85, 2794], [86, 87], [86, 117], [86, 134], [86, 135], [86, 194], [86, 200], [86, 202], [86, 221], [86, 602], [86, 625], [86, 633], [86, 675], [86, 676], [86, 726], [86, 1027], [86, 2144], [86, 2217], [86, 2278], [86, 2347], [86, 2732], [86, 2795], [86, 2796], [87, 113], [87, 128], [87, 162], [87, 195], [87, 235], [87, 241], [87, 289], [87, 300], [87, 412], [87, 420], [87, 425], [87, 430], [87, 439], [87, 461], [87, 625], [87, 659], [87, 677], [87, 685], [87, 963], [87, 964], [87, 1013], [87, 2143], [87, 2145], [87, 2168], [87, 2218], [87, 2250], [87, 2366], [87, 2371], [87, 2410], [87, 2425], [87, 2555], [87, 2591], [87, 2727], [87, 2733], [87, 2734], [87, 2760], [87, 2852], [88, 89], [88, 91], [88, 175], [88, 199], [88, 314], [88, 320], [88, 462], [88, 553], [88, 554], [88, 684], [88, 688], [88, 714], [88, 719], [88, 728], [88, 752], [88, 780], [88, 821], [88, 865], [88, 884], [88, 983], [88, 988], [88, 1030], [88, 2156], [88, 2215], [88, 2219], [88, 2594], [88, 2691], [88, 2804], [88, 2818], [88, 2858], [89, 184], [89, 301], [89, 321], [89, 675], [89, 714], [89, 905], [89, 929], [89, 2220], [89, 2324], [89, 2593], [89, 2804], [89, 2806], [90, 91], [90, 271], [90, 333], [90, 463], [90, 482], [90, 487], [90, 490], [90, 587], [90, 630], [90, 631], [90, 686], [90, 2221], [90, 2594], [90, 2672], [91, 230], [91, 412], [91, 413], [91, 463], [91, 484], [91, 485], [91, 487], [91, 536], [91, 589], [91, 621], [91, 688], [91, 801], [91, 888], [91, 2221], [91, 2222], [91, 2593], [91, 2595], [91, 2635], [91, 2719], [91, 2760], [91, 2818], [92, 93], [92, 185], [92, 508], [92, 637], [92, 692], [92, 2223], [93, 94], [93, 99], [93, 111], [93, 169], [93, 177], [93, 194], [93, 419], [93, 649], [93, 673], [93, 2133], [93, 2224], [93, 2732], [94, 99], [94, 103], [94, 136], [94, 192], [94, 317], [94, 393], [94, 507], [94, 552], [94, 624], [94, 650], [94, 660], [94, 811], [94, 2132], [94, 2181], [94, 2225], [95, 96], [95, 503], [95, 695], [95, 699], [95, 702], [95, 705], [95, 2226], [96, 97], [96, 206], [96, 210], [96, 550], [96, 613], [96, 617], [96, 658], [96, 695], [96, 971], [96, 2156], [96, 2226], [96, 2227], [96, 2680], [96, 2793], [97, 301], [97, 375], [97, 551], [97, 646], [97, 697], [97, 699], [97, 707], [97, 971], [97, 1013], [97, 2228], [97, 2232], [97, 2776], [97, 2830], [98, 99], [98, 100], [98, 211], [98, 213], [98, 538], [98, 619], [98, 708], [98, 713], [98, 715], [98, 2229], [99, 129], [99, 191], [99, 203], [99, 214], [99, 252], [99, 455], [99, 456], [99, 539], [99, 626], [99, 709], [99, 713], [99, 716], [99, 2230], [99, 2369], [99, 2586], [100, 101], [100, 172], [100, 297], [100, 359], [100, 456], [100, 548], [100, 555], [100, 674], [100, 708], [100, 711], [100, 2156], [100, 2229], [100, 2231], [100, 2586], [101, 174], [101, 211], [101, 312], [101, 371], [101, 422], [101, 549], [101, 551], [101, 811], [101, 2156], [101, 2232], [101, 2342], [102, 103], [102, 105], [102, 216], [102, 287], [102, 405], [102, 489], [102, 509], [102, 606], [102, 608], [102, 706], [102, 718], [102, 721], [102, 723], [102, 726], [102, 729], [102, 732], [102, 735], [102, 738], [102, 2233], [103, 104], [103, 124], [103, 146], [103, 257], [103, 274], [103, 405], [103, 638], [103, 850], [103, 2234], [103, 2387], [103, 2596], [103, 2650], [104, 514], [104, 572], [104, 594], [104, 729], [104, 850], [104, 926], [104, 2157], [104, 2234], [104, 2235], [104, 2644], [104, 2724], [105, 165], [105, 380], [105, 474], [105, 503], [105, 586], [105, 596], [105, 614], [105, 633], [105, 726], [105, 2236], [106, 107], [106, 217], [106, 323], [106, 492], [106, 522], [106, 545], [106, 753], [106, 2132], [106, 2237], [107, 108], [107, 133], [107, 191], [107, 212], [107, 239], [107, 248], [107, 265], [107, 266], [107, 937], [107, 2238], [107, 2396], [108, 109], [108, 123], [108, 633], [108, 723], [108, 724], [108, 2142], [108, 2239], [108, 2854], [109, 110], [109, 162], [109, 204], [109, 205], [109, 206], [109, 378], [109, 633], [109, 695], [109, 772], [109, 824], [109, 970], [109, 1010], [109, 1028], [109, 2132], [109, 2143], [109, 2216], [109, 2226], [109, 2240], [109, 2331], [109, 2334], [109, 2337], [109, 2340], [109, 2409], [110, 206], [110, 351], [110, 379], [110, 724], [110, 725], [110, 736], [110, 772], [110, 1009], [110, 2142], [110, 2145], [110, 2227], [110, 2240], [110, 2241], [110, 2338], [110, 2439], [110, 2481], [110, 2598], [110, 2850], [110, 2854], [110, 2866], [111, 112], [111, 115], [111, 164], [111, 214], [111, 419], [111, 483], [111, 579], [111, 692], [111, 758], [111, 955], [111, 993], [111, 2132], [111, 2134], [111, 2211], [111, 2242], [111, 2245], [111, 2249], [111, 2369], [111, 2614], [112, 113], [112, 115], [112, 118], [112, 172], [112, 226], [112, 386], [112, 411], [112, 447], [112, 460], [112, 673], [112, 674], [112, 688], [112, 901], [112, 2133], [112, 2167], [112, 2243], [112, 2245], [112, 2250], [112, 2541], [112, 2818], [113, 119], [113, 121], [113, 134], [113, 139], [113, 213], [113, 214], [113, 241], [113, 244], [113, 386], [113, 437], [113, 575], [113, 627], [113, 713], [113, 1029], [113, 2144], [113, 2167], [113, 2244], [113, 2249], [113, 2251], [113, 2369], [113, 2371], [114, 115], [114, 118], [114, 190], [114, 316], [114, 332], [114, 333], [114, 601], [114, 603], [114, 654], [114, 682], [114, 791], [114, 2245], [115, 116], [115, 194], [115, 195], [115, 341], [115, 435], [115, 488], [115, 654], [115, 683], [115, 790], [115, 809], [115, 833], [115, 909], [115, 1010], [115, 2133], [115, 2146], [115, 2246], [115, 2249], [115, 2321], [115, 2447], [115, 2463], [115, 2464], [115, 2732], [115, 2734], [115, 2785], [115, 2813], [115, 2922], [116, 117], [116, 172], [116, 186], [116, 190], [116, 191], [116, 215], [116, 243], [116, 266], [116, 335], [116, 358], [116, 360], [116, 456], [116, 697], [116, 980], [116, 1007], [116, 2245], [116, 2247], [116, 2316], [116, 2322], [116, 2396], [116, 2429], [116, 2462], [116, 2469], [116, 2586], [116, 2795], [116, 2830], [117, 138], [117, 192], [117, 199], [117, 202], [117, 249], [117, 337], [117, 340], [117, 346], [117, 358], [117, 388], [117, 524], [117, 618], [117, 646], [117, 651], [117, 664], [117, 671], [117, 683], [117, 714], [117, 762], [117, 827], [117, 976], [117, 2145], [117, 2248], [117, 2278], [117, 2321], [117, 2323], [117, 2469], [117, 2518], [117, 2578], [117, 2661], [117, 2716], [117, 2733], [117, 2776], [117, 2794], [117, 2800], [117, 2804], [117, 2813], [118, 119], [118, 410], [118, 687], [118, 2249], [119, 120], [119, 238], [119, 240], [119, 2250], [120, 121], [120, 254], [120, 286], [120, 350], [120, 381], [120, 415], [120, 457], [120, 500], [120, 506], [120, 542], [120, 560], [120, 562], [120, 687], [120, 2156], [120, 2251], [121, 255], [121, 277], [121, 282], [121, 286], [121, 320], [121, 343], [121, 346], [121, 351], [121, 356], [121, 383], [121, 416], [121, 458], [121, 501], [121, 517], [121, 575], [121, 628], [121, 688], [121, 717], [121, 841], [121, 842], [121, 888], [121, 2145], [121, 2182], [121, 2250], [121, 2252], [121, 2385], [121, 2417], [121, 2481], [121, 2512], [121, 2546], [121, 2588], [121, 2631], [121, 2637], [121, 2673], [121, 2691], [121, 2693], [121, 2818], [122, 123], [122, 126], [122, 129], [122, 133], [122, 134], [122, 136], [122, 139], [122, 242], [122, 443], [122, 450], [122, 451], [122, 624], [122, 672], [122, 765], [122, 2253], [122, 2582], [123, 124], [123, 129], [123, 148], [123, 256], [123, 309], [123, 310], [123, 378], [123, 706], [123, 956], [123, 2254], [123, 2409], [123, 2440], [124, 125], [124, 136], [124, 174], [124, 706], [124, 1014], [124, 1016], [124, 2255], [124, 2342], [124, 2528], [124, 2650], [125, 146], [125, 205], [125, 237], [125, 335], [125, 358], [125, 378], [125, 391], [125, 397], [125, 449], [125, 628], [125, 1032], [125, 1060], [125, 1064], [125, 2150], [125, 2155], [125, 2256], [125, 2340], [125, 2409], [125, 2436], [125, 2446], [125, 2469], [125, 2529], [125, 2596], [125, 2629], [125, 2637], [126, 127], [126, 139], [126, 235], [126, 283], [126, 334], [126, 460], [126, 461], [126, 1035], [126, 2257], [126, 2461], [126, 2591], [127, 128], [127, 148], [127, 151], [127, 235], [127, 236], [127, 279], [127, 446], [127, 1034], [127, 2258], [127, 2378], [127, 2410], [128, 149], [128, 168], [128, 237], [128, 279], [128, 280], [128, 348], [128, 378], [128, 411], [128, 461], [128, 720], [128, 838], [128, 1011], [128, 1034], [128, 1063], [128, 2138], [128, 2144], [128, 2258], [128, 2259], [128, 2264], [128, 2409], [128, 2411], [128, 2436], [128, 2541], [128, 2591], [128, 2764], [128, 2812], [129, 130], [129, 203], [129, 316], [129, 368], [129, 369], [129, 377], [129, 434], [129, 452], [129, 453], [129, 498], [129, 546], [129, 547], [129, 2195], [129, 2260], [129, 2583], [130, 131], [130, 204], [130, 219], [130, 428], [130, 435], [130, 453], [130, 709], [130, 765], [130, 906], [130, 956], [130, 999], [130, 1010], [130, 2230], [130, 2253], [130, 2254], [130, 2261], [130, 2334], [130, 2447], [130, 2499], [130, 2500], [130, 2508], [130, 2565], [130, 2583], [130, 2584], [130, 2629], [130, 2677], [130, 2678], [131, 132], [131, 219], [131, 224], [131, 369], [131, 374], [131, 458], [131, 628], [131, 693], [131, 694], [131, 698], [131, 794], [131, 806], [131, 867], [131, 1021], [131, 2135], [131, 2208], [131, 2260], [131, 2262], [131, 2373], [131, 2407], [131, 2460], [131, 2499], [131, 2501], [131, 2505], [131, 2588], [131, 2612], [131, 2630], [131, 2637], [131, 2835], [132, 321], [132, 379], [132, 396], [132, 413], [132, 499], [132, 571], [132, 746], [132, 794], [132, 837], [132, 2145], [132, 2263], [132, 2298], [132, 2412], [132, 2439], [132, 2460], [132, 2500], [132, 2527], [132, 2540], [132, 2595], [132, 2628], [132, 2629], [133, 134], [133, 240], [133, 279], [133, 380], [133, 381], [133, 434], [133, 492], [133, 495], [133, 623], [133, 633], [133, 1063], [133, 2264], [134, 135], [134, 148], [134, 191], [134, 270], [134, 316], [134, 346], [134, 360], [134, 713], [134, 958], [134, 2250], [134, 2265], [134, 2733], [134, 2796], [135, 158], [135, 231], [135, 241], [135, 271], [135, 493], [135, 505], [135, 604], [135, 665], [135, 671], [135, 759], [135, 1006], [135, 2132], [135, 2192], [135, 2266], [135, 2288], [135, 2371], [135, 2636], [135, 2733], [135, 2797], [135, 2800], [136, 137], [136, 138], [136, 334], [136, 346], [136, 394], [136, 507], [136, 517], [136, 817], [136, 923], [136, 2182], [136, 2267], [136, 2461], [137, 138], [137, 305], [137, 335], [137, 387], [137, 453], [137, 508], [137, 2268], [137, 2583], [138, 249], [138, 336], [138, 405], [138, 919], [138, 980], [138, 2183], [138, 2269], [138, 2716], [138, 2795], [138, 2837], [139, 140], [139, 417], [139, 431], [139, 517], [139, 620], [139, 627], [139, 816], [139, 2182], [139, 2250], [139, 2270], [139, 2750], [140, 141], [140, 227], [140, 353], [140, 423], [140, 450], [140, 461], [140, 816], [140, 995], [140, 2270], [140, 2271], [140, 2582], [140, 2591], [140, 2805], [141, 142], [141, 234], [141, 307], [141, 527], [141, 528], [141, 996], [141, 2164], [141, 2178], [141, 2272], [141, 2666], [141, 2669], [141, 2679], [142, 156], [142, 249], [142, 364], [142, 511], [142, 583], [142, 844], [142, 861], [142, 883], [142, 2145], [142, 2175], [142, 2179], [142, 2183], [142, 2273], [142, 2530], [142, 2627], [142, 2642], [142, 2685], [142, 2716], [142, 2717], [142, 2723], [143, 144], [143, 147], [143, 151], [143, 154], [143, 247], [143, 349], [143, 656], [143, 663], [143, 2274], [144, 145], [144, 202], [144, 203], [144, 236], [144, 349], [144, 429], [144, 468], [144, 618], [144, 657], [144, 1003], [144, 1026], [144, 2132], [144, 2275], [144, 2278], [144, 2282], [144, 2285], [144, 2378], [144, 2480], [144, 2787], [144, 2794], [145, 146], [145, 203], [145, 217], [145, 465], [145, 650], [145, 2274], [145, 2276], [145, 2596], [146, 187], [146, 221], [146, 259], [146, 354], [146, 366], [146, 392], [146, 465], [146, 467], [146, 468], [146, 469], [146, 735], [146, 849], [146, 950], [146, 2155], [146, 2159], [146, 2276], [146, 2277], [146, 2328], [146, 2347], [146, 2478], [146, 2480], [146, 2528], [146, 2597], [146, 2791], [147, 148], [147, 197], [147, 202], [147, 242], [147, 294], [147, 602], [147, 664], [147, 666], [147, 681], [147, 2278], [148, 149], [148, 151], [148, 158], [148, 285], [148, 294], [148, 316], [148, 447], [148, 769], [148, 1016], [148, 2279], [148, 2288], [149, 150], [149, 163], [149, 282], [149, 295], [149, 303], [149, 310], [149, 313], [149, 348], [149, 360], [149, 383], [149, 769], [149, 2279], [149, 2280], [149, 2296], [149, 2309], [149, 2410], [149, 2440], [149, 2512], [150, 158], [150, 165], [150, 231], [150, 383], [150, 474], [150, 514], [150, 768], [150, 773], [150, 1011], [150, 2145], [150, 2236], [150, 2281], [150, 2288], [150, 2289], [150, 2293], [150, 2297], [150, 2310], [150, 2419], [150, 2512], [150, 2636], [150, 2644], [150, 2746], [150, 2764], [151, 152], [151, 154], [151, 217], [151, 283], [151, 348], [151, 1026], [151, 2282], [152, 153], [152, 446], [152, 447], [152, 649], [152, 663], [152, 705], [152, 2283], [153, 168], [153, 337], [153, 534], [153, 578], [153, 618], [153, 889], [153, 910], [153, 1002], [153, 1026], [153, 1033], [153, 2138], [153, 2153], [153, 2186], [153, 2282], [153, 2284], [153, 2577], [153, 2578], [153, 2780], [153, 2794], [153, 2836], [154, 155], [154, 294], [154, 417], [154, 429], [154, 434], [154, 623], [154, 2285], [155, 156], [155, 295], [155, 348], [155, 349], [155, 423], [155, 429], [155, 2285], [155, 2286], [156, 282], [156, 305], [156, 398], [156, 423], [156, 428], [156, 434], [156, 453], [156, 495], [156, 571], [156, 844], [156, 2178], [156, 2287], [156, 2565], [156, 2583], [156, 2628], [157, 158], [157, 161], [157, 165], [157, 256], [157, 384], [157, 665], [157, 2288], [158, 159], [158, 165], [158, 257], [158, 285], [158, 768], [158, 972], [158, 973], [158, 992], [158, 1059], [158, 2132], [158, 2289], [158, 2292], [158, 2296], [158, 2387], [158, 2515], [158, 2796], [159, 160], [159, 384], [159, 391], [159, 442], [159, 582], [159, 594], [159, 615], [159, 671], [159, 788], [159, 789], [159, 954], [159, 984], [159, 1016], [159, 1042], [159, 1059], [159, 2150], [159, 2288], [159, 2290], [159, 2343], [159, 2379], [159, 2422], [159, 2516], [159, 2643], [159, 2718], [159, 2724], [159, 2746], [159, 2765], [159, 2800], [160, 411], [160, 436], [160, 516], [160, 582], [160, 622], [160, 634], [160, 757], [160, 777], [160, 795], [160, 854], [160, 892], [160, 894], [160, 895], [160, 899], [160, 901], [160, 904], [160, 929], [160, 975], [160, 1011], [160, 1048], [160, 2174], [160, 2180], [160, 2200], [160, 2209], [160, 2291], [160, 2344], [160, 2515], [160, 2517], [160, 2541], [160, 2610], [160, 2646], [160, 2718], [160, 2763], [160, 2764], [160, 2798], [160, 2806], [160, 2909], [161, 162], [161, 235], [161, 252], [161, 291], [161, 323], [161, 692], [161, 732], [161, 972], [161, 2143], [161, 2292], [161, 2421], [162, 163], [162, 165], [162, 223], [162, 244], [162, 272], [162, 289], [162, 327], [162, 339], [162, 372], [162, 693], [162, 696], [162, 722], [162, 773], [162, 825], [162, 877], [162, 972], [162, 1013], [162, 2142], [162, 2144], [162, 2292], [162, 2293], [162, 2331], [162, 2453], [162, 2501], [162, 2502], [162, 2585], [162, 2612], [162, 2857], [163, 164], [163, 272], [163, 313], [163, 724], [163, 947], [163, 2143], [163, 2294], [163, 2309], [163, 2614], [163, 2854], [163, 2857], [164, 177], [164, 215], [164, 483], [164, 580], [164, 693], [164, 993], [164, 996], [164, 997], [164, 2133], [164, 2135], [164, 2145], [164, 2164], [164, 2242], [164, 2295], [164, 2429], [164, 2612], [164, 2613], [164, 2622], [164, 2783], [164, 2785], [165, 166], [165, 179], [165, 288], [165, 381], [165, 505], [165, 513], [165, 615], [165, 633], [165, 2143], [165, 2156], [165, 2288], [165, 2296], [166, 167], [166, 431], [166, 491], [166, 693], [166, 778], [166, 2297], [166, 2612], [167, 431], [167, 499], [167, 592], [167, 641], [167, 894], [167, 2155], [167, 2209], [167, 2298], [167, 2650], [167, 2775], [168, 169], [168, 170], [168, 175], [168, 182], [168, 225], [168, 234], [168, 266], [168, 313], [168, 411], [168, 471], [168, 677], [168, 752], [168, 946], [168, 975], [168, 1008], [168, 2132], [168, 2139], [168, 2160], [168, 2164], [168, 2196], [168, 2215], [168, 2224], [168, 2283], [168, 2299], [168, 2303], [168, 2306], [168, 2309], [168, 2312], [168, 2396], [168, 2410], [168, 2541], [168, 2549], [168, 2679], [168, 2708], [168, 2727], [168, 2763], [168, 2849], [169, 170], [169, 177], [169, 183], [169, 393], [169, 637], [169, 709], [169, 714], [169, 821], [169, 953], [169, 980], [169, 985], [169, 993], [169, 1000], [169, 1002], [169, 2138], [169, 2196], [169, 2199], [169, 2223], [169, 2225], [169, 2230], [169, 2242], [169, 2300], [169, 2308], [169, 2325], [169, 2550], [169, 2780], [169, 2804], [170, 171], [170, 176], [170, 547], [170, 740], [170, 900], [170, 925], [170, 2138], [170, 2195], [170, 2197], [170, 2224], [170, 2301], [170, 2307], [170, 2651], [170, 2738], [170, 2836], [171, 182], [171, 183], [171, 205], [171, 208], [171, 445], [171, 622], [171, 832], [171, 845], [171, 862], [171, 913], [171, 987], [171, 1011], [171, 1033], [171, 1051], [171, 2145], [171, 2186], [171, 2196], [171, 2302], [171, 2312], [171, 2325], [171, 2339], [171, 2340], [171, 2448], [171, 2575], [171, 2577], [171, 2581], [171, 2610], [171, 2629], [171, 2764], [171, 2870], [171, 2880], [171, 2964], [171, 3058], [172, 173], [172, 178], [172, 180], [172, 225], [172, 242], [172, 306], [172, 360], [172, 447], [172, 675], [172, 711], [172, 2249], [172, 2303], [172, 2321], [173, 174], [173, 358], [173, 383], [173, 385], [173, 386], [173, 505], [173, 666], [173, 670], [173, 712], [173, 717], [173, 1006], [173, 1017], [173, 2157], [173, 2167], [173, 2304], [173, 2342], [173, 2469], [173, 2512], [173, 2797], [174, 201], [174, 211], [174, 291], [174, 358], [174, 551], [174, 640], [174, 699], [174, 702], [174, 708], [174, 716], [174, 1012], [174, 1042], [174, 2132], [174, 2229], [174, 2232], [174, 2305], [174, 2343], [174, 2421], [174, 2469], [174, 2552], [175, 176], [175, 177], [175, 280], [175, 281], [175, 485], [175, 891], [175, 2138], [175, 2156], [175, 2306], [175, 2593], [176, 177], [176, 208], [176, 553], [176, 2196], [176, 2307], [177, 222], [177, 306], [177, 307], [177, 456], [177, 508], [177, 996], [177, 2224], [177, 2308], [177, 2586], [177, 2614], [177, 2669], [178, 179], [178, 313], [178, 322], [178, 520], [178, 731], [178, 2309], [179, 180], [179, 230], [179, 613], [179, 632], [179, 696], [179, 2310], [179, 2635], [180, 231], [180, 307], [180, 632], [180, 697], [180, 901], [180, 1059], [180, 2311], [180, 2636], [180, 2651], [180, 2669], [180, 2830], [181, 182], [181, 194], [181, 348], [181, 350], [181, 356], [181, 360], [181, 494], [181, 574], [181, 575], [181, 719], [181, 731], [181, 2312], [181, 2634], [181, 2732], [182, 183], [182, 326], [182, 341], [182, 351], [182, 855], [182, 1008], [182, 1009], [182, 1022], [182, 1046], [182, 2138], [182, 2139], [182, 2146], [182, 2197], [182, 2313], [182, 2325], [182, 2479], [182, 2481], [182, 2487], [182, 2491], [182, 2625], [182, 2705], [182, 2706], [182, 2850], [182, 2862], [183, 184], [183, 194], [183, 714], [183, 728], [183, 775], [183, 902], [183, 968], [183, 1027], [183, 1066], [183, 2197], [183, 2217], [183, 2224], [183, 2246], [183, 2312], [183, 2314], [183, 2324], [183, 2326], [183, 2352], [183, 2732], [183, 2735], [183, 2736], [183, 2804], [183, 2858], [183, 3196], [184, 186], [184, 193], [184, 326], [184, 390], [184, 728], [184, 770], [184, 804], [184, 822], [184, 977], [184, 2145], [184, 2315], [184, 2316], [184, 2317], [184, 2325], [184, 2394], [184, 2476], [184, 2639], [184, 2777], [184, 2858], [184, 2862], [184, 2940], [185, 186], [185, 190], [185, 193], [185, 197], [185, 262], [185, 270], [185, 635], [185, 2316], [186, 187], [186, 188], [186, 193], [186, 228], [186, 243], [186, 637], [186, 675], [186, 678], [186, 763], [186, 770], [186, 931], [186, 2132], [186, 2223], [186, 2317], [186, 2321], [186, 2324], [186, 2328], [186, 2393], [186, 2401], [186, 2766], [187, 188], [187, 320], [187, 456], [187, 552], [187, 558], [187, 628], [187, 1065], [187, 2181], [187, 2316], [187, 2318], [187, 2393], [187, 2586], [187, 2596], [187, 2637], [187, 2691], [188, 189], [188, 215], [188, 259], [188, 262], [188, 263], [188, 412], [188, 427], [188, 435], [188, 559], [188, 800], [188, 801], [188, 885], [188, 948], [188, 967], [188, 999], [188, 1055], [188, 2316], [188, 2319], [188, 2392], [188, 2394], [188, 2429], [188, 2445], [188, 2447], [188, 2459], [188, 2478], [188, 2548], [188, 2568], [188, 2613], [188, 2757], [188, 2758], [188, 2760], [188, 2832], [189, 314], [189, 343], [189, 544], [189, 583], [189, 609], [189, 736], [189, 737], [189, 761], [189, 780], [189, 810], [189, 836], [189, 851], [189, 959], [189, 987], [189, 1064], [189, 2145], [189, 2155], [189, 2219], [189, 2320], [189, 2393], [189, 2446], [189, 2448], [189, 2526], [189, 2613], [189, 2673], [189, 2675], [189, 2723], [189, 2739], [189, 2748], [189, 2767], [189, 2866], [189, 2942], [189, 3021], [189, 3043], [190, 191], [190, 265], [190, 298], [190, 331], [190, 338], [190, 455], [190, 664], [190, 699], [190, 2321], [191, 192], [191, 214], [191, 2321], [191, 2322], [191, 2369], [192, 198], [192, 248], [192, 523], [192, 569], [192, 635], [192, 664], [192, 672], [192, 762], [192, 2132], [192, 2323], [192, 2795], [193, 194], [193, 263], [193, 345], [193, 508], [193, 646], [193, 727], [193, 731], [193, 809], [193, 2156], [193, 2316], [193, 2324], [193, 2732], [193, 2776], [194, 195], [194, 221], [194, 601], [194, 604], [194, 605], [194, 673], [194, 727], [194, 1065], [194, 2192], [194, 2193], [194, 2245], [194, 2325], [194, 2347], [194, 2733], [194, 2734], [195, 196], [195, 223], [195, 263], [195, 367], [195, 419], [195, 425], [195, 456], [195, 603], [195, 626], [195, 674], [195, 872], [195, 913], [195, 967], [195, 968], [195, 1053], [195, 2144], [195, 2172], [195, 2245], [195, 2326], [195, 2555], [195, 2585], [195, 2586], [195, 2732], [195, 2757], [195, 2880], [196, 220], [196, 251], [196, 480], [196, 790], [196, 857], [196, 858], [196, 869], [196, 872], [196, 2145], [196, 2171], [196, 2173], [196, 2327], [196, 2504], [196, 2641], [196, 2671], [196, 2734], [196, 2773], [196, 2922], [197, 198], [197, 347], [197, 465], [197, 681], [197, 2328], [198, 199], [198, 290], [198, 660], [198, 691], [198, 981], [198, 2329], [199, 371], [199, 404], [199, 405], [199, 614], [199, 618], [199, 726], [199, 981], [199, 2329], [199, 2330], [199, 2593], [199, 2794], [199, 2795], [200, 201], [200, 242], [200, 340], [200, 699], [200, 2331], [200, 2733], [201, 202], [201, 203], [201, 340], [201, 370], [201, 378], [201, 2278], [201, 2332], [201, 2342], [201, 2409], [202, 224], [202, 430], [202, 769], [202, 838], [202, 991], [202, 1006], [202, 1065], [202, 2274], [202, 2279], [202, 2328], [202, 2333], [202, 2373], [202, 2425], [202, 2733], [202, 2795], [202, 2797], [202, 2812], [203, 204], [203, 209], [203, 274], [203, 370], [203, 466], [203, 467], [203, 702], [203, 732], [203, 2274], [203, 2334], [203, 2597], [204, 205], [204, 467], [204, 709], [204, 733], [204, 734], [204, 748], [204, 985], [204, 991], [204, 1003], [204, 1019], [204, 2142], [204, 2177], [204, 2230], [204, 2260], [204, 2275], [204, 2276], [204, 2332], [204, 2335], [204, 2340], [204, 2405], [204, 2501], [204, 2597], [204, 2598], [204, 2833], [204, 2863], [205, 207], [205, 209], [205, 249], [205, 250], [205, 372], [205, 391], [205, 846], [205, 864], [205, 921], [205, 1051], [205, 2142], [205, 2150], [205, 2197], [205, 2334], [205, 2336], [205, 2338], [205, 2341], [205, 2471], [205, 2502], [205, 2528], [205, 2530], [205, 2670], [205, 2716], [205, 2749], [205, 2781], [206, 207], [206, 308], [206, 350], [206, 467], [206, 719], [206, 723], [206, 735], [206, 2142], [206, 2156], [206, 2337], [206, 2597], [207, 208], [207, 209], [207, 366], [207, 467], [207, 2338], [207, 2340], [207, 2597], [208, 304], [208, 367], [208, 480], [208, 720], [208, 725], [208, 844], [208, 2197], [208, 2339], [209, 210], [209, 340], [209, 371], [209, 397], [209, 399], [209, 539], [209, 585], [209, 618], [209, 650], [209, 2340], [209, 2794], [210, 454], [210, 476], [210, 617], [210, 644], [210, 648], [210, 732], [210, 749], [210, 2341], [210, 2694], [210, 2793], [211, 212], [211, 290], [211, 338], [211, 421], [211, 2342], [212, 213], [212, 244], [212, 254], [212, 283], [212, 312], [212, 323], [212, 331], [212, 381], [212, 384], [212, 421], [212, 1042], [212, 2343], [213, 214], [213, 283], [213, 416], [213, 420], [213, 630], [213, 634], [213, 667], [213, 711], [213, 1048], [213, 2250], [213, 2344], [213, 2369], [213, 2546], [214, 215], [214, 238], [214, 255], [214, 298], [214, 300], [214, 668], [214, 781], [214, 1050], [214, 2132], [214, 2133], [214, 2250], [214, 2345], [214, 2370], [214, 2385], [214, 2429], [214, 2449], [214, 2852], [215, 246], [215, 298], [215, 456], [215, 559], [215, 711], [215, 781], [215, 949], [215, 995], [215, 1044], [215, 1049], [215, 2145], [215, 2321], [215, 2345], [215, 2346], [215, 2369], [215, 2376], [215, 2393], [215, 2430], [215, 2586], [215, 2614], [215, 2831], [216, 217], [216, 221], [216, 387], [216, 424], [216, 465], [216, 481], [216, 601], [216, 602], [216, 2347], [217, 218], [217, 221], [217, 649], [217, 681], [217, 692], [217, 1025], [217, 2347], [217, 2348], [218, 219], [218, 235], [218, 424], [218, 425], [218, 626], [218, 655], [218, 2349], [218, 2499], [218, 2555], [219, 220], [219, 257], [219, 368], [219, 373], [219, 571], [219, 655], [219, 818], [219, 820], [219, 824], [219, 873], [219, 907], [219, 2132], [219, 2171], [219, 2207], [219, 2260], [219, 2350], [219, 2387], [219, 2500], [219, 2504], [219, 2507], [219, 2628], [220, 373], [220, 388], [220, 425], [220, 426], [220, 441], [220, 514], [220, 818], [220, 870], [220, 2145], [220, 2172], [220, 2350], [220, 2351], [220, 2499], [220, 2505], [220, 2518], [220, 2555], [220, 2571], [220, 2584], [220, 2644], [220, 2722], [221, 222], [221, 316], [221, 335], [221, 348], [221, 388], [221, 425], [221, 606], [221, 684], [221, 693], [221, 739], [221, 775], [221, 778], [221, 821], [221, 950], [221, 1025], [221, 2132], [221, 2233], [221, 2348], [221, 2352], [221, 2518], [221, 2555], [221, 2596], [221, 2612], [221, 2732], [221, 2733], [222, 223], [222, 366], [222, 388], [222, 426], [222, 510], [222, 526], [222, 649], [222, 652], [222, 653], [222, 791], [222, 1002], [222, 1023], [222, 2347], [222, 2353], [222, 2518], [222, 2585], [222, 2652], [222, 2780], [223, 252], [223, 328], [223, 329], [223, 366], [223, 399], [223, 454], [223, 456], [223, 704], [223, 728], [223, 743], [223, 774], [223, 822], [223, 825], [223, 920], [223, 950], [223, 962], [223, 1066], [223, 2143], [223, 2156], [223, 2341], [223, 2354], [223, 2383], [223, 2586], [223, 2734], [223, 2858], [224, 225], [224, 242], [224, 301], [224, 327], [224, 383], [224, 432], [224, 516], [224, 524], [224, 556], [224, 567], [224, 568], [224, 628], [224, 697], [224, 765], [224, 771], [224, 929], [224, 954], [224, 2132], [224, 2154], [224, 2189], [224, 2253], [224, 2278], [224, 2303], [224, 2331], [224, 2355], [224, 2374], [224, 2432], [224, 2453], [224, 2475], [224, 2500], [224, 2512], [224, 2525], [224, 2562], [224, 2637], [224, 2646], [224, 2661], [224, 2686], [224, 2806], [224, 2830], [225, 226], [225, 297], [225, 313], [225, 337], [225, 929], [225, 989], [225, 1007], [225, 1017], [225, 1046], [225, 2135], [225, 2138], [225, 2231], [225, 2243], [225, 2247], [225, 2304], [225, 2309], [225, 2311], [225, 2356], [225, 2373], [225, 2437], [225, 2491], [225, 2578], [225, 2806], [225, 2842], [226, 227], [226, 337], [226, 461], [226, 567], [226, 714], [226, 813], [226, 854], [226, 993], [226, 1029], [226, 1030], [226, 2189], [226, 2242], [226, 2244], [226, 2246], [226, 2249], [226, 2303], [226, 2357], [226, 2517], [226, 2542], [226, 2578], [226, 2591], [226, 2804], [226, 2805], [226, 2819], [226, 3032], [227, 297], [227, 620], [227, 674], [227, 714], [227, 750], [227, 968], [227, 988], [227, 1000], [227, 2145], [227, 2231], [227, 2243], [227, 2326], [227, 2358], [227, 2550], [227, 2611], [227, 2704], [227, 2750], [227, 2751], [227, 2804], [227, 3005], [228, 229], [228, 244], [228, 270], [228, 272], [228, 320], [228, 324], [228, 326], [228, 402], [228, 731], [228, 785], [228, 808], [228, 889], [228, 958], [228, 982], [228, 2132], [228, 2265], [228, 2316], [228, 2359], [228, 2402], [228, 2511], [228, 2568], [228, 2691], [228, 2857], [228, 2862], [229, 230], [229, 271], [229, 272], [229, 342], [229, 487], [229, 582], [229, 661], [229, 759], [229, 763], [229, 798], [229, 877], [229, 973], [229, 2201], [229, 2221], [229, 2266], [229, 2360], [229, 2361], [229, 2401], [229, 2569], [229, 2624], [229, 2635], [229, 2692], [229, 2718], [229, 2747], [229, 2766], [229, 2822], [229, 2857], [230, 231], [230, 271], [230, 494], [230, 504], [230, 589], [230, 641], [230, 643], [230, 646], [230, 696], [230, 730], [230, 731], [230, 827], [230, 1018], [230, 2361], [230, 2402], [230, 2594], [230, 2634], [230, 2636], [230, 2719], [230, 2775], [230, 2776], [231, 251], [231, 360], [231, 494], [231, 505], [231, 697], [231, 759], [231, 898], [231, 905], [231, 1017], [231, 1058], [231, 2145], [231, 2266], [231, 2296], [231, 2304], [231, 2362], [231, 2634], [231, 2635], [231, 2671], [231, 2796], [231, 2801], [231, 2830], [232, 233], [232, 493], [232, 570], [232, 678], [232, 679], [232, 691], [232, 702], [232, 881], [232, 2132], [232, 2186], [232, 2363], [233, 234], [233, 545], [233, 548], [233, 690], [233, 702], [233, 708], [233, 2229], [233, 2364], [233, 2679], [234, 296], [234, 297], [234, 548], [234, 550], [234, 552], [234, 874], [234, 2138], [234, 2181], [234, 2186], [234, 2231], [234, 2364], [234, 2365], [234, 2549], [234, 2680], [235, 333], [235, 624], [235, 626], [235, 630], [235, 963], [235, 2132], [235, 2144], [235, 2366], [236, 237], [236, 247], [236, 305], [236, 334], [236, 428], [236, 446], [236, 927], [236, 984], [236, 985], [236, 1016], [236, 1031], [236, 2132], [236, 2274], [236, 2367], [236, 2379], [236, 2436], [236, 2461], [236, 2565], [237, 269], [237, 305], [237, 468], [237, 720], [237, 880], [237, 1031], [237, 1064], [237, 2145], [237, 2268], [237, 2287], [237, 2367], [237, 2368], [237, 2378], [237, 2380], [237, 2399], [237, 2410], [237, 2437], [237, 2446], [237, 2480], [237, 2486], [237, 2528], [238, 239], [238, 254], [238, 298], [238, 318], [238, 721], [238, 2369], [239, 240], [239, 639], [239, 700], [239, 703], [239, 721], [239, 1050], [239, 2370], [240, 241], [240, 417], [240, 562], [240, 623], [240, 665], [240, 667], [240, 2371], [241, 427], [241, 438], [241, 795], [241, 877], [241, 901], [241, 1050], [241, 1063], [241, 2144], [241, 2250], [241, 2264], [241, 2370], [241, 2372], [241, 2548], [241, 2693], [241, 2754], [241, 2796], [241, 2798], [242, 243], [242, 301], [242, 322], [242, 344], [242, 369], [242, 381], [242, 394], [242, 431], [242, 506], [242, 515], [242, 530], [242, 555], [242, 675], [242, 699], [242, 2373], [243, 244], [243, 252], [243, 344], [243, 558], [243, 559], [243, 627], [243, 651], [243, 771], [243, 822], [243, 2316], [243, 2321], [243, 2374], [244, 245], [244, 324], [244, 338], [244, 352], [244, 381], [244, 385], [244, 447], [244, 786], [244, 877], [244, 2143], [244, 2250], [244, 2375], [244, 2401], [245, 246], [245, 298], [245, 318], [245, 320], [245, 338], [245, 355], [245, 398], [245, 421], [245, 475], [245, 558], [245, 668], [245, 722], [245, 2156], [245, 2376], [245, 2449], [245, 2691], [246, 358], [246, 408], [246, 640], [246, 668], [246, 712], [246, 741], [246, 786], [246, 1052], [246, 2145], [246, 2375], [246, 2377], [246, 2429], [246, 2449], [246, 2451], [246, 2469], [246, 2486], [246, 2529], [246, 2552], [246, 2606], [246, 2689], [246, 2799], [246, 2853], [247, 248], [247, 305], [247, 330], [247, 434], [247, 2378], [248, 249], [248, 384], [248, 511], [248, 585], [248, 984], [248, 2175], [248, 2379], [248, 2716], [249, 250], [249, 266], [249, 305], [249, 307], [249, 470], [249, 581], [249, 585], [249, 614], [249, 615], [249, 633], [249, 641], [249, 827], [249, 880], [249, 897], [249, 984], [249, 991], [249, 2178], [249, 2340], [249, 2379], [249, 2380], [249, 2396], [249, 2669], [249, 2707], [249, 2708], [249, 2717], [249, 2775], [249, 2795], [250, 251], [250, 351], [250, 501], [250, 720], [250, 1056], [250, 2340], [250, 2381], [250, 2481], [250, 2631], [250, 2671], [250, 2716], [251, 307], [251, 540], [251, 604], [251, 643], [251, 710], [251, 805], [251, 843], [251, 902], [251, 2145], [251, 2172], [251, 2192], [251, 2194], [251, 2382], [251, 2636], [251, 2669], [251, 2670], [251, 2706], [251, 2735], [251, 2844], [251, 2848], [252, 253], [252, 454], [252, 626], [252, 649], [252, 650], [252, 678], [252, 962], [252, 2383], [252, 2585], [253, 254], [253, 323], [253, 328], [253, 541], [253, 542], [253, 611], [253, 636], [253, 878], [253, 2384], [254, 255], [254, 331], [254, 415], [254, 703], [254, 704], [254, 2385], [255, 391], [255, 416], [255, 698], [255, 716], [255, 820], [255, 878], [255, 916], [255, 1042], [255, 1060], [255, 2150], [255, 2251], [255, 2343], [255, 2369], [255, 2384], [255, 2386], [255, 2462], [255, 2546], [255, 2834], [255, 2835], [256, 257], [256, 261], [256, 347], [256, 368], [256, 513], [256, 532], [256, 2387], [257, 258], [257, 259], [257, 284], [257, 287], [257, 293], [257, 513], [257, 514], [257, 956], [257, 957], [257, 1037], [257, 2132], [257, 2254], [257, 2288], [257, 2388], [257, 2392], [257, 2478], [257, 2499], [257, 2644], [257, 2663], [258, 259], [258, 288], [258, 386], [258, 412], [258, 474], [258, 571], [258, 589], [258, 892], [258, 2167], [258, 2236], [258, 2387], [258, 2389], [258, 2478], [258, 2628], [258, 2719], [258, 2760], [259, 260], [259, 289], [259, 341], [259, 347], [259, 430], [259, 661], [259, 787], [259, 828], [259, 834], [259, 886], [259, 973], [259, 1022], [259, 1044], [259, 2146], [259, 2148], [259, 2328], [259, 2387], [259, 2390], [259, 2393], [259, 2425], [259, 2479], [259, 2501], [259, 2507], [259, 2509], [259, 2579], [259, 2596], [259, 2603], [259, 2604], [259, 2822], [260, 289], [260, 293], [260, 366], [260, 372], [260, 448], [260, 534], [260, 589], [260, 801], [260, 835], [260, 839], [260, 1054], [260, 1058], [260, 2145], [260, 2153], [260, 2391], [260, 2433], [260, 2478], [260, 2502], [260, 2580], [260, 2663], [260, 2665], [260, 2704], [260, 2719], [261, 262], [261, 275], [261, 608], [261, 629], [261, 721], [261, 2392], [262, 263], [262, 298], [262, 314], [262, 316], [262, 328], [262, 347], [262, 417], [262, 437], [262, 482], [262, 626], [262, 627], [262, 629], [262, 701], [262, 2393], [262, 2726], [263, 264], [263, 282], [263, 423], [263, 544], [263, 809], [263, 888], [263, 2393], [263, 2394], [263, 2729], [263, 2734], [264, 273], [264, 525], [264, 598], [264, 646], [264, 701], [264, 802], [264, 807], [264, 885], [264, 888], [264, 913], [264, 2145], [264, 2395], [264, 2534], [264, 2615], [264, 2726], [264, 2728], [264, 2730], [264, 2776], [264, 2832], [264, 2880], [265, 266], [265, 350], [265, 414], [265, 415], [265, 585], [265, 723], [265, 2396], [266, 267], [266, 312], [266, 351], [266, 416], [266, 470], [266, 495], [266, 658], [266, 700], [266, 724], [266, 782], [266, 783], [266, 833], [266, 937], [266, 980], [266, 2132], [266, 2138], [266, 2238], [266, 2321], [266, 2397], [266, 2481], [266, 2545], [266, 2546], [266, 2716], [266, 2854], [267, 268], [267, 294], [267, 310], [267, 396], [267, 411], [267, 414], [267, 416], [267, 431], [267, 433], [267, 434], [267, 457], [267, 460], [267, 516], [267, 1060], [267, 2396], [267, 2398], [267, 2440], [267, 2540], [267, 2541], [267, 2546], [267, 2646], [268, 269], [268, 305], [268, 330], [268, 334], [268, 395], [268, 460], [268, 484], [268, 688], [268, 707], [268, 897], [268, 2156], [268, 2399], [268, 2461], [268, 2545], [268, 2818], [269, 334], [269, 433], [269, 461], [269, 525], [269, 812], [269, 813], [269, 836], [269, 923], [269, 2145], [269, 2398], [269, 2400], [269, 2436], [269, 2461], [269, 2465], [269, 2526], [269, 2591], [269, 2615], [269, 2819], [269, 2838], [269, 3028], [270, 271], [270, 380], [270, 437], [270, 560], [270, 726], [270, 731], [270, 2401], [271, 438], [271, 493], [271, 504], [271, 561], [271, 587], [271, 616], [271, 635], [271, 691], [271, 726], [271, 2402], [271, 2635], [271, 2796], [272, 273], [272, 337], [272, 474], [272, 606], [272, 726], [272, 728], [272, 730], [272, 807], [272, 965], [272, 1027], [272, 1028], [272, 2143], [272, 2216], [272, 2217], [272, 2233], [272, 2236], [272, 2330], [272, 2401], [272, 2402], [272, 2403], [272, 2570], [272, 2578], [272, 2613], [272, 2728], [272, 2858], [272, 2861], [273, 320], [273, 468], [273, 572], [273, 597], [273, 657], [273, 722], [273, 756], [273, 947], [273, 965], [273, 2145], [273, 2404], [273, 2480], [273, 2535], [273, 2570], [273, 2691], [273, 2692], [273, 2729], [273, 2756], [273, 2787], [273, 2788], [273, 2857], [274, 275], [274, 276], [274, 284], [274, 354], [274, 570], [274, 638], [274, 666], [274, 702], [274, 748], [274, 2157], [274, 2405], [275, 276], [275, 323], [275, 421], [275, 545], [275, 595], [275, 599], [275, 608], [275, 648], [275, 690], [275, 732], [275, 936], [275, 2406], [276, 277], [276, 369], [276, 416], [276, 546], [276, 600], [276, 631], [276, 703], [276, 867], [276, 2407], [276, 2546], [276, 2650], [276, 2672], [277, 329], [277, 343], [277, 354], [277, 395], [277, 743], [277, 2155], [277, 2251], [277, 2408], [277, 2673], [278, 279], [278, 308], [278, 378], [278, 397], [278, 409], [278, 681], [278, 2409], [279, 280], [279, 305], [279, 410], [279, 460], [279, 633], [279, 681], [279, 2410], [280, 281], [280, 308], [280, 495], [280, 684], [280, 725], [280, 2410], [280, 2411], [281, 282], [281, 410], [281, 411], [281, 499], [281, 2412], [281, 2541], [282, 316], [282, 329], [282, 360], [282, 435], [282, 453], [282, 502], [282, 654], [282, 876], [282, 2145], [282, 2251], [282, 2413], [282, 2447], [282, 2583], [283, 284], [283, 287], [283, 311], [283, 332], [283, 400], [283, 417], [283, 459], [283, 472], [283, 587], [283, 692], [283, 1036], [283, 2414], [283, 2590], [284, 285], [284, 286], [284, 400], [284, 457], [284, 458], [284, 1005], [284, 2387], [284, 2415], [284, 2588], [285, 286], [285, 381], [285, 383], [285, 666], [285, 667], [285, 1004], [285, 2288], [285, 2416], [285, 2512], [286, 316], [286, 319], [286, 332], [286, 546], [286, 584], [286, 686], [286, 841], [286, 2132], [286, 2251], [286, 2417], [287, 288], [287, 289], [287, 293], [287, 333], [287, 420], [287, 473], [287, 655], [287, 738], [287, 1039], [287, 1040], [287, 2387], [287, 2418], [287, 2663], [288, 289], [288, 573], [288, 589], [288, 2419], [288, 2719], [289, 735], [289, 890], [289, 910], [289, 2143], [289, 2144], [289, 2420], [289, 2478], [289, 2579], [290, 291], [290, 371], [290, 563], [290, 564], [290, 2421], [291, 292], [291, 371], [291, 372], [291, 384], [291, 385], [291, 476], [291, 534], [291, 788], [291, 978], [291, 981], [291, 2132], [291, 2153], [291, 2329], [291, 2342], [291, 2422], [291, 2502], [291, 2694], [291, 2695], [292, 293], [292, 363], [292, 427], [292, 476], [292, 512], [292, 564], [292, 565], [292, 640], [292, 661], [292, 734], [292, 743], [292, 823], [292, 825], [292, 877], [292, 1043], [292, 2176], [292, 2177], [292, 2421], [292, 2423], [292, 2493], [292, 2548], [292, 2552], [292, 2649], [292, 2663], [292, 2694], [292, 2696], [292, 2822], [292, 2914], [293, 336], [293, 400], [293, 405], [293, 526], [293, 532], [293, 533], [293, 589], [293, 591], [293, 696], [293, 1038], [293, 1059], [293, 2212], [293, 2387], [293, 2424], [293, 2579], [293, 2651], [293, 2652], [293, 2664], [293, 2695], [293, 2719], [293, 2836], [293, 2837], [294, 295], [294, 347], [294, 370], [294, 430], [294, 431], [294, 658], [294, 667], [294, 690], [294, 713], [294, 778], [294, 950], [294, 992], [294, 2156], [294, 2157], [294, 2425], [294, 2545], [295, 296], [295, 304], [295, 359], [295, 430], [295, 712], [295, 947], [295, 2425], [295, 2426], [296, 297], [296, 313], [296, 422], [296, 599], [296, 690], [296, 767], [296, 947], [296, 988], [296, 2231], [296, 2309], [296, 2427], [296, 2679], [296, 2821], [297, 551], [297, 556], [297, 708], [297, 766], [297, 989], [297, 2145], [297, 2229], [297, 2232], [297, 2303], [297, 2428], [297, 2490], [297, 2587], [297, 2679], [297, 2686], [297, 2805], [297, 2839], [297, 2842], [298, 299], [298, 455], [298, 483], [298, 700], [298, 2156], [298, 2369], [298, 2429], [299, 300], [299, 375], [299, 700], [299, 721], [299, 722], [299, 729], [299, 2430], [299, 2852], [300, 420], [300, 606], [300, 698], [300, 721], [300, 733], [300, 994], [300, 1050], [300, 2144], [300, 2233], [300, 2369], [300, 2370], [300, 2392], [300, 2430], [300, 2431], [300, 2835], [300, 2863], [301, 302], [301, 374], [301, 382], [301, 450], [301, 822], [301, 2156], [301, 2373], [301, 2432], [301, 2582], [302, 303], [302, 448], [302, 449], [302, 568], [302, 2154], [302, 2433], [303, 304], [303, 337], [303, 352], [303, 447], [303, 707], [303, 720], [303, 730], [303, 1056], [303, 2434], [303, 2578], [304, 375], [304, 486], [304, 544], [304, 1369], [304, 2145], [304, 2156], [304, 2435], [305, 306], [305, 315], [305, 349], [305, 355], [305, 397], [305, 2156], [305, 2378], [305, 2436], [305, 2716], [306, 307], [306, 506], [306, 717], [306, 791], [306, 2437], [306, 2669], [307, 450], [307, 507], [307, 526], [307, 538], [307, 613], [307, 621], [307, 708], [307, 858], [307, 911], [307, 953], [307, 971], [307, 990], [307, 2164], [307, 2199], [307, 2229], [307, 2438], [307, 2582], [307, 2651], [307, 2652], [307, 2670], [307, 2671], [307, 2716], [307, 2846], [308, 309], [308, 378], [308, 379], [308, 499], [308, 684], [308, 745], [308, 2156], [308, 2409], [308, 2439], [309, 310], [309, 414], [309, 451], [309, 452], [309, 515], [309, 520], [309, 723], [309, 778], [309, 2440], [310, 379], [310, 450], [310, 453], [310, 513], [310, 516], [310, 707], [310, 724], [310, 739], [310, 777], [310, 952], [310, 956], [310, 1013], [310, 1021], [310, 2254], [310, 2439], [310, 2441], [310, 2545], [310, 2582], [310, 2583], [310, 2646], [310, 2651], [310, 2854], [310, 2909], [311, 312], [311, 348], [311, 363], [311, 423], [311, 439], [311, 447], [311, 458], [311, 459], [311, 461], [311, 575], [311, 589], [311, 693], [311, 711], [311, 809], [311, 830], [311, 833], [311, 890], [311, 893], [311, 1030], [311, 1036], [311, 1040], [311, 2156], [311, 2414], [311, 2442], [311, 2493], [311, 2588], [311, 2591], [311, 2612], [311, 2719], [312, 313], [312, 327], [312, 352], [312, 382], [312, 422], [312, 615], [312, 711], [312, 783], [312, 1041], [312, 1042], [312, 2309], [312, 2343], [312, 2396], [312, 2443], [312, 2453], [312, 2590], [313, 326], [313, 327], [313, 1018], [313, 1030], [313, 2138], [313, 2303], [313, 2310], [313, 2444], [313, 2453], [313, 2651], [313, 2862], [314, 315], [314, 317], [314, 395], [314, 482], [314, 542], [314, 544], [314, 592], [314, 608], [314, 617], [314, 636], [314, 735], [314, 811], [314, 890], [314, 912], [314, 2156], [314, 2445], [314, 2593], [314, 2793], [315, 316], [315, 354], [315, 395], [315, 397], [315, 502], [315, 571], [315, 581], [315, 584], [315, 618], [315, 681], [315, 682], [315, 889], [315, 1064], [315, 2446], [315, 2628], [315, 2707], [315, 2794], [316, 354], [316, 410], [316, 434], [316, 435], [316, 791], [316, 2132], [316, 2347], [316, 2447], [317, 367], [317, 444], [317, 566], [317, 569], [317, 607], [317, 626], [317, 650], [317, 679], [317, 912], [317, 987], [317, 2188], [317, 2448], [318, 319], [318, 668], [318, 732], [318, 2449], [319, 320], [319, 545], [319, 560], [319, 570], [319, 616], [319, 657], [319, 945], [319, 2450], [319, 2691], [319, 2787], [320, 321], [320, 356], [320, 552], [320, 560], [320, 571], [320, 574], [320, 643], [320, 690], [320, 710], [320, 820], [320, 888], [320, 945], [320, 983], [320, 1052], [320, 2181], [320, 2251], [320, 2401], [320, 2450], [320, 2451], [320, 2568], [320, 2593], [320, 2628], [320, 2692], [320, 2728], [320, 2844], [321, 325], [321, 556], [321, 710], [321, 712], [321, 805], [321, 2452], [321, 2492], [321, 2630], [321, 2686], [321, 2691], [321, 2844], [322, 323], [322, 327], [322, 361], [322, 518], [322, 519], [322, 629], [322, 2453], [323, 324], [323, 327], [323, 630], [323, 930], [323, 2453], [323, 2454], [324, 325], [324, 361], [324, 472], [324, 2401], [324, 2455], [324, 2492], [325, 326], [325, 327], [325, 341], [325, 352], [325, 361], [325, 363], [325, 809], [325, 2146], [325, 2155], [325, 2453], [325, 2455], [325, 2456], [325, 2475], [325, 2493], [325, 2862], [326, 352], [326, 364], [326, 731], [326, 785], [326, 1046], [326, 2145], [326, 2309], [326, 2312], [326, 2324], [326, 2359], [326, 2361], [326, 2401], [326, 2457], [326, 2491], [326, 2492], [326, 2627], [326, 2861], [327, 328], [327, 345], [327, 412], [327, 512], [327, 599], [327, 691], [327, 696], [327, 799], [327, 811], [327, 890], [327, 930], [327, 1016], [327, 2132], [327, 2143], [327, 2309], [327, 2373], [327, 2454], [327, 2458], [327, 2492], [327, 2649], [327, 2650], [327, 2760], [328, 329], [328, 345], [328, 389], [328, 440], [328, 464], [328, 543], [328, 612], [328, 628], [328, 800], [328, 878], [328, 2171], [328, 2384], [328, 2453], [328, 2459], [328, 2585], [328, 2637], [328, 2779], [329, 344], [329, 369], [329, 376], [329, 377], [329, 398], [329, 499], [329, 794], [329, 1062], [329, 2155], [329, 2460], [329, 2585], [330, 331], [330, 334], [330, 686], [330, 706], [330, 2461], [331, 332], [331, 669], [331, 699], [331, 2462], [332, 333], [332, 575], [332, 909], [332, 2463], [333, 446], [333, 488], [333, 526], [333, 655], [333, 699], [333, 749], [333, 2132], [333, 2464], [333, 2652], [334, 335], [334, 336], [334, 400], [334, 517], [334, 531], [334, 812], [334, 939], [334, 940], [334, 2132], [334, 2163], [334, 2182], [334, 2378], [334, 2399], [334, 2462], [334, 2465], [334, 2817], [334, 2837], [335, 336], [335, 453], [335, 693], [335, 1060], [335, 2321], [335, 2347], [335, 2461], [335, 2466], [335, 2528], [335, 2583], [335, 2612], [335, 2837], [336, 337], [336, 606], [336, 697], [336, 706], [336, 707], [336, 734], [336, 755], [336, 898], [336, 956], [336, 985], [336, 994], [336, 1014], [336, 1054], [336, 2177], [336, 2183], [336, 2233], [336, 2254], [336, 2255], [336, 2461], [336, 2467], [336, 2578], [336, 2580], [336, 2620], [336, 2663], [336, 2710], [336, 2717], [336, 2830], [336, 2836], [336, 2838], [337, 447], [337, 568], [337, 769], [337, 786], [337, 789], [337, 790], [337, 807], [337, 830], [337, 1015], [337, 1033], [337, 2145], [337, 2154], [337, 2243], [337, 2279], [337, 2283], [337, 2303], [337, 2375], [337, 2434], [337, 2442], [337, 2468], [337, 2516], [337, 2577], [337, 2717], [337, 2795], [337, 2837], [337, 2857], [337, 2922], [337, 3147], [337, 3190], [338, 339], [338, 358], [338, 370], [338, 397], [338, 506], [338, 664], [338, 713], [338, 2469], [339, 340], [339, 508], [339, 625], [339, 722], [339, 735], [339, 811], [339, 2143], [339, 2470], [340, 353], [340, 371], [340, 394], [340, 449], [340, 450], [340, 651], [340, 846], [340, 978], [340, 2471], [340, 2582], [340, 2795], [341, 342], [341, 343], [341, 351], [341, 358], [341, 386], [341, 445], [341, 683], [341, 710], [341, 736], [341, 856], [341, 962], [341, 973], [341, 1061], [341, 2132], [341, 2147], [341, 2167], [341, 2205], [341, 2245], [341, 2312], [341, 2383], [341, 2469], [341, 2472], [341, 2475], [341, 2478], [341, 2481], [341, 2485], [341, 2492], [341, 2496], [341, 2507], [341, 2575], [341, 2673], [341, 2747], [341, 2813], [341, 2844], [341, 2866], [342, 343], [342, 610], [342, 616], [342, 643], [342, 661], [342, 728], [342, 737], [342, 743], [342, 804], [342, 884], [342, 902], [342, 903], [342, 934], [342, 945], [342, 2146], [342, 2198], [342, 2402], [342, 2450], [342, 2473], [342, 2476], [342, 2673], [342, 2735], [342, 2748], [342, 2774], [342, 2778], [342, 2822], [342, 2858], [343, 542], [343, 543], [343, 631], [343, 834], [343, 878], [343, 2146], [343, 2251], [343, 2384], [343, 2445], [343, 2474], [343, 2609], [343, 2672], [343, 2674], [343, 2747], [344, 345], [344, 360], [344, 361], [344, 745], [344, 2475], [345, 346], [345, 385], [345, 395], [345, 507], [345, 562], [345, 604], [345, 616], [345, 646], [345, 804], [345, 805], [345, 808], [345, 888], [345, 931], [345, 2192], [345, 2453], [345, 2476], [345, 2776], [346, 671], [346, 717], [346, 1016], [346, 2251], [346, 2477], [346, 2795], [346, 2800], [347, 348], [347, 370], [347, 376], [347, 378], [347, 448], [347, 465], [347, 472], [347, 473], [347, 691], [347, 2409], [347, 2478], [348, 349], [348, 402], [348, 439], [348, 474], [348, 739], [348, 1022], [348, 1026], [348, 2236], [348, 2282], [348, 2347], [348, 2410], [348, 2479], [348, 2511], [348, 2590], [349, 465], [349, 466], [349, 468], [349, 597], [349, 2156], [349, 2274], [349, 2480], [350, 351], [350, 355], [350, 2481], [351, 352], [351, 495], [351, 720], [351, 1044], [351, 1066], [351, 2146], [351, 2251], [351, 2312], [351, 2337], [351, 2396], [351, 2482], [351, 2486], [352, 353], [352, 382], [352, 786], [352, 822], [352, 1056], [352, 2375], [352, 2481], [352, 2483], [352, 2492], [352, 2862], [353, 399], [353, 467], [353, 528], [353, 565], [353, 600], [353, 707], [353, 722], [353, 734], [353, 822], [353, 2145], [353, 2155], [353, 2177], [353, 2484], [353, 2597], [354, 355], [354, 356], [354, 357], [354, 365], [354, 469], [354, 876], [354, 1061], [354, 2159], [354, 2485], [354, 2596], [355, 356], [355, 467], [355, 470], [355, 2486], [355, 2597], [356, 468], [356, 471], [356, 927], [356, 1065], [356, 2160], [356, 2251], [356, 2480], [356, 2487], [356, 2691], [357, 358], [357, 618], [357, 702], [357, 743], [357, 859], [357, 2171], [357, 2469], [357, 2488], [357, 2794], [358, 470], [358, 628], [358, 710], [358, 786], [358, 919], [358, 920], [358, 2146], [358, 2321], [358, 2342], [358, 2375], [358, 2376], [358, 2470], [358, 2489], [358, 2501], [358, 2528], [358, 2637], [358, 2795], [358, 2844], [359, 360], [359, 456], [359, 710], [359, 713], [359, 2490], [359, 2586], [359, 2844], [360, 450], [360, 495], [360, 731], [360, 958], [360, 1046], [360, 2265], [360, 2321], [360, 2491], [360, 2582], [360, 2636], [361, 362], [361, 731], [361, 2492], [362, 363], [362, 459], [362, 472], [362, 496], [362, 518], [362, 564], [362, 588], [362, 745], [362, 2493], [363, 364], [363, 385], [363, 472], [363, 475], [363, 512], [363, 589], [363, 712], [363, 744], [363, 828], [363, 829], [363, 851], [363, 888], [363, 2155], [363, 2492], [363, 2494], [363, 2590], [363, 2603], [363, 2627], [363, 2649], [363, 2695], [363, 2719], [363, 2876], [364, 402], [364, 496], [364, 826], [364, 827], [364, 851], [364, 1043], [364, 1045], [364, 2145], [364, 2178], [364, 2493], [364, 2495], [364, 2511], [364, 2513], [364, 2533], [364, 2626], [364, 2862], [364, 2914], [365, 366], [365, 385], [365, 500], [365, 717], [365, 791], [365, 2496], [366, 367], [366, 405], [366, 650], [366, 735], [366, 921], [366, 1024], [366, 2497], [366, 2579], [366, 2585], [366, 2596], [366, 2781], [367, 544], [367, 553], [367, 572], [367, 680], [367, 683], [367, 685], [367, 987], [367, 2168], [367, 2448], [367, 2498], [367, 2640], [367, 2704], [367, 2734], [367, 2813], [368, 369], [368, 373], [368, 376], [368, 497], [368, 2499], [369, 370], [369, 374], [369, 457], [369, 481], [369, 499], [369, 506], [369, 704], [369, 2500], [369, 2650], [370, 371], [370, 2501], [371, 372], [371, 448], [371, 565], [371, 2156], [371, 2421], [371, 2502], [372, 551], [372, 615], [372, 694], [372, 788], [372, 846], [372, 1056], [372, 2143], [372, 2145], [372, 2232], [372, 2330], [372, 2340], [372, 2421], [372, 2422], [372, 2471], [372, 2501], [372, 2503], [372, 2579], [372, 2696], [373, 374], [373, 387], [373, 424], [373, 440], [373, 453], [373, 513], [373, 591], [373, 2156], [373, 2499], [373, 2504], [373, 2583], [374, 375], [374, 453], [374, 600], [374, 2500], [374, 2505], [374, 2583], [375, 389], [375, 426], [375, 528], [375, 698], [375, 704], [375, 996], [375, 2506], [375, 2835], [376, 377], [376, 477], [376, 497], [376, 2185], [376, 2507], [377, 378], [377, 445], [377, 498], [377, 2171], [377, 2409], [377, 2508], [377, 2575], [378, 379], [378, 396], [378, 449], [378, 627], [378, 787], [378, 838], [378, 2132], [378, 2142], [378, 2410], [378, 2439], [378, 2509], [378, 2528], [378, 2540], [378, 2812], [379, 744], [379, 776], [379, 787], [379, 1021], [379, 1053], [379, 2145], [379, 2337], [379, 2409], [379, 2411], [379, 2440], [379, 2509], [379, 2510], [379, 2630], [379, 2815], [379, 2876], [380, 381], [380, 402], [380, 496], [380, 2511], [381, 382], [381, 383], [381, 457], [381, 515], [381, 2512], [382, 383], [382, 495], [382, 496], [382, 2512], [382, 2513], [383, 402], [383, 458], [383, 516], [383, 712], [383, 786], [383, 1004], [383, 1042], [383, 1063], [383, 2251], [383, 2264], [383, 2296], [383, 2343], [383, 2373], [383, 2375], [383, 2416], [383, 2511], [383, 2513], [383, 2514], [383, 2588], [383, 2646], [384, 385], [384, 512], [384, 587], [384, 593], [384, 615], [384, 634], [384, 669], [384, 2421], [384, 2515], [384, 2649], [385, 386], [385, 395], [385, 447], [385, 475], [385, 651], [385, 789], [385, 827], [385, 933], [385, 982], [385, 1056], [385, 2167], [385, 2421], [385, 2493], [385, 2516], [386, 492], [386, 494], [386, 495], [386, 501], [386, 571], [386, 634], [386, 684], [386, 685], [386, 745], [386, 854], [386, 884], [386, 912], [386, 932], [386, 2132], [386, 2146], [386, 2168], [386, 2249], [386, 2250], [386, 2517], [386, 2623], [386, 2628], [386, 2631], [386, 2634], [387, 388], [387, 405], [387, 664], [387, 2518], [388, 389], [388, 391], [388, 406], [388, 502], [388, 694], [388, 920], [388, 943], [388, 2150], [388, 2268], [388, 2347], [388, 2504], [388, 2519], [388, 2536], [388, 2795], [389, 390], [389, 544], [389, 591], [389, 612], [389, 636], [389, 646], [389, 680], [389, 694], [389, 959], [389, 2518], [389, 2520], [389, 2639], [389, 2640], [389, 2767], [389, 2776], [389, 2779], [390, 508], [390, 628], [390, 637], [390, 680], [390, 760], [390, 853], [390, 924], [390, 2145], [390, 2183], [390, 2223], [390, 2268], [390, 2308], [390, 2324], [390, 2470], [390, 2521], [390, 2637], [390, 2638], [390, 2640], [390, 2689], [390, 2768], [390, 2824], [391, 392], [391, 406], [391, 441], [391, 501], [391, 526], [391, 612], [391, 622], [391, 671], [391, 748], [391, 814], [391, 817], [391, 903], [391, 2132], [391, 2151], [391, 2267], [391, 2340], [391, 2385], [391, 2405], [391, 2496], [391, 2515], [391, 2518], [391, 2522], [391, 2525], [391, 2528], [391, 2531], [391, 2536], [391, 2571], [391, 2610], [391, 2631], [391, 2652], [391, 2779], [391, 2791], [391, 2800], [391, 2846], [392, 393], [392, 404], [392, 476], [392, 550], [392, 612], [392, 617], [392, 657], [392, 660], [392, 661], [392, 810], [392, 820], [392, 821], [392, 981], [392, 983], [392, 2150], [392, 2225], [392, 2329], [392, 2523], [392, 2535], [392, 2596], [392, 2680], [392, 2694], [392, 2779], [392, 2787], [392, 2793], [392, 2822], [392, 2942], [393, 436], [393, 471], [393, 605], [393, 709], [393, 762], [393, 810], [393, 817], [393, 850], [393, 868], [393, 919], [393, 921], [393, 922], [393, 924], [393, 925], [393, 969], [393, 987], [393, 1051], [393, 1065], [393, 2160], [393, 2162], [393, 2174], [393, 2193], [393, 2224], [393, 2230], [393, 2234], [393, 2267], [393, 2323], [393, 2448], [393, 2524], [393, 2638], [393, 2683], [393, 2755], [393, 2781], [393, 2791], [393, 2942], [394, 395], [394, 530], [394, 600], [394, 811], [394, 2525], [395, 396], [395, 449], [395, 526], [395, 688], [395, 836], [395, 837], [395, 896], [395, 940], [395, 941], [395, 2526], [395, 2540], [395, 2652], [395, 2818], [396, 409], [396, 411], [396, 499], [396, 570], [396, 631], [396, 746], [396, 894], [396, 1062], [396, 2132], [396, 2209], [396, 2409], [396, 2527], [396, 2541], [396, 2545], [396, 2630], [396, 2672], [397, 398], [397, 465], [397, 498], [397, 506], [397, 2528], [398, 399], [398, 440], [398, 571], [398, 745], [398, 2529], [398, 2628], [399, 440], [399, 477], [399, 565], [399, 823], [399, 2176], [399, 2185], [399, 2186], [399, 2530], [399, 2585], [400, 401], [400, 416], [400, 440], [400, 691], [400, 814], [400, 2461], [400, 2531], [400, 2546], [400, 2663], [401, 402], [401, 416], [401, 479], [401, 686], [401, 689], [401, 831], [401, 833], [401, 2212], [401, 2511], [401, 2532], [401, 2546], [402, 438], [402, 472], [402, 474], [402, 496], [402, 511], [402, 578], [402, 826], [402, 1063], [402, 2132], [402, 2175], [402, 2236], [402, 2264], [402, 2401], [402, 2512], [402, 2533], [402, 2627], [403, 404], [403, 557], [403, 598], [403, 599], [403, 612], [403, 642], [403, 648], [403, 2534], [403, 2779], [404, 597], [404, 660], [404, 2535], [404, 2791], [405, 406], [405, 476], [405, 553], [405, 591], [405, 614], [405, 704], [405, 2536], [405, 2663], [405, 2694], [406, 407], [406, 606], [406, 698], [406, 850], [406, 870], [406, 942], [406, 1024], [406, 1038], [406, 2150], [406, 2169], [406, 2233], [406, 2234], [406, 2269], [406, 2330], [406, 2424], [406, 2497], [406, 2518], [406, 2537], [406, 2607], [406, 2684], [406, 2722], [406, 2745], [406, 2835], [407, 408], [407, 476], [407, 594], [407, 698], [407, 733], [407, 788], [407, 840], [407, 868], [407, 920], [407, 1020], [407, 2341], [407, 2422], [407, 2423], [407, 2523], [407, 2536], [407, 2538], [407, 2606], [407, 2608], [407, 2681], [407, 2683], [407, 2694], [407, 2724], [407, 2835], [407, 2863], [407, 3050], [408, 469], [408, 475], [408, 594], [408, 789], [408, 829], [408, 848], [408, 876], [408, 2145], [408, 2159], [408, 2161], [408, 2169], [408, 2376], [408, 2494], [408, 2516], [408, 2539], [408, 2600], [408, 2607], [408, 2724], [408, 3114], [409, 410], [409, 414], [409, 499], [409, 541], [409, 2540], [410, 411], [410, 414], [410, 629], [410, 634], [410, 687], [410, 778], [410, 2541], [411, 412], [411, 435], [411, 688], [411, 758], [411, 777], [411, 874], [411, 889], [411, 1016], [411, 2134], [411, 2138], [411, 2249], [411, 2410], [411, 2412], [411, 2447], [411, 2540], [411, 2542], [411, 2545], [411, 2760], [411, 2765], [411, 2818], [411, 2909], [412, 413], [412, 431], [412, 620], [412, 629], [412, 630], [412, 675], [412, 677], [412, 689], [412, 711], [412, 890], [412, 928], [412, 2144], [412, 2392], [412, 2393], [412, 2453], [412, 2541], [412, 2543], [412, 2594], [412, 2595], [412, 2727], [412, 2750], [412, 2761], [413, 464], [413, 529], [413, 537], [413, 600], [413, 622], [413, 631], [413, 800], [413, 835], [413, 866], [413, 905], [413, 1018], [413, 2145], [413, 2459], [413, 2544], [413, 2594], [413, 2610], [413, 2630], [413, 2672], [413, 2674], [413, 2744], [413, 2760], [413, 2762], [414, 415], [414, 515], [414, 2545], [415, 416], [415, 520], [415, 541], [415, 2546], [416, 631], [416, 690], [416, 715], [416, 1047], [416, 2251], [416, 2385], [416, 2396], [416, 2545], [416, 2547], [416, 2651], [416, 2672], [417, 418], [417, 421], [417, 423], [417, 424], [417, 427], [417, 431], [417, 434], [417, 437], [417, 440], [417, 535], [417, 564], [417, 592], [417, 689], [417, 738], [417, 2548], [418, 419], [418, 548], [418, 554], [418, 689], [418, 711], [418, 783], [418, 2549], [419, 420], [419, 552], [419, 553], [419, 655], [419, 658], [419, 674], [419, 704], [419, 739], [419, 858], [419, 873], [419, 996], [419, 1000], [419, 1001], [419, 2133], [419, 2181], [419, 2550], [419, 2734], [420, 491], [420, 992], [420, 2144], [420, 2551], [420, 2651], [420, 2852], [421, 422], [421, 520], [421, 564], [421, 593], [421, 639], [421, 640], [421, 690], [421, 2552], [422, 423], [422, 565], [422, 599], [422, 640], [422, 2552], [422, 2553], [423, 427], [423, 565], [423, 590], [423, 988], [423, 2156], [423, 2548], [423, 2554], [423, 2590], [424, 425], [424, 603], [424, 738], [424, 2555], [425, 426], [425, 427], [425, 572], [425, 825], [425, 2144], [425, 2347], [425, 2349], [425, 2504], [425, 2548], [425, 2556], [425, 2734], [425, 2869], [426, 510], [426, 527], [426, 655], [426, 908], [426, 996], [426, 1040], [426, 2504], [426, 2555], [426, 2557], [426, 2666], [426, 2786], [427, 428], [427, 429], [427, 432], [427, 441], [427, 527], [427, 552], [427, 583], [427, 640], [427, 751], [427, 816], [427, 999], [427, 1036], [427, 2132], [427, 2181], [427, 2212], [427, 2270], [427, 2285], [427, 2371], [427, 2393], [427, 2414], [427, 2549], [427, 2552], [427, 2554], [427, 2555], [427, 2558], [427, 2562], [427, 2565], [427, 2568], [427, 2571], [427, 2666], [427, 2695], [427, 2723], [427, 2820], [427, 2869], [428, 429], [428, 433], [428, 434], [428, 435], [428, 458], [428, 747], [428, 833], [428, 861], [428, 862], [428, 927], [428, 946], [428, 1063], [428, 2260], [428, 2264], [428, 2285], [428, 2287], [428, 2378], [428, 2398], [428, 2447], [428, 2548], [428, 2559], [428, 2588], [428, 2642], [428, 2701], [428, 2870], [429, 430], [429, 659], [429, 948], [429, 1026], [429, 2186], [429, 2274], [429, 2282], [429, 2286], [429, 2425], [429, 2548], [429, 2560], [429, 2565], [429, 2754], [430, 432], [430, 433], [430, 710], [430, 767], [430, 769], [430, 777], [430, 784], [430, 795], [430, 847], [430, 872], [430, 887], [430, 2144], [430, 2145], [430, 2158], [430, 2278], [430, 2279], [430, 2285], [430, 2398], [430, 2426], [430, 2478], [430, 2501], [430, 2561], [430, 2562], [430, 2789], [430, 2798], [430, 2821], [430, 2844], [430, 2909], [430, 3081], [430, 3123], [431, 432], [431, 484], [431, 555], [431, 559], [431, 600], [431, 700], [431, 711], [431, 2545], [431, 2562], [431, 2760], [432, 433], [432, 525], [432, 556], [432, 803], [432, 816], [432, 928], [432, 989], [432, 1049], [432, 2270], [432, 2297], [432, 2298], [432, 2373], [432, 2398], [432, 2425], [432, 2543], [432, 2548], [432, 2563], [432, 2615], [432, 2686], [432, 2690], [432, 2731], [432, 2831], [432, 2842], [433, 458], [433, 461], [433, 746], [433, 782], [433, 952], [433, 1032], [433, 1047], [433, 2397], [433, 2399], [433, 2425], [433, 2441], [433, 2527], [433, 2542], [433, 2545], [433, 2547], [433, 2562], [433, 2564], [433, 2565], [433, 2588], [433, 2591], [433, 2647], [433, 3191], [434, 457], [434, 511], [434, 570], [434, 739], [434, 2175], [434, 2545], [434, 2565], [435, 436], [435, 758], [435, 769], [435, 775], [435, 790], [435, 841], [435, 872], [435, 948], [435, 958], [435, 1008], [435, 1060], [435, 1061], [435, 1062], [435, 1064], [435, 1065], [435, 2134], [435, 2139], [435, 2174], [435, 2245], [435, 2260], [435, 2265], [435, 2279], [435, 2352], [435, 2393], [435, 2413], [435, 2417], [435, 2446], [435, 2485], [435, 2541], [435, 2565], [435, 2566], [435, 2922], [436, 488], [436, 661], [436, 734], [436, 741], [436, 746], [436, 750], [436, 753], [436, 755], [436, 756], [436, 757], [436, 759], [436, 760], [436, 762], [436, 766], [436, 768], [436, 770], [436, 772], [436, 775], [436, 781], [436, 782], [436, 785], [436, 787], [436, 788], [436, 792], [436, 799], [436, 812], [436, 818], [436, 826], [436, 839], [436, 841], [436, 848], [436, 852], [436, 854], [436, 857], [436, 860], [436, 861], [436, 863], [436, 866], [436, 868], [436, 875], [436, 879], [436, 881], [436, 885], [436, 902], [436, 906], [436, 915], [436, 963], [436, 974], [436, 993], [436, 1003], [436, 1012], [436, 1031], [436, 1037], [436, 1501], [436, 2132], [436, 2137], [436, 2145], [436, 2173], [436, 2177], [436, 2200], [436, 2210], [436, 2225], [436, 2237], [436, 2240], [436, 2242], [436, 2266], [436, 2275], [436, 2289], [436, 2299], [436, 2305], [436, 2317], [436, 2323], [436, 2345], [436, 2350], [436, 2352], [436, 2355], [436, 2359], [436, 2363], [436, 2366], [436, 2367], [436, 2388], [436, 2397], [436, 2417], [436, 2422], [436, 2447], [436, 2458], [436, 2464], [436, 2465], [436, 2472], [436, 2509], [436, 2517], [436, 2522], [436, 2527], [436, 2533], [436, 2558], [436, 2567], [436, 2600], [436, 2642], [436, 2657], [436, 2662], [436, 2665], [436, 2678], [436, 2683], [436, 2697], [436, 2710], [436, 2712], [436, 2735], [436, 2737], [436, 2741], [436, 2751], [436, 2762], [436, 2765], [436, 2768], [436, 2788], [436, 2799], [436, 2822], [436, 2826], [436, 2832], [436, 2839], [436, 2879], [436, 2883], [436, 2884], [436, 2896], [436, 2929], [436, 3093], [436, 3167], [437, 438], [437, 439], [437, 560], [437, 2250], [437, 2568], [438, 439], [438, 561], [438, 701], [438, 2371], [438, 2511], [438, 2569], [438, 2726], [439, 574], [439, 597], [439, 659], [439, 726], [439, 738], [439, 965], [439, 966], [439, 2144], [439, 2570], [439, 2590], [440, 441], [440, 512], [440, 517], [440, 535], [440, 537], [440, 562], [440, 591], [440, 600], [440, 2182], [440, 2571], [440, 2649], [441, 442], [441, 527], [441, 800], [441, 803], [441, 814], [441, 815], [441, 870], [441, 2150], [441, 2459], [441, 2504], [441, 2529], [441, 2530], [441, 2531], [441, 2548], [441, 2572], [441, 2643], [441, 2648], [441, 2666], [441, 2668], [441, 2693], [441, 2722], [441, 2731], [442, 458], [442, 512], [442, 514], [442, 594], [442, 799], [442, 806], [442, 829], [442, 861], [442, 893], [442, 2208], [442, 2423], [442, 2458], [442, 2494], [442, 2515], [442, 2571], [442, 2573], [442, 2588], [442, 2642], [442, 2644], [442, 2647], [442, 2649], [442, 2693], [442, 2724], [442, 3023], [443, 444], [443, 446], [443, 449], [443, 522], [443, 569], [443, 979], [443, 2574], [444, 445], [444, 478], [444, 647], [444, 2575], [445, 477], [445, 547], [445, 919], [445, 934], [445, 979], [445, 987], [445, 1010], [445, 2146], [445, 2155], [445, 2185], [445, 2195], [445, 2197], [445, 2448], [445, 2574], [445, 2576], [445, 2609], [445, 2778], [446, 447], [446, 569], [446, 686], [446, 720], [446, 833], [446, 1033], [446, 2378], [446, 2577], [447, 586], [447, 664], [447, 706], [447, 726], [447, 791], [447, 1016], [447, 1059], [447, 2156], [447, 2249], [447, 2578], [447, 2590], [448, 449], [448, 532], [448, 534], [448, 573], [448, 588], [448, 2153], [448, 2156], [448, 2579], [449, 450], [449, 526], [449, 683], [449, 706], [449, 720], [449, 735], [449, 979], [449, 1054], [449, 2409], [449, 2528], [449, 2574], [449, 2580], [449, 2582], [449, 2652], [449, 2813], [450, 451], [450, 453], [450, 461], [450, 495], [450, 625], [450, 714], [450, 765], [450, 845], [450, 912], [450, 923], [450, 971], [450, 980], [450, 1032], [450, 1066], [450, 2156], [450, 2253], [450, 2440], [450, 2581], [450, 2583], [450, 2591], [450, 2669], [450, 2804], [451, 452], [451, 460], [451, 538], [451, 673], [451, 2582], [452, 453], [452, 455], [452, 466], [452, 2583], [453, 456], [453, 467], [453, 517], [453, 858], [453, 943], [453, 2182], [453, 2183], [453, 2260], [453, 2440], [453, 2582], [453, 2584], [453, 2586], [453, 2597], [454, 455], [454, 603], [454, 727], [454, 2585], [455, 456], [455, 466], [455, 603], [455, 2586], [456, 467], [456, 558], [456, 709], [456, 914], [456, 1013], [456, 2230], [456, 2321], [456, 2429], [456, 2583], [456, 2585], [456, 2587], [456, 2597], [456, 2734], [457, 458], [457, 459], [457, 512], [457, 513], [457, 783], [457, 2545], [457, 2588], [457, 2649], [458, 514], [458, 834], [458, 1005], [458, 1016], [458, 1043], [458, 1044], [458, 2157], [458, 2251], [458, 2398], [458, 2415], [458, 2500], [458, 2512], [458, 2565], [458, 2589], [458, 2590], [458, 2643], [458, 2644], [458, 2914], [459, 460], [459, 481], [459, 588], [459, 2590], [460, 461], [460, 745], [460, 2249], [460, 2545], [460, 2591], [461, 567], [461, 744], [461, 833], [461, 898], [461, 994], [461, 1035], [461, 2144], [461, 2164], [461, 2189], [461, 2243], [461, 2257], [461, 2398], [461, 2399], [461, 2410], [461, 2582], [461, 2590], [461, 2592], [461, 2876], [462, 463], [462, 560], [462, 673], [462, 687], [462, 727], [462, 2593], [463, 464], [463, 504], [463, 588], [463, 629], [463, 687], [463, 2594], [464, 479], [464, 499], [464, 541], [464, 543], [464, 613], [464, 629], [464, 631], [464, 2156], [464, 2595], [464, 2672], [465, 466], [465, 660], [465, 2596], [466, 467], [466, 549], [466, 2597], [467, 468], [467, 470], [467, 550], [467, 722], [467, 1044], [467, 1053], [467, 2157], [467, 2334], [467, 2480], [467, 2583], [467, 2586], [467, 2596], [467, 2598], [467, 2680], [468, 471], [468, 719], [468, 926], [468, 943], [468, 949], [468, 1003], [468, 1022], [468, 2145], [468, 2160], [468, 2274], [468, 2275], [468, 2286], [468, 2436], [468, 2479], [468, 2596], [468, 2597], [468, 2599], [468, 2728], [469, 470], [469, 471], [469, 475], [469, 581], [469, 638], [469, 668], [469, 828], [469, 848], [469, 933], [469, 2132], [469, 2160], [469, 2449], [469, 2596], [469, 2600], [469, 2603], [469, 2606], [469, 2609], [469, 2707], [470, 471], [470, 517], [470, 919], [470, 2159], [470, 2160], [470, 2182], [470, 2396], [470, 2469], [470, 2597], [470, 2601], [470, 2716], [471, 494], [471, 622], [471, 826], [471, 859], [471, 925], [471, 946], [471, 1017], [471, 1018], [471, 1019], [471, 2138], [471, 2159], [471, 2161], [471, 2225], [471, 2304], [471, 2480], [471, 2488], [471, 2533], [471, 2602], [471, 2610], [471, 2634], [471, 2692], [471, 2693], [471, 2708], [471, 2754], [471, 2833], [472, 473], [472, 587], [472, 667], [472, 828], [472, 2493], [472, 2511], [472, 2603], [473, 474], [473, 477], [473, 701], [473, 886], [473, 992], [473, 2185], [473, 2236], [473, 2604], [473, 2726], [474, 494], [474, 606], [474, 677], [474, 892], [474, 893], [474, 1011], [474, 2186], [474, 2212], [474, 2233], [474, 2296], [474, 2511], [474, 2605], [474, 2634], [474, 2717], [474, 2727], [474, 2745], [474, 2764], [474, 2857], [475, 476], [475, 593], [475, 983], [475, 2156], [475, 2159], [475, 2493], [475, 2606], [475, 2694], [476, 477], [476, 550], [476, 552], [476, 563], [476, 593], [476, 704], [476, 732], [476, 919], [476, 2181], [476, 2185], [476, 2421], [476, 2607], [476, 2680], [476, 2695], [476, 2791], [477, 534], [477, 552], [477, 592], [477, 650], [477, 655], [477, 840], [477, 2153], [477, 2155], [477, 2181], [477, 2186], [477, 2575], [477, 2608], [477, 2694], [478, 479], [478, 542], [478, 647], [478, 2609], [479, 480], [479, 498], [479, 553], [479, 619], [479, 622], [479, 634], [479, 686], [479, 715], [479, 2171], [479, 2610], [480, 622], [480, 674], [480, 688], [480, 858], [480, 874], [480, 2172], [480, 2610], [480, 2611], [480, 2818], [481, 482], [481, 483], [481, 489], [481, 692], [481, 693], [481, 2612], [482, 483], [482, 490], [482, 726], [482, 2613], [483, 491], [483, 652], [483, 654], [483, 2133], [483, 2156], [483, 2614], [484, 485], [484, 525], [484, 530], [484, 531], [484, 535], [484, 598], [484, 600], [484, 2156], [484, 2163], [484, 2594], [484, 2615], [485, 486], [485, 711], [485, 2164], [485, 2594], [485, 2616], [486, 528], [486, 557], [486, 562], [486, 670], [486, 712], [486, 805], [486, 2617], [486, 2693], [487, 488], [487, 531], [487, 567], [487, 582], [487, 752], [487, 866], [487, 911], [487, 939], [487, 954], [487, 2163], [487, 2164], [487, 2189], [487, 2215], [487, 2222], [487, 2402], [487, 2464], [487, 2594], [487, 2613], [487, 2618], [487, 2621], [487, 2718], [487, 2761], [487, 2762], [487, 2817], [488, 567], [488, 605], [488, 697], [488, 807], [488, 856], [488, 863], [488, 901], [488, 908], [488, 909], [488, 910], [488, 913], [488, 955], [488, 963], [488, 1028], [488, 1033], [488, 1039], [488, 1057], [488, 2174], [488, 2189], [488, 2191], [488, 2193], [488, 2205], [488, 2216], [488, 2221], [488, 2245], [488, 2366], [488, 2418], [488, 2463], [488, 2577], [488, 2619], [488, 2657], [488, 2786], [488, 2830], [488, 2880], [489, 490], [489, 491], [489, 706], [489, 2620], [489, 2650], [490, 491], [490, 579], [490, 609], [490, 701], [490, 2211], [490, 2621], [490, 2726], [490, 2739], [491, 641], [491, 700], [491, 718], [491, 955], [491, 997], [491, 2622], [491, 2651], [491, 2775], [492, 493], [492, 570], [492, 584], [492, 666], [492, 681], [492, 932], [492, 2167], [492, 2623], [493, 494], [493, 503], [493, 2624], [493, 2634], [493, 2796], [494, 503], [494, 605], [494, 621], [494, 632], [494, 695], [494, 719], [494, 855], [494, 884], [494, 2160], [494, 2167], [494, 2186], [494, 2193], [494, 2226], [494, 2236], [494, 2624], [494, 2625], [494, 2635], [494, 2636], [494, 2754], [495, 496], [495, 725], [495, 745], [495, 1045], [495, 1063], [495, 2167], [495, 2264], [495, 2396], [495, 2481], [495, 2582], [495, 2626], [496, 731], [496, 783], [496, 2156], [496, 2511], [496, 2627], [497, 498], [497, 499], [497, 560], [497, 570], [497, 571], [497, 592], [497, 2628], [498, 499], [498, 713], [498, 2629], [499, 2156], [499, 2540], [499, 2630], [500, 501], [500, 584], [500, 687], [500, 2631], [501, 502], [501, 584], [501, 688], [501, 983], [501, 2150], [501, 2167], [501, 2169], [501, 2251], [501, 2496], [501, 2632], [501, 2715], [501, 2818], [502, 544], [502, 653], [502, 684], [502, 837], [502, 876], [502, 1064], [502, 2169], [502, 2446], [502, 2518], [502, 2631], [502, 2633], [503, 504], [503, 505], [503, 623], [503, 2634], [504, 505], [504, 588], [504, 644], [504, 645], [504, 2635], [505, 540], [505, 665], [505, 670], [505, 699], [505, 2156], [505, 2636], [505, 2796], [506, 507], [506, 508], [506, 539], [506, 628], [506, 2637], [507, 508], [507, 523], [507, 531], [507, 557], [507, 562], [507, 647], [507, 648], [507, 911], [507, 924], [507, 2163], [507, 2638], [507, 2650], [507, 2669], [508, 509], [508, 558], [508, 637], [508, 693], [508, 2156], [508, 2223], [508, 2612], [508, 2639], [509, 510], [509, 647], [509, 649], [509, 655], [509, 679], [509, 680], [509, 727], [509, 2640], [510, 642], [510, 680], [510, 729], [510, 2640], [510, 2641], [511, 512], [511, 516], [511, 547], [511, 739], [511, 823], [511, 861], [511, 899], [511, 2132], [511, 2176], [511, 2178], [511, 2195], [511, 2511], [511, 2642], [511, 2646], [511, 2649], [512, 513], [512, 516], [512, 518], [512, 562], [512, 593], [512, 892], [512, 2175], [512, 2453], [512, 2493], [512, 2643], [512, 2646], [512, 2650], [512, 2695], [513, 514], [513, 739], [513, 825], [513, 2156], [513, 2387], [513, 2440], [513, 2644], [513, 2649], [514, 862], [514, 952], [514, 1037], [514, 1040], [514, 2145], [514, 2148], [514, 2296], [514, 2387], [514, 2388], [514, 2441], [514, 2504], [514, 2588], [514, 2643], [514, 2645], [514, 2870], [514, 2956], [515, 516], [515, 518], [515, 634], [515, 745], [515, 2646], [516, 517], [516, 734], [516, 744], [516, 894], [516, 897], [516, 991], [516, 992], [516, 2175], [516, 2177], [516, 2179], [516, 2182], [516, 2209], [516, 2373], [516, 2440], [516, 2512], [516, 2545], [516, 2647], [516, 2649], [516, 2765], [516, 2876], [517, 546], [517, 552], [517, 600], [517, 622], [517, 628], [517, 815], [517, 927], [517, 2181], [517, 2183], [517, 2251], [517, 2461], [517, 2583], [517, 2609], [517, 2610], [517, 2637], [517, 2646], [517, 2648], [517, 2677], [518, 519], [518, 564], [518, 2649], [519, 520], [519, 608], [519, 2650], [520, 532], [520, 538], [520, 593], [520, 608], [520, 704], [520, 723], [520, 2156], [520, 2651], [521, 522], [521, 526], [521, 530], [521, 532], [521, 535], [521, 538], [521, 662], [521, 663], [521, 2652], [522, 523], [522, 526], [522, 533], [522, 647], [522, 649], [522, 754], [522, 2652], [522, 2653], [523, 524], [523, 530], [523, 531], [523, 635], [523, 2163], [523, 2654], [523, 2661], [524, 525], [524, 526], [524, 530], [524, 557], [524, 646], [524, 694], [524, 897], [524, 2373], [524, 2525], [524, 2615], [524, 2652], [524, 2654], [524, 2655], [524, 2776], [524, 2795], [525, 527], [525, 531], [525, 557], [525, 803], [525, 852], [525, 2145], [525, 2163], [525, 2222], [525, 2399], [525, 2562], [525, 2616], [525, 2656], [525, 2661], [525, 2662], [525, 2666], [525, 2729], [525, 2731], [526, 527], [526, 617], [526, 618], [526, 621], [526, 642], [526, 743], [526, 754], [526, 791], [526, 863], [526, 1059], [526, 2132], [526, 2150], [526, 2653], [526, 2657], [526, 2661], [526, 2663], [526, 2666], [526, 2669], [526, 2793], [526, 2794], [527, 528], [527, 535], [527, 642], [527, 801], [527, 893], [527, 2186], [527, 2548], [527, 2571], [527, 2615], [527, 2652], [527, 2658], [527, 2667], [528, 529], [528, 550], [528, 600], [528, 612], [528, 640], [528, 700], [528, 703], [528, 916], [528, 2552], [528, 2659], [528, 2666], [528, 2680], [528, 2744], [528, 2779], [528, 2834], [529, 610], [529, 612], [529, 613], [529, 677], [529, 767], [529, 915], [529, 990], [529, 2145], [529, 2198], [529, 2227], [529, 2310], [529, 2438], [529, 2595], [529, 2660], [529, 2667], [529, 2727], [529, 2741], [529, 2745], [529, 2774], [529, 2779], [529, 2821], [530, 645], [530, 664], [530, 2661], [531, 693], [531, 701], [531, 808], [531, 852], [531, 867], [531, 877], [531, 896], [531, 2132], [531, 2164], [531, 2221], [531, 2407], [531, 2461], [531, 2612], [531, 2615], [531, 2620], [531, 2662], [531, 2726], [532, 533], [532, 564], [532, 588], [532, 705], [532, 706], [532, 2663], [533, 534], [533, 587], [533, 2153], [533, 2663], [533, 2664], [534, 566], [534, 568], [534, 587], [534, 650], [534, 839], [534, 979], [534, 2132], [534, 2154], [534, 2185], [534, 2188], [534, 2283], [534, 2421], [534, 2574], [534, 2579], [534, 2665], [535, 536], [535, 2666], [536, 537], [536, 573], [536, 613], [536, 621], [536, 689], [536, 2594], [536, 2667], [537, 571], [537, 622], [537, 689], [537, 884], [537, 892], [537, 2186], [537, 2595], [537, 2610], [537, 2628], [537, 2668], [538, 539], [538, 540], [538, 585], [538, 715], [538, 2669], [539, 540], [539, 699], [539, 704], [539, 716], [539, 2670], [540, 575], [540, 604], [540, 713], [540, 717], [540, 2156], [540, 2192], [540, 2671], [541, 542], [541, 608], [541, 631], [541, 2672], [542, 543], [542, 616], [542, 2673], [543, 544], [543, 643], [543, 2673], [543, 2674], [544, 590], [544, 837], [544, 2445], [544, 2675], [545, 546], [545, 552], [545, 607], [545, 647], [545, 935], [545, 2181], [545, 2676], [546, 547], [546, 2182], [546, 2195], [546, 2677], [547, 637], [547, 900], [547, 906], [547, 2132], [547, 2175], [547, 2183], [547, 2196], [547, 2223], [547, 2575], [547, 2650], [547, 2678], [547, 2738], [548, 549], [548, 2679], [549, 550], [549, 563], [549, 660], [549, 702], [549, 783], [549, 2680], [550, 551], [550, 658], [550, 702], [550, 858], [550, 1019], [550, 1020], [550, 1043], [550, 1051], [550, 2157], [550, 2186], [550, 2232], [550, 2597], [550, 2679], [550, 2681], [550, 2694], [550, 2791], [550, 2833], [550, 2914], [551, 810], [551, 1012], [551, 1013], [551, 1041], [551, 1051], [551, 2145], [551, 2231], [551, 2305], [551, 2342], [551, 2443], [551, 2502], [551, 2553], [551, 2680], [551, 2682], [551, 2942], [552, 553], [552, 556], [552, 599], [552, 642], [552, 749], [552, 868], [552, 927], [552, 935], [552, 992], [552, 2132], [552, 2182], [552, 2185], [552, 2548], [552, 2676], [552, 2679], [552, 2683], [552, 2686], [552, 2691], [552, 2694], [553, 554], [553, 579], [553, 607], [553, 680], [553, 900], [553, 942], [553, 2181], [553, 2211], [553, 2212], [553, 2593], [553, 2640], [553, 2684], [553, 2738], [554, 577], [554, 578], [554, 592], [554, 614], [554, 740], [554, 883], [554, 899], [554, 2186], [554, 2593], [554, 2685], [555, 556], [555, 558], [555, 2686], [556, 557], [556, 599], [556, 712], [556, 914], [556, 1032], [556, 2155], [556, 2181], [556, 2231], [556, 2373], [556, 2562], [556, 2687], [556, 2689], [557, 642], [557, 805], [557, 923], [557, 924], [557, 2615], [557, 2638], [557, 2661], [557, 2686], [557, 2688], [558, 559], [558, 2586], [558, 2689], [559, 693], [559, 950], [559, 1060], [559, 2155], [559, 2393], [559, 2429], [559, 2612], [559, 2690], [560, 561], [560, 597], [560, 713], [560, 2691], [561, 562], [561, 592], [561, 596], [561, 597], [561, 2692], [562, 641], [562, 667], [562, 669], [562, 703], [562, 743], [562, 745], [562, 884], [562, 2156], [562, 2649], [562, 2693], [562, 2775], [563, 564], [563, 660], [563, 2694], [564, 565], [564, 691], [564, 783], [564, 2695], [565, 2695], [565, 2696], [566, 567], [566, 569], [566, 573], [566, 620], [566, 747], [566, 833], [566, 875], [566, 890], [566, 903], [566, 2132], [566, 2153], [566, 2189], [566, 2697], [566, 2701], [566, 2704], [566, 2750], [567, 568], [567, 746], [567, 836], [567, 838], [567, 890], [567, 912], [567, 973], [567, 998], [567, 2154], [567, 2188], [567, 2190], [567, 2221], [567, 2243], [567, 2373], [567, 2464], [567, 2526], [567, 2527], [567, 2591], [567, 2698], [567, 2812], [567, 2823], [568, 651], [568, 979], [568, 1057], [568, 2153], [568, 2155], [568, 2189], [568, 2328], [568, 2373], [568, 2433], [568, 2574], [568, 2578], [568, 2699], [569, 672], [569, 682], [569, 683], [569, 986], [569, 2188], [569, 2700], [569, 2813], [570, 571], [570, 747], [570, 2540], [570, 2628], [570, 2701], [571, 572], [571, 573], [571, 583], [571, 747], [571, 819], [571, 874], [571, 889], [571, 1053], [571, 2157], [571, 2167], [571, 2186], [571, 2499], [571, 2507], [571, 2629], [571, 2630], [571, 2691], [571, 2701], [571, 2702], [571, 2723], [572, 583], [572, 685], [572, 844], [572, 2168], [572, 2555], [572, 2628], [572, 2703], [572, 2723], [572, 2728], [573, 574], [573, 655], [573, 674], [573, 682], [573, 2156], [573, 2188], [573, 2628], [573, 2704], [574, 575], [574, 596], [574, 643], [574, 749], [574, 2691], [574, 2705], [575, 654], [575, 909], [575, 2250], [575, 2251], [575, 2463], [575, 2590], [575, 2706], [576, 577], [576, 581], [576, 585], [576, 587], [576, 592], [576, 650], [576, 2707], [577, 578], [577, 585], [577, 618], [577, 2708], [577, 2794], [578, 579], [578, 586], [578, 587], [578, 2211], [578, 2283], [578, 2511], [578, 2709], [579, 580], [579, 581], [579, 652], [579, 676], [579, 683], [579, 686], [579, 706], [579, 755], [579, 1002], [579, 2132], [579, 2133], [579, 2212], [579, 2707], [579, 2710], [579, 2780], [579, 2783], [579, 2813], [580, 652], [580, 688], [580, 707], [580, 755], [580, 777], [580, 895], [580, 941], [580, 1023], [580, 2145], [580, 2169], [580, 2211], [580, 2353], [580, 2614], [580, 2710], [580, 2711], [580, 2784], [580, 2818], [580, 2909], [581, 582], [581, 583], [581, 584], [581, 879], [581, 896], [581, 911], [581, 921], [581, 982], [581, 2132], [581, 2159], [581, 2169], [581, 2211], [581, 2708], [581, 2712], [581, 2716], [581, 2718], [581, 2723], [581, 2781], [582, 583], [582, 587], [582, 589], [582, 828], [582, 839], [582, 889], [582, 903], [582, 1036], [582, 2221], [582, 2402], [582, 2414], [582, 2515], [582, 2603], [582, 2664], [582, 2665], [582, 2707], [582, 2709], [582, 2713], [582, 2717], [582, 2719], [582, 2720], [582, 2723], [582, 2765], [583, 590], [583, 592], [583, 594], [583, 840], [583, 851], [583, 883], [583, 907], [583, 2178], [583, 2207], [583, 2212], [583, 2298], [583, 2445], [583, 2548], [583, 2608], [583, 2628], [583, 2685], [583, 2692], [583, 2707], [583, 2714], [583, 2718], [583, 2721], [583, 2724], [584, 686], [584, 2631], [584, 2707], [584, 2715], [585, 586], [585, 644], [585, 664], [585, 2716], [586, 587], [586, 706], [586, 2717], [587, 588], [587, 589], [587, 592], [587, 634], [587, 2153], [587, 2718], [587, 2719], [588, 589], [588, 2719], [589, 590], [589, 615], [589, 898], [589, 941], [589, 2493], [589, 2579], [589, 2590], [589, 2594], [589, 2635], [589, 2663], [589, 2718], [589, 2720], [590, 591], [590, 592], [590, 2719], [590, 2721], [590, 2723], [591, 641], [591, 642], [591, 738], [591, 870], [591, 907], [591, 2183], [591, 2207], [591, 2663], [591, 2722], [591, 2775], [592, 593], [592, 2185], [592, 2723], [593, 594], [593, 638], [593, 644], [593, 811], [593, 2649], [593, 2694], [593, 2724], [594, 638], [594, 640], [594, 641], [594, 797], [594, 810], [594, 925], [594, 992], [594, 1018], [594, 2157], [594, 2515], [594, 2552], [594, 2606], [594, 2607], [594, 2643], [594, 2651], [594, 2723], [594, 2725], [594, 2769], [594, 2775], [594, 2942], [595, 596], [595, 598], [595, 648], [595, 656], [595, 701], [595, 2726], [596, 613], [596, 629], [596, 677], [596, 690], [596, 2727], [597, 598], [597, 625], [597, 656], [597, 657], [597, 726], [597, 2156], [597, 2728], [597, 2787], [598, 599], [598, 645], [598, 701], [598, 749], [598, 2156], [598, 2726], [598, 2729], [599, 600], [599, 722], [599, 802], [599, 811], [599, 936], [599, 2157], [599, 2181], [599, 2406], [599, 2453], [599, 2686], [599, 2730], [600, 803], [600, 867], [600, 2157], [600, 2182], [600, 2407], [600, 2595], [600, 2731], [601, 602], [601, 603], [601, 2732], [602, 664], [602, 665], [602, 2733], [603, 626], [603, 749], [603, 2734], [604, 605], [604, 616], [604, 902], [604, 909], [604, 922], [604, 2132], [604, 2171], [604, 2193], [604, 2463], [604, 2671], [604, 2732], [604, 2735], [604, 2796], [605, 606], [605, 617], [605, 646], [605, 680], [605, 911], [605, 912], [605, 913], [605, 969], [605, 971], [605, 2192], [605, 2194], [605, 2225], [605, 2233], [605, 2464], [605, 2634], [605, 2640], [605, 2732], [605, 2736], [605, 2755], [605, 2776], [605, 2793], [605, 2880], [606, 609], [606, 680], [606, 685], [606, 724], [606, 729], [606, 733], [606, 736], [606, 798], [606, 850], [606, 985], [606, 1039], [606, 2132], [606, 2168], [606, 2193], [606, 2201], [606, 2234], [606, 2236], [606, 2347], [606, 2418], [606, 2536], [606, 2620], [606, 2640], [606, 2737], [606, 2739], [606, 2837], [606, 2849], [606, 2852], [606, 2854], [606, 2857], [606, 2860], [606, 2863], [606, 2866], [606, 2869], [607, 608], [607, 679], [607, 900], [607, 2738], [608, 609], [608, 636], [608, 2739], [609, 631], [609, 892], [609, 900], [609, 911], [609, 936], [609, 959], [609, 2233], [609, 2392], [609, 2406], [609, 2445], [609, 2650], [609, 2651], [609, 2672], [609, 2738], [609, 2740], [609, 2767], [610, 611], [610, 613], [610, 631], [610, 695], [610, 703], [610, 821], [610, 884], [610, 915], [610, 953], [610, 954], [610, 2132], [610, 2199], [610, 2226], [610, 2672], [610, 2741], [610, 2744], [610, 2747], [611, 612], [611, 636], [611, 648], [611, 703], [611, 917], [611, 2198], [611, 2742], [611, 2779], [612, 648], [612, 701], [612, 917], [612, 924], [612, 934], [612, 936], [612, 2150], [612, 2155], [612, 2341], [612, 2406], [612, 2534], [612, 2638], [612, 2726], [612, 2742], [612, 2743], [612, 2744], [612, 2778], [612, 2791], [613, 614], [613, 643], [613, 648], [613, 690], [613, 2156], [613, 2198], [613, 2669], [613, 2744], [614, 615], [614, 783], [614, 2716], [614, 2745], [615, 670], [615, 1056], [615, 2502], [615, 2515], [615, 2716], [615, 2719], [615, 2746], [616, 617], [616, 643], [616, 647], [616, 691], [616, 727], [616, 2192], [616, 2747], [616, 2793], [617, 618], [617, 646], [617, 657], [617, 662], [617, 727], [617, 735], [617, 737], [617, 910], [617, 2193], [617, 2652], [617, 2748], [617, 2776], [617, 2787], [617, 2791], [617, 2794], [618, 650], [618, 663], [618, 864], [618, 2186], [618, 2274], [618, 2283], [618, 2652], [618, 2749], [618, 2793], [618, 2795], [619, 620], [619, 623], [619, 626], [619, 629], [619, 674], [619, 2750], [620, 621], [620, 622], [620, 672], [620, 674], [620, 689], [620, 708], [620, 750], [620, 967], [620, 1001], [620, 2132], [620, 2188], [620, 2229], [620, 2610], [620, 2751], [620, 2754], [620, 2757], [620, 2760], [620, 2805], [621, 622], [621, 901], [621, 2212], [621, 2594], [621, 2610], [621, 2634], [621, 2652], [621, 2669], [621, 2750], [621, 2752], [622, 831], [622, 857], [622, 903], [622, 904], [622, 939], [622, 942], [622, 943], [622, 2150], [622, 2160], [622, 2173], [622, 2182], [622, 2197], [622, 2532], [622, 2595], [622, 2609], [622, 2611], [622, 2629], [622, 2684], [622, 2750], [622, 2753], [622, 2765], [622, 2817], [622, 2846], [623, 624], [623, 2754], [624, 625], [624, 656], [624, 969], [624, 2755], [625, 713], [625, 735], [625, 969], [625, 2144], [625, 2582], [625, 2733], [625, 2755], [625, 2756], [626, 627], [626, 967], [626, 2171], [626, 2734], [626, 2757], [627, 628], [627, 912], [627, 1055], [627, 2250], [627, 2409], [627, 2637], [627, 2758], [628, 924], [628, 1065], [628, 1066], [628, 2171], [628, 2182], [628, 2251], [628, 2373], [628, 2437], [628, 2469], [628, 2500], [628, 2528], [628, 2638], [628, 2639], [628, 2670], [628, 2759], [629, 630], [629, 2760], [630, 631], [630, 2672], [630, 2760], [630, 2761], [631, 866], [631, 878], [631, 2132], [631, 2198], [631, 2384], [631, 2540], [631, 2546], [631, 2595], [631, 2673], [631, 2739], [631, 2762], [632, 633], [632, 634], [632, 676], [632, 695], [632, 718], [632, 740], [632, 891], [632, 975], [632, 2226], [632, 2634], [632, 2763], [633, 634], [633, 725], [633, 739], [633, 991], [633, 1011], [633, 2142], [633, 2716], [633, 2733], [633, 2764], [634, 667], [634, 675], [634, 778], [634, 2132], [634, 2167], [634, 2765], [635, 636], [635, 645], [635, 646], [635, 676], [635, 699], [635, 701], [635, 763], [635, 2726], [635, 2766], [635, 2776], [636, 637], [636, 679], [636, 691], [636, 704], [636, 821], [636, 959], [636, 2223], [636, 2767], [637, 679], [637, 760], [637, 912], [637, 980], [637, 998], [637, 999], [637, 2132], [637, 2195], [637, 2224], [637, 2316], [637, 2639], [637, 2768], [637, 2823], [638, 639], [638, 797], [638, 2159], [638, 2724], [638, 2769], [639, 640], [639, 667], [639, 668], [639, 703], [639, 796], [639, 2449], [639, 2552], [639, 2770], [640, 700], [640, 712], [640, 767], [640, 796], [640, 936], [640, 1042], [640, 2342], [640, 2343], [640, 2376], [640, 2406], [640, 2548], [640, 2553], [640, 2651], [640, 2695], [640, 2724], [640, 2770], [640, 2771], [640, 2821], [641, 642], [641, 644], [641, 646], [641, 700], [641, 729], [641, 798], [641, 865], [641, 871], [641, 891], [641, 893], [641, 897], [641, 992], [641, 1059], [641, 2156], [641, 2201], [641, 2341], [641, 2635], [641, 2716], [641, 2724], [641, 2772], [641, 2776], [642, 643], [642, 647], [642, 749], [642, 869], [642, 934], [642, 2181], [642, 2652], [642, 2666], [642, 2773], [642, 2775], [642, 2778], [643, 805], [643, 2635], [643, 2671], [643, 2691], [643, 2747], [643, 2774], [644, 645], [644, 2775], [645, 646], [645, 662], [645, 664], [645, 811], [645, 2776], [646, 763], [646, 807], [646, 810], [646, 941], [646, 977], [646, 2193], [646, 2635], [646, 2661], [646, 2729], [646, 2766], [646, 2775], [646, 2777], [646, 2793], [646, 2795], [646, 2942], [647, 648], [647, 934], [647, 2778], [648, 660], [648, 2779], [649, 650], [649, 655], [649, 682], [649, 705], [649, 1002], [649, 2780], [650, 651], [650, 921], [650, 2153], [650, 2185], [650, 2781], [650, 2794], [651, 912], [651, 2154], [651, 2155], [651, 2782], [651, 2795], [652, 653], [652, 687], [652, 778], [652, 2156], [652, 2211], [652, 2783], [653, 654], [653, 682], [653, 683], [653, 684], [653, 2784], [653, 2813], [654, 2245], [654, 2785], [655, 693], [655, 704], [655, 908], [655, 1010], [655, 2185], [655, 2499], [655, 2612], [655, 2786], [656, 657], [656, 660], [656, 662], [656, 2787], [657, 658], [657, 701], [657, 732], [657, 756], [657, 966], [657, 969], [657, 2132], [657, 2274], [657, 2726], [657, 2728], [657, 2755], [657, 2788], [657, 2791], [657, 2793], [658, 659], [658, 690], [658, 700], [658, 718], [658, 732], [658, 739], [658, 784], [658, 938], [658, 947], [658, 970], [658, 1013], [658, 2396], [658, 2680], [658, 2787], [658, 2789], [659, 689], [659, 833], [659, 2144], [659, 2186], [659, 2285], [659, 2790], [660, 662], [660, 691], [660, 811], [660, 2791], [661, 691], [661, 751], [661, 767], [661, 799], [661, 814], [661, 834], [661, 851], [661, 881], [661, 892], [661, 907], [661, 910], [661, 959], [661, 966], [661, 981], [661, 1043], [661, 2149], [661, 2174], [661, 2207], [661, 2329], [661, 2363], [661, 2402], [661, 2458], [661, 2478], [661, 2531], [661, 2695], [661, 2747], [661, 2767], [661, 2791], [661, 2792], [661, 2820], [661, 2821], [661, 2914], [662, 663], [662, 2793], [663, 664], [663, 2794], [664, 669], [664, 673], [664, 682], [664, 2156], [664, 2795], [665, 666], [665, 669], [665, 2796], [666, 667], [666, 669], [666, 1006], [666, 2797], [667, 668], [667, 712], [667, 795], [667, 992], [667, 2449], [667, 2798], [668, 733], [668, 741], [668, 743], [668, 945], [668, 946], [668, 950], [668, 2132], [668, 2159], [668, 2369], [668, 2376], [668, 2450], [668, 2799], [668, 2863], [669, 670], [669, 671], [669, 2800], [670, 671], [670, 2800], [670, 2801], [671, 904], [671, 1006], [671, 2150], [671, 2462], [671, 2515], [671, 2693], [671, 2795], [671, 2796], [671, 2797], [671, 2801], [671, 2802], [672, 673], [672, 675], [672, 714], [672, 764], [672, 2750], [672, 2803], [672, 2804], [673, 674], [673, 682], [673, 713], [673, 714], [673, 778], [673, 2249], [673, 2732], [673, 2804], [674, 874], [674, 2156], [674, 2249], [674, 2734], [674, 2750], [674, 2805], [675, 676], [675, 929], [675, 2316], [675, 2733], [675, 2760], [675, 2806], [676, 677], [676, 699], [676, 821], [676, 944], [676, 2211], [676, 2727], [676, 2733], [676, 2807], [677, 696], [677, 700], [677, 701], [677, 767], [677, 865], [677, 910], [677, 2138], [677, 2144], [677, 2164], [677, 2212], [677, 2236], [677, 2692], [677, 2705], [677, 2726], [677, 2744], [677, 2760], [677, 2808], [677, 2821], [678, 679], [678, 727], [678, 728], [678, 961], [678, 2316], [678, 2809], [678, 2858], [679, 680], [679, 960], [679, 2223], [679, 2640], [679, 2810], [680, 728], [680, 908], [680, 934], [680, 960], [680, 1002], [680, 2171], [680, 2186], [680, 2193], [680, 2233], [680, 2639], [680, 2641], [680, 2778], [680, 2780], [680, 2786], [680, 2810], [680, 2811], [680, 2858], [681, 682], [681, 684], [681, 838], [681, 2812], [682, 683], [682, 2813], [683, 714], [683, 720], [683, 838], [683, 986], [683, 1002], [683, 1064], [683, 2146], [683, 2211], [683, 2245], [683, 2446], [683, 2700], [683, 2704], [683, 2780], [683, 2784], [683, 2795], [683, 2804], [683, 2812], [683, 2814], [684, 685], [684, 719], [684, 776], [684, 778], [684, 838], [684, 2167], [684, 2168], [684, 2347], [684, 2593], [684, 2812], [684, 2815], [685, 725], [685, 736], [685, 865], [685, 898], [685, 2144], [685, 2167], [685, 2169], [685, 2233], [685, 2419], [685, 2816], [685, 2866], [686, 687], [686, 688], [686, 939], [686, 2211], [686, 2817], [686, 2818], [687, 688], [687, 791], [687, 2818], [688, 720], [688, 790], [688, 813], [688, 827], [688, 895], [688, 939], [688, 2249], [688, 2251], [688, 2541], [688, 2593], [688, 2594], [688, 2631], [688, 2783], [688, 2817], [688, 2819], [688, 2922], [689, 690], [689, 691], [689, 751], [689, 988], [689, 2750], [689, 2760], [689, 2820], [690, 691], [690, 708], [690, 767], [690, 820], [690, 2229], [690, 2546], [690, 2691], [690, 2821], [691, 783], [691, 2132], [691, 2453], [691, 2822], [692, 693], [692, 998], [692, 2133], [692, 2612], [692, 2823], [693, 694], [693, 809], [693, 853], [693, 941], [693, 998], [693, 1010], [693, 2143], [693, 2155], [693, 2163], [693, 2347], [693, 2500], [693, 2590], [693, 2613], [693, 2614], [693, 2620], [693, 2823], [693, 2824], [694, 837], [694, 2500], [694, 2502], [694, 2518], [694, 2612], [694, 2661], [694, 2825], [695, 696], [695, 697], [695, 938], [695, 974], [695, 1019], [695, 2132], [695, 2142], [695, 2198], [695, 2227], [695, 2634], [695, 2826], [695, 2830], [695, 2833], [695, 2836], [696, 697], [696, 910], [696, 2143], [696, 2226], [696, 2453], [696, 2635], [696, 2663], [696, 2727], [696, 2827], [696, 2830], [697, 698], [697, 699], [697, 763], [697, 914], [697, 944], [697, 954], [697, 1012], [697, 1049], [697, 1057], [697, 2226], [697, 2228], [697, 2305], [697, 2321], [697, 2331], [697, 2373], [697, 2462], [697, 2464], [697, 2636], [697, 2670], [697, 2766], [697, 2807], [697, 2828], [697, 2831], [697, 2835], [697, 2837], [698, 704], [698, 774], [698, 779], [698, 908], [698, 916], [698, 959], [698, 1000], [698, 1021], [698, 1043], [698, 2145], [698, 2354], [698, 2385], [698, 2500], [698, 2506], [698, 2536], [698, 2550], [698, 2607], [698, 2651], [698, 2670], [698, 2767], [698, 2786], [698, 2829], [698, 2830], [698, 2834], [698, 2852], [698, 2914], [698, 2952], [698, 3111], [699, 700], [699, 704], [699, 706], [699, 2342], [699, 2830], [700, 701], [700, 1049], [700, 1050], [700, 2370], [700, 2396], [700, 2552], [700, 2726], [700, 2727], [700, 2775], [700, 2831], [701, 885], [701, 936], [701, 2132], [701, 2163], [701, 2406], [701, 2727], [701, 2729], [701, 2779], [701, 2787], [701, 2832], [702, 703], [702, 1019], [702, 2342], [702, 2680], [702, 2833], [703, 704], [703, 743], [703, 916], [703, 2198], [703, 2834], [704, 721], [704, 783], [704, 821], [704, 980], [704, 2156], [704, 2585], [704, 2694], [704, 2835], [705, 706], [705, 2836], [706, 707], [706, 2211], [706, 2837], [707, 729], [707, 897], [707, 941], [707, 2440], [707, 2783], [707, 2837], [707, 2838], [708, 709], [708, 710], [708, 766], [708, 873], [708, 970], [708, 991], [708, 1048], [708, 2132], [708, 2230], [708, 2231], [708, 2342], [708, 2344], [708, 2669], [708, 2750], [708, 2839], [708, 2844], [708, 2846], [709, 710], [709, 781], [709, 820], [709, 946], [709, 962], [709, 967], [709, 2224], [709, 2225], [709, 2229], [709, 2260], [709, 2322], [709, 2334], [709, 2345], [709, 2383], [709, 2586], [709, 2587], [709, 2670], [709, 2757], [709, 2840], [709, 2844], [709, 2847], [710, 713], [710, 714], [710, 843], [710, 872], [710, 958], [710, 1029], [710, 2146], [710, 2229], [710, 2230], [710, 2244], [710, 2265], [710, 2425], [710, 2469], [710, 2490], [710, 2629], [710, 2671], [710, 2691], [710, 2756], [710, 2804], [710, 2841], [710, 2848], [711, 712], [711, 989], [711, 1048], [711, 2344], [711, 2429], [711, 2590], [711, 2760], [711, 2842], [712, 795], [712, 2376], [712, 2493], [712, 2512], [712, 2552], [712, 2686], [712, 2798], [712, 2843], [713, 717], [713, 2250], [713, 2844], [714, 764], [714, 777], [714, 904], [714, 912], [714, 943], [714, 2224], [714, 2243], [714, 2325], [714, 2582], [714, 2593], [714, 2795], [714, 2803], [714, 2805], [714, 2813], [714, 2844], [714, 2845], [714, 2909], [715, 716], [715, 717], [715, 2546], [715, 2846], [716, 717], [716, 2171], [716, 2342], [716, 2385], [716, 2847], [717, 843], [717, 922], [717, 2157], [717, 2251], [717, 2848], [718, 719], [718, 723], [718, 778], [718, 2849], [719, 720], [719, 730], [719, 735], [719, 1009], [719, 1028], [719, 2216], [719, 2480], [719, 2593], [719, 2634], [719, 2850], [720, 1033], [720, 2410], [720, 2436], [720, 2481], [720, 2577], [720, 2813], [720, 2818], [720, 2851], [721, 732], [721, 2852], [722, 729], [722, 732], [722, 733], [722, 947], [722, 2143], [722, 2597], [722, 2728], [722, 2853], [722, 2863], [723, 724], [723, 2854], [724, 725], [724, 971], [724, 2233], [724, 2239], [724, 2337], [724, 2396], [724, 2440], [724, 2651], [724, 2849], [724, 2855], [725, 844], [725, 891], [725, 1011], [725, 2168], [725, 2337], [725, 2764], [725, 2854], [725, 2856], [726, 727], [726, 730], [726, 2733], [726, 2857], [727, 728], [727, 791], [727, 2732], [727, 2793], [727, 2858], [728, 737], [728, 790], [728, 801], [728, 821], [728, 948], [728, 961], [728, 2186], [728, 2324], [728, 2325], [728, 2585], [728, 2593], [728, 2640], [728, 2747], [728, 2748], [728, 2809], [728, 2857], [728, 2859], [728, 2922], [729, 730], [729, 1040], [729, 2156], [729, 2233], [729, 2775], [729, 2860], [730, 731], [730, 2635], [730, 2857], [730, 2861], [731, 2156], [731, 2401], [731, 2635], [731, 2862], [732, 733], [732, 2694], [732, 2787], [732, 2863], [733, 734], [733, 756], [733, 784], [733, 936], [733, 950], [733, 972], [733, 992], [733, 999], [733, 2177], [733, 2233], [733, 2292], [733, 2334], [733, 2341], [733, 2406], [733, 2449], [733, 2607], [733, 2788], [733, 2789], [733, 2852], [733, 2853], [733, 2864], [734, 744], [734, 771], [734, 782], [734, 786], [734, 816], [734, 823], [734, 860], [734, 867], [734, 916], [734, 919], [734, 978], [734, 994], [734, 1021], [734, 1060], [734, 2174], [734, 2176], [734, 2210], [734, 2270], [734, 2334], [734, 2374], [734, 2375], [734, 2397], [734, 2407], [734, 2484], [734, 2646], [734, 2695], [734, 2834], [734, 2837], [734, 2863], [734, 2865], [734, 2876], [735, 736], [735, 2596], [735, 2793], [735, 2866], [736, 737], [736, 849], [736, 856], [736, 1009], [736, 1024], [736, 1054], [736, 2146], [736, 2168], [736, 2205], [736, 2233], [736, 2277], [736, 2337], [736, 2420], [736, 2445], [736, 2470], [736, 2497], [736, 2580], [736, 2748], [736, 2756], [736, 2850], [736, 2867], [737, 756], [737, 863], [737, 864], [737, 865], [737, 977], [737, 2227], [737, 2341], [737, 2445], [737, 2523], [737, 2657], [737, 2736], [737, 2747], [737, 2749], [737, 2777], [737, 2788], [737, 2793], [737, 2858], [737, 2866], [737, 2868], [737, 3041], [738, 739], [738, 749], [738, 2869], [739, 740], [739, 824], [739, 844], [739, 862], [739, 957], [739, 1053], [739, 2175], [739, 2347], [739, 2440], [739, 2870], [740, 821], [740, 2186], [740, 2196], [740, 2651], [740, 2871], [741, 742], [741, 781], [741, 795], [741, 796], [741, 848], [741, 2174], [741, 2345], [741, 2376], [741, 2377], [741, 2449], [741, 2600], [741, 2770], [741, 2798], [741, 2864], [741, 2872], [741, 2874], [741, 3076], [741, 3077], [741, 3081], [742, 743], [742, 774], [742, 859], [742, 863], [742, 916], [742, 2203], [742, 2354], [742, 2408], [742, 2423], [742, 2473], [742, 2488], [742, 2657], [742, 2693], [742, 2799], [742, 2834], [742, 2873], [742, 3008], [743, 877], [743, 2341], [743, 2449], [743, 2585], [743, 2652], [743, 2695], [743, 2747], [743, 2874], [744, 745], [744, 854], [744, 898], [744, 1045], [744, 2177], [744, 2439], [744, 2475], [744, 2493], [744, 2517], [744, 2529], [744, 2591], [744, 2626], [744, 2646], [744, 2693], [744, 2875], [745, 2167], [745, 2876], [746, 747], [746, 787], [746, 834], [746, 836], [746, 866], [746, 2174], [746, 2189], [746, 2263], [746, 2398], [746, 2509], [746, 2526], [746, 2540], [746, 2542], [746, 2630], [746, 2701], [746, 2762], [746, 2877], [746, 3025], [746, 3193], [747, 748], [747, 819], [747, 834], [747, 881], [747, 903], [747, 932], [747, 945], [747, 2188], [747, 2363], [747, 2405], [747, 2450], [747, 2527], [747, 2565], [747, 2623], [747, 2628], [747, 2702], [747, 2878], [748, 758], [748, 797], [748, 847], [748, 850], [748, 867], [748, 922], [748, 936], [748, 1005], [748, 1006], [748, 1019], [748, 1061], [748, 2132], [748, 2134], [748, 2150], [748, 2157], [748, 2158], [748, 2234], [748, 2334], [748, 2406], [748, 2407], [748, 2415], [748, 2485], [748, 2701], [748, 2769], [748, 2797], [748, 2833], [748, 2879], [749, 913], [749, 2181], [749, 2880], [750, 751], [750, 764], [750, 766], [750, 816], [750, 875], [750, 928], [750, 2174], [750, 2270], [750, 2358], [750, 2543], [750, 2697], [750, 2750], [750, 2752], [750, 2753], [750, 2803], [750, 2805], [750, 2820], [750, 2839], [750, 2881], [750, 3098], [750, 3132], [751, 752], [751, 767], [751, 831], [751, 928], [751, 988], [751, 2215], [751, 2532], [751, 2543], [751, 2548], [751, 2549], [751, 2667], [751, 2668], [751, 2751], [751, 2790], [751, 2821], [751, 2822], [751, 2882], [751, 3119], [752, 764], [752, 780], [752, 838], [752, 899], [752, 900], [752, 929], [752, 939], [752, 945], [752, 961], [752, 981], [752, 1028], [752, 2132], [752, 2138], [752, 2216], [752, 2219], [752, 2221], [752, 2329], [752, 2450], [752, 2593], [752, 2738], [752, 2803], [752, 2806], [752, 2809], [752, 2812], [752, 2817], [752, 2820], [752, 2883], [753, 754], [753, 930], [753, 932], [753, 935], [753, 937], [753, 1025], [753, 2132], [753, 2174], [753, 2238], [753, 2348], [753, 2454], [753, 2623], [753, 2653], [753, 2676], [753, 2884], [754, 863], [754, 934], [754, 940], [754, 953], [754, 979], [754, 1002], [754, 2199], [754, 2237], [754, 2574], [754, 2652], [754, 2654], [754, 2657], [754, 2664], [754, 2778], [754, 2780], [754, 2885], [755, 879], [755, 939], [755, 942], [755, 944], [755, 993], [755, 2174], [755, 2211], [755, 2213], [755, 2242], [755, 2621], [755, 2684], [755, 2709], [755, 2711], [755, 2712], [755, 2783], [755, 2807], [755, 2814], [755, 2817], [755, 2837], [755, 2886], [755, 3133], [756, 784], [756, 885], [756, 945], [756, 948], [756, 1003], [756, 2174], [756, 2275], [756, 2404], [756, 2450], [756, 2523], [756, 2728], [756, 2748], [756, 2787], [756, 2789], [756, 2832], [756, 2863], [756, 2887], [756, 3097], [756, 3100], [757, 758], [757, 796], [757, 797], [757, 951], [757, 953], [757, 955], [757, 956], [757, 2134], [757, 2136], [757, 2174], [757, 2199], [757, 2239], [757, 2254], [757, 2651], [757, 2664], [757, 2765], [757, 2769], [757, 2770], [757, 2888], [758, 873], [758, 2133], [758, 2135], [758, 2200], [758, 2392], [758, 2405], [758, 2447], [758, 2541], [758, 2889], [759, 768], [759, 902], [759, 958], [759, 1027], [759, 2174], [759, 2217], [759, 2265], [759, 2289], [759, 2362], [759, 2372], [759, 2402], [759, 2624], [759, 2636], [759, 2735], [759, 2796], [759, 2802], [759, 2890], [759, 3137], [760, 761], [760, 770], [760, 906], [760, 959], [760, 960], [760, 2174], [760, 2223], [760, 2300], [760, 2317], [760, 2521], [760, 2639], [760, 2678], [760, 2767], [760, 2810], [760, 2891], [760, 3043], [760, 3111], [760, 3129], [760, 3130], [761, 845], [761, 854], [761, 912], [761, 987], [761, 1055], [761, 2445], [761, 2448], [761, 2517], [761, 2581], [761, 2698], [761, 2736], [761, 2758], [761, 2768], [761, 2782], [761, 2845], [761, 2892], [761, 3042], [762, 763], [762, 764], [762, 976], [762, 978], [762, 981], [762, 984], [762, 986], [762, 2174], [762, 2225], [762, 2248], [762, 2322], [762, 2329], [762, 2379], [762, 2654], [762, 2700], [762, 2766], [762, 2795], [762, 2803], [762, 2893], [763, 798], [763, 885], [763, 931], [763, 944], [763, 959], [763, 977], [763, 2201], [763, 2316], [763, 2323], [763, 2402], [763, 2654], [763, 2767], [763, 2776], [763, 2777], [763, 2807], [763, 2830], [763, 2832], [763, 2894], [764, 765], [764, 929], [764, 986], [764, 2215], [764, 2253], [764, 2323], [764, 2700], [764, 2751], [764, 2804], [764, 2806], [764, 2845], [764, 2895], [765, 816], [765, 817], [765, 845], [765, 953], [765, 956], [765, 958], [765, 969], [765, 978], [765, 979], [765, 1035], [765, 1063], [765, 2132], [765, 2199], [765, 2254], [765, 2257], [765, 2260], [765, 2264], [765, 2265], [765, 2267], [765, 2270], [765, 2373], [765, 2574], [765, 2581], [765, 2582], [765, 2755], [765, 2803], [765, 2896], [766, 767], [766, 990], [766, 1012], [766, 2174], [766, 2229], [766, 2231], [766, 2305], [766, 2364], [766, 2428], [766, 2438], [766, 2751], [766, 2821], [766, 2840], [766, 2841], [766, 2897], [766, 3004], [766, 3101], [766, 3122], [766, 3179], [767, 784], [767, 936], [767, 1047], [767, 1052], [767, 2364], [767, 2406], [767, 2425], [767, 2427], [767, 2451], [767, 2547], [767, 2552], [767, 2727], [767, 2744], [767, 2789], [767, 2820], [767, 2822], [767, 2839], [767, 2898], [767, 2951], [768, 769], [768, 887], [768, 1004], [768, 1037], [768, 2174], [768, 2266], [768, 2279], [768, 2281], [768, 2288], [768, 2290], [768, 2296], [768, 2388], [768, 2416], [768, 2899], [768, 3103], [768, 3104], [768, 3123], [768, 3190], [769, 956], [769, 958], [769, 1004], [769, 1015], [769, 1026], [769, 1034], [769, 2254], [769, 2258], [769, 2265], [769, 2278], [769, 2280], [769, 2282], [769, 2289], [769, 2416], [769, 2425], [769, 2447], [769, 2578], [769, 2900], [769, 3147], [770, 771], [770, 785], [770, 929], [770, 961], [770, 1007], [770, 2174], [770, 2247], [770, 2315], [770, 2316], [770, 2318], [770, 2319], [770, 2324], [770, 2359], [770, 2374], [770, 2768], [770, 2806], [770, 2809], [770, 2894], [770, 2901], [770, 3062], [771, 786], [771, 822], [771, 962], [771, 1007], [771, 1055], [771, 2177], [771, 2247], [771, 2317], [771, 2373], [771, 2375], [771, 2383], [771, 2475], [771, 2689], [771, 2690], [771, 2758], [771, 2782], [771, 2902], [771, 2953], [772, 773], [772, 787], [772, 974], [772, 1011], [772, 2142], [772, 2174], [772, 2239], [772, 2241], [772, 2293], [772, 2335], [772, 2336], [772, 2337], [772, 2509], [772, 2764], [772, 2826], [772, 2903], [772, 2955], [772, 3101], [772, 3141], [772, 3159], [773, 774], [773, 786], [773, 799], [773, 853], [773, 964], [773, 972], [773, 2143], [773, 2218], [773, 2240], [773, 2292], [773, 2294], [773, 2296], [773, 2354], [773, 2375], [773, 2403], [773, 2420], [773, 2458], [773, 2470], [773, 2503], [773, 2824], [773, 2827], [773, 2853], [773, 2904], [773, 2956], [773, 3008], [773, 3103], [773, 3144], [774, 794], [774, 800], [774, 962], [774, 968], [774, 1023], [774, 1024], [774, 2140], [774, 2145], [774, 2147], [774, 2203], [774, 2293], [774, 2326], [774, 2353], [774, 2383], [774, 2459], [774, 2460], [774, 2497], [774, 2530], [774, 2585], [774, 2587], [774, 2835], [774, 2859], [774, 2874], [774, 2905], [774, 2953], [774, 2956], [774, 3051], [774, 3081], [774, 3093], [774, 3197], [775, 776], [775, 777], [775, 779], [775, 849], [775, 853], [775, 862], [775, 1021], [775, 1022], [775, 1023], [775, 1025], [775, 1027], [775, 2174], [775, 2217], [775, 2277], [775, 2325], [775, 2347], [775, 2348], [775, 2353], [775, 2447], [775, 2466], [775, 2479], [775, 2519], [775, 2556], [775, 2737], [775, 2815], [775, 2824], [775, 2870], [775, 2906], [775, 2909], [775, 2952], [775, 3081], [775, 3156], [776, 777], [776, 780], [776, 838], [776, 854], [776, 1009], [776, 2190], [776, 2219], [776, 2352], [776, 2411], [776, 2439], [776, 2517], [776, 2633], [776, 2784], [776, 2812], [776, 2816], [776, 2850], [776, 2907], [776, 2909], [776, 2969], [777, 778], [777, 943], [777, 2297], [777, 2352], [777, 2425], [777, 2440], [777, 2541], [777, 2765], [777, 2783], [777, 2804], [777, 2815], [777, 2849], [777, 2908], [778, 2347], [778, 2909], [779, 780], [779, 821], [779, 915], [779, 944], [779, 959], [779, 2219], [779, 2300], [779, 2352], [779, 2523], [779, 2741], [779, 2767], [779, 2807], [779, 2835], [779, 2859], [779, 2871], [779, 2910], [779, 2951], [780, 813], [780, 883], [780, 942], [780, 1009], [780, 1052], [780, 2145], [780, 2215], [780, 2220], [780, 2222], [780, 2306], [780, 2330], [780, 2445], [780, 2451], [780, 2593], [780, 2684], [780, 2685], [780, 2815], [780, 2819], [780, 2845], [780, 2850], [780, 2859], [780, 2883], [780, 2911], [780, 2952], [780, 2996], [780, 3015], [780, 3114], [780, 3119], [780, 3161], [781, 993], [781, 1029], [781, 1048], [781, 2174], [781, 2230], [781, 2242], [781, 2244], [781, 2322], [781, 2344], [781, 2346], [781, 2369], [781, 2386], [781, 2429], [781, 2431], [781, 2799], [781, 2912], [781, 3181], [782, 784], [782, 832], [782, 880], [782, 937], [782, 1007], [782, 1041], [782, 1043], [782, 1045], [782, 1047], [782, 1049], [782, 1051], [782, 2174], [782, 2177], [782, 2238], [782, 2247], [782, 2299], [782, 2380], [782, 2396], [782, 2398], [782, 2443], [782, 2482], [782, 2547], [782, 2601], [782, 2626], [782, 2789], [782, 2831], [782, 2855], [782, 2913], [782, 2914], [782, 2964], [782, 3068], [782, 3111], [783, 1043], [783, 2396], [783, 2914], [784, 862], [784, 938], [784, 946], [784, 947], [784, 949], [784, 1000], [784, 1020], [784, 1049], [784, 2227], [784, 2397], [784, 2425], [784, 2550], [784, 2681], [784, 2788], [784, 2790], [784, 2821], [784, 2831], [784, 2849], [784, 2863], [784, 2870], [784, 2915], [784, 3069], [784, 3078], [784, 3101], [784, 3144], [785, 786], [785, 826], [785, 1052], [785, 2174], [785, 2317], [785, 2360], [785, 2375], [785, 2401], [785, 2403], [785, 2451], [785, 2455], [785, 2457], [785, 2533], [785, 2862], [785, 2916], [785, 2939], [785, 3020], [785, 3089], [785, 3113], [786, 789], [786, 1029], [786, 1042], [786, 2177], [786, 2244], [786, 2293], [786, 2343], [786, 2359], [786, 2374], [786, 2376], [786, 2455], [786, 2469], [786, 2483], [786, 2512], [786, 2516], [786, 2578], [786, 2917], [786, 3008], [787, 956], [787, 1054], [787, 1055], [787, 2174], [787, 2240], [787, 2254], [787, 2256], [787, 2259], [787, 2332], [787, 2409], [787, 2439], [787, 2478], [787, 2508], [787, 2510], [787, 2527], [787, 2580], [787, 2758], [787, 2918], [787, 2969], [788, 789], [788, 839], [788, 972], [788, 1012], [788, 1057], [788, 2174], [788, 2292], [788, 2305], [788, 2421], [788, 2423], [788, 2502], [788, 2503], [788, 2515], [788, 2516], [788, 2607], [788, 2665], [788, 2919], [788, 3109], [788, 3112], [789, 804], [789, 829], [789, 836], [789, 854], [789, 933], [789, 973], [789, 1017], [789, 1056], [789, 1058], [789, 2304], [789, 2375], [789, 2422], [789, 2476], [789, 2494], [789, 2496], [789, 2515], [789, 2517], [789, 2526], [789, 2578], [789, 2606], [789, 2782], [789, 2920], [789, 2958], [789, 3064], [789, 3113], [789, 3187], [790, 791], [790, 801], [790, 863], [790, 1023], [790, 2172], [790, 2245], [790, 2353], [790, 2437], [790, 2447], [790, 2496], [790, 2578], [790, 2657], [790, 2818], [790, 2858], [790, 2921], [791, 2652], [791, 2922], [792, 793], [792, 842], [792, 882], [792, 951], [792, 964], [792, 976], [792, 2136], [792, 2141], [792, 2145], [792, 2149], [792, 2162], [792, 2166], [792, 2170], [792, 2174], [792, 2180], [792, 2184], [792, 2187], [792, 2191], [792, 2204], [792, 2214], [792, 2218], [792, 2241], [792, 2248], [792, 2252], [792, 2263], [792, 2273], [792, 2281], [792, 2295], [792, 2302], [792, 2315], [792, 2320], [792, 2327], [792, 2346], [792, 2351], [792, 2358], [792, 2362], [792, 2368], [792, 2377], [792, 2382], [792, 2391], [792, 2395], [792, 2400], [792, 2404], [792, 2413], [792, 2428], [792, 2435], [792, 2457], [792, 2468], [792, 2484], [792, 2495], [792, 2503], [792, 2510], [792, 2521], [792, 2539], [792, 2544], [792, 2561], [792, 2567], [792, 2599], [792, 2645], [792, 2656], [792, 2660], [792, 2682], [792, 2711], [792, 2829], [792, 2905], [792, 2911], [792, 2923], [792, 2961], [792, 2976], [792, 2978], [792, 3002], [793, 794], [793, 806], [793, 829], [793, 840], [793, 849], [793, 853], [793, 860], [793, 2137], [793, 2155], [793, 2203], [793, 2208], [793, 2210], [793, 2256], [793, 2277], [793, 2298], [793, 2320], [793, 2408], [793, 2456], [793, 2460], [793, 2484], [793, 2494], [793, 2576], [793, 2608], [793, 2687], [793, 2690], [793, 2699], [793, 2743], [793, 2782], [793, 2824], [793, 2924], [793, 2991], [793, 3072], [794, 800], [794, 1062], [794, 2204], [794, 2354], [794, 2408], [794, 2413], [794, 2459], [794, 2475], [794, 2500], [794, 2507], [794, 2508], [794, 2529], [794, 2630], [794, 2925], [794, 3193], [795, 796], [795, 828], [795, 887], [795, 1004], [795, 1006], [795, 1048], [795, 2344], [795, 2371], [795, 2416], [795, 2425], [795, 2603], [795, 2693], [795, 2765], [795, 2770], [795, 2797], [795, 2799], [795, 2843], [795, 2926], [795, 3123], [796, 797], [796, 916], [796, 1050], [796, 2200], [796, 2370], [796, 2552], [796, 2769], [796, 2771], [796, 2798], [796, 2799], [796, 2834], [796, 2927], [797, 798], [797, 848], [797, 850], [797, 2200], [797, 2201], [797, 2234], [797, 2405], [797, 2600], [797, 2724], [797, 2725], [797, 2770], [797, 2928], [798, 856], [798, 871], [798, 894], [798, 907], [798, 918], [798, 934], [798, 955], [798, 975], [798, 984], [798, 1050], [798, 2132], [798, 2202], [798, 2205], [798, 2207], [798, 2209], [798, 2233], [798, 2370], [798, 2379], [798, 2402], [798, 2763], [798, 2766], [798, 2769], [798, 2772], [798, 2775], [798, 2778], [798, 2929], [799, 800], [799, 802], [799, 804], [799, 806], [799, 807], [799, 810], [799, 928], [799, 930], [799, 1015], [799, 1041], [799, 2174], [799, 2208], [799, 2293], [799, 2355], [799, 2443], [799, 2444], [799, 2453], [799, 2454], [799, 2456], [799, 2459], [799, 2476], [799, 2543], [799, 2643], [799, 2730], [799, 2822], [799, 2827], [799, 2930], [799, 2942], [799, 3021], [799, 3061], [799, 3147], [800, 801], [800, 804], [800, 857], [800, 878], [800, 2173], [800, 2354], [800, 2384], [800, 2393], [800, 2458], [800, 2460], [800, 2476], [800, 2520], [800, 2571], [800, 2595], [800, 2674], [800, 2743], [800, 2759], [800, 2931], [800, 3009], [801, 807], [801, 2393], [801, 2459], [801, 2579], [801, 2594], [801, 2666], [801, 2858], [801, 2922], [801, 2932], [802, 803], [802, 810], [802, 847], [802, 868], [802, 936], [802, 2158], [802, 2406], [802, 2427], [802, 2458], [802, 2534], [802, 2553], [802, 2683], [802, 2687], [802, 2729], [802, 2731], [802, 2853], [802, 2933], [802, 2942], [802, 3067], [803, 815], [803, 847], [803, 867], [803, 2158], [803, 2407], [803, 2484], [803, 2505], [803, 2525], [803, 2544], [803, 2562], [803, 2571], [803, 2615], [803, 2648], [803, 2659], [803, 2730], [803, 2934], [803, 2998], [804, 805], [804, 836], [804, 902], [804, 905], [804, 924], [804, 931], [804, 954], [804, 977], [804, 2324], [804, 2458], [804, 2459], [804, 2475], [804, 2477], [804, 2516], [804, 2526], [804, 2638], [804, 2693], [804, 2735], [804, 2747], [804, 2777], [804, 2935], [804, 2936], [804, 2939], [804, 3019], [804, 3062], [805, 837], [805, 1056], [805, 2476], [805, 2671], [805, 2936], [806, 850], [806, 867], [806, 906], [806, 907], [806, 924], [806, 951], [806, 1014], [806, 2136], [806, 2204], [806, 2207], [806, 2234], [806, 2255], [806, 2298], [806, 2407], [806, 2458], [806, 2500], [806, 2620], [806, 2638], [806, 2643], [806, 2650], [806, 2678], [806, 2740], [806, 2937], [807, 808], [807, 809], [807, 893], [807, 910], [807, 2458], [807, 2464], [807, 2578], [807, 2620], [807, 2729], [807, 2776], [807, 2857], [807, 2938], [808, 809], [808, 888], [808, 909], [808, 2163], [808, 2401], [808, 2463], [808, 2939], [809, 2245], [809, 2492], [809, 2590], [809, 2612], [809, 2940], [810, 811], [810, 1018], [810, 2225], [810, 2232], [810, 2445], [810, 2458], [810, 2470], [810, 2525], [810, 2724], [810, 2730], [810, 2776], [810, 2791], [810, 2941], [811, 2453], [811, 2942], [812, 814], [812, 815], [812, 817], [812, 852], [812, 1031], [812, 1035], [812, 2174], [812, 2257], [812, 2267], [812, 2367], [812, 2399], [812, 2400], [812, 2461], [812, 2466], [812, 2467], [812, 2531], [812, 2648], [812, 2662], [812, 2943], [812, 3070], [812, 3071], [813, 836], [813, 842], [813, 939], [813, 2219], [813, 2222], [813, 2243], [813, 2252], [813, 2399], [813, 2526], [813, 2542], [813, 2611], [813, 2632], [813, 2711], [813, 2817], [813, 2818], [813, 2851], [813, 2921], [813, 2944], [813, 2958], [813, 3026], [813, 3070], [814, 831], [814, 1005], [814, 1036], [814, 1038], [814, 1047], [814, 2150], [814, 2152], [814, 2414], [814, 2415], [814, 2424], [814, 2465], [814, 2532], [814, 2547], [814, 2571], [814, 2822], [814, 2945], [815, 816], [815, 817], [815, 842], [815, 868], [815, 2182], [815, 2184], [815, 2252], [815, 2267], [815, 2270], [815, 2465], [815, 2571], [815, 2584], [815, 2601], [815, 2647], [815, 2677], [815, 2683], [815, 2731], [815, 2753], [815, 2759], [815, 2946], [815, 3058], [816, 1029], [816, 1035], [816, 1055], [816, 2177], [816, 2244], [816, 2253], [816, 2257], [816, 2271], [816, 2548], [816, 2562], [816, 2648], [816, 2751], [816, 2758], [816, 2947], [817, 923], [817, 924], [817, 1014], [817, 2150], [817, 2225], [817, 2253], [817, 2255], [817, 2268], [817, 2269], [817, 2465], [817, 2477], [817, 2525], [817, 2638], [817, 2648], [817, 2948], [817, 3054], [818, 819], [818, 857], [818, 908], [818, 1037], [818, 2173], [818, 2174], [818, 2261], [818, 2262], [818, 2349], [818, 2351], [818, 2388], [818, 2499], [818, 2504], [818, 2702], [818, 2786], [818, 2949], [818, 2951], [818, 2955], [818, 3004], [818, 3038], [819, 847], [819, 854], [819, 882], [819, 1052], [819, 1064], [819, 2158], [819, 2187], [819, 2263], [819, 2287], [819, 2350], [819, 2389], [819, 2446], [819, 2451], [819, 2517], [819, 2529], [819, 2628], [819, 2668], [819, 2701], [819, 2703], [819, 2704], [819, 2714], [819, 2878], [819, 2950], [819, 3005], [819, 3020], [819, 3184], [820, 821], [820, 950], [820, 2230], [820, 2385], [820, 2499], [820, 2691], [820, 2791], [820, 2951], [821, 2198], [821, 2224], [821, 2347], [821, 2593], [821, 2791], [821, 2858], [821, 2952], [822, 1066], [822, 2324], [822, 2374], [822, 2585], [822, 2953], [823, 824], [823, 881], [823, 962], [823, 2175], [823, 2177], [823, 2363], [823, 2383], [823, 2530], [823, 2695], [823, 2836], [823, 2954], [824, 825], [824, 1021], [824, 1053], [824, 2142], [824, 2176], [824, 2499], [824, 2955], [825, 1021], [825, 2143], [825, 2555], [825, 2585], [825, 2695], [825, 2956], [826, 828], [826, 831], [826, 861], [826, 1022], [826, 2160], [826, 2174], [826, 2359], [826, 2479], [826, 2495], [826, 2511], [826, 2514], [826, 2532], [826, 2569], [826, 2603], [826, 2605], [826, 2627], [826, 2642], [826, 2709], [826, 2957], [826, 3194], [827, 982], [827, 2627], [827, 2635], [827, 2716], [827, 2795], [827, 2818], [827, 2958], [828, 829], [828, 860], [828, 886], [828, 933], [828, 1036], [828, 2159], [828, 2210], [828, 2414], [828, 2455], [828, 2478], [828, 2493], [828, 2494], [828, 2533], [828, 2604], [828, 2718], [828, 2798], [828, 2959], [829, 830], [829, 2148], [829, 2204], [829, 2423], [829, 2442], [829, 2456], [829, 2493], [829, 2495], [829, 2516], [829, 2603], [829, 2606], [829, 2643], [829, 2720], [829, 2843], [829, 2875], [829, 2959], [829, 2960], [829, 2982], [829, 3019], [830, 832], [830, 853], [830, 965], [830, 989], [830, 1022], [830, 1036], [830, 1041], [830, 2145], [830, 2152], [830, 2414], [830, 2443], [830, 2479], [830, 2494], [830, 2554], [830, 2570], [830, 2578], [830, 2589], [830, 2590], [830, 2592], [830, 2706], [830, 2720], [830, 2824], [830, 2842], [830, 2940], [830, 2961], [830, 2964], [830, 3021], [830, 3024], [830, 3161], [830, 3167], [830, 3171], [831, 832], [831, 939], [831, 1047], [831, 2213], [831, 2531], [831, 2533], [831, 2547], [831, 2610], [831, 2817], [831, 2820], [831, 2962], [831, 2964], [832, 833], [832, 875], [832, 1033], [832, 2197], [832, 2246], [832, 2397], [832, 2442], [832, 2532], [832, 2559], [832, 2577], [832, 2592], [832, 2697], [832, 2790], [832, 2963], [832, 3141], [833, 1010], [833, 2188], [833, 2245], [833, 2396], [833, 2565], [833, 2590], [833, 2591], [833, 2964], [834, 835], [834, 890], [834, 2478], [834, 2527], [834, 2588], [834, 2673], [834, 2701], [834, 2822], [834, 2965], [835, 836], [835, 890], [835, 903], [835, 2526], [835, 2579], [835, 2595], [835, 2966], [836, 837], [836, 863], [836, 903], [836, 940], [836, 1054], [836, 1064], [836, 2189], [836, 2399], [836, 2408], [836, 2445], [836, 2446], [836, 2476], [836, 2516], [836, 2525], [836, 2527], [836, 2580], [836, 2657], [836, 2819], [836, 2967], [836, 2968], [836, 3027], [836, 3071], [836, 3072], [837, 1056], [837, 2526], [837, 2630], [837, 2968], [838, 932], [838, 1025], [838, 1028], [838, 1064], [838, 2189], [838, 2215], [838, 2216], [838, 2278], [838, 2328], [838, 2348], [838, 2409], [838, 2410], [838, 2446], [838, 2623], [838, 2813], [838, 2815], [838, 2969], [839, 840], [839, 875], [839, 921], [839, 2153], [839, 2174], [839, 2284], [839, 2391], [839, 2422], [839, 2579], [839, 2608], [839, 2664], [839, 2697], [839, 2699], [839, 2718], [839, 2781], [839, 2970], [839, 3110], [840, 856], [840, 868], [840, 882], [840, 886], [840, 907], [840, 908], [840, 921], [840, 2185], [840, 2187], [840, 2204], [840, 2205], [840, 2207], [840, 2507], [840, 2530], [840, 2576], [840, 2604], [840, 2607], [840, 2665], [840, 2683], [840, 2723], [840, 2781], [840, 2786], [840, 2971], [841, 842], [841, 909], [841, 922], [841, 939], [841, 945], [841, 1004], [841, 1005], [841, 2174], [841, 2251], [841, 2252], [841, 2415], [841, 2416], [841, 2447], [841, 2450], [841, 2463], [841, 2677], [841, 2715], [841, 2817], [841, 2972], [842, 843], [842, 1029], [842, 1047], [842, 1052], [842, 2137], [842, 2166], [842, 2244], [842, 2251], [842, 2386], [842, 2408], [842, 2413], [842, 2417], [842, 2451], [842, 2474], [842, 2477], [842, 2482], [842, 2487], [842, 2514], [842, 2547], [842, 2589], [842, 2632], [842, 2648], [842, 2706], [842, 2759], [842, 2819], [842, 2848], [842, 2972], [842, 2973], [842, 3019], [843, 847], [843, 922], [843, 1017], [843, 2158], [843, 2162], [843, 2252], [843, 2304], [843, 2437], [843, 2477], [843, 2496], [843, 2671], [843, 2844], [843, 2846], [843, 2847], [843, 2974], [843, 3053], [844, 862], [844, 947], [844, 996], [844, 2178], [844, 2870], [844, 2975], [845, 846], [845, 952], [845, 990], [845, 1045], [845, 1046], [845, 1054], [845, 2145], [845, 2197], [845, 2253], [845, 2271], [845, 2432], [845, 2438], [845, 2441], [845, 2471], [845, 2491], [845, 2580], [845, 2582], [845, 2584], [845, 2592], [845, 2626], [845, 2756], [845, 2845], [845, 2896], [845, 2976], [845, 3043], [845, 3054], [845, 3102], [845, 3111], [845, 3163], [845, 3197], [846, 976], [846, 978], [846, 1054], [846, 2248], [846, 2331], [846, 2332], [846, 2340], [846, 2470], [846, 2484], [846, 2502], [846, 2525], [846, 2580], [846, 2581], [846, 2782], [846, 2977], [846, 3109], [847, 1015], [847, 1017], [847, 1020], [847, 2135], [847, 2145], [847, 2151], [847, 2157], [847, 2235], [847, 2304], [847, 2405], [847, 2425], [847, 2589], [847, 2598], [847, 2681], [847, 2702], [847, 2725], [847, 2730], [847, 2731], [847, 2848], [847, 2879], [847, 2978], [847, 3005], [847, 3007], [847, 3051], [847, 3058], [847, 3147], [847, 3163], [848, 849], [848, 856], [848, 879], [848, 1061], [848, 2159], [848, 2174], [848, 2205], [848, 2277], [848, 2485], [848, 2539], [848, 2601], [848, 2602], [848, 2606], [848, 2712], [848, 2769], [848, 2799], [848, 2959], [848, 2979], [848, 3064], [849, 850], [849, 1024], [849, 1061], [849, 2204], [849, 2234], [849, 2256], [849, 2276], [849, 2318], [849, 2352], [849, 2390], [849, 2485], [849, 2497], [849, 2523], [849, 2596], [849, 2598], [849, 2599], [849, 2600], [849, 2866], [849, 2980], [849, 3081], [850, 1014], [850, 1037], [850, 2208], [850, 2225], [850, 2233], [850, 2235], [850, 2255], [850, 2277], [850, 2388], [850, 2405], [850, 2536], [850, 2769], [850, 2981], [851, 854], [851, 892], [851, 982], [851, 983], [851, 1018], [851, 1030], [851, 2445], [851, 2493], [851, 2517], [851, 2568], [851, 2627], [851, 2723], [851, 2822], [851, 2982], [852, 853], [852, 885], [852, 924], [852, 2163], [852, 2165], [852, 2174], [852, 2465], [852, 2615], [852, 2618], [852, 2638], [852, 2654], [852, 2656], [852, 2824], [852, 2832], [852, 2939], [852, 2983], [852, 2998], [852, 3008], [852, 3027], [853, 908], [853, 998], [853, 2190], [853, 2204], [853, 2262], [853, 2293], [853, 2295], [853, 2297], [853, 2352], [853, 2442], [853, 2466], [853, 2612], [853, 2639], [853, 2662], [853, 2690], [853, 2786], [853, 2823], [853, 2825], [853, 2940], [853, 2984], [853, 3072], [853, 3129], [853, 3141], [854, 855], [854, 932], [854, 1017], [854, 1029], [854, 1045], [854, 2167], [854, 2174], [854, 2243], [854, 2244], [854, 2304], [854, 2389], [854, 2472], [854, 2516], [854, 2623], [854, 2625], [854, 2626], [854, 2632], [854, 2702], [854, 2765], [854, 2815], [854, 2816], [854, 2876], [854, 2985], [854, 3015], [854, 3043], [854, 3063], [855, 856], [855, 882], [855, 974], [855, 975], [855, 1009], [855, 2187], [855, 2205], [855, 2312], [855, 2361], [855, 2362], [855, 2517], [855, 2602], [855, 2605], [855, 2624], [855, 2634], [855, 2736], [855, 2752], [855, 2763], [855, 2826], [855, 2850], [855, 2986], [855, 3015], [856, 886], [856, 918], [856, 2146], [856, 2201], [856, 2202], [856, 2206], [856, 2464], [856, 2600], [856, 2604], [856, 2608], [856, 2625], [856, 2866], [856, 2987], [857, 859], [857, 902], [857, 967], [857, 2171], [857, 2172], [857, 2174], [857, 2327], [857, 2350], [857, 2459], [857, 2488], [857, 2508], [857, 2610], [857, 2735], [857, 2757], [857, 2759], [857, 2811], [857, 2847], [857, 2988], [857, 3065], [858, 873], [858, 2172], [858, 2583], [858, 2651], [858, 2669], [858, 2680], [858, 2989], [859, 864], [859, 1008], [859, 1019], [859, 1061], [859, 2139], [859, 2160], [859, 2173], [859, 2485], [859, 2489], [859, 2749], [859, 2833], [859, 2874], [859, 2990], [860, 894], [860, 917], [860, 918], [860, 998], [860, 1062], [860, 2155], [860, 2174], [860, 2177], [860, 2202], [860, 2204], [860, 2209], [860, 2276], [860, 2455], [860, 2603], [860, 2742], [860, 2823], [860, 2991], [861, 862], [861, 906], [861, 925], [861, 984], [861, 2174], [861, 2175], [861, 2178], [861, 2273], [861, 2379], [861, 2533], [861, 2565], [861, 2643], [861, 2647], [861, 2678], [861, 2870], [861, 2954], [861, 2992], [861, 3030], [862, 926], [862, 952], [862, 957], [862, 985], [862, 1000], [862, 1011], [862, 1022], [862, 2197], [862, 2352], [862, 2441], [862, 2479], [862, 2550], [862, 2565], [862, 2642], [862, 2644], [862, 2764], [862, 2789], [862, 2869], [862, 2871], [862, 2955], [862, 2975], [862, 2993], [862, 3088], [862, 3184], [863, 864], [863, 869], [863, 895], [863, 990], [863, 1023], [863, 1038], [863, 1054], [863, 2174], [863, 2353], [863, 2424], [863, 2438], [863, 2464], [863, 2522], [863, 2526], [863, 2580], [863, 2652], [863, 2653], [863, 2655], [863, 2658], [863, 2748], [863, 2749], [863, 2752], [863, 2773], [863, 2874], [863, 2885], [863, 2922], [863, 2994], [863, 3190], [864, 882], [864, 921], [864, 976], [864, 1003], [864, 1008], [864, 1028], [864, 1064], [864, 2139], [864, 2187], [864, 2216], [864, 2248], [864, 2275], [864, 2284], [864, 2330], [864, 2340], [864, 2446], [864, 2488], [864, 2657], [864, 2708], [864, 2748], [864, 2781], [864, 2794], [864, 2995], [865, 884], [865, 891], [865, 895], [865, 2168], [865, 2593], [865, 2692], [865, 2708], [865, 2727], [865, 2748], [865, 2775], [865, 2996], [866, 867], [866, 915], [866, 1047], [866, 2174], [866, 2221], [866, 2407], [866, 2474], [866, 2527], [866, 2544], [866, 2547], [866, 2595], [866, 2672], [866, 2740], [866, 2741], [866, 2761], [866, 2997], [866, 3009], [867, 916], [867, 936], [867, 1047], [867, 2163], [867, 2177], [867, 2208], [867, 2405], [867, 2406], [867, 2408], [867, 2500], [867, 2547], [867, 2677], [867, 2731], [867, 2762], [867, 2834], [867, 2998], [868, 869], [868, 872], [868, 887], [868, 913], [868, 935], [868, 942], [868, 1000], [868, 1052], [868, 2174], [868, 2181], [868, 2225], [868, 2318], [868, 2365], [868, 2451], [868, 2550], [868, 2558], [868, 2607], [868, 2608], [868, 2648], [868, 2676], [868, 2684], [868, 2687], [868, 2730], [868, 2773], [868, 2880], [868, 2999], [868, 3058], [868, 3066], [868, 3123], [869, 870], [869, 871], [869, 913], [869, 934], [869, 2172], [869, 2534], [869, 2641], [869, 2657], [869, 2658], [869, 2683], [869, 2688], [869, 2722], [869, 2772], [869, 2774], [869, 2778], [869, 2880], [869, 3000], [869, 3065], [870, 871], [870, 907], [870, 1038], [870, 2149], [870, 2184], [870, 2207], [870, 2424], [870, 2504], [870, 2520], [870, 2536], [870, 2571], [870, 2721], [870, 2772], [870, 2773], [870, 2869], [870, 3001], [870, 3038], [871, 880], [871, 887], [871, 977], [871, 997], [871, 1049], [871, 2145], [871, 2201], [871, 2203], [871, 2206], [871, 2298], [871, 2361], [871, 2380], [871, 2622], [871, 2693], [871, 2722], [871, 2725], [871, 2773], [871, 2775], [871, 2777], [871, 2831], [871, 2860], [871, 2929], [871, 2996], [871, 3002], [871, 3022], [871, 3024], [871, 3028], [871, 3123], [871, 3190], [872, 873], [872, 874], [872, 920], [872, 943], [872, 1065], [872, 2172], [872, 2425], [872, 2447], [872, 2507], [872, 2683], [872, 2734], [872, 2844], [872, 3003], [873, 874], [873, 2134], [873, 2229], [873, 2499], [873, 3004], [874, 2157], [874, 2541], [874, 2628], [874, 2679], [874, 3005], [875, 986], [875, 987], [875, 2174], [875, 2188], [875, 2214], [875, 2448], [875, 2665], [875, 2698], [875, 2700], [875, 2704], [875, 2751], [875, 2878], [875, 2964], [875, 3006], [875, 3021], [875, 3034], [876, 920], [876, 1061], [876, 2157], [876, 2485], [876, 2606], [876, 3007], [877, 1059], [877, 2143], [877, 2163], [877, 2371], [877, 2402], [877, 2695], [877, 3008], [878, 917], [878, 930], [878, 931], [878, 959], [878, 962], [878, 1062], [878, 2383], [878, 2385], [878, 2454], [878, 2459], [878, 2672], [878, 2673], [878, 2742], [878, 2767], [878, 3009], [879, 880], [879, 1064], [879, 2169], [879, 2170], [879, 2174], [879, 2380], [879, 2446], [879, 2600], [879, 2707], [879, 2710], [879, 2713], [879, 2714], [879, 2715], [879, 3010], [879, 3027], [879, 3042], [879, 3052], [879, 3113], [880, 976], [880, 984], [880, 990], [880, 1011], [880, 2248], [880, 2269], [880, 2273], [880, 2336], [880, 2379], [880, 2381], [880, 2397], [880, 2436], [880, 2438], [880, 2601], [880, 2712], [880, 2716], [880, 2745], [880, 2746], [880, 2764], [880, 2772], [880, 2958], [880, 3011], [880, 3028], [880, 3115], [880, 3122], [881, 882], [881, 899], [881, 960], [881, 961], [881, 1019], [881, 2174], [881, 2176], [881, 2186], [881, 2187], [881, 2364], [881, 2624], [881, 2701], [881, 2809], [881, 2810], [881, 2822], [881, 2833], [881, 3012], [882, 883], [882, 1020], [882, 2137], [882, 2149], [882, 2186], [882, 2284], [882, 2302], [882, 2363], [882, 2365], [882, 2530], [882, 2560], [882, 2605], [882, 2608], [882, 2625], [882, 2658], [882, 2668], [882, 2681], [882, 2685], [882, 2702], [882, 2749], [882, 2790], [882, 2811], [882, 2859], [882, 2871], [882, 3012], [882, 3013], [882, 3184], [883, 899], [883, 942], [883, 2178], [883, 2180], [883, 2187], [883, 2219], [883, 2549], [883, 2684], [883, 2708], [883, 2709], [883, 2723], [883, 2745], [883, 2871], [883, 3014], [883, 3030], [884, 892], [884, 904], [884, 905], [884, 911], [884, 2167], [884, 2198], [884, 2593], [884, 2634], [884, 2747], [884, 3015], [885, 886], [885, 1049], [885, 2174], [885, 2393], [885, 2395], [885, 2569], [885, 2604], [885, 2621], [885, 2662], [885, 2726], [885, 2729], [885, 2743], [885, 2766], [885, 2788], [885, 2808], [885, 2831], [885, 3016], [885, 3067], [886, 887], [886, 1039], [886, 2205], [886, 2418], [886, 2478], [886, 2603], [886, 2605], [886, 2608], [886, 2832], [886, 3017], [886, 3123], [887, 992], [887, 2289], [887, 2425], [887, 2551], [887, 2604], [887, 2647], [887, 2683], [887, 2725], [887, 2772], [887, 2798], [887, 2864], [887, 3018], [887, 3122], [888, 2251], [888, 2493], [888, 2594], [888, 2691], [888, 2729], [888, 3019], [889, 890], [889, 1016], [889, 2283], [889, 2401], [889, 2541], [889, 2628], [889, 2718], [889, 3020], [890, 910], [890, 1030], [890, 2188], [890, 2189], [890, 2453], [890, 2590], [890, 2760], [890, 3021], [891, 975], [891, 2227], [891, 2763], [891, 2775], [891, 3022], [892, 899], [892, 2236], [892, 2649], [892, 2739], [892, 2765], [892, 2822], [892, 3023], [893, 1040], [893, 1059], [893, 2236], [893, 2590], [893, 2620], [893, 2643], [893, 2666], [893, 2717], [893, 2775], [893, 3024], [894, 896], [894, 2201], [894, 2210], [894, 2298], [894, 2540], [894, 2609], [894, 2646], [894, 2765], [894, 3025], [895, 896], [895, 897], [895, 898], [895, 901], [895, 2164], [895, 2657], [895, 2693], [895, 2708], [895, 2765], [895, 2783], [895, 2818], [895, 3026], [896, 897], [896, 941], [896, 2163], [896, 2209], [896, 2707], [896, 3027], [897, 2646], [897, 2661], [897, 2716], [897, 2775], [897, 3028], [898, 1054], [898, 2168], [898, 2580], [898, 2591], [898, 2636], [898, 2719], [898, 2837], [898, 2876], [898, 3029], [899, 900], [899, 2175], [899, 2215], [899, 2363], [899, 2685], [899, 2738], [899, 2765], [899, 3030], [900, 935], [900, 942], [900, 960], [900, 987], [900, 1001], [900, 2195], [900, 2196], [900, 2215], [900, 2448], [900, 2676], [900, 2684], [900, 2739], [900, 2810], [900, 3031], [901, 955], [901, 1059], [901, 2212], [901, 2249], [901, 2371], [901, 2464], [901, 2765], [901, 3032], [902, 904], [902, 953], [902, 2173], [902, 2174], [902, 2192], [902, 2199], [902, 2266], [902, 2325], [902, 2382], [902, 2476], [902, 2671], [902, 2736], [902, 2747], [902, 3033], [902, 3040], [902, 3053], [903, 2150], [903, 2188], [903, 2526], [903, 2610], [903, 2701], [903, 2718], [903, 2747], [903, 3034], [904, 905], [904, 953], [904, 2199], [904, 2610], [904, 2735], [904, 2765], [904, 2800], [904, 2804], [904, 3035], [905, 954], [905, 2476], [905, 2595], [905, 2636], [905, 3036], [906, 907], [906, 2174], [906, 2183], [906, 2184], [906, 2195], [906, 2207], [906, 2208], [906, 2260], [906, 2301], [906, 2576], [906, 2642], [906, 2677], [906, 2768], [906, 3031], [906, 3037], [907, 934], [907, 959], [907, 2201], [907, 2208], [907, 2499], [907, 2608], [907, 2664], [907, 2678], [907, 2722], [907, 2723], [907, 2767], [907, 2778], [907, 2822], [907, 2871], [907, 3038], [908, 1000], [908, 1002], [908, 1039], [908, 2212], [908, 2349], [908, 2350], [908, 2418], [908, 2464], [908, 2550], [908, 2557], [908, 2608], [908, 2640], [908, 2704], [908, 2780], [908, 2824], [908, 2835], [908, 3039], [908, 3141], [909, 1036], [909, 2192], [909, 2245], [909, 2414], [909, 2417], [909, 2462], [909, 2464], [909, 2706], [909, 3040], [910, 966], [910, 2283], [910, 2464], [910, 2727], [910, 2793], [910, 2822], [910, 3041], [911, 912], [911, 2193], [911, 2221], [911, 2669], [911, 2707], [911, 2739], [911, 3042], [912, 2167], [912, 2189], [912, 2193], [912, 2223], [912, 2582], [912, 2804], [912, 3043], [913, 914], [913, 2193], [913, 2197], [913, 2341], [913, 2464], [913, 2683], [913, 2705], [913, 2729], [913, 2734], [913, 2773], [913, 2869], [913, 3044], [914, 1049], [914, 1051], [914, 2183], [914, 2341], [914, 2586], [914, 2686], [914, 2830], [914, 2831], [914, 2880], [914, 3045], [915, 916], [915, 917], [915, 974], [915, 2174], [915, 2198], [915, 2473], [915, 2660], [915, 2742], [915, 2744], [915, 2762], [915, 2826], [915, 2834], [915, 2952], [915, 3015], [915, 3046], [915, 3084], [915, 3085], [916, 917], [916, 1019], [916, 1050], [916, 2177], [916, 2370], [916, 2385], [916, 2407], [916, 2659], [916, 2693], [916, 2741], [916, 2742], [916, 2770], [916, 2833], [916, 2835], [916, 2874], [916, 3047], [917, 918], [917, 959], [917, 2202], [917, 2210], [917, 2384], [917, 2741], [917, 2743], [917, 2767], [917, 2779], [917, 2834], [917, 3048], [918, 962], [918, 974], [918, 2201], [918, 2203], [918, 2205], [918, 2210], [918, 2341], [918, 2383], [918, 2742], [918, 2826], [918, 3049], [919, 978], [919, 980], [919, 1051], [919, 2177], [919, 2225], [919, 2469], [919, 2575], [919, 2694], [919, 3050], [920, 950], [920, 2157], [920, 2469], [920, 2501], [920, 2507], [920, 2518], [920, 2585], [920, 2607], [920, 3051], [921, 962], [921, 987], [921, 1002], [921, 1024], [921, 2225], [921, 2276], [921, 2340], [921, 2383], [921, 2448], [921, 2497], [921, 2608], [921, 2665], [921, 2707], [921, 2749], [921, 2780], [921, 2782], [921, 3052], [922, 1006], [922, 1065], [922, 2192], [922, 2225], [922, 2405], [922, 2417], [922, 2797], [922, 2848], [922, 3053], [923, 1032], [923, 2267], [923, 2399], [923, 2582], [923, 3054], [924, 934], [924, 990], [924, 2208], [924, 2225], [924, 2267], [924, 2438], [924, 2476], [924, 2637], [924, 2639], [924, 2654], [924, 2662], [924, 2688], [924, 2693], [924, 2778], [924, 2779], [924, 3042], [924, 3055], [925, 926], [925, 927], [925, 984], [925, 2160], [925, 2196], [925, 2225], [925, 2379], [925, 2642], [925, 2724], [925, 3056], [926, 927], [926, 985], [926, 2183], [926, 2480], [926, 2870], [926, 3057], [927, 991], [927, 1032], [927, 1065], [927, 2157], [927, 2181], [927, 2182], [927, 2378], [927, 2565], [927, 3058], [928, 929], [928, 964], [928, 989], [928, 2218], [928, 2222], [928, 2319], [928, 2389], [928, 2458], [928, 2542], [928, 2544], [928, 2562], [928, 2751], [928, 2760], [928, 2761], [928, 2806], [928, 2808], [928, 2820], [928, 2842], [928, 3021], [928, 3059], [929, 944], [929, 1027], [929, 2215], [929, 2217], [929, 2220], [929, 2303], [929, 2317], [929, 2373], [929, 2543], [929, 2765], [929, 2803], [929, 2807], [929, 3060], [930, 931], [930, 936], [930, 972], [930, 1042], [930, 2237], [930, 2292], [930, 2343], [930, 2384], [930, 2406], [930, 2453], [930, 2455], [930, 2458], [930, 2761], [930, 3061], [931, 933], [931, 940], [931, 2316], [931, 2384], [931, 2454], [931, 2462], [931, 2476], [931, 2766], [931, 3062], [932, 933], [932, 1006], [932, 1063], [932, 2167], [932, 2237], [932, 2264], [932, 2517], [932, 2624], [932, 2701], [932, 2715], [932, 2797], [932, 2812], [932, 3063], [933, 940], [933, 1006], [933, 2159], [933, 2328], [933, 2516], [933, 2603], [933, 2623], [933, 2797], [933, 3064], [934, 935], [934, 2171], [934, 2201], [934, 2207], [934, 2575], [934, 2609], [934, 2638], [934, 2640], [934, 2653], [934, 2676], [934, 2747], [934, 2773], [934, 2779], [934, 3065], [935, 936], [935, 945], [935, 1001], [935, 2181], [935, 2237], [935, 2364], [935, 2406], [935, 2450], [935, 2677], [935, 2683], [935, 2738], [935, 2778], [935, 3066], [936, 2392], [936, 2405], [936, 2407], [936, 2454], [936, 2552], [936, 2676], [936, 2726], [936, 2730], [936, 2739], [936, 2779], [936, 2821], [936, 2863], [936, 3067], [937, 938], [937, 984], [937, 1042], [937, 1050], [937, 1063], [937, 2237], [937, 2239], [937, 2264], [937, 2322], [937, 2343], [937, 2370], [937, 2379], [937, 2396], [937, 2397], [937, 3068], [938, 957], [938, 1001], [938, 1019], [938, 1050], [938, 2226], [938, 2238], [938, 2370], [938, 2754], [938, 2789], [938, 2833], [938, 3069], [939, 940], [939, 1033], [939, 2215], [939, 2221], [939, 2417], [939, 2461], [939, 2532], [939, 2577], [939, 2610], [939, 2710], [939, 2715], [939, 2818], [939, 2819], [939, 3070], [940, 979], [940, 2461], [940, 2526], [940, 2574], [940, 2609], [940, 2653], [940, 2817], [940, 3071], [941, 2155], [941, 2612], [941, 2719], [941, 2776], [941, 2783], [941, 3072], [942, 943], [942, 1000], [942, 2183], [942, 2213], [942, 2219], [942, 2307], [942, 2498], [942, 2536], [942, 2550], [942, 2610], [942, 2683], [942, 2685], [942, 2710], [942, 2738], [942, 2811], [942, 3031], [942, 3073], [943, 2480], [943, 2518], [943, 2583], [943, 2610], [943, 2684], [943, 2804], [943, 2909], [943, 3074], [944, 975], [944, 1027], [944, 2217], [944, 2710], [944, 2763], [944, 2766], [944, 2806], [944, 2808], [944, 2830], [944, 2952], [944, 3075], [945, 946], [945, 948], [945, 1052], [945, 2215], [945, 2417], [945, 2449], [945, 2451], [945, 2676], [945, 2691], [945, 2701], [945, 2747], [945, 2788], [945, 3076], [946, 949], [946, 2138], [946, 2160], [946, 2230], [946, 2449], [946, 2450], [946, 2565], [946, 2789], [946, 3077], [947, 996], [947, 2728], [947, 2789], [947, 3078], [948, 949], [948, 950], [948, 2285], [948, 2393], [948, 2447], [948, 2450], [948, 2788], [948, 2858], [948, 3079], [949, 950], [949, 2429], [949, 2480], [949, 2789], [949, 3080], [950, 1060], [950, 2347], [950, 2449], [950, 2585], [950, 2596], [950, 2863], [950, 3081], [951, 952], [951, 990], [951, 997], [951, 1038], [951, 1047], [951, 2135], [951, 2137], [951, 2180], [951, 2200], [951, 2208], [951, 2213], [951, 2301], [951, 2311], [951, 2424], [951, 2438], [951, 2441], [951, 2444], [951, 2547], [951, 2551], [951, 2622], [951, 2651], [951, 2725], [951, 2740], [951, 2771], [951, 2829], [951, 2855], [951, 2871], [951, 2888], [951, 2989], [951, 3082], [952, 956], [952, 2136], [952, 2254], [952, 2280], [952, 2398], [952, 2440], [952, 2510], [952, 2581], [952, 2584], [952, 2644], [952, 2647], [952, 2838], [952, 2855], [952, 2870], [952, 2908], [952, 3083], [952, 3087], [952, 3144], [952, 3152], [953, 954], [953, 984], [953, 990], [953, 2198], [953, 2200], [953, 2224], [953, 2253], [953, 2379], [953, 2438], [953, 2653], [953, 2669], [953, 2735], [953, 3084], [954, 2198], [954, 2199], [954, 2221], [954, 2373], [954, 2476], [954, 2515], [954, 2830], [954, 3085], [955, 997], [955, 1050], [955, 2133], [955, 2200], [955, 2201], [955, 2370], [955, 2464], [955, 2622], [955, 3086], [956, 957], [956, 1014], [956, 2200], [956, 2239], [956, 2253], [956, 2255], [956, 2260], [956, 2279], [956, 2387], [956, 2440], [956, 2441], [956, 2509], [956, 2837], [956, 3087], [957, 1001], [957, 1025], [957, 1026], [957, 2254], [957, 2282], [957, 2348], [957, 2387], [957, 2836], [957, 2870], [957, 3088], [958, 1027], [958, 1029], [958, 1046], [958, 1063], [958, 2217], [958, 2244], [958, 2253], [958, 2264], [958, 2266], [958, 2279], [958, 2322], [958, 2401], [958, 2447], [958, 2477], [958, 2491], [958, 2844], [958, 3089], [959, 960], [959, 2207], [959, 2384], [959, 2445], [959, 2520], [959, 2739], [959, 2742], [959, 2766], [959, 2768], [959, 2810], [959, 2822], [959, 2835], [959, 2952], [959, 3090], [960, 961], [960, 987], [960, 2363], [960, 2448], [960, 2640], [960, 2738], [960, 2767], [960, 2768], [960, 2809], [960, 2811], [960, 3091], [961, 962], [961, 1008], [961, 2139], [961, 2215], [961, 2317], [961, 2363], [961, 2383], [961, 2810], [961, 2858], [961, 2859], [961, 3092], [962, 967], [962, 972], [962, 1002], [962, 1008], [962, 1062], [962, 2132], [962, 2139], [962, 2146], [962, 2176], [962, 2202], [962, 2230], [962, 2292], [962, 2354], [962, 2374], [962, 2384], [962, 2585], [962, 2757], [962, 2780], [962, 2781], [962, 2809], [962, 3093], [963, 964], [963, 966], [963, 967], [963, 969], [963, 972], [963, 1034], [963, 1035], [963, 2144], [963, 2174], [963, 2218], [963, 2257], [963, 2258], [963, 2292], [963, 2349], [963, 2464], [963, 2755], [963, 2757], [963, 2761], [963, 3094], [964, 965], [964, 968], [964, 1027], [964, 1029], [964, 2137], [964, 2144], [964, 2191], [964, 2217], [964, 2244], [964, 2259], [964, 2293], [964, 2326], [964, 2366], [964, 2372], [964, 2420], [964, 2431], [964, 2543], [964, 2551], [964, 2556], [964, 2561], [964, 2570], [964, 2592], [964, 2756], [964, 2790], [964, 2808], [964, 2816], [964, 3094], [964, 3095], [964, 3144], [965, 966], [965, 1022], [965, 2149], [965, 2218], [965, 2442], [965, 2479], [965, 2568], [965, 2569], [965, 2705], [965, 2728], [965, 2790], [965, 2857], [965, 2869], [965, 3096], [965, 3097], [966, 1026], [966, 1036], [966, 2282], [966, 2366], [966, 2414], [966, 2570], [966, 2787], [966, 2822], [966, 3097], [967, 968], [967, 987], [967, 1001], [967, 1055], [967, 2173], [967, 2230], [967, 2326], [967, 2349], [967, 2366], [967, 2383], [967, 2393], [967, 2448], [967, 2734], [967, 2750], [967, 2758], [967, 3098], [968, 1000], [968, 2218], [968, 2246], [968, 2325], [968, 2327], [968, 2354], [968, 2394], [968, 2498], [968, 2550], [968, 2556], [968, 2587], [968, 2734], [968, 2757], [968, 2805], [968, 3003], [968, 3044], [968, 3098], [968, 3099], [968, 3184], [969, 970], [969, 2193], [969, 2225], [969, 2253], [969, 2366], [969, 2754], [969, 2756], [969, 2787], [969, 3100], [970, 971], [970, 1013], [970, 1051], [970, 2142], [970, 2229], [970, 2755], [970, 3101], [971, 1051], [971, 2193], [971, 2582], [971, 2669], [971, 2854], [971, 3102], [972, 973], [972, 998], [972, 1057], [972, 2143], [972, 2288], [972, 2293], [972, 2366], [972, 2383], [972, 2422], [972, 2454], [972, 2823], [972, 2863], [972, 3103], [973, 1058], [973, 2146], [973, 2189], [973, 2288], [973, 2292], [973, 2402], [973, 2478], [973, 2516], [973, 3104], [974, 975], [974, 1008], [974, 2139], [974, 2141], [974, 2174], [974, 2202], [974, 2226], [974, 2227], [974, 2240], [974, 2625], [974, 2741], [974, 2763], [974, 2827], [974, 2828], [974, 3069], [974, 3105], [974, 3150], [975, 1011], [975, 2138], [975, 2201], [975, 2310], [975, 2311], [975, 2625], [975, 2764], [975, 2765], [975, 2807], [975, 2826], [975, 2849], [975, 2871], [975, 3022], [975, 3106], [976, 977], [976, 1007], [976, 1027], [976, 2137], [976, 2162], [976, 2217], [976, 2247], [976, 2269], [976, 2323], [976, 2330], [976, 2333], [976, 2380], [976, 2468], [976, 2471], [976, 2477], [976, 2489], [976, 2519], [976, 2655], [976, 2749], [976, 2777], [976, 2782], [976, 2795], [976, 2802], [976, 2814], [976, 2845], [976, 2893], [976, 2958], [976, 3107], [977, 2228], [977, 2248], [977, 2324], [977, 2361], [977, 2395], [977, 2476], [977, 2520], [977, 2655], [977, 2736], [977, 2748], [977, 2766], [977, 2772], [977, 2776], [977, 2894], [977, 2938], [977, 2941], [977, 3072], [977, 3108], [978, 979], [978, 2177], [978, 2253], [978, 2323], [978, 2421], [978, 2471], [978, 2574], [978, 3109], [979, 986], [979, 1033], [979, 1054], [979, 2153], [979, 2154], [979, 2253], [979, 2575], [979, 2577], [979, 2580], [979, 2653], [979, 2700], [979, 3110], [980, 994], [980, 995], [980, 1010], [980, 1066], [980, 2223], [980, 2224], [980, 2321], [980, 2396], [980, 2582], [980, 3111], [981, 982], [981, 2215], [981, 2323], [981, 2328], [981, 2330], [981, 2421], [981, 2791], [981, 2822], [981, 3112], [982, 983], [982, 2329], [982, 2401], [982, 2707], [982, 3113], [983, 2593], [983, 2631], [983, 2691], [983, 2791], [983, 3114], [984, 985], [984, 2199], [984, 2201], [984, 2238], [984, 2323], [984, 2378], [984, 2380], [984, 2515], [984, 2642], [984, 2716], [984, 3115], [985, 2224], [985, 2233], [985, 2334], [985, 2378], [985, 2379], [985, 2837], [985, 2870], [985, 3116], [986, 987], [986, 1033], [986, 2323], [986, 2448], [986, 2574], [986, 2577], [986, 2697], [986, 2803], [986, 2813], [986, 2814], [986, 3117], [987, 2197], [987, 2225], [987, 2445], [987, 2498], [987, 2575], [987, 2697], [987, 2700], [987, 2738], [987, 2757], [987, 2781], [987, 2810], [987, 3043], [987, 3118], [988, 1030], [988, 2593], [988, 2805], [988, 2820], [988, 3119], [989, 1041], [989, 1048], [989, 2180], [989, 2231], [989, 2303], [989, 2344], [989, 2346], [989, 2442], [989, 2443], [989, 2543], [989, 2549], [989, 2562], [989, 2616], [989, 2843], [989, 3120], [989, 3179], [990, 2136], [990, 2165], [990, 2199], [990, 2272], [990, 2308], [990, 2311], [990, 2380], [990, 2382], [990, 2437], [990, 2581], [990, 2638], [990, 2657], [990, 2669], [990, 2744], [990, 2752], [990, 2839], [990, 2989], [990, 3042], [990, 3084], [990, 3102], [990, 3121], [991, 992], [991, 2229], [991, 2278], [991, 2334], [991, 2646], [991, 2716], [991, 3122], [992, 2181], [992, 2288], [992, 2646], [992, 2724], [992, 2775], [992, 2863], [992, 3123], [993, 994], [993, 998], [993, 1000], [993, 2133], [993, 2174], [993, 2224], [993, 2243], [993, 2246], [993, 2295], [993, 2345], [993, 2550], [993, 2614], [993, 2710], [993, 2823], [993, 2889], [993, 3086], [993, 3124], [994, 995], [994, 998], [994, 2177], [994, 2242], [994, 2591], [994, 2823], [994, 2837], [994, 2852], [994, 3125], [995, 999], [995, 1000], [995, 2183], [995, 2429], [995, 2550], [995, 3126], [996, 1000], [996, 2550], [996, 2614], [996, 3127], [997, 1049], [997, 2136], [997, 2164], [997, 2191], [997, 2297], [997, 2551], [997, 2614], [997, 2620], [997, 2621], [997, 2772], [997, 2831], [997, 2849], [997, 3086], [997, 3128], [998, 999], [998, 1025], [998, 1036], [998, 2189], [998, 2210], [998, 2223], [998, 2242], [998, 2292], [998, 2348], [998, 2414], [998, 2612], [998, 2824], [998, 3129], [999, 1000], [999, 2223], [999, 2260], [999, 2393], [999, 2548], [999, 2550], [999, 2823], [999, 2863], [999, 3130], [1000, 1001], [1000, 2224], [1000, 2242], [1000, 2326], [1000, 2549], [1000, 2551], [1000, 2683], [1000, 2684], [1000, 2786], [1000, 2789], [1000, 2805], [1000, 2835], [1000, 2870], [1000, 2989], [1000, 3004], [1000, 3127], [1000, 3131], [1000, 3132], [1001, 2392], [1001, 2550], [1001, 2676], [1001, 2738], [1001, 2750], [1001, 2757], [1001, 3132], [1002, 1023], [1002, 1025], [1002, 2211], [1002, 2224], [1002, 2283], [1002, 2348], [1002, 2353], [1002, 2383], [1002, 2640], [1002, 2653], [1002, 2781], [1002, 2786], [1002, 2813], [1002, 2836], [1002, 3133], [1003, 1028], [1003, 1031], [1003, 2174], [1003, 2216], [1003, 2274], [1003, 2276], [1003, 2333], [1003, 2334], [1003, 2367], [1003, 2480], [1003, 2560], [1003, 2599], [1003, 2749], [1003, 2788], [1003, 3134], [1003, 3157], [1004, 1005], [1004, 1006], [1004, 2279], [1004, 2289], [1004, 2415], [1004, 2417], [1004, 2512], [1004, 2514], [1004, 2797], [1004, 2798], [1004, 3135], [1005, 1036], [1005, 1037], [1005, 2388], [1005, 2405], [1005, 2414], [1005, 2416], [1005, 2417], [1005, 2531], [1005, 2588], [1005, 2589], [1005, 3136], [1006, 1017], [1006, 2278], [1006, 2304], [1006, 2405], [1006, 2416], [1006, 2623], [1006, 2796], [1006, 2798], [1006, 2800], [1006, 3137], [1007, 1008], [1007, 1046], [1007, 2139], [1007, 2246], [1007, 2248], [1007, 2303], [1007, 2317], [1007, 2321], [1007, 2322], [1007, 2346], [1007, 2374], [1007, 2397], [1007, 2466], [1007, 2489], [1007, 2491], [1007, 2587], [1007, 2828], [1007, 3111], [1007, 3138], [1008, 2138], [1008, 2140], [1008, 2247], [1008, 2312], [1008, 2383], [1008, 2447], [1008, 2488], [1008, 2749], [1008, 2809], [1008, 2826], [1008, 3139], [1009, 1028], [1009, 2191], [1009, 2216], [1009, 2219], [1009, 2312], [1009, 2337], [1009, 2599], [1009, 2625], [1009, 2815], [1009, 2849], [1009, 2851], [1009, 2861], [1009, 2866], [1009, 3140], [1009, 3159], [1010, 2142], [1010, 2245], [1010, 2260], [1010, 2575], [1010, 2612], [1010, 3141], [1011, 1027], [1011, 1063], [1011, 2197], [1011, 2217], [1011, 2236], [1011, 2239], [1011, 2240], [1011, 2264], [1011, 2296], [1011, 2380], [1011, 2410], [1011, 2763], [1011, 2765], [1011, 2856], [1011, 2870], [1011, 3122], [1011, 3142], [1012, 1014], [1012, 1017], [1012, 1019], [1012, 2174], [1012, 2232], [1012, 2255], [1012, 2304], [1012, 2332], [1012, 2342], [1012, 2422], [1012, 2489], [1012, 2682], [1012, 2771], [1012, 2830], [1012, 2833], [1012, 2839], [1012, 2847], [1012, 3143], [1012, 3173], [1013, 2143], [1013, 2144], [1013, 2232], [1013, 2440], [1013, 2586], [1013, 3144], [1014, 1015], [1014, 2208], [1014, 2234], [1014, 2254], [1014, 2256], [1014, 2267], [1014, 2305], [1014, 2837], [1014, 3145], [1014, 3147], [1015, 1016], [1015, 1031], [1015, 2158], [1015, 2255], [1015, 2279], [1015, 2290], [1015, 2367], [1015, 2458], [1015, 2477], [1015, 2542], [1015, 2578], [1015, 2589], [1015, 3020], [1015, 3146], [1016, 2157], [1016, 2378], [1016, 2453], [1016, 2515], [1016, 2541], [1016, 2588], [1016, 3147], [1017, 1018], [1017, 2158], [1017, 2160], [1017, 2303], [1017, 2305], [1017, 2489], [1017, 2514], [1017, 2516], [1017, 2517], [1017, 2636], [1017, 2797], [1017, 2801], [1017, 2843], [1017, 2848], [1017, 3137], [1017, 3148], [1018, 2160], [1018, 2304], [1018, 2309], [1018, 2595], [1018, 2635], [1018, 2724], [1018, 2942], [1018, 3149], [1019, 1020], [1019, 2160], [1019, 2226], [1019, 2305], [1019, 2334], [1019, 2363], [1019, 2364], [1019, 2405], [1019, 2488], [1019, 2680], [1019, 2681], [1019, 2834], [1019, 3150], [1020, 2158], [1020, 2161], [1020, 2187], [1020, 2227], [1020, 2365], [1020, 2523], [1020, 2598], [1020, 2607], [1020, 2659], [1020, 2680], [1020, 2682], [1020, 2789], [1020, 2833], [1020, 2989], [1020, 3150], [1020, 3151], [1020, 3174], [1020, 3182], [1021, 1044], [1021, 1060], [1021, 1066], [1021, 2177], [1021, 2331], [1021, 2352], [1021, 2439], [1021, 2440], [1021, 2500], [1021, 2835], [1021, 3152], [1022, 1026], [1022, 1044], [1022, 2259], [1022, 2280], [1022, 2282], [1022, 2286], [1022, 2312], [1022, 2352], [1022, 2442], [1022, 2478], [1022, 2480], [1022, 2533], [1022, 2570], [1022, 2605], [1022, 2870], [1022, 3153], [1022, 3157], [1023, 1024], [1023, 2308], [1023, 2352], [1023, 2354], [1023, 2497], [1023, 2519], [1023, 2557], [1023, 2641], [1023, 2657], [1023, 2780], [1023, 2783], [1023, 2784], [1023, 2922], [1023, 3133], [1023, 3154], [1024, 2162], [1024, 2169], [1024, 2277], [1024, 2338], [1024, 2353], [1024, 2354], [1024, 2391], [1024, 2496], [1024, 2498], [1024, 2536], [1024, 2781], [1024, 2866], [1024, 3052], [1024, 3155], [1025, 1026], [1025, 2237], [1025, 2276], [1025, 2282], [1025, 2347], [1025, 2349], [1025, 2352], [1025, 2780], [1025, 2812], [1025, 2823], [1025, 3156], [1026, 1034], [1026, 1036], [1026, 2258], [1026, 2274], [1026, 2279], [1026, 2283], [1026, 2285], [1026, 2348], [1026, 2414], [1026, 2479], [1026, 3157], [1027, 1028], [1027, 2216], [1027, 2218], [1027, 2248], [1027, 2265], [1027, 2266], [1027, 2325], [1027, 2331], [1027, 2333], [1027, 2352], [1027, 2733], [1027, 2756], [1027, 2764], [1027, 2806], [1027, 2807], [1027, 2857], [1027, 3158], [1028, 1033], [1028, 2142], [1028, 2215], [1028, 2217], [1028, 2275], [1028, 2464], [1028, 2577], [1028, 2624], [1028, 2749], [1028, 2790], [1028, 2812], [1028, 2850], [1028, 2857], [1028, 3159], [1029, 1048], [1029, 1055], [1029, 2218], [1029, 2243], [1029, 2250], [1029, 2252], [1029, 2265], [1029, 2270], [1029, 2344], [1029, 2345], [1029, 2372], [1029, 2375], [1029, 2517], [1029, 2568], [1029, 2706], [1029, 2758], [1029, 2844], [1029, 3160], [1030, 2212], [1030, 2243], [1030, 2309], [1030, 2568], [1030, 2590], [1030, 2593], [1030, 2613], [1030, 3161], [1031, 1033], [1031, 1034], [1031, 2174], [1031, 2258], [1031, 2275], [1031, 2368], [1031, 2378], [1031, 2436], [1031, 2465], [1031, 2559], [1031, 2577], [1031, 3058], [1031, 3115], [1031, 3116], [1031, 3147], [1031, 3162], [1032, 1051], [1032, 2157], [1032, 2398], [1032, 2528], [1032, 2582], [1032, 2629], [1032, 2686], [1032, 2846], [1032, 3163], [1033, 1034], [1033, 2197], [1033, 2216], [1033, 2258], [1033, 2283], [1033, 2367], [1033, 2464], [1033, 2574], [1033, 2578], [1033, 2700], [1033, 2817], [1033, 2851], [1033, 2964], [1033, 3164], [1034, 1035], [1034, 2257], [1034, 2259], [1034, 2279], [1034, 2282], [1034, 2366], [1034, 2367], [1034, 2410], [1034, 2577], [1034, 3165], [1035, 1036], [1035, 2253], [1035, 2258], [1035, 2270], [1035, 2366], [1035, 2414], [1035, 2465], [1035, 2591], [1035, 2592], [1035, 3166], [1036, 1039], [1036, 1042], [1036, 1048], [1036, 2132], [1036, 2257], [1036, 2282], [1036, 2343], [1036, 2344], [1036, 2415], [1036, 2418], [1036, 2442], [1036, 2463], [1036, 2531], [1036, 2548], [1036, 2590], [1036, 2603], [1036, 2718], [1036, 2823], [1036, 3167], [1037, 1038], [1037, 1039], [1037, 2174], [1037, 2234], [1037, 2289], [1037, 2350], [1037, 2387], [1037, 2389], [1037, 2390], [1037, 2415], [1037, 2418], [1037, 2424], [1037, 2644], [1037, 2645], [1037, 3087], [1037, 3088], [1037, 3168], [1038, 1039], [1038, 2136], [1038, 2213], [1038, 2388], [1038, 2391], [1038, 2418], [1038, 2423], [1038, 2467], [1038, 2531], [1038, 2536], [1038, 2657], [1038, 2663], [1038, 2664], [1038, 2720], [1038, 2722], [1038, 2827], [1038, 3169], [1038, 3190], [1039, 1040], [1039, 2233], [1039, 2388], [1039, 2414], [1039, 2419], [1039, 2420], [1039, 2424], [1039, 2464], [1039, 2551], [1039, 2604], [1039, 2786], [1039, 2869], [1039, 3170], [1039, 3171], [1040, 2418], [1040, 2590], [1040, 2644], [1040, 3171], [1041, 1042], [1041, 1043], [1041, 2232], [1041, 2343], [1041, 2397], [1041, 2442], [1041, 2444], [1041, 2458], [1041, 2483], [1041, 2513], [1041, 2553], [1041, 2746], [1041, 2842], [1041, 2914], [1041, 3172], [1041, 3173], [1042, 1048], [1042, 2238], [1042, 2342], [1042, 2344], [1042, 2375], [1042, 2385], [1042, 2414], [1042, 2443], [1042, 2454], [1042, 2462], [1042, 2512], [1042, 2515], [1042, 2552], [1042, 3173], [1043, 1044], [1043, 2397], [1043, 2443], [1043, 2549], [1043, 2588], [1043, 2627], [1043, 2680], [1043, 2695], [1043, 2745], [1043, 2822], [1043, 2835], [1043, 3174], [1044, 2429], [1044, 2478], [1044, 2479], [1044, 2481], [1044, 2588], [1044, 2597], [1044, 2914], [1044, 3175], [1045, 1046], [1045, 1063], [1045, 2264], [1045, 2287], [1045, 2397], [1045, 2411], [1045, 2482], [1045, 2491], [1045, 2513], [1045, 2517], [1045, 2581], [1045, 2627], [1045, 2856], [1045, 2876], [1045, 3176], [1045, 3194], [1046, 2247], [1046, 2265], [1046, 2280], [1046, 2303], [1046, 2312], [1046, 2362], [1046, 2413], [1046, 2475], [1046, 2490], [1046, 2581], [1046, 2626], [1046, 2862], [1046, 3089], [1046, 3177], [1047, 1048], [1047, 2135], [1047, 2136], [1047, 2164], [1047, 2252], [1047, 2344], [1047, 2386], [1047, 2397], [1047, 2398], [1047, 2407], [1047, 2531], [1047, 2532], [1047, 2546], [1047, 2762], [1047, 2821], [1047, 2846], [1047, 3178], [1048, 2164], [1048, 2229], [1048, 2244], [1048, 2343], [1048, 2345], [1048, 2414], [1048, 2547], [1048, 2551], [1048, 2761], [1048, 2765], [1048, 2798], [1048, 2842], [1048, 3179], [1049, 1050], [1049, 2370], [1049, 2397], [1049, 2429], [1049, 2430], [1049, 2562], [1049, 2622], [1049, 2659], [1049, 2771], [1049, 2772], [1049, 2789], [1049, 2808], [1049, 2830], [1049, 2832], [1049, 3180], [1049, 3181], [1050, 2201], [1050, 2238], [1050, 2369], [1050, 2371], [1050, 2770], [1050, 2831], [1050, 2834], [1050, 2852], [1050, 3181], [1051, 2197], [1051, 2225], [1051, 2232], [1051, 2340], [1051, 2397], [1051, 2680], [1051, 2846], [1051, 3182], [1052, 2219], [1052, 2252], [1052, 2318], [1052, 2359], [1052, 2376], [1052, 2404], [1052, 2450], [1052, 2452], [1052, 2487], [1052, 2683], [1052, 2691], [1052, 2702], [1052, 2705], [1052, 2774], [1052, 2821], [1052, 2841], [1052, 2951], [1052, 3019], [1052, 3076], [1052, 3114], [1052, 3183], [1053, 2186], [1053, 2439], [1053, 2597], [1053, 2628], [1053, 2734], [1053, 3184], [1054, 2256], [1054, 2433], [1054, 2471], [1054, 2509], [1054, 2526], [1054, 2574], [1054, 2579], [1054, 2581], [1054, 2657], [1054, 2814], [1054, 2837], [1054, 2851], [1054, 2866], [1054, 3110], [1054, 3185], [1055, 2244], [1055, 2270], [1055, 2374], [1055, 2393], [1055, 2509], [1055, 2757], [1055, 2759], [1055, 3043], [1055, 3186], [1056, 2502], [1056, 2516], [1056, 3187], [1057, 1058], [1057, 1059], [1057, 2154], [1057, 2292], [1057, 2341], [1057, 2422], [1057, 2464], [1057, 2830], [1057, 3188], [1058, 1059], [1058, 2516], [1058, 2579], [1058, 2636], [1058, 3189], [1059, 2288], [1059, 2515], [1059, 2652], [1059, 2663], [1059, 2775], [1059, 3190], [1060, 1062], [1060, 2177], [1060, 2385], [1060, 2447], [1060, 2528], [1060, 2545], [1060, 3191], [1061, 1064], [1061, 2146], [1061, 2277], [1061, 2405], [1061, 2408], [1061, 2446], [1061, 2447], [1061, 2486], [1061, 2487], [1061, 2488], [1061, 2496], [1061, 2600], [1061, 3007], [1061, 3192], [1062, 2210], [1062, 2383], [1062, 2384], [1062, 2447], [1062, 2460], [1062, 2540], [1062, 3193], [1063, 2238], [1063, 2253], [1063, 2265], [1063, 2371], [1063, 2410], [1063, 2511], [1063, 2512], [1063, 2565], [1063, 2623], [1063, 2626], [1063, 2754], [1063, 2764], [1063, 3194], [1064, 2436], [1064, 2445], [1064, 2447], [1064, 2485], [1064, 2526], [1064, 2528], [1064, 2633], [1064, 2702], [1064, 2712], [1064, 2715], [1064, 2749], [1064, 2812], [1064, 2813], [1064, 3020], [1064, 3195], [1065, 2225], [1065, 2278], [1065, 2447], [1065, 2637], [1065, 2732], [1065, 3196], [1066, 2325], [1066, 2331], [1066, 2481], [1066, 2582], [1066, 2585], [1066, 2637], [1066, 2670], [1066, 3197], [1067, 1068], [1067, 1145], [1067, 1176], [1067, 1179], [1067, 1183], [1067, 1303], [1067, 1548], [1067, 3198], [1068, 1069], [1068, 1134], [1068, 1326], [1068, 1339], [1068, 1381], [1068, 1475], [1068, 1823], [1068, 3199], [1069, 1070], [1069, 1092], [1069, 1237], [1069, 1346], [1069, 1347], [1069, 1371], [1069, 1434], [1069, 1481], [1069, 1548], [1069, 1823], [1069, 3200], [1070, 1071], [1070, 1114], [1070, 1134], [1070, 1142], [1070, 1147], [1070, 1235], [1070, 1245], [1070, 1358], [1070, 1372], [1070, 1375], [1070, 1378], [1070, 1481], [1070, 1485], [1070, 1556], [1070, 1585], [1070, 1659], [1070, 1674], [1070, 1705], [1070, 1763], [1070, 1789], [1070, 1805], [1070, 1822], [1070, 1923], [1070, 2016], [1070, 3201], [1071, 1075], [1071, 1079], [1071, 1083], [1071, 1096], [1071, 1100], [1071, 1104], [1071, 1108], [1071, 1114], [1071, 1118], [1071, 1121], [1071, 1125], [1071, 1138], [1071, 1148], [1071, 1152], [1071, 1175], [1071, 1182], [1071, 1186], [1071, 1197], [1071, 1207], [1071, 1215], [1071, 1229], [1071, 1236], [1071, 1249], [1071, 1254], [1071, 1261], [1071, 1280], [1071, 1285], [1071, 1292], [1071, 1296], [1071, 1302], [1071, 1311], [1071, 1316], [1071, 1325], [1071, 1329], [1071, 1334], [1071, 1338], [1071, 1347], [1071, 1362], [1071, 1369], [1071, 1391], [1071, 1402], [1071, 1418], [1071, 1429], [1071, 1437], [1071, 1444], [1071, 1455], [1071, 1473], [1071, 1478], [1071, 1495], [1071, 1501], [1071, 1533], [1071, 1579], [1071, 1590], [1071, 1594], [1071, 1616], [1071, 1645], [1071, 1763], [1071, 1839], [1071, 1845], [1071, 1857], [1071, 1895], [1071, 1910], [1071, 1912], [1071, 1936], [1071, 3202], [1072, 1073], [1072, 1094], [1072, 1098], [1072, 1130], [1072, 1149], [1072, 1158], [1072, 1217], [1072, 1233], [1072, 1237], [1072, 1240], [1072, 1243], [1072, 1246], [1072, 1330], [1072, 1344], [1072, 1475], [1072, 1483], [1072, 1613], [1072, 1642], [1072, 1661], [1072, 1697], [1072, 1783], [1072, 3203], [1073, 1074], [1073, 1181], [1073, 1246], [1073, 1317], [1073, 1381], [1073, 1422], [1073, 1683], [1073, 1743], [1073, 1760], [1073, 2073], [1073, 3204], [1074, 1075], [1074, 1228], [1074, 1240], [1074, 1288], [1074, 1347], [1074, 1793], [1074, 2014], [1074, 2073], [1074, 3205], [1075, 1137], [1075, 1161], [1075, 1175], [1075, 1594], [1075, 1760], [1075, 1802], [1075, 1849], [1075, 1956], [1075, 2039], [1075, 2085], [1075, 3206], [1076, 1077], [1076, 1150], [1076, 1160], [1076, 1174], [1076, 1265], [1076, 1268], [1076, 1271], [1076, 1274], [1076, 1343], [1076, 3207], [1077, 1078], [1077, 1226], [1077, 1227], [1077, 1265], [1077, 1387], [1077, 1435], [1077, 1436], [1077, 1519], [1077, 1546], [1077, 1791], [1077, 3208], [1078, 1079], [1078, 1102], [1078, 1152], [1078, 1184], [1078, 1300], [1078, 1305], [1078, 1344], [1078, 1359], [1078, 1489], [1078, 1525], [1078, 1661], [1078, 1667], [1078, 1668], [1078, 1694], [1078, 1786], [1078, 3209], [1079, 1081], [1079, 1085], [1079, 1089], [1079, 1090], [1079, 1092], [1079, 1103], [1079, 1106], [1079, 1108], [1079, 1112], [1079, 1117], [1079, 1120], [1079, 1131], [1079, 1153], [1079, 1161], [1079, 1165], [1079, 1166], [1079, 1185], [1079, 1230], [1079, 1240], [1079, 1258], [1079, 1271], [1079, 1288], [1079, 1310], [1079, 1333], [1079, 1359], [1079, 1363], [1079, 1366], [1079, 1369], [1079, 1370], [1079, 1373], [1079, 1376], [1079, 1379], [1079, 1414], [1079, 1436], [1079, 1438], [1079, 1488], [1079, 1512], [1079, 1513], [1079, 1515], [1079, 1529], [1079, 1540], [1079, 1548], [1079, 1549], [1079, 1561], [1079, 1564], [1079, 1570], [1079, 1573], [1079, 1578], [1079, 1585], [1079, 1605], [1079, 1627], [1079, 1638], [1079, 1662], [1079, 1663], [1079, 1678], [1079, 1706], [1079, 1717], [1079, 1729], [1079, 1739], [1079, 1769], [1079, 1794], [1079, 1796], [1079, 3210], [1080, 1081], [1080, 1101], [1080, 1139], [1080, 1179], [1080, 1246], [1080, 1317], [1080, 1403], [1080, 1406], [1080, 1409], [1080, 1412], [1080, 1415], [1080, 1419], [1080, 1426], [1080, 1430], [1080, 1441], [1080, 1509], [1080, 1607], [1080, 1681], [1080, 1747], [1080, 1778], [1080, 1800], [1080, 3211], [1081, 1082], [1081, 1140], [1081, 1288], [1081, 1406], [1081, 1424], [1081, 1608], [1081, 1708], [1081, 1718], [1081, 1719], [1081, 1941], [1081, 3212], [1082, 1083], [1082, 1324], [1082, 1328], [1082, 1360], [1082, 1412], [1082, 1428], [1082, 1444], [1082, 1578], [1082, 1890], [1082, 2118], [1082, 3213], [1083, 1086], [1083, 1121], [1083, 1264], [1083, 1295], [1083, 1361], [1083, 1454], [1083, 1469], [1083, 1504], [1083, 1630], [1083, 1656], [1083, 1708], [1083, 1726], [1083, 1756], [1083, 2053], [1083, 3214], [1084, 1085], [1084, 1201], [1084, 1274], [1084, 1319], [1084, 1339], [1084, 1430], [1084, 1449], [1084, 1452], [1084, 1456], [1084, 1459], [1084, 1462], [1084, 1465], [1084, 1470], [1084, 1505], [1084, 1544], [1084, 1565], [1084, 1586], [1084, 1713], [1084, 1725], [1084, 1734], [1084, 1780], [1084, 3215], [1085, 1086], [1085, 1092], [1085, 1272], [1085, 1456], [1085, 1468], [1085, 1469], [1085, 1545], [1085, 1680], [1085, 1735], [1085, 1988], [1085, 3216], [1086, 1334], [1086, 1376], [1086, 1465], [1086, 1523], [1086, 1759], [1086, 1879], [1086, 1900], [1086, 3217], [1087, 1088], [1087, 1119], [1087, 1122], [1087, 1217], [1087, 1355], [1087, 1508], [1087, 1513], [1087, 1599], [1087, 3218], [1088, 1089], [1088, 1123], [1088, 1262], [1088, 1307], [1088, 1367], [1088, 1508], [1088, 1512], [1088, 1633], [1088, 3219], [1089, 1119], [1089, 1138], [1089, 1144], [1089, 1275], [1089, 1379], [1089, 1426], [1089, 1427], [1089, 1462], [1089, 1509], [1089, 1530], [1089, 1546], [1089, 1584], [1089, 1620], [1089, 1713], [1089, 3220], [1090, 1091], [1090, 1516], [1090, 1519], [1090, 1524], [1090, 1527], [1090, 1709], [1090, 3221], [1091, 1092], [1091, 1339], [1091, 1522], [1091, 1531], [1091, 1562], [1091, 1614], [1091, 1658], [1091, 3222], [1092, 1169], [1092, 1238], [1092, 1339], [1092, 1359], [1092, 1523], [1092, 1532], [1092, 1615], [1092, 1636], [1092, 1659], [1092, 1664], [1092, 1665], [1092, 1782], [1092, 1813], [1092, 1912], [1092, 1939], [1092, 1941], [1092, 1985], [1092, 1992], [1092, 2081], [1092, 2097], [1092, 3223], [1093, 1094], [1093, 1383], [1093, 1530], [1093, 1534], [1093, 1537], [1093, 1540], [1093, 1543], [1093, 1641], [1093, 3224], [1094, 1095], [1094, 1159], [1094, 1238], [1094, 1414], [1094, 1422], [1094, 1467], [1094, 1536], [1094, 1544], [1094, 1568], [1094, 1626], [1094, 1627], [1094, 1642], [1094, 1688], [1094, 1767], [1094, 3225], [1095, 1096], [1095, 1240], [1095, 1429], [1095, 1536], [1095, 1540], [1095, 1545], [1095, 1551], [1095, 1615], [1095, 1892], [1095, 2048], [1095, 3226], [1096, 1128], [1096, 1159], [1096, 1169], [1096, 1182], [1096, 1242], [1096, 1431], [1096, 1432], [1096, 1458], [1096, 1469], [1096, 1521], [1096, 1622], [1096, 1690], [1096, 1782], [1096, 1988], [1096, 3227], [1097, 1098], [1097, 1155], [1097, 1341], [1097, 1395], [1097, 1546], [1097, 1549], [1097, 1554], [1097, 1596], [1097, 1660], [1097, 3228], [1098, 1099], [1098, 1155], [1098, 1278], [1098, 1481], [1098, 1525], [1098, 1548], [1098, 1550], [1098, 1556], [1098, 1603], [1098, 1627], [1098, 1640], [1098, 1651], [1098, 1661], [1098, 3229], [1099, 1100], [1099, 1206], [1099, 1229], [1099, 1233], [1099, 1372], [1099, 1526], [1099, 1550], [1099, 1552], [1099, 1596], [1099, 1742], [1099, 1960], [1099, 2062], [1099, 2112], [1099, 2113], [1099, 3230], [1100, 1186], [1100, 1306], [1100, 1506], [1100, 1507], [1100, 1536], [1100, 1551], [1100, 1627], [1100, 1736], [1100, 1807], [1100, 1809], [1100, 1860], [1100, 1869], [1100, 1936], [1100, 1960], [1100, 1981], [1100, 1989], [1100, 3231], [1101, 1102], [1101, 1183], [1101, 1184], [1101, 1451], [1101, 1557], [1101, 1562], [1101, 1565], [1101, 1568], [1101, 3232], [1102, 1103], [1102, 1167], [1102, 1353], [1102, 1750], [1102, 1800], [1102, 3233], [1103, 1104], [1103, 1431], [1103, 1470], [1103, 1540], [1103, 1565], [1103, 1641], [1103, 1646], [1103, 1654], [1103, 1655], [1103, 1717], [1103, 3234], [1104, 1471], [1104, 1473], [1104, 1566], [1104, 1567], [1104, 1645], [1104, 1646], [1104, 1750], [1104, 1944], [1104, 2089], [1104, 3235], [1105, 1106], [1105, 1107], [1105, 1126], [1105, 1433], [1105, 1571], [1105, 1574], [1105, 1712], [1105, 3236], [1106, 1107], [1106, 1261], [1106, 1438], [1106, 1575], [1106, 1605], [1106, 1668], [1106, 1707], [1106, 1856], [1106, 3237], [1107, 1108], [1107, 1261], [1107, 1284], [1107, 1393], [1107, 1422], [1107, 1442], [1107, 1544], [1107, 1669], [1107, 1691], [1107, 1693], [1107, 1745], [1107, 1781], [1107, 1922], [1107, 1999], [1107, 3238], [1108, 1111], [1108, 1134], [1108, 1144], [1108, 1159], [1108, 1171], [1108, 1174], [1108, 1176], [1108, 1200], [1108, 1209], [1108, 1223], [1108, 1233], [1108, 1239], [1108, 1251], [1108, 1257], [1108, 1279], [1108, 1284], [1108, 1286], [1108, 1289], [1108, 1293], [1108, 1297], [1108, 1300], [1108, 1301], [1108, 1322], [1108, 1331], [1108, 1351], [1108, 1356], [1108, 1381], [1108, 1392], [1108, 1398], [1108, 1399], [1108, 1406], [1108, 1443], [1108, 1451], [1108, 1456], [1108, 1461], [1108, 1467], [1108, 1492], [1108, 1501], [1108, 1534], [1108, 1576], [1108, 1591], [1108, 1596], [1108, 1599], [1108, 1612], [1108, 1617], [1108, 1631], [1108, 1644], [1108, 1646], [1108, 1669], [1108, 1671], [1108, 1675], [1108, 1685], [1108, 1696], [1108, 1699], [1108, 1702], [1108, 1722], [1108, 1733], [1108, 1756], [1108, 1760], [1108, 1766], [1108, 1773], [1108, 1813], [1108, 1817], [1108, 1818], [1108, 1830], [1108, 1863], [1108, 2027], [1108, 2101], [1108, 3239], [1109, 1110], [1109, 1112], [1109, 1129], [1109, 1445], [1109, 1576], [1109, 1580], [1109, 1583], [1109, 3240], [1110, 1111], [1110, 1297], [1110, 1317], [1110, 1464], [1110, 1629], [1110, 1770], [1110, 1888], [1110, 3241], [1111, 1144], [1111, 1204], [1111, 1268], [1111, 1308], [1111, 1309], [1111, 1331], [1111, 1341], [1111, 1418], [1111, 1580], [1111, 1629], [1111, 1768], [1111, 1771], [1111, 1797], [1111, 1799], [1111, 1810], [1111, 3242], [1112, 1113], [1112, 1117], [1112, 1207], [1112, 1464], [1112, 1561], [1112, 1576], [1112, 1619], [1112, 1650], [1112, 1651], [1112, 1657], [1112, 3243], [1113, 1114], [1113, 1232], [1113, 1366], [1113, 1418], [1113, 1447], [1113, 1580], [1113, 1581], [1113, 3244], [1114, 1154], [1114, 1225], [1114, 1232], [1114, 1346], [1114, 1545], [1114, 1619], [1114, 1654], [1114, 1680], [1114, 1699], [1114, 1776], [1114, 1777], [1114, 1790], [1114, 1956], [1114, 3245], [1115, 1116], [1115, 1119], [1115, 1482], [1115, 1610], [1115, 1613], [1115, 1617], [1115, 1620], [1115, 1625], [1115, 1628], [1115, 3246], [1116, 1117], [1116, 1185], [1116, 1395], [1116, 1517], [1116, 1543], [1116, 1544], [1116, 1571], [1116, 1580], [1116, 1582], [1116, 1611], [1116, 3247], [1117, 1118], [1117, 1129], [1117, 1241], [1117, 1517], [1117, 1573], [1117, 1612], [1117, 1618], [1117, 1771], [1117, 3248], [1118, 1203], [1118, 1207], [1118, 1401], [1118, 1455], [1118, 1518], [1118, 1582], [1118, 1612], [1118, 1656], [1118, 1971], [1118, 1979], [1118, 1991], [1118, 2007], [1118, 2060], [1118, 3249], [1119, 1120], [1119, 1509], [1119, 1542], [1119, 1628], [1119, 3250], [1120, 1121], [1120, 1131], [1120, 1170], [1120, 1217], [1120, 1219], [1120, 1297], [1120, 1562], [1120, 1568], [1120, 1574], [1120, 1600], [1120, 1613], [1120, 1614], [1120, 1728], [1120, 1792], [1120, 3251], [1121, 1218], [1121, 1236], [1121, 1297], [1121, 1299], [1121, 1464], [1121, 1494], [1121, 1539], [1121, 1542], [1121, 1559], [1121, 1592], [1121, 1602], [1121, 1615], [1121, 1619], [1121, 1636], [1121, 1683], [1121, 1724], [1121, 1745], [1121, 1793], [1121, 1805], [1121, 1946], [1121, 1947], [1121, 2118], [1121, 3252], [1122, 1123], [1122, 1631], [1122, 1635], [1122, 1638], [1122, 1684], [1122, 3253], [1123, 1124], [1123, 1155], [1123, 1177], [1123, 1307], [1123, 1398], [1123, 1460], [1123, 1461], [1123, 1525], [1123, 1632], [1123, 1746], [1123, 1757], [1123, 3254], [1124, 1125], [1124, 1156], [1124, 1197], [1124, 1231], [1124, 1366], [1124, 1367], [1124, 1632], [1124, 1638], [1124, 1749], [1124, 1758], [1124, 1902], [1124, 3255], [1125, 1128], [1125, 1140], [1125, 1152], [1125, 1156], [1125, 1162], [1125, 1398], [1125, 1491], [1125, 1553], [1125, 1556], [1125, 1640], [1125, 1719], [1125, 1784], [1125, 1785], [1125, 2105], [1125, 3256], [1126, 1127], [1126, 1397], [1126, 1605], [1126, 1666], [1126, 1669], [1126, 1730], [1126, 3257], [1127, 1128], [1127, 1159], [1127, 1167], [1127, 1398], [1127, 1568], [1127, 1574], [1127, 1666], [1127, 1670], [1127, 1689], [1127, 1710], [1127, 1727], [1127, 1814], [1127, 3258], [1128, 1575], [1128, 1605], [1128, 1670], [1128, 1690], [1128, 1794], [1128, 3259], [1129, 1130], [1129, 1157], [1129, 1509], [1129, 1584], [1129, 1612], [1129, 1672], [1129, 3260], [1130, 1131], [1130, 1158], [1130, 1235], [1130, 1241], [1130, 1585], [1130, 1672], [1130, 1770], [1130, 3261], [1131, 1236], [1131, 1246], [1131, 1259], [1131, 1273], [1131, 1274], [1131, 1382], [1131, 1509], [1131, 1511], [1131, 1515], [1131, 1544], [1131, 1563], [1131, 1698], [1131, 1804], [1131, 1814], [1131, 1898], [1131, 1992], [1131, 3262], [1132, 1133], [1132, 1160], [1132, 1606], [1132, 1675], [1132, 1678], [1132, 1681], [1132, 3263], [1133, 1134], [1133, 1158], [1133, 1187], [1133, 1313], [1133, 1372], [1133, 1587], [1133, 1603], [1133, 1669], [1133, 2018], [1133, 3264], [1134, 1173], [1134, 1188], [1134, 1585], [1134, 1598], [1134, 1699], [1134, 1703], [1134, 1704], [1134, 1822], [1134, 3265], [1135, 1136], [1135, 1139], [1135, 1141], [1135, 1143], [1135, 1167], [1135, 1304], [1135, 1313], [1135, 1336], [1135, 1697], [1135, 1700], [1135, 1703], [1135, 1706], [1135, 1709], [1135, 1712], [1135, 1863], [1135, 3266], [1136, 1137], [1136, 1139], [1136, 1144], [1136, 1275], [1136, 1317], [1136, 1676], [1136, 1760], [1136, 1983], [1136, 3267], [1137, 1138], [1137, 1140], [1137, 1270], [1137, 1275], [1137, 1288], [1137, 1472], [1137, 1677], [1137, 1706], [1137, 1798], [1137, 1802], [1137, 1808], [1137, 1978], [1137, 1979], [1137, 1983], [1137, 2122], [1137, 3268], [1138, 1142], [1138, 1144], [1138, 1190], [1138, 1211], [1138, 1232], [1138, 1254], [1138, 1342], [1138, 1390], [1138, 1394], [1138, 1418], [1138, 1428], [1138, 1510], [1138, 1542], [1138, 1621], [1138, 1624], [1138, 1633], [1138, 1677], [1138, 1716], [1138, 1758], [1138, 1858], [1138, 1925], [1138, 2006], [1138, 3269], [1139, 1140], [1139, 1398], [1139, 1534], [1139, 1538], [1139, 1542], [1139, 1559], [1139, 1800], [1139, 1921], [1139, 3270], [1140, 1315], [1140, 1473], [1140, 1706], [1140, 1921], [1140, 2123], [1140, 3271], [1141, 1142], [1141, 1433], [1141, 1542], [1141, 1598], [1141, 1612], [1141, 1656], [1141, 1657], [1141, 1701], [1141, 1712], [1141, 1756], [1141, 1805], [1141, 1972], [1141, 3272], [1142, 1168], [1142, 1189], [1142, 1232], [1142, 1341], [1142, 1392], [1142, 1434], [1142, 1554], [1142, 1572], [1142, 1577], [1142, 1584], [1142, 1612], [1142, 1674], [1142, 1871], [1142, 3273], [1143, 1144], [1143, 1232], [1143, 1474], [1143, 1543], [1143, 1580], [1143, 1699], [1143, 1959], [1143, 3274], [1144, 1210], [1144, 1389], [1144, 1537], [1144, 1676], [1144, 1757], [1144, 1925], [1144, 3275], [1145, 1146], [1145, 1641], [1145, 1644], [1145, 1714], [1145, 1717], [1145, 1747], [1145, 3276], [1146, 1147], [1146, 1170], [1146, 1231], [1146, 1482], [1146, 1547], [1146, 1585], [1146, 1597], [1146, 1638], [1146, 1657], [1146, 1661], [1146, 1720], [1146, 3277], [1147, 1148], [1147, 1358], [1147, 1466], [1147, 1492], [1147, 1539], [1147, 1618], [1147, 1644], [1147, 1648], [1147, 1686], [1147, 1742], [1147, 1966], [1147, 1973], [1147, 2095], [1147, 3278], [1148, 1292], [1148, 1325], [1148, 1432], [1148, 1631], [1148, 1638], [1148, 1748], [1148, 1884], [1148, 1940], [1148, 1973], [1148, 3279], [1149, 1150], [1149, 1153], [1149, 1155], [1149, 1263], [1149, 1384], [1149, 1527], [1149, 1672], [1149, 1737], [1149, 1740], [1149, 1743], [1149, 1746], [1149, 1751], [1149, 1754], [1149, 1817], [1149, 3280], [1150, 1151], [1150, 1209], [1150, 1398], [1150, 1511], [1150, 1558], [1150, 1683], [1150, 1724], [1150, 1746], [1150, 1784], [1150, 1791], [1150, 2093], [1150, 3281], [1151, 1152], [1151, 1182], [1151, 1199], [1151, 1200], [1151, 1259], [1151, 1265], [1151, 1267], [1151, 1286], [1151, 1667], [1151, 1690], [1151, 1698], [1151, 1740], [1151, 1741], [1151, 1791], [1151, 2092], [1151, 3282], [1152, 1178], [1152, 1193], [1152, 1227], [1152, 1260], [1152, 1300], [1152, 1306], [1152, 1354], [1152, 1365], [1152, 1477], [1152, 1485], [1152, 1490], [1152, 1495], [1152, 1504], [1152, 1526], [1152, 1690], [1152, 1724], [1152, 1742], [1152, 1750], [1152, 2028], [1152, 2029], [1152, 2078], [1152, 3283], [1153, 1154], [1153, 1156], [1153, 1240], [1153, 1264], [1153, 1379], [1153, 1385], [1153, 1527], [1153, 1618], [1153, 1619], [1153, 1749], [1153, 1753], [1153, 1779], [1153, 1784], [1153, 1793], [1153, 1817], [1153, 1845], [1153, 1886], [1153, 1930], [1153, 1949], [1153, 2048], [1153, 2053], [1153, 2095], [1153, 3284], [1154, 1249], [1154, 1366], [1154, 1386], [1154, 1740], [1154, 1779], [1154, 1970], [1154, 1994], [1154, 3285], [1155, 1156], [1155, 1336], [1155, 1398], [1155, 1528], [1155, 1547], [1155, 1552], [1155, 1555], [1155, 1652], [1155, 1695], [1155, 1696], [1155, 1751], [1155, 3286], [1156, 1295], [1156, 1477], [1156, 1478], [1156, 1528], [1156, 1549], [1156, 1550], [1156, 1552], [1156, 1601], [1156, 1654], [1156, 1686], [1156, 1753], [1156, 1866], [1156, 1953], [1156, 3287], [1157, 1158], [1157, 1250], [1157, 1573], [1157, 1702], [1157, 1757], [1157, 3288], [1158, 1159], [1158, 1164], [1158, 1176], [1158, 1234], [1158, 1242], [1158, 1259], [1158, 1484], [1158, 1714], [1158, 1738], [1158, 3289], [1159, 1164], [1159, 1168], [1159, 1201], [1159, 1257], [1159, 1382], [1159, 1458], [1159, 1572], [1159, 1617], [1159, 1689], [1159, 1715], [1159, 1725], [1159, 1876], [1159, 3290], [1160, 1161], [1160, 1568], [1160, 1760], [1160, 1764], [1160, 1767], [1160, 1770], [1160, 3291], [1161, 1162], [1161, 1271], [1161, 1275], [1161, 1615], [1161, 1678], [1161, 1682], [1161, 1723], [1161, 1760], [1161, 2036], [1161, 3292], [1162, 1366], [1162, 1440], [1162, 1616], [1162, 1711], [1162, 1762], [1162, 1764], [1162, 1772], [1162, 2036], [1162, 2078], [1162, 3293], [1163, 1164], [1163, 1165], [1163, 1276], [1163, 1278], [1163, 1603], [1163, 1684], [1163, 1773], [1163, 1778], [1163, 1780], [1163, 3294], [1164, 1194], [1164, 1256], [1164, 1268], [1164, 1279], [1164, 1317], [1164, 1520], [1164, 1521], [1164, 1604], [1164, 1691], [1164, 1774], [1164, 1778], [1164, 1781], [1164, 3295], [1165, 1166], [1165, 1237], [1165, 1362], [1165, 1424], [1165, 1521], [1165, 1613], [1165, 1620], [1165, 1739], [1165, 1773], [1165, 1776], [1165, 3296], [1166, 1239], [1166, 1276], [1166, 1377], [1166, 1436], [1166, 1487], [1166, 1614], [1166, 1616], [1166, 1876], [1166, 3297], [1167, 1168], [1167, 1170], [1167, 1281], [1167, 1352], [1167, 1470], [1167, 1554], [1167, 1574], [1167, 1671], [1167, 1673], [1167, 1771], [1167, 1783], [1167, 1786], [1167, 1788], [1167, 1791], [1167, 1794], [1167, 1797], [1167, 1800], [1167, 1803], [1167, 3298], [1168, 1169], [1168, 1189], [1168, 1211], [1168, 1322], [1168, 1339], [1168, 1470], [1168, 1703], [1168, 1915], [1168, 3299], [1169, 1579], [1169, 1637], [1169, 1659], [1169, 1794], [1169, 1915], [1169, 1991], [1169, 3300], [1170, 1230], [1170, 1445], [1170, 1539], [1170, 1568], [1170, 1651], [1170, 1661], [1170, 1679], [1170, 1698], [1170, 1791], [1170, 3301], [1171, 1172], [1171, 1282], [1171, 1388], [1171, 1557], [1171, 1587], [1171, 1610], [1171, 1818], [1171, 3302], [1172, 1173], [1172, 1198], [1172, 1256], [1172, 1277], [1172, 1304], [1172, 1313], [1172, 1330], [1172, 1331], [1172, 2002], [1172, 3303], [1173, 1174], [1173, 1188], [1173, 1698], [1173, 1788], [1173, 1789], [1173, 3304], [1174, 1175], [1174, 1227], [1174, 1269], [1174, 1270], [1174, 1271], [1174, 1443], [1174, 1698], [1174, 1760], [1174, 1837], [1174, 1889], [1174, 2035], [1174, 2075], [1174, 2093], [1174, 3305], [1175, 1271], [1175, 1416], [1175, 1444], [1175, 1789], [1175, 1790], [1175, 1801], [1175, 1837], [1175, 2074], [1175, 3306], [1176, 1177], [1176, 1180], [1176, 1229], [1176, 1279], [1176, 1484], [1176, 1548], [1176, 1644], [1176, 1757], [1176, 1823], [1176, 2020], [1176, 2058], [1176, 3307], [1177, 1178], [1177, 1180], [1177, 1183], [1177, 1237], [1177, 1291], [1177, 1451], [1177, 1476], [1177, 1512], [1177, 1525], [1177, 1738], [1177, 1739], [1177, 1753], [1177, 1966], [1177, 3308], [1178, 1184], [1178, 1186], [1178, 1199], [1178, 1204], [1178, 1278], [1178, 1279], [1178, 1306], [1178, 1309], [1178, 1451], [1178, 1502], [1178, 1640], [1178, 1692], [1178, 1778], [1178, 2094], [1178, 3309], [1179, 1180], [1179, 1183], [1179, 1255], [1179, 1381], [1179, 1397], [1179, 1398], [1179, 1666], [1179, 1668], [1179, 1719], [1179, 1747], [1179, 1856], [1179, 3310], [1180, 1181], [1180, 1259], [1180, 1260], [1180, 1406], [1180, 1500], [1180, 1553], [1180, 1719], [1180, 1748], [1180, 1855], [1180, 1874], [1180, 1898], [1180, 1974], [1180, 2075], [1180, 3311], [1181, 1182], [1181, 1237], [1181, 1251], [1181, 1255], [1181, 1256], [1181, 1280], [1181, 1308], [1181, 1331], [1181, 1400], [1181, 1423], [1181, 1425], [1181, 1521], [1181, 1762], [1181, 2045], [1181, 2072], [1181, 3312], [1182, 1203], [1182, 1257], [1182, 1264], [1182, 1267], [1182, 1314], [1182, 1402], [1182, 1405], [1182, 1411], [1182, 1423], [1182, 1453], [1182, 1589], [1182, 1683], [1182, 1711], [1182, 1716], [1182, 1729], [1182, 1736], [1182, 1748], [1182, 1779], [1182, 1827], [1182, 1892], [1182, 2041], [1182, 3313], [1183, 1184], [1183, 1475], [1183, 1752], [1183, 3314], [1184, 1185], [1184, 1303], [1184, 1305], [1184, 3315], [1185, 1186], [1185, 1319], [1185, 1351], [1185, 1415], [1185, 1446], [1185, 1480], [1185, 1522], [1185, 1565], [1185, 1571], [1185, 1607], [1185, 1625], [1185, 1627], [1185, 1752], [1185, 3316], [1186, 1320], [1186, 1342], [1186, 1347], [1186, 1351], [1186, 1385], [1186, 1408], [1186, 1411], [1186, 1416], [1186, 1421], [1186, 1448], [1186, 1481], [1186, 1523], [1186, 1566], [1186, 1582], [1186, 1640], [1186, 1693], [1186, 1753], [1186, 1782], [1186, 1906], [1186, 1907], [1186, 1953], [1186, 3317], [1187, 1188], [1187, 1191], [1187, 1194], [1187, 1198], [1187, 1199], [1187, 1201], [1187, 1204], [1187, 1307], [1187, 1508], [1187, 1515], [1187, 1516], [1187, 1689], [1187, 1737], [1187, 1830], [1187, 3318], [1188, 1189], [1188, 1194], [1188, 1213], [1188, 1321], [1188, 1374], [1188, 1375], [1188, 1443], [1188, 1771], [1188, 2021], [1188, 3319], [1189, 1190], [1189, 1201], [1189, 1239], [1189, 1771], [1189, 2079], [1189, 2081], [1189, 3320], [1190, 1211], [1190, 1270], [1190, 1302], [1190, 1400], [1190, 1423], [1190, 1443], [1190, 1456], [1190, 1462], [1190, 1514], [1190, 1693], [1190, 2097], [1190, 2125], [1190, 2129], [1190, 3321], [1191, 1192], [1191, 1204], [1191, 1300], [1191, 1348], [1191, 1399], [1191, 1525], [1191, 1526], [1191, 2100], [1191, 3322], [1192, 1193], [1192, 1213], [1192, 1216], [1192, 1300], [1192, 1301], [1192, 1344], [1192, 1511], [1192, 2099], [1192, 3323], [1193, 1214], [1193, 1233], [1193, 1302], [1193, 1344], [1193, 1345], [1193, 1413], [1193, 1443], [1193, 1476], [1193, 1526], [1193, 1785], [1193, 1903], [1193, 2076], [1193, 2099], [1193, 2128], [1193, 3324], [1194, 1195], [1194, 1268], [1194, 1381], [1194, 1433], [1194, 1434], [1194, 1442], [1194, 1499], [1194, 1517], [1194, 1518], [1194, 1563], [1194, 1611], [1194, 1612], [1194, 3325], [1195, 1196], [1195, 1269], [1195, 1284], [1195, 1493], [1195, 1500], [1195, 1518], [1195, 1774], [1195, 1830], [1195, 1971], [1195, 2021], [1195, 2064], [1195, 2075], [1195, 3326], [1196, 1197], [1196, 1284], [1196, 1289], [1196, 1434], [1196, 1439], [1196, 1523], [1196, 1693], [1196, 1758], [1196, 1759], [1196, 1763], [1196, 1859], [1196, 1871], [1196, 1932], [1196, 2086], [1196, 3327], [1197, 1386], [1197, 1444], [1197, 1461], [1197, 1478], [1197, 1564], [1197, 1636], [1197, 1811], [1197, 1859], [1197, 1902], [1197, 3328], [1198, 1199], [1198, 1305], [1198, 1344], [1198, 1445], [1198, 1446], [1198, 1499], [1198, 1557], [1198, 1560], [1198, 1688], [1198, 1698], [1198, 2128], [1198, 3329], [1199, 1200], [1199, 1213], [1199, 1256], [1199, 1335], [1199, 1381], [1199, 1411], [1199, 1425], [1199, 1778], [1199, 2023], [1199, 3330], [1200, 1223], [1200, 1296], [1200, 1306], [1200, 1336], [1200, 1558], [1200, 1570], [1200, 1669], [1200, 1730], [1200, 1736], [1200, 1824], [1200, 2071], [1200, 3331], [1201, 1202], [1201, 1203], [1201, 1399], [1201, 1411], [1201, 1459], [1201, 1572], [1201, 1582], [1201, 1882], [1201, 1988], [1201, 3332], [1202, 1203], [1202, 1370], [1202, 1400], [1202, 1452], [1202, 1518], [1202, 1573], [1202, 3333], [1203, 1314], [1203, 1401], [1203, 1470], [1203, 1984], [1203, 2045], [1203, 3334], [1204, 1205], [1204, 1482], [1204, 1496], [1204, 1582], [1204, 1685], [1204, 1692], [1204, 1881], [1204, 3335], [1205, 1206], [1205, 1292], [1205, 1418], [1205, 1488], [1205, 1515], [1205, 1526], [1205, 1881], [1205, 2060], [1205, 3336], [1206, 1207], [1206, 1299], [1206, 1372], [1206, 1592], [1206, 1593], [1206, 2061], [1206, 3337], [1207, 1221], [1207, 1314], [1207, 1429], [1207, 1576], [1207, 1648], [1207, 1909], [1207, 1926], [1207, 1948], [1207, 3338], [1208, 1209], [1208, 1212], [1208, 1216], [1208, 1219], [1208, 1312], [1208, 1414], [1208, 1721], [1208, 1728], [1208, 3339], [1209, 1210], [1209, 1267], [1209, 1268], [1209, 1301], [1209, 1414], [1209, 1494], [1209, 1533], [1209, 1683], [1209, 1722], [1209, 2068], [1209, 2091], [1209, 3340], [1210, 1211], [1210, 1268], [1210, 1282], [1210, 1530], [1210, 1715], [1210, 3341], [1211, 1252], [1211, 1286], [1211, 1324], [1211, 1419], [1211, 1431], [1211, 1457], [1211, 1530], [1211, 1532], [1211, 1533], [1211, 1534], [1211, 1800], [1211, 1914], [1211, 2015], [1211, 3342], [1212, 1213], [1212, 1262], [1212, 1267], [1212, 1307], [1212, 1359], [1212, 1667], [1212, 1729], [1212, 1731], [1212, 1746], [1212, 3343], [1213, 1214], [1213, 1216], [1213, 1223], [1213, 1350], [1213, 1359], [1213, 1381], [1213, 1512], [1213, 1834], [1213, 2081], [1213, 3344], [1214, 1215], [1214, 1228], [1214, 1347], [1214, 1360], [1214, 1368], [1214, 1375], [1214, 1378], [1214, 1413], [1214, 1425], [1214, 1448], [1214, 1834], [1214, 3345], [1215, 1223], [1215, 1230], [1215, 1296], [1215, 1448], [1215, 1539], [1215, 1579], [1215, 1833], [1215, 1838], [1215, 2076], [1215, 3346], [1216, 1217], [1216, 1219], [1216, 1282], [1216, 1348], [1216, 1413], [1216, 2091], [1216, 3347], [1217, 1218], [1217, 1511], [1217, 1512], [1217, 1714], [1217, 1728], [1217, 1770], [1217, 3348], [1218, 1233], [1218, 1402], [1218, 1599], [1218, 1643], [1218, 1683], [1218, 1954], [1218, 1975], [1218, 2067], [1218, 2091], [1218, 2098], [1218, 3349], [1219, 1220], [1219, 1359], [1219, 1482], [1219, 1494], [1219, 1499], [1219, 1688], [1219, 3350], [1220, 1221], [1220, 1360], [1220, 1413], [1220, 1414], [1220, 1488], [1220, 1494], [1220, 3351], [1221, 1347], [1221, 1370], [1221, 1463], [1221, 1488], [1221, 1493], [1221, 1499], [1221, 1518], [1221, 1560], [1221, 1636], [1221, 1909], [1221, 3352], [1222, 1223], [1222, 1226], [1222, 1230], [1222, 1321], [1222, 1449], [1222, 1730], [1222, 3353], [1223, 1224], [1223, 1230], [1223, 1322], [1223, 1350], [1223, 1833], [1223, 2037], [1223, 2038], [1223, 2057], [1223, 2124], [1223, 3354], [1224, 1225], [1224, 1449], [1224, 1456], [1224, 1507], [1224, 1647], [1224, 1659], [1224, 1680], [1224, 1736], [1224, 1853], [1224, 1854], [1224, 2019], [1224, 2049], [1224, 2081], [1224, 2107], [1224, 2124], [1224, 3355], [1225, 1476], [1225, 1501], [1225, 1581], [1225, 1647], [1225, 1687], [1225, 1699], [1225, 1822], [1225, 1842], [1225, 1860], [1225, 1919], [1225, 1957], [1225, 1959], [1225, 1960], [1225, 1964], [1225, 1966], [1225, 1969], [1225, 1994], [1225, 2040], [1225, 2076], [1225, 2113], [1225, 3356], [1226, 1227], [1226, 1300], [1226, 1317], [1226, 1356], [1226, 1388], [1226, 1757], [1226, 1797], [1226, 2037], [1226, 3357], [1227, 1228], [1227, 1230], [1227, 1288], [1227, 1309], [1227, 1337], [1227, 1354], [1227, 1392], [1227, 1404], [1227, 1437], [1227, 1758], [1227, 1761], [1227, 1787], [1227, 1838], [1227, 1890], [1227, 1942], [1227, 2037], [1227, 2078], [1227, 3358], [1228, 1229], [1228, 1337], [1228, 1378], [1228, 1789], [1228, 2012], [1228, 3359], [1229, 1242], [1229, 1280], [1229, 1548], [1229, 1645], [1229, 1758], [1229, 2058], [1229, 2061], [1229, 2062], [1229, 3360], [1230, 1231], [1230, 1244], [1230, 1353], [1230, 1446], [1230, 1570], [1230, 1578], [1230, 1680], [1230, 1698], [1230, 3361], [1231, 1232], [1231, 1496], [1231, 1556], [1231, 1758], [1231, 1843], [1231, 3362], [1232, 1496], [1232, 1564], [1232, 1657], [1232, 1706], [1232, 1959], [1232, 3363], [1233, 1234], [1233, 1235], [1233, 1240], [1233, 1247], [1233, 1290], [1233, 1299], [1233, 1331], [1233, 1378], [1233, 1476], [1233, 1536], [1233, 1742], [1233, 1817], [1233, 2011], [1233, 2040], [1233, 2073], [1233, 3364], [1234, 1235], [1234, 1242], [1234, 1248], [1234, 1458], [1234, 1702], [1234, 1774], [1234, 1779], [1234, 1886], [1234, 2018], [1234, 2045], [1234, 2050], [1234, 2058], [1234, 2065], [1234, 2067], [1234, 3365], [1235, 1236], [1235, 1241], [1235, 1612], [1235, 1805], [1235, 1965], [1235, 1990], [1235, 3366], [1236, 1247], [1236, 1248], [1236, 1270], [1236, 1273], [1236, 1510], [1236, 1687], [1236, 1897], [1236, 1910], [1236, 1927], [1236, 1978], [1236, 2052], [1236, 2076], [1236, 2098], [1236, 2116], [1236, 3367], [1237, 1238], [1237, 1243], [1237, 1245], [1237, 1290], [1237, 1307], [1237, 1371], [1237, 1425], [1237, 1512], [1237, 1740], [1237, 1776], [1237, 3368], [1238, 1239], [1238, 1423], [1238, 1448], [1238, 1450], [1238, 1451], [1238, 1570], [1238, 1731], [1238, 1735], [1238, 1777], [1238, 1782], [1238, 2071], [1238, 2082], [1238, 3369], [1239, 1266], [1239, 1276], [1239, 1356], [1239, 1423], [1239, 1616], [1239, 1705], [1239, 1764], [1239, 1767], [1239, 1773], [1239, 1781], [1239, 2077], [1239, 2107], [1239, 3370], [1240, 1241], [1240, 1242], [1240, 1345], [1240, 1346], [1240, 1550], [1240, 1956], [1240, 3371], [1241, 1242], [1241, 1273], [1241, 1618], [1241, 3372], [1242, 1287], [1242, 1371], [1242, 1372], [1242, 1521], [1242, 1573], [1242, 2061], [1242, 3373], [1243, 1244], [1243, 1378], [1243, 1387], [1243, 1585], [1243, 1796], [1243, 3374], [1244, 1245], [1244, 1295], [1244, 1678], [1244, 1697], [1244, 1761], [1244, 3375], [1245, 1296], [1245, 1372], [1245, 1697], [1245, 1762], [1245, 1966], [1245, 2124], [1245, 3376], [1246, 1247], [1246, 1259], [1246, 1413], [1246, 1415], [1246, 1421], [1246, 1425], [1246, 1559], [1246, 1639], [1246, 1640], [1246, 1784], [1246, 1796], [1246, 3377], [1247, 1248], [1247, 1391], [1247, 1406], [1247, 1416], [1247, 1920], [1247, 2073], [1247, 2074], [1247, 2087], [1247, 2111], [1247, 3378], [1248, 1249], [1248, 1259], [1248, 1779], [1248, 1793], [1248, 1840], [1248, 1967], [1248, 2033], [1248, 2092], [1248, 2131], [1248, 3379], [1249, 1251], [1249, 1258], [1249, 1391], [1249, 1455], [1249, 1793], [1249, 1835], [1249, 1869], [1249, 1887], [1249, 2042], [1249, 3380], [1250, 1251], [1250, 1255], [1250, 1258], [1250, 1262], [1250, 1327], [1250, 1335], [1250, 1700], [1250, 3381], [1251, 1252], [1251, 1253], [1251, 1258], [1251, 1293], [1251, 1308], [1251, 1702], [1251, 1740], [1251, 1743], [1251, 1828], [1251, 1835], [1251, 1996], [1251, 3382], [1252, 1253], [1252, 1385], [1252, 1521], [1252, 1617], [1252, 1623], [1252, 1693], [1252, 2130], [1252, 3383], [1253, 1254], [1253, 1280], [1253, 1324], [1253, 1327], [1253, 1328], [1253, 1477], [1253, 1492], [1253, 1500], [1253, 1624], [1253, 1865], [1253, 1866], [1253, 1950], [1253, 2013], [1253, 2032], [1253, 2064], [1253, 2120], [1253, 3384], [1254, 1379], [1254, 1408], [1254, 1609], [1254, 1648], [1254, 1674], [1254, 1801], [1254, 1802], [1254, 1826], [1254, 1845], [1254, 1875], [1254, 1901], [1254, 1916], [1254, 2024], [1254, 2052], [1254, 2129], [1254, 3385], [1255, 1256], [1255, 1330], [1255, 1363], [1255, 1396], [1255, 1403], [1255, 1520], [1255, 1729], [1255, 1764], [1255, 3386], [1256, 1257], [1256, 1279], [1256, 3387], [1257, 1263], [1257, 1313], [1257, 1588], [1257, 1634], [1257, 1700], [1257, 1729], [1257, 1737], [1257, 1827], [1257, 3388], [1258, 1259], [1258, 1328], [1258, 1410], [1258, 1573], [1258, 1711], [1258, 1792], [1258, 1796], [1258, 1874], [1258, 3389], [1259, 1260], [1259, 1286], [1259, 1666], [1259, 1669], [1259, 1670], [1259, 1738], [1259, 1792], [1259, 2130], [1259, 3390], [1260, 1261], [1260, 1288], [1260, 1328], [1260, 1432], [1260, 1484], [1260, 1490], [1260, 1521], [1260, 1668], [1260, 1691], [1260, 1739], [1260, 1937], [1260, 1978], [1260, 2032], [1260, 2033], [1260, 2118], [1260, 3391], [1261, 1285], [1261, 1316], [1261, 1545], [1261, 1855], [1261, 1922], [1261, 1923], [1261, 1934], [1261, 1937], [1261, 3392], [1262, 1263], [1262, 1412], [1262, 1530], [1262, 1746], [1262, 3393], [1263, 1264], [1263, 1355], [1263, 1725], [1263, 1756], [1263, 2046], [1263, 3394], [1264, 1436], [1264, 1469], [1264, 1470], [1264, 1679], [1264, 1683], [1264, 1791], [1264, 2046], [1264, 3395], [1265, 1266], [1265, 1307], [1265, 1405], [1265, 1764], [1265, 3396], [1266, 1267], [1266, 1268], [1266, 1405], [1266, 1435], [1266, 1443], [1266, 3397], [1267, 1289], [1267, 1495], [1267, 1834], [1267, 1903], [1267, 2056], [1267, 2071], [1267, 2130], [1267, 3398], [1268, 1269], [1268, 1274], [1268, 1339], [1268, 1435], [1268, 1531], [1268, 1532], [1268, 1767], [1268, 1797], [1268, 3399], [1269, 1270], [1269, 1532], [1269, 1774], [1269, 1798], [1269, 1799], [1269, 1813], [1269, 2050], [1269, 2056], [1269, 2068], [1269, 2084], [1269, 3400], [1270, 1272], [1270, 1274], [1270, 1314], [1270, 1315], [1270, 1437], [1270, 1456], [1270, 1911], [1270, 1929], [1270, 1986], [1270, 2116], [1270, 3401], [1271, 1272], [1271, 1373], [1271, 1415], [1271, 1532], [1271, 1784], [1271, 1788], [1271, 1800], [1271, 3402], [1272, 1273], [1272, 1274], [1272, 1431], [1272, 1532], [1272, 3403], [1273, 1369], [1273, 1432], [1273, 1545], [1273, 1785], [1273, 1790], [1273, 1909], [1273, 3404], [1274, 1275], [1274, 1405], [1274, 1436], [1274, 1462], [1274, 1464], [1274, 1604], [1274, 1650], [1274, 1683], [1274, 1715], [1274, 3405], [1275, 1519], [1275, 1541], [1275, 1682], [1275, 1709], [1275, 1713], [1275, 1797], [1275, 1814], [1275, 3406], [1276, 1277], [1276, 1355], [1276, 1403], [1276, 1486], [1276, 3407], [1277, 1278], [1277, 1309], [1277, 1319], [1277, 1348], [1277, 1377], [1277, 1388], [1277, 1396], [1277, 1446], [1277, 1449], [1277, 1486], [1277, 2107], [1277, 3408], [1278, 1279], [1278, 1348], [1278, 1481], [1278, 1485], [1278, 1695], [1278, 1699], [1278, 1732], [1278, 1776], [1278, 2113], [1278, 3409], [1279, 1280], [1279, 1303], [1279, 1320], [1279, 1363], [1279, 1365], [1279, 1733], [1279, 1846], [1279, 2115], [1279, 3410], [1280, 1311], [1280, 1363], [1280, 1521], [1280, 1624], [1280, 1776], [1280, 1846], [1280, 2014], [1280, 2060], [1280, 2109], [1280, 2114], [1280, 3411], [1281, 1282], [1281, 1286], [1281, 1452], [1281, 1489], [1281, 1530], [1281, 1546], [1281, 1666], [1281, 1667], [1281, 3412], [1282, 1283], [1282, 1286], [1282, 1714], [1282, 1746], [1282, 1757], [1282, 2090], [1282, 3413], [1283, 1284], [1283, 1300], [1283, 1489], [1283, 1490], [1283, 1691], [1283, 1720], [1283, 3414], [1284, 1285], [1284, 1322], [1284, 1433], [1284, 1438], [1284, 1636], [1284, 1720], [1284, 1883], [1284, 1885], [1284, 1889], [1284, 1938], [1284, 1972], [1284, 3415], [1285, 1438], [1285, 1453], [1285, 1490], [1285, 1491], [1285, 1506], [1285, 1579], [1285, 1883], [1285, 1935], [1285, 3416], [1286, 1287], [1286, 1381], [1286, 1400], [1286, 1413], [1286, 1453], [1286, 1490], [1286, 1671], [1286, 1749], [1286, 1758], [1286, 1804], [1286, 1840], [1286, 1843], [1286, 1886], [1286, 2015], [1286, 2090], [1286, 3417], [1287, 1288], [1287, 1431], [1287, 1453], [1287, 1491], [1287, 1575], [1287, 1591], [1287, 1714], [1287, 1717], [1287, 1718], [1287, 1856], [1287, 2067], [1287, 2088], [1287, 3418], [1288, 1317], [1288, 1393], [1288, 1394], [1288, 1431], [1288, 1464], [1288, 1519], [1288, 1521], [1288, 1769], [1288, 1793], [1288, 1808], [1288, 1839], [1288, 1887], [1288, 1890], [1288, 1985], [1288, 2015], [1288, 2027], [1288, 2131], [1288, 3419], [1289, 1290], [1289, 1307], [1289, 1366], [1289, 1392], [1289, 1448], [1289, 1497], [1289, 1581], [1289, 1589], [1289, 1621], [1289, 1632], [1289, 1633], [1289, 1693], [1289, 1762], [1289, 1830], [1289, 1836], [1289, 1994], [1289, 2019], [1289, 3420], [1290, 1291], [1290, 1362], [1290, 1378], [1290, 1402], [1290, 1994], [1290, 2054], [1290, 2072], [1290, 2082], [1290, 2111], [1290, 3421], [1291, 1292], [1291, 1402], [1291, 1526], [1291, 1632], [1291, 1779], [1291, 1878], [1291, 1919], [1291, 2058], [1291, 2094], [1291, 2095], [1291, 3422], [1292, 1362], [1292, 1685], [1292, 1739], [1292, 1779], [1292, 1815], [1292, 2033], [1292, 2053], [1292, 2065], [1292, 3423], [1293, 1294], [1293, 1309], [1293, 1335], [1293, 1337], [1293, 1385], [1293, 1389], [1293, 1391], [1293, 1467], [1293, 1796], [1293, 1850], [1293, 1873], [1293, 1954], [1293, 2023], [1293, 2047], [1293, 3424], [1294, 1295], [1294, 1336], [1294, 1337], [1294, 1407], [1294, 1552], [1294, 1647], [1294, 1726], [1294, 1824], [1294, 1828], [1294, 1863], [1294, 1942], [1294, 2038], [1294, 3425], [1295, 1296], [1295, 1336], [1295, 1559], [1295, 1569], [1295, 1654], [1295, 1706], [1295, 1708], [1295, 1711], [1295, 1761], [1295, 1795], [1295, 1796], [1295, 1892], [1295, 2083], [1295, 3426], [1296, 1316], [1296, 1425], [1296, 1559], [1296, 1570], [1296, 1762], [1296, 1824], [1296, 1963], [1296, 1970], [1296, 2082], [1296, 2123], [1296, 3427], [1297, 1298], [1297, 1558], [1297, 1635], [1297, 1743], [1297, 1744], [1297, 1756], [1297, 1767], [1297, 1946], [1297, 3428], [1298, 1299], [1298, 1610], [1298, 1613], [1298, 1755], [1298, 1767], [1298, 1773], [1298, 3429], [1299, 1361], [1299, 1362], [1299, 1613], [1299, 1615], [1299, 1617], [1299, 1939], [1299, 3430], [1300, 1398], [1300, 1689], [1300, 1691], [1300, 1695], [1300, 2028], [1300, 3431], [1301, 1302], [1301, 1312], [1301, 1370], [1301, 1399], [1301, 1493], [1301, 1511], [1301, 1992], [1301, 2049], [1301, 2050], [1301, 2081], [1301, 2096], [1301, 3432], [1302, 1334], [1302, 1370], [1302, 1533], [1302, 1785], [1302, 1945], [1302, 2096], [1302, 2129], [1302, 3433], [1303, 1304], [1303, 1319], [1303, 1363], [1303, 1383], [1303, 1786], [1303, 3434], [1304, 1305], [1304, 1704], [1304, 1765], [1304, 1768], [1304, 1786], [1304, 2115], [1304, 3435], [1305, 1306], [1305, 1482], [1305, 1627], [1305, 1688], [1305, 1730], [1305, 1732], [1305, 3436], [1306, 1492], [1306, 1503], [1306, 1860], [1306, 1942], [1306, 1966], [1306, 2115], [1306, 2128], [1306, 3437], [1307, 1308], [1307, 1366], [1307, 1387], [1307, 1409], [1307, 1434], [1307, 1446], [1307, 1459], [1307, 1496], [1307, 1571], [1307, 1580], [1307, 1595], [1307, 1620], [1307, 1740], [1307, 1764], [1307, 3438], [1308, 1309], [1308, 1317], [1308, 1409], [1308, 1623], [1308, 1624], [1308, 1692], [1308, 1716], [1308, 1836], [1308, 1887], [1308, 3439], [1309, 1310], [1309, 1389], [1309, 1403], [1309, 1417], [1309, 1446], [1309, 1450], [1309, 1512], [1309, 1851], [1309, 1942], [1309, 3440], [1310, 1311], [1310, 1363], [1310, 1383], [1310, 1385], [1310, 1403], [1310, 1420], [1310, 1463], [1310, 1486], [1310, 1540], [1310, 1623], [1310, 1733], [1310, 1787], [1310, 3441], [1311, 1423], [1311, 1473], [1311, 1705], [1311, 1733], [1311, 1777], [1311, 1806], [1311, 1851], [1311, 2117], [1311, 3442], [1312, 1313], [1312, 1370], [1312, 1395], [1312, 1499], [1312, 3443], [1313, 1314], [1313, 1449], [1313, 1576], [1313, 1650], [1313, 2049], [1313, 3444], [1314, 1315], [1314, 1331], [1314, 1370], [1314, 1372], [1314, 1535], [1314, 1646], [1314, 1650], [1314, 1679], [1314, 1680], [1314, 1698], [1314, 1706], [1314, 1892], [1314, 1945], [1314, 1962], [1314, 2049], [1314, 2056], [1314, 3445], [1315, 1316], [1315, 1416], [1315, 1566], [1315, 1785], [1315, 2121], [1315, 3446], [1316, 1372], [1316, 1605], [1316, 1669], [1316, 1708], [1316, 1775], [1316, 1870], [1316, 1908], [1316, 1967], [1316, 3447], [1317, 1318], [1317, 1519], [1317, 1691], [1317, 1714], [1317, 1715], [1317, 1743], [1317, 2027], [1317, 3448], [1318, 1319], [1318, 1388], [1318, 1393], [1318, 1606], [1318, 1607], [1318, 1676], [1318, 1701], [1318, 1943], [1318, 3449], [1319, 1320], [1319, 1396], [1319, 1480], [1319, 1768], [1319, 1769], [1319, 3450], [1320, 1456], [1320, 1481], [1320, 1763], [1320, 1781], [1320, 1885], [1320, 1943], [1320, 1981], [1320, 2107], [1320, 2125], [1320, 3451], [1321, 1322], [1321, 1326], [1321, 1412], [1321, 1433], [1321, 1578], [1321, 1597], [1321, 3452], [1322, 1323], [1322, 1324], [1322, 1349], [1322, 1352], [1322, 1358], [1322, 1578], [1322, 1579], [1322, 2021], [1322, 2022], [1322, 2102], [1322, 3453], [1323, 1324], [1323, 1353], [1323, 1451], [1323, 1477], [1323, 1539], [1323, 1636], [1323, 1654], [1323, 1957], [1323, 3454], [1324, 1325], [1324, 1354], [1324, 1406], [1324, 1412], [1324, 1495], [1324, 1726], [1324, 1852], [1324, 1893], [1324, 1899], [1324, 1951], [1324, 2038], [1324, 2087], [1324, 2109], [1324, 3455], [1325, 1354], [1325, 1358], [1325, 1431], [1325, 1437], [1325, 1513], [1325, 1599], [1325, 1654], [1325, 1866], [1325, 1900], [1325, 1904], [1325, 2119], [1325, 2123], [1325, 3456], [1326, 1327], [1326, 1340], [1326, 1673], [1326, 1694], [1326, 1786], [1326, 3457], [1327, 1328], [1327, 1363], [1327, 1379], [1327, 1381], [1327, 1393], [1327, 1412], [1327, 1482], [1327, 1502], [1327, 1547], [1327, 1691], [1327, 1692], [1327, 1694], [1327, 1766], [1327, 3458], [1328, 1329], [1328, 1347], [1328, 1488], [1328, 1609], [1328, 1874], [1328, 1953], [1328, 3459], [1329, 1338], [1329, 1590], [1329, 1663], [1329, 1711], [1329, 1766], [1329, 1867], [1329, 1872], [1329, 1950], [1329, 1953], [1329, 1978], [1329, 3460], [1330, 1331], [1330, 1415], [1330, 1479], [1330, 1480], [1330, 1650], [1330, 1788], [1330, 3461], [1331, 1332], [1331, 1377], [1331, 1416], [1331, 1481], [1331, 1535], [1331, 1560], [1331, 1723], [1331, 1765], [1331, 1789], [1331, 1847], [1331, 1848], [1331, 1898], [1331, 2002], [1331, 2045], [1331, 3462], [1332, 1333], [1332, 1359], [1332, 1375], [1332, 1461], [1332, 1476], [1332, 1479], [1332, 1481], [1332, 1496], [1332, 1498], [1332, 1499], [1332, 1522], [1332, 1525], [1332, 1581], [1332, 2125], [1332, 3463], [1333, 1334], [1333, 1370], [1333, 1395], [1333, 1399], [1333, 1460], [1333, 1525], [1333, 1549], [1333, 1753], [1333, 1772], [1333, 1962], [1333, 3464], [1334, 1399], [1334, 1498], [1334, 1526], [1334, 1590], [1334, 1877], [1334, 1878], [1334, 1901], [1334, 1988], [1334, 3465], [1335, 1336], [1335, 1445], [1335, 1502], [1335, 1625], [1335, 1791], [1335, 1796], [1335, 3466], [1336, 1503], [1336, 1558], [1336, 1569], [1336, 1626], [1336, 1652], [1336, 1681], [1336, 1700], [1336, 1756], [1336, 1791], [1336, 3467], [1337, 1338], [1337, 1402], [1337, 1539], [1337, 1671], [1337, 1791], [1337, 1793], [1337, 1795], [1337, 1872], [1337, 2030], [1337, 2092], [1337, 2093], [1337, 3468], [1338, 1385], [1338, 1533], [1338, 1637], [1338, 1662], [1338, 1722], [1338, 1787], [1338, 1821], [1338, 2012], [1338, 2030], [1338, 3469], [1339, 1340], [1339, 1341], [1339, 1349], [1339, 1419], [1339, 1635], [1339, 1703], [1339, 1731], [1339, 1767], [1339, 1813], [1339, 3470], [1340, 1341], [1340, 1388], [1340, 1486], [1340, 1610], [1340, 1660], [1340, 1664], [1340, 1673], [1340, 1713], [1340, 1755], [1340, 1797], [1340, 2001], [1340, 3471], [1341, 1342], [1341, 1434], [1341, 1481], [1341, 1611], [1341, 1665], [1341, 1696], [1341, 1768], [1341, 1932], [1341, 3472], [1342, 1394], [1342, 1408], [1342, 1419], [1342, 1460], [1342, 1808], [1342, 3473], [1343, 1344], [1343, 1373], [1343, 1443], [1343, 1462], [1343, 1474], [1343, 1746], [1343, 3474], [1344, 1345], [1344, 1370], [1344, 1475], [1344, 1525], [1344, 1698], [1344, 1746], [1344, 3475], [1345, 1346], [1345, 1373], [1345, 1560], [1345, 1749], [1345, 1790], [1345, 3476], [1346, 1347], [1346, 1475], [1346, 1476], [1346, 1564], [1346, 3477], [1347, 1381], [1347, 1394], [1347, 1425], [1347, 1500], [1347, 1518], [1347, 1567], [1347, 1719], [1347, 1941], [1347, 3478], [1348, 1349], [1348, 1352], [1348, 1376], [1348, 1397], [1348, 1465], [1348, 1482], [1348, 1524], [1348, 1537], [1348, 1652], [1348, 1757], [1348, 2101], [1348, 3479], [1349, 1350], [1349, 1351], [1349, 1465], [1349, 1522], [1349, 1523], [1349, 2070], [1349, 3480], [1350, 1351], [1350, 1446], [1350, 1448], [1350, 1731], [1350, 1732], [1350, 2069], [1350, 3481], [1351, 1381], [1351, 1384], [1351, 1397], [1351, 1611], [1351, 1649], [1351, 1751], [1351, 1906], [1351, 3482], [1352, 1353], [1352, 1354], [1352, 1358], [1352, 1398], [1352, 1485], [1352, 1538], [1352, 1720], [1352, 1803], [1352, 2104], [1352, 2105], [1352, 3483], [1353, 1354], [1353, 1638], [1353, 1654], [1353, 3484], [1354, 1800], [1354, 1955], [1354, 1975], [1354, 3485], [1355, 1356], [1355, 1436], [1355, 1628], [1355, 1629], [1355, 3486], [1356, 1357], [1356, 1436], [1356, 1437], [1356, 1449], [1356, 1450], [1356, 1541], [1356, 1599], [1356, 1853], [1356, 2043], [1356, 2046], [1356, 3487], [1357, 1358], [1357, 1428], [1357, 1492], [1357, 1541], [1357, 1577], [1357, 1629], [1357, 1630], [1357, 1705], [1357, 1726], [1357, 1799], [1357, 1808], [1357, 1888], [1357, 1890], [1357, 1942], [1357, 2108], [1357, 3488], [1358, 1401], [1358, 1465], [1358, 1470], [1358, 1591], [1358, 1597], [1358, 1598], [1358, 1654], [1358, 1656], [1358, 1761], [1358, 2103], [1358, 2124], [1358, 3489], [1359, 1360], [1359, 1412], [1359, 1435], [1359, 1495], [1359, 1496], [1359, 1723], [1359, 1732], [1359, 1755], [1359, 1778], [1359, 1843], [1359, 2015], [1359, 2057], [1359, 3490], [1360, 1361], [1360, 1369], [1360, 1424], [1360, 1495], [1360, 1777], [1360, 2012], [1360, 3491], [1361, 1362], [1361, 1378], [1361, 1487], [1361, 1664], [1361, 1755], [1361, 1832], [1361, 2012], [1361, 2053], [1361, 3492], [1362, 1616], [1362, 1621], [1362, 1773], [1362, 1831], [1362, 2054], [1362, 3493], [1363, 1364], [1363, 1520], [1363, 1548], [1363, 1765], [1363, 3494], [1364, 1365], [1364, 1440], [1364, 1765], [1364, 1786], [1364, 1787], [1364, 1794], [1364, 3495], [1365, 1485], [1365, 1671], [1365, 1763], [1365, 1786], [1365, 1798], [1365, 2059], [1365, 2115], [1365, 3496], [1366, 1367], [1366, 1439], [1366, 1447], [1366, 1515], [1366, 1887], [1366, 3497], [1367, 1368], [1367, 1513], [1367, 1514], [1367, 1633], [1367, 3498], [1368, 1369], [1368, 1402], [1368, 1417], [1368, 1512], [1368, 1772], [1368, 1785], [1368, 1795], [1368, 2121], [1368, 3499], [1369, 1440], [1369, 1551], [1369, 1609], [1369, 3500], [1370, 1371], [1370, 1380], [1370, 1414], [1370, 1420], [1370, 1462], [1370, 3501], [1371, 1372], [1371, 1571], [1371, 1782], [1371, 1856], [1371, 3502], [1372, 1515], [1372, 1572], [1372, 1591], [1372, 1603], [1372, 1678], [1372, 1686], [1372, 1773], [1372, 1923], [1372, 1976], [1372, 2018], [1372, 2036], [1372, 2055], [1372, 3503], [1373, 1374], [1373, 1443], [1373, 1444], [1373, 1564], [1373, 1749], [1373, 1810], [1373, 3504], [1374, 1375], [1374, 1479], [1374, 1516], [1374, 1517], [1374, 1580], [1374, 1585], [1374, 1788], [1374, 1843], [1374, 3505], [1375, 1444], [1375, 1515], [1375, 1518], [1375, 1578], [1375, 1581], [1375, 1772], [1375, 1789], [1375, 1804], [1375, 1842], [1375, 2017], [1375, 2021], [1375, 2078], [1375, 2086], [1375, 3506], [1376, 1377], [1376, 1413], [1376, 1428], [1376, 1488], [1376, 1504], [1376, 1512], [1376, 1523], [1376, 1524], [1376, 1526], [1376, 1640], [1376, 1654], [1376, 1758], [1376, 1776], [1376, 1874], [1376, 1895], [1376, 1898], [1376, 1955], [1376, 1958], [1376, 2095], [1376, 2101], [1376, 2105], [1376, 3507], [1377, 1378], [1377, 1392], [1377, 1417], [1377, 1447], [1377, 1487], [1377, 1680], [1377, 1776], [1377, 1848], [1377, 2106], [1377, 2107], [1377, 3508], [1378, 1391], [1378, 1392], [1378, 2083], [1378, 2095], [1378, 3509], [1379, 1380], [1379, 1382], [1379, 1460], [1379, 1547], [1379, 1607], [1379, 1609], [1379, 1657], [1379, 1673], [1379, 1682], [1379, 1701], [1379, 1800], [1379, 1876], [1379, 1955], [1379, 1977], [1379, 3510], [1380, 1381], [1380, 1419], [1380, 1460], [1380, 1462], [1380, 1567], [1380, 1636], [1380, 1646], [1380, 1649], [1380, 1683], [1380, 1746], [1380, 1747], [1380, 1954], [1380, 2129], [1380, 3511], [1381, 1419], [1381, 1475], [1381, 1499], [1381, 1500], [1381, 1856], [1381, 3512], [1382, 1432], [1382, 1509], [1382, 1631], [1382, 1634], [1382, 1672], [1382, 1691], [1382, 1715], [1382, 1744], [1382, 1977], [1382, 2052], [1382, 3513], [1383, 1384], [1383, 1733], [1383, 1797], [1383, 3514], [1384, 1385], [1384, 1610], [1384, 1625], [1384, 1635], [1384, 1681], [1384, 1722], [1384, 2010], [1384, 3515], [1385, 1386], [1385, 1421], [1385, 1617], [1385, 1625], [1385, 1636], [1385, 1639], [1385, 1708], [1385, 1755], [1385, 1775], [1385, 1885], [1385, 1953], [1385, 2010], [1385, 2048], [1385, 2117], [1385, 3516], [1386, 1390], [1386, 1621], [1386, 1775], [1386, 1777], [1386, 1870], [1386, 3517], [1387, 1388], [1387, 1392], [1387, 1426], [1387, 1583], [1387, 1584], [1387, 1694], [1387, 3518], [1388, 1389], [1388, 1392], [1388, 1695], [1388, 1995], [1388, 3519], [1389, 1390], [1389, 1426], [1389, 1537], [1389, 3520], [1390, 1391], [1390, 1392], [1390, 1406], [1390, 1417], [1390, 1426], [1390, 1428], [1390, 1874], [1390, 3521], [1391, 1417], [1391, 1429], [1391, 1796], [1391, 1850], [1391, 2111], [1391, 3522], [1392, 1393], [1392, 1410], [1392, 1477], [1392, 1577], [1392, 1664], [1392, 1756], [1392, 1761], [1392, 1864], [1392, 1876], [1392, 1955], [1392, 1995], [1392, 2081], [1392, 3523], [1393, 1394], [1393, 1410], [1393, 1454], [1393, 1505], [1393, 1529], [1393, 1608], [1393, 1677], [1393, 1693], [1393, 1865], [1393, 1943], [1393, 3524], [1394, 1409], [1394, 1434], [1394, 1441], [1394, 1442], [1394, 1463], [1394, 1564], [1394, 1859], [1394, 2127], [1394, 3525], [1395, 1396], [1395, 1399], [1395, 1751], [1395, 1771], [1395, 3526], [1396, 1397], [1396, 1734], [1396, 1764], [1396, 3527], [1397, 1398], [1397, 1640], [1397, 1974], [1397, 3528], [1398, 1511], [1398, 1553], [1398, 1591], [1398, 1720], [1398, 1764], [1398, 1814], [1398, 3529], [1399, 1400], [1399, 1401], [1399, 1465], [1399, 1582], [1399, 1596], [1399, 1877], [1399, 2004], [1399, 2005], [1399, 3530], [1400, 1401], [1400, 1518], [1400, 1758], [1400, 2125], [1400, 3531], [1401, 1402], [1401, 1671], [1401, 1762], [1401, 1771], [1401, 1772], [1401, 1799], [1401, 1820], [1401, 1963], [1401, 2021], [1401, 2050], [1401, 2059], [1401, 2079], [1401, 2119], [1401, 3532], [1402, 1512], [1402, 1633], [1402, 1834], [1402, 1851], [1402, 1854], [1402, 1855], [1402, 1872], [1402, 1895], [1402, 2080], [1402, 2098], [1402, 3533], [1403, 1404], [1403, 1423], [1403, 1435], [1403, 1462], [1403, 1571], [1403, 1729], [1403, 1778], [1403, 3534], [1404, 1405], [1404, 1573], [1404, 1690], [1404, 1787], [1404, 1800], [1404, 1876], [1404, 3535], [1405, 1418], [1405, 1436], [1405, 1459], [1405, 1514], [1405, 1515], [1405, 1716], [1405, 1911], [1405, 2043], [1405, 3536], [1406, 1407], [1406, 1408], [1406, 1416], [1406, 1423], [1406, 1451], [1406, 1510], [1406, 1748], [1406, 1775], [1406, 1801], [1406, 1921], [1406, 2027], [1406, 2038], [1406, 2126], [1406, 3537], [1407, 1408], [1407, 1675], [1407, 1681], [1407, 1708], [1407, 1726], [1407, 1793], [1407, 1802], [1407, 1808], [1407, 1869], [1407, 1949], [1407, 1967], [1407, 1968], [1407, 1999], [1407, 2010], [1407, 3538], [1408, 1607], [1408, 1608], [1408, 1696], [1408, 1899], [1408, 1943], [1408, 3539], [1409, 1410], [1409, 1425], [1409, 1426], [1409, 1810], [1409, 3540], [1410, 1411], [1410, 1450], [1410, 1460], [1410, 1572], [1410, 1627], [1410, 1669], [1410, 1681], [1410, 1711], [1410, 1869], [1410, 1870], [1410, 1873], [1410, 1953], [1410, 1996], [1410, 3541], [1411, 1736], [1411, 1782], [1411, 2081], [1411, 3542], [1412, 1413], [1412, 1435], [1412, 1441], [1412, 1443], [1412, 1513], [1412, 1530], [1412, 1537], [1412, 1538], [1412, 1756], [1412, 3543], [1413, 1414], [1413, 1467], [1413, 1504], [1413, 1539], [1413, 1804], [1413, 2087], [1413, 2091], [1413, 3544], [1414, 1530], [1414, 1531], [1414, 1533], [1414, 1662], [1414, 3545], [1415, 1416], [1415, 1420], [1415, 3546], [1416, 1417], [1416, 1560], [1416, 1785], [1416, 2109], [1416, 2131], [1416, 3547], [1417, 1418], [1417, 1447], [1417, 1851], [1417, 1887], [1417, 2121], [1417, 3548], [1418, 1464], [1418, 1532], [1418, 1593], [1418, 1630], [1418, 1665], [1418, 1772], [1418, 1787], [1418, 1799], [1418, 1887], [1418, 3549], [1419, 1420], [1419, 1421], [1419, 1422], [1419, 1430], [1419, 1534], [1419, 1941], [1419, 2126], [1419, 3550], [1420, 1421], [1420, 1532], [1420, 1535], [1420, 3551], [1421, 1533], [1421, 1536], [1421, 1992], [1421, 2130], [1421, 3552], [1422, 1423], [1422, 1683], [1422, 1767], [1422, 1808], [1422, 1924], [1422, 3553], [1423, 1535], [1423, 1693], [1423, 1775], [1423, 1851], [1423, 1984], [1423, 1985], [1423, 3554], [1424, 1425], [1424, 1521], [1424, 1775], [1424, 1778], [1424, 3555], [1425, 1515], [1425, 1560], [1425, 1796], [1425, 2023], [1425, 2111], [1425, 3556], [1426, 1427], [1426, 1796], [1426, 3557], [1427, 1428], [1427, 1524], [1427, 1537], [1427, 1561], [1427, 1583], [1427, 1629], [1427, 1653], [1427, 1810], [1427, 3558], [1428, 1429], [1428, 1450], [1428, 1537], [1428, 1540], [1428, 1577], [1428, 1654], [1428, 1777], [1428, 1809], [1428, 1893], [1428, 1894], [1428, 1916], [1428, 1953], [1428, 3559], [1429, 1467], [1429, 1561], [1429, 1891], [1429, 1892], [1429, 1916], [1429, 2108], [1429, 2110], [1429, 3560], [1430, 1431], [1430, 1450], [1430, 1565], [1430, 1782], [1430, 1856], [1430, 3561], [1431, 1432], [1431, 1470], [1431, 1715], [1431, 1800], [1431, 1986], [1431, 2089], [1431, 3562], [1432, 1609], [1432, 1618], [1432, 1637], [1432, 1745], [1432, 1748], [1432, 1750], [1432, 2052], [1432, 3563], [1433, 1434], [1433, 1438], [1433, 1441], [1433, 1562], [1433, 3564], [1434, 1435], [1434, 1439], [1434, 1522], [1434, 1546], [1434, 1564], [1434, 1571], [1434, 1769], [1434, 3565], [1435, 1436], [1435, 3566], [1436, 1437], [1436, 1513], [1436, 1630], [1436, 3567], [1437, 1616], [1437, 1680], [1437, 1759], [1437, 1853], [1437, 1911], [1437, 2121], [1437, 3568], [1438, 1439], [1438, 1452], [1438, 1489], [1438, 1505], [1438, 1518], [1438, 1578], [1438, 1656], [1438, 3569], [1439, 1440], [1439, 1518], [1439, 1665], [1439, 3570], [1440, 1454], [1440, 1491], [1440, 1593], [1440, 1763], [1440, 1769], [1440, 2061], [1440, 3571], [1441, 1442], [1441, 1542], [1441, 1562], [1441, 3572], [1442, 1443], [1442, 1510], [1442, 1563], [1442, 3573], [1443, 1444], [1443, 1461], [1443, 1514], [1443, 1692], [1443, 1852], [1443, 1903], [1443, 3574], [1444, 1809], [1444, 1841], [1444, 1852], [1444, 2086], [1444, 2118], [1444, 3575], [1445, 1446], [1445, 1467], [1445, 1561], [1445, 3576], [1446, 1447], [1446, 1448], [1446, 1522], [1446, 1580], [1446, 3577], [1447, 1448], [1447, 1560], [1447, 1561], [1447, 3578], [1448, 1467], [1448, 1523], [1448, 1581], [1448, 1777], [1448, 1851], [1448, 2069], [1448, 2107], [1448, 2128], [1448, 3579], [1449, 1450], [1449, 1577], [1449, 1652], [1449, 1658], [1449, 1680], [1449, 1699], [1449, 1734], [1449, 3580], [1450, 1451], [1450, 1460], [1450, 1512], [1450, 1540], [1450, 1716], [1450, 1854], [1450, 1892], [1450, 1998], [1450, 2047], [1450, 2121], [1450, 3581], [1451, 1557], [1451, 1559], [1451, 1560], [1451, 1566], [1451, 1636], [1451, 1699], [1451, 1749], [1451, 1750], [1451, 1810], [1451, 1919], [1451, 1949], [1451, 1977], [1451, 1997], [1451, 3582], [1452, 1453], [1452, 1470], [1452, 1729], [1452, 3583], [1453, 1454], [1453, 1456], [1453, 1471], [1453, 1567], [1453, 1759], [1453, 1985], [1453, 2008], [1453, 3584], [1454, 1455], [1454, 1609], [1454, 1656], [1454, 1677], [1454, 1701], [1454, 1711], [1454, 1745], [1454, 1759], [1454, 2024], [1454, 3585], [1455, 1573], [1455, 1693], [1455, 1702], [1455, 1745], [1455, 1825], [1455, 1918], [1455, 1989], [1455, 3586], [1456, 1457], [1456, 1471], [1456, 1506], [1456, 1566], [1456, 1591], [1456, 1677], [1456, 1687], [1456, 1736], [1456, 1813], [1456, 1879], [1456, 1882], [1456, 1968], [1456, 3587], [1457, 1458], [1457, 1469], [1457, 1541], [1457, 1615], [1457, 1677], [1457, 1682], [1457, 1722], [1457, 1725], [1457, 1726], [1457, 1875], [1457, 1885], [1457, 1886], [1457, 2046], [1457, 2048], [1457, 3588], [1458, 1501], [1458, 1536], [1458, 1670], [1458, 1774], [1458, 1827], [1458, 1875], [1458, 1882], [1458, 1915], [1458, 1933], [1458, 1984], [1458, 1986], [1458, 1987], [1458, 1989], [1458, 1990], [1458, 2034], [1458, 2052], [1458, 2116], [1458, 2130], [1458, 3589], [1459, 1460], [1459, 1595], [1459, 1665], [1459, 1876], [1459, 3590], [1460, 1461], [1460, 1514], [1460, 1591], [1460, 1753], [1460, 1901], [1460, 1902], [1460, 1961], [1460, 2005], [1460, 2006], [1460, 3591], [1461, 1474], [1461, 1476], [1461, 1564], [1461, 1635], [1461, 1696], [1461, 1811], [1461, 1959], [1461, 2127], [1461, 3592], [1462, 1463], [1462, 1530], [1462, 1563], [1462, 1571], [1462, 3593], [1463, 1464], [1463, 1505], [1463, 1636], [1463, 1810], [1463, 3594], [1464, 1505], [1464, 1542], [1464, 1630], [1464, 1888], [1464, 3595], [1465, 1466], [1465, 1481], [1465, 1505], [1465, 1756], [1465, 1879], [1465, 3596], [1466, 1467], [1466, 1481], [1466, 1544], [1466, 1751], [1466, 1754], [1466, 1896], [1466, 1898], [1466, 3597], [1467, 1503], [1467, 1537], [1467, 1539], [1467, 1561], [1467, 1576], [1467, 1643], [1467, 1891], [1467, 2128], [1467, 3598], [1468, 1469], [1468, 1622], [1468, 1663], [1468, 1664], [1468, 1677], [1468, 1707], [1468, 1713], [1468, 3599], [1469, 1662], [1469, 1725], [1469, 3600], [1470, 1471], [1470, 1541], [1470, 1618], [1470, 1656], [1470, 1679], [1470, 1769], [1470, 3601], [1471, 1472], [1471, 1671], [1471, 1763], [1471, 1915], [1471, 1935], [1471, 2007], [1471, 2089], [1471, 2103], [1471, 3602], [1472, 1473], [1472, 1541], [1472, 1659], [1472, 1763], [1472, 1798], [1472, 1853], [1472, 1905], [1472, 1933], [1472, 1985], [1472, 2085], [1472, 3603], [1473, 1534], [1473, 1540], [1473, 1659], [1473, 1854], [1473, 1894], [1473, 1913], [1473, 1941], [1473, 3604], [1474, 1475], [1474, 1479], [1474, 1564], [1474, 1606], [1474, 3605], [1475, 1476], [1475, 1479], [1475, 1694], [1475, 1699], [1475, 1752], [1475, 1843], [1475, 3606], [1476, 1477], [1476, 1500], [1476, 1753], [1476, 1823], [1476, 1842], [1476, 1939], [1476, 1954], [1476, 2081], [1476, 3607], [1477, 1478], [1477, 1496], [1477, 1685], [1477, 1694], [1477, 1695], [1477, 1740], [1477, 1742], [1477, 1754], [1477, 1776], [1477, 1955], [1477, 1993], [1477, 3608], [1478, 1529], [1478, 1594], [1478, 1602], [1478, 1665], [1478, 1687], [1478, 1696], [1478, 1865], [1478, 1900], [1478, 1931], [1478, 1970], [1478, 2083], [1478, 3609], [1479, 1480], [1479, 1580], [1479, 3610], [1480, 1481], [1480, 1585], [1480, 1606], [1480, 3611], [1481, 1696], [1481, 1755], [1481, 1780], [1481, 2112], [1481, 3612], [1482, 1483], [1482, 1486], [1482, 1488], [1482, 1489], [1482, 1492], [1482, 1496], [1482, 1499], [1482, 1502], [1482, 1505], [1482, 1600], [1482, 1629], [1482, 1657], [1482, 1754], [1482, 1803], [1482, 3613], [1483, 1484], [1483, 1613], [1483, 1619], [1483, 1754], [1483, 1776], [1483, 1848], [1483, 3614], [1484, 1485], [1484, 1617], [1484, 1618], [1484, 1720], [1484, 1723], [1484, 1739], [1484, 1769], [1484, 1804], [1484, 1923], [1484, 1938], [1484, 2061], [1484, 2065], [1484, 2066], [1484, 3615], [1485, 1556], [1485, 2057], [1485, 3616], [1486, 1487], [1486, 1585], [1486, 1629], [1486, 1658], [1486, 1704], [1486, 1705], [1486, 1755], [1486, 3617], [1487, 1488], [1487, 1630], [1487, 1664], [1487, 1705], [1487, 3618], [1488, 1492], [1488, 1630], [1488, 1655], [1488, 2053], [1488, 3619], [1489, 1490], [1489, 1668], [1489, 1803], [1489, 3620], [1490, 1491], [1490, 1492], [1490, 1637], [1490, 1890], [1490, 3621], [1491, 1575], [1491, 1592], [1491, 1720], [1491, 1973], [1491, 2061], [1491, 2105], [1491, 3622], [1492, 1493], [1492, 1494], [1492, 1497], [1492, 1506], [1492, 1592], [1492, 1617], [1492, 1648], [1492, 1705], [1492, 1816], [1492, 1881], [1492, 2064], [1492, 2101], [1492, 3623], [1493, 1494], [1493, 1498], [1493, 1499], [1493, 1500], [1493, 1523], [1493, 1812], [1493, 1898], [1493, 1926], [1493, 1927], [1493, 1992], [1493, 2011], [1493, 2128], [1493, 3624], [1494, 1495], [1494, 1724], [1494, 2013], [1494, 2091], [1494, 3625], [1495, 1497], [1495, 1498], [1495, 1775], [1495, 1832], [1495, 1834], [1495, 1842], [1495, 1849], [1495, 1860], [1495, 1912], [1495, 1937], [1495, 1952], [1495, 3626], [1496, 1497], [1496, 1549], [1496, 1620], [1496, 1624], [1496, 1665], [1496, 1765], [1496, 1776], [1496, 3627], [1497, 1498], [1497, 1590], [1497, 1621], [1497, 1868], [1497, 1881], [1497, 1993], [1497, 2054], [1497, 2114], [1497, 3628], [1498, 1523], [1498, 1526], [1498, 1811], [1498, 1847], [1498, 2017], [1498, 2097], [1498, 2112], [1498, 3629], [1499, 1522], [1499, 1576], [1499, 1635], [1499, 1804], [1499, 3630], [1500, 1501], [1500, 1823], [1500, 1834], [1500, 1840], [1500, 1855], [1500, 1906], [1500, 1937], [1500, 2013], [1500, 2023], [1500, 2073], [1500, 2125], [1500, 2126], [1500, 2127], [1500, 2129], [1500, 2130], [1500, 3631], [1501, 1553], [1501, 1726], [1501, 1799], [1501, 1806], [1501, 1811], [1501, 1815], [1501, 1818], [1501, 1820], [1501, 1821], [1501, 1822], [1501, 1824], [1501, 1825], [1501, 1827], [1501, 1831], [1501, 1833], [1501, 1835], [1501, 1837], [1501, 1840], [1501, 1846], [1501, 1847], [1501, 1850], [1501, 1852], [1501, 1853], [1501, 1857], [1501, 1864], [1501, 1877], [1501, 1883], [1501, 1891], [1501, 1904], [1501, 1906], [1501, 1913], [1501, 1917], [1501, 1919], [1501, 1922], [1501, 1925], [1501, 1926], [1501, 1928], [1501, 1931], [1501, 1933], [1501, 1940], [1501, 1944], [1501, 1946], [1501, 1950], [1501, 1967], [1501, 1971], [1501, 1980], [1501, 2028], [1501, 2039], [1501, 2058], [1501, 2068], [1501, 2077], [1501, 2096], [1501, 2102], [1501, 2174], [1501, 3632], [1502, 1503], [1502, 1504], [1502, 1625], [1502, 3633], [1503, 1504], [1503, 1626], [1503, 1766], [1503, 3634], [1504, 1639], [1504, 1662], [1504, 1724], [1504, 1791], [1504, 1803], [1504, 2030], [1504, 2031], [1504, 3635], [1505, 1506], [1505, 1577], [1505, 1582], [1505, 1600], [1505, 1602], [1505, 1627], [1505, 1656], [1505, 1665], [1505, 3636], [1506, 1507], [1506, 1592], [1506, 1865], [1506, 1868], [1506, 1879], [1506, 1880], [1506, 1935], [1506, 3637], [1507, 1523], [1507, 1577], [1507, 1579], [1507, 1659], [1507, 1864], [1507, 1871], [1507, 1894], [1507, 1926], [1507, 1958], [1507, 3638], [1508, 1509], [1508, 1511], [1508, 1514], [1508, 1587], [1508, 1634], [1508, 2044], [1508, 3639], [1509, 1510], [1509, 1543], [1509, 1712], [1509, 3640], [1510, 1542], [1510, 1612], [1510, 1984], [1510, 1999], [1510, 2044], [1510, 2052], [1510, 2075], [1510, 3641], [1511, 1512], [1511, 1634], [1511, 1751], [1511, 1785], [1511, 1898], [1511, 2098], [1511, 3642], [1512, 1651], [1512, 1729], [1512, 1771], [1512, 1791], [1512, 1856], [1512, 2081], [1512, 2124], [1512, 3643], [1513, 1514], [1513, 1597], [1513, 1599], [1513, 1638], [1513, 1653], [1513, 3644], [1514, 1515], [1514, 1591], [1514, 1748], [1514, 1771], [1514, 1785], [1514, 1800], [1514, 2044], [1514, 2119], [1514, 3645], [1515, 1516], [1515, 1518], [1515, 1526], [1515, 1560], [1515, 1690], [1515, 1779], [1515, 1830], [1515, 1910], [1515, 1977], [1515, 1988], [1515, 2036], [1515, 2045], [1515, 2097], [1515, 2131], [1515, 3646], [1516, 1517], [1516, 1525], [1516, 1603], [1516, 1738], [1516, 3647], [1517, 1518], [1517, 1520], [1517, 1531], [1517, 3648], [1518, 1521], [1518, 1532], [1518, 1582], [1518, 1923], [1518, 2008], [1518, 3649], [1519, 1520], [1519, 1668], [1519, 1792], [1519, 3650], [1520, 1521], [1520, 1531], [1520, 1668], [1520, 3651], [1521, 1532], [1521, 1623], [1521, 1774], [1521, 1979], [1521, 2078], [1521, 3652], [1522, 1523], [1522, 1524], [1522, 1577], [1522, 1578], [1522, 1848], [1522, 3653], [1523, 1579], [1523, 1899], [1523, 2070], [1523, 2081], [1523, 2108], [1523, 2109], [1523, 3654], [1524, 1525], [1524, 1546], [1524, 1653], [1524, 3655], [1525, 1526], [1525, 1810], [1525, 3656], [1526, 1632], [1526, 1809], [1526, 1898], [1526, 1963], [1526, 2059], [1526, 2100], [1526, 3657], [1527, 1528], [1527, 1625], [1527, 1738], [1527, 1752], [1527, 1792], [1527, 3658], [1528, 1529], [1528, 1569], [1528, 1653], [1528, 1694], [1528, 1752], [1528, 3659], [1529, 1544], [1529, 1564], [1529, 1606], [1529, 1608], [1529, 1678], [1529, 1694], [1529, 1696], [1529, 3660], [1530, 1531], [1530, 1725], [1530, 3661], [1531, 1532], [1531, 1614], [1531, 3662], [1532, 1533], [1532, 1535], [1532, 1615], [1532, 1787], [1532, 2109], [1532, 2118], [1532, 3663], [1533, 1536], [1533, 1784], [1533, 1991], [1533, 2008], [1533, 2014], [1533, 2068], [1533, 2087], [1533, 3664], [1534, 1535], [1534, 1536], [1534, 1540], [1534, 1646], [1534, 1703], [1534, 1733], [1534, 1893], [1534, 1913], [1534, 1998], [1534, 3665], [1535, 1536], [1535, 1582], [1535, 1984], [1535, 3666], [1536, 1559], [1536, 1687], [1536, 1891], [1536, 1924], [1536, 1990], [1536, 2011], [1536, 2082], [1536, 2083], [1536, 2084], [1536, 3667], [1537, 1538], [1537, 1652], [1537, 1732], [1537, 1893], [1537, 3668], [1538, 1539], [1538, 1542], [1538, 1766], [1538, 1951], [1538, 2057], [1538, 3669], [1539, 1559], [1539, 1671], [1539, 1742], [1539, 1957], [1539, 1958], [1539, 2076], [1539, 3670], [1540, 1541], [1540, 1658], [1540, 2048], [1540, 3671], [1541, 1542], [1541, 1615], [1541, 1617], [1541, 1628], [1541, 1658], [1541, 1769], [1541, 1797], [1541, 1984], [1541, 3672], [1542, 1599], [1542, 1617], [1542, 1657], [1542, 1715], [1542, 1720], [1542, 1905], [1542, 3673], [1543, 1544], [1543, 1607], [1543, 1712], [1543, 3674], [1544, 1545], [1544, 1563], [1544, 1618], [1544, 1684], [1544, 1687], [1544, 1699], [1544, 1751], [1544, 1780], [1544, 3675], [1545, 1687], [1545, 1739], [1545, 1753], [1545, 1923], [1545, 1939], [1545, 3676], [1546, 1547], [1546, 1548], [1546, 1554], [1546, 1757], [1546, 1758], [1546, 3677], [1547, 1548], [1547, 1555], [1547, 1791], [1547, 3678], [1548, 1556], [1548, 1717], [1548, 1719], [1548, 3679], [1549, 1550], [1549, 1590], [1549, 1595], [1549, 1596], [1549, 1600], [1549, 1663], [1549, 1665], [1549, 3680], [1550, 1551], [1550, 1776], [1550, 3681], [1551, 1593], [1551, 1622], [1551, 1627], [1551, 1735], [1551, 1777], [1551, 1870], [1551, 3682], [1552, 1553], [1552, 1596], [1552, 1632], [1552, 1647], [1552, 1817], [1552, 1931], [1552, 1976], [1552, 2004], [1552, 2019], [1552, 3683], [1553, 1632], [1553, 1670], [1553, 1762], [1553, 1872], [1553, 1921], [1553, 1928], [1553, 1966], [1553, 1973], [1553, 1974], [1553, 1975], [1553, 1978], [1553, 2020], [1553, 2028], [1553, 2093], [1553, 2098], [1553, 2104], [1553, 2122], [1553, 3684], [1554, 1555], [1554, 1556], [1554, 1771], [1554, 3685], [1555, 1556], [1555, 1644], [1555, 1674], [1555, 1766], [1555, 3686], [1556, 1706], [1556, 1765], [1556, 1783], [1556, 2020], [1556, 2062], [1556, 3687], [1557, 1558], [1557, 1635], [1557, 1649], [1557, 1731], [1557, 1746], [1557, 1997], [1557, 3688], [1558, 1559], [1558, 1568], [1558, 3689], [1559, 1568], [1559, 1670], [1559, 1686], [1559, 1697], [1559, 1760], [1559, 1784], [1559, 1920], [1559, 1949], [1559, 3690], [1560, 1561], [1560, 1790], [1560, 1810], [1560, 2110], [1560, 2128], [1560, 3691], [1561, 1796], [1561, 1848], [1561, 3692], [1562, 1563], [1562, 1564], [1562, 1625], [1562, 1635], [1562, 1636], [1562, 1657], [1562, 3693], [1563, 1564], [1563, 1778], [1563, 3694], [1564, 3695], [1565, 1566], [1565, 1649], [1565, 1752], [1565, 3696], [1566, 1567], [1566, 1649], [1566, 1753], [1566, 2048], [1566, 3697], [1567, 1609], [1567, 1718], [1567, 1749], [1567, 1902], [1567, 1941], [1567, 2129], [1567, 3698], [1568, 1569], [1568, 1570], [1568, 1688], [1568, 3699], [1569, 1570], [1569, 1653], [1569, 1709], [1569, 1710], [1569, 3700], [1570, 1605], [1570, 1730], [1570, 1735], [1570, 1764], [1570, 3701], [1571, 1572], [1571, 1573], [1571, 1604], [1571, 1693], [1571, 3702], [1572, 1573], [1572, 1588], [1572, 1596], [1572, 1622], [1572, 1627], [1572, 1712], [1572, 1713], [1572, 1976], [1572, 1989], [1572, 3703], [1573, 1574], [1573, 1623], [1573, 1702], [1573, 1758], [1573, 3704], [1574, 1575], [1574, 1712], [1574, 1714], [1574, 1720], [1574, 1744], [1574, 1745], [1574, 1792], [1574, 3705], [1575, 1707], [1575, 1745], [1575, 1794], [1575, 3706], [1576, 1577], [1576, 1581], [1576, 1612], [1576, 1804], [1576, 1888], [1576, 1926], [1576, 1964], [1576, 3707], [1577, 1578], [1577, 1581], [1577, 1583], [1577, 1627], [1577, 1658], [1577, 1957], [1577, 3708], [1578, 1579], [1578, 1804], [1578, 1890], [1578, 3709], [1579, 1927], [1579, 2017], [1579, 2102], [1579, 2105], [1579, 3710], [1580, 1581], [1580, 1583], [1580, 1699], [1580, 1810], [1580, 3711], [1581, 1582], [1581, 1799], [1581, 1809], [1581, 1959], [1581, 1962], [1581, 2056], [1581, 2057], [1581, 3712], [1582, 1611], [1582, 1617], [1582, 1665], [1582, 1687], [1582, 1693], [1582, 1880], [1582, 1992], [1582, 3713], [1583, 1584], [1583, 1629], [1583, 3714], [1584, 1585], [1584, 1673], [1584, 3715], [1585, 1597], [1585, 1603], [1585, 1658], [1585, 1673], [1585, 1769], [1585, 1788], [1585, 3716], [1586, 1587], [1586, 1591], [1586, 1595], [1586, 1597], [1586, 1600], [1586, 1603], [1586, 1727], [1586, 1728], [1586, 3717], [1587, 1588], [1587, 1591], [1587, 1598], [1587, 1712], [1587, 1714], [1587, 1819], [1587, 3718], [1588, 1589], [1588, 1595], [1588, 1596], [1588, 1700], [1588, 3719], [1589, 1590], [1589, 1591], [1589, 1595], [1589, 1622], [1589, 1711], [1589, 1759], [1589, 1962], [1589, 3720], [1590, 1592], [1590, 1596], [1590, 1622], [1590, 1868], [1590, 1917], [1590, 3721], [1591, 1592], [1591, 1682], [1591, 1683], [1591, 1686], [1591, 1707], [1591, 1808], [1591, 1819], [1591, 1856], [1591, 1928], [1591, 2124], [1591, 3722], [1592, 1593], [1592, 1600], [1592, 1707], [1592, 1866], [1592, 1958], [1592, 3723], [1593, 1594], [1593, 1615], [1593, 1665], [1593, 1677], [1593, 1705], [1593, 1765], [1593, 1768], [1593, 1981], [1593, 3724], [1594, 1675], [1594, 1677], [1594, 1678], [1594, 1742], [1594, 1832], [1594, 1980], [1594, 2055], [1594, 3725], [1595, 1710], [1595, 1729], [1595, 3726], [1596, 1758], [1596, 1766], [1596, 1873], [1596, 1917], [1596, 1932], [1596, 1942], [1596, 1961], [1596, 3727], [1597, 1598], [1597, 1629], [1597, 1653], [1597, 1770], [1597, 1771], [1597, 3728], [1598, 1599], [1598, 1652], [1598, 3729], [1599, 1631], [1599, 1633], [1599, 1652], [1599, 1715], [1599, 1904], [1599, 2044], [1599, 3730], [1600, 1601], [1600, 3731], [1601, 1602], [1601, 1638], [1601, 1678], [1601, 1686], [1601, 1754], [1601, 3732], [1602, 1636], [1602, 1687], [1602, 1754], [1602, 1949], [1602, 1957], [1602, 3733], [1603, 1604], [1603, 1605], [1603, 1650], [1603, 1780], [1603, 3734], [1604, 1605], [1604, 1764], [1604, 1769], [1604, 1781], [1604, 3735], [1605, 1640], [1605, 1669], [1605, 1778], [1605, 1782], [1605, 3736], [1606, 1607], [1606, 1673], [1606, 1696], [1606, 3737], [1607, 1608], [1607, 1681], [1607, 3738], [1608, 1609], [1608, 1708], [1608, 3739], [1609, 1655], [1609, 1902], [1609, 3740], [1610, 1611], [1610, 1617], [1610, 1672], [1610, 1712], [1610, 2000], [1610, 3741], [1611, 1612], [1611, 3742], [1612, 1702], [1612, 1965], [1612, 1971], [1612, 3743], [1613, 1614], [1613, 3744], [1614, 1615], [1614, 1628], [1614, 1725], [1614, 1767], [1614, 1848], [1614, 3745], [1615, 1616], [1615, 1723], [1615, 1767], [1615, 1923], [1615, 2084], [1615, 2085], [1615, 2108], [1615, 2116], [1615, 3746], [1616, 1875], [1616, 2077], [1616, 2078], [1616, 2106], [1616, 2116], [1616, 3747], [1617, 1618], [1617, 1621], [1617, 1664], [1617, 1707], [1617, 1814], [1617, 1933], [1617, 1992], [1617, 2000], [1617, 2057], [1617, 3748], [1618, 1619], [1618, 1644], [1618, 1672], [1618, 1745], [1618, 1965], [1618, 2007], [1618, 3749], [1619, 1642], [1619, 1643], [1619, 1657], [1619, 1679], [1619, 1805], [1619, 1948], [1619, 1964], [1619, 3750], [1620, 1621], [1620, 1623], [1620, 3751], [1621, 1622], [1621, 1664], [1621, 1777], [1621, 1979], [1621, 2097], [1621, 3752], [1622, 1707], [1622, 1870], [1622, 1988], [1622, 1989], [1622, 3753], [1623, 1624], [1623, 3754], [1624, 1758], [1624, 2015], [1624, 2125], [1624, 3755], [1625, 1626], [1625, 1662], [1625, 1778], [1625, 3756], [1626, 1627], [1626, 1657], [1626, 1661], [1626, 1662], [1626, 3757], [1627, 1706], [1627, 1732], [1627, 1734], [1627, 1768], [1627, 1808], [1627, 1810], [1627, 1949], [1627, 3758], [1628, 1629], [1628, 1725], [1628, 3759], [1629, 1630], [1629, 1756], [1629, 1848], [1629, 3760], [1630, 3761], [1631, 1632], [1631, 1634], [1631, 1638], [1631, 1685], [1631, 1812], [1631, 1898], [1631, 1940], [1631, 1955], [1631, 1968], [1631, 3762], [1632, 1633], [1632, 1811], [1632, 1901], [1632, 1903], [1632, 1955], [1632, 1977], [1632, 2038], [1632, 2063], [1632, 3763], [1633, 1716], [1633, 2044], [1633, 2122], [1633, 3764], [1634, 1737], [1634, 1747], [1634, 1748], [1634, 2051], [1634, 3765], [1635, 1636], [1635, 1812], [1635, 3766], [1636, 1637], [1636, 1638], [1636, 1648], [1636, 1812], [1636, 1884], [1636, 1939], [1636, 1954], [1636, 2118], [1636, 3767], [1637, 1648], [1637, 1750], [1637, 1909], [1637, 3768], [1638, 1639], [1638, 1720], [1638, 1739], [1638, 1747], [1638, 3769], [1639, 1640], [1639, 1661], [1639, 1708], [1639, 1814], [1639, 3770], [1640, 1719], [1640, 1974], [1640, 3771], [1641, 1642], [1641, 1646], [1641, 1650], [1641, 1652], [1641, 1657], [1641, 1715], [1641, 3772], [1642, 1643], [1642, 1650], [1642, 1683], [1642, 3773], [1643, 1644], [1643, 1651], [1643, 1652], [1643, 3774], [1644, 1645], [1644, 1646], [1644, 1717], [1644, 1741], [1644, 1748], [1644, 1751], [1644, 1771], [1644, 1820], [1644, 2067], [1644, 3775], [1645, 1717], [1645, 1753], [1645, 1772], [1645, 1820], [1645, 1842], [1645, 1960], [1645, 2006], [1645, 2088], [1645, 3776], [1646, 1647], [1646, 1648], [1646, 1649], [1646, 1944], [1646, 1961], [1646, 1976], [1646, 1986], [1646, 2047], [1646, 3777], [1647, 1648], [1647, 1652], [1647, 1654], [1647, 1893], [1647, 1904], [1647, 1954], [1647, 1968], [1647, 2101], [1647, 3778], [1648, 1655], [1648, 1657], [1648, 1659], [1648, 1905], [1648, 1916], [1648, 1948], [1648, 1972], [1648, 3779], [1649, 1751], [1649, 3780], [1650, 1651], [1650, 1709], [1650, 1729], [1650, 3781], [1651, 1652], [1651, 1771], [1651, 3782], [1652, 1653], [1652, 1654], [1652, 1657], [1652, 1699], [1652, 3783], [1653, 1654], [1653, 3784], [1654, 1655], [1654, 1680], [1654, 1963], [1654, 2006], [1654, 3785], [1655, 1656], [1655, 1657], [1655, 3786], [1656, 1706], [1656, 1707], [1656, 1803], [1656, 1935], [1656, 1972], [1656, 3787], [1657, 1658], [1657, 3788], [1658, 1659], [1658, 1703], [1658, 1709], [1658, 1876], [1658, 3789], [1659, 1703], [1659, 1705], [1659, 1706], [1659, 1862], [1659, 1875], [1659, 1990], [1659, 2057], [1659, 2083], [1659, 3790], [1660, 1661], [1660, 1663], [1660, 1713], [1660, 1721], [1660, 1766], [1660, 3791], [1661, 1678], [1661, 1694], [1661, 1742], [1661, 1755], [1661, 3792], [1662, 1663], [1662, 1690], [1662, 1721], [1662, 1722], [1662, 1791], [1662, 3793], [1663, 1664], [1663, 1710], [1663, 1766], [1663, 1814], [1663, 3794], [1664, 1665], [1664, 1787], [1664, 1867], [1664, 1876], [1664, 2001], [1664, 3795], [1665, 1868], [1665, 1932], [1665, 3796], [1666, 1667], [1666, 1668], [1666, 3797], [1667, 1729], [1667, 1730], [1667, 3798], [1668, 1691], [1668, 1814], [1668, 3799], [1669, 1670], [1669, 1681], [1669, 1967], [1669, 1974], [1669, 1987], [1669, 3800], [1670, 1671], [1670, 1682], [1670, 1711], [1670, 1745], [1670, 1976], [1670, 1977], [1670, 1978], [1670, 2034], [1670, 2036], [1670, 3801], [1671, 1674], [1671, 1745], [1671, 1750], [1671, 1789], [1671, 1794], [1671, 1798], [1671, 1801], [1671, 1863], [1671, 1915], [1671, 2050], [1671, 2104], [1671, 3802], [1672, 1673], [1672, 1744], [1672, 1965], [1672, 3803], [1673, 1674], [1673, 1701], [1673, 3804], [1674, 1696], [1674, 1957], [1674, 1965], [1674, 1976], [1674, 2001], [1674, 2024], [1674, 3805], [1675, 1676], [1675, 1678], [1675, 1696], [1675, 1760], [1675, 1768], [1675, 1886], [1675, 1949], [1675, 1980], [1675, 2018], [1675, 2019], [1675, 3806], [1676, 1677], [1676, 1701], [1676, 1713], [1676, 1768], [1676, 1982], [1676, 3807], [1677, 1713], [1677, 1766], [1677, 1982], [1677, 1989], [1677, 1999], [1677, 2001], [1677, 3808], [1678, 1679], [1678, 1708], [1678, 1713], [1678, 1755], [1678, 3809], [1679, 1680], [1679, 1848], [1679, 3810], [1680, 1735], [1680, 2121], [1680, 3811], [1681, 1682], [1681, 1708], [1681, 1712], [1681, 1756], [1681, 1792], [1681, 3812], [1682, 1683], [1682, 1711], [1682, 1722], [1682, 1727], [1682, 1792], [1682, 1800], [1682, 1802], [1682, 1975], [1682, 3813], [1683, 1715], [1683, 1728], [1683, 1929], [1683, 3814], [1684, 1685], [1684, 1688], [1684, 1691], [1684, 1694], [1684, 1739], [1684, 3815], [1685, 1686], [1685, 1687], [1685, 1737], [1685, 1739], [1685, 1754], [1685, 1773], [1685, 1815], [1685, 2032], [1685, 2066], [1685, 3816], [1686, 1687], [1686, 1966], [1686, 3817], [1687, 1896], [1687, 1922], [1687, 1968], [1687, 1969], [1687, 2004], [1687, 2007], [1687, 2008], [1687, 3818], [1688, 1689], [1688, 3819], [1689, 1690], [1689, 1721], [1689, 2034], [1689, 3820], [1690, 1778], [1690, 1800], [1690, 2034], [1690, 3821], [1691, 1692], [1691, 2032], [1691, 3822], [1692, 1693], [1692, 1977], [1692, 2120], [1692, 3823], [1693, 1989], [1693, 2130], [1693, 2131], [1693, 3824], [1694, 1695], [1694, 3825], [1695, 1696], [1695, 3826], [1696, 1931], [1696, 1943], [1696, 3827], [1697, 1698], [1697, 1699], [1697, 1741], [1697, 1760], [1697, 1783], [1697, 1805], [1697, 1956], [1697, 2040], [1697, 3828], [1698, 1699], [1698, 1790], [1698, 1804], [1698, 2056], [1698, 2076], [1698, 3829], [1699, 1732], [1699, 1740], [1699, 1843], [1699, 3830], [1700, 1701], [1700, 1710], [1700, 1711], [1700, 1741], [1700, 1764], [1700, 1766], [1700, 1828], [1700, 3831], [1701, 1702], [1701, 1744], [1701, 1756], [1701, 1769], [1701, 1886], [1701, 2024], [1701, 3832], [1702, 1744], [1702, 1825], [1702, 1977], [1702, 2045], [1702, 2063], [1702, 2064], [1702, 3833], [1703, 1704], [1703, 1862], [1703, 3834], [1704, 1705], [1704, 1732], [1704, 1733], [1704, 1768], [1704, 1861], [1704, 3835], [1705, 1765], [1705, 1777], [1705, 1832], [1705, 1861], [1705, 2001], [1705, 2107], [1705, 3836], [1706, 1707], [1706, 1709], [1706, 1711], [1706, 1765], [1706, 1794], [1706, 1863], [1706, 1930], [1706, 1936], [1706, 1956], [1706, 1958], [1706, 1962], [1706, 2057], [1706, 2124], [1706, 3837], [1707, 1708], [1707, 1712], [1707, 1814], [1707, 1934], [1707, 1999], [1707, 3838], [1708, 1870], [1708, 3839], [1709, 1710], [1709, 3840], [1710, 1711], [1710, 1727], [1710, 1729], [1710, 1876], [1710, 3841], [1711, 1828], [1711, 1872], [1711, 1875], [1711, 2006], [1711, 2042], [1711, 3842], [1712, 1713], [1712, 1999], [1712, 3843], [1713, 1725], [1713, 3844], [1714, 1715], [1714, 1720], [1714, 1747], [1714, 1770], [1714, 2067], [1714, 3845], [1715, 1716], [1715, 1986], [1715, 3846], [1716, 1977], [1716, 3847], [1717, 1718], [1717, 1752], [1717, 1843], [1717, 3848], [1718, 1719], [1718, 1747], [1718, 1748], [1718, 1749], [1718, 3849], [1719, 3850], [1720, 1758], [1720, 1769], [1720, 1973], [1720, 2075], [1720, 3851], [1721, 1722], [1721, 1725], [1721, 1727], [1721, 3852], [1722, 1723], [1722, 1766], [1722, 1797], [1722, 1821], [1722, 2031], [1722, 2034], [1722, 3853], [1723, 1724], [1723, 1755], [1723, 1765], [1723, 1783], [1723, 1797], [1723, 1804], [1723, 1849], [1723, 2003], [1723, 2012], [1723, 2035], [1723, 2078], [1723, 3854], [1724, 1754], [1724, 1898], [1724, 3855], [1725, 1727], [1725, 1756], [1725, 1876], [1725, 3856], [1726, 1756], [1726, 1816], [1726, 1832], [1726, 1864], [1726, 1879], [1726, 1899], [1726, 1916], [1726, 1946], [1726, 1957], [1726, 1972], [1726, 1975], [1726, 2024], [1726, 2031], [1726, 2046], [1726, 2108], [1726, 3857], [1727, 1728], [1727, 3858], [1728, 1729], [1728, 3859], [1729, 1734], [1729, 1738], [1729, 1747], [1729, 3860], [1730, 1731], [1730, 1734], [1730, 3861], [1731, 1732], [1731, 1734], [1731, 2071], [1731, 3862], [1732, 1733], [1732, 1777], [1732, 1860], [1732, 2057], [1732, 3863], [1733, 1798], [1733, 1806], [1733, 1808], [1733, 2010], [1733, 2011], [1733, 2015], [1733, 3864], [1734, 1735], [1734, 1736], [1734, 3865], [1735, 1736], [1735, 3866], [1736, 1969], [1736, 2071], [1736, 3867], [1737, 1738], [1737, 1740], [1737, 1779], [1737, 1829], [1737, 3868], [1738, 1739], [1738, 1747], [1738, 1778], [1738, 1779], [1738, 1843], [1738, 3869], [1739, 1939], [1739, 3870], [1740, 1741], [1740, 1994], [1740, 3871], [1741, 1742], [1741, 1764], [1741, 1886], [1741, 2009], [1741, 3872], [1742, 1761], [1742, 1765], [1742, 1766], [1742, 1832], [1742, 1930], [1742, 1975], [1742, 3873], [1743, 1744], [1743, 1792], [1743, 1793], [1743, 2026], [1743, 3874], [1744, 1745], [1744, 2025], [1744, 3875], [1745, 1793], [1745, 1973], [1745, 1999], [1745, 2025], [1745, 2067], [1745, 3876], [1746, 1747], [1746, 1749], [1746, 1903], [1746, 3877], [1747, 1748], [1747, 3878], [1748, 1779], [1748, 1785], [1748, 1903], [1748, 2051], [1748, 2067], [1748, 2129], [1748, 3879], [1749, 1750], [1749, 1784], [1749, 1841], [1749, 1843], [1749, 1903], [1749, 3880], [1750, 1790], [1750, 1801], [1750, 1930], [1750, 1963], [1750, 3881], [1751, 1752], [1751, 1753], [1751, 2004], [1751, 3882], [1752, 1753], [1752, 1856], [1752, 3883], [1753, 1785], [1753, 1855], [1753, 1878], [1753, 1892], [1753, 1960], [1753, 2004], [1753, 3884], [1754, 1755], [1754, 1756], [1754, 1816], [1754, 2053], [1754, 3885], [1755, 1756], [1755, 1773], [1755, 1832], [1755, 1885], [1755, 3886], [1756, 1848], [1756, 3887], [1757, 1758], [1757, 2063], [1757, 3888], [1758, 1759], [1758, 1874], [1758, 1918], [1758, 2006], [1758, 2063], [1758, 2075], [1758, 3889], [1759, 1902], [1759, 3890], [1760, 1761], [1760, 1762], [1760, 2003], [1760, 2039], [1760, 2084], [1760, 3891], [1761, 1762], [1761, 1975], [1761, 3892], [1762, 1763], [1762, 1764], [1762, 1828], [1762, 1979], [1762, 2009], [1762, 2019], [1762, 2077], [1762, 2114], [1762, 2122], [1762, 3893], [1763, 1769], [1763, 1839], [1763, 1844], [1763, 1973], [1763, 1981], [1763, 2024], [1763, 2065], [1763, 2086], [1763, 2108], [1763, 3894], [1764, 1765], [1764, 1769], [1764, 1771], [1764, 3895], [1765, 1766], [1765, 2114], [1765, 2115], [1765, 3896], [1766, 1950], [1766, 2001], [1766, 3897], [1767, 1768], [1767, 2084], [1767, 3898], [1768, 1769], [1768, 1808], [1768, 1981], [1768, 3899], [1769, 1786], [1769, 1848], [1769, 1886], [1769, 2045], [1769, 3900], [1770, 1771], [1770, 3901], [1771, 1772], [1771, 3902], [1772, 1794], [1772, 1962], [1772, 2006], [1772, 3903], [1773, 1774], [1773, 1775], [1773, 1831], [1773, 1938], [1773, 2035], [1773, 2056], [1773, 2113], [1773, 3904], [1774, 1775], [1774, 1846], [1774, 1885], [1774, 2011], [1774, 2027], [1774, 2032], [1774, 3905], [1775, 1778], [1775, 1779], [1775, 1908], [1775, 1937], [1775, 2023], [1775, 2094], [1775, 3906], [1776, 1777], [1776, 2054], [1776, 2113], [1776, 3907], [1777, 1860], [1777, 3908], [1778, 1782], [1778, 3909], [1779, 1829], [1779, 1842], [1779, 1969], [1779, 1977], [1779, 2008], [1779, 3910], [1780, 1781], [1780, 1782], [1780, 3911], [1781, 1782], [1781, 3912], [1782, 1908], [1782, 1987], [1782, 3913], [1783, 1784], [1783, 1788], [1783, 1843], [1783, 3914], [1784, 1785], [1784, 1795], [1784, 1800], [1784, 2074], [1784, 2093], [1784, 3915], [1785, 2098], [1785, 3916], [1786, 1797], [1786, 3917], [1787, 1794], [1787, 1797], [1787, 1798], [1787, 2012], [1787, 3918], [1788, 1789], [1788, 3919], [1789, 1790], [1789, 2036], [1789, 3920], [1790, 1909], [1790, 1956], [1790, 2076], [1790, 3921], [1791, 1792], [1791, 1795], [1791, 3922], [1792, 1793], [1792, 1856], [1792, 3923], [1793, 1802], [1793, 1855], [1793, 1866], [1793, 1886], [1793, 2013], [1793, 2026], [1793, 3924], [1794, 1795], [1794, 2105], [1794, 3925], [1795, 1796], [1795, 3926], [1796, 3927], [1797, 1798], [1797, 3928], [1798, 1799], [1798, 1821], [1798, 1849], [1798, 2001], [1798, 2015], [1798, 2037], [1798, 2057], [1798, 2064], [1798, 3929], [1799, 1809], [1799, 1836], [1799, 1847], [1799, 1851], [1799, 1881], [1799, 1888], [1799, 1925], [1799, 1932], [1799, 1981], [1799, 1984], [1799, 2043], [1799, 2059], [1799, 2086], [1799, 2125], [1799, 3930], [1800, 1801], [1800, 3931], [1801, 1802], [1801, 1914], [1801, 1921], [1801, 2074], [1801, 2089], [1801, 2119], [1801, 3932], [1802, 1821], [1802, 1928], [1802, 1929], [1802, 1930], [1802, 2042], [1802, 3933], [1803, 1804], [1803, 1814], [1803, 3934], [1804, 1805], [1804, 1889], [1804, 1909], [1804, 1927], [1804, 2022], [1804, 2118], [1804, 3935], [1805, 1886], [1805, 3936], [1806, 1807], [1806, 1846], [1806, 1860], [1806, 1861], [1806, 1913], [1806, 3937], [1807, 1808], [1807, 1839], [1807, 1924], [1807, 1928], [1807, 1981], [1807, 3938], [1808, 1942], [1808, 3939], [1809, 1810], [1809, 1919], [1809, 1963], [1809, 2110], [1809, 3940], [1810, 3941], [1811, 1812], [1811, 1852], [1811, 1899], [1811, 1901], [1811, 1931], [1811, 3942], [1812, 1813], [1812, 1884], [1812, 1899], [1812, 1946], [1812, 1968], [1812, 1997], [1812, 2010], [1812, 3943], [1813, 1823], [1813, 1862], [1813, 1912], [1813, 1915], [1813, 1932], [1813, 1987], [1813, 2001], [1813, 2070], [1813, 2071], [1813, 2084], [1813, 2126], [1813, 3944], [1814, 1978], [1814, 3945], [1815, 1816], [1815, 1829], [1815, 1831], [1815, 1881], [1815, 1940], [1815, 1993], [1815, 3946], [1816, 1817], [1816, 1832], [1816, 1896], [1816, 1993], [1816, 2053], [1816, 3947], [1817, 1829], [1817, 1845], [1817, 1903], [1817, 1964], [1817, 1965], [1817, 1994], [1817, 2004], [1817, 2010], [1817, 2026], [1817, 2046], [1817, 2093], [1817, 3948], [1818, 1819], [1818, 1995], [1818, 1997], [1818, 2000], [1818, 2002], [1818, 2090], [1818, 3949], [1819, 1928], [1819, 1999], [1819, 2005], [1819, 2018], [1819, 2044], [1819, 2067], [1819, 3950], [1820, 1944], [1820, 2004], [1820, 2007], [1820, 2009], [1820, 2058], [1820, 3951], [1821, 1849], [1821, 1950], [1821, 2010], [1821, 2013], [1821, 2068], [1821, 3952], [1822, 1823], [1822, 1861], [1822, 1862], [1822, 2016], [1822, 2018], [1822, 2020], [1822, 2021], [1822, 3953], [1823, 1938], [1823, 3954], [1824, 1833], [1824, 1967], [1824, 2023], [1824, 2092], [1824, 3955], [1825, 1826], [1825, 1835], [1825, 1971], [1825, 2024], [1825, 2025], [1825, 3956], [1826, 1910], [1826, 1919], [1826, 1977], [1826, 2052], [1826, 2120], [1826, 3957], [1827, 1828], [1827, 1829], [1827, 2041], [1827, 2043], [1827, 2046], [1827, 2049], [1827, 2051], [1827, 3958], [1828, 1863], [1828, 1950], [1828, 1996], [1828, 2009], [1828, 2024], [1828, 2042], [1828, 3959], [1829, 1830], [1829, 1994], [1829, 2051], [1829, 3960], [1830, 1881], [1830, 1882], [1830, 1910], [1830, 2018], [1830, 2021], [1830, 2023], [1830, 2034], [1830, 2043], [1830, 2044], [1830, 2100], [1830, 2128], [1830, 3961], [1831, 1832], [1831, 2055], [1831, 2077], [1831, 3962], [1832, 1849], [1832, 2001], [1832, 2112], [1832, 2117], [1832, 3963], [1833, 1834], [1833, 1952], [1833, 2069], [1833, 2102], [1833, 3964], [1834, 2021], [1834, 2023], [1834, 2069], [1834, 2080], [1834, 2091], [1834, 2099], [1834, 3965], [1835, 1836], [1835, 1850], [1835, 1994], [1835, 2026], [1835, 2072], [1835, 3966], [1836, 1851], [1836, 1887], [1836, 2027], [1836, 2072], [1836, 2120], [1836, 3967], [1837, 1838], [1837, 1852], [1837, 2039], [1837, 2076], [1837, 3968], [1838, 1839], [1838, 1851], [1838, 1864], [1838, 1918], [1838, 2029], [1838, 2037], [1838, 3969], [1839, 1859], [1839, 1865], [1839, 2027], [1839, 2033], [1839, 2088], [1839, 2089], [1839, 3970], [1840, 1841], [1840, 1842], [1840, 1844], [1840, 1914], [1840, 1918], [1840, 1927], [1840, 2086], [1840, 2087], [1840, 2088], [1840, 2090], [1840, 2092], [1840, 3971], [1841, 1842], [1841, 1845], [1841, 1903], [1841, 1919], [1841, 2074], [1841, 3972], [1842, 1843], [1842, 2008], [1842, 3973], [1843, 3974], [1844, 1845], [1844, 1886], [1844, 1980], [1844, 2009], [1844, 2024], [1844, 3975], [1845, 1878], [1845, 1948], [1845, 2007], [1845, 2074], [1845, 2117], [1845, 3976], [1846, 2058], [1846, 2094], [1846, 2113], [1846, 3977], [1847, 1849], [1847, 1897], [1847, 1945], [1847, 2002], [1847, 2072], [1847, 2106], [1847, 2108], [1847, 2110], [1847, 2112], [1847, 2114], [1847, 2116], [1847, 3978], [1848, 2108], [1848, 3979], [1849, 1927], [1849, 2003], [1849, 2011], [1849, 2012], [1849, 2014], [1849, 2065], [1849, 2085], [1849, 2114], [1849, 3980], [1850, 1851], [1850, 1891], [1850, 2117], [1850, 3981], [1851, 1854], [1851, 2094], [1851, 2107], [1851, 3982], [1852, 2021], [1852, 2119], [1852, 2120], [1852, 3983], [1853, 1854], [1853, 1904], [1853, 2037], [1853, 2077], [1853, 2122], [1853, 3984], [1854, 1869], [1854, 1894], [1854, 1901], [1854, 1919], [1854, 1998], [1854, 2038], [1854, 2082], [1854, 2121], [1854, 2123], [1854, 3985], [1855, 1856], [1855, 1866], [1855, 1928], [1855, 2088], [1855, 3986], [1856, 3987], [1857, 1858], [1857, 1907], [1857, 1947], [1857, 2016], [1857, 2029], [1857, 2041], [1857, 3988], [1858, 1859], [1858, 1871], [1858, 1894], [1858, 1905], [1858, 1914], [1858, 1918], [1858, 1925], [1858, 3989], [1859, 1865], [1859, 2127], [1859, 3990], [1860, 1861], [1860, 1893], [1860, 1952], [1860, 2069], [1860, 2071], [1860, 2113], [1860, 3991], [1861, 1862], [1861, 1981], [1861, 2115], [1861, 3992], [1862, 1863], [1862, 1913], [1862, 1915], [1862, 3993], [1863, 1921], [1863, 1936], [1863, 1959], [1863, 1972], [1863, 1983], [1863, 1999], [1863, 2020], [1863, 2040], [1863, 2049], [1863, 2115], [1863, 3994], [1864, 1865], [1864, 1867], [1864, 1869], [1864, 1871], [1864, 1872], [1864, 1875], [1864, 1993], [1864, 1995], [1864, 2080], [1864, 2106], [1864, 3995], [1865, 1866], [1865, 1869], [1865, 1922], [1865, 1943], [1865, 3996], [1866, 1872], [1866, 3997], [1867, 1868], [1867, 1875], [1867, 1912], [1867, 1933], [1867, 2001], [1867, 3998], [1868, 1880], [1868, 1912], [1868, 1932], [1868, 3999], [1869, 1870], [1869, 1901], [1869, 1967], [1869, 1970], [1869, 1989], [1869, 1996], [1869, 2019], [1869, 2042], [1869, 4000], [1870, 1902], [1870, 2121], [1870, 4001], [1871, 1915], [1871, 1932], [1871, 1971], [1871, 1972], [1871, 1989], [1871, 2016], [1871, 2079], [1871, 4002], [1872, 1873], [1872, 1874], [1872, 1958], [1872, 1975], [1872, 4003], [1873, 1874], [1873, 1953], [1873, 1974], [1873, 4004], [1874, 4005], [1875, 1876], [1875, 2083], [1875, 4006], [1876, 4007], [1877, 1879], [1877, 1880], [1877, 1882], [1877, 1917], [1877, 2096], [1877, 2100], [1877, 4008], [1878, 1901], [1878, 1907], [1878, 2004], [1878, 4009], [1879, 1896], [1879, 2070], [1879, 2101], [1879, 2103], [1879, 2112], [1879, 4010], [1880, 1881], [1880, 1882], [1880, 1907], [1880, 1933], [1880, 4011], [1881, 2094], [1881, 2100], [1881, 2120], [1881, 4012], [1882, 1988], [1882, 1989], [1882, 2079], [1882, 4013], [1883, 1884], [1883, 1922], [1883, 1973], [1883, 2102], [1883, 4014], [1884, 1912], [1884, 1919], [1884, 1947], [1884, 2117], [1884, 2129], [1884, 4015], [1885, 1886], [1885, 2015], [1885, 4016], [1886, 4017], [1887, 2131], [1887, 4018], [1888, 1889], [1888, 1946], [1888, 2027], [1888, 4019], [1889, 1890], [1889, 2086], [1889, 2118], [1889, 4020], [1890, 2086], [1890, 4021], [1891, 1893], [1891, 1896], [1891, 1926], [1891, 2087], [1891, 4022], [1892, 2047], [1892, 4023], [1893, 1894], [1893, 1925], [1893, 1951], [1893, 1998], [1893, 2101], [1893, 4024], [1894, 1895], [1894, 4025], [1895, 1897], [1895, 1918], [1895, 2030], [1895, 2054], [1895, 2087], [1895, 2101], [1895, 2106], [1895, 4026], [1896, 1897], [1896, 2004], [1896, 2112], [1896, 4027], [1897, 1898], [1897, 1940], [1897, 2098], [1897, 4028], [1898, 2075], [1898, 4029], [1899, 1900], [1899, 1955], [1899, 4030], [1900, 1901], [1900, 1955], [1900, 1968], [1900, 4031], [1901, 1902], [1901, 1928], [1901, 1968], [1901, 2005], [1901, 2119], [1901, 2129], [1901, 4032], [1902, 2121], [1902, 4033], [1903, 1997], [1903, 2090], [1903, 2093], [1903, 2129], [1903, 4034], [1904, 1905], [1904, 1940], [1904, 1986], [1904, 4035], [1905, 1921], [1905, 1933], [1905, 1947], [1905, 1951], [1905, 1972], [1905, 1973], [1905, 1986], [1905, 4036], [1906, 1907], [1906, 1974], [1906, 1987], [1906, 2004], [1906, 2010], [1906, 2069], [1906, 2070], [1906, 4037], [1907, 1908], [1907, 2094], [1907, 2112], [1907, 2117], [1907, 4038], [1908, 1912], [1908, 1987], [1908, 2082], [1908, 4039], [1909, 1927], [1909, 2012], [1909, 2061], [1909, 4040], [1910, 1911], [1910, 2017], [1910, 2055], [1910, 2110], [1910, 2111], [1910, 2119], [1910, 4041], [1911, 2041], [1911, 2043], [1911, 2119], [1911, 4042], [1912, 2080], [1912, 2082], [1912, 2085], [1912, 4043], [1913, 1914], [1913, 1921], [1913, 1944], [1913, 2126], [1913, 4044], [1914, 1915], [1914, 2089], [1914, 2126], [1914, 4045], [1915, 2079], [1915, 2102], [1915, 4046], [1916, 1919], [1916, 1957], [1916, 2047], [1916, 2048], [1916, 2083], [1916, 2095], [1916, 4047], [1917, 1918], [1917, 1950], [1917, 1989], [1917, 4048], [1918, 1973], [1918, 2063], [1918, 4049], [1919, 1920], [1919, 1997], [1919, 2082], [1919, 2094], [1919, 2110], [1919, 4050], [1920, 1921], [1920, 1947], [1920, 2039], [1920, 2040], [1920, 2074], [1920, 4051], [1921, 1951], [1921, 1983], [1921, 4052], [1922, 1924], [1922, 1967], [1922, 2032], [1922, 4053], [1923, 1938], [1923, 4054], [1924, 1929], [1924, 2073], [1924, 2084], [1924, 2126], [1924, 4055], [1925, 1959], [1925, 1982], [1925, 1983], [1925, 2063], [1925, 2127], [1925, 4056], [1926, 1927], [1926, 1971], [1926, 1990], [1926, 2049], [1926, 4057], [1927, 1991], [1927, 2017], [1927, 2022], [1927, 2050], [1927, 2065], [1927, 2076], [1927, 2087], [1927, 4058], [1928, 1929], [1928, 1934], [1928, 1960], [1928, 2055], [1928, 2088], [1928, 2103], [1928, 2119], [1928, 4059], [1929, 1947], [1929, 1986], [1929, 2041], [1929, 2068], [1929, 2073], [1929, 2093], [1929, 2129], [1929, 4060], [1930, 1949], [1930, 1956], [1930, 1960], [1930, 4061], [1931, 1932], [1931, 1980], [1931, 2112], [1931, 4062], [1932, 1981], [1932, 2001], [1932, 2112], [1932, 4063], [1933, 1934], [1933, 1937], [1933, 1952], [1933, 1978], [1933, 2000], [1933, 2007], [1933, 2065], [1933, 2117], [1933, 4064], [1934, 1935], [1934, 1936], [1934, 1978], [1934, 1999], [1934, 4065], [1935, 1936], [1935, 1972], [1935, 2103], [1935, 4066], [1936, 1945], [1936, 1952], [1936, 2042], [1936, 2062], [1936, 2114], [1936, 4067], [1937, 1938], [1937, 1939], [1937, 1985], [1937, 2008], [1937, 2130], [1937, 4068], [1938, 1939], [1938, 4069], [1939, 4070], [1940, 2051], [1940, 2052], [1940, 4071], [1941, 1985], [1941, 2126], [1941, 4072], [1942, 2124], [1942, 4073], [1943, 1982], [1943, 1995], [1943, 1996], [1943, 2024], [1943, 2027], [1943, 2127], [1943, 4074], [1944, 1945], [1944, 2129], [1944, 4075], [1945, 2041], [1945, 2049], [1945, 2055], [1945, 2076], [1945, 4076], [1946, 1947], [1946, 1964], [1946, 2025], [1946, 2026], [1946, 2084], [1946, 4077], [1947, 1948], [1947, 2085], [1947, 4078], [1948, 1964], [1948, 2007], [1948, 4079], [1949, 1957], [1949, 1969], [1949, 1970], [1949, 1976], [1949, 4080], [1950, 1951], [1950, 2114], [1950, 4081], [1951, 1952], [1951, 2104], [1951, 4082], [1952, 2057], [1952, 4083], [1953, 4084], [1954, 1955], [1954, 2081], [1954, 4085], [1955, 1975], [1955, 2095], [1955, 4086], [1956, 2040], [1956, 4087], [1957, 1964], [1957, 4088], [1958, 2105], [1958, 2124], [1958, 4089], [1959, 1961], [1959, 4090], [1960, 1961], [1960, 1962], [1960, 1963], [1960, 1966], [1960, 4091], [1961, 1962], [1961, 2006], [1961, 4092], [1962, 4093], [1963, 2119], [1963, 4094], [1964, 1965], [1964, 4095], [1965, 2000], [1965, 2007], [1965, 2025], [1965, 2052], [1965, 2066], [1965, 4096], [1966, 2020], [1966, 2124], [1966, 4097], [1967, 1969], [1967, 2018], [1967, 4098], [1968, 4099], [1969, 1970], [1969, 2018], [1969, 4100], [1970, 2019], [1970, 4101], [1971, 1972], [1971, 4102], [1972, 1999], [1972, 2024], [1972, 4103], [1973, 2065], [1973, 2067], [1973, 2104], [1973, 4104], [1974, 2101], [1974, 4105], [1975, 2031], [1975, 4106], [1976, 1977], [1976, 4107], [1977, 4108], [1978, 1979], [1978, 4109], [1979, 2114], [1979, 2116], [1979, 4110], [1980, 1981], [1980, 1982], [1980, 2039], [1980, 4111], [1981, 1982], [1981, 2084], [1981, 2115], [1981, 4112], [1982, 1983], [1982, 2024], [1982, 4113], [1983, 2027], [1983, 2039], [1983, 4114], [1984, 2043], [1984, 2045], [1984, 2116], [1984, 4115], [1985, 2015], [1985, 4116], [1986, 2027], [1986, 2052], [1986, 2067], [1986, 2089], [1986, 4117], [1987, 2071], [1987, 2130], [1987, 4118], [1988, 2097], [1988, 4119], [1989, 1999], [1989, 2055], [1989, 4120], [1990, 1991], [1990, 1992], [1990, 2049], [1990, 4121], [1991, 1992], [1991, 2050], [1991, 4122], [1992, 2056], [1992, 2097], [1992, 2130], [1992, 4123], [1993, 1994], [1993, 2029], [1993, 2054], [1993, 4124], [1994, 2009], [1994, 2092], [1994, 4125], [1995, 1996], [1995, 2001], [1995, 2037], [1995, 2107], [1995, 4126], [1996, 1998], [1996, 2005], [1996, 4127], [1997, 1998], [1997, 2071], [1997, 2128], [1997, 4128], [1998, 2005], [1998, 2071], [1998, 4129], [1999, 2000], [1999, 4130], [2000, 2001], [2000, 2010], [2000, 2066], [2000, 4131], [2001, 4132], [2002, 2003], [2002, 2049], [2002, 2107], [2002, 2115], [2002, 2128], [2002, 4133], [2003, 2022], [2003, 2066], [2003, 2084], [2003, 2115], [2003, 4134], [2004, 2005], [2004, 2098], [2004, 4135], [2005, 2044], [2005, 4136], [2006, 4137], [2007, 2008], [2007, 2065], [2007, 4138], [2008, 4139], [2009, 2040], [2009, 2092], [2009, 4140], [2010, 2011], [2010, 2013], [2010, 2117], [2010, 4141], [2011, 2014], [2011, 4142], [2012, 2061], [2012, 4143], [2013, 2014], [2013, 2015], [2013, 4144], [2014, 2015], [2014, 4145], [2015, 2125], [2015, 4146], [2016, 2017], [2016, 2055], [2016, 2062], [2016, 2103], [2016, 2112], [2016, 4147], [2017, 2021], [2017, 4148], [2018, 2019], [2018, 2049], [2018, 2055], [2018, 4149], [2019, 4150], [2020, 2062], [2020, 2115], [2020, 4151], [2021, 2022], [2021, 2079], [2021, 4152], [2022, 2066], [2022, 2090], [2022, 2091], [2022, 4153], [2023, 2092], [2023, 2094], [2023, 2111], [2023, 2128], [2023, 4154], [2024, 2025], [2024, 4155], [2025, 2026], [2025, 2052], [2025, 4156], [2026, 2027], [2026, 2073], [2026, 4157], [2027, 2032], [2027, 2037], [2027, 2067], [2027, 2073], [2027, 2127], [2027, 4158], [2028, 2029], [2028, 2031], [2028, 2032], [2028, 2034], [2028, 2037], [2028, 2099], [2028, 2100], [2028, 4159], [2029, 2030], [2029, 2033], [2029, 2092], [2029, 2094], [2029, 4160], [2030, 2031], [2030, 2087], [2030, 4161], [2031, 2091], [2031, 2101], [2031, 4162], [2032, 2033], [2032, 2052], [2032, 2066], [2032, 2120], [2032, 4163], [2033, 2065], [2033, 4164], [2034, 2035], [2034, 4165], [2035, 2036], [2035, 2078], [2035, 2116], [2035, 4166], [2036, 2116], [2036, 4167], [2037, 2038], [2037, 2063], [2037, 2122], [2037, 4168], [2038, 2123], [2038, 4169], [2039, 2040], [2039, 2073], [2039, 4170], [2040, 2076], [2040, 4171], [2041, 2042], [2041, 2072], [2041, 2092], [2041, 4172], [2042, 4173], [2043, 2044], [2043, 4174], [2044, 2051], [2044, 2098], [2044, 2119], [2044, 4175], [2045, 2059], [2045, 2060], [2045, 2075], [2045, 2131], [2045, 4176], [2046, 2047], [2046, 4177], [2047, 2048], [2047, 4178], [2048, 4179], [2049, 2050], [2049, 4180], [2050, 4181], [2051, 2052], [2051, 2098], [2051, 4182], [2052, 4183], [2053, 2095], [2053, 4184], [2054, 2106], [2054, 2113], [2054, 4185], [2055, 4186], [2056, 2057], [2056, 4187], [2057, 4188], [2058, 2059], [2058, 2063], [2058, 2065], [2058, 4189], [2059, 2060], [2059, 2063], [2059, 4190], [2060, 2064], [2060, 2065], [2060, 4191], [2061, 2065], [2061, 4192], [2062, 2114], [2062, 4193], [2063, 2064], [2063, 2090], [2063, 2101], [2063, 4194], [2064, 2065], [2064, 4195], [2065, 2066], [2065, 4196], [2066, 4197], [2067, 2088], [2067, 2090], [2067, 4198], [2068, 2093], [2068, 2096], [2068, 4199], [2069, 2070], [2069, 2071], [2069, 4200], [2070, 2101], [2070, 2102], [2070, 4201], [2071, 2082], [2071, 4202], [2072, 2073], [2072, 2111], [2072, 4203], [2073, 4204], [2074, 2093], [2074, 4205], [2075, 4206], [2076, 2092], [2076, 2128], [2076, 4207], [2077, 2079], [2077, 2082], [2077, 2084], [2077, 4208], [2078, 4209], [2079, 2080], [2079, 4210], [2080, 2081], [2080, 2096], [2080, 4211], [2081, 4212], [2082, 2083], [2082, 4213], [2083, 4214], [2084, 2085], [2084, 4215], [2085, 4216], [2086, 2109], [2086, 2125], [2086, 2131], [2086, 4217], [2087, 2091], [2087, 2109], [2087, 4218], [2088, 2089], [2088, 4219], [2089, 4220], [2090, 2091], [2090, 4221], [2091, 2099], [2091, 2101], [2091, 4222], [2092, 2093], [2092, 4223], [2093, 2098], [2093, 4224], [2094, 2113], [2094, 2120], [2094, 4225], [2095, 4226], [2096, 2098], [2096, 2099], [2096, 4227], [2097, 2116], [2097, 4228], [2098, 2099], [2098, 4229], [2099, 2100], [2099, 4230], [2100, 2101], [2100, 4231], [2101, 2104], [2101, 2107], [2101, 2113], [2101, 4232], [2102, 2103], [2102, 2104], [2102, 4233], [2103, 2104], [2103, 4234], [2104, 2105], [2104, 4235], [2105, 4236], [2106, 2107], [2106, 2108], [2106, 4237], [2107, 2113], [2107, 4238], [2108, 2109], [2108, 4239], [2109, 4240], [2110, 2111], [2110, 2128], [2110, 4241], [2111, 4242], [2112, 2113], [2112, 4243], [2113, 4244], [2114, 2115], [2114, 4245], [2115, 4246], [2116, 4247], [2117, 4248], [2118, 4249], [2119, 4250], [2120, 4251], [2121, 4252], [2122, 2123], [2122, 2124], [2122, 4253], [2123, 2124], [2123, 4254], [2124, 4255], [2125, 2127], [2125, 4256], [2126, 2129], [2126, 4257], [2127, 4258], [2128, 4259], [2129, 4260], [2130, 4261], [2131, 4262], [2132, 2133], [2132, 2138], [2132, 2142], [2132, 2145], [2132, 2146], [2132, 2150], [2132, 2153], [2132, 2156], [2132, 2159], [2132, 2163], [2132, 2167], [2132, 2171], [2132, 2174], [2132, 2175], [2132, 2181], [2132, 2188], [2132, 2192], [2132, 2195], [2132, 2198], [2132, 2201], [2132, 2211], [2132, 2215], [2132, 2223], [2132, 2226], [2132, 2229], [2132, 2233], [2132, 2237], [2132, 2253], [2132, 2274], [2132, 2288], [2132, 2316], [2132, 2342], [2132, 2347], [2132, 2369], [2132, 2373], [2132, 2378], [2132, 2383], [2132, 2387], [2132, 2396], [2132, 2401], [2132, 2405], [2132, 2409], [2132, 2414], [2132, 2421], [2132, 2449], [2132, 2453], [2132, 2461], [2132, 2499], [2132, 2511], [2132, 2540], [2132, 2548], [2132, 2652], [2132, 2672], [2132, 2707], [2132, 2726], [2132, 2750], [2132, 2787], [2132, 2796], [2132, 3198], [2132, 3203], [2132, 3207], [2132, 3210], [2132, 3211], [2132, 3215], [2132, 3218], [2132, 3221], [2132, 3224], [2132, 3228], [2132, 3232], [2132, 3236], [2132, 3239], [2132, 3240], [2132, 3246], [2132, 3253], [2132, 3257], [2132, 3260], [2132, 3263], [2132, 3266], [2132, 3276], [2132, 3280], [2132, 3288], [2132, 3291], [2132, 3294], [2132, 3298], [2132, 3302], [2132, 3318], [2132, 3339], [2132, 3353], [2132, 3381], [2132, 3407], [2132, 3412], [2132, 3434], [2132, 3438], [2132, 3443], [2132, 3448], [2132, 3452], [2132, 3461], [2132, 3466], [2132, 3470], [2132, 3474], [2132, 3479], [2132, 3486], [2132, 3514], [2132, 3518], [2132, 3526], [2132, 3564], [2132, 3576], [2132, 3605], [2132, 3613], [2132, 3717], [2132, 3737], [2132, 3772], [2132, 3791], [2132, 3815], [2132, 3852], [2132, 3861], [2133, 2134], [2133, 2211], [2133, 2242], [2133, 2245], [2133, 2249], [2133, 2369], [2133, 2614], [2134, 2135], [2134, 2200], [2134, 2392], [2134, 2405], [2134, 2447], [2134, 2541], [2134, 2889], [2135, 2136], [2135, 2158], [2135, 2303], [2135, 2412], [2135, 2413], [2135, 2437], [2135, 2500], [2135, 2547], [2135, 2614], [2135, 2889], [2136, 2137], [2136, 2180], [2136, 2200], [2136, 2208], [2136, 2213], [2136, 2301], [2136, 2311], [2136, 2424], [2136, 2438], [2136, 2441], [2136, 2444], [2136, 2547], [2136, 2551], [2136, 2622], [2136, 2651], [2136, 2725], [2136, 2740], [2136, 2771], [2136, 2829], [2136, 2855], [2136, 2871], [2136, 2888], [2136, 2989], [2136, 3082], [2137, 2141], [2137, 2145], [2137, 2149], [2137, 2162], [2137, 2166], [2137, 2170], [2137, 2174], [2137, 2180], [2137, 2184], [2137, 2187], [2137, 2191], [2137, 2204], [2137, 2214], [2137, 2218], [2137, 2241], [2137, 2248], [2137, 2252], [2137, 2263], [2137, 2273], [2137, 2281], [2137, 2295], [2137, 2302], [2137, 2315], [2137, 2320], [2137, 2327], [2137, 2346], [2137, 2351], [2137, 2358], [2137, 2362], [2137, 2368], [2137, 2377], [2137, 2382], [2137, 2391], [2137, 2395], [2137, 2400], [2137, 2404], [2137, 2413], [2137, 2428], [2137, 2435], [2137, 2457], [2137, 2468], [2137, 2484], [2137, 2495], [2137, 2503], [2137, 2510], [2137, 2521], [2137, 2539], [2137, 2544], [2137, 2561], [2137, 2567], [2137, 2599], [2137, 2645], [2137, 2656], [2137, 2660], [2137, 2682], [2137, 2711], [2137, 2829], [2137, 2905], [2137, 2911], [2137, 2923], [2137, 2961], [2137, 2976], [2137, 2978], [2137, 3002], [2138, 2139], [2138, 2160], [2138, 2164], [2138, 2196], [2138, 2215], [2138, 2224], [2138, 2283], [2138, 2299], [2138, 2303], [2138, 2306], [2138, 2309], [2138, 2312], [2138, 2396], [2138, 2410], [2138, 2541], [2138, 2549], [2138, 2679], [2138, 2708], [2138, 2727], [2138, 2763], [2138, 2849], [2139, 2140], [2139, 2247], [2139, 2312], [2139, 2383], [2139, 2447], [2139, 2488], [2139, 2749], [2139, 2809], [2139, 2826], [2139, 3139], [2140, 2141], [2140, 2294], [2140, 2306], [2140, 2354], [2140, 2413], [2140, 2859], [2140, 3080], [2140, 3139], [2141, 2203], [2141, 2227], [2141, 2241], [2141, 2660], [2141, 2826], [2141, 2868], [2141, 2915], [2141, 3022], [2141, 3105], [2141, 3151], [2142, 2143], [2142, 2216], [2142, 2226], [2142, 2240], [2142, 2331], [2142, 2334], [2142, 2337], [2142, 2340], [2142, 2409], [2143, 2144], [2143, 2292], [2143, 2293], [2143, 2331], [2143, 2453], [2143, 2501], [2143, 2502], [2143, 2585], [2143, 2612], [2143, 2857], [2144, 2145], [2144, 2168], [2144, 2218], [2144, 2250], [2144, 2366], [2144, 2371], [2144, 2410], [2144, 2425], [2144, 2555], [2144, 2591], [2144, 2727], [2144, 2733], [2144, 2734], [2144, 2760], [2144, 2852], [2145, 2147], [2145, 2151], [2145, 2155], [2145, 2156], [2145, 2158], [2145, 2169], [2145, 2172], [2145, 2174], [2145, 2178], [2145, 2183], [2145, 2186], [2145, 2197], [2145, 2219], [2145, 2227], [2145, 2231], [2145, 2232], [2145, 2251], [2145, 2296], [2145, 2306], [2145, 2324], [2145, 2337], [2145, 2354], [2145, 2376], [2145, 2399], [2145, 2425], [2145, 2429], [2145, 2432], [2145, 2435], [2145, 2436], [2145, 2439], [2145, 2442], [2145, 2445], [2145, 2480], [2145, 2502], [2145, 2504], [2145, 2554], [2145, 2578], [2145, 2579], [2145, 2581], [2145, 2595], [2145, 2606], [2145, 2614], [2145, 2615], [2145, 2627], [2145, 2630], [2145, 2636], [2145, 2639], [2145, 2644], [2145, 2651], [2145, 2671], [2145, 2693], [2145, 2704], [2145, 2728], [2145, 2729], [2145, 2744], [2145, 2772], [2145, 2783], [2145, 2795], [2145, 2805], [2145, 2835], [2145, 2860], [2145, 2862], [2146, 2147], [2146, 2167], [2146, 2205], [2146, 2245], [2146, 2312], [2146, 2383], [2146, 2469], [2146, 2472], [2146, 2475], [2146, 2478], [2146, 2481], [2146, 2485], [2146, 2492], [2146, 2496], [2146, 2507], [2146, 2575], [2146, 2673], [2146, 2747], [2146, 2813], [2146, 2844], [2146, 2866], [2147, 2148], [2147, 2206], [2147, 2354], [2147, 2472], [2147, 2490], [2147, 2674], [2147, 2774], [2147, 2784], [2147, 2785], [2147, 3007], [2148, 2149], [2148, 2390], [2148, 2394], [2148, 2426], [2148, 2478], [2148, 2494], [2148, 2510], [2148, 2644], [2148, 2956], [2148, 3184], [2149, 2152], [2149, 2187], [2149, 2330], [2149, 2361], [2149, 2427], [2149, 2520], [2149, 2535], [2149, 2570], [2149, 2696], [2149, 2722], [2149, 2774], [2149, 2792], [2149, 2822], [2149, 3119], [2150, 2151], [2150, 2267], [2150, 2340], [2150, 2385], [2150, 2405], [2150, 2496], [2150, 2515], [2150, 2518], [2150, 2522], [2150, 2525], [2150, 2528], [2150, 2531], [2150, 2536], [2150, 2571], [2150, 2610], [2150, 2631], [2150, 2652], [2150, 2779], [2150, 2791], [2150, 2800], [2150, 2846], [2151, 2152], [2151, 2158], [2151, 2338], [2151, 2522], [2151, 2534], [2151, 2535], [2151, 2611], [2151, 2746], [2151, 2801], [2151, 3054], [2152, 2400], [2152, 2442], [2152, 2531], [2152, 2589], [2152, 2825], [2152, 2945], [2152, 2966], [2153, 2154], [2153, 2185], [2153, 2188], [2153, 2283], [2153, 2421], [2153, 2574], [2153, 2579], [2153, 2665], [2154, 2155], [2154, 2189], [2154, 2328], [2154, 2373], [2154, 2433], [2154, 2574], [2154, 2578], [2154, 2699], [2155, 2185], [2155, 2204], [2155, 2210], [2155, 2341], [2155, 2445], [2155, 2492], [2155, 2493], [2155, 2528], [2155, 2575], [2155, 2596], [2155, 2612], [2155, 2650], [2155, 2686], [2155, 2779], [2156, 2157], [2156, 2582], [2156, 2585], [2156, 2590], [2156, 2593], [2156, 2775], [2157, 2158], [2157, 2405], [2157, 2588], [2157, 2597], [2157, 2628], [2157, 2680], [2157, 2724], [2158, 2235], [2158, 2304], [2158, 2405], [2158, 2425], [2158, 2589], [2158, 2598], [2158, 2681], [2158, 2702], [2158, 2725], [2158, 2730], [2158, 2731], [2158, 2848], [2158, 2879], [2158, 2978], [2158, 3005], [2158, 3007], [2158, 3051], [2158, 3058], [2158, 3147], [2158, 3163], [2159, 2160], [2159, 2449], [2159, 2596], [2159, 2600], [2159, 2603], [2159, 2606], [2159, 2609], [2159, 2707], [2160, 2161], [2160, 2225], [2160, 2304], [2160, 2480], [2160, 2488], [2160, 2533], [2160, 2602], [2160, 2610], [2160, 2634], [2160, 2692], [2160, 2693], [2160, 2708], [2160, 2754], [2160, 2833], [2161, 2162], [2161, 2306], [2161, 2495], [2161, 2602], [2161, 2606], [2161, 2611], [2161, 2617], [2161, 2681], [2161, 2958], [2161, 3114], [2162, 2194], [2162, 2225], [2162, 2235], [2162, 2248], [2162, 2308], [2162, 2497], [2162, 2498], [2162, 2524], [2162, 2535], [2162, 2587], [2162, 2688], [2162, 2756], [2162, 2848], [2162, 3054], [2163, 2164], [2163, 2221], [2163, 2407], [2163, 2461], [2163, 2612], [2163, 2615], [2163, 2620], [2163, 2662], [2163, 2726], [2164, 2165], [2164, 2221], [2164, 2344], [2164, 2547], [2164, 2591], [2164, 2614], [2164, 2616], [2164, 2622], [2164, 2669], [2164, 2693], [2164, 2706], [2164, 2717], [2164, 2727], [2165, 2166], [2165, 2272], [2165, 2295], [2165, 2299], [2165, 2438], [2165, 2592], [2165, 2616], [2165, 2618], [2165, 2662], [2165, 2808], [2165, 3026], [2165, 3128], [2165, 3178], [2165, 3179], [2166, 2252], [2166, 2372], [2166, 2572], [2166, 2573], [2166, 2602], [2166, 2617], [2166, 2693], [2166, 2802], [2166, 2873], [2166, 2875], [2166, 2926], [2166, 2935], [2166, 3002], [2166, 3026], [2166, 3047], [2166, 3055], [2167, 2168], [2167, 2249], [2167, 2250], [2167, 2517], [2167, 2623], [2167, 2628], [2167, 2631], [2167, 2634], [2168, 2169], [2168, 2233], [2168, 2419], [2168, 2816], [2168, 2866], [2169, 2170], [2169, 2497], [2169, 2536], [2169, 2606], [2169, 2631], [2169, 2707], [2169, 2712], [2169, 2720], [2169, 2721], [2169, 2783], [2170, 2537], [2170, 2539], [2170, 2632], [2170, 2633], [2170, 2711], [2170, 2712], [2170, 2816], [2170, 3010], [2170, 3155], [2171, 2172], [2171, 2173], [2171, 2192], [2171, 2499], [2171, 2637], [2171, 2640], [2171, 2778], [2172, 2173], [2172, 2327], [2172, 2504], [2172, 2641], [2172, 2671], [2172, 2734], [2172, 2773], [2172, 2922], [2173, 2174], [2173, 2327], [2173, 2350], [2173, 2459], [2173, 2488], [2173, 2508], [2173, 2610], [2173, 2735], [2173, 2757], [2173, 2759], [2173, 2811], [2173, 2847], [2173, 2988], [2173, 3065], [2174, 2177], [2174, 2200], [2174, 2210], [2174, 2225], [2174, 2237], [2174, 2240], [2174, 2242], [2174, 2266], [2174, 2275], [2174, 2289], [2174, 2299], [2174, 2305], [2174, 2317], [2174, 2323], [2174, 2345], [2174, 2350], [2174, 2352], [2174, 2355], [2174, 2359], [2174, 2363], [2174, 2366], [2174, 2367], [2174, 2388], [2174, 2397], [2174, 2417], [2174, 2422], [2174, 2447], [2174, 2458], [2174, 2464], [2174, 2465], [2174, 2472], [2174, 2509], [2174, 2517], [2174, 2522], [2174, 2527], [2174, 2533], [2174, 2558], [2174, 2567], [2174, 2600], [2174, 2642], [2174, 2657], [2174, 2662], [2174, 2665], [2174, 2678], [2174, 2683], [2174, 2697], [2174, 2710], [2174, 2712], [2174, 2735], [2174, 2737], [2174, 2741], [2174, 2751], [2174, 2762], [2174, 2765], [2174, 2768], [2174, 2788], [2174, 2799], [2174, 2822], [2174, 2826], [2174, 2832], [2174, 2839], [2174, 2879], [2174, 2883], [2174, 2884], [2174, 2896], [2174, 2929], [2174, 3093], [2174, 3167], [2175, 2176], [2175, 2178], [2175, 2195], [2175, 2511], [2175, 2642], [2175, 2646], [2175, 2649], [2176, 2177], [2176, 2363], [2176, 2383], [2176, 2530], [2176, 2695], [2176, 2836], [2176, 2954], [2177, 2210], [2177, 2270], [2177, 2334], [2177, 2374], [2177, 2375], [2177, 2397], [2177, 2407], [2177, 2484], [2177, 2646], [2177, 2695], [2177, 2834], [2177, 2837], [2177, 2863], [2177, 2865], [2177, 2876], [2178, 2179], [2178, 2183], [2178, 2273], [2178, 2530], [2178, 2627], [2178, 2642], [2178, 2685], [2178, 2716], [2178, 2717], [2178, 2723], [2179, 2180], [2179, 2298], [2179, 2432], [2179, 2484], [2179, 2513], [2179, 2646], [2179, 2647], [2180, 2220], [2180, 2291], [2180, 2298], [2180, 2412], [2180, 2611], [2180, 2685], [2180, 2720], [2180, 2746], [2180, 2765], [2180, 2842], [2180, 2843], [2180, 2856], [2180, 3022], [2181, 2182], [2181, 2185], [2181, 2548], [2181, 2676], [2181, 2679], [2181, 2683], [2181, 2686], [2181, 2691], [2181, 2694], [2182, 2183], [2182, 2251], [2182, 2461], [2182, 2583], [2182, 2609], [2182, 2610], [2182, 2637], [2182, 2646], [2182, 2648], [2182, 2677], [2183, 2184], [2183, 2195], [2183, 2307], [2183, 2583], [2183, 2639], [2183, 2678], [2183, 2684], [2183, 2837], [2184, 2269], [2184, 2273], [2184, 2467], [2184, 2521], [2184, 2584], [2184, 2648], [2184, 2678], [2184, 2722], [2184, 3037], [2184, 3045], [2184, 3057], [2184, 3073], [2184, 3126], [2185, 2186], [2185, 2575], [2185, 2608], [2185, 2694], [2186, 2187], [2186, 2197], [2186, 2236], [2186, 2283], [2186, 2285], [2186, 2363], [2186, 2628], [2186, 2634], [2186, 2640], [2186, 2666], [2186, 2679], [2186, 2680], [2186, 2794], [2186, 2858], [2187, 2284], [2187, 2302], [2187, 2363], [2187, 2365], [2187, 2530], [2187, 2560], [2187, 2605], [2187, 2608], [2187, 2625], [2187, 2658], [2187, 2668], [2187, 2681], [2187, 2685], [2187, 2702], [2187, 2749], [2187, 2790], [2187, 2811], [2187, 2859], [2187, 2871], [2187, 3012], [2187, 3013], [2187, 3184], [2188, 2189], [2188, 2697], [2188, 2701], [2188, 2704], [2188, 2750], [2189, 2190], [2189, 2221], [2189, 2243], [2189, 2373], [2189, 2464], [2189, 2526], [2189, 2527], [2189, 2591], [2189, 2698], [2189, 2812], [2189, 2823], [2190, 2191], [2190, 2222], [2190, 2263], [2190, 2297], [2190, 2432], [2190, 2433], [2190, 2698], [2190, 2704], [2190, 2815], [2190, 2824], [2190, 2968], [2191, 2194], [2191, 2206], [2191, 2218], [2191, 2222], [2191, 2228], [2191, 2464], [2191, 2557], [2191, 2619], [2191, 2622], [2191, 2706], [2191, 2785], [2191, 2850], [2191, 2851], [2191, 3171], [2192, 2193], [2192, 2463], [2192, 2671], [2192, 2732], [2192, 2735], [2192, 2796], [2193, 2194], [2193, 2225], [2193, 2233], [2193, 2464], [2193, 2634], [2193, 2640], [2193, 2732], [2193, 2736], [2193, 2755], [2193, 2776], [2193, 2793], [2193, 2880], [2194, 2641], [2194, 2671], [2194, 2736], [2194, 2756], [2194, 2860], [2195, 2196], [2195, 2223], [2195, 2575], [2195, 2650], [2195, 2678], [2195, 2738], [2196, 2197], [2196, 2224], [2196, 2301], [2196, 2307], [2196, 2651], [2196, 2738], [2196, 2836], [2197, 2302], [2197, 2312], [2197, 2325], [2197, 2339], [2197, 2340], [2197, 2448], [2197, 2575], [2197, 2577], [2197, 2581], [2197, 2610], [2197, 2629], [2197, 2764], [2197, 2870], [2197, 2880], [2197, 2964], [2197, 3058], [2198, 2199], [2198, 2226], [2198, 2672], [2198, 2741], [2198, 2744], [2198, 2747], [2199, 2200], [2199, 2224], [2199, 2253], [2199, 2379], [2199, 2438], [2199, 2653], [2199, 2669], [2199, 2735], [2199, 3084], [2200, 2239], [2200, 2254], [2200, 2651], [2200, 2664], [2200, 2765], [2200, 2769], [2200, 2770], [2200, 2888], [2201, 2202], [2201, 2205], [2201, 2207], [2201, 2209], [2201, 2233], [2201, 2370], [2201, 2379], [2201, 2402], [2201, 2763], [2201, 2766], [2201, 2769], [2201, 2772], [2201, 2775], [2201, 2778], [2201, 2929], [2202, 2203], [2202, 2205], [2202, 2210], [2202, 2341], [2202, 2383], [2202, 2742], [2202, 2826], [2202, 3049], [2203, 2204], [2203, 2206], [2203, 2336], [2203, 2341], [2203, 2354], [2203, 2538], [2203, 2743], [2203, 2772], [2203, 2864], [2203, 2868], [2203, 2874], [2203, 3044], [2203, 3045], [2203, 3049], [2203, 3188], [2204, 2208], [2204, 2210], [2204, 2256], [2204, 2277], [2204, 2298], [2204, 2320], [2204, 2408], [2204, 2456], [2204, 2460], [2204, 2484], [2204, 2494], [2204, 2576], [2204, 2608], [2204, 2687], [2204, 2690], [2204, 2699], [2204, 2743], [2204, 2782], [2204, 2824], [2204, 2924], [2204, 2991], [2204, 3072], [2205, 2206], [2205, 2464], [2205, 2600], [2205, 2604], [2205, 2608], [2205, 2625], [2205, 2866], [2205, 2987], [2206, 2381], [2206, 2539], [2206, 2772], [2206, 2987], [2206, 3189], [2207, 2208], [2207, 2499], [2207, 2608], [2207, 2664], [2207, 2678], [2207, 2722], [2207, 2723], [2207, 2767], [2207, 2778], [2207, 2822], [2207, 2871], [2207, 3038], [2208, 2234], [2208, 2255], [2208, 2298], [2208, 2407], [2208, 2458], [2208, 2500], [2208, 2620], [2208, 2638], [2208, 2643], [2208, 2650], [2208, 2678], [2208, 2740], [2208, 2937], [2209, 2210], [2209, 2298], [2209, 2540], [2209, 2609], [2209, 2646], [2209, 2765], [2209, 3025], [2210, 2276], [2210, 2455], [2210, 2603], [2210, 2742], [2210, 2823], [2210, 2991], [2211, 2212], [2211, 2707], [2211, 2710], [2211, 2780], [2211, 2783], [2211, 2813], [2212, 2213], [2212, 2236], [2212, 2297], [2212, 2548], [2212, 2613], [2212, 2651], [2212, 2663], [2212, 2704], [2212, 2723], [2212, 2727], [2212, 2786], [2213, 2214], [2213, 2424], [2213, 2532], [2213, 2558], [2213, 2605], [2213, 2684], [2213, 2710], [2213, 2714], [2213, 2752], [2213, 2808], [2213, 3032], [2213, 3039], [2213, 3161], [2214, 2358], [2214, 2391], [2214, 2498], [2214, 2697], [2214, 2704], [2214, 2814], [2214, 2950], [2214, 3006], [2214, 3039], [2215, 2216], [2215, 2219], [2215, 2221], [2215, 2329], [2215, 2450], [2215, 2593], [2215, 2738], [2215, 2803], [2215, 2806], [2215, 2809], [2215, 2812], [2215, 2817], [2215, 2820], [2215, 2883], [2216, 2217], [2216, 2275], [2216, 2464], [2216, 2577], [2216, 2624], [2216, 2749], [2216, 2790], [2216, 2812], [2216, 2850], [2216, 2857], [2216, 3159], [2217, 2218], [2217, 2248], [2217, 2265], [2217, 2266], [2217, 2325], [2217, 2331], [2217, 2333], [2217, 2352], [2217, 2733], [2217, 2756], [2217, 2764], [2217, 2806], [2217, 2807], [2217, 2857], [2217, 3158], [2218, 2244], [2218, 2259], [2218, 2293], [2218, 2326], [2218, 2366], [2218, 2372], [2218, 2420], [2218, 2431], [2218, 2543], [2218, 2551], [2218, 2556], [2218, 2561], [2218, 2570], [2218, 2592], [2218, 2756], [2218, 2790], [2218, 2808], [2218, 2816], [2218, 3094], [2218, 3095], [2218, 3144], [2219, 2220], [2219, 2222], [2219, 2306], [2219, 2330], [2219, 2445], [2219, 2451], [2219, 2593], [2219, 2684], [2219, 2685], [2219, 2815], [2219, 2819], [2219, 2845], [2219, 2850], [2219, 2859], [2219, 2883], [2219, 2911], [2219, 2952], [2219, 2996], [2219, 3015], [2219, 3114], [2219, 3119], [2219, 3161], [2220, 2315], [2220, 2432], [2220, 2452], [2220, 2806], [2220, 2845], [2220, 3036], [2220, 3060], [2221, 2222], [2221, 2402], [2221, 2464], [2221, 2594], [2221, 2613], [2221, 2618], [2221, 2621], [2221, 2718], [2221, 2761], [2221, 2762], [2221, 2817], [2222, 2361], [2222, 2543], [2222, 2544], [2222, 2594], [2222, 2615], [2222, 2616], [2222, 2618], [2222, 2667], [2222, 2720], [2222, 2752], [2222, 2819], [2222, 2932], [2222, 3019], [2223, 2224], [2223, 2316], [2223, 2639], [2223, 2768], [2223, 2823], [2224, 2225], [2224, 2230], [2224, 2242], [2224, 2300], [2224, 2308], [2224, 2325], [2224, 2550], [2224, 2780], [2224, 2804], [2225, 2230], [2225, 2234], [2225, 2267], [2225, 2323], [2225, 2448], [2225, 2524], [2225, 2638], [2225, 2683], [2225, 2755], [2225, 2781], [2225, 2791], [2225, 2942], [2226, 2227], [2226, 2634], [2226, 2826], [2226, 2830], [2226, 2833], [2226, 2836], [2227, 2228], [2227, 2337], [2227, 2341], [2227, 2681], [2227, 2744], [2227, 2748], [2227, 2789], [2227, 2826], [2227, 3102], [2228, 2432], [2228, 2506], [2228, 2682], [2228, 2777], [2228, 2828], [2228, 2830], [2228, 2838], [2228, 3102], [2228, 3144], [2229, 2230], [2229, 2231], [2229, 2342], [2229, 2344], [2229, 2669], [2229, 2750], [2229, 2839], [2229, 2844], [2229, 2846], [2230, 2260], [2230, 2322], [2230, 2334], [2230, 2345], [2230, 2383], [2230, 2586], [2230, 2587], [2230, 2670], [2230, 2757], [2230, 2840], [2230, 2844], [2230, 2847], [2231, 2232], [2231, 2303], [2231, 2428], [2231, 2490], [2231, 2587], [2231, 2679], [2231, 2686], [2231, 2805], [2231, 2839], [2231, 2842], [2232, 2305], [2232, 2342], [2232, 2443], [2232, 2502], [2232, 2553], [2232, 2680], [2232, 2682], [2232, 2942], [2233, 2234], [2233, 2236], [2233, 2347], [2233, 2418], [2233, 2536], [2233, 2620], [2233, 2640], [2233, 2737], [2233, 2739], [2233, 2837], [2233, 2849], [2233, 2852], [2233, 2854], [2233, 2857], [2233, 2860], [2233, 2863], [2233, 2866], [2233, 2869], [2234, 2235], [2234, 2255], [2234, 2277], [2234, 2388], [2234, 2405], [2234, 2536], [2234, 2769], [2234, 2981], [2235, 2645], [2235, 2703], [2235, 2725], [2235, 2860], [2235, 2981], [2235, 3057], [2236, 2296], [2236, 2511], [2236, 2605], [2236, 2634], [2236, 2717], [2236, 2727], [2236, 2745], [2236, 2764], [2236, 2857], [2237, 2238], [2237, 2348], [2237, 2454], [2237, 2623], [2237, 2653], [2237, 2676], [2237, 2884], [2238, 2239], [2238, 2264], [2238, 2322], [2238, 2343], [2238, 2370], [2238, 2379], [2238, 2396], [2238, 2397], [2238, 3068], [2239, 2240], [2239, 2254], [2239, 2764], [2239, 2854], [2239, 2855], [2240, 2241], [2240, 2293], [2240, 2335], [2240, 2336], [2240, 2337], [2240, 2509], [2240, 2764], [2240, 2826], [2240, 2903], [2240, 2955], [2240, 3101], [2240, 3141], [2240, 3159], [2241, 2337], [2241, 2482], [2241, 2510], [2241, 2855], [2241, 2856], [2241, 2867], [2241, 2903], [2241, 3140], [2242, 2243], [2242, 2246], [2242, 2295], [2242, 2345], [2242, 2550], [2242, 2614], [2242, 2710], [2242, 2823], [2242, 2889], [2242, 3086], [2242, 3124], [2243, 2244], [2243, 2246], [2243, 2249], [2243, 2303], [2243, 2357], [2243, 2517], [2243, 2542], [2243, 2578], [2243, 2591], [2243, 2804], [2243, 2805], [2243, 2819], [2243, 3032], [2244, 2250], [2244, 2252], [2244, 2265], [2244, 2270], [2244, 2344], [2244, 2345], [2244, 2372], [2244, 2375], [2244, 2517], [2244, 2568], [2244, 2706], [2244, 2758], [2244, 2844], [2244, 3160], [2245, 2246], [2245, 2249], [2245, 2321], [2245, 2447], [2245, 2463], [2245, 2464], [2245, 2732], [2245, 2734], [2245, 2785], [2245, 2813], [2245, 2922], [2246, 2247], [2246, 2325], [2246, 2326], [2246, 2472], [2246, 2566], [2246, 2619], [2246, 2785], [2246, 2814], [2246, 2921], [2246, 2940], [2246, 2964], [2246, 3040], [2246, 3141], [2247, 2248], [2247, 2303], [2247, 2317], [2247, 2321], [2247, 2322], [2247, 2346], [2247, 2374], [2247, 2397], [2247, 2466], [2247, 2489], [2247, 2491], [2247, 2587], [2247, 2828], [2247, 3111], [2247, 3138], [2248, 2269], [2248, 2323], [2248, 2330], [2248, 2333], [2248, 2380], [2248, 2468], [2248, 2471], [2248, 2477], [2248, 2489], [2248, 2519], [2248, 2655], [2248, 2749], [2248, 2777], [2248, 2782], [2248, 2795], [2248, 2802], [2248, 2814], [2248, 2845], [2248, 2893], [2248, 2958], [2248, 3107], [2249, 2250], [2249, 2541], [2249, 2818], [2250, 2251], [2250, 2369], [2250, 2371], [2251, 2252], [2251, 2385], [2251, 2417], [2251, 2481], [2251, 2512], [2251, 2546], [2251, 2588], [2251, 2631], [2251, 2637], [2251, 2673], [2251, 2691], [2251, 2693], [2251, 2818], [2252, 2386], [2252, 2408], [2252, 2413], [2252, 2417], [2252, 2451], [2252, 2474], [2252, 2477], [2252, 2482], [2252, 2487], [2252, 2514], [2252, 2547], [2252, 2589], [2252, 2632], [2252, 2648], [2252, 2706], [2252, 2759], [2252, 2819], [2252, 2848], [2252, 2972], [2252, 2973], [2252, 3019], [2253, 2254], [2253, 2257], [2253, 2260], [2253, 2264], [2253, 2265], [2253, 2267], [2253, 2270], [2253, 2373], [2253, 2574], [2253, 2581], [2253, 2582], [2253, 2755], [2253, 2803], [2253, 2896], [2254, 2255], [2254, 2260], [2254, 2279], [2254, 2387], [2254, 2440], [2254, 2441], [2254, 2509], [2254, 2837], [2254, 3087], [2255, 2256], [2255, 2267], [2255, 2305], [2255, 2837], [2255, 3145], [2255, 3147], [2256, 2277], [2256, 2336], [2256, 2368], [2256, 2466], [2256, 2489], [2256, 2509], [2256, 2522], [2256, 2528], [2256, 2580], [2256, 2759], [2256, 3163], [2256, 3191], [2256, 3195], [2257, 2258], [2257, 2270], [2257, 2366], [2257, 2414], [2257, 2465], [2257, 2591], [2257, 2592], [2257, 3166], [2258, 2259], [2258, 2279], [2258, 2282], [2258, 2366], [2258, 2367], [2258, 2410], [2258, 2577], [2258, 3165], [2259, 2280], [2259, 2299], [2259, 2368], [2259, 2410], [2259, 2411], [2259, 2479], [2259, 2509], [2259, 2542], [2259, 2592], [2259, 2851], [2259, 2969], [2259, 3142], [2259, 3165], [2259, 3194], [2260, 2261], [2260, 2334], [2260, 2447], [2260, 2499], [2260, 2500], [2260, 2508], [2260, 2565], [2260, 2583], [2260, 2584], [2260, 2629], [2260, 2677], [2260, 2678], [2261, 2262], [2261, 2335], [2261, 2350], [2261, 2559], [2261, 2566], [2261, 2584], [2261, 2840], [2261, 2896], [2261, 3037], [2261, 3087], [2261, 3130], [2261, 3141], [2262, 2263], [2262, 2350], [2262, 2355], [2262, 2500], [2262, 2505], [2262, 2589], [2262, 2759], [2262, 2824], [2262, 2825], [2262, 2829], [2262, 2925], [2262, 2937], [2262, 2998], [2262, 3152], [2263, 2452], [2263, 2510], [2263, 2527], [2263, 2544], [2263, 2630], [2263, 2702], [2263, 2877], [2263, 2925], [2263, 2968], [2264, 2265], [2264, 2371], [2264, 2410], [2264, 2511], [2264, 2512], [2264, 2565], [2264, 2623], [2264, 2626], [2264, 2754], [2264, 2764], [2264, 3194], [2265, 2266], [2265, 2279], [2265, 2322], [2265, 2401], [2265, 2447], [2265, 2477], [2265, 2491], [2265, 2844], [2265, 3089], [2266, 2289], [2266, 2362], [2266, 2372], [2266, 2402], [2266, 2624], [2266, 2636], [2266, 2735], [2266, 2796], [2266, 2802], [2266, 2890], [2266, 3137], [2267, 2268], [2267, 2269], [2267, 2465], [2267, 2477], [2267, 2525], [2267, 2638], [2267, 2648], [2267, 2948], [2267, 3054], [2268, 2269], [2268, 2436], [2268, 2466], [2268, 2518], [2268, 2584], [2268, 2639], [2269, 2380], [2269, 2467], [2269, 2536], [2269, 3050], [2269, 3111], [2270, 2271], [2270, 2548], [2270, 2562], [2270, 2648], [2270, 2751], [2270, 2758], [2270, 2947], [2271, 2272], [2271, 2358], [2271, 2484], [2271, 2554], [2271, 2581], [2271, 2592], [2271, 2947], [2271, 3126], [2272, 2273], [2272, 2365], [2272, 2438], [2272, 2658], [2272, 2659], [2272, 3127], [2273, 2287], [2273, 2380], [2273, 2495], [2273, 2642], [2273, 2714], [2273, 2975], [2273, 2992], [2273, 3014], [2274, 2275], [2274, 2278], [2274, 2282], [2274, 2285], [2274, 2378], [2274, 2480], [2274, 2787], [2274, 2794], [2275, 2276], [2275, 2333], [2275, 2334], [2275, 2367], [2275, 2480], [2275, 2560], [2275, 2599], [2275, 2749], [2275, 2788], [2275, 3134], [2275, 3157], [2276, 2277], [2276, 2334], [2276, 2348], [2276, 2596], [2276, 2781], [2277, 2318], [2277, 2352], [2277, 2390], [2277, 2485], [2277, 2497], [2277, 2523], [2277, 2596], [2277, 2598], [2277, 2599], [2277, 2600], [2277, 2866], [2277, 2980], [2277, 3081], [2278, 2279], [2278, 2328], [2278, 2333], [2278, 2373], [2278, 2425], [2278, 2733], [2278, 2795], [2278, 2797], [2278, 2812], [2279, 2280], [2279, 2282], [2279, 2289], [2279, 2416], [2279, 2425], [2279, 2447], [2279, 2578], [2279, 2900], [2279, 3147], [2280, 2281], [2280, 2294], [2280, 2413], [2280, 2426], [2280, 2434], [2280, 2441], [2280, 2444], [2280, 2479], [2280, 2491], [2280, 2514], [2280, 2900], [2281, 2289], [2281, 2296], [2281, 2362], [2281, 2514], [2281, 2605], [2281, 2645], [2281, 2899], [2281, 2904], [2281, 3142], [2282, 2283], [2282, 2285], [2282, 2348], [2282, 2414], [2282, 2479], [2282, 3157], [2283, 2284], [2283, 2577], [2283, 2578], [2283, 2780], [2283, 2794], [2283, 2836], [2284, 2299], [2284, 2468], [2284, 2665], [2284, 2709], [2284, 2749], [2284, 3020], [2284, 3041], [2284, 3133], [2284, 3157], [2284, 3164], [2285, 2286], [2285, 2425], [2285, 2548], [2285, 2560], [2285, 2565], [2285, 2754], [2286, 2287], [2286, 2426], [2286, 2479], [2286, 2480], [2286, 2554], [2286, 2560], [2287, 2413], [2287, 2436], [2287, 2529], [2287, 2554], [2287, 2559], [2287, 2565], [2287, 2584], [2287, 2626], [2287, 2702], [2287, 2975], [2288, 2289], [2288, 2292], [2288, 2296], [2288, 2387], [2288, 2515], [2288, 2796], [2289, 2290], [2289, 2296], [2289, 2388], [2289, 2416], [2289, 2899], [2289, 3103], [2289, 3104], [2289, 3123], [2289, 3190], [2290, 2291], [2290, 2515], [2290, 2522], [2290, 2573], [2290, 2713], [2290, 2725], [2290, 2746], [2290, 2802], [2290, 2919], [2290, 2920], [2290, 3085], [2290, 3115], [2290, 3147], [2290, 3173], [2290, 3190], [2291, 2542], [2291, 2567], [2291, 2647], [2291, 2713], [2291, 2753], [2291, 2765], [2291, 2888], [2291, 2908], [2291, 2926], [2291, 2985], [2291, 3023], [2291, 3025], [2291, 3026], [2291, 3030], [2291, 3032], [2291, 3035], [2291, 3060], [2291, 3106], [2291, 3142], [2291, 3179], [2292, 2293], [2292, 2366], [2292, 2383], [2292, 2422], [2292, 2454], [2292, 2823], [2292, 2863], [2292, 3103], [2293, 2294], [2293, 2296], [2293, 2354], [2293, 2375], [2293, 2403], [2293, 2420], [2293, 2458], [2293, 2470], [2293, 2503], [2293, 2824], [2293, 2827], [2293, 2853], [2293, 2904], [2293, 2956], [2293, 3008], [2293, 3103], [2293, 3144], [2294, 2295], [2294, 2403], [2294, 2444], [2294, 2855], [2294, 3078], [2295, 2308], [2295, 2346], [2295, 2614], [2295, 2711], [2295, 2824], [2295, 3124], [2295, 3127], [2295, 3128], [2296, 2297], [2296, 2310], [2296, 2419], [2296, 2512], [2296, 2636], [2296, 2644], [2296, 2746], [2296, 2764], [2297, 2298], [2297, 2562], [2297, 2622], [2297, 2824], [2297, 2909], [2298, 2562], [2298, 2630], [2298, 2723], [2298, 2772], [2298, 3025], [2299, 2300], [2299, 2301], [2299, 2306], [2299, 2313], [2299, 2356], [2299, 2365], [2299, 2397], [2299, 2444], [2299, 2542], [2299, 2602], [2299, 2808], [2299, 2883], [2299, 3077], [2299, 3106], [2299, 3139], [2300, 2301], [2300, 2308], [2300, 2314], [2300, 2524], [2300, 2768], [2300, 2840], [2300, 2845], [2300, 2952], [2300, 3084], [2300, 3111], [2300, 3116], [2300, 3124], [2300, 3131], [2300, 3133], [2301, 2302], [2301, 2307], [2301, 2678], [2301, 2871], [2301, 3031], [2301, 3056], [2302, 2313], [2302, 2314], [2302, 2336], [2302, 2339], [2302, 2576], [2302, 2753], [2302, 2963], [2302, 2976], [2302, 2993], [2302, 3044], [2302, 3118], [2302, 3142], [2302, 3164], [2302, 3182], [2303, 2304], [2303, 2309], [2303, 2311], [2303, 2356], [2303, 2373], [2303, 2437], [2303, 2491], [2303, 2578], [2303, 2806], [2303, 2842], [2304, 2305], [2304, 2489], [2304, 2514], [2304, 2516], [2304, 2517], [2304, 2636], [2304, 2797], [2304, 2801], [2304, 2843], [2304, 2848], [2304, 3137], [2304, 3148], [2305, 2332], [2305, 2342], [2305, 2422], [2305, 2489], [2305, 2682], [2305, 2771], [2305, 2830], [2305, 2833], [2305, 2839], [2305, 2847], [2305, 3143], [2305, 3173], [2306, 2307], [2306, 2308], [2306, 2411], [2306, 2412], [2306, 2616], [2306, 3022], [2307, 2308], [2307, 2339], [2307, 2684], [2308, 2353], [2308, 2437], [2308, 2438], [2308, 2587], [2308, 2639], [2308, 3127], [2309, 2310], [2309, 2444], [2309, 2453], [2309, 2651], [2309, 2862], [2310, 2311], [2310, 2361], [2310, 2744], [2310, 2763], [2310, 2827], [2311, 2362], [2311, 2438], [2311, 2763], [2311, 2828], [2311, 3032], [2311, 3190], [2312, 2313], [2312, 2325], [2312, 2479], [2312, 2481], [2312, 2487], [2312, 2491], [2312, 2625], [2312, 2705], [2312, 2706], [2312, 2850], [2312, 2862], [2313, 2314], [2313, 2457], [2313, 2472], [2313, 2482], [2313, 2986], [2313, 3139], [2313, 3140], [2313, 3153], [2313, 3177], [2314, 2315], [2314, 2325], [2314, 2845], [2314, 2859], [2314, 2906], [2314, 3033], [2314, 3099], [2314, 3158], [2314, 3197], [2315, 2317], [2315, 2324], [2315, 2457], [2315, 2521], [2315, 2859], [2315, 2901], [2315, 2935], [2315, 2953], [2315, 3108], [2316, 2317], [2316, 2321], [2316, 2324], [2316, 2328], [2316, 2393], [2316, 2401], [2316, 2766], [2317, 2318], [2317, 2319], [2317, 2324], [2317, 2359], [2317, 2374], [2317, 2768], [2317, 2806], [2317, 2809], [2317, 2894], [2317, 2901], [2317, 3062], [2318, 2319], [2318, 2451], [2318, 2587], [2318, 2683], [2318, 2689], [2318, 2759], [2318, 3196], [2319, 2320], [2319, 2346], [2319, 2390], [2319, 2393], [2319, 2394], [2319, 2543], [2319, 2558], [2319, 2566], [2319, 2690], [2319, 2931], [2319, 2932], [2319, 3016], [2319, 3079], [2319, 3098], [2319, 3130], [2319, 3186], [2320, 2445], [2320, 2474], [2320, 2675], [2320, 2714], [2320, 2740], [2320, 2867], [2320, 2868], [2320, 2892], [2320, 2911], [2320, 2941], [2320, 2967], [2320, 2982], [2320, 3090], [2320, 3118], [2320, 3195], [2321, 2322], [2321, 2396], [2321, 2429], [2321, 2462], [2321, 2469], [2321, 2586], [2321, 2795], [2321, 2830], [2322, 2323], [2322, 2345], [2323, 2329], [2323, 2379], [2323, 2654], [2323, 2700], [2323, 2766], [2323, 2795], [2323, 2803], [2323, 2893], [2324, 2325], [2324, 2394], [2324, 2476], [2324, 2639], [2324, 2777], [2324, 2858], [2324, 2862], [2324, 2940], [2325, 2326], [2325, 2352], [2325, 2732], [2325, 2735], [2325, 2736], [2325, 2804], [2325, 2858], [2325, 3196], [2326, 2327], [2326, 2354], [2326, 2394], [2326, 2498], [2326, 2550], [2326, 2556], [2326, 2587], [2326, 2734], [2326, 2757], [2326, 2805], [2326, 3003], [2326, 3044], [2326, 3098], [2326, 3099], [2326, 3184], [2327, 2351], [2327, 2382], [2327, 2611], [2327, 2921], [2327, 2988], [2327, 2989], [2327, 3000], [2327, 3003], [2328, 2329], [2328, 2478], [2328, 2596], [2328, 2812], [2329, 2330], [2329, 2421], [2329, 2791], [2329, 2822], [2329, 3112], [2330, 2502], [2330, 2535], [2330, 2536], [2330, 2745], [2330, 2749], [2330, 2857], [2330, 3112], [2331, 2332], [2331, 2373], [2331, 2471], [2331, 2830], [2332, 2333], [2332, 2334], [2332, 2471], [2332, 2501], [2332, 2509], [2333, 2355], [2333, 2561], [2333, 2900], [2333, 2969], [2333, 3122], [2333, 3137], [2333, 3196], [2334, 2335], [2334, 2340], [2334, 2405], [2334, 2501], [2334, 2597], [2334, 2598], [2334, 2833], [2334, 2863], [2335, 2336], [2335, 2598], [2335, 2840], [2335, 2864], [2335, 2865], [2335, 2879], [2335, 3116], [2335, 3122], [2335, 3134], [2335, 3150], [2336, 2338], [2336, 2340], [2336, 2380], [2336, 2381], [2336, 2503], [2336, 2522], [2336, 2977], [2336, 2995], [2336, 3052], [2336, 3182], [2337, 2338], [2337, 2439], [2337, 2481], [2337, 2598], [2337, 2850], [2337, 2854], [2337, 2866], [2338, 2339], [2338, 2340], [2338, 2497], [2338, 2598], [2339, 2435], [2339, 2498], [2339, 2611], [2339, 2851], [2339, 2856], [2339, 2975], [2340, 2341], [2340, 2471], [2340, 2502], [2340, 2528], [2340, 2530], [2340, 2670], [2340, 2716], [2340, 2749], [2340, 2781], [2341, 2585], [2341, 2607], [2341, 2748], [2341, 2775], [2341, 2779], [2341, 2863], [2341, 2880], [2342, 2343], [2342, 2421], [2342, 2469], [2342, 2552], [2343, 2344], [2343, 2375], [2343, 2385], [2343, 2414], [2343, 2443], [2343, 2454], [2343, 2462], [2343, 2512], [2343, 2515], [2343, 2552], [2343, 3173], [2344, 2345], [2344, 2414], [2344, 2547], [2344, 2551], [2344, 2761], [2344, 2765], [2344, 2798], [2344, 2842], [2344, 3179], [2345, 2346], [2345, 2369], [2345, 2386], [2345, 2429], [2345, 2431], [2345, 2799], [2345, 2912], [2345, 3181], [2346, 2377], [2346, 2429], [2346, 2587], [2346, 2690], [2346, 2842], [2346, 2912], [2346, 3080], [2346, 3126], [2346, 3175], [2346, 3180], [2347, 2348], [2347, 2352], [2347, 2518], [2347, 2555], [2347, 2596], [2347, 2612], [2347, 2732], [2347, 2733], [2348, 2349], [2348, 2352], [2348, 2780], [2348, 2812], [2348, 2823], [2348, 3156], [2349, 2350], [2349, 2366], [2349, 2555], [2349, 2556], [2349, 2757], [2349, 2786], [2350, 2351], [2350, 2388], [2350, 2499], [2350, 2504], [2350, 2702], [2350, 2786], [2350, 2949], [2350, 2951], [2350, 2955], [2350, 3004], [2350, 3038], [2351, 2504], [2351, 2519], [2351, 2556], [2351, 2557], [2351, 2572], [2351, 2645], [2351, 2949], [2351, 3001], [2352, 2353], [2352, 2447], [2352, 2466], [2352, 2479], [2352, 2519], [2352, 2556], [2352, 2737], [2352, 2815], [2352, 2824], [2352, 2870], [2352, 2906], [2352, 2909], [2352, 2952], [2352, 3081], [2352, 3156], [2353, 2354], [2353, 2497], [2353, 2519], [2353, 2557], [2353, 2641], [2353, 2657], [2353, 2780], [2353, 2783], [2353, 2784], [2353, 2922], [2353, 3133], [2353, 3154], [2354, 2383], [2354, 2459], [2354, 2460], [2354, 2497], [2354, 2530], [2354, 2585], [2354, 2587], [2354, 2835], [2354, 2859], [2354, 2874], [2354, 2905], [2354, 2953], [2354, 2956], [2354, 3051], [2354, 3081], [2354, 3093], [2354, 3197], [2355, 2356], [2355, 2373], [2355, 2432], [2355, 2458], [2355, 2514], [2355, 2563], [2355, 2647], [2355, 2655], [2355, 2687], [2355, 2698], [2355, 2699], [2355, 2759], [2355, 2828], [2355, 2896], [2355, 2902], [2355, 3060], [2355, 3085], [2356, 2357], [2356, 2428], [2356, 2444], [2356, 2468], [2356, 3060], [2356, 3120], [2356, 3138], [2356, 3148], [2356, 3177], [2357, 2358], [2357, 2468], [2357, 2592], [2357, 2698], [2357, 2845], [2357, 2944], [2357, 2985], [2357, 3124], [2357, 3160], [2357, 3161], [2358, 2428], [2358, 2751], [2358, 2805], [2358, 2845], [2358, 2881], [2358, 3099], [2358, 3119], [2358, 3131], [2359, 2360], [2359, 2375], [2359, 2401], [2359, 2403], [2359, 2451], [2359, 2455], [2359, 2457], [2359, 2533], [2359, 2862], [2359, 2916], [2359, 2939], [2359, 3020], [2359, 3089], [2359, 3113], [2360, 2361], [2360, 2402], [2360, 2403], [2360, 2473], [2360, 2618], [2360, 2713], [2360, 2792], [2360, 2890], [2360, 2894], [2360, 2929], [2360, 3008], [2360, 3104], [2361, 2362], [2361, 2402], [2361, 2625], [2361, 2635], [2361, 2720], [2361, 2772], [2361, 2774], [2361, 2777], [2361, 2827], [2361, 2861], [2361, 2862], [2361, 2958], [2361, 3149], [2362, 2382], [2362, 2491], [2362, 2625], [2362, 2636], [2362, 2828], [2362, 2890], [2362, 3029], [2362, 3036], [2362, 3148], [2362, 3189], [2363, 2364], [2363, 2624], [2363, 2701], [2363, 2809], [2363, 2810], [2363, 2822], [2363, 2833], [2363, 3012], [2364, 2365], [2364, 2676], [2364, 2679], [2364, 2821], [2364, 2833], [2364, 2839], [2365, 2427], [2365, 2428], [2365, 2679], [2365, 2681], [2365, 2683], [2365, 3005], [2366, 2464], [2366, 2755], [2366, 2757], [2366, 2761], [2366, 3094], [2367, 2368], [2367, 2378], [2367, 2436], [2367, 2465], [2367, 2559], [2367, 2577], [2367, 3058], [2367, 3115], [2367, 3116], [2367, 3147], [2367, 3162], [2368, 2400], [2368, 2436], [2368, 2599], [2368, 2851], [2368, 3011], [2368, 3162], [2368, 3195], [2369, 2370], [2369, 2385], [2369, 2429], [2369, 2449], [2369, 2852], [2370, 2371], [2370, 2770], [2370, 2831], [2370, 2834], [2370, 2852], [2370, 3181], [2371, 2372], [2371, 2548], [2371, 2693], [2371, 2754], [2371, 2796], [2371, 2798], [2372, 2558], [2372, 2569], [2372, 2926], [2372, 3008], [2372, 3032], [2372, 3181], [2372, 3194], [2373, 2374], [2373, 2432], [2373, 2453], [2373, 2475], [2373, 2500], [2373, 2512], [2373, 2525], [2373, 2562], [2373, 2637], [2373, 2646], [2373, 2661], [2373, 2686], [2373, 2806], [2373, 2830], [2374, 2375], [2374, 2383], [2374, 2475], [2374, 2689], [2374, 2690], [2374, 2758], [2374, 2782], [2374, 2902], [2374, 2953], [2375, 2376], [2375, 2455], [2375, 2469], [2375, 2483], [2375, 2512], [2375, 2516], [2375, 2578], [2375, 2917], [2375, 3008], [2376, 2377], [2376, 2429], [2376, 2449], [2376, 2451], [2376, 2469], [2376, 2486], [2376, 2529], [2376, 2552], [2376, 2606], [2376, 2689], [2376, 2799], [2376, 2853], [2377, 2489], [2377, 2539], [2377, 2771], [2377, 2799], [2377, 2843], [2377, 2872], [2377, 2917], [2377, 3183], [2378, 2379], [2378, 2436], [2378, 2461], [2378, 2565], [2379, 2380], [2379, 2515], [2379, 2642], [2379, 2716], [2379, 3115], [2380, 2381], [2380, 2397], [2380, 2436], [2380, 2438], [2380, 2601], [2380, 2712], [2380, 2716], [2380, 2745], [2380, 2746], [2380, 2764], [2380, 2772], [2380, 2958], [2380, 3011], [2380, 3028], [2380, 3115], [2380, 3122], [2381, 2382], [2381, 2482], [2381, 2632], [2381, 2851], [2381, 3187], [2382, 2438], [2382, 2671], [2382, 2735], [2382, 2774], [2382, 2841], [2382, 2936], [2382, 2974], [2382, 3033], [2383, 2384], [2383, 2585], [2383, 2757], [2383, 2780], [2383, 2781], [2383, 2809], [2383, 3093], [2384, 2385], [2384, 2454], [2384, 2459], [2384, 2672], [2384, 2673], [2384, 2742], [2384, 2767], [2384, 3009], [2385, 2386], [2385, 2462], [2385, 2546], [2385, 2834], [2385, 2835], [2386, 2522], [2386, 2547], [2386, 2829], [2386, 2847], [2386, 2951], [2386, 3009], [2386, 3047], [2386, 3173], [2386, 3191], [2387, 2388], [2387, 2392], [2387, 2478], [2387, 2499], [2387, 2644], [2387, 2663], [2388, 2389], [2388, 2390], [2388, 2415], [2388, 2418], [2388, 2424], [2388, 2644], [2388, 2645], [2388, 3087], [2388, 3088], [2388, 3168], [2389, 2390], [2389, 2419], [2389, 2517], [2389, 2543], [2389, 2605], [2389, 2702], [2389, 2720], [2389, 3023], [2390, 2391], [2390, 2420], [2390, 2472], [2390, 2478], [2390, 2561], [2390, 2792], [2390, 2918], [2390, 2959], [2390, 2965], [2390, 3017], [2390, 3104], [2390, 3153], [2390, 3175], [2391, 2420], [2391, 2424], [2391, 2497], [2391, 2503], [2391, 2579], [2391, 2665], [2391, 2720], [2391, 2932], [2391, 2966], [2391, 2970], [2391, 3185], [2391, 3189], [2392, 2393], [2392, 2406], [2392, 2739], [2392, 2760], [2392, 2852], [2393, 2394], [2393, 2429], [2393, 2445], [2393, 2447], [2393, 2459], [2393, 2478], [2393, 2548], [2393, 2568], [2393, 2613], [2393, 2757], [2393, 2758], [2393, 2760], [2393, 2832], [2394, 2395], [2394, 2413], [2394, 2554], [2394, 2675], [2394, 2940], [2394, 3019], [2395, 2404], [2395, 2656], [2395, 2729], [2395, 2777], [2395, 2832], [2395, 2933], [2395, 2938], [2395, 3016], [2395, 3019], [2395, 3044], [2396, 2397], [2396, 2481], [2396, 2545], [2396, 2546], [2396, 2716], [2396, 2854], [2397, 2398], [2397, 2443], [2397, 2482], [2397, 2547], [2397, 2601], [2397, 2626], [2397, 2789], [2397, 2831], [2397, 2855], [2397, 2913], [2397, 2914], [2397, 2964], [2397, 3068], [2397, 3111], [2398, 2399], [2398, 2425], [2398, 2441], [2398, 2527], [2398, 2542], [2398, 2545], [2398, 2547], [2398, 2562], [2398, 2564], [2398, 2565], [2398, 2588], [2398, 2591], [2398, 2647], [2398, 3191], [2399, 2400], [2399, 2436], [2399, 2461], [2399, 2465], [2399, 2526], [2399, 2591], [2399, 2615], [2399, 2819], [2399, 2838], [2399, 3028], [2400, 2465], [2400, 2564], [2400, 2592], [2400, 2656], [2400, 2943], [2400, 2944], [2400, 2967], [2400, 3054], [2401, 2402], [2401, 2511], [2401, 2568], [2401, 2691], [2401, 2857], [2401, 2862], [2402, 2569], [2402, 2624], [2402, 2635], [2402, 2692], [2402, 2718], [2402, 2747], [2402, 2766], [2402, 2822], [2402, 2857], [2403, 2404], [2403, 2468], [2403, 2605], [2403, 2737], [2403, 2857], [2403, 2859], [2403, 2861], [2403, 2938], [2403, 3096], [2403, 3158], [2403, 3159], [2404, 2451], [2404, 2599], [2404, 2703], [2404, 2728], [2404, 2788], [2404, 2853], [2404, 2887], [2404, 3078], [2404, 3096], [2405, 2406], [2405, 2407], [2405, 2415], [2405, 2485], [2405, 2701], [2405, 2769], [2405, 2797], [2405, 2833], [2405, 2879], [2406, 2407], [2406, 2454], [2406, 2552], [2406, 2676], [2406, 2726], [2406, 2730], [2406, 2739], [2406, 2779], [2406, 2821], [2406, 2863], [2406, 3067], [2407, 2408], [2407, 2500], [2407, 2547], [2407, 2677], [2407, 2731], [2407, 2762], [2407, 2834], [2407, 2998], [2408, 2460], [2408, 2474], [2408, 2485], [2408, 2526], [2408, 2874], [2409, 2410], [2409, 2439], [2409, 2509], [2409, 2528], [2409, 2540], [2409, 2812], [2410, 2411], [2410, 2436], [2410, 2541], [2410, 2591], [2410, 2764], [2410, 2812], [2411, 2412], [2411, 2439], [2411, 2626], [2411, 2815], [2411, 2856], [2412, 2413], [2412, 2541], [2412, 2542], [2412, 2630], [2413, 2447], [2413, 2460], [2413, 2491], [2413, 2566], [2413, 2584], [2413, 2633], [2413, 2785], [2413, 3007], [2414, 2415], [2414, 2418], [2414, 2442], [2414, 2463], [2414, 2531], [2414, 2548], [2414, 2590], [2414, 2603], [2414, 2718], [2414, 2823], [2414, 3167], [2415, 2416], [2415, 2417], [2415, 2531], [2415, 2588], [2415, 2589], [2415, 3136], [2416, 2417], [2416, 2512], [2416, 2514], [2416, 2797], [2416, 2798], [2416, 3135], [2417, 2447], [2417, 2450], [2417, 2463], [2417, 2677], [2417, 2715], [2417, 2817], [2417, 2972], [2418, 2419], [2418, 2420], [2418, 2424], [2418, 2464], [2418, 2551], [2418, 2604], [2418, 2786], [2418, 2869], [2418, 3170], [2418, 3171], [2419, 2420], [2419, 2704], [2419, 2720], [2420, 2866], [2420, 3021], [2420, 3041], [2421, 2422], [2421, 2502], [2421, 2694], [2421, 2695], [2422, 2423], [2422, 2502], [2422, 2503], [2422, 2515], [2422, 2516], [2422, 2607], [2422, 2665], [2422, 2919], [2422, 3109], [2422, 3112], [2423, 2424], [2423, 2494], [2423, 2558], [2423, 2607], [2423, 2643], [2423, 2695], [2423, 2696], [2423, 2771], [2423, 2792], [2423, 2865], [2423, 2874], [2423, 2954], [2423, 2956], [2423, 3008], [2423, 3174], [2424, 2467], [2424, 2531], [2424, 2536], [2424, 2657], [2424, 2663], [2424, 2664], [2424, 2720], [2424, 2722], [2424, 2827], [2424, 3169], [2424, 3190], [2425, 2426], [2425, 2478], [2425, 2501], [2425, 2561], [2425, 2562], [2425, 2789], [2425, 2798], [2425, 2821], [2425, 2844], [2425, 2909], [2425, 3081], [2425, 3123], [2426, 2427], [2426, 2435], [2426, 2490], [2426, 2561], [2426, 2843], [2426, 3078], [2427, 2428], [2427, 2444], [2427, 2553], [2427, 2730], [2427, 2821], [2427, 2898], [2427, 3078], [2427, 3119], [2428, 2682], [2428, 2687], [2428, 2839], [2428, 2897], [2428, 3120], [2429, 2430], [2429, 2586], [2429, 2614], [2429, 2831], [2430, 2431], [2430, 2506], [2430, 2831], [2430, 2852], [2430, 2853], [2430, 2860], [2431, 2551], [2431, 2737], [2431, 2829], [2431, 2852], [2431, 2864], [2431, 3125], [2431, 3181], [2432, 2433], [2432, 2505], [2432, 2513], [2432, 2581], [2432, 2953], [2433, 2434], [2433, 2579], [2433, 2580], [2433, 2699], [2434, 2435], [2434, 2468], [2434, 2483], [2434, 2578], [2434, 2838], [2434, 2851], [2434, 2861], [2434, 3187], [2435, 2506], [2435, 2617], [2435, 2675], [2435, 3500], [2436, 2437], [2436, 2446], [2436, 2480], [2436, 2486], [2436, 2528], [2437, 2438], [2437, 2637], [2437, 2848], [2437, 2922], [2438, 2581], [2438, 2638], [2438, 2657], [2438, 2669], [2438, 2744], [2438, 2752], [2438, 2839], [2438, 2989], [2438, 3042], [2438, 3084], [2438, 3102], [2438, 3121], [2439, 2440], [2439, 2509], [2439, 2510], [2439, 2630], [2439, 2815], [2439, 2876], [2440, 2441], [2440, 2545], [2440, 2582], [2440, 2583], [2440, 2646], [2440, 2651], [2440, 2854], [2440, 2909], [2441, 2510], [2441, 2581], [2441, 2584], [2441, 2644], [2441, 2647], [2441, 2838], [2441, 2855], [2441, 2870], [2441, 2908], [2441, 3083], [2441, 3087], [2441, 3144], [2441, 3152], [2442, 2443], [2442, 2479], [2442, 2494], [2442, 2554], [2442, 2570], [2442, 2578], [2442, 2589], [2442, 2590], [2442, 2592], [2442, 2706], [2442, 2720], [2442, 2824], [2442, 2842], [2442, 2940], [2442, 2961], [2442, 2964], [2442, 3021], [2442, 3024], [2442, 3161], [2442, 3167], [2442, 3171], [2443, 2444], [2443, 2458], [2443, 2483], [2443, 2513], [2443, 2553], [2443, 2746], [2443, 2842], [2443, 2914], [2443, 3172], [2443, 3173], [2444, 2457], [2444, 2458], [2444, 3149], [2444, 3161], [2445, 2446], [2445, 2448], [2445, 2526], [2445, 2613], [2445, 2673], [2445, 2675], [2445, 2723], [2445, 2739], [2445, 2748], [2445, 2767], [2445, 2866], [2445, 2942], [2445, 3021], [2445, 3043], [2446, 2447], [2446, 2485], [2446, 2526], [2446, 2528], [2446, 2633], [2446, 2702], [2446, 2712], [2446, 2715], [2446, 2749], [2446, 2812], [2446, 2813], [2446, 3020], [2446, 3195], [2447, 2485], [2447, 2541], [2447, 2565], [2447, 2566], [2447, 2922], [2448, 2498], [2448, 2575], [2448, 2697], [2448, 2700], [2448, 2738], [2448, 2757], [2448, 2781], [2448, 2810], [2448, 3043], [2448, 3118], [2449, 2450], [2449, 2799], [2449, 2863], [2450, 2451], [2450, 2676], [2450, 2691], [2450, 2701], [2450, 2747], [2450, 2788], [2450, 3076], [2451, 2452], [2451, 2487], [2451, 2683], [2451, 2691], [2451, 2702], [2451, 2705], [2451, 2774], [2451, 2821], [2451, 2841], [2451, 2951], [2451, 3019], [2451, 3076], [2451, 3114], [2451, 3183], [2452, 2456], [2452, 2687], [2452, 2841], [2452, 2843], [2452, 2936], [2453, 2454], [2453, 2458], [2453, 2492], [2453, 2649], [2453, 2650], [2453, 2760], [2454, 2455], [2454, 2458], [2454, 2761], [2454, 3061], [2455, 2456], [2455, 2492], [2455, 2603], [2456, 2457], [2456, 2458], [2456, 2472], [2456, 2483], [2456, 2492], [2456, 2494], [2456, 2940], [2457, 2483], [2457, 2495], [2457, 2862], [2457, 2916], [2457, 3177], [2458, 2459], [2458, 2476], [2458, 2543], [2458, 2643], [2458, 2730], [2458, 2822], [2458, 2827], [2458, 2930], [2458, 2942], [2458, 3021], [2458, 3061], [2458, 3147], [2459, 2460], [2459, 2476], [2459, 2520], [2459, 2571], [2459, 2595], [2459, 2674], [2459, 2743], [2459, 2759], [2459, 2931], [2459, 3009], [2460, 2475], [2460, 2500], [2460, 2507], [2460, 2508], [2460, 2529], [2460, 2630], [2460, 2925], [2460, 3193], [2461, 2462], [2461, 2465], [2461, 2817], [2461, 2837], [2462, 2463], [2462, 2800], [2462, 2830], [2463, 2464], [2463, 2706], [2463, 3040], [2464, 2577], [2464, 2619], [2464, 2657], [2464, 2786], [2464, 2830], [2464, 2880], [2465, 2466], [2465, 2467], [2465, 2531], [2465, 2648], [2465, 2662], [2465, 2943], [2465, 3070], [2465, 3071], [2466, 2467], [2466, 2584], [2466, 2824], [2466, 3191], [2467, 2468], [2467, 2737], [2467, 2828], [2467, 2837], [2467, 2838], [2467, 2865], [2467, 2886], [2467, 3029], [2467, 3087], [2467, 3116], [2467, 3125], [2467, 3145], [2467, 3185], [2468, 2578], [2468, 2699], [2468, 2900], [2468, 2917], [2468, 2920], [2468, 2921], [2468, 2938], [2468, 2961], [2468, 3146], [2468, 3164], [2469, 2470], [2469, 2489], [2469, 2501], [2469, 2528], [2469, 2637], [2469, 2795], [2469, 2844], [2470, 2471], [2470, 2639], [2470, 2756], [2470, 2853], [2470, 2866], [2470, 2942], [2471, 2484], [2471, 2502], [2471, 2525], [2471, 2580], [2471, 2581], [2471, 2782], [2471, 2977], [2471, 3109], [2472, 2473], [2472, 2474], [2472, 2482], [2472, 2489], [2472, 2517], [2472, 2576], [2472, 2814], [2472, 2841], [2472, 2867], [2472, 2987], [2472, 3093], [2472, 3104], [2472, 3192], [2473, 2474], [2473, 2741], [2473, 2747], [2473, 2774], [2473, 2792], [2473, 2859], [2473, 2868], [2473, 2874], [2473, 2935], [2473, 3015], [2473, 3033], [2473, 3034], [2473, 3065], [2473, 3076], [2474, 2673], [2474, 2674], [2474, 2762], [2474, 2965], [2474, 3009], [2475, 2476], [2475, 2491], [2475, 2492], [2475, 2876], [2476, 2477], [2476, 2516], [2476, 2526], [2476, 2638], [2476, 2693], [2476, 2735], [2476, 2747], [2476, 2777], [2476, 2935], [2476, 2936], [2476, 2939], [2476, 3019], [2476, 3062], [2477, 2802], [2477, 2848], [2477, 3147], [2478, 2479], [2478, 2501], [2478, 2507], [2478, 2509], [2478, 2579], [2478, 2596], [2478, 2603], [2478, 2604], [2478, 2822], [2479, 2480], [2479, 2533], [2479, 2570], [2479, 2605], [2479, 2870], [2479, 3153], [2479, 3157], [2480, 2596], [2480, 2597], [2480, 2599], [2480, 2728], [2481, 2482], [2481, 2486], [2482, 2483], [2482, 2626], [2482, 2851], [2482, 3175], [2482, 3197], [2483, 2484], [2483, 2513], [2483, 2917], [2483, 2953], [2483, 3187], [2484, 2530], [2484, 2598], [2484, 2659], [2484, 2696], [2484, 2731], [2484, 2838], [2484, 2853], [2484, 2865], [2484, 2953], [2485, 2486], [2485, 2487], [2485, 2488], [2485, 2496], [2485, 2600], [2485, 3007], [2485, 3192], [2486, 2487], [2486, 2598], [2486, 2601], [2487, 2599], [2487, 2602], [2487, 3058], [2487, 3196], [2488, 2489], [2488, 2749], [2488, 2833], [2488, 2874], [2488, 2990], [2489, 2601], [2489, 2759], [2489, 2841], [2489, 2917], [2489, 3050], [2489, 3051], [2490, 2491], [2490, 2587], [2490, 2841], [2490, 2844], [2491, 2581], [2491, 2626], [2491, 2862], [2491, 3089], [2491, 3177], [2492, 2493], [2492, 2862], [2493, 2494], [2493, 2590], [2493, 2603], [2493, 2627], [2493, 2649], [2493, 2695], [2493, 2719], [2493, 2876], [2494, 2495], [2494, 2516], [2494, 2603], [2494, 2606], [2494, 2643], [2494, 2720], [2494, 2843], [2494, 2875], [2494, 2959], [2494, 2960], [2494, 2982], [2494, 3019], [2495, 2533], [2495, 2627], [2495, 2957], [2495, 2958], [2495, 2982], [2495, 3174], [2495, 3176], [2496, 2497], [2496, 2516], [2496, 2631], [2496, 2848], [2496, 2922], [2497, 2498], [2497, 2536], [2497, 2781], [2497, 2866], [2497, 3052], [2497, 3155], [2498, 2675], [2498, 2684], [2498, 2703], [2498, 2811], [2498, 2814], [2498, 2816], [2498, 3118], [2499, 2500], [2499, 2504], [2499, 2507], [2499, 2628], [2500, 2501], [2500, 2505], [2500, 2588], [2500, 2612], [2500, 2630], [2500, 2637], [2500, 2835], [2501, 2502], [2502, 2503], [2502, 2579], [2502, 2696], [2503, 2682], [2503, 2746], [2503, 2825], [2503, 2919], [2503, 2977], [2503, 3187], [2504, 2505], [2504, 2518], [2504, 2555], [2504, 2571], [2504, 2584], [2504, 2644], [2504, 2722], [2505, 2506], [2505, 2584], [2505, 2731], [2506, 2520], [2506, 2557], [2506, 2659], [2506, 2829], [2506, 2835], [2506, 3127], [2507, 2508], [2507, 2608], [2507, 2628], [2508, 2509], [2508, 2576], [2508, 2629], [2509, 2510], [2509, 2527], [2509, 2580], [2509, 2758], [2509, 2918], [2509, 2969], [2510, 2875], [2510, 2907], [2510, 2918], [2510, 3152], [2510, 3184], [2511, 2512], [2511, 2533], [2511, 2627], [2512, 2513], [2512, 2514], [2512, 2588], [2512, 2646], [2513, 2514], [2513, 2626], [2513, 2627], [2514, 2533], [2514, 2589], [2514, 2647], [2514, 2843], [2514, 2917], [2514, 3135], [2514, 3173], [2514, 3194], [2515, 2516], [2515, 2643], [2515, 2718], [2515, 2724], [2515, 2746], [2515, 2765], [2515, 2800], [2516, 2517], [2516, 2526], [2516, 2578], [2516, 2606], [2516, 2782], [2516, 2920], [2516, 2958], [2516, 3064], [2516, 3113], [2516, 3187], [2517, 2623], [2517, 2625], [2517, 2626], [2517, 2632], [2517, 2702], [2517, 2765], [2517, 2815], [2517, 2816], [2517, 2876], [2517, 2985], [2517, 3015], [2517, 3043], [2517, 3063], [2518, 2519], [2518, 2536], [2518, 2795], [2519, 2520], [2519, 2522], [2519, 2537], [2519, 2633], [2519, 2825], [2519, 3051], [2519, 3074], [2520, 2521], [2520, 2675], [2520, 2722], [2520, 2743], [2520, 2767], [2520, 2777], [2520, 2811], [2520, 2825], [2520, 3090], [2521, 2639], [2521, 2759], [2521, 2768], [2521, 2811], [2521, 2891], [2521, 2984], [2521, 3055], [2522, 2523], [2522, 2537], [2522, 2572], [2522, 2632], [2522, 2657], [2522, 2743], [2522, 2753], [2522, 2802], [2522, 2879], [2522, 2945], [2522, 2948], [2522, 3034], [2523, 2524], [2523, 2535], [2523, 2607], [2523, 2681], [2523, 2743], [2523, 2748], [2523, 2788], [2523, 2791], [2523, 2792], [2523, 2941], [2523, 2951], [2523, 2952], [2523, 3112], [2523, 3114], [2524, 2567], [2524, 2602], [2524, 2736], [2524, 2840], [2524, 2893], [2524, 2941], [2524, 2948], [2524, 2981], [2524, 2999], [2524, 3050], [2524, 3052], [2524, 3053], [2524, 3055], [2524, 3056], [2524, 3100], [2524, 3118], [2524, 3182], [2524, 3196], [2525, 2526], [2525, 2661], [2525, 2731], [2525, 2942], [2526, 2527], [2526, 2580], [2526, 2657], [2526, 2819], [2526, 2967], [2526, 2968], [2526, 3027], [2526, 3071], [2526, 3072], [2527, 2540], [2527, 2542], [2527, 2630], [2527, 2701], [2527, 2762], [2527, 2877], [2527, 3025], [2527, 3193], [2528, 2529], [2528, 2596], [2528, 2629], [2528, 2637], [2529, 2530], [2529, 2571], [2529, 2702], [2529, 2876], [2530, 2571], [2530, 2608], [2530, 2696], [2530, 2954], [2531, 2532], [2531, 2547], [2531, 2571], [2531, 2822], [2531, 2945], [2532, 2533], [2532, 2547], [2532, 2610], [2532, 2817], [2532, 2820], [2532, 2962], [2532, 2964], [2533, 2569], [2533, 2603], [2533, 2605], [2533, 2627], [2533, 2642], [2533, 2709], [2533, 2957], [2533, 3194], [2534, 2535], [2534, 2688], [2534, 2729], [2534, 2730], [2534, 2743], [2534, 2773], [2534, 2779], [2535, 2728], [2535, 2791], [2536, 2537], [2536, 2607], [2536, 2684], [2536, 2722], [2536, 2745], [2536, 2835], [2537, 2538], [2537, 2737], [2537, 2829], [2537, 2981], [2537, 3001], [2537, 3073], [2537, 3155], [2537, 3169], [2538, 2539], [2538, 2607], [2538, 2725], [2538, 2829], [2538, 2864], [2538, 2919], [2538, 2971], [2538, 2999], [2538, 3051], [2538, 3151], [2539, 2600], [2539, 2606], [2539, 2725], [2539, 2920], [2539, 2960], [2539, 2979], [2539, 3007], [2540, 2541], [2540, 2545], [2540, 2630], [2540, 2672], [2541, 2542], [2541, 2545], [2541, 2760], [2541, 2765], [2541, 2818], [2541, 2909], [2542, 2543], [2542, 2566], [2542, 2819], [2542, 2889], [2542, 2908], [2542, 3005], [2542, 3020], [2542, 3147], [2543, 2544], [2543, 2562], [2543, 2751], [2543, 2760], [2543, 2761], [2543, 2806], [2543, 2808], [2543, 2820], [2543, 2842], [2543, 3021], [2543, 3059], [2544, 2595], [2544, 2660], [2544, 2668], [2544, 2731], [2544, 2753], [2544, 2762], [2544, 2931], [2544, 2966], [2544, 2997], [2544, 3036], [2544, 3149], [2545, 2546], [2545, 2646], [2546, 2547], [2546, 2651], [2546, 2672], [2547, 2762], [2547, 2821], [2547, 2846], [2547, 3178], [2548, 2549], [2548, 2552], [2548, 2554], [2548, 2555], [2548, 2558], [2548, 2562], [2548, 2565], [2548, 2568], [2548, 2571], [2548, 2666], [2548, 2695], [2548, 2723], [2548, 2820], [2548, 2869], [2549, 2550], [2549, 2679], [2549, 2685], [2549, 2820], [2549, 2842], [2549, 2914], [2550, 2551], [2550, 2683], [2550, 2684], [2550, 2786], [2550, 2789], [2550, 2805], [2550, 2835], [2550, 2870], [2550, 2989], [2550, 3004], [2550, 3127], [2550, 3131], [2550, 3132], [2551, 2622], [2551, 3123], [2552, 2553], [2552, 2651], [2552, 2695], [2552, 2724], [2552, 2770], [2552, 2771], [2552, 2821], [2553, 2554], [2553, 2696], [2553, 2730], [2553, 2771], [2554, 2558], [2554, 2696], [2554, 2721], [2554, 3119], [2555, 2556], [2555, 2734], [2555, 2869], [2556, 2557], [2556, 2558], [2556, 2703], [2556, 2956], [2557, 2641], [2557, 2658], [2557, 2786], [2557, 3039], [2557, 3127], [2557, 3171], [2558, 2559], [2558, 2560], [2558, 2563], [2558, 2572], [2558, 2658], [2558, 2683], [2558, 2714], [2558, 2771], [2558, 2882], [2558, 2947], [2558, 3130], [2558, 3167], [2559, 2560], [2559, 2564], [2559, 2565], [2559, 2566], [2559, 2589], [2559, 2878], [2559, 2964], [2559, 2992], [2559, 2993], [2559, 3058], [2559, 3077], [2559, 3194], [2560, 2561], [2560, 2790], [2560, 3079], [2560, 3157], [2561, 2563], [2561, 2564], [2561, 2841], [2561, 2898], [2561, 2900], [2561, 2908], [2561, 2915], [2561, 2926], [2561, 2978], [2561, 3003], [2561, 3018], [2562, 2563], [2562, 2615], [2562, 2686], [2562, 2690], [2562, 2731], [2562, 2831], [2562, 2842], [2563, 2564], [2563, 2656], [2563, 2687], [2563, 2934], [2563, 2947], [2563, 3059], [2563, 3120], [2563, 3180], [2564, 2589], [2564, 2592], [2564, 2877], [2564, 2913], [2564, 3083], [2564, 3163], [2564, 3178], [2565, 2588], [2565, 2642], [2565, 2701], [2565, 2870], [2566, 2567], [2566, 2889], [2566, 2900], [2566, 2906], [2566, 2921], [2566, 2972], [2566, 3003], [2566, 3079], [2566, 3089], [2566, 3139], [2566, 3191], [2566, 3192], [2566, 3193], [2566, 3195], [2566, 3196], [2567, 2619], [2567, 2792], [2567, 2865], [2567, 2872], [2567, 2877], [2567, 2881], [2567, 2884], [2567, 2886], [2567, 2887], [2567, 2888], [2567, 2890], [2567, 2891], [2567, 2893], [2567, 2897], [2567, 2899], [2567, 2901], [2567, 2903], [2567, 2906], [2567, 2912], [2567, 2913], [2567, 2916], [2567, 2918], [2567, 2919], [2567, 2923], [2567, 2930], [2567, 2943], [2567, 2949], [2567, 2957], [2567, 2970], [2567, 2972], [2567, 2979], [2567, 2983], [2567, 2985], [2567, 2988], [2567, 2991], [2567, 2992], [2567, 2994], [2567, 2997], [2567, 2999], [2567, 3006], [2567, 3010], [2567, 3012], [2567, 3016], [2567, 3033], [2567, 3037], [2567, 3046], [2567, 3094], [2567, 3105], [2567, 3124], [2567, 3134], [2567, 3143], [2567, 3162], [2567, 3168], [2567, 3632], [2568, 2569], [2568, 2570], [2568, 2691], [2569, 2570], [2569, 2692], [2569, 2832], [2570, 2705], [2570, 2728], [2570, 2790], [2570, 2857], [2570, 2869], [2570, 3096], [2570, 3097], [2571, 2572], [2571, 2643], [2571, 2648], [2571, 2666], [2571, 2668], [2571, 2693], [2571, 2722], [2571, 2731], [2572, 2573], [2572, 2658], [2572, 2931], [2572, 2934], [2572, 2945], [2572, 2946], [2572, 3001], [2573, 2589], [2573, 2643], [2573, 2645], [2573, 2725], [2573, 2930], [2573, 2937], [2573, 2960], [2573, 2992], [2573, 3024], [2574, 2575], [2574, 2577], [2574, 2580], [2574, 2653], [2574, 2700], [2574, 3110], [2575, 2576], [2575, 2609], [2575, 2778], [2576, 2608], [2576, 2678], [2576, 3050], [2576, 3065], [2576, 3110], [2576, 3118], [2576, 3141], [2577, 2578], [2577, 2700], [2577, 2817], [2577, 2851], [2577, 2964], [2577, 3164], [2578, 2717], [2578, 2795], [2578, 2837], [2578, 2857], [2578, 2922], [2578, 3147], [2578, 3190], [2579, 2580], [2579, 2663], [2579, 2665], [2579, 2704], [2579, 2719], [2580, 2581], [2580, 2657], [2580, 2814], [2580, 2837], [2580, 2851], [2580, 2866], [2580, 3110], [2580, 3185], [2581, 2582], [2581, 2584], [2581, 2592], [2581, 2626], [2581, 2756], [2581, 2845], [2581, 2896], [2581, 2976], [2581, 3043], [2581, 3054], [2581, 3102], [2581, 3111], [2581, 3163], [2581, 3197], [2582, 2583], [2582, 2591], [2582, 2669], [2582, 2804], [2583, 2584], [2583, 2586], [2583, 2597], [2584, 2587], [2584, 2598], [2584, 2648], [2584, 2989], [2584, 3074], [2585, 2586], [2585, 2734], [2585, 2858], [2586, 2587], [2586, 2597], [2586, 2734], [2587, 2598], [2587, 2689], [2587, 2840], [2587, 3045], [2587, 3144], [2588, 2589], [2588, 2590], [2588, 2643], [2588, 2644], [2588, 2914], [2589, 2645], [2589, 2965], [2589, 3136], [2589, 3147], [2589, 3174], [2589, 3175], [2590, 2591], [2590, 2612], [2590, 2719], [2591, 2592], [2591, 2876], [2592, 2698], [2592, 2875], [2592, 2964], [2592, 3029], [2592, 3125], [2592, 3166], [2593, 2594], [2593, 2691], [2593, 2804], [2593, 2818], [2593, 2858], [2594, 2595], [2594, 2635], [2594, 2719], [2594, 2760], [2594, 2818], [2595, 2610], [2595, 2630], [2595, 2672], [2595, 2674], [2595, 2744], [2595, 2760], [2595, 2762], [2596, 2597], [2596, 2791], [2597, 2598], [2597, 2680], [2598, 2599], [2598, 2601], [2598, 2681], [2598, 2853], [2598, 3175], [2598, 3184], [2599, 2602], [2599, 2850], [2599, 3057], [2599, 3074], [2599, 3080], [2599, 3134], [2599, 3153], [2600, 2601], [2600, 2602], [2600, 2606], [2600, 2712], [2600, 2769], [2600, 2799], [2600, 2959], [2600, 2979], [2600, 3064], [2601, 2602], [2601, 2648], [2601, 3050], [2602, 2625], [2602, 2753], [2602, 2957], [2602, 2990], [2602, 3056], [2602, 3077], [2602, 3148], [2602, 3149], [2602, 3150], [2603, 2604], [2603, 2718], [2603, 2798], [2603, 2959], [2604, 2605], [2604, 2608], [2604, 2832], [2604, 3017], [2604, 3123], [2605, 2625], [2605, 2737], [2605, 2808], [2605, 3023], [2605, 3024], [2605, 3142], [2606, 2607], [2606, 2724], [2606, 3114], [2607, 2608], [2607, 2681], [2607, 2683], [2607, 2694], [2607, 2724], [2607, 2835], [2607, 2863], [2607, 3050], [2608, 2665], [2608, 2683], [2608, 2723], [2608, 2781], [2608, 2786], [2608, 2971], [2609, 2610], [2609, 2673], [2609, 2778], [2610, 2611], [2610, 2629], [2610, 2684], [2610, 2750], [2610, 2753], [2610, 2765], [2610, 2817], [2610, 2846], [2611, 2753], [2611, 2805], [2611, 2819], [2611, 2989], [2611, 3005], [2612, 2613], [2612, 2614], [2612, 2620], [2612, 2823], [2612, 2824], [2613, 2614], [2613, 2621], [2613, 2857], [2614, 2622], [2614, 2783], [2614, 2785], [2615, 2616], [2615, 2656], [2615, 2661], [2615, 2662], [2615, 2666], [2615, 2729], [2615, 2731], [2616, 2617], [2616, 2842], [2617, 2659], [2617, 2688], [2617, 2693], [2617, 2801], [2617, 2843], [2617, 2936], [2618, 2619], [2618, 2662], [2618, 2698], [2618, 2713], [2618, 2883], [2618, 2997], [2618, 3042], [2618, 3070], [2618, 3085], [2619, 2698], [2619, 2736], [2619, 2828], [2619, 2938], [2619, 2987], [2619, 2994], [2619, 3032], [2619, 3039], [2619, 3040], [2619, 3041], [2619, 3044], [2619, 3086], [2619, 3094], [2619, 3159], [2619, 3164], [2619, 3170], [2619, 3188], [2620, 2621], [2620, 2622], [2620, 2837], [2621, 2622], [2621, 2710], [2621, 2740], [2621, 2832], [2622, 2772], [2622, 2831], [2622, 2849], [2622, 3086], [2622, 3128], [2623, 2624], [2623, 2701], [2623, 2715], [2623, 2797], [2623, 2812], [2623, 3063], [2624, 2625], [2624, 2634], [2625, 2634], [2625, 2736], [2625, 2752], [2625, 2763], [2625, 2826], [2625, 2850], [2625, 2986], [2625, 3015], [2626, 2627], [2626, 2856], [2626, 2876], [2626, 3176], [2626, 3194], [2627, 2862], [2627, 2914], [2628, 2629], [2628, 2630], [2628, 2691], [2628, 2701], [2628, 2702], [2628, 2723], [2629, 2630], [2629, 2844], [2631, 2632], [2631, 2715], [2631, 2818], [2632, 2633], [2632, 2715], [2632, 2819], [2632, 3114], [2633, 2675], [2633, 2784], [2633, 2815], [2633, 2968], [2633, 3007], [2633, 3195], [2634, 2635], [2634, 2636], [2634, 2754], [2635, 2636], [2635, 2719], [2635, 2775], [2635, 2776], [2636, 2671], [2636, 2796], [2636, 2801], [2636, 2830], [2637, 2638], [2637, 2639], [2637, 2670], [2637, 2759], [2638, 2639], [2638, 2654], [2638, 2662], [2638, 2688], [2638, 2693], [2638, 2778], [2638, 2779], [2638, 3042], [2638, 3055], [2639, 2640], [2639, 2689], [2639, 2768], [2639, 2824], [2640, 2641], [2640, 2778], [2640, 2780], [2640, 2786], [2640, 2810], [2640, 2811], [2640, 2858], [2641, 2773], [2641, 2811], [2641, 2860], [2642, 2643], [2642, 2647], [2642, 2678], [2642, 2870], [2642, 2954], [2642, 2992], [2642, 3030], [2643, 2644], [2643, 2647], [2643, 2649], [2643, 2693], [2643, 2724], [2643, 3023], [2644, 2645], [2644, 2870], [2644, 2956], [2645, 2993], [2645, 3083], [2645, 3168], [2645, 3171], [2646, 2647], [2646, 2649], [2646, 2765], [2646, 2876], [2647, 2648], [2647, 2865], [2647, 2875], [2647, 3025], [2647, 3028], [2647, 3122], [2647, 3123], [2648, 2677], [2648, 2683], [2648, 2731], [2648, 2753], [2648, 2759], [2648, 2946], [2648, 3058], [2649, 2650], [2649, 2695], [2650, 2651], [2650, 2739], [2651, 2663], [2651, 2669], [2651, 2724], [2651, 2739], [2651, 2835], [2651, 2854], [2652, 2653], [2652, 2657], [2652, 2661], [2652, 2663], [2652, 2666], [2652, 2669], [2652, 2793], [2652, 2794], [2653, 2654], [2653, 2657], [2653, 2664], [2653, 2778], [2653, 2780], [2653, 2885], [2654, 2655], [2654, 2661], [2654, 2662], [2654, 2766], [2655, 2656], [2655, 2657], [2655, 2661], [2655, 2688], [2655, 2777], [2655, 2825], [2655, 3028], [2656, 2658], [2656, 2662], [2656, 2688], [2656, 2934], [2656, 2983], [2657, 2658], [2657, 2748], [2657, 2749], [2657, 2752], [2657, 2773], [2657, 2874], [2657, 2885], [2657, 2922], [2657, 2994], [2657, 3190], [2658, 2659], [2658, 2666], [2658, 2773], [2658, 2932], [2658, 3024], [2659, 2660], [2659, 2681], [2659, 2731], [2659, 2743], [2659, 2771], [2659, 2831], [2659, 2834], [2659, 3047], [2660, 2741], [2660, 2743], [2660, 2744], [2660, 2808], [2660, 2898], [2660, 3046], [2660, 3121], [2661, 2776], [2661, 2795], [2662, 2824], [2662, 2832], [2662, 2939], [2662, 2983], [2662, 2998], [2662, 3008], [2662, 3027], [2663, 2664], [2663, 2695], [2663, 2719], [2663, 2836], [2663, 2837], [2664, 2665], [2664, 2718], [2665, 2697], [2665, 2699], [2665, 2718], [2665, 2781], [2665, 2970], [2665, 3110], [2666, 2667], [2667, 2668], [2667, 2704], [2667, 2744], [2667, 2752], [2667, 2820], [2668, 2702], [2668, 2753], [2668, 2820], [2668, 3015], [2668, 3023], [2669, 2670], [2669, 2671], [2669, 2716], [2669, 2846], [2670, 2671], [2670, 2830], [2670, 2835], [2670, 2847], [2671, 2706], [2671, 2735], [2671, 2844], [2671, 2848], [2672, 2673], [2672, 2739], [2672, 2762], [2673, 2674], [2673, 2747], [2674, 2675], [2674, 2774], [2675, 2721], [2675, 2968], [2676, 2677], [2676, 2683], [2676, 2738], [2676, 2778], [2676, 3066], [2677, 2678], [2678, 2768], [2678, 3031], [2678, 3037], [2679, 2680], [2680, 2681], [2680, 2694], [2680, 2791], [2680, 2833], [2680, 2914], [2681, 2682], [2681, 2789], [2681, 2833], [2681, 2989], [2681, 3150], [2681, 3151], [2681, 3174], [2681, 3182], [2682, 2941], [2682, 3143], [2682, 3144], [2682, 3172], [2682, 3182], [2683, 2684], [2683, 2687], [2683, 2730], [2683, 2773], [2683, 2880], [2683, 2999], [2683, 3058], [2683, 3066], [2683, 3123], [2684, 2685], [2684, 2710], [2684, 2738], [2684, 2811], [2684, 3031], [2684, 3073], [2685, 2708], [2685, 2709], [2685, 2723], [2685, 2745], [2685, 2871], [2685, 3014], [2685, 3030], [2686, 2687], [2686, 2689], [2687, 2688], [2687, 2730], [2687, 2843], [2687, 3045], [2687, 3163], [2688, 2773], [2688, 2936], [2688, 3054], [2688, 3055], [2689, 2690], [2690, 2824], [2690, 3081], [2690, 3191], [2691, 2692], [2691, 2728], [2691, 2844], [2692, 2693], [2692, 2723], [2692, 2727], [2692, 2728], [2693, 2772], [2693, 2798], [2693, 2800], [2693, 2834], [2693, 2874], [2693, 2876], [2693, 3015], [2694, 2695], [2694, 2791], [2695, 2696], [2695, 2822], [2695, 2914], [2697, 2698], [2697, 2700], [2697, 2704], [2697, 2751], [2697, 2878], [2697, 2964], [2697, 3006], [2697, 3021], [2697, 3034], [2698, 2699], [2698, 2877], [2698, 2967], [2698, 2969], [2698, 3021], [2698, 3043], [2698, 3104], [2698, 3129], [2699, 2782], [2699, 3110], [2699, 3188], [2700, 2803], [2700, 2813], [2700, 2814], [2700, 3117], [2701, 2702], [2701, 2878], [2702, 2703], [2702, 2704], [2702, 2714], [2702, 2878], [2702, 2950], [2702, 3005], [2702, 3020], [2702, 3184], [2703, 2714], [2703, 2816], [2703, 2975], [2704, 2705], [2704, 2786], [2704, 2805], [2704, 2813], [2705, 2706], [2705, 2727], [2705, 2774], [2705, 2880], [2706, 2785], [2706, 3040], [2707, 2708], [2707, 2712], [2707, 2716], [2707, 2718], [2707, 2723], [2707, 2781], [2708, 2709], [2708, 2716], [2708, 2749], [2709, 2710], [2709, 2717], [2709, 2718], [2710, 2711], [2710, 2712], [2710, 2783], [2710, 2807], [2710, 2814], [2710, 2817], [2710, 2837], [2710, 2886], [2710, 3133], [2711, 2783], [2711, 2819], [2711, 2838], [2711, 2886], [2711, 2908], [2711, 3026], [2711, 3072], [2711, 3154], [2712, 2713], [2712, 2714], [2712, 2715], [2712, 3010], [2712, 3027], [2712, 3042], [2712, 3052], [2712, 3113], [2713, 2714], [2713, 2718], [2713, 2720], [2713, 2959], [2713, 2970], [2713, 3020], [2713, 3034], [2713, 3167], [2714, 2721], [2714, 2723], [2714, 2725], [2714, 2971], [2714, 2982], [2714, 3014], [2714, 3038], [2715, 2817], [2716, 2717], [2716, 2775], [2716, 2795], [2717, 2718], [2717, 2837], [2718, 2719], [2718, 2720], [2718, 2723], [2718, 2765], [2719, 2720], [2720, 2721], [2720, 2746], [2720, 3029], [2720, 3072], [2721, 2722], [2721, 2723], [2722, 2772], [2722, 2773], [2722, 2869], [2722, 3001], [2722, 3038], [2723, 2724], [2724, 2725], [2724, 2769], [2724, 2775], [2724, 2942], [2725, 2769], [2725, 2771], [2725, 2772], [2725, 2928], [2725, 2941], [2725, 3056], [2725, 3123], [2725, 3149], [2726, 2727], [2726, 2729], [2726, 2779], [2726, 2787], [2726, 2832], [2727, 2744], [2727, 2760], [2727, 2808], [2727, 2821], [2728, 2729], [2728, 2756], [2728, 2787], [2728, 2788], [2728, 2857], [2729, 2730], [2729, 2776], [2729, 2832], [2729, 2880], [2730, 2731], [2730, 2853], [2730, 2933], [2730, 2942], [2730, 3067], [2731, 2934], [2731, 2998], [2732, 2733], [2732, 2734], [2733, 2795], [2733, 2796], [2734, 2757], [2734, 2880], [2735, 2736], [2735, 2747], [2735, 3033], [2735, 3040], [2735, 3053], [2736, 2737], [2736, 2748], [2736, 2777], [2736, 2811], [2736, 3042], [2736, 3043], [2736, 3044], [2736, 3100], [2736, 3102], [2737, 2740], [2737, 2811], [2737, 2816], [2737, 2855], [2737, 2860], [2737, 2864], [2737, 2867], [2737, 2929], [2737, 2981], [2737, 3116], [2737, 3170], [2738, 2739], [2738, 2810], [2738, 3031], [2739, 2740], [2739, 2767], [2740, 2762], [2740, 3023], [2740, 3031], [2740, 3042], [2740, 3067], [2740, 3090], [2741, 2742], [2741, 2744], [2741, 2762], [2741, 2826], [2741, 2834], [2741, 2952], [2741, 3015], [2741, 3046], [2741, 3084], [2741, 3085], [2742, 2743], [2742, 2767], [2742, 2779], [2742, 2834], [2742, 3048], [2743, 2779], [2743, 2832], [2743, 3048], [2743, 3055], [2743, 3065], [2743, 3067], [2744, 2745], [2744, 2774], [2744, 2779], [2744, 2821], [2745, 2746], [2745, 2914], [2746, 2801], [2746, 3187], [2747, 2748], [2747, 2774], [2747, 2778], [2747, 2822], [2747, 2858], [2748, 2749], [2748, 2777], [2748, 2788], [2748, 2793], [2748, 2858], [2748, 2866], [2748, 2868], [2748, 3041], [2749, 2781], [2749, 2794], [2749, 2995], [2750, 2751], [2750, 2754], [2750, 2757], [2750, 2760], [2750, 2805], [2751, 2752], [2751, 2753], [2751, 2803], [2751, 2805], [2751, 2820], [2751, 2839], [2751, 2881], [2751, 3098], [2751, 3132], [2752, 2753], [2752, 3032], [2753, 2962], [2753, 2988], [2753, 3034], [2753, 3035], [2753, 3070], [2753, 3073], [2753, 3074], [2754, 2755], [2755, 2756], [2755, 2787], [2755, 3100], [2756, 2844], [2756, 2866], [2756, 3100], [2757, 2758], [2757, 3098], [2758, 2759], [2758, 3043], [2758, 3186], [2759, 3055], [2759, 3196], [2759, 3197], [2760, 2761], [2761, 2762], [2762, 2997], [2762, 3009], [2763, 2764], [2763, 2765], [2763, 2807], [2763, 2826], [2763, 2849], [2763, 2871], [2763, 3022], [2763, 3106], [2764, 2765], [2764, 2856], [2764, 2870], [2764, 3122], [2764, 3142], [2765, 2798], [2765, 2806], [2765, 2909], [2766, 2767], [2766, 2776], [2766, 2777], [2766, 2807], [2766, 2830], [2766, 2832], [2766, 2894], [2767, 2768], [2767, 2810], [2767, 2822], [2767, 2835], [2767, 2952], [2767, 3090], [2768, 2810], [2768, 2891], [2768, 3043], [2768, 3111], [2768, 3129], [2768, 3130], [2769, 2770], [2769, 2928], [2770, 2771], [2770, 2798], [2770, 2799], [2770, 2834], [2770, 2927], [2771, 2831], [2771, 2843], [2771, 2898], [2771, 2927], [2771, 3067], [2771, 3173], [2772, 2773], [2772, 2775], [2772, 2777], [2772, 2831], [2772, 2860], [2772, 2929], [2772, 2996], [2772, 3002], [2772, 3022], [2772, 3024], [2772, 3028], [2772, 3123], [2772, 3190], [2773, 2774], [2773, 2778], [2773, 2880], [2773, 3000], [2773, 3065], [2774, 2936], [2775, 2776], [2776, 2777], [2776, 2793], [2776, 2795], [2776, 2942], [2777, 2894], [2777, 2938], [2777, 2941], [2777, 3072], [2777, 3108], [2778, 2779], [2778, 3065], [2779, 2791], [2780, 2781], [2780, 2786], [2780, 2813], [2780, 2836], [2780, 3133], [2781, 2782], [2781, 3052], [2782, 3043], [2783, 2784], [2783, 2818], [2783, 2909], [2784, 2785], [2784, 2813], [2784, 2814], [2784, 2815], [2786, 2824], [2786, 2835], [2786, 3039], [2786, 3141], [2787, 2788], [2787, 2791], [2787, 2793], [2788, 2789], [2788, 2832], [2788, 2863], [2788, 2887], [2788, 3097], [2788, 3100], [2789, 2790], [2789, 2821], [2789, 2831], [2789, 2849], [2789, 2863], [2789, 2870], [2789, 2915], [2789, 3069], [2789, 3078], [2789, 3101], [2789, 3144], [2790, 2820], [2790, 2964], [2791, 2793], [2791, 2822], [2791, 2942], [2792, 2822], [2792, 2882], [2792, 2898], [2792, 2930], [2792, 2945], [2792, 2965], [2792, 2982], [2792, 3012], [2792, 3023], [2792, 3038], [2792, 3041], [2792, 3090], [2792, 3097], [2792, 3112], [2792, 3174], [2793, 2794], [2794, 2795], [2795, 2800], [2795, 2804], [2795, 2813], [2796, 2797], [2796, 2800], [2797, 2798], [2797, 2800], [2797, 3137], [2798, 2799], [2798, 2843], [2798, 2926], [2798, 3123], [2799, 2864], [2799, 2872], [2799, 2874], [2799, 3076], [2799, 3077], [2799, 3081], [2800, 2801], [2800, 2802], [2801, 2802], [2802, 3035], [2802, 3137], [2803, 2804], [2803, 2806], [2803, 2845], [2803, 2895], [2804, 2805], [2804, 2813], [2804, 2844], [2804, 2845], [2804, 2909], [2805, 3005], [2806, 2807], [2806, 3060], [2807, 2808], [2807, 2830], [2807, 2952], [2807, 3075], [2808, 2827], [2808, 2831], [2808, 2832], [2808, 2898], [2808, 2996], [2808, 3041], [2809, 2810], [2809, 2858], [2809, 2859], [2809, 3092], [2810, 2811], [2810, 3091], [2811, 2859], [2811, 3039], [2811, 3065], [2811, 3091], [2811, 3133], [2812, 2813], [2812, 2815], [2812, 2969], [2813, 2814], [2814, 2845], [2814, 2851], [2814, 2969], [2814, 3117], [2814, 3133], [2814, 3195], [2815, 2816], [2815, 2850], [2815, 2907], [2815, 2909], [2815, 2969], [2816, 2856], [2816, 2867], [2816, 2996], [2816, 3029], [2817, 2818], [2817, 2819], [2817, 3070], [2818, 2819], [2818, 2922], [2819, 2851], [2819, 2921], [2819, 2944], [2819, 2958], [2819, 3026], [2819, 3070], [2820, 2821], [2820, 2822], [2820, 2882], [2820, 3119], [2821, 2822], [2821, 2839], [2821, 2898], [2821, 2951], [2822, 2914], [2823, 2824], [2823, 3129], [2824, 2825], [2824, 2940], [2824, 2984], [2824, 3072], [2824, 3129], [2824, 3141], [2825, 2968], [2826, 2827], [2826, 2828], [2826, 3069], [2826, 3105], [2826, 3150], [2827, 2828], [2827, 3041], [2828, 2829], [2828, 2830], [2828, 2894], [2828, 3045], [2828, 3075], [2828, 3085], [2828, 3143], [2828, 3180], [2828, 3188], [2829, 2835], [2829, 2905], [2829, 2910], [2829, 3039], [2829, 3047], [2829, 3090], [2829, 3131], [2829, 3152], [2829, 3174], [2830, 2831], [2830, 2835], [2830, 2837], [2831, 2832], [2831, 3180], [2831, 3181], [2832, 3016], [2832, 3067], [2833, 2834], [2833, 3150], [2834, 2835], [2834, 2874], [2834, 3047], [2835, 2852], [2835, 2914], [2835, 2952], [2835, 3111], [2836, 2837], [2837, 2838], [2838, 2860], [2838, 3028], [2838, 3072], [2839, 2840], [2839, 2841], [2839, 2897], [2839, 3004], [2839, 3101], [2839, 3122], [2839, 3179], [2840, 2841], [2840, 2912], [2840, 2951], [2840, 3077], [2840, 3093], [2840, 3098], [2841, 2844], [2841, 2845], [2841, 2974], [2841, 3003], [2841, 3089], [2841, 3160], [2842, 2843], [2842, 3120], [2842, 3179], [2843, 2926], [2844, 2848], [2845, 2895], [2845, 2908], [2845, 3035], [2845, 3043], [2845, 3074], [2846, 2847], [2846, 2848], [2847, 2848], [2848, 2974], [2848, 3053], [2849, 2850], [2849, 2854], [2849, 2909], [2850, 2851], [2850, 2861], [2850, 2866], [2850, 3140], [2850, 3159], [2851, 3164], [2852, 2863], [2853, 2860], [2853, 2863], [2853, 2864], [2853, 3078], [2854, 2855], [2855, 2856], [2855, 3102], [2856, 2975], [2856, 3022], [2856, 3142], [2857, 2858], [2857, 2861], [2858, 2859], [2858, 2922], [2859, 2868], [2859, 2921], [2859, 2932], [2859, 2952], [2859, 3079], [2859, 3092], [2860, 2861], [2860, 3171], [2861, 2862], [2863, 2864], [2864, 2865], [2864, 2887], [2864, 2915], [2864, 3067], [2864, 3081], [2864, 3103], [2864, 3123], [2864, 3130], [2865, 2875], [2865, 2902], [2865, 2913], [2865, 2917], [2865, 2947], [2865, 2954], [2865, 2991], [2865, 2998], [2865, 3047], [2865, 3050], [2865, 3109], [2865, 3125], [2865, 3152], [2865, 3191], [2866, 2867], [2867, 2868], [2867, 2980], [2867, 2987], [2867, 3140], [2867, 3155], [2867, 3185], [2868, 2887], [2868, 2994], [2868, 2995], [2868, 2996], [2868, 3108], [2869, 2870], [2869, 2880], [2870, 2871], [2870, 2955], [2870, 2975], [2870, 2993], [2870, 3088], [2870, 3184], [2871, 2952], [2872, 2873], [2872, 2912], [2872, 2926], [2872, 2927], [2872, 2979], [2873, 2874], [2873, 2905], [2873, 2990], [2873, 2994], [2873, 3047], [2874, 3008], [2875, 2876], [2875, 2985], [2875, 3029], [2875, 3176], [2877, 2878], [2877, 2918], [2877, 2965], [2877, 2967], [2877, 2997], [2878, 2879], [2878, 2950], [2878, 2965], [2878, 3012], [2878, 3034], [2878, 3063], [2878, 3076], [2879, 2889], [2879, 2928], [2879, 2978], [2879, 2981], [2879, 2998], [2879, 3053], [2879, 3067], [2879, 3136], [2879, 3137], [2879, 3150], [2879, 3192], [2880, 3044], [2881, 2882], [2881, 2895], [2881, 2897], [2881, 2947], [2881, 3006], [2881, 3059], [2882, 2883], [2882, 2898], [2882, 2962], [2882, 3059], [2882, 3119], [2883, 2895], [2883, 2911], [2883, 2969], [2883, 3030], [2883, 3031], [2883, 3060], [2883, 3070], [2883, 3076], [2883, 3092], [2883, 3112], [2883, 3159], [2884, 2885], [2884, 3061], [2884, 3063], [2884, 3066], [2884, 3068], [2884, 3156], [2885, 2994], [2885, 3065], [2885, 3071], [2885, 3084], [2885, 3110], [2885, 3133], [2886, 3010], [2886, 3070], [2886, 3073], [2886, 3075], [2886, 3124], [2887, 2915], [2887, 3016], [2887, 3076], [2887, 3079], [2887, 3134], [2888, 2889], [2888, 2927], [2888, 2928], [2888, 3082], [2888, 3084], [2888, 3086], [2888, 3087], [2889, 3004], [2890, 2899], [2890, 3033], [2890, 3089], [2890, 3158], [2891, 2892], [2891, 2901], [2891, 3037], [2891, 3090], [2891, 3091], [2892, 2976], [2892, 2985], [2892, 3043], [2892, 3118], [2892, 3186], [2893, 2894], [2893, 2895], [2893, 3107], [2893, 3109], [2893, 3112], [2893, 3115], [2893, 3117], [2894, 2929], [2894, 3016], [2894, 3062], [2894, 3075], [2894, 3090], [2894, 3108], [2895, 2896], [2895, 3060], [2895, 3117], [2896, 2947], [2896, 2948], [2896, 2976], [2896, 3084], [2896, 3087], [2896, 3089], [2896, 3100], [2896, 3109], [2896, 3110], [2896, 3166], [2896, 3194], [2897, 2898], [2897, 3121], [2897, 3143], [2898, 2915], [2898, 3067], [2898, 3178], [2898, 3183], [2899, 2900], [2899, 3018], [2899, 3135], [2899, 3168], [2900, 3087], [2900, 3089], [2900, 3135], [2900, 3146], [2900, 3157], [2900, 3165], [2901, 2902], [2901, 2916], [2901, 3060], [2901, 3092], [2901, 3138], [2902, 2917], [2902, 2953], [2902, 3093], [2902, 3138], [2902, 3186], [2903, 2904], [2903, 2918], [2903, 3105], [2903, 3142], [2904, 2905], [2904, 2917], [2904, 2930], [2904, 2984], [2904, 3095], [2904, 3103], [2905, 2925], [2905, 2931], [2905, 3093], [2905, 3099], [2905, 3154], [2905, 3155], [2906, 2907], [2906, 2908], [2906, 2910], [2906, 2980], [2906, 2984], [2906, 2993], [2906, 3152], [2906, 3153], [2906, 3154], [2906, 3156], [2906, 3158], [2907, 2908], [2907, 2911], [2907, 2969], [2907, 2985], [2907, 3140], [2908, 2909], [2908, 3074], [2910, 2911], [2910, 2952], [2910, 3046], [2910, 3075], [2910, 3090], [2911, 2944], [2911, 3014], [2911, 3073], [2911, 3140], [2911, 3183], [2912, 3124], [2912, 3160], [2912, 3179], [2913, 2915], [2913, 2963], [2913, 3011], [2913, 3068], [2913, 3138], [2913, 3172], [2913, 3174], [2913, 3176], [2913, 3178], [2913, 3180], [2913, 3182], [2914, 3174], [2915, 2993], [2915, 3069], [2915, 3077], [2915, 3078], [2915, 3080], [2915, 3131], [2915, 3151], [2915, 3180], [2916, 2917], [2916, 2957], [2916, 3183], [2917, 2920], [2917, 3160], [2917, 3173], [2918, 3087], [2918, 3185], [2918, 3186], [2919, 2920], [2919, 2970], [2919, 3103], [2919, 3143], [2919, 3188], [2920, 2935], [2920, 2960], [2920, 2967], [2920, 2985], [2920, 3064], [2920, 3104], [2920, 3148], [2920, 3187], [2920, 3189], [2921, 2922], [2921, 2932], [2921, 2994], [2921, 3154], [2923, 2924], [2923, 2973], [2923, 3013], [2923, 3082], [2923, 3095], [2923, 3107], [2924, 2925], [2924, 2937], [2924, 2960], [2924, 2971], [2924, 2980], [2924, 2984], [2924, 2991], [2925, 2931], [2925, 3193], [2926, 2927], [2926, 2959], [2926, 3018], [2926, 3135], [2926, 3137], [2926, 3179], [2927, 2928], [2927, 3047], [2927, 3181], [2928, 2929], [2928, 2979], [2928, 2981], [2929, 2987], [2929, 3002], [2929, 3025], [2929, 3038], [2929, 3049], [2929, 3065], [2929, 3086], [2929, 3106], [2929, 3115], [2929, 3181], [2930, 2931], [2930, 2933], [2930, 2935], [2930, 2937], [2930, 2938], [2930, 2941], [2930, 3059], [2930, 3061], [2930, 3146], [2930, 3172], [2931, 2932], [2931, 2935], [2931, 2988], [2931, 3009], [2932, 2938], [2933, 2934], [2933, 2941], [2933, 2978], [2933, 2999], [2933, 3067], [2934, 2946], [2934, 2978], [2934, 2998], [2935, 2936], [2935, 2967], [2935, 3033], [2935, 3036], [2935, 3055], [2935, 3062], [2935, 3085], [2935, 3108], [2936, 2968], [2936, 3187], [2937, 2981], [2937, 2998], [2937, 3037], [2937, 3038], [2937, 3055], [2937, 3082], [2937, 3145], [2938, 2939], [2938, 2940], [2938, 3024], [2938, 3041], [2939, 2940], [2939, 3019], [2939, 3040], [2941, 2942], [2941, 3149], [2943, 2945], [2943, 2946], [2943, 2948], [2943, 2983], [2943, 3162], [2943, 3166], [2944, 2967], [2944, 2973], [2944, 3070], [2945, 2962], [2945, 3136], [2945, 3167], [2945, 3169], [2945, 3178], [2946, 2947], [2946, 2948], [2946, 2973], [2946, 2999], [2947, 3160], [2947, 3166], [2947, 3186], [2948, 3054], [2948, 3055], [2948, 3145], [2949, 2950], [2949, 2988], [2949, 3039], [2949, 3168], [2950, 2978], [2950, 2985], [2950, 3013], [2950, 3183], [2950, 3195], [2951, 2952], [2951, 3081], [2953, 3197], [2954, 2955], [2954, 3012], [2954, 3093], [2955, 2956], [2955, 3152], [2955, 3184], [2956, 3152], [2957, 2959], [2957, 2962], [2957, 2992], [2957, 3153], [2958, 3113], [2959, 2960], [2959, 2991], [2959, 3017], [2959, 3064], [2959, 3167], [2960, 2961], [2961, 2963], [2961, 2984], [2961, 3096], [2961, 3120], [2961, 3153], [2961, 3167], [2961, 3172], [2962, 2963], [2962, 3070], [2962, 3178], [2963, 2964], [2963, 3006], [2963, 3164], [2964, 3141], [2965, 2966], [2965, 3021], [2966, 2967], [2966, 3021], [2966, 3034], [2967, 2968], [2967, 2994], [2967, 3034], [2967, 3071], [2967, 3185], [2967, 3195], [2968, 3187], [2969, 3063], [2969, 3156], [2969, 3159], [2969, 3195], [2970, 2971], [2970, 3006], [2970, 3052], [2971, 2987], [2971, 2999], [2971, 3013], [2971, 3017], [2971, 3038], [2971, 3039], [2971, 3052], [2972, 2973], [2972, 3040], [2972, 3053], [2972, 3070], [2972, 3076], [2972, 3135], [2972, 3136], [2973, 2974], [2973, 3160], [2973, 3178], [2973, 3183], [2974, 2978], [2974, 3053], [2974, 3148], [2975, 2993], [2975, 3078], [2975, 3127], [2976, 2977], [2976, 3083], [2976, 3121], [2976, 3176], [2976, 3177], [2976, 3185], [2977, 3107], [2977, 3109], [2977, 3185], [2978, 3146], [2978, 3148], [2978, 3151], [2979, 2980], [2979, 2987], [2979, 3010], [2979, 3192], [2980, 2981], [2980, 3155], [2980, 3192], [2981, 3145], [2981, 3168], [2982, 2985], [2982, 3023], [2982, 3113], [2982, 3114], [2982, 3149], [2982, 3161], [2983, 2984], [2983, 3016], [2983, 3055], [2984, 3039], [2984, 3129], [2985, 2986], [2985, 3063], [2985, 3148], [2985, 3160], [2985, 3176], [2986, 2987], [2986, 3013], [2986, 3105], [2986, 3106], [2986, 3140], [2987, 3017], [2987, 3049], [2988, 2990], [2988, 3033], [2988, 3098], [2989, 3004], [2990, 2995], [2990, 3139], [2990, 3150], [2990, 3192], [2991, 3025], [2991, 3048], [2991, 3049], [2991, 3129], [2991, 3193], [2992, 2993], [2992, 3037], [2992, 3056], [2992, 3115], [2993, 3057], [2993, 3083], [2993, 3088], [2993, 3116], [2993, 3131], [2993, 3142], [2993, 3153], [2994, 2995], [2994, 3000], [2994, 3026], [2994, 3121], [2994, 3154], [2994, 3169], [2994, 3185], [2995, 3013], [2995, 3052], [2995, 3107], [2995, 3134], [2995, 3139], [2995, 3159], [2995, 3195], [2996, 3015], [2996, 3022], [2996, 3026], [2997, 2998], [2997, 3046], [2997, 3178], [2998, 3047], [2998, 3067], [2998, 3178], [2999, 3000], [2999, 3003], [2999, 3018], [2999, 3044], [2999, 3066], [2999, 3073], [2999, 3131], [2999, 3183], [3000, 3001], [3000, 3002], [3000, 3044], [3000, 3065], [3001, 3002], [3001, 3038], [3001, 3169], [3002, 3011], [3002, 3018], [3002, 3108], [3002, 3128], [3002, 3180], [3003, 3004], [3003, 3005], [3003, 3051], [3003, 3074], [3003, 3196], [3004, 3005], [3006, 3117], [3006, 3118], [3007, 3051], [3007, 3192], [3008, 3190], [3009, 3048], [3009, 3061], [3009, 3062], [3009, 3090], [3009, 3093], [3009, 3193], [3010, 3011], [3010, 3195], [3011, 3107], [3011, 3115], [3011, 3121], [3011, 3142], [3012, 3013], [3012, 3030], [3012, 3091], [3012, 3092], [3012, 3150], [3013, 3014], [3013, 3151], [3014, 3030], [3014, 3073], [3015, 3023], [3015, 3035], [3015, 3036], [3015, 3042], [3016, 3017], [3016, 3180], [3017, 3018], [3017, 3170], [3018, 3123], [3020, 3021], [3020, 3147], [3021, 3041], [3021, 3161], [3022, 3106], [3023, 3030], [3024, 3171], [3024, 3190], [3025, 3027], [3026, 3027], [3026, 3028], [3026, 3029], [3026, 3032], [3027, 3028], [3027, 3072], [3029, 3185], [3030, 3031], [3031, 3066], [3031, 3073], [3031, 3091], [3031, 3118], [3031, 3132], [3032, 3086], [3032, 3190], [3033, 3035], [3033, 3084], [3035, 3036], [3035, 3084], [3036, 3085], [3037, 3038], [3038, 3065], [3038, 3090], [3039, 3131], [3039, 3133], [3039, 3170], [3040, 3167], [3041, 3097], [3042, 3043], [3044, 3045], [3045, 3180], [3045, 3182], [3046, 3047], [3046, 3048], [3046, 3105], [3047, 3048], [3047, 3150], [3047, 3181], [3048, 3049], [3048, 3090], [3049, 3093], [3049, 3105], [3050, 3109], [3050, 3111], [3050, 3182], [3051, 3081], [3052, 3093], [3052, 3118], [3052, 3133], [3052, 3155], [3053, 3137], [3053, 3196], [3054, 3163], [3055, 3065], [3055, 3121], [3056, 3057], [3056, 3058], [3056, 3115], [3057, 3058], [3057, 3116], [3058, 3122], [3058, 3163], [3058, 3196], [3059, 3060], [3059, 3095], [3059, 3120], [3060, 3075], [3060, 3158], [3061, 3062], [3061, 3067], [3061, 3103], [3061, 3173], [3062, 3064], [3062, 3071], [3063, 3064], [3063, 3137], [3063, 3194], [3064, 3071], [3064, 3137], [3065, 3066], [3066, 3067], [3066, 3076], [3066, 3132], [3068, 3069], [3068, 3115], [3068, 3173], [3068, 3181], [3068, 3194], [3069, 3088], [3069, 3132], [3069, 3150], [3069, 3181], [3070, 3071], [3070, 3164], [3071, 3110], [3073, 3074], [3073, 3131], [3075, 3106], [3075, 3158], [3076, 3077], [3076, 3079], [3076, 3183], [3077, 3080], [3078, 3127], [3079, 3080], [3079, 3081], [3080, 3081], [3081, 3191], [3082, 3083], [3082, 3121], [3082, 3128], [3082, 3169], [3082, 3178], [3083, 3087], [3084, 3085], [3084, 3115], [3084, 3121], [3086, 3128], [3086, 3181], [3087, 3088], [3087, 3145], [3088, 3132], [3088, 3156], [3088, 3157], [3089, 3158], [3089, 3160], [3089, 3177], [3089, 3194], [3090, 3091], [3091, 3092], [3091, 3118], [3092, 3093], [3092, 3139], [3093, 3098], [3093, 3103], [3093, 3133], [3093, 3139], [3093, 3193], [3094, 3095], [3094, 3097], [3094, 3098], [3094, 3100], [3094, 3103], [3094, 3165], [3094, 3166], [3095, 3096], [3095, 3099], [3095, 3158], [3095, 3160], [3096, 3097], [3096, 3153], [3097, 3157], [3097, 3167], [3098, 3099], [3098, 3118], [3098, 3132], [3098, 3186], [3099, 3131], [3100, 3101], [3101, 3102], [3101, 3144], [3101, 3182], [3102, 3182], [3103, 3104], [3103, 3129], [3103, 3188], [3104, 3189], [3105, 3106], [3105, 3139], [3106, 3142], [3107, 3108], [3107, 3138], [3107, 3158], [3109, 3110], [3110, 3117], [3110, 3164], [3110, 3185], [3111, 3125], [3111, 3126], [3111, 3141], [3111, 3197], [3112, 3113], [3113, 3114], [3115, 3116], [3117, 3118], [3117, 3164], [3119, 3161], [3120, 3172], [3120, 3179], [3122, 3123], [3124, 3125], [3124, 3129], [3124, 3131], [3125, 3126], [3125, 3129], [3126, 3130], [3126, 3131], [3127, 3131], [3128, 3180], [3129, 3130], [3129, 3156], [3129, 3167], [3130, 3131], [3131, 3132], [3133, 3154], [3133, 3156], [3134, 3159], [3134, 3162], [3135, 3136], [3135, 3137], [3136, 3167], [3136, 3168], [3137, 3148], [3138, 3139], [3138, 3177], [3140, 3159], [3142, 3158], [3142, 3194], [3143, 3145], [3143, 3148], [3143, 3150], [3145, 3146], [3146, 3147], [3146, 3162], [3148, 3149], [3150, 3151], [3152, 3175], [3152, 3191], [3152, 3197], [3153, 3157], [3153, 3175], [3154, 3155], [3156, 3157], [3157, 3165], [3157, 3167], [3158, 3159], [3159, 3164], [3160, 3179], [3160, 3186], [3162, 3164], [3162, 3165], [3163, 3182], [3164, 3165], [3165, 3166], [3166, 3167], [3167, 3170], [3167, 3173], [3167, 3179], [3168, 3169], [3168, 3170], [3169, 3170], [3170, 3171], [3172, 3173], [3172, 3174], [3173, 3179], [3174, 3175], [3176, 3177], [3176, 3194], [3178, 3179], [3180, 3181], [3188, 3189], [3188, 3190], [3189, 3190], [3191, 3193], [3192, 3195], [3198, 3199], [3198, 3276], [3198, 3307], [3198, 3310], [3198, 3314], [3198, 3434], [3198, 3679], [3199, 3200], [3199, 3265], [3199, 3457], [3199, 3470], [3199, 3512], [3199, 3606], [3199, 3954], [3200, 3201], [3200, 3223], [3200, 3368], [3200, 3477], [3200, 3478], [3200, 3502], [3200, 3565], [3200, 3612], [3200, 3679], [3200, 3954], [3201, 3202], [3201, 3245], [3201, 3265], [3201, 3273], [3201, 3278], [3201, 3366], [3201, 3376], [3201, 3489], [3201, 3503], [3201, 3506], [3201, 3509], [3201, 3612], [3201, 3616], [3201, 3687], [3201, 3716], [3201, 3790], [3201, 3805], [3201, 3836], [3201, 3894], [3201, 3920], [3201, 3936], [3201, 3953], [3201, 4054], [3201, 4147], [3202, 3206], [3202, 3210], [3202, 3214], [3202, 3227], [3202, 3231], [3202, 3235], [3202, 3239], [3202, 3245], [3202, 3249], [3202, 3252], [3202, 3256], [3202, 3269], [3202, 3279], [3202, 3283], [3202, 3306], [3202, 3313], [3202, 3317], [3202, 3328], [3202, 3338], [3202, 3346], [3202, 3360], [3202, 3367], [3202, 3380], [3202, 3385], [3202, 3392], [3202, 3411], [3202, 3416], [3202, 3423], [3202, 3427], [3202, 3433], [3202, 3442], [3202, 3447], [3202, 3456], [3202, 3460], [3202, 3465], [3202, 3469], [3202, 3478], [3202, 3493], [3202, 3500], [3202, 3522], [3202, 3533], [3202, 3549], [3202, 3560], [3202, 3568], [3202, 3575], [3202, 3586], [3202, 3604], [3202, 3609], [3202, 3626], [3202, 3632], [3202, 3664], [3202, 3710], [3202, 3721], [3202, 3725], [3202, 3747], [3202, 3776], [3202, 3894], [3202, 3970], [3202, 3976], [3202, 3988], [3202, 4026], [3202, 4041], [3202, 4043], [3202, 4067], [3203, 3204], [3203, 3225], [3203, 3229], [3203, 3261], [3203, 3280], [3203, 3289], [3203, 3348], [3203, 3364], [3203, 3368], [3203, 3371], [3203, 3374], [3203, 3377], [3203, 3461], [3203, 3475], [3203, 3606], [3203, 3614], [3203, 3744], [3203, 3773], [3203, 3792], [3203, 3828], [3203, 3914], [3204, 3205], [3204, 3312], [3204, 3377], [3204, 3448], [3204, 3512], [3204, 3553], [3204, 3814], [3204, 3874], [3204, 3891], [3204, 4204], [3205, 3206], [3205, 3359], [3205, 3371], [3205, 3419], [3205, 3478], [3205, 3924], [3205, 4145], [3205, 4204], [3206, 3268], [3206, 3292], [3206, 3306], [3206, 3725], [3206, 3891], [3206, 3933], [3206, 3980], [3206, 4087], [3206, 4170], [3206, 4216], [3207, 3208], [3207, 3281], [3207, 3291], [3207, 3305], [3207, 3396], [3207, 3399], [3207, 3402], [3207, 3405], [3207, 3474], [3208, 3209], [3208, 3357], [3208, 3358], [3208, 3396], [3208, 3518], [3208, 3566], [3208, 3567], [3208, 3650], [3208, 3677], [3208, 3922], [3209, 3210], [3209, 3233], [3209, 3283], [3209, 3315], [3209, 3431], [3209, 3436], [3209, 3475], [3209, 3490], [3209, 3620], [3209, 3656], [3209, 3792], [3209, 3798], [3209, 3799], [3209, 3825], [3209, 3917], [3210, 3212], [3210, 3216], [3210, 3220], [3210, 3221], [3210, 3223], [3210, 3234], [3210, 3237], [3210, 3239], [3210, 3243], [3210, 3248], [3210, 3251], [3210, 3262], [3210, 3284], [3210, 3292], [3210, 3296], [3210, 3297], [3210, 3316], [3210, 3361], [3210, 3371], [3210, 3389], [3210, 3402], [3210, 3419], [3210, 3441], [3210, 3464], [3210, 3490], [3210, 3494], [3210, 3497], [3210, 3500], [3210, 3501], [3210, 3504], [3210, 3507], [3210, 3510], [3210, 3545], [3210, 3567], [3210, 3569], [3210, 3619], [3210, 3643], [3210, 3644], [3210, 3646], [3210, 3660], [3210, 3671], [3210, 3679], [3210, 3680], [3210, 3692], [3210, 3695], [3210, 3701], [3210, 3704], [3210, 3709], [3210, 3716], [3210, 3736], [3210, 3758], [3210, 3769], [3210, 3793], [3210, 3794], [3210, 3809], [3210, 3837], [3210, 3848], [3210, 3860], [3210, 3870], [3210, 3900], [3210, 3925], [3210, 3927], [3211, 3212], [3211, 3232], [3211, 3270], [3211, 3310], [3211, 3377], [3211, 3448], [3211, 3534], [3211, 3537], [3211, 3540], [3211, 3543], [3211, 3546], [3211, 3550], [3211, 3557], [3211, 3561], [3211, 3572], [3211, 3640], [3211, 3738], [3211, 3812], [3211, 3878], [3211, 3909], [3211, 3931], [3212, 3213], [3212, 3271], [3212, 3419], [3212, 3537], [3212, 3555], [3212, 3739], [3212, 3839], [3212, 3849], [3212, 3850], [3212, 4072], [3213, 3214], [3213, 3455], [3213, 3459], [3213, 3491], [3213, 3543], [3213, 3559], [3213, 3575], [3213, 3709], [3213, 4021], [3213, 4249], [3214, 3217], [3214, 3252], [3214, 3395], [3214, 3426], [3214, 3492], [3214, 3585], [3214, 3600], [3214, 3635], [3214, 3761], [3214, 3787], [3214, 3839], [3214, 3857], [3214, 3887], [3214, 4184], [3215, 3216], [3215, 3332], [3215, 3405], [3215, 3450], [3215, 3470], [3215, 3561], [3215, 3580], [3215, 3583], [3215, 3587], [3215, 3590], [3215, 3593], [3215, 3596], [3215, 3601], [3215, 3636], [3215, 3675], [3215, 3696], [3215, 3717], [3215, 3844], [3215, 3856], [3215, 3865], [3215, 3911], [3216, 3217], [3216, 3223], [3216, 3403], [3216, 3587], [3216, 3599], [3216, 3600], [3216, 3676], [3216, 3811], [3216, 3866], [3216, 4119], [3217, 3465], [3217, 3507], [3217, 3596], [3217, 3654], [3217, 3890], [3217, 4010], [3217, 4031], [3218, 3219], [3218, 3250], [3218, 3253], [3218, 3348], [3218, 3486], [3218, 3639], [3218, 3644], [3218, 3730], [3219, 3220], [3219, 3254], [3219, 3393], [3219, 3438], [3219, 3498], [3219, 3639], [3219, 3643], [3219, 3764], [3220, 3250], [3220, 3269], [3220, 3275], [3220, 3406], [3220, 3510], [3220, 3557], [3220, 3558], [3220, 3593], [3220, 3640], [3220, 3661], [3220, 3677], [3220, 3715], [3220, 3751], [3220, 3844], [3221, 3222], [3221, 3647], [3221, 3650], [3221, 3655], [3221, 3658], [3221, 3840], [3222, 3223], [3222, 3470], [3222, 3653], [3222, 3662], [3222, 3693], [3222, 3745], [3222, 3789], [3223, 3300], [3223, 3369], [3223, 3470], [3223, 3490], [3223, 3654], [3223, 3663], [3223, 3746], [3223, 3767], [3223, 3790], [3223, 3795], [3223, 3796], [3223, 3913], [3223, 3944], [3223, 4043], [3223, 4070], [3223, 4072], [3223, 4116], [3223, 4123], [3223, 4212], [3223, 4228], [3224, 3225], [3224, 3514], [3224, 3661], [3224, 3665], [3224, 3668], [3224, 3671], [3224, 3674], [3224, 3772], [3225, 3226], [3225, 3290], [3225, 3369], [3225, 3545], [3225, 3553], [3225, 3598], [3225, 3667], [3225, 3675], [3225, 3699], [3225, 3757], [3225, 3758], [3225, 3773], [3225, 3819], [3225, 3898], [3226, 3227], [3226, 3371], [3226, 3560], [3226, 3667], [3226, 3671], [3226, 3676], [3226, 3682], [3226, 3746], [3226, 4023], [3226, 4179], [3227, 3259], [3227, 3290], [3227, 3300], [3227, 3313], [3227, 3373], [3227, 3562], [3227, 3563], [3227, 3589], [3227, 3600], [3227, 3652], [3227, 3753], [3227, 3821], [3227, 3913], [3227, 4119], [3228, 3229], [3228, 3286], [3228, 3472], [3228, 3526], [3228, 3677], [3228, 3680], [3228, 3685], [3228, 3727], [3228, 3791], [3229, 3230], [3229, 3286], [3229, 3409], [3229, 3612], [3229, 3656], [3229, 3679], [3229, 3681], [3229, 3687], [3229, 3734], [3229, 3758], [3229, 3771], [3229, 3782], [3229, 3792], [3230, 3231], [3230, 3337], [3230, 3360], [3230, 3364], [3230, 3503], [3230, 3657], [3230, 3681], [3230, 3683], [3230, 3727], [3230, 3873], [3230, 4091], [3230, 4193], [3230, 4243], [3230, 4244], [3231, 3317], [3231, 3437], [3231, 3637], [3231, 3638], [3231, 3667], [3231, 3682], [3231, 3758], [3231, 3867], [3231, 3938], [3231, 3940], [3231, 3991], [3231, 4000], [3231, 4067], [3231, 4091], [3231, 4112], [3231, 4120], [3232, 3233], [3232, 3314], [3232, 3315], [3232, 3582], [3232, 3688], [3232, 3693], [3232, 3696], [3232, 3699], [3233, 3234], [3233, 3298], [3233, 3484], [3233, 3881], [3233, 3931], [3234, 3235], [3234, 3562], [3234, 3601], [3234, 3671], [3234, 3696], [3234, 3772], [3234, 3777], [3234, 3785], [3234, 3786], [3234, 3848], [3235, 3602], [3235, 3604], [3235, 3697], [3235, 3698], [3235, 3776], [3235, 3777], [3235, 3881], [3235, 4075], [3235, 4220], [3236, 3237], [3236, 3238], [3236, 3257], [3236, 3564], [3236, 3702], [3236, 3705], [3236, 3843], [3237, 3238], [3237, 3392], [3237, 3569], [3237, 3706], [3237, 3736], [3237, 3799], [3237, 3838], [3237, 3987], [3238, 3239], [3238, 3392], [3238, 3415], [3238, 3524], [3238, 3553], [3238, 3573], [3238, 3675], [3238, 3800], [3238, 3822], [3238, 3824], [3238, 3876], [3238, 3912], [3238, 4053], [3238, 4130], [3239, 3242], [3239, 3265], [3239, 3275], [3239, 3290], [3239, 3302], [3239, 3305], [3239, 3307], [3239, 3331], [3239, 3340], [3239, 3354], [3239, 3364], [3239, 3370], [3239, 3382], [3239, 3388], [3239, 3410], [3239, 3415], [3239, 3417], [3239, 3420], [3239, 3424], [3239, 3428], [3239, 3431], [3239, 3432], [3239, 3453], [3239, 3462], [3239, 3482], [3239, 3487], [3239, 3512], [3239, 3523], [3239, 3529], [3239, 3530], [3239, 3537], [3239, 3574], [3239, 3582], [3239, 3587], [3239, 3592], [3239, 3598], [3239, 3623], [3239, 3632], [3239, 3665], [3239, 3707], [3239, 3722], [3239, 3727], [3239, 3730], [3239, 3743], [3239, 3748], [3239, 3762], [3239, 3775], [3239, 3777], [3239, 3800], [3239, 3802], [3239, 3806], [3239, 3816], [3239, 3827], [3239, 3830], [3239, 3833], [3239, 3853], [3239, 3864], [3239, 3887], [3239, 3891], [3239, 3897], [3239, 3904], [3239, 3944], [3239, 3948], [3239, 3949], [3239, 3961], [3239, 3994], [3239, 4158], [3239, 4232], [3240, 3241], [3240, 3243], [3240, 3260], [3240, 3576], [3240, 3707], [3240, 3711], [3240, 3714], [3241, 3242], [3241, 3428], [3241, 3448], [3241, 3595], [3241, 3760], [3241, 3901], [3241, 4019], [3242, 3275], [3242, 3335], [3242, 3399], [3242, 3439], [3242, 3440], [3242, 3462], [3242, 3472], [3242, 3549], [3242, 3711], [3242, 3760], [3242, 3899], [3242, 3902], [3242, 3928], [3242, 3930], [3242, 3941], [3243, 3244], [3243, 3248], [3243, 3338], [3243, 3595], [3243, 3692], [3243, 3707], [3243, 3750], [3243, 3781], [3243, 3782], [3243, 3788], [3244, 3245], [3244, 3363], [3244, 3497], [3244, 3549], [3244, 3578], [3244, 3711], [3244, 3712], [3245, 3285], [3245, 3356], [3245, 3363], [3245, 3477], [3245, 3676], [3245, 3750], [3245, 3785], [3245, 3811], [3245, 3830], [3245, 3907], [3245, 3908], [3245, 3921], [3245, 4087], [3246, 3247], [3246, 3250], [3246, 3613], [3246, 3741], [3246, 3744], [3246, 3748], [3246, 3751], [3246, 3756], [3246, 3759], [3247, 3248], [3247, 3316], [3247, 3526], [3247, 3648], [3247, 3674], [3247, 3675], [3247, 3702], [3247, 3711], [3247, 3713], [3247, 3742], [3248, 3249], [3248, 3260], [3248, 3372], [3248, 3648], [3248, 3704], [3248, 3743], [3248, 3749], [3248, 3902], [3249, 3334], [3249, 3338], [3249, 3532], [3249, 3586], [3249, 3649], [3249, 3713], [3249, 3743], [3249, 3787], [3249, 4102], [3249, 4110], [3249, 4122], [3249, 4138], [3249, 4191], [3250, 3251], [3250, 3640], [3250, 3673], [3250, 3759], [3251, 3252], [3251, 3262], [3251, 3301], [3251, 3348], [3251, 3350], [3251, 3428], [3251, 3693], [3251, 3699], [3251, 3705], [3251, 3731], [3251, 3744], [3251, 3745], [3251, 3859], [3251, 3923], [3252, 3349], [3252, 3367], [3252, 3428], [3252, 3430], [3252, 3595], [3252, 3625], [3252, 3670], [3252, 3673], [3252, 3690], [3252, 3723], [3252, 3733], [3252, 3746], [3252, 3750], [3252, 3767], [3252, 3814], [3252, 3855], [3252, 3876], [3252, 3924], [3252, 3936], [3252, 4077], [3252, 4078], [3252, 4249], [3253, 3254], [3253, 3762], [3253, 3766], [3253, 3769], [3253, 3815], [3254, 3255], [3254, 3286], [3254, 3308], [3254, 3438], [3254, 3529], [3254, 3591], [3254, 3592], [3254, 3656], [3254, 3763], [3254, 3877], [3254, 3888], [3255, 3256], [3255, 3287], [3255, 3328], [3255, 3362], [3255, 3497], [3255, 3498], [3255, 3763], [3255, 3769], [3255, 3880], [3255, 3889], [3255, 4033], [3256, 3259], [3256, 3271], [3256, 3283], [3256, 3287], [3256, 3293], [3256, 3529], [3256, 3622], [3256, 3684], [3256, 3687], [3256, 3771], [3256, 3850], [3256, 3915], [3256, 3916], [3256, 4236], [3257, 3258], [3257, 3528], [3257, 3736], [3257, 3797], [3257, 3800], [3257, 3861], [3258, 3259], [3258, 3290], [3258, 3298], [3258, 3529], [3258, 3699], [3258, 3705], [3258, 3797], [3258, 3801], [3258, 3820], [3258, 3841], [3258, 3858], [3258, 3945], [3259, 3706], [3259, 3736], [3259, 3801], [3259, 3821], [3259, 3925], [3260, 3261], [3260, 3288], [3260, 3640], [3260, 3715], [3260, 3743], [3260, 3803], [3261, 3262], [3261, 3289], [3261, 3366], [3261, 3372], [3261, 3716], [3261, 3803], [3261, 3901], [3262, 3367], [3262, 3377], [3262, 3390], [3262, 3404], [3262, 3405], [3262, 3513], [3262, 3640], [3262, 3642], [3262, 3646], [3262, 3675], [3262, 3694], [3262, 3829], [3262, 3935], [3262, 3945], [3262, 4029], [3262, 4123], [3263, 3264], [3263, 3291], [3263, 3737], [3263, 3806], [3263, 3809], [3263, 3812], [3264, 3265], [3264, 3289], [3264, 3318], [3264, 3444], [3264, 3503], [3264, 3718], [3264, 3734], [3264, 3800], [3264, 4149], [3265, 3304], [3265, 3319], [3265, 3716], [3265, 3729], [3265, 3830], [3265, 3834], [3265, 3835], [3265, 3953], [3266, 3267], [3266, 3270], [3266, 3272], [3266, 3274], [3266, 3298], [3266, 3435], [3266, 3444], [3266, 3467], [3266, 3828], [3266, 3831], [3266, 3834], [3266, 3837], [3266, 3840], [3266, 3843], [3266, 3994], [3267, 3268], [3267, 3270], [3267, 3275], [3267, 3406], [3267, 3448], [3267, 3807], [3267, 3891], [3267, 4114], [3268, 3269], [3268, 3271], [3268, 3401], [3268, 3406], [3268, 3419], [3268, 3603], [3268, 3808], [3268, 3837], [3268, 3929], [3268, 3933], [3268, 3939], [3268, 4109], [3268, 4110], [3268, 4114], [3268, 4253], [3269, 3273], [3269, 3275], [3269, 3321], [3269, 3342], [3269, 3363], [3269, 3385], [3269, 3473], [3269, 3521], [3269, 3525], [3269, 3549], [3269, 3559], [3269, 3641], [3269, 3673], [3269, 3752], [3269, 3755], [3269, 3764], [3269, 3808], [3269, 3847], [3269, 3889], [3269, 3989], [3269, 4056], [3269, 4137], [3270, 3271], [3270, 3529], [3270, 3665], [3270, 3669], [3270, 3673], [3270, 3690], [3270, 3931], [3270, 4052], [3271, 3446], [3271, 3604], [3271, 3837], [3271, 4052], [3271, 4254], [3272, 3273], [3272, 3564], [3272, 3673], [3272, 3729], [3272, 3743], [3272, 3787], [3272, 3788], [3272, 3832], [3272, 3843], [3272, 3887], [3272, 3936], [3272, 4103], [3273, 3299], [3273, 3320], [3273, 3363], [3273, 3472], [3273, 3523], [3273, 3565], [3273, 3685], [3273, 3703], [3273, 3708], [3273, 3715], [3273, 3743], [3273, 3805], [3273, 4002], [3274, 3275], [3274, 3363], [3274, 3605], [3274, 3674], [3274, 3711], [3274, 3830], [3274, 4090], [3275, 3341], [3275, 3520], [3275, 3668], [3275, 3807], [3275, 3888], [3275, 4056], [3276, 3277], [3276, 3772], [3276, 3775], [3276, 3845], [3276, 3848], [3276, 3878], [3277, 3278], [3277, 3301], [3277, 3362], [3277, 3613], [3277, 3678], [3277, 3716], [3277, 3728], [3277, 3769], [3277, 3788], [3277, 3792], [3277, 3851], [3278, 3279], [3278, 3489], [3278, 3597], [3278, 3623], [3278, 3670], [3278, 3749], [3278, 3775], [3278, 3779], [3278, 3817], [3278, 3873], [3278, 4097], [3278, 4104], [3278, 4226], [3279, 3423], [3279, 3456], [3279, 3563], [3279, 3762], [3279, 3769], [3279, 3879], [3279, 4015], [3279, 4071], [3279, 4104], [3280, 3281], [3280, 3284], [3280, 3286], [3280, 3394], [3280, 3515], [3280, 3658], [3280, 3803], [3280, 3868], [3280, 3871], [3280, 3874], [3280, 3877], [3280, 3882], [3280, 3885], [3280, 3948], [3281, 3282], [3281, 3340], [3281, 3529], [3281, 3642], [3281, 3689], [3281, 3814], [3281, 3855], [3281, 3877], [3281, 3915], [3281, 3922], [3281, 4224], [3282, 3283], [3282, 3313], [3282, 3330], [3282, 3331], [3282, 3390], [3282, 3396], [3282, 3398], [3282, 3417], [3282, 3798], [3282, 3821], [3282, 3829], [3282, 3871], [3282, 3872], [3282, 3922], [3282, 4223], [3283, 3309], [3283, 3324], [3283, 3358], [3283, 3391], [3283, 3431], [3283, 3437], [3283, 3485], [3283, 3496], [3283, 3608], [3283, 3616], [3283, 3621], [3283, 3626], [3283, 3635], [3283, 3657], [3283, 3821], [3283, 3855], [3283, 3873], [3283, 3881], [3283, 4159], [3283, 4160], [3283, 4209], [3284, 3285], [3284, 3287], [3284, 3371], [3284, 3395], [3284, 3510], [3284, 3516], [3284, 3658], [3284, 3749], [3284, 3750], [3284, 3880], [3284, 3884], [3284, 3910], [3284, 3915], [3284, 3924], [3284, 3948], [3284, 3976], [3284, 4017], [3284, 4061], [3284, 4080], [3284, 4179], [3284, 4184], [3284, 4226], [3285, 3380], [3285, 3497], [3285, 3517], [3285, 3871], [3285, 3910], [3285, 4101], [3285, 4125], [3286, 3287], [3286, 3467], [3286, 3529], [3286, 3659], [3286, 3678], [3286, 3683], [3286, 3686], [3286, 3783], [3286, 3826], [3286, 3827], [3286, 3882], [3287, 3426], [3287, 3608], [3287, 3609], [3287, 3659], [3287, 3680], [3287, 3681], [3287, 3683], [3287, 3732], [3287, 3785], [3287, 3817], [3287, 3884], [3287, 3997], [3287, 4084], [3288, 3289], [3288, 3381], [3288, 3704], [3288, 3833], [3288, 3888], [3289, 3290], [3289, 3295], [3289, 3307], [3289, 3365], [3289, 3373], [3289, 3390], [3289, 3615], [3289, 3845], [3289, 3869], [3290, 3295], [3290, 3299], [3290, 3332], [3290, 3388], [3290, 3513], [3290, 3589], [3290, 3703], [3290, 3748], [3290, 3820], [3290, 3846], [3290, 3856], [3290, 4007], [3291, 3292], [3291, 3699], [3291, 3891], [3291, 3895], [3291, 3898], [3291, 3901], [3292, 3293], [3292, 3402], [3292, 3406], [3292, 3746], [3292, 3809], [3292, 3813], [3292, 3854], [3292, 3891], [3292, 4167], [3293, 3497], [3293, 3571], [3293, 3747], [3293, 3842], [3293, 3893], [3293, 3895], [3293, 3903], [3293, 4167], [3293, 4209], [3294, 3295], [3294, 3296], [3294, 3407], [3294, 3409], [3294, 3734], [3294, 3815], [3294, 3904], [3294, 3909], [3294, 3911], [3295, 3325], [3295, 3387], [3295, 3399], [3295, 3410], [3295, 3448], [3295, 3651], [3295, 3652], [3295, 3735], [3295, 3822], [3295, 3905], [3295, 3909], [3295, 3912], [3296, 3297], [3296, 3368], [3296, 3493], [3296, 3555], [3296, 3652], [3296, 3744], [3296, 3751], [3296, 3870], [3296, 3904], [3296, 3907], [3297, 3370], [3297, 3407], [3297, 3508], [3297, 3567], [3297, 3618], [3297, 3745], [3297, 3747], [3297, 4007], [3298, 3299], [3298, 3301], [3298, 3412], [3298, 3483], [3298, 3601], [3298, 3685], [3298, 3705], [3298, 3802], [3298, 3804], [3298, 3902], [3298, 3914], [3298, 3917], [3298, 3919], [3298, 3922], [3298, 3925], [3298, 3928], [3298, 3931], [3298, 3934], [3299, 3300], [3299, 3320], [3299, 3342], [3299, 3453], [3299, 3470], [3299, 3601], [3299, 3834], [3299, 4046], [3300, 3710], [3300, 3768], [3300, 3790], [3300, 3925], [3300, 4046], [3300, 4122], [3301, 3361], [3301, 3576], [3301, 3670], [3301, 3699], [3301, 3782], [3301, 3792], [3301, 3810], [3301, 3829], [3301, 3922], [3302, 3303], [3302, 3413], [3302, 3519], [3302, 3688], [3302, 3718], [3302, 3741], [3302, 3949], [3303, 3304], [3303, 3329], [3303, 3387], [3303, 3408], [3303, 3435], [3303, 3444], [3303, 3461], [3303, 3462], [3303, 4133], [3304, 3305], [3304, 3319], [3304, 3829], [3304, 3919], [3304, 3920], [3305, 3306], [3305, 3358], [3305, 3400], [3305, 3401], [3305, 3402], [3305, 3574], [3305, 3829], [3305, 3891], [3305, 3968], [3305, 4020], [3305, 4166], [3305, 4206], [3305, 4224], [3306, 3402], [3306, 3547], [3306, 3575], [3306, 3920], [3306, 3921], [3306, 3932], [3306, 3968], [3306, 4205], [3307, 3308], [3307, 3311], [3307, 3360], [3307, 3410], [3307, 3615], [3307, 3679], [3307, 3775], [3307, 3888], [3307, 3954], [3307, 4151], [3307, 4189], [3308, 3309], [3308, 3311], [3308, 3314], [3308, 3368], [3308, 3422], [3308, 3582], [3308, 3607], [3308, 3643], [3308, 3656], [3308, 3869], [3308, 3870], [3308, 3884], [3308, 4097], [3309, 3315], [3309, 3317], [3309, 3330], [3309, 3335], [3309, 3409], [3309, 3410], [3309, 3437], [3309, 3440], [3309, 3582], [3309, 3633], [3309, 3771], [3309, 3823], [3309, 3909], [3309, 4225], [3310, 3311], [3310, 3314], [3310, 3386], [3310, 3512], [3310, 3528], [3310, 3529], [3310, 3797], [3310, 3799], [3310, 3850], [3310, 3878], [3310, 3987], [3311, 3312], [3311, 3390], [3311, 3391], [3311, 3537], [3311, 3631], [3311, 3684], [3311, 3850], [3311, 3879], [3311, 3986], [3311, 4005], [3311, 4029], [3311, 4105], [3311, 4206], [3312, 3313], [3312, 3368], [3312, 3382], [3312, 3386], [3312, 3387], [3312, 3411], [3312, 3439], [3312, 3462], [3312, 3531], [3312, 3554], [3312, 3556], [3312, 3652], [3312, 3893], [3312, 4176], [3312, 4203], [3313, 3334], [3313, 3388], [3313, 3395], [3313, 3398], [3313, 3445], [3313, 3533], [3313, 3536], [3313, 3542], [3313, 3554], [3313, 3584], [3313, 3720], [3313, 3814], [3313, 3842], [3313, 3847], [3313, 3860], [3313, 3867], [3313, 3879], [3313, 3910], [3313, 3958], [3313, 4023], [3313, 4172], [3314, 3315], [3314, 3606], [3314, 3883], [3315, 3316], [3315, 3434], [3315, 3436], [3316, 3317], [3316, 3450], [3316, 3482], [3316, 3546], [3316, 3577], [3316, 3611], [3316, 3653], [3316, 3696], [3316, 3702], [3316, 3738], [3316, 3756], [3316, 3758], [3316, 3883], [3317, 3451], [3317, 3473], [3317, 3478], [3317, 3482], [3317, 3516], [3317, 3539], [3317, 3542], [3317, 3547], [3317, 3552], [3317, 3579], [3317, 3612], [3317, 3654], [3317, 3697], [3317, 3713], [3317, 3771], [3317, 3824], [3317, 3884], [3317, 3913], [3317, 4037], [3317, 4038], [3317, 4084], [3318, 3319], [3318, 3322], [3318, 3325], [3318, 3329], [3318, 3330], [3318, 3332], [3318, 3335], [3318, 3438], [3318, 3639], [3318, 3646], [3318, 3647], [3318, 3820], [3318, 3868], [3318, 3961], [3319, 3320], [3319, 3325], [3319, 3344], [3319, 3452], [3319, 3505], [3319, 3506], [3319, 3574], [3319, 3902], [3319, 4152], [3320, 3321], [3320, 3332], [3320, 3370], [3320, 3902], [3320, 4210], [3320, 4212], [3321, 3342], [3321, 3401], [3321, 3433], [3321, 3531], [3321, 3554], [3321, 3574], [3321, 3587], [3321, 3593], [3321, 3645], [3321, 3824], [3321, 4228], [3321, 4256], [3321, 4260], [3322, 3323], [3322, 3335], [3322, 3431], [3322, 3479], [3322, 3530], [3322, 3656], [3322, 3657], [3322, 4231], [3323, 3324], [3323, 3344], [3323, 3347], [3323, 3431], [3323, 3432], [3323, 3475], [3323, 3642], [3323, 4230], [3324, 3345], [3324, 3364], [3324, 3433], [3324, 3475], [3324, 3476], [3324, 3544], [3324, 3574], [3324, 3607], [3324, 3657], [3324, 3916], [3324, 4034], [3324, 4207], [3324, 4230], [3324, 4259], [3325, 3326], [3325, 3399], [3325, 3512], [3325, 3564], [3325, 3565], [3325, 3573], [3325, 3630], [3325, 3648], [3325, 3649], [3325, 3694], [3325, 3742], [3325, 3743], [3326, 3327], [3326, 3400], [3326, 3415], [3326, 3624], [3326, 3631], [3326, 3649], [3326, 3905], [3326, 3961], [3326, 4102], [3326, 4152], [3326, 4195], [3326, 4206], [3327, 3328], [3327, 3415], [3327, 3420], [3327, 3565], [3327, 3570], [3327, 3654], [3327, 3824], [3327, 3889], [3327, 3890], [3327, 3894], [3327, 3990], [3327, 4002], [3327, 4063], [3327, 4217], [3328, 3517], [3328, 3575], [3328, 3592], [3328, 3609], [3328, 3695], [3328, 3767], [3328, 3942], [3328, 3990], [3328, 4033], [3329, 3330], [3329, 3436], [3329, 3475], [3329, 3576], [3329, 3577], [3329, 3630], [3329, 3688], [3329, 3691], [3329, 3819], [3329, 3829], [3329, 4259], [3330, 3331], [3330, 3344], [3330, 3387], [3330, 3466], [3330, 3512], [3330, 3542], [3330, 3556], [3330, 3909], [3330, 4154], [3331, 3354], [3331, 3427], [3331, 3437], [3331, 3467], [3331, 3689], [3331, 3701], [3331, 3800], [3331, 3861], [3331, 3867], [3331, 3955], [3331, 4202], [3332, 3333], [3332, 3334], [3332, 3530], [3332, 3542], [3332, 3590], [3332, 3703], [3332, 3713], [3332, 4013], [3332, 4119], [3333, 3334], [3333, 3501], [3333, 3531], [3333, 3583], [3333, 3649], [3333, 3704], [3334, 3445], [3334, 3532], [3334, 3601], [3334, 4115], [3334, 4176], [3335, 3336], [3335, 3613], [3335, 3627], [3335, 3713], [3335, 3816], [3335, 3823], [3335, 4012], [3336, 3337], [3336, 3423], [3336, 3549], [3336, 3619], [3336, 3646], [3336, 3657], [3336, 4012], [3336, 4191], [3337, 3338], [3337, 3430], [3337, 3503], [3337, 3723], [3337, 3724], [3337, 4192], [3338, 3352], [3338, 3445], [3338, 3560], [3338, 3707], [3338, 3779], [3338, 4040], [3338, 4057], [3338, 4079], [3339, 3340], [3339, 3343], [3339, 3347], [3339, 3350], [3339, 3443], [3339, 3545], [3339, 3852], [3339, 3859], [3340, 3341], [3340, 3398], [3340, 3399], [3340, 3432], [3340, 3545], [3340, 3625], [3340, 3664], [3340, 3814], [3340, 3853], [3340, 4199], [3340, 4222], [3341, 3342], [3341, 3399], [3341, 3413], [3341, 3661], [3341, 3846], [3342, 3383], [3342, 3417], [3342, 3455], [3342, 3550], [3342, 3562], [3342, 3588], [3342, 3661], [3342, 3663], [3342, 3664], [3342, 3665], [3342, 3931], [3342, 4045], [3342, 4146], [3343, 3344], [3343, 3393], [3343, 3398], [3343, 3438], [3343, 3490], [3343, 3798], [3343, 3860], [3343, 3862], [3343, 3877], [3344, 3345], [3344, 3347], [3344, 3354], [3344, 3481], [3344, 3490], [3344, 3512], [3344, 3643], [3344, 3965], [3344, 4212], [3345, 3346], [3345, 3359], [3345, 3478], [3345, 3491], [3345, 3499], [3345, 3506], [3345, 3509], [3345, 3544], [3345, 3556], [3345, 3579], [3345, 3965], [3346, 3354], [3346, 3361], [3346, 3427], [3346, 3579], [3346, 3670], [3346, 3710], [3346, 3964], [3346, 3969], [3346, 4207], [3347, 3348], [3347, 3350], [3347, 3413], [3347, 3479], [3347, 3544], [3347, 4222], [3348, 3349], [3348, 3642], [3348, 3643], [3348, 3845], [3348, 3859], [3348, 3901], [3349, 3364], [3349, 3533], [3349, 3730], [3349, 3774], [3349, 3814], [3349, 4085], [3349, 4106], [3349, 4198], [3349, 4222], [3349, 4229], [3350, 3351], [3350, 3490], [3350, 3613], [3350, 3625], [3350, 3630], [3350, 3819], [3351, 3352], [3351, 3491], [3351, 3544], [3351, 3545], [3351, 3619], [3351, 3625], [3352, 3478], [3352, 3501], [3352, 3594], [3352, 3619], [3352, 3624], [3352, 3630], [3352, 3649], [3352, 3691], [3352, 3767], [3352, 4040], [3353, 3354], [3353, 3357], [3353, 3361], [3353, 3452], [3353, 3580], [3353, 3861], [3354, 3355], [3354, 3361], [3354, 3453], [3354, 3481], [3354, 3964], [3354, 4168], [3354, 4169], [3354, 4188], [3354, 4255], [3355, 3356], [3355, 3580], [3355, 3587], [3355, 3638], [3355, 3778], [3355, 3790], [3355, 3811], [3355, 3867], [3355, 3984], [3355, 3985], [3355, 4150], [3355, 4180], [3355, 4212], [3355, 4238], [3355, 4255], [3356, 3607], [3356, 3632], [3356, 3712], [3356, 3778], [3356, 3818], [3356, 3830], [3356, 3953], [3356, 3973], [3356, 3991], [3356, 4050], [3356, 4088], [3356, 4090], [3356, 4091], [3356, 4095], [3356, 4097], [3356, 4100], [3356, 4125], [3356, 4171], [3356, 4207], [3356, 4244], [3357, 3358], [3357, 3431], [3357, 3448], [3357, 3487], [3357, 3519], [3357, 3888], [3357, 3928], [3357, 4168], [3358, 3359], [3358, 3361], [3358, 3419], [3358, 3440], [3358, 3468], [3358, 3485], [3358, 3523], [3358, 3535], [3358, 3568], [3358, 3889], [3358, 3892], [3358, 3918], [3358, 3969], [3358, 4021], [3358, 4073], [3358, 4168], [3358, 4209], [3359, 3360], [3359, 3468], [3359, 3509], [3359, 3920], [3359, 4143], [3360, 3373], [3360, 3411], [3360, 3679], [3360, 3776], [3360, 3889], [3360, 4189], [3360, 4192], [3360, 4193], [3361, 3362], [3361, 3375], [3361, 3484], [3361, 3577], [3361, 3701], [3361, 3709], [3361, 3811], [3361, 3829], [3362, 3363], [3362, 3627], [3362, 3687], [3362, 3889], [3362, 3974], [3363, 3627], [3363, 3695], [3363, 3788], [3363, 3837], [3363, 4090], [3364, 3365], [3364, 3366], [3364, 3371], [3364, 3378], [3364, 3421], [3364, 3430], [3364, 3462], [3364, 3509], [3364, 3607], [3364, 3667], [3364, 3873], [3364, 3948], [3364, 4142], [3364, 4171], [3364, 4204], [3365, 3366], [3365, 3373], [3365, 3379], [3365, 3589], [3365, 3833], [3365, 3905], [3365, 3910], [3365, 4017], [3365, 4149], [3365, 4176], [3365, 4181], [3365, 4189], [3365, 4196], [3365, 4198], [3366, 3367], [3366, 3372], [3366, 3743], [3366, 3936], [3366, 4096], [3366, 4121], [3367, 3378], [3367, 3379], [3367, 3401], [3367, 3404], [3367, 3641], [3367, 3818], [3367, 4028], [3367, 4041], [3367, 4058], [3367, 4109], [3367, 4183], [3367, 4207], [3367, 4229], [3367, 4247], [3368, 3369], [3368, 3374], [3368, 3376], [3368, 3421], [3368, 3438], [3368, 3502], [3368, 3556], [3368, 3643], [3368, 3871], [3368, 3907], [3369, 3370], [3369, 3554], [3369, 3579], [3369, 3581], [3369, 3582], [3369, 3701], [3369, 3862], [3369, 3866], [3369, 3908], [3369, 3913], [3369, 4202], [3369, 4213], [3370, 3397], [3370, 3407], [3370, 3487], [3370, 3554], [3370, 3747], [3370, 3836], [3370, 3895], [3370, 3898], [3370, 3904], [3370, 3912], [3370, 4208], [3370, 4238], [3371, 3372], [3371, 3373], [3371, 3476], [3371, 3477], [3371, 3681], [3371, 4087], [3372, 3373], [3372, 3404], [3372, 3749], [3373, 3418], [3373, 3502], [3373, 3503], [3373, 3652], [3373, 3704], [3373, 4192], [3374, 3375], [3374, 3509], [3374, 3518], [3374, 3716], [3374, 3927], [3375, 3376], [3375, 3426], [3375, 3809], [3375, 3828], [3375, 3892], [3376, 3427], [3376, 3503], [3376, 3828], [3376, 3893], [3376, 4097], [3376, 4255], [3377, 3378], [3377, 3390], [3377, 3544], [3377, 3546], [3377, 3552], [3377, 3556], [3377, 3690], [3377, 3770], [3377, 3771], [3377, 3915], [3377, 3927], [3378, 3379], [3378, 3522], [3378, 3537], [3378, 3547], [3378, 4051], [3378, 4204], [3378, 4205], [3378, 4218], [3378, 4242], [3379, 3380], [3379, 3390], [3379, 3910], [3379, 3924], [3379, 3971], [3379, 4098], [3379, 4164], [3379, 4223], [3379, 4262], [3380, 3382], [3380, 3389], [3380, 3522], [3380, 3586], [3380, 3924], [3380, 3966], [3380, 4000], [3380, 4018], [3380, 4173], [3381, 3382], [3381, 3386], [3381, 3389], [3381, 3393], [3381, 3458], [3381, 3466], [3381, 3831], [3382, 3383], [3382, 3384], [3382, 3389], [3382, 3424], [3382, 3439], [3382, 3833], [3382, 3871], [3382, 3874], [3382, 3959], [3382, 3966], [3382, 4127], [3383, 3384], [3383, 3516], [3383, 3652], [3383, 3748], [3383, 3754], [3383, 3824], [3383, 4261], [3384, 3385], [3384, 3411], [3384, 3455], [3384, 3458], [3384, 3459], [3384, 3608], [3384, 3623], [3384, 3631], [3384, 3755], [3384, 3996], [3384, 3997], [3384, 4081], [3384, 4144], [3384, 4163], [3384, 4195], [3384, 4251], [3385, 3510], [3385, 3539], [3385, 3740], [3385, 3779], [3385, 3805], [3385, 3932], [3385, 3933], [3385, 3957], [3385, 3976], [3385, 4006], [3385, 4032], [3385, 4047], [3385, 4155], [3385, 4183], [3385, 4260], [3386, 3387], [3386, 3461], [3386, 3494], [3386, 3527], [3386, 3534], [3386, 3651], [3386, 3860], [3386, 3895], [3387, 3388], [3387, 3410], [3388, 3394], [3388, 3444], [3388, 3719], [3388, 3765], [3388, 3831], [3388, 3860], [3388, 3868], [3388, 3958], [3389, 3390], [3389, 3459], [3389, 3541], [3389, 3704], [3389, 3842], [3389, 3923], [3389, 3927], [3389, 4005], [3390, 3391], [3390, 3417], [3390, 3797], [3390, 3800], [3390, 3801], [3390, 3869], [3390, 3923], [3390, 4261], [3391, 3392], [3391, 3419], [3391, 3459], [3391, 3563], [3391, 3615], [3391, 3621], [3391, 3652], [3391, 3799], [3391, 3822], [3391, 3870], [3391, 4068], [3391, 4109], [3391, 4163], [3391, 4164], [3391, 4249], [3392, 3416], [3392, 3447], [3392, 3676], [3392, 3986], [3392, 4053], [3392, 4054], [3392, 4065], [3392, 4068], [3393, 3394], [3393, 3543], [3393, 3661], [3393, 3877], [3394, 3395], [3394, 3486], [3394, 3856], [3394, 3887], [3394, 4177], [3395, 3567], [3395, 3600], [3395, 3601], [3395, 3810], [3395, 3814], [3395, 3922], [3395, 4177], [3396, 3397], [3396, 3438], [3396, 3536], [3396, 3895], [3397, 3398], [3397, 3399], [3397, 3536], [3397, 3566], [3397, 3574], [3398, 3420], [3398, 3626], [3398, 3965], [3398, 4034], [3398, 4187], [3398, 4202], [3398, 4261], [3399, 3400], [3399, 3405], [3399, 3470], [3399, 3566], [3399, 3662], [3399, 3663], [3399, 3898], [3399, 3928], [3400, 3401], [3400, 3663], [3400, 3905], [3400, 3929], [3400, 3930], [3400, 3944], [3400, 4181], [3400, 4187], [3400, 4199], [3400, 4215], [3401, 3403], [3401, 3405], [3401, 3445], [3401, 3446], [3401, 3568], [3401, 3587], [3401, 4042], [3401, 4060], [3401, 4117], [3401, 4247], [3402, 3403], [3402, 3504], [3402, 3546], [3402, 3663], [3402, 3915], [3402, 3919], [3402, 3931], [3403, 3404], [3403, 3405], [3403, 3562], [3403, 3663], [3404, 3500], [3404, 3563], [3404, 3676], [3404, 3916], [3404, 3921], [3404, 4040], [3405, 3406], [3405, 3536], [3405, 3567], [3405, 3593], [3405, 3595], [3405, 3735], [3405, 3781], [3405, 3814], [3405, 3846], [3406, 3650], [3406, 3672], [3406, 3813], [3406, 3840], [3406, 3844], [3406, 3928], [3406, 3945], [3407, 3408], [3407, 3486], [3407, 3534], [3407, 3617], [3408, 3409], [3408, 3440], [3408, 3450], [3408, 3479], [3408, 3508], [3408, 3519], [3408, 3527], [3408, 3577], [3408, 3580], [3408, 3617], [3408, 4238], [3409, 3410], [3409, 3479], [3409, 3612], [3409, 3616], [3409, 3826], [3409, 3830], [3409, 3863], [3409, 3907], [3409, 4244], [3410, 3411], [3410, 3434], [3410, 3451], [3410, 3494], [3410, 3496], [3410, 3864], [3410, 3977], [3410, 4246], [3411, 3442], [3411, 3494], [3411, 3652], [3411, 3755], [3411, 3907], [3411, 3977], [3411, 4145], [3411, 4191], [3411, 4240], [3411, 4245], [3412, 3413], [3412, 3417], [3412, 3583], [3412, 3620], [3412, 3661], [3412, 3677], [3412, 3797], [3412, 3798], [3413, 3414], [3413, 3417], [3413, 3845], [3413, 3877], [3413, 3888], [3413, 4221], [3414, 3415], [3414, 3431], [3414, 3620], [3414, 3621], [3414, 3822], [3414, 3851], [3415, 3416], [3415, 3453], [3415, 3564], [3415, 3569], [3415, 3767], [3415, 3851], [3415, 4014], [3415, 4016], [3415, 4020], [3415, 4069], [3415, 4103], [3416, 3569], [3416, 3584], [3416, 3621], [3416, 3622], [3416, 3637], [3416, 3710], [3416, 4014], [3416, 4066], [3417, 3418], [3417, 3512], [3417, 3531], [3417, 3544], [3417, 3584], [3417, 3621], [3417, 3802], [3417, 3880], [3417, 3889], [3417, 3935], [3417, 3971], [3417, 3974], [3417, 4017], [3417, 4146], [3417, 4221], [3418, 3419], [3418, 3562], [3418, 3584], [3418, 3622], [3418, 3706], [3418, 3722], [3418, 3845], [3418, 3848], [3418, 3849], [3418, 3987], [3418, 4198], [3418, 4219], [3419, 3448], [3419, 3524], [3419, 3525], [3419, 3562], [3419, 3595], [3419, 3650], [3419, 3652], [3419, 3900], [3419, 3924], [3419, 3939], [3419, 3970], [3419, 4018], [3419, 4021], [3419, 4116], [3419, 4146], [3419, 4158], [3419, 4262], [3420, 3421], [3420, 3438], [3420, 3497], [3420, 3523], [3420, 3579], [3420, 3628], [3420, 3712], [3420, 3720], [3420, 3752], [3420, 3763], [3420, 3764], [3420, 3824], [3420, 3893], [3420, 3961], [3420, 3967], [3420, 4125], [3420, 4150], [3421, 3422], [3421, 3493], [3421, 3509], [3421, 3533], [3421, 4125], [3421, 4185], [3421, 4203], [3421, 4213], [3421, 4242], [3422, 3423], [3422, 3533], [3422, 3657], [3422, 3763], [3422, 3910], [3422, 4009], [3422, 4050], [3422, 4189], [3422, 4225], [3422, 4226], [3423, 3493], [3423, 3816], [3423, 3870], [3423, 3910], [3423, 3946], [3423, 4164], [3423, 4184], [3423, 4196], [3424, 3425], [3424, 3440], [3424, 3466], [3424, 3468], [3424, 3516], [3424, 3520], [3424, 3522], [3424, 3598], [3424, 3927], [3424, 3981], [3424, 4004], [3424, 4085], [3424, 4154], [3424, 4178], [3425, 3426], [3425, 3467], [3425, 3468], [3425, 3538], [3425, 3683], [3425, 3778], [3425, 3857], [3425, 3955], [3425, 3959], [3425, 3994], [3425, 4073], [3425, 4169], [3426, 3427], [3426, 3467], [3426, 3690], [3426, 3700], [3426, 3785], [3426, 3837], [3426, 3839], [3426, 3842], [3426, 3892], [3426, 3926], [3426, 3927], [3426, 4023], [3426, 4214], [3427, 3447], [3427, 3556], [3427, 3690], [3427, 3701], [3427, 3893], [3427, 3955], [3427, 4094], [3427, 4101], [3427, 4213], [3427, 4254], [3428, 3429], [3428, 3689], [3428, 3766], [3428, 3874], [3428, 3875], [3428, 3887], [3428, 3898], [3428, 4077], [3429, 3430], [3429, 3741], [3429, 3744], [3429, 3886], [3429, 3898], [3429, 3904], [3430, 3492], [3430, 3493], [3430, 3744], [3430, 3746], [3430, 3748], [3430, 4070], [3431, 3529], [3431, 3820], [3431, 3822], [3431, 3826], [3431, 4159], [3432, 3433], [3432, 3443], [3432, 3501], [3432, 3530], [3432, 3624], [3432, 3642], [3432, 4123], [3432, 4180], [3432, 4181], [3432, 4212], [3432, 4227], [3433, 3465], [3433, 3501], [3433, 3664], [3433, 3916], [3433, 4076], [3433, 4227], [3433, 4260], [3434, 3435], [3434, 3450], [3434, 3494], [3434, 3514], [3434, 3917], [3435, 3436], [3435, 3835], [3435, 3896], [3435, 3899], [3435, 3917], [3435, 4246], [3436, 3437], [3436, 3613], [3436, 3758], [3436, 3819], [3436, 3861], [3436, 3863], [3437, 3623], [3437, 3634], [3437, 3991], [3437, 4073], [3437, 4097], [3437, 4246], [3437, 4259], [3438, 3439], [3438, 3497], [3438, 3518], [3438, 3540], [3438, 3565], [3438, 3577], [3438, 3590], [3438, 3627], [3438, 3702], [3438, 3711], [3438, 3726], [3438, 3751], [3438, 3871], [3438, 3895], [3439, 3440], [3439, 3448], [3439, 3540], [3439, 3754], [3439, 3755], [3439, 3823], [3439, 3847], [3439, 3967], [3439, 4018], [3440, 3441], [3440, 3520], [3440, 3534], [3440, 3548], [3440, 3577], [3440, 3581], [3440, 3643], [3440, 3982], [3440, 4073], [3441, 3442], [3441, 3494], [3441, 3514], [3441, 3516], [3441, 3534], [3441, 3551], [3441, 3594], [3441, 3617], [3441, 3671], [3441, 3754], [3441, 3864], [3441, 3918], [3442, 3554], [3442, 3604], [3442, 3836], [3442, 3864], [3442, 3908], [3442, 3937], [3442, 3982], [3442, 4248], [3443, 3444], [3443, 3501], [3443, 3526], [3443, 3630], [3444, 3445], [3444, 3580], [3444, 3707], [3444, 3781], [3444, 4180], [3445, 3446], [3445, 3462], [3445, 3501], [3445, 3503], [3445, 3666], [3445, 3777], [3445, 3781], [3445, 3810], [3445, 3811], [3445, 3829], [3445, 3837], [3445, 4023], [3445, 4076], [3445, 4093], [3445, 4180], [3445, 4187], [3446, 3447], [3446, 3547], [3446, 3697], [3446, 3916], [3446, 4252], [3447, 3503], [3447, 3736], [3447, 3800], [3447, 3839], [3447, 3906], [3447, 4001], [3447, 4039], [3447, 4098], [3448, 3449], [3448, 3650], [3448, 3822], [3448, 3845], [3448, 3846], [3448, 3874], [3448, 4158], [3449, 3450], [3449, 3519], [3449, 3524], [3449, 3737], [3449, 3738], [3449, 3807], [3449, 3832], [3449, 4074], [3450, 3451], [3450, 3527], [3450, 3611], [3450, 3899], [3450, 3900], [3451, 3587], [3451, 3612], [3451, 3894], [3451, 3912], [3451, 4016], [3451, 4074], [3451, 4112], [3451, 4238], [3451, 4256], [3452, 3453], [3452, 3457], [3452, 3543], [3452, 3564], [3452, 3709], [3452, 3728], [3453, 3454], [3453, 3455], [3453, 3480], [3453, 3483], [3453, 3489], [3453, 3709], [3453, 3710], [3453, 4152], [3453, 4153], [3453, 4233], [3454, 3455], [3454, 3484], [3454, 3582], [3454, 3608], [3454, 3670], [3454, 3767], [3454, 3785], [3454, 4088], [3455, 3456], [3455, 3485], [3455, 3537], [3455, 3543], [3455, 3626], [3455, 3857], [3455, 3983], [3455, 4024], [3455, 4030], [3455, 4082], [3455, 4169], [3455, 4218], [3455, 4240], [3456, 3485], [3456, 3489], [3456, 3562], [3456, 3568], [3456, 3644], [3456, 3730], [3456, 3785], [3456, 3997], [3456, 4031], [3456, 4035], [3456, 4250], [3456, 4254], [3457, 3458], [3457, 3471], [3457, 3804], [3457, 3825], [3457, 3917], [3458, 3459], [3458, 3494], [3458, 3510], [3458, 3512], [3458, 3524], [3458, 3543], [3458, 3613], [3458, 3633], [3458, 3678], [3458, 3822], [3458, 3823], [3458, 3825], [3458, 3897], [3459, 3460], [3459, 3478], [3459, 3619], [3459, 3740], [3459, 4005], [3459, 4084], [3460, 3469], [3460, 3721], [3460, 3794], [3460, 3842], [3460, 3897], [3460, 3998], [3460, 4003], [3460, 4081], [3460, 4084], [3460, 4109], [3461, 3462], [3461, 3546], [3461, 3610], [3461, 3611], [3461, 3781], [3461, 3919], [3462, 3463], [3462, 3508], [3462, 3547], [3462, 3612], [3462, 3666], [3462, 3691], [3462, 3854], [3462, 3896], [3462, 3920], [3462, 3978], [3462, 3979], [3462, 4029], [3462, 4133], [3462, 4176], [3463, 3464], [3463, 3490], [3463, 3506], [3463, 3592], [3463, 3607], [3463, 3610], [3463, 3612], [3463, 3627], [3463, 3629], [3463, 3630], [3463, 3653], [3463, 3656], [3463, 3712], [3463, 4256], [3464, 3465], [3464, 3501], [3464, 3526], [3464, 3530], [3464, 3591], [3464, 3656], [3464, 3680], [3464, 3884], [3464, 3903], [3464, 4093], [3465, 3530], [3465, 3629], [3465, 3657], [3465, 3721], [3465, 4008], [3465, 4009], [3465, 4032], [3465, 4119], [3466, 3467], [3466, 3576], [3466, 3633], [3466, 3756], [3466, 3922], [3466, 3927], [3467, 3634], [3467, 3689], [3467, 3700], [3467, 3757], [3467, 3783], [3467, 3812], [3467, 3831], [3467, 3887], [3467, 3922], [3468, 3469], [3468, 3533], [3468, 3670], [3468, 3802], [3468, 3922], [3468, 3924], [3468, 3926], [3468, 4003], [3468, 4161], [3468, 4223], [3468, 4224], [3469, 3516], [3469, 3664], [3469, 3768], [3469, 3793], [3469, 3853], [3469, 3918], [3469, 3952], [3469, 4143], [3469, 4161], [3470, 3471], [3470, 3472], [3470, 3480], [3470, 3550], [3470, 3766], [3470, 3834], [3470, 3862], [3470, 3898], [3470, 3944], [3471, 3472], [3471, 3519], [3471, 3617], [3471, 3741], [3471, 3791], [3471, 3795], [3471, 3804], [3471, 3844], [3471, 3886], [3471, 3928], [3471, 4132], [3472, 3473], [3472, 3565], [3472, 3612], [3472, 3742], [3472, 3796], [3472, 3827], [3472, 3899], [3472, 4063], [3473, 3525], [3473, 3539], [3473, 3550], [3473, 3591], [3473, 3939], [3474, 3475], [3474, 3504], [3474, 3574], [3474, 3593], [3474, 3605], [3474, 3877], [3475, 3476], [3475, 3501], [3475, 3606], [3475, 3656], [3475, 3829], [3475, 3877], [3476, 3477], [3476, 3504], [3476, 3691], [3476, 3880], [3476, 3921], [3477, 3478], [3477, 3606], [3477, 3607], [3477, 3695], [3478, 3512], [3478, 3525], [3478, 3556], [3478, 3631], [3478, 3649], [3478, 3698], [3478, 3850], [3478, 4072], [3479, 3480], [3479, 3483], [3479, 3507], [3479, 3528], [3479, 3596], [3479, 3613], [3479, 3655], [3479, 3668], [3479, 3783], [3479, 3888], [3479, 4232], [3480, 3481], [3480, 3482], [3480, 3596], [3480, 3653], [3480, 3654], [3480, 4201], [3481, 3482], [3481, 3577], [3481, 3579], [3481, 3862], [3481, 3863], [3481, 4200], [3482, 3512], [3482, 3515], [3482, 3528], [3482, 3742], [3482, 3780], [3482, 3882], [3482, 4037], [3483, 3484], [3483, 3485], [3483, 3489], [3483, 3529], [3483, 3616], [3483, 3669], [3483, 3851], [3483, 3934], [3483, 4235], [3483, 4236], [3484, 3485], [3484, 3769], [3484, 3785], [3485, 3931], [3485, 4086], [3485, 4106], [3486, 3487], [3486, 3567], [3486, 3759], [3486, 3760], [3487, 3488], [3487, 3567], [3487, 3568], [3487, 3580], [3487, 3581], [3487, 3672], [3487, 3730], [3487, 3984], [3487, 4174], [3487, 4177], [3488, 3489], [3488, 3559], [3488, 3623], [3488, 3672], [3488, 3708], [3488, 3760], [3488, 3761], [3488, 3836], [3488, 3857], [3488, 3930], [3488, 3939], [3488, 4019], [3488, 4021], [3488, 4073], [3488, 4239], [3489, 3532], [3489, 3596], [3489, 3601], [3489, 3722], [3489, 3728], [3489, 3729], [3489, 3785], [3489, 3787], [3489, 3892], [3489, 4234], [3489, 4255], [3490, 3491], [3490, 3543], [3490, 3566], [3490, 3626], [3490, 3627], [3490, 3854], [3490, 3863], [3490, 3886], [3490, 3909], [3490, 3974], [3490, 4146], [3490, 4188], [3491, 3492], [3491, 3500], [3491, 3555], [3491, 3626], [3491, 3908], [3491, 4143], [3492, 3493], [3492, 3509], [3492, 3618], [3492, 3795], [3492, 3886], [3492, 3963], [3492, 4143], [3492, 4184], [3493, 3747], [3493, 3752], [3493, 3904], [3493, 3962], [3493, 4185], [3494, 3495], [3494, 3651], [3494, 3679], [3494, 3896], [3495, 3496], [3495, 3571], [3495, 3896], [3495, 3917], [3495, 3918], [3495, 3925], [3496, 3616], [3496, 3802], [3496, 3894], [3496, 3917], [3496, 3929], [3496, 4190], [3496, 4246], [3497, 3498], [3497, 3570], [3497, 3578], [3497, 3646], [3497, 4018], [3498, 3499], [3498, 3644], [3498, 3645], [3498, 3764], [3499, 3500], [3499, 3533], [3499, 3548], [3499, 3643], [3499, 3903], [3499, 3916], [3499, 3926], [3499, 4252], [3500, 3571], [3500, 3682], [3500, 3740], [3501, 3502], [3501, 3511], [3501, 3545], [3501, 3551], [3501, 3593], [3502, 3503], [3502, 3702], [3502, 3913], [3502, 3987], [3503, 3646], [3503, 3703], [3503, 3722], [3503, 3734], [3503, 3809], [3503, 3817], [3503, 3904], [3503, 4054], [3503, 4107], [3503, 4149], [3503, 4167], [3503, 4186], [3504, 3505], [3504, 3574], [3504, 3575], [3504, 3695], [3504, 3880], [3504, 3941], [3505, 3506], [3505, 3610], [3505, 3647], [3505, 3648], [3505, 3711], [3505, 3716], [3505, 3919], [3505, 3974], [3506, 3575], [3506, 3646], [3506, 3649], [3506, 3709], [3506, 3712], [3506, 3903], [3506, 3920], [3506, 3935], [3506, 3973], [3506, 4148], [3506, 4152], [3506, 4209], [3506, 4217], [3507, 3508], [3507, 3544], [3507, 3559], [3507, 3619], [3507, 3635], [3507, 3643], [3507, 3654], [3507, 3655], [3507, 3657], [3507, 3771], [3507, 3785], [3507, 3889], [3507, 3907], [3507, 4005], [3507, 4026], [3507, 4029], [3507, 4086], [3507, 4089], [3507, 4226], [3507, 4232], [3507, 4236], [3508, 3509], [3508, 3523], [3508, 3548], [3508, 3578], [3508, 3618], [3508, 3811], [3508, 3907], [3508, 3979], [3508, 4237], [3508, 4238], [3509, 3522], [3509, 3523], [3509, 4214], [3509, 4226], [3510, 3511], [3510, 3513], [3510, 3591], [3510, 3678], [3510, 3738], [3510, 3740], [3510, 3788], [3510, 3804], [3510, 3813], [3510, 3832], [3510, 3931], [3510, 4007], [3510, 4086], [3510, 4108], [3511, 3512], [3511, 3550], [3511, 3591], [3511, 3593], [3511, 3698], [3511, 3767], [3511, 3777], [3511, 3780], [3511, 3814], [3511, 3877], [3511, 3878], [3511, 4085], [3511, 4260], [3512, 3550], [3512, 3606], [3512, 3630], [3512, 3631], [3512, 3987], [3513, 3563], [3513, 3640], [3513, 3762], [3513, 3765], [3513, 3803], [3513, 3822], [3513, 3846], [3513, 3875], [3513, 4108], [3513, 4183], [3514, 3515], [3514, 3864], [3514, 3928], [3515, 3516], [3515, 3741], [3515, 3756], [3515, 3766], [3515, 3812], [3515, 3853], [3515, 4141], [3516, 3517], [3516, 3552], [3516, 3748], [3516, 3756], [3516, 3767], [3516, 3770], [3516, 3839], [3516, 3886], [3516, 3906], [3516, 4016], [3516, 4084], [3516, 4141], [3516, 4179], [3516, 4248], [3517, 3521], [3517, 3752], [3517, 3906], [3517, 3908], [3517, 4001], [3518, 3519], [3518, 3523], [3518, 3557], [3518, 3714], [3518, 3715], [3518, 3825], [3519, 3520], [3519, 3523], [3519, 3826], [3519, 4126], [3520, 3521], [3520, 3557], [3520, 3668], [3521, 3522], [3521, 3523], [3521, 3537], [3521, 3548], [3521, 3557], [3521, 3559], [3521, 4005], [3522, 3548], [3522, 3560], [3522, 3927], [3522, 3981], [3522, 4242], [3523, 3524], [3523, 3541], [3523, 3608], [3523, 3708], [3523, 3795], [3523, 3887], [3523, 3892], [3523, 3995], [3523, 4007], [3523, 4086], [3523, 4126], [3523, 4212], [3524, 3525], [3524, 3541], [3524, 3585], [3524, 3636], [3524, 3660], [3524, 3739], [3524, 3808], [3524, 3824], [3524, 3996], [3524, 4074], [3525, 3540], [3525, 3565], [3525, 3572], [3525, 3573], [3525, 3594], [3525, 3695], [3525, 3990], [3525, 4258], [3526, 3527], [3526, 3530], [3526, 3882], [3526, 3902], [3527, 3528], [3527, 3865], [3527, 3895], [3528, 3529], [3528, 3771], [3528, 4105], [3529, 3642], [3529, 3684], [3529, 3722], [3529, 3851], [3529, 3895], [3529, 3945], [3530, 3531], [3530, 3532], [3530, 3596], [3530, 3713], [3530, 3727], [3530, 4008], [3530, 4135], [3530, 4136], [3531, 3532], [3531, 3649], [3531, 3889], [3531, 4256], [3532, 3533], [3532, 3802], [3532, 3893], [3532, 3902], [3532, 3903], [3532, 3930], [3532, 3951], [3532, 4094], [3532, 4152], [3532, 4181], [3532, 4190], [3532, 4210], [3532, 4250], [3533, 3643], [3533, 3764], [3533, 3965], [3533, 3982], [3533, 3985], [3533, 3986], [3533, 4003], [3533, 4026], [3533, 4211], [3533, 4229], [3534, 3535], [3534, 3554], [3534, 3566], [3534, 3593], [3534, 3702], [3534, 3860], [3534, 3909], [3535, 3536], [3535, 3704], [3535, 3821], [3535, 3918], [3535, 3931], [3535, 4007], [3536, 3549], [3536, 3567], [3536, 3590], [3536, 3645], [3536, 3646], [3536, 3847], [3536, 4042], [3536, 4174], [3537, 3538], [3537, 3539], [3537, 3547], [3537, 3554], [3537, 3582], [3537, 3641], [3537, 3879], [3537, 3906], [3537, 3932], [3537, 4052], [3537, 4158], [3537, 4169], [3537, 4257], [3538, 3539], [3538, 3806], [3538, 3812], [3538, 3839], [3538, 3857], [3538, 3924], [3538, 3933], [3538, 3939], [3538, 4000], [3538, 4080], [3538, 4098], [3538, 4099], [3538, 4130], [3538, 4141], [3539, 3738], [3539, 3739], [3539, 3827], [3539, 4030], [3539, 4074], [3540, 3541], [3540, 3556], [3540, 3557], [3540, 3941], [3541, 3542], [3541, 3581], [3541, 3591], [3541, 3703], [3541, 3758], [3541, 3800], [3541, 3812], [3541, 3842], [3541, 4000], [3541, 4001], [3541, 4004], [3541, 4084], [3541, 4127], [3542, 3867], [3542, 3913], [3542, 4212], [3543, 3544], [3543, 3566], [3543, 3572], [3543, 3574], [3543, 3644], [3543, 3661], [3543, 3668], [3543, 3669], [3543, 3887], [3544, 3545], [3544, 3598], [3544, 3635], [3544, 3670], [3544, 3935], [3544, 4218], [3544, 4222], [3545, 3661], [3545, 3662], [3545, 3664], [3545, 3793], [3546, 3547], [3546, 3551], [3547, 3548], [3547, 3691], [3547, 3916], [3547, 4240], [3547, 4262], [3548, 3549], [3548, 3578], [3548, 3982], [3548, 4018], [3548, 4252], [3549, 3595], [3549, 3663], [3549, 3724], [3549, 3761], [3549, 3796], [3549, 3903], [3549, 3918], [3549, 3930], [3549, 4018], [3550, 3551], [3550, 3552], [3550, 3553], [3550, 3561], [3550, 3665], [3550, 4072], [3550, 4257], [3551, 3552], [3551, 3663], [3551, 3666], [3552, 3664], [3552, 3667], [3552, 4123], [3552, 4261], [3553, 3554], [3553, 3814], [3553, 3898], [3553, 3939], [3553, 4055], [3554, 3666], [3554, 3824], [3554, 3906], [3554, 3982], [3554, 4115], [3554, 4116], [3555, 3556], [3555, 3652], [3555, 3906], [3555, 3909], [3556, 3646], [3556, 3691], [3556, 3927], [3556, 4154], [3556, 4242], [3557, 3558], [3557, 3927], [3558, 3559], [3558, 3655], [3558, 3668], [3558, 3692], [3558, 3714], [3558, 3760], [3558, 3784], [3558, 3941], [3559, 3560], [3559, 3581], [3559, 3668], [3559, 3671], [3559, 3708], [3559, 3785], [3559, 3908], [3559, 3940], [3559, 4024], [3559, 4025], [3559, 4047], [3559, 4084], [3560, 3598], [3560, 3692], [3560, 4022], [3560, 4023], [3560, 4047], [3560, 4239], [3560, 4241], [3561, 3562], [3561, 3581], [3561, 3696], [3561, 3913], [3561, 3987], [3562, 3563], [3562, 3601], [3562, 3846], [3562, 3931], [3562, 4117], [3562, 4220], [3563, 3740], [3563, 3749], [3563, 3768], [3563, 3876], [3563, 3879], [3563, 3881], [3563, 4183], [3564, 3565], [3564, 3569], [3564, 3572], [3564, 3693], [3565, 3566], [3565, 3570], [3565, 3653], [3565, 3677], [3565, 3695], [3565, 3702], [3565, 3900], [3566, 3567], [3567, 3568], [3567, 3644], [3567, 3761], [3568, 3747], [3568, 3811], [3568, 3890], [3568, 3984], [3568, 4042], [3568, 4252], [3569, 3570], [3569, 3583], [3569, 3620], [3569, 3636], [3569, 3649], [3569, 3709], [3569, 3787], [3570, 3571], [3570, 3649], [3570, 3796], [3571, 3585], [3571, 3622], [3571, 3724], [3571, 3894], [3571, 3900], [3571, 4192], [3572, 3573], [3572, 3673], [3572, 3693], [3573, 3574], [3573, 3641], [3573, 3694], [3574, 3575], [3574, 3592], [3574, 3645], [3574, 3823], [3574, 3983], [3574, 4034], [3575, 3940], [3575, 3972], [3575, 3983], [3575, 4217], [3575, 4249], [3576, 3577], [3576, 3598], [3576, 3692], [3577, 3578], [3577, 3579], [3577, 3653], [3577, 3711], [3578, 3579], [3578, 3691], [3578, 3692], [3579, 3598], [3579, 3654], [3579, 3712], [3579, 3908], [3579, 3982], [3579, 4200], [3579, 4238], [3579, 4259], [3580, 3581], [3580, 3708], [3580, 3783], [3580, 3789], [3580, 3811], [3580, 3830], [3580, 3865], [3581, 3582], [3581, 3591], [3581, 3643], [3581, 3671], [3581, 3847], [3581, 3985], [3581, 4023], [3581, 4129], [3581, 4178], [3581, 4252], [3582, 3688], [3582, 3690], [3582, 3691], [3582, 3697], [3582, 3767], [3582, 3830], [3582, 3880], [3582, 3881], [3582, 3941], [3582, 4050], [3582, 4080], [3582, 4108], [3582, 4128], [3583, 3584], [3583, 3601], [3583, 3860], [3584, 3585], [3584, 3587], [3584, 3602], [3584, 3698], [3584, 3890], [3584, 4116], [3584, 4139], [3585, 3586], [3585, 3740], [3585, 3787], [3585, 3808], [3585, 3832], [3585, 3842], [3585, 3876], [3585, 3890], [3585, 4155], [3586, 3704], [3586, 3824], [3586, 3833], [3586, 3876], [3586, 3956], [3586, 4049], [3586, 4120], [3587, 3588], [3587, 3602], [3587, 3637], [3587, 3697], [3587, 3722], [3587, 3808], [3587, 3818], [3587, 3867], [3587, 3944], [3587, 4010], [3587, 4013], [3587, 4099], [3588, 3589], [3588, 3600], [3588, 3672], [3588, 3746], [3588, 3808], [3588, 3813], [3588, 3853], [3588, 3856], [3588, 3857], [3588, 4006], [3588, 4016], [3588, 4017], [3588, 4177], [3588, 4179], [3589, 3632], [3589, 3667], [3589, 3801], [3589, 3905], [3589, 3958], [3589, 4006], [3589, 4013], [3589, 4046], [3589, 4064], [3589, 4115], [3589, 4117], [3589, 4118], [3589, 4120], [3589, 4121], [3589, 4165], [3589, 4183], [3589, 4247], [3589, 4261], [3590, 3591], [3590, 3726], [3590, 3796], [3590, 4007], [3591, 3592], [3591, 3645], [3591, 3722], [3591, 3884], [3591, 4032], [3591, 4033], [3591, 4092], [3591, 4136], [3591, 4137], [3592, 3605], [3592, 3607], [3592, 3695], [3592, 3766], [3592, 3827], [3592, 3942], [3592, 4090], [3592, 4258], [3593, 3594], [3593, 3661], [3593, 3694], [3593, 3702], [3594, 3595], [3594, 3636], [3594, 3767], [3594, 3941], [3595, 3636], [3595, 3673], [3595, 3761], [3595, 4019], [3596, 3597], [3596, 3612], [3596, 3636], [3596, 3887], [3596, 4010], [3597, 3598], [3597, 3612], [3597, 3675], [3597, 3882], [3597, 3885], [3597, 4027], [3597, 4029], [3598, 3634], [3598, 3668], [3598, 3670], [3598, 3692], [3598, 3707], [3598, 3774], [3598, 4022], [3598, 4259], [3599, 3600], [3599, 3753], [3599, 3794], [3599, 3795], [3599, 3808], [3599, 3838], [3599, 3844], [3600, 3793], [3600, 3856], [3601, 3602], [3601, 3672], [3601, 3749], [3601, 3787], [3601, 3810], [3601, 3900], [3602, 3603], [3602, 3802], [3602, 3894], [3602, 4046], [3602, 4066], [3602, 4138], [3602, 4220], [3602, 4234], [3603, 3604], [3603, 3672], [3603, 3790], [3603, 3894], [3603, 3929], [3603, 3984], [3603, 4036], [3603, 4064], [3603, 4116], [3603, 4216], [3604, 3665], [3604, 3671], [3604, 3790], [3604, 3985], [3604, 4025], [3604, 4044], [3604, 4072], [3605, 3606], [3605, 3610], [3605, 3695], [3605, 3737], [3606, 3607], [3606, 3610], [3606, 3825], [3606, 3830], [3606, 3883], [3606, 3974], [3607, 3608], [3607, 3631], [3607, 3884], [3607, 3954], [3607, 3973], [3607, 4070], [3607, 4085], [3607, 4212], [3608, 3609], [3608, 3627], [3608, 3816], [3608, 3825], [3608, 3826], [3608, 3871], [3608, 3873], [3608, 3885], [3608, 3907], [3608, 4086], [3608, 4124], [3609, 3660], [3609, 3725], [3609, 3733], [3609, 3796], [3609, 3818], [3609, 3827], [3609, 3996], [3609, 4031], [3609, 4062], [3609, 4101], [3609, 4214], [3610, 3611], [3610, 3711], [3611, 3612], [3611, 3716], [3611, 3737], [3612, 3827], [3612, 3886], [3612, 3911], [3612, 4243], [3613, 3614], [3613, 3617], [3613, 3619], [3613, 3620], [3613, 3623], [3613, 3627], [3613, 3630], [3613, 3633], [3613, 3636], [3613, 3731], [3613, 3760], [3613, 3788], [3613, 3885], [3613, 3934], [3614, 3615], [3614, 3744], [3614, 3750], [3614, 3885], [3614, 3907], [3614, 3979], [3615, 3616], [3615, 3748], [3615, 3749], [3615, 3851], [3615, 3854], [3615, 3870], [3615, 3900], [3615, 3935], [3615, 4054], [3615, 4069], [3615, 4192], [3615, 4196], [3615, 4197], [3616, 3687], [3616, 4188], [3617, 3618], [3617, 3716], [3617, 3760], [3617, 3789], [3617, 3835], [3617, 3836], [3617, 3886], [3618, 3619], [3618, 3761], [3618, 3795], [3618, 3836], [3619, 3623], [3619, 3761], [3619, 3786], [3619, 4184], [3620, 3621], [3620, 3799], [3620, 3934], [3621, 3622], [3621, 3623], [3621, 3768], [3621, 4021], [3622, 3706], [3622, 3723], [3622, 3851], [3622, 4104], [3622, 4192], [3622, 4236], [3623, 3624], [3623, 3625], [3623, 3628], [3623, 3637], [3623, 3723], [3623, 3748], [3623, 3779], [3623, 3836], [3623, 3947], [3623, 4012], [3623, 4195], [3623, 4232], [3624, 3625], [3624, 3629], [3624, 3630], [3624, 3631], [3624, 3654], [3624, 3943], [3624, 4029], [3624, 4057], [3624, 4058], [3624, 4123], [3624, 4142], [3624, 4259], [3625, 3626], [3625, 3855], [3625, 4144], [3625, 4222], [3626, 3628], [3626, 3629], [3626, 3906], [3626, 3963], [3626, 3965], [3626, 3973], [3626, 3980], [3626, 3991], [3626, 4043], [3626, 4068], [3626, 4083], [3627, 3628], [3627, 3680], [3627, 3751], [3627, 3755], [3627, 3796], [3627, 3896], [3627, 3907], [3628, 3629], [3628, 3721], [3628, 3752], [3628, 3999], [3628, 4012], [3628, 4124], [3628, 4185], [3628, 4245], [3629, 3654], [3629, 3657], [3629, 3942], [3629, 3978], [3629, 4148], [3629, 4228], [3629, 4243], [3630, 3653], [3630, 3707], [3630, 3766], [3630, 3935], [3631, 3632], [3631, 3954], [3631, 3965], [3631, 3971], [3631, 3986], [3631, 4037], [3631, 4068], [3631, 4144], [3631, 4154], [3631, 4204], [3631, 4256], [3631, 4257], [3631, 4258], [3631, 4260], [3631, 4261], [3632, 3684], [3632, 3857], [3632, 3930], [3632, 3937], [3632, 3942], [3632, 3946], [3632, 3949], [3632, 3951], [3632, 3952], [3632, 3953], [3632, 3955], [3632, 3956], [3632, 3958], [3632, 3962], [3632, 3964], [3632, 3966], [3632, 3968], [3632, 3971], [3632, 3977], [3632, 3978], [3632, 3981], [3632, 3983], [3632, 3984], [3632, 3988], [3632, 3995], [3632, 4008], [3632, 4014], [3632, 4022], [3632, 4035], [3632, 4037], [3632, 4044], [3632, 4048], [3632, 4050], [3632, 4053], [3632, 4056], [3632, 4057], [3632, 4059], [3632, 4062], [3632, 4064], [3632, 4071], [3632, 4075], [3632, 4077], [3632, 4081], [3632, 4098], [3632, 4102], [3632, 4111], [3632, 4159], [3632, 4170], [3632, 4189], [3632, 4199], [3632, 4208], [3632, 4227], [3632, 4233], [3633, 3634], [3633, 3635], [3633, 3756], [3634, 3635], [3634, 3757], [3634, 3897], [3635, 3770], [3635, 3793], [3635, 3855], [3635, 3922], [3635, 3934], [3635, 4161], [3635, 4162], [3636, 3637], [3636, 3708], [3636, 3713], [3636, 3731], [3636, 3733], [3636, 3758], [3636, 3787], [3636, 3796], [3637, 3638], [3637, 3723], [3637, 3996], [3637, 3999], [3637, 4010], [3637, 4011], [3637, 4066], [3638, 3654], [3638, 3708], [3638, 3710], [3638, 3790], [3638, 3995], [3638, 4002], [3638, 4025], [3638, 4057], [3638, 4089], [3639, 3640], [3639, 3642], [3639, 3645], [3639, 3718], [3639, 3765], [3639, 4175], [3640, 3641], [3640, 3674], [3640, 3843], [3641, 3673], [3641, 3743], [3641, 4115], [3641, 4130], [3641, 4175], [3641, 4183], [3641, 4206], [3642, 3643], [3642, 3765], [3642, 3882], [3642, 3916], [3642, 4029], [3642, 4229], [3643, 3782], [3643, 3860], [3643, 3902], [3643, 3922], [3643, 3987], [3643, 4212], [3643, 4255], [3644, 3645], [3644, 3728], [3644, 3730], [3644, 3769], [3644, 3784], [3645, 3646], [3645, 3722], [3645, 3879], [3645, 3902], [3645, 3916], [3645, 3931], [3645, 4175], [3645, 4250], [3646, 3647], [3646, 3649], [3646, 3657], [3646, 3691], [3646, 3821], [3646, 3910], [3646, 3961], [3646, 4041], [3646, 4108], [3646, 4119], [3646, 4167], [3646, 4176], [3646, 4228], [3646, 4262], [3647, 3648], [3647, 3656], [3647, 3734], [3647, 3869], [3648, 3649], [3648, 3651], [3648, 3662], [3649, 3652], [3649, 3663], [3649, 3713], [3649, 4054], [3649, 4139], [3650, 3651], [3650, 3799], [3650, 3923], [3651, 3652], [3651, 3662], [3651, 3799], [3652, 3663], [3652, 3754], [3652, 3905], [3652, 4110], [3652, 4209], [3653, 3654], [3653, 3655], [3653, 3708], [3653, 3709], [3653, 3979], [3654, 3710], [3654, 4030], [3654, 4201], [3654, 4212], [3654, 4239], [3654, 4240], [3655, 3656], [3655, 3677], [3655, 3784], [3656, 3657], [3656, 3941], [3657, 3763], [3657, 3940], [3657, 4029], [3657, 4094], [3657, 4190], [3657, 4231], [3658, 3659], [3658, 3756], [3658, 3869], [3658, 3883], [3658, 3923], [3659, 3660], [3659, 3700], [3659, 3784], [3659, 3825], [3659, 3883], [3660, 3675], [3660, 3695], [3660, 3737], [3660, 3739], [3660, 3809], [3660, 3825], [3660, 3827], [3661, 3662], [3661, 3856], [3662, 3663], [3662, 3745], [3663, 3664], [3663, 3666], [3663, 3746], [3663, 3918], [3663, 4240], [3663, 4249], [3664, 3667], [3664, 3915], [3664, 4122], [3664, 4139], [3664, 4145], [3664, 4199], [3664, 4218], [3665, 3666], [3665, 3667], [3665, 3671], [3665, 3777], [3665, 3834], [3665, 3864], [3665, 4024], [3665, 4044], [3665, 4129], [3666, 3667], [3666, 3713], [3666, 4115], [3667, 3690], [3667, 3818], [3667, 4022], [3667, 4055], [3667, 4121], [3667, 4142], [3667, 4213], [3667, 4214], [3667, 4215], [3668, 3669], [3668, 3783], [3668, 3863], [3668, 4024], [3669, 3670], [3669, 3673], [3669, 3897], [3669, 4082], [3669, 4188], [3670, 3690], [3670, 3802], [3670, 3873], [3670, 4088], [3670, 4089], [3670, 4207], [3671, 3672], [3671, 3789], [3671, 4179], [3672, 3673], [3672, 3746], [3672, 3748], [3672, 3759], [3672, 3789], [3672, 3900], [3672, 3928], [3672, 4115], [3673, 3730], [3673, 3748], [3673, 3788], [3673, 3846], [3673, 3851], [3673, 4036], [3674, 3675], [3674, 3738], [3674, 3843], [3675, 3676], [3675, 3694], [3675, 3749], [3675, 3815], [3675, 3818], [3675, 3830], [3675, 3882], [3675, 3911], [3676, 3818], [3676, 3870], [3676, 3884], [3676, 4054], [3676, 4070], [3677, 3678], [3677, 3679], [3677, 3685], [3677, 3888], [3677, 3889], [3678, 3679], [3678, 3686], [3678, 3922], [3679, 3687], [3679, 3848], [3679, 3850], [3680, 3681], [3680, 3721], [3680, 3726], [3680, 3727], [3680, 3731], [3680, 3794], [3680, 3796], [3681, 3682], [3681, 3907], [3682, 3724], [3682, 3753], [3682, 3758], [3682, 3866], [3682, 3908], [3682, 4001], [3683, 3684], [3683, 3727], [3683, 3763], [3683, 3778], [3683, 3948], [3683, 4062], [3683, 4107], [3683, 4135], [3683, 4150], [3684, 3763], [3684, 3801], [3684, 3893], [3684, 4003], [3684, 4052], [3684, 4059], [3684, 4097], [3684, 4104], [3684, 4105], [3684, 4106], [3684, 4109], [3684, 4151], [3684, 4159], [3684, 4224], [3684, 4229], [3684, 4235], [3684, 4253], [3685, 3686], [3685, 3687], [3685, 3902], [3686, 3687], [3686, 3775], [3686, 3805], [3686, 3897], [3687, 3837], [3687, 3896], [3687, 3914], [3687, 4151], [3687, 4193], [3688, 3689], [3688, 3766], [3688, 3780], [3688, 3862], [3688, 3877], [3688, 4128], [3689, 3690], [3689, 3699], [3690, 3699], [3690, 3801], [3690, 3817], [3690, 3828], [3690, 3891], [3690, 3915], [3690, 4051], [3690, 4080], [3691, 3692], [3691, 3921], [3691, 3941], [3691, 4241], [3691, 4259], [3692, 3927], [3692, 3979], [3693, 3694], [3693, 3695], [3693, 3756], [3693, 3766], [3693, 3767], [3693, 3788], [3694, 3695], [3694, 3909], [3696, 3697], [3696, 3780], [3696, 3883], [3697, 3698], [3697, 3780], [3697, 3884], [3697, 4179], [3698, 3740], [3698, 3849], [3698, 3880], [3698, 4033], [3698, 4072], [3698, 4260], [3699, 3700], [3699, 3701], [3699, 3819], [3700, 3701], [3700, 3784], [3700, 3840], [3700, 3841], [3701, 3736], [3701, 3861], [3701, 3866], [3701, 3895], [3702, 3703], [3702, 3704], [3702, 3735], [3702, 3824], [3703, 3704], [3703, 3719], [3703, 3727], [3703, 3753], [3703, 3758], [3703, 3843], [3703, 3844], [3703, 4107], [3703, 4120], [3704, 3705], [3704, 3754], [3704, 3833], [3704, 3889], [3705, 3706], [3705, 3843], [3705, 3845], [3705, 3851], [3705, 3875], [3705, 3876], [3705, 3923], [3706, 3838], [3706, 3876], [3706, 3925], [3707, 3708], [3707, 3712], [3707, 3743], [3707, 3935], [3707, 4019], [3707, 4057], [3707, 4095], [3708, 3709], [3708, 3712], [3708, 3714], [3708, 3758], [3708, 3789], [3708, 4088], [3709, 3710], [3709, 3935], [3709, 4021], [3710, 4058], [3710, 4148], [3710, 4233], [3710, 4236], [3711, 3712], [3711, 3714], [3711, 3830], [3711, 3941], [3712, 3713], [3712, 3930], [3712, 3940], [3712, 4090], [3712, 4093], [3712, 4187], [3712, 4188], [3713, 3742], [3713, 3748], [3713, 3796], [3713, 3818], [3713, 3824], [3713, 4011], [3713, 4123], [3714, 3715], [3714, 3760], [3715, 3716], [3715, 3804], [3716, 3728], [3716, 3734], [3716, 3789], [3716, 3804], [3716, 3900], [3716, 3919], [3717, 3718], [3717, 3722], [3717, 3726], [3717, 3728], [3717, 3731], [3717, 3734], [3717, 3858], [3717, 3859], [3718, 3719], [3718, 3722], [3718, 3729], [3718, 3843], [3718, 3845], [3718, 3950], [3719, 3720], [3719, 3726], [3719, 3727], [3719, 3831], [3720, 3721], [3720, 3722], [3720, 3726], [3720, 3753], [3720, 3842], [3720, 3890], [3720, 4093], [3721, 3723], [3721, 3727], [3721, 3753], [3721, 3999], [3721, 4048], [3722, 3723], [3722, 3813], [3722, 3814], [3722, 3817], [3722, 3838], [3722, 3939], [3722, 3950], [3722, 3987], [3722, 4059], [3722, 4255], [3723, 3724], [3723, 3731], [3723, 3838], [3723, 3997], [3723, 4089], [3724, 3725], [3724, 3746], [3724, 3796], [3724, 3808], [3724, 3836], [3724, 3896], [3724, 3899], [3724, 4112], [3725, 3806], [3725, 3808], [3725, 3809], [3725, 3873], [3725, 3963], [3725, 4111], [3725, 4186], [3726, 3841], [3726, 3860], [3727, 3889], [3727, 3897], [3727, 4004], [3727, 4048], [3727, 4063], [3727, 4073], [3727, 4092], [3728, 3729], [3728, 3760], [3728, 3784], [3728, 3901], [3728, 3902], [3729, 3730], [3729, 3783], [3730, 3762], [3730, 3764], [3730, 3783], [3730, 3846], [3730, 4035], [3730, 4175], [3731, 3732], [3732, 3733], [3732, 3769], [3732, 3809], [3732, 3817], [3732, 3885], [3733, 3767], [3733, 3818], [3733, 3885], [3733, 4080], [3733, 4088], [3734, 3735], [3734, 3736], [3734, 3781], [3734, 3911], [3735, 3736], [3735, 3895], [3735, 3900], [3735, 3912], [3736, 3771], [3736, 3800], [3736, 3909], [3736, 3913], [3737, 3738], [3737, 3804], [3737, 3827], [3738, 3739], [3738, 3812], [3739, 3740], [3739, 3839], [3740, 3786], [3740, 4033], [3741, 3742], [3741, 3748], [3741, 3803], [3741, 3843], [3741, 4131], [3742, 3743], [3743, 3833], [3743, 4096], [3743, 4102], [3744, 3745], [3745, 3746], [3745, 3759], [3745, 3856], [3745, 3898], [3745, 3979], [3746, 3747], [3746, 3854], [3746, 3898], [3746, 4054], [3746, 4215], [3746, 4216], [3746, 4239], [3746, 4247], [3747, 4006], [3747, 4208], [3747, 4209], [3747, 4237], [3747, 4247], [3748, 3749], [3748, 3752], [3748, 3795], [3748, 3838], [3748, 3945], [3748, 4064], [3748, 4123], [3748, 4131], [3748, 4188], [3749, 3750], [3749, 3775], [3749, 3803], [3749, 3876], [3749, 4096], [3749, 4138], [3750, 3773], [3750, 3774], [3750, 3788], [3750, 3810], [3750, 3936], [3750, 4079], [3750, 4095], [3751, 3752], [3751, 3754], [3752, 3753], [3752, 3795], [3752, 3908], [3752, 4110], [3752, 4228], [3753, 3838], [3753, 4001], [3753, 4119], [3753, 4120], [3754, 3755], [3755, 3889], [3755, 4146], [3755, 4256], [3756, 3757], [3756, 3793], [3756, 3909], [3757, 3758], [3757, 3788], [3757, 3792], [3757, 3793], [3758, 3837], [3758, 3863], [3758, 3865], [3758, 3899], [3758, 3939], [3758, 3941], [3758, 4080], [3759, 3760], [3759, 3856], [3760, 3761], [3760, 3887], [3760, 3979], [3762, 3763], [3762, 3765], [3762, 3769], [3762, 3816], [3762, 3943], [3762, 4029], [3762, 4071], [3762, 4086], [3762, 4099], [3763, 3764], [3763, 3942], [3763, 4032], [3763, 4034], [3763, 4086], [3763, 4108], [3763, 4169], [3763, 4194], [3764, 3847], [3764, 4175], [3764, 4253], [3765, 3868], [3765, 3878], [3765, 3879], [3765, 4182], [3766, 3767], [3766, 3943], [3767, 3768], [3767, 3769], [3767, 3779], [3767, 3943], [3767, 4015], [3767, 4070], [3767, 4085], [3767, 4249], [3768, 3779], [3768, 3881], [3768, 4040], [3769, 3770], [3769, 3851], [3769, 3870], [3769, 3878], [3770, 3771], [3770, 3792], [3770, 3839], [3770, 3945], [3771, 3850], [3771, 4105], [3772, 3773], [3772, 3777], [3772, 3781], [3772, 3783], [3772, 3788], [3772, 3846], [3773, 3774], [3773, 3781], [3773, 3814], [3774, 3775], [3774, 3782], [3774, 3783], [3775, 3776], [3775, 3777], [3775, 3848], [3775, 3872], [3775, 3879], [3775, 3882], [3775, 3902], [3775, 3951], [3775, 4198], [3776, 3848], [3776, 3884], [3776, 3903], [3776, 3951], [3776, 3973], [3776, 4091], [3776, 4137], [3776, 4219], [3777, 3778], [3777, 3779], [3777, 3780], [3777, 4075], [3777, 4092], [3777, 4107], [3777, 4117], [3777, 4178], [3778, 3779], [3778, 3783], [3778, 3785], [3778, 4024], [3778, 4035], [3778, 4085], [3778, 4099], [3778, 4232], [3779, 3786], [3779, 3788], [3779, 3790], [3779, 4036], [3779, 4047], [3779, 4079], [3779, 4103], [3780, 3882], [3781, 3782], [3781, 3840], [3781, 3860], [3782, 3783], [3782, 3902], [3783, 3784], [3783, 3785], [3783, 3788], [3783, 3830], [3784, 3785], [3785, 3786], [3785, 3811], [3785, 4094], [3785, 4137], [3786, 3787], [3786, 3788], [3787, 3837], [3787, 3838], [3787, 3934], [3787, 4066], [3787, 4103], [3788, 3789], [3789, 3790], [3789, 3834], [3789, 3840], [3789, 4007], [3790, 3834], [3790, 3836], [3790, 3837], [3790, 3993], [3790, 4006], [3790, 4121], [3790, 4188], [3790, 4214], [3791, 3792], [3791, 3794], [3791, 3844], [3791, 3852], [3791, 3897], [3792, 3809], [3792, 3825], [3792, 3873], [3792, 3886], [3793, 3794], [3793, 3821], [3793, 3852], [3793, 3853], [3793, 3922], [3794, 3795], [3794, 3841], [3794, 3897], [3794, 3945], [3795, 3796], [3795, 3918], [3795, 3998], [3795, 4007], [3795, 4132], [3796, 3999], [3796, 4063], [3797, 3798], [3797, 3799], [3798, 3860], [3798, 3861], [3799, 3822], [3799, 3945], [3800, 3801], [3800, 3812], [3800, 4098], [3800, 4105], [3800, 4118], [3801, 3802], [3801, 3813], [3801, 3842], [3801, 3876], [3801, 4107], [3801, 4108], [3801, 4109], [3801, 4165], [3801, 4167], [3802, 3805], [3802, 3876], [3802, 3881], [3802, 3920], [3802, 3925], [3802, 3929], [3802, 3932], [3802, 3994], [3802, 4046], [3802, 4181], [3802, 4235], [3803, 3804], [3803, 3875], [3803, 4096], [3804, 3805], [3804, 3832], [3805, 3827], [3805, 4088], [3805, 4096], [3805, 4107], [3805, 4132], [3805, 4155], [3806, 3807], [3806, 3809], [3806, 3827], [3806, 3891], [3806, 3899], [3806, 4017], [3806, 4080], [3806, 4111], [3806, 4149], [3806, 4150], [3807, 3808], [3807, 3832], [3807, 3844], [3807, 3899], [3807, 4113], [3808, 3844], [3808, 3897], [3808, 4113], [3808, 4120], [3808, 4130], [3808, 4132], [3809, 3810], [3809, 3839], [3809, 3844], [3809, 3886], [3810, 3811], [3810, 3979], [3811, 3866], [3811, 4252], [3812, 3813], [3812, 3839], [3812, 3843], [3812, 3887], [3812, 3923], [3813, 3814], [3813, 3842], [3813, 3853], [3813, 3858], [3813, 3923], [3813, 3931], [3813, 3933], [3813, 4106], [3814, 3846], [3814, 3859], [3814, 4060], [3815, 3816], [3815, 3819], [3815, 3822], [3815, 3825], [3815, 3870], [3816, 3817], [3816, 3818], [3816, 3868], [3816, 3870], [3816, 3885], [3816, 3904], [3816, 3946], [3816, 4163], [3816, 4197], [3817, 3818], [3817, 4097], [3818, 4027], [3818, 4053], [3818, 4099], [3818, 4100], [3818, 4135], [3818, 4138], [3818, 4139], [3819, 3820], [3820, 3821], [3820, 3852], [3820, 4165], [3821, 3909], [3821, 3931], [3821, 4165], [3822, 3823], [3822, 4163], [3823, 3824], [3823, 4108], [3823, 4251], [3824, 4120], [3824, 4261], [3824, 4262], [3825, 3826], [3826, 3827], [3827, 4062], [3827, 4074], [3828, 3829], [3828, 3830], [3828, 3872], [3828, 3891], [3828, 3914], [3828, 3936], [3828, 4087], [3828, 4171], [3829, 3830], [3829, 3921], [3829, 3935], [3829, 4187], [3829, 4207], [3830, 3863], [3830, 3871], [3830, 3974], [3831, 3832], [3831, 3841], [3831, 3842], [3831, 3872], [3831, 3895], [3831, 3897], [3831, 3959], [3832, 3833], [3832, 3875], [3832, 3887], [3832, 3900], [3832, 4017], [3832, 4155], [3833, 3875], [3833, 3956], [3833, 4108], [3833, 4176], [3833, 4194], [3833, 4195], [3834, 3835], [3834, 3993], [3835, 3836], [3835, 3863], [3835, 3864], [3835, 3899], [3835, 3992], [3836, 3896], [3836, 3908], [3836, 3963], [3836, 3992], [3836, 4132], [3836, 4238], [3837, 3838], [3837, 3840], [3837, 3842], [3837, 3896], [3837, 3925], [3837, 3994], [3837, 4061], [3837, 4067], [3837, 4087], [3837, 4089], [3837, 4093], [3837, 4188], [3837, 4255], [3838, 3839], [3838, 3843], [3838, 3945], [3838, 4065], [3838, 4130], [3839, 4001], [3840, 3841], [3841, 3842], [3841, 3858], [3841, 3860], [3841, 4007], [3842, 3959], [3842, 4003], [3842, 4006], [3842, 4137], [3842, 4173], [3843, 3844], [3843, 4130], [3844, 3856], [3845, 3846], [3845, 3851], [3845, 3878], [3845, 3901], [3845, 4198], [3846, 3847], [3846, 4117], [3847, 4108], [3848, 3849], [3848, 3883], [3848, 3974], [3849, 3850], [3849, 3878], [3849, 3879], [3849, 3880], [3851, 3889], [3851, 3900], [3851, 4104], [3851, 4206], [3852, 3853], [3852, 3856], [3852, 3858], [3853, 3854], [3853, 3897], [3853, 3928], [3853, 3952], [3853, 4162], [3853, 4165], [3854, 3855], [3854, 3886], [3854, 3896], [3854, 3914], [3854, 3928], [3854, 3935], [3854, 3980], [3854, 4134], [3854, 4143], [3854, 4166], [3854, 4209], [3855, 3885], [3855, 4029], [3856, 3858], [3856, 3887], [3856, 4007], [3857, 3887], [3857, 3947], [3857, 3963], [3857, 3995], [3857, 4010], [3857, 4030], [3857, 4047], [3857, 4077], [3857, 4088], [3857, 4103], [3857, 4106], [3857, 4155], [3857, 4162], [3857, 4177], [3857, 4239], [3858, 3859], [3859, 3860], [3860, 3865], [3860, 3869], [3860, 3878], [3861, 3862], [3861, 3865], [3862, 3863], [3862, 3865], [3862, 4202], [3863, 3864], [3863, 3908], [3863, 3991], [3863, 4188], [3864, 3929], [3864, 3937], [3864, 3939], [3864, 4141], [3864, 4142], [3864, 4146], [3865, 3866], [3865, 3867], [3866, 3867], [3867, 4100], [3867, 4202], [3868, 3869], [3868, 3871], [3868, 3910], [3868, 3960], [3869, 3870], [3869, 3878], [3869, 3909], [3869, 3910], [3869, 3974], [3870, 4070], [3871, 3872], [3871, 4125], [3872, 3873], [3872, 3895], [3872, 4017], [3872, 4140], [3873, 3892], [3873, 3896], [3873, 3897], [3873, 3963], [3873, 4061], [3873, 4106], [3874, 3875], [3874, 3923], [3874, 3924], [3874, 4157], [3875, 3876], [3875, 4156], [3876, 3924], [3876, 4104], [3876, 4130], [3876, 4156], [3876, 4198], [3877, 3878], [3877, 3880], [3877, 4034], [3878, 3879], [3879, 3910], [3879, 3916], [3879, 4034], [3879, 4182], [3879, 4198], [3879, 4260], [3880, 3881], [3880, 3915], [3880, 3972], [3880, 3974], [3880, 4034], [3881, 3921], [3881, 3932], [3881, 4061], [3881, 4094], [3882, 3883], [3882, 3884], [3882, 4135], [3883, 3884], [3883, 3987], [3884, 3916], [3884, 3986], [3884, 4009], [3884, 4023], [3884, 4091], [3884, 4135], [3885, 3886], [3885, 3887], [3885, 3947], [3885, 4184], [3886, 3887], [3886, 3904], [3886, 3963], [3886, 4016], [3887, 3979], [3888, 3889], [3888, 4194], [3889, 3890], [3889, 4005], [3889, 4049], [3889, 4137], [3889, 4194], [3889, 4206], [3890, 4033], [3891, 3892], [3891, 3893], [3891, 4134], [3891, 4170], [3891, 4215], [3892, 3893], [3892, 4106], [3893, 3894], [3893, 3895], [3893, 3959], [3893, 4110], [3893, 4140], [3893, 4150], [3893, 4208], [3893, 4245], [3893, 4253], [3894, 3900], [3894, 3970], [3894, 3975], [3894, 4104], [3894, 4112], [3894, 4155], [3894, 4196], [3894, 4217], [3894, 4239], [3895, 3896], [3895, 3900], [3895, 3902], [3896, 3897], [3896, 4245], [3896, 4246], [3897, 4081], [3897, 4132], [3898, 3899], [3898, 4215], [3899, 3900], [3899, 3939], [3899, 4112], [3900, 3917], [3900, 3979], [3900, 4017], [3900, 4176], [3901, 3902], [3902, 3903], [3903, 3925], [3903, 4093], [3903, 4137], [3904, 3905], [3904, 3906], [3904, 3962], [3904, 4069], [3904, 4166], [3904, 4187], [3904, 4244], [3905, 3906], [3905, 3977], [3905, 4016], [3905, 4142], [3905, 4158], [3905, 4163], [3906, 3909], [3906, 3910], [3906, 4039], [3906, 4068], [3906, 4154], [3906, 4225], [3907, 3908], [3907, 4185], [3907, 4244], [3908, 3991], [3909, 3913], [3910, 3960], [3910, 3973], [3910, 4100], [3910, 4108], [3910, 4139], [3911, 3912], [3911, 3913], [3912, 3913], [3913, 4039], [3913, 4118], [3914, 3915], [3914, 3919], [3914, 3974], [3915, 3916], [3915, 3926], [3915, 3931], [3915, 4205], [3915, 4224], [3916, 4229], [3917, 3928], [3918, 3925], [3918, 3928], [3918, 3929], [3918, 4143], [3919, 3920], [3920, 3921], [3920, 4167], [3921, 4040], [3921, 4087], [3921, 4207], [3922, 3923], [3922, 3926], [3923, 3924], [3923, 3987], [3924, 3933], [3924, 3986], [3924, 3997], [3924, 4017], [3924, 4144], [3924, 4157], [3925, 3926], [3925, 4236], [3926, 3927], [3928, 3929], [3929, 3930], [3929, 3952], [3929, 3980], [3929, 4132], [3929, 4146], [3929, 4168], [3929, 4188], [3929, 4195], [3930, 3940], [3930, 3967], [3930, 3978], [3930, 3982], [3930, 4012], [3930, 4019], [3930, 4056], [3930, 4063], [3930, 4112], [3930, 4115], [3930, 4174], [3930, 4190], [3930, 4217], [3930, 4256], [3931, 3932], [3932, 3933], [3932, 4045], [3932, 4052], [3932, 4205], [3932, 4220], [3932, 4250], [3933, 3952], [3933, 4059], [3933, 4060], [3933, 4061], [3933, 4173], [3934, 3935], [3934, 3945], [3935, 3936], [3935, 4020], [3935, 4040], [3935, 4058], [3935, 4153], [3935, 4249], [3936, 4017], [3937, 3938], [3937, 3977], [3937, 3991], [3937, 3992], [3937, 4044], [3938, 3939], [3938, 3970], [3938, 4055], [3938, 4059], [3938, 4112], [3939, 4073], [3940, 3941], [3940, 4050], [3940, 4094], [3940, 4241], [3942, 3943], [3942, 3983], [3942, 4030], [3942, 4032], [3942, 4062], [3943, 3944], [3943, 4015], [3943, 4030], [3943, 4077], [3943, 4099], [3943, 4128], [3943, 4141], [3944, 3954], [3944, 3993], [3944, 4043], [3944, 4046], [3944, 4063], [3944, 4118], [3944, 4132], [3944, 4201], [3944, 4202], [3944, 4215], [3944, 4257], [3945, 4109], [3946, 3947], [3946, 3960], [3946, 3962], [3946, 4012], [3946, 4071], [3946, 4124], [3947, 3948], [3947, 3963], [3947, 4027], [3947, 4124], [3947, 4184], [3948, 3960], [3948, 3976], [3948, 4034], [3948, 4095], [3948, 4096], [3948, 4125], [3948, 4135], [3948, 4141], [3948, 4157], [3948, 4177], [3948, 4224], [3949, 3950], [3949, 4126], [3949, 4128], [3949, 4131], [3949, 4133], [3949, 4221], [3950, 4059], [3950, 4130], [3950, 4136], [3950, 4149], [3950, 4175], [3950, 4198], [3951, 4075], [3951, 4135], [3951, 4138], [3951, 4140], [3951, 4189], [3952, 3980], [3952, 4081], [3952, 4141], [3952, 4144], [3952, 4199], [3953, 3954], [3953, 3992], [3953, 3993], [3953, 4147], [3953, 4149], [3953, 4151], [3953, 4152], [3954, 4069], [3955, 3964], [3955, 4098], [3955, 4154], [3955, 4223], [3956, 3957], [3956, 3966], [3956, 4102], [3956, 4155], [3956, 4156], [3957, 4041], [3957, 4050], [3957, 4108], [3957, 4183], [3957, 4251], [3958, 3959], [3958, 3960], [3958, 4172], [3958, 4174], [3958, 4177], [3958, 4180], [3958, 4182], [3959, 3994], [3959, 4081], [3959, 4127], [3959, 4140], [3959, 4155], [3959, 4173], [3960, 3961], [3960, 4125], [3960, 4182], [3961, 4012], [3961, 4013], [3961, 4041], [3961, 4149], [3961, 4152], [3961, 4154], [3961, 4165], [3961, 4174], [3961, 4175], [3961, 4231], [3961, 4259], [3962, 3963], [3962, 4186], [3962, 4208], [3963, 3980], [3963, 4132], [3963, 4243], [3963, 4248], [3964, 3965], [3964, 4083], [3964, 4200], [3964, 4233], [3965, 4152], [3965, 4154], [3965, 4200], [3965, 4211], [3965, 4222], [3965, 4230], [3966, 3967], [3966, 3981], [3966, 4125], [3966, 4157], [3966, 4203], [3967, 3982], [3967, 4018], [3967, 4158], [3967, 4203], [3967, 4251], [3968, 3969], [3968, 3983], [3968, 4170], [3968, 4207], [3969, 3970], [3969, 3982], [3969, 3995], [3969, 4049], [3969, 4160], [3969, 4168], [3970, 3990], [3970, 3996], [3970, 4158], [3970, 4164], [3970, 4219], [3970, 4220], [3971, 3972], [3971, 3973], [3971, 3975], [3971, 4045], [3971, 4049], [3971, 4058], [3971, 4217], [3971, 4218], [3971, 4219], [3971, 4221], [3971, 4223], [3972, 3973], [3972, 3976], [3972, 4034], [3972, 4050], [3972, 4205], [3973, 3974], [3973, 4139], [3975, 3976], [3975, 4017], [3975, 4111], [3975, 4140], [3975, 4155], [3976, 4009], [3976, 4079], [3976, 4138], [3976, 4205], [3976, 4248], [3977, 4189], [3977, 4225], [3977, 4244], [3978, 3980], [3978, 4028], [3978, 4076], [3978, 4133], [3978, 4203], [3978, 4237], [3978, 4239], [3978, 4241], [3978, 4243], [3978, 4245], [3978, 4247], [3979, 4239], [3980, 4058], [3980, 4134], [3980, 4142], [3980, 4143], [3980, 4145], [3980, 4196], [3980, 4216], [3980, 4245], [3981, 3982], [3981, 4022], [3981, 4248], [3982, 3985], [3982, 4225], [3982, 4238], [3983, 4152], [3983, 4250], [3983, 4251], [3984, 3985], [3984, 4035], [3984, 4168], [3984, 4208], [3984, 4253], [3985, 4000], [3985, 4025], [3985, 4032], [3985, 4050], [3985, 4129], [3985, 4169], [3985, 4213], [3985, 4252], [3985, 4254], [3986, 3987], [3986, 3997], [3986, 4059], [3986, 4219], [3988, 3989], [3988, 4038], [3988, 4078], [3988, 4147], [3988, 4160], [3988, 4172], [3989, 3990], [3989, 4002], [3989, 4025], [3989, 4036], [3989, 4045], [3989, 4049], [3989, 4056], [3990, 3996], [3990, 4258], [3991, 3992], [3991, 4024], [3991, 4083], [3991, 4200], [3991, 4202], [3991, 4244], [3992, 3993], [3992, 4112], [3992, 4246], [3993, 3994], [3993, 4044], [3993, 4046], [3994, 4052], [3994, 4067], [3994, 4090], [3994, 4103], [3994, 4114], [3994, 4130], [3994, 4151], [3994, 4171], [3994, 4180], [3994, 4246], [3995, 3996], [3995, 3998], [3995, 4000], [3995, 4002], [3995, 4003], [3995, 4006], [3995, 4124], [3995, 4126], [3995, 4211], [3995, 4237], [3996, 3997], [3996, 4000], [3996, 4053], [3996, 4074], [3997, 4003], [3998, 3999], [3998, 4006], [3998, 4043], [3998, 4064], [3998, 4132], [3999, 4011], [3999, 4043], [3999, 4063], [4000, 4001], [4000, 4032], [4000, 4098], [4000, 4101], [4000, 4120], [4000, 4127], [4000, 4150], [4000, 4173], [4001, 4033], [4001, 4252], [4002, 4046], [4002, 4063], [4002, 4102], [4002, 4103], [4002, 4120], [4002, 4147], [4002, 4210], [4003, 4004], [4003, 4005], [4003, 4089], [4003, 4106], [4004, 4005], [4004, 4084], [4004, 4105], [4006, 4007], [4006, 4214], [4008, 4010], [4008, 4011], [4008, 4013], [4008, 4048], [4008, 4227], [4008, 4231], [4009, 4032], [4009, 4038], [4009, 4135], [4010, 4027], [4010, 4201], [4010, 4232], [4010, 4234], [4010, 4243], [4011, 4012], [4011, 4013], [4011, 4038], [4011, 4064], [4012, 4225], [4012, 4231], [4012, 4251], [4013, 4119], [4013, 4120], [4013, 4210], [4014, 4015], [4014, 4053], [4014, 4104], [4014, 4233], [4015, 4043], [4015, 4050], [4015, 4078], [4015, 4248], [4015, 4260], [4016, 4017], [4016, 4146], [4018, 4262], [4019, 4020], [4019, 4077], [4019, 4158], [4020, 4021], [4020, 4217], [4020, 4249], [4021, 4217], [4022, 4024], [4022, 4027], [4022, 4057], [4022, 4218], [4023, 4178], [4024, 4025], [4024, 4056], [4024, 4082], [4024, 4129], [4024, 4232], [4025, 4026], [4026, 4028], [4026, 4049], [4026, 4161], [4026, 4185], [4026, 4218], [4026, 4232], [4026, 4237], [4027, 4028], [4027, 4135], [4027, 4243], [4028, 4029], [4028, 4071], [4028, 4229], [4029, 4206], [4030, 4031], [4030, 4086], [4031, 4032], [4031, 4086], [4031, 4099], [4032, 4033], [4032, 4059], [4032, 4099], [4032, 4136], [4032, 4250], [4032, 4260], [4033, 4252], [4034, 4128], [4034, 4221], [4034, 4224], [4034, 4260], [4035, 4036], [4035, 4071], [4035, 4117], [4036, 4052], [4036, 4064], [4036, 4078], [4036, 4082], [4036, 4103], [4036, 4104], [4036, 4117], [4037, 4038], [4037, 4105], [4037, 4118], [4037, 4135], [4037, 4141], [4037, 4200], [4037, 4201], [4038, 4039], [4038, 4225], [4038, 4243], [4038, 4248], [4039, 4043], [4039, 4118], [4039, 4213], [4040, 4058], [4040, 4143], [4040, 4192], [4041, 4042], [4041, 4148], [4041, 4186], [4041, 4241], [4041, 4242], [4041, 4250], [4042, 4172], [4042, 4174], [4042, 4250], [4043, 4211], [4043, 4213], [4043, 4216], [4044, 4045], [4044, 4052], [4044, 4075], [4044, 4257], [4045, 4046], [4045, 4220], [4045, 4257], [4046, 4210], [4046, 4233], [4047, 4050], [4047, 4088], [4047, 4178], [4047, 4179], [4047, 4214], [4047, 4226], [4048, 4049], [4048, 4081], [4048, 4120], [4049, 4104], [4049, 4194], [4050, 4051], [4050, 4128], [4050, 4213], [4050, 4225], [4050, 4241], [4051, 4052], [4051, 4078], [4051, 4170], [4051, 4171], [4051, 4205], [4052, 4082], [4052, 4114], [4053, 4055], [4053, 4098], [4053, 4163], [4054, 4069], [4055, 4060], [4055, 4204], [4055, 4215], [4055, 4257], [4056, 4090], [4056, 4113], [4056, 4114], [4056, 4194], [4056, 4258], [4057, 4058], [4057, 4102], [4057, 4121], [4057, 4180], [4058, 4122], [4058, 4148], [4058, 4153], [4058, 4181], [4058, 4196], [4058, 4207], [4058, 4218], [4059, 4060], [4059, 4065], [4059, 4091], [4059, 4186], [4059, 4219], [4059, 4234], [4059, 4250], [4060, 4078], [4060, 4117], [4060, 4172], [4060, 4199], [4060, 4204], [4060, 4224], [4060, 4260], [4061, 4080], [4061, 4087], [4061, 4091], [4062, 4063], [4062, 4111], [4062, 4243], [4063, 4112], [4063, 4132], [4063, 4243], [4064, 4065], [4064, 4068], [4064, 4083], [4064, 4109], [4064, 4131], [4064, 4138], [4064, 4196], [4064, 4248], [4065, 4066], [4065, 4067], [4065, 4109], [4065, 4130], [4066, 4067], [4066, 4103], [4066, 4234], [4067, 4076], [4067, 4083], [4067, 4173], [4067, 4193], [4067, 4245], [4068, 4069], [4068, 4070], [4068, 4116], [4068, 4139], [4068, 4261], [4069, 4070], [4071, 4182], [4071, 4183], [4072, 4116], [4072, 4257], [4073, 4255], [4074, 4113], [4074, 4126], [4074, 4127], [4074, 4155], [4074, 4158], [4074, 4258], [4075, 4076], [4075, 4260], [4076, 4172], [4076, 4180], [4076, 4186], [4076, 4207], [4077, 4078], [4077, 4095], [4077, 4156], [4077, 4157], [4077, 4215], [4078, 4079], [4078, 4216], [4079, 4095], [4079, 4138], [4080, 4088], [4080, 4100], [4080, 4101], [4080, 4107], [4081, 4082], [4081, 4245], [4082, 4083], [4082, 4235], [4083, 4188], [4085, 4086], [4085, 4212], [4086, 4106], [4086, 4226], [4087, 4171], [4088, 4095], [4089, 4236], [4089, 4255], [4090, 4092], [4091, 4092], [4091, 4093], [4091, 4094], [4091, 4097], [4092, 4093], [4092, 4137], [4094, 4250], [4095, 4096], [4096, 4131], [4096, 4138], [4096, 4156], [4096, 4183], [4096, 4197], [4097, 4151], [4097, 4255], [4098, 4100], [4098, 4149], [4100, 4101], [4100, 4149], [4101, 4150], [4102, 4103], [4103, 4130], [4103, 4155], [4104, 4196], [4104, 4198], [4104, 4235], [4105, 4232], [4106, 4162], [4107, 4108], [4109, 4110], [4110, 4245], [4110, 4247], [4111, 4112], [4111, 4113], [4111, 4170], [4112, 4113], [4112, 4215], [4112, 4246], [4113, 4114], [4113, 4155], [4114, 4158], [4114, 4170], [4115, 4174], [4115, 4176], [4115, 4247], [4116, 4146], [4117, 4158], [4117, 4183], [4117, 4198], [4117, 4220], [4118, 4202], [4118, 4261], [4119, 4228], [4120, 4130], [4120, 4186], [4121, 4122], [4121, 4123], [4121, 4180], [4122, 4123], [4122, 4181], [4123, 4187], [4123, 4228], [4123, 4261], [4124, 4125], [4124, 4160], [4124, 4185], [4125, 4140], [4125, 4223], [4126, 4127], [4126, 4132], [4126, 4168], [4126, 4238], [4127, 4129], [4127, 4136], [4128, 4129], [4128, 4202], [4128, 4259], [4129, 4136], [4129, 4202], [4130, 4131], [4131, 4132], [4131, 4141], [4131, 4197], [4133, 4134], [4133, 4180], [4133, 4238], [4133, 4246], [4133, 4259], [4134, 4153], [4134, 4197], [4134, 4215], [4134, 4246], [4135, 4136], [4135, 4229], [4136, 4175], [4138, 4139], [4138, 4196], [4140, 4171], [4140, 4223], [4141, 4142], [4141, 4144], [4141, 4248], [4142, 4145], [4143, 4192], [4144, 4145], [4144, 4146], [4145, 4146], [4146, 4256], [4147, 4148], [4147, 4186], [4147, 4193], [4147, 4234], [4147, 4243], [4148, 4152], [4149, 4150], [4149, 4180], [4149, 4186], [4151, 4193], [4151, 4246], [4152, 4153], [4152, 4210], [4153, 4197], [4153, 4221], [4153, 4222], [4154, 4223], [4154, 4225], [4154, 4242], [4154, 4259], [4155, 4156], [4156, 4157], [4156, 4183], [4157, 4158], [4157, 4204], [4158, 4163], [4158, 4168], [4158, 4198], [4158, 4204], [4158, 4258], [4159, 4160], [4159, 4162], [4159, 4163], [4159, 4165], [4159, 4168], [4159, 4230], [4159, 4231], [4160, 4161], [4160, 4164], [4160, 4223], [4160, 4225], [4161, 4162], [4161, 4218], [4162, 4222], [4162, 4232], [4163, 4164], [4163, 4183], [4163, 4197], [4163, 4251], [4164, 4196], [4165, 4166], [4166, 4167], [4166, 4209], [4166, 4247], [4167, 4247], [4168, 4169], [4168, 4194], [4168, 4253], [4169, 4254], [4170, 4171], [4170, 4204], [4171, 4207], [4172, 4173], [4172, 4203], [4172, 4223], [4174, 4175], [4175, 4182], [4175, 4229], [4175, 4250], [4176, 4190], [4176, 4191], [4176, 4206], [4176, 4262], [4177, 4178], [4178, 4179], [4180, 4181], [4182, 4183], [4182, 4229], [4184, 4226], [4185, 4237], [4185, 4244], [4187, 4188], [4189, 4190], [4189, 4194], [4189, 4196], [4190, 4191], [4190, 4194], [4191, 4195], [4191, 4196], [4192, 4196], [4193, 4245], [4194, 4195], [4194, 4221], [4194, 4232], [4195, 4196], [4196, 4197], [4198, 4219], [4198, 4221], [4199, 4224], [4199, 4227], [4200, 4201], [4200, 4202], [4201, 4232], [4201, 4233], [4202, 4213], [4203, 4204], [4203, 4242], [4205, 4224], [4207, 4223], [4207, 4259], [4208, 4210], [4208, 4213], [4208, 4215], [4210, 4211], [4211, 4212], [4211, 4227], [4213, 4214], [4215, 4216], [4217, 4240], [4217, 4256], [4217, 4262], [4218, 4222], [4218, 4240], [4219, 4220], [4221, 4222], [4222, 4230], [4222, 4232], [4223, 4224], [4224, 4229], [4225, 4244], [4225, 4251], [4227, 4229], [4227, 4230], [4228, 4247], [4229, 4230], [4230, 4231], [4231, 4232], [4232, 4235], [4232, 4238], [4232, 4244], [4233, 4234], [4233, 4235], [4234, 4235], [4235, 4236], [4237, 4238], [4237, 4239], [4238, 4244], [4239, 4240], [4241, 4242], [4241, 4259], [4243, 4244], [4245, 4246], [4253, 4254], [4253, 4255], [4254, 4255], [4256, 4258], [4257, 4260]], \"cross\": 9119}]"
open('/tmp/newcand2.json','w').write(BUNDLE)
import sys; sys.path.insert(0,'/tmp')
print('bundle:', len(open('/tmp/newcand2.json').read()), 'bytes')


In [ ]:
import sys, traceback
sys.argv=["nn_siren.py","--demo"]
import nn_siren
try:
    _ = nn_siren.main()
    print('HUMO OK')
except Exception:
    traceback.print_exc()
    print('HUMO FALLO')


## C5_874x2+10


In [ ]:
import json, sys, traceback
C = [c for c in json.load(open('/tmp/newcand2.json')) if c['name']=='C5_874x2+10'][0]
print('candidato:', C['name'], 'n=', C['n'], 'E=', C['edges'], 'cross=', C.get('cross'))
open('/tmp/c.vtx.json','w').write(json.dumps(C['vtx']))
open('/tmp/c.edge','w').write('\n'.join(f'e {a} {b}' for a,b in C['edge']))
sys.argv=['nn_siren.py','--vtx','/tmp/c.vtx.json','--edges','/tmp/c.edge','--k','5','--restarts','8','--steps','4000','--kicks','50','--kick-size','15','--tabu-iters','30000','--slim-rounds','5','--slim-radius','2','--seed','7']
import nn_siren
try:
    _ = nn_siren.main()
    print('CRIBA C5_874x2+10 OK')
except Exception:
    traceback.print_exc()
    print('CRIBA C5_874x2+10 FALLO')


## C6_509x2+10


In [ ]:
import json, sys, traceback
C = [c for c in json.load(open('/tmp/newcand2.json')) if c['name']=='C6_509x2+10'][0]
print('candidato:', C['name'], 'n=', C['n'], 'E=', C['edges'], 'cross=', C.get('cross'))
open('/tmp/c.vtx.json','w').write(json.dumps(C['vtx']))
open('/tmp/c.edge','w').write('\n'.join(f'e {a} {b}' for a,b in C['edge']))
sys.argv=['nn_siren.py','--vtx','/tmp/c.vtx.json','--edges','/tmp/c.edge','--k','5','--restarts','8','--steps','4000','--kicks','50','--kick-size','15','--tabu-iters','30000','--slim-rounds','5','--slim-radius','2','--seed','7']
import nn_siren
try:
    _ = nn_siren.main()
    print('CRIBA C6_509x2+10 OK')
except Exception:
    traceback.print_exc()
    print('CRIBA C6_509x2+10 FALLO')


## C7_G3x2+10


In [ ]:
import json, sys, traceback
C = [c for c in json.load(open('/tmp/newcand2.json')) if c['name']=='C7_G3x2+10'][0]
print('candidato:', C['name'], 'n=', C['n'], 'E=', C['edges'], 'cross=', C.get('cross'))
open('/tmp/c.vtx.json','w').write(json.dumps(C['vtx']))
open('/tmp/c.edge','w').write('\n'.join(f'e {a} {b}' for a,b in C['edge']))
sys.argv=['nn_siren.py','--vtx','/tmp/c.vtx.json','--edges','/tmp/c.edge','--k','5','--restarts','8','--steps','6000','--kicks','50','--kick-size','15','--tabu-iters','30000','--slim-rounds','5','--slim-radius','2','--seed','7']
import nn_siren
try:
    _ = nn_siren.main()
    print('CRIBA C7_G3x2+10 OK')
except Exception:
    traceback.print_exc()
    print('CRIBA C7_G3x2+10 FALLO')
